In [ ]:
# ============================================================
# 0. INSTALACIÓN
# ============================================================
# gurobipy instala la API oficial de Gurobi Optimizer.
!pip -q install --upgrade gurobipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 66.3 MB/s eta 0:00:00


## Licencia académica de Gurobi

La configuración predeterminada usa una licencia **Academic WLS**, apropiada para Google Colab y modelos grandes. El notebook nunca guarda las credenciales dentro del archivo ni en los resultados.

Antes de ejecutar la optimización:

1. Genera una licencia **Academic WLS** en el portal de Gurobi usando tu cuenta académica.
2. En Colab, guarda estos tres secretos en el panel **Secrets**:
   - `GUROBI_WLS_ACCESS_ID`
   - `GUROBI_WLS_SECRET`
   - `GUROBI_WLS_LICENSE_ID`
3. Si los secretos no existen, el notebook los solicitará de forma oculta mediante `getpass`.

Para usar una licencia Named-User local, cambia `GUROBI_LICENSE_MODE = "FILE"` y coloca `gurobi.lic` en la ruta indicada. `AUTO` permite que Gurobi busque una licencia ya instalada en las ubicaciones estándar.

> La licencia académica sólo debe utilizarse para investigación, docencia o trabajo académico, no para operación comercial.


In [ ]:
from google.colab import drive
import shutil
import os

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir las rutas
# Asegúrate de que la ruta coincida exactamente con la estructura de tu Drive
ruta_origen = '/content/drive/MyDrive/2026/FEP Ing. Civil Quimica/Datos_Biometano.zip'
ruta_destino = '/content/Datos_Biometano.zip'

# 3. Copiar el archivo a la carpeta /content
if os.path.exists(ruta_origen):
    shutil.copy(ruta_origen, ruta_destino)
    print("Archivo copiado exitosamente a /content")
else:
    print(f"No se encontró el archivo en: {ruta_origen}. Verifica la ruta.")

# 4. (Opcional) Descomprimir el archivo
# !unzip /content/Datos_Biometano.zip -d /content/datos_extraidos

Mounted at /content/drive
Archivo copiado exitosamente a /content


## Variables Relevantes

In [ ]:
NUMERO_TOTAL_PLANTAS = False

NUMERO_EMPRESAS_A_UTILIZAR = 5

NUMERO_CLIENTES_A_UTILIZAR = False

compra_porcentaje = True

venta_porcentaje = False

NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR = False

ALPHA_COMPRA_PORCENTAJE = 0.25

BETA_VENTA_PORCENTAJE = 0.25

LARGO_MAXIMO_GASODUCTO_Corto = 10

TIME_LIMIT_SECONDS = 3000

Capacidad_Camiones_Alterada = False
CAPACIDAD_CAMIONES_MANUAL = {
    "BIO": 0,
    "GNL": 1000,
    "GNC": 1000,
   "BC":  0,
}

# ============================================================
# CAPACIDADES DE PLANTAS DEFINIDAS MANUALMENTE
# Sobrescriben tipos_plantas.csv sin modificar el ZIP
# ============================================================

Capacidad_Plantas_Alterada = True
CAPACIDAD_PLANTAS_MANUAL = {
    "L": 9999999,   # BioGNL
    "BG": 9999999,  # BioGBC / GasBC
}



# Otras Variables

In [ ]:
# ============================================================
# 1. CONFIGURACIÓN PRINCIPAL
# ============================================================
from pathlib import Path
import math

NOMBRE_ZIP_DATOS = "Datos_Biometano_Excedentes_Actualizados_precio_mercado_biogas_crudo.zip"
CARPETA_RAIZ_DATOS = "Datos_Biometano_Corredores_Bidireccionales_COMPLETO"

CONTENT = Path("/content")
DIR_TRABAJO = CONTENT / "biometano_modelo_pdf"
DIR_CACHE = CONTENT / "cache_google_maps_biometano"
DIR_RESULTADOS = CONTENT / "Resultados_Biometano_Modelo_PDF"

# ============================================================
# LICENCIA Y MOTOR PROFESIONAL GUROBI
# ============================================================
# "WLS"  : recomendado para Google Colab con licencia Academic WLS completa.
# "FILE" : usa un archivo gurobi.lic, por ejemplo una licencia Named-User local.
# "AUTO" : deja que Gurobi encuentre una licencia ya instalada; puede caer en
#          la licencia limitada incluida con gurobipy si no existe otra licencia.
GUROBI_LICENSE_MODE = "WLS"
GUROBI_LICENSE_FILE = CONTENT / "gurobi.lic"

# Nombres de secretos compatibles con Colab Secrets y variables de entorno.
GUROBI_WLS_ACCESS_ID_SECRET_NAME = "GUROBI_WLS_ACCESS_ID"
GUROBI_WLS_SECRET_SECRET_NAME = "GUROBI_WLS_SECRET"
GUROBI_WLS_LICENSE_ID_SECRET_NAME = "GUROBI_WLS_LICENSE_ID"

# Auditoría del optimizador.
GUROBI_GUARDAR_LOG = True
GUROBI_EXPORTAR_MODELO_MPS = False
GUROBI_CALCULAR_IIS_SI_INFACTIBLE = False

NOMBRE_ZIP_CACHE_API = "Cache_Google_Routes_Biometano_COMPLETA.zip"
NOMBRE_ZIP_DATOS_CON_CACHE = "Datos_Biometano_Corredores_Bidireccionales_TRANSPORTE_VARIABLE_COMPLETO.zip"

# La caché general no se completa mediante internet salvo las precargas
# explícitas de pares faltantes. Las API keys se piden con getpass y nunca se
# guardan en el notebook, en los resultados ni en el ZIP definitivo.
NO_CONSULTAR_API = True
USAR_CACHE_LOCAL_AUNQUE_HAYA_VENCIDO = False

# Precarga dirigida cliente -> cliente ya existente en el modelo.
CONSULTAR_API_CLIENTE_CLIENTE_SI_FALTA = False

# Matriz y trayectorias explícitas empresa -> empresa. Se consultan solo
# los pares faltantes y se guardan en archivos dedicados dentro del ZIP.
CONSULTAR_API_EMPRESA_EMPRESA_SI_FALTA = False

# Las rutas empresa -> cliente ya vienen precargadas en E2C. No se reconstruyen
# ni se sustituyen por un sitio estratégico: cada ABC|NE_Ei|Cj usa exactamente
# la distancia y polilínea E2C|Ei|Cj del ZIP.
EXIGIR_EMPRESA_CLIENTE_E2C_COMPLETO = False

# Fallback para cualquier arco nuevo de nodos NE_E que no pueda reutilizar una
# ruta existente por igualdad exacta de coordenadas. En los datos entregados,
# la cadena principal empresa -> empresa queda cubierta por la matriz E2E.
CONSULTAR_API_NODOS_EMPRESA_SI_FALTA = False


# Consultas nuevas asociadas exclusivamente a los puntos de gasoducto.
CONSULTAR_API_EMPRESA_GASODUCTO_SI_FALTA = False
CONSULTAR_API_UBICACION_GASODUCTO_SI_FALTA = False
CONSULTAR_API_GASODUCTO_CLIENTE_SI_FALTA = False

# True elimina de la caché únicamente EIG/UIG/GJC y vuelve a consultar Google
# Routes con las coordenadas actuales de los 18 extremos direccionales de los
# nueve gasoductos. No elimina C2C, E2E ni E2C.
RECONSTRUIR_RUTAS_GASODUCTOS_DESDE_CERO = False

# Nunca se construyen ni consultan rutas Google gasoducto -> gasoducto.
CONSULTAR_API_GASODUCTO_GASODUCTO = False

# El ZIP definitivo se crea y descarga siempre al terminar la precarga, incluso
# cuando no fue necesario consumir nuevamente la API.
DESCARGAR_ZIP_DEFINITIVO_COMPLETO = False
NOMBRE_ZIP_DATOS_DEFINITIVO = "Datos_Biometano_9_Gasoductos_RUTAS_ACTUALIZADAS.zip"

# Control opcional del número TOTAL de unidades de planta activas.
# False: el MILP decide libremente.
# 1, 2, 3, ...: obliga a activar exactamente esa cantidad total, sumando todos los tipos y sitios.


# Control GLOBAL opcional del número EXACTO TOTAL de empresas proveedoras a utilizar.
# False: el MILP decide libremente cuántas empresas utiliza en toda la solución.
# 1, 2, 3, ...: obliga a utilizar exactamente esa cantidad TOTAL de empresas
# activadas por binaria, sumando toda la solución.
# IMPORTANTE: NO exige que cada cliente sea abastecido por ese número de empresas.





# Control GLOBAL opcional del número EXACTO TOTAL de clientes a utilizar.
# False: el MILP decide libremente cuántos clientes atiende en toda la solución.
# 1, 2, 3, ...: obliga a atender exactamente esa cantidad TOTAL de clientes
# activados por binaria, sumando toda la solución.
# IMPORTANTE: es una sola cardinalidad global; no se aplica por empresa ni por ruta.


# Igualdad porcentual opcional entre participantes ACTIVADOS.
# True: cada empresa proveedora activada suministra el mismo porcentaje de su
# capacidad máxima disponible .
# False: el MILP decide independientemente cuánto comprar a cada empresa.


# True: cada cliente activado recibe el mismo porcentaje de su demanda máxima
# disponible .
# False: el MILP decide independientemente qué porcentaje vender a cada cliente.


# Control de rutas compartidas.
# True habilita giras Origen -> A -> B -> Origen además de viajes directos.
PERMITIR_RUTAS_COMPARTIDAS = True

# Restricción física de modos de transporte solicitada:
# - el biogás crudo NO puede transportarse en vehículos; sólo puede moverse
#   entre ubicaciones mediante gasoductos cortos (la transferencia local en la
#   misma ubicación sigue siendo local y no constituye transporte).
# - el GasBC NO puede transportarse en vehículos; sólo puede moverse entre
#   ubicaciones mediante gasoductos cortos o gasoductos convencionales.
# BioGNL y BioGNC conservan exactamente su transporte vehicular existente.
PERMITIR_VEHICULOS_BIOGAS_CRUDO = False#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
PERMITIR_VEHICULOS_GASBC = False#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# Excepción puntual solicitada: se mantiene prohibido el transporte vehicular
# general de GasBC, pero se permite retirar GasBC desde la SALIDA de un gasoducto
# convencional y llevarlo a un cliente exclusivamente como HUB de tránsito. La
# entrega final ZG por carretera continúa prohibida como antes.
PERMITIR_VEHICULO_GASBC_SALIDA_CONVENCIONAL_A_CLIENTE_HUB = False#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# El biogás crudo transportado por vehículos puede visitar como máximo dos
# plantas BG por viaje: Empresa -> BG_A -> BG_B -> Empresa. El valor 2 se
# mantiene explícito para impedir que se generen recorridos de tres o más paradas.
MAXIMO_PARADAS_BIOGAS_CRUDO_VEHICULO = 2
if MAXIMO_PARADAS_BIOGAS_CRUDO_VEHICULO != 2:
    raise ValueError("MAXIMO_PARADAS_BIOGAS_CRUDO_VEHICULO debe mantenerse en 2.")

# True: sólo se generan rutas de dos paradas.
# False: el MILP puede escoger rutas directas o rutas compartidas.
FORZAR_EXACTAMENTE_DOS_PARADAS = False

# En rutas compartidas de dos paradas, cada parada debe descargar
# una cantidad minima por viaje para representar dos destinos reales.
# Usa False o 0.0 si quieres permitir paradas con descarga cero.
DESCARGA_MINIMA_POR_PARADA_POR_VIAJE = 0

# El remanente después de la última parada puede volver al origen.
PERMITIR_RETORNO_CON_REMANENTE = False

# Rompe la degeneración de cargar producto innecesario y devolverlo. Es tan
# pequeña que no altera materialmente la economía de las rutas.
PENALIZACION_RETORNO_REMANENTE_USD_UNIDAD = False

if FORZAR_EXACTAMENTE_DOS_PARADAS and not PERMITIR_RUTAS_COMPARTIDAS:
    raise ValueError(
        "FORZAR_EXACTAMENTE_DOS_PARADAS=True requiere "
        "PERMITIR_RUTAS_COMPARTIDAS=True."
    )
if DESCARGA_MINIMA_POR_PARADA_POR_VIAJE < 0:
    raise ValueError(
        "DESCARGA_MINIMA_POR_PARADA_POR_VIAJE debe ser mayor o igual que cero."
    )
if PENALIZACION_RETORNO_REMANENTE_USD_UNIDAD < 0:
    raise ValueError(
        "PENALIZACION_RETORNO_REMANENTE_USD_UNIDAD no puede ser negativa."
    )


# Control opcional de exclusión de empresas según su oferta ORIGINAL.
# False: considera todas las empresas.
# 1: excluye la empresa con mayor capacidad_max_biogas_nm3.
# 2, 3, ...: excluye esa cantidad, empezando por las de mayor oferta.


# ============================================================
# PRECIOS ENDOGENOS DISCRETIZADOS (MILP)
# ============================================================
# El PDF reemplaza los factores anteriores por funciones de
# precio dependientes de la cantidad comprada/vendida.
#
# Compra de biogas crudo al proveedor i:
#   P_i(q) = P0_i * (1 + alpha * max(0, q - E_i) / (Qmax_i - E_i))
#
# Venta al cliente j:
#   P_j(q) = max(P_MIN, P0_j * (1 - beta * max(0, q - DNC_j) / (Dmax_j - DNC_j)))
#
# Para mantener el modelo como MILP, el notebook no multiplica precio(q)
# por q dentro del solver. En su lugar crea variables continuas por tramo:
#   QBUY_SEG[i,k]  = compra al proveedor i en el tramo k
#   QSELL_SEG[j,k] = venta al cliente j en el tramo k
# y usa en la funcion objetivo el costo/ingreso incremental precalculado
# de cada tramo.

COL_PRECIO_BIOGAS_CRUDO = "precio_mercado_biogas_crudo_usd_nm3"
COL_OFERTA_EXCEDENTE_BIOGAS = "oferta_excedente_biogas_crudo_nm3_dia"
COL_DEMANDA_NO_CUBIERTA = "demanda_no_cubrida_nm3_dia"

# Discretizacion inteligente: punto exacto en el umbral y tramos mas densos
# inmediatamente despues del excedente/demanda no cubierta, donde cambia la
# pendiente economica.
N_TRAMOS_PRECIO = 8
FRACCIONES_TRAMOS_PRECIO = (0.05, 0.10, 0.20, 0.35, 0.55, 0.75, 1.00)

# Porcentaje del precio base aplicado al alcanzar la cantidad maxima. Si se cuenta con valores
# calibrados externos, estos dos parametros son los que deben reemplazarse.

PRECIO_VENTA_MIN_USD_NM3 = 0.0

# La transferencia local se determina mediante relaciones explícitas de nodo físico,
# no mediante tolerancias ni comparaciones de latitud/longitud dentro del MILP.
CRITERIO_TRANSFERENCIA_LOCAL = "MISMO_NODO_FISICO_DECLARADO"

# En los nodos exactos de clientes se puede instalar BG y, si el modelo lo
# decide, transformar localmente su GasBC a BioGNL/BioGNC mediante L/C antes de
# reconvertirlo con RL/RC. Esto permite concentrar biogás crudo de varias empresas
# en la ubicación del cliente sin imponer que la cadena sea económicamente viable.
TECNOLOGIAS_EN_NODO_CLIENTE = ("BG", "L", "RL", "C", "RC")

# ============================================================
# GASODUCTOS CORTOS DE BIOGÁS CRUDO
# EMPRESA -> NODO FÍSICO; CLIENTE/HUB -> CLIENTE; EMPRESA/HUB -> NODO FÍSICO
# ============================================================
# Estos candidatos NO se leen ni se escriben dentro del ZIP. Antes de construir
# el MILP, el notebook calcula desde cero la distancia geodésica recta desde cada
# empresa hacia: clientes, ubicaciones posibles P#, ubicaciones de empresas y
# entradas de gasoductos convencionales. En el destino puede instalarse una BG y
# concentrarse simultáneamente el biogás crudo procedente de varias empresas.
# Se crea una alternativa únicamente cuando la distancia calculada es menor que
# LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM. El trazado es siempre una línea recta.
HABILITAR_GASODUCTOS_BIOGAS_CORTOS = True
# NUEVO: un cliente puede actuar como HUB de biogás crudo y reenviarlo por
# gasoductos cortos a otros clientes sin obligar a procesarlo en el primer cliente.
HABILITAR_RELEVO_BIOGAS_CRUDO_CLIENTE_CLIENTE = True
# NUEVO: una ubicación física de empresa NE_E puede actuar como HUB de biogás
# crudo. El flujo recibido desde otra empresa puede salir por otro gasoducto corto
# sin contabilizarse como producción de la empresa intermedia.
HABILITAR_RELEVO_BIOGAS_CRUDO_EN_EMPRESAS = True
# NUEVO: una ubicación potencial P# puede actuar como HUB de biogás crudo sin
# procesarlo obligatoriamente. El biogás que llega a P# puede continuar por un
# segundo (o posterior) gasoducto corto hacia otro nodo dentro del largo máximo.
HABILITAR_RELEVO_BIOGAS_CRUDO_EN_UBICACIONES_P = True
# NUEVO: GasBC recibido en un cliente puede mantenerse como GasBC y reenviarse
# por otro gasoducto corto a uno o varios clientes aguas abajo. Esto incluye GasBC
# recibido desde un gasoducto convencional por carretera o por gasoducto corto, y
# GasBC generado por RL/RC que deba continuar por un segundo salto.
HABILITAR_REENVIO_GASBC_EN_CLIENTES = True
# NUEVO: una ubicación física de empresa NE_E puede recibir GasBC desde la salida
# de un gasoducto convencional y actuar como HUB puramente físico. No es necesario
# que la empresa produzca ni que exista una planta BG para reenviar ese GasBC: el
# flujo entra al balance nodal de GasBC y puede repartirse por varios gasoductos
# cortos hacia clientes/nodos aguas abajo. Incluye coincidencias de 0 km.
HABILITAR_REENVIO_GASBC_EN_UBICACIONES_EMPRESA = True
# NUEVO: una RL/RC instalada exactamente en el nodo de entrada de una dirección
# del gasoducto convencional puede inyectar allí el GasBC reconvertido.


HABILITAR_INYECCION_RL_RC_GASODUCTO_CONVENCIONAL = True
COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA = 500.0  # costo fijo por activar cada gasoducto corto de biogas crudo
COSTO_VARIABLE_GASODUCTO_BIOGAS_CORTO_USD_NM3_KM = 0.00050
CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA = 185_000#185_000#185_000.0
LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM = LARGO_MAXIMO_GASODUCTO_Corto#40#40.0

if COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA < 0:
    raise ValueError("El costo fijo del gasoducto corto no puede ser negativo.")
if COSTO_VARIABLE_GASODUCTO_BIOGAS_CORTO_USD_NM3_KM < 0:
    raise ValueError("El costo variable del gasoducto corto no puede ser negativo.")
if CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA <= 0:
    raise ValueError("La capacidad del gasoducto corto debe ser positiva.")
if LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM <= 0:
    raise ValueError("El largo máximo del gasoducto corto debe ser positivo.")

# Gasoductos cortos de GasBC. Son alternativas físicas distintas de los camiones:
#   1) BG situada en una empresa -> entrada del corredor principal;
#   2) salida del corredor principal -> cliente o ubicación física de empresa NE_E;
#   3) BG situada en cualquier nodo físico permitido -> uno o varios clientes;
#   4) BG en empresa -> ubicación posible (alternativa previa);
#   5) GasBC producido por RL/RC en cualquier nodo compatible -> ubicación P#.
# Cada conexión posee su propia binaria, por lo que una misma fuente de GasBC
# concentradora puede activar múltiples gasoductos cortos hacia clientes distintos.
# Estas alternativas no activan ni dibujan un corredor convencional por sí solas.
# En todos estos casos se transporta GasBC, nunca biogás crudo.
HABILITAR_GASODUCTOS_GASBC_CORTOS_EMPRESA_ENTRADA = True
HABILITAR_GASODUCTOS_GASBC_CORTOS_SALIDA_CLIENTE = True
HABILITAR_GASODUCTOS_GASBC_CORTOS_EMPRESA_DESTINO = True
COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA = 625.0  # costo fijo por activar cada gasoducto corto de GasBC
COSTO_VARIABLE_GASODUCTO_GASBC_CORTO_USD_NM3_KM = 0.00120
CAPACIDAD_GASODUCTO_GASBC_CORTO_NM3_DIA = CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA
LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM = LARGO_MAXIMO_GASODUCTO_Corto

if COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA < 0:
    raise ValueError("El costo fijo del gasoducto corto de GasBC no puede ser negativo.")
if COSTO_VARIABLE_GASODUCTO_GASBC_CORTO_USD_NM3_KM < 0:
    raise ValueError("El costo variable del gasoducto corto de GasBC no puede ser negativo.")
if CAPACIDAD_GASODUCTO_GASBC_CORTO_NM3_DIA <= 0:
    raise ValueError("La capacidad del gasoducto corto de GasBC debe ser positiva.")
if LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM <= 0:
    raise ValueError("El largo máximo del gasoducto corto de GasBC debe ser positivo.")

# Compatibilidad amistosa: también se aceptan "False" y enteros positivos escritos como texto.
if isinstance(NUMERO_TOTAL_PLANTAS, str):
    valor_plantas = NUMERO_TOTAL_PLANTAS.strip()
    if valor_plantas.lower() == "false":
        NUMERO_TOTAL_PLANTAS = False
    elif valor_plantas.isdigit():
        NUMERO_TOTAL_PLANTAS = int(valor_plantas)
    else:
        raise ValueError(
            "NUMERO_TOTAL_PLANTAS debe ser False o un entero positivo: 1, 2, 3, ..."
        )

if NUMERO_TOTAL_PLANTAS is not False:
    if type(NUMERO_TOTAL_PLANTAS) is not int or NUMERO_TOTAL_PLANTAS < 1:
        raise ValueError(
            "NUMERO_TOTAL_PLANTAS debe ser False o un entero positivo: 1, 2, 3, ..."
        )

# Compatibilidad amistosa para el número exacto de empresas a utilizar.
if isinstance(NUMERO_EMPRESAS_A_UTILIZAR, str):
    valor_empresas = NUMERO_EMPRESAS_A_UTILIZAR.strip()
    if valor_empresas.lower() == "false":
        NUMERO_EMPRESAS_A_UTILIZAR = False
    elif valor_empresas.isdigit():
        NUMERO_EMPRESAS_A_UTILIZAR = int(valor_empresas)
    else:
        raise ValueError(
            "NUMERO_EMPRESAS_A_UTILIZAR debe ser False o un entero positivo: 1, 2, 3, ..."
        )

if NUMERO_EMPRESAS_A_UTILIZAR is not False:
    if type(NUMERO_EMPRESAS_A_UTILIZAR) is not int or NUMERO_EMPRESAS_A_UTILIZAR < 1:
        raise ValueError(
            "NUMERO_EMPRESAS_A_UTILIZAR debe ser False o un entero positivo: 1, 2, 3, ..."
        )

# Compatibilidad amistosa para el número exacto GLOBAL de clientes a utilizar.
if isinstance(NUMERO_CLIENTES_A_UTILIZAR, str):
    valor_clientes = NUMERO_CLIENTES_A_UTILIZAR.strip()
    if valor_clientes.lower() == "false":
        NUMERO_CLIENTES_A_UTILIZAR = False
    elif valor_clientes.isdigit():
        NUMERO_CLIENTES_A_UTILIZAR = int(valor_clientes)
    else:
        raise ValueError(
            "NUMERO_CLIENTES_A_UTILIZAR debe ser False o un entero positivo: 1, 2, 3, ..."
        )

if NUMERO_CLIENTES_A_UTILIZAR is not False:
    if type(NUMERO_CLIENTES_A_UTILIZAR) is not int or NUMERO_CLIENTES_A_UTILIZAR < 1:
        raise ValueError(
            "NUMERO_CLIENTES_A_UTILIZAR debe ser False o un entero positivo: 1, 2, 3, ..."
        )

# Compatibilidad amistosa para los interruptores booleanos de igualdad porcentual.
def _normalizar_booleano_configuracion(nombre, valor):
    if type(valor) is bool:
        return valor
    if isinstance(valor, str):
        texto = valor.strip().lower()
        if texto == "true":
            return True
        if texto == "false":
            return False
    raise ValueError(f"{nombre} debe ser True o False.")

compra_porcentaje = _normalizar_booleano_configuracion(
    "compra_porcentaje", compra_porcentaje
)
venta_porcentaje = _normalizar_booleano_configuracion(
    "venta_porcentaje", venta_porcentaje
)

# Compatibilidad amistosa para la exclusión de empresas.
if isinstance(NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR, str):
    valor_exclusion = NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR.strip()
    if valor_exclusion.lower() == "false":
        NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR = False
    elif valor_exclusion.isdigit():
        NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR = int(valor_exclusion)
    else:
        raise ValueError(
            "NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR debe ser False o un entero positivo: 1, 2, 3, ..."
        )

if NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR is not False:
    if (
        type(NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR) is not int
        or NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR < 1
    ):
        raise ValueError(
            "NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR debe ser False o un entero positivo: 1, 2, 3, ..."
        )

print(
    "Número total de plantas:",
    "decisión libre del MILP" if NUMERO_TOTAL_PLANTAS is False
    else f"exactamente {NUMERO_TOTAL_PLANTAS} unidades activas"
)
print(
    "Número de empresas a utilizar:",
    "decisión libre del MILP" if NUMERO_EMPRESAS_A_UTILIZAR is False
    else f"exactamente {NUMERO_EMPRESAS_A_UTILIZAR} empresa(s) activada(s) por binaria"
)
print(
    "Número de clientes a utilizar:",
    "decisión libre del MILP" if NUMERO_CLIENTES_A_UTILIZAR is False
    else f"exactamente {NUMERO_CLIENTES_A_UTILIZAR} cliente(s) activado(s) por binaria"
)
print(
    "Compra en igual porcentaje:",
    "sí, entre empresas proveedoras activadas" if compra_porcentaje
    else "no, porcentaje libre por empresa"
)
print(
    "Venta en igual porcentaje:",
    "sí, entre clientes activados" if venta_porcentaje
    else "no, porcentaje libre por cliente"
)
print(
    "Exclusión por mayor oferta:",
    "ninguna empresa" if NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR is False
    else f"{NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR} empresa(s) con mayor capacidad original"
)

# Los factores de conversión no se definen aquí: se leen exclusivamente desde el CSV del ZIP.

EPS = 1e-7
# Cantidades inferiores a este valor se consideran cero exclusivamente al
# interpretar y exportar flujos físicos. No modifica demandas, capacidades,
# costos ni cantidades mínimas del problema económico.
TOL_FLUJO_CERO_NM3 = 1e-5
# Evita que un NTR numericamente casi cero transporte flujo cuando
# la capacidad del camion se define absurdamente grande. No agrega
# ninguna capacidad maxima por ruta.
ACTIVAR_BINARIA_GIRA_TRANSPORTE = True
TOL_ENTERO_RESULTADO = 1e-6

MIP_REL_GAP = 0.0000001

# ============================================================
# FUNCIÓN OBJETIVO ECONÓMICA
# ============================================================
# El MILP maximiza exclusivamente el beneficio total:
# ingresos - costos variables - costos fijos de plantas - costos de transporte.
# En este escenario se anulan los costos fijos de activacion de empresas y gasoductos.
# Gurobi resuelve la forma equivalente de minimización costo - ingreso.
# Los valores -1/0 dejan que el motor profesional seleccione automáticamente
# presolve, cortes, heurísticas, algoritmo de relajación y paralelismo.


# Escenario solicitado: no cobrar costos fijos de activacion de empresas.
# Los gasoductos cortos dinamicos SI cobran su costo fijo propio al activar
# la binaria correspondiente en la funcion objetivo del MILP.
# ANULAR_COSTO_FIJO_ACTIVACION_GASODUCTOS solo afecta gasoductos convencionales
# leidos desde el ZIP; no anula COSTO_FIJO_GASODUCTO_*_CORTO_USD_DIA.
ANULAR_COSTO_FIJO_ACTIVACION_EMPRESAS = True
ANULAR_COSTO_FIJO_ACTIVACION_GASODUCTOS = True

SOLVER_OPTIONS = {
    "TimeLimit": float(TIME_LIMIT_SECONDS),
    "MIPGap": float(MIP_REL_GAP),
    "Presolve": -1,
    "Cuts": -1,
    "MIPFocus": 0,
    "Threads": 0,
    "OutputFlag": 1,
    # Reduce residuos de factibilidad en vínculos planta-flujo.
    "FeasibilityTol": 1e-9,
    "IntFeasTol": 1e-9,
    "NumericFocus": 2,
}


Número total de plantas: decisión libre del MILP
Número de empresas a utilizar: exactamente 5 empresa(s) activada(s) por binaria
Número de clientes a utilizar: decisión libre del MILP
Compra en igual porcentaje: sí, entre empresas proveedoras activadas
Venta en igual porcentaje: no, porcentaje libre por cliente
Exclusión por mayor oferta: ninguna empresa


In [ ]:
# ============================================================
# 1B. INSTALACIÓN Y REPARACIÓN AUTOMÁTICA DEL ENTORNO
#     Corrige instalaciones mezcladas de NumPy/SciPy en Google Colab.
# ============================================================
import os
import sys
import subprocess

# Este trío fue fijado conjuntamente para Python 3.12 y evita que SciPy cargue
# contra archivos binarios pertenecientes a otra versión de NumPy.
NUMPY_COMPATIBLE = "2.3.5"
SCIPY_COMPATIBLE = "1.17.1"
PANDAS_COMPATIBLE = "2.3.3"


def _ejecutar_pip(*paquetes, force_reinstall=False):
    comando = [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "--no-cache-dir",
    ]
    if force_reinstall:
        comando.append("--force-reinstall")
    comando.extend(paquetes)
    subprocess.check_call(comando)


def _probar_stack_cientifico():
    prueba = subprocess.run(
        [
            sys.executable,
            "-c",
            (
                "import numpy, scipy, pandas; "
                "from scipy.sparse import coo_matrix; "
                "A=coo_matrix(([1.0],([0],[0])),shape=(1,1)).tocsr(); "
                "print(numpy.__version__, scipy.__version__, pandas.__version__)"
            ),
        ],
        text=True,
        capture_output=True,
    )
    return prueba


# Gurobi no necesita reemplazar NumPy ni SciPy; se instala por separado.
_ejecutar_pip("gurobipy")

_prueba = _probar_stack_cientifico()
if _prueba.returncode != 0:
    print("Se detectó una instalación inconsistente de NumPy/SciPy/Pandas.")
    print((_prueba.stderr or _prueba.stdout).strip())
    print("Reinstalando conjuntamente las versiones compatibles...")
    _ejecutar_pip(
        f"numpy=={NUMPY_COMPATIBLE}",
        f"scipy=={SCIPY_COMPATIBLE}",
        f"pandas=={PANDAS_COMPATIBLE}",
        force_reinstall=True,
    )
    print(
        "Entorno reparado. Colab se reiniciará automáticamente. "
        "Cuando vuelva a conectarse, ejecuta nuevamente desde la primera celda."
    )
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
else:
    print("Entorno científico correcto:", _prueba.stdout.strip())


Entorno científico correcto: 2.0.2 1.16.3 2.2.3


# 2

In [ ]:
# ============================================================
# 2. LECTURA MANUAL DEL ZIP YA SUBIDO A /content
#    No se usa un método interactivo de subida y no aparece ningún botón de subida.
# ============================================================
import os, io, re, math, json, time, shutil, zipfile, html, getpass
from datetime import datetime, timezone
from collections import defaultdict, deque

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from types import SimpleNamespace
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display, HTML

try:
    from google.colab import files
    EN_COLAB = True
except Exception:
    files = None
    EN_COLAB = False



def _leer_secreto_gurobi(nombre, prompt):
    """Lee un secreto sin incrustarlo ni imprimirlo en el notebook."""
    valor = os.environ.get(nombre, "").strip()
    if valor:
        return valor

    if EN_COLAB:
        try:
            from google.colab import userdata
            valor = str(userdata.get(nombre) or "").strip()
        except Exception:
            valor = ""
        if valor:
            return valor

    valor = getpass.getpass(prompt).strip()
    if not valor:
        raise ValueError(f"No se proporcionó el secreto requerido: {nombre}")
    return valor


def crear_entorno_gurobi():
    """Crea un entorno Gurobi completo según WLS, archivo o auto-detección."""
    modo = str(GUROBI_LICENSE_MODE).strip().upper()

    if modo == "WLS":
        access_id = _leer_secreto_gurobi(
            GUROBI_WLS_ACCESS_ID_SECRET_NAME,
            "Gurobi WLS Access ID: ",
        )
        secret = _leer_secreto_gurobi(
            GUROBI_WLS_SECRET_SECRET_NAME,
            "Gurobi WLS Secret: ",
        )
        license_id_text = _leer_secreto_gurobi(
            GUROBI_WLS_LICENSE_ID_SECRET_NAME,
            "Gurobi WLS License ID: ",
        )
        try:
            license_id = int(license_id_text)
        except ValueError as exc:
            raise ValueError("GUROBI_WLS_LICENSE_ID debe ser un número entero.") from exc

        return gp.Env(params={
            "WLSACCESSID": access_id,
            "WLSSECRET": secret,
            "LICENSEID": license_id,
        })

    if modo == "FILE":
        licencia = Path(GUROBI_LICENSE_FILE).expanduser().resolve()
        if not licencia.exists():
            raise FileNotFoundError(
                f"No existe el archivo de licencia Gurobi: {licencia}. "
                "Sube gurobi.lic o cambia GUROBI_LICENSE_MODE."
            )
        os.environ["GRB_LICENSE_FILE"] = str(licencia)
        return gp.Env()

    if modo == "AUTO":
        print(
            "Gurobi buscará una licencia instalada en las ubicaciones estándar. "
            "Para Colab y modelos grandes se recomienda GUROBI_LICENSE_MODE='WLS'."
        )
        return gp.Env()

    raise ValueError(
        "GUROBI_LICENSE_MODE debe ser 'WLS', 'FILE' o 'AUTO'; "
        f"se recibió {GUROBI_LICENSE_MODE!r}."
    )

CONTENT.mkdir(parents=True, exist_ok=True)
zip_path = CONTENT / NOMBRE_ZIP_DATOS

# Admite cualquier nombre de archivo: identifica el ZIP de datos por su
# estructura interna, no por el texto del nombre. Así también acepta copias como
# "Copia de Final_Datos_...zip" sin confundirlas con un ZIP de resultados.
if not zip_path.exists():
    candidatos=[]
    marcador=f"{CARPETA_RAIZ_DATOS}/Archivos_Datos_Empresas/clientes.csv"
    marcador_c2c=f"{CARPETA_RAIZ_DATOS}/Archivos_Datos_API/google_client_to_client_polylines.json"
    marcador_e2e_matriz=f"{CARPETA_RAIZ_DATOS}/Archivos_Datos_API/google_company_to_company_matrix.jsonl"
    marcador_e2e_trayectorias=f"{CARPETA_RAIZ_DATOS}/Archivos_Datos_API/google_company_to_company_polylines.json"
    marcador_e2c_matriz=f"{CARPETA_RAIZ_DATOS}/Archivos_Datos_API/google_company_to_client_matrix.jsonl"
    marcador_e2c_trayectorias=f"{CARPETA_RAIZ_DATOS}/Archivos_Datos_API/google_company_to_client_polylines.json"
    for candidato in sorted(CONTENT.glob("*.zip")):
        try:
            with zipfile.ZipFile(candidato) as zf_test:
                nombres=set(zf_test.namelist())
            if {
                marcador,
                marcador_c2c,
                marcador_e2e_matriz,
                marcador_e2e_trayectorias,
                marcador_e2c_matriz,
                marcador_e2c_trayectorias,
            }.issubset(nombres):
                candidatos.append(candidato)
        except zipfile.BadZipFile:
            continue
    if len(candidatos) == 1:
        zip_path = candidatos[0]
        print("ZIP de datos detectado por contenido en /content:", zip_path.name)
    else:
        encontrados = "\n".join(f" - {p.name}" for p in sorted(CONTENT.glob("*.zip"))) or " - ninguno"
        compatibles = "\n".join(f" - {p.name}" for p in candidatos) or " - ninguno"
        raise FileNotFoundError(
            "No se pudo identificar de forma única el ZIP E2E/E2C/C2C de datos y trayectorias.\n"
            "Súbelo manualmente desde el panel Archivos de Colab y vuelve a ejecutar esta celda.\n"
            f"Nombre esperado opcional: {NOMBRE_ZIP_DATOS}\n"
            f"ZIP compatibles por contenido:\n{compatibles}\n"
            f"Todos los ZIP encontrados en /content:\n{encontrados}"
        )

for carpeta in [DIR_TRABAJO, DIR_CACHE, DIR_RESULTADOS]:
    if carpeta.exists():
        shutil.rmtree(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(DIR_TRABAJO)

BASE_DATOS = DIR_TRABAJO / CARPETA_RAIZ_DATOS
if not BASE_DATOS.exists():
    encontrados = list(DIR_TRABAJO.rglob("LEEME_PRIMERO.txt"))
    if len(encontrados) != 1:
        raise FileNotFoundError("No se pudo identificar la carpeta raíz del ZIP de datos.")
    BASE_DATOS = encontrados[0].parent

DIR_INPUT = BASE_DATOS / "Archivos_Datos_Empresas"
DIR_API_SEED = BASE_DATOS / "Archivos_Datos_API"

archivos_cache_obligatorios = [
    "google_route_matrix_cache.jsonl",
    "google_route_polylines_cache.json",
    "cache_manifest.json",
    # Las trayectorias entre clientes ya vienen dentro del ZIP y son la fuente
    # de verdad del tramo parada 1 -> parada 2 del vehículo.
    "google_client_to_client_matrix.jsonl",
    "google_client_to_client_polylines.json",
    "google_client_to_client_manifest.json",
    # El ZIP definitivo también debe contener completa la matriz y las
    # trayectorias dirigidas empresa -> empresa.
    "google_company_to_company_matrix.jsonl",
    "google_company_to_company_polylines.json",
    "google_company_to_company_manifest.json",
    # Distancias y trayectorias empresa -> cliente para entregas directas de GasBC.
    "google_company_to_client_matrix.jsonl",
    "google_company_to_client_polylines.json",
    "google_company_to_client_manifest.json",
]
faltantes_cache = [n for n in archivos_cache_obligatorios if not (DIR_API_SEED / n).exists()]
if faltantes_cache:
    raise FileNotFoundError(
        "El ZIP no contiene la caché completa requerida: " + ", ".join(faltantes_cache)
    )

for name in archivos_cache_obligatorios:
    shutil.copy2(DIR_API_SEED / name, DIR_CACHE / name)

# Los tres archivos empresa-empresa son obligatorios en este notebook y ya
# fueron validados dentro de `archivos_cache_obligatorios`.
archivos_cache_empresa_empresa = [
    "google_company_to_company_matrix.jsonl",
    "google_company_to_company_polylines.json",
    "google_company_to_company_manifest.json",
]

MATRIX_CACHE_PATH = DIR_CACHE / "google_route_matrix_cache.jsonl"
POLYLINE_CACHE_PATH = DIR_CACHE / "google_route_polylines_cache.json"
MANIFEST_CACHE_PATH = DIR_CACHE / "cache_manifest.json"
CLIENT_MATRIX_SEED_PATH = DIR_CACHE / "google_client_to_client_matrix.jsonl"
CLIENT_POLYLINE_SEED_PATH = DIR_CACHE / "google_client_to_client_polylines.json"
CLIENT_MANIFEST_SEED_PATH = DIR_CACHE / "google_client_to_client_manifest.json"
COMPANY_MATRIX_SEED_PATH = DIR_CACHE / "google_company_to_company_matrix.jsonl"
COMPANY_POLYLINE_SEED_PATH = DIR_CACHE / "google_company_to_company_polylines.json"
COMPANY_MANIFEST_SEED_PATH = DIR_CACHE / "google_company_to_company_manifest.json"
COMPANY_CLIENT_MATRIX_SEED_PATH = DIR_CACHE / "google_company_to_client_matrix.jsonl"
COMPANY_CLIENT_POLYLINE_SEED_PATH = DIR_CACHE / "google_company_to_client_polylines.json"
COMPANY_CLIENT_MANIFEST_SEED_PATH = DIR_CACHE / "google_company_to_client_manifest.json"


def _contar_jsonl(path):
    return sum(1 for linea in path.read_text(encoding="utf-8").splitlines() if linea.strip())


def _contar_json_dict(path):
    contenido = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(contenido, dict):
        raise ValueError(f"Se esperaba un objeto JSON en {path.name}.")
    return len(contenido)


conteos_zip = {
    "distancias_cliente_cliente": _contar_jsonl(CLIENT_MATRIX_SEED_PATH),
    "trayectorias_cliente_cliente": _contar_json_dict(CLIENT_POLYLINE_SEED_PATH),
    "distancias_empresa_empresa": _contar_jsonl(COMPANY_MATRIX_SEED_PATH),
    "trayectorias_empresa_empresa": _contar_json_dict(COMPANY_POLYLINE_SEED_PATH),
    "distancias_empresa_cliente": _contar_jsonl(COMPANY_CLIENT_MATRIX_SEED_PATH),
    "trayectorias_empresa_cliente": _contar_json_dict(COMPANY_CLIENT_POLYLINE_SEED_PATH),
}
if conteos_zip["distancias_cliente_cliente"] != conteos_zip["trayectorias_cliente_cliente"]:
    raise ValueError("El ZIP tiene cantidades distintas de distancias y trayectorias cliente-cliente.")
if conteos_zip["distancias_empresa_empresa"] != conteos_zip["trayectorias_empresa_empresa"]:
    raise ValueError("El ZIP tiene cantidades distintas de distancias y trayectorias empresa-empresa.")
if conteos_zip["distancias_empresa_empresa"] == 0:
    raise ValueError("El ZIP no contiene registros empresa-empresa.")
if conteos_zip["distancias_empresa_cliente"] != conteos_zip["trayectorias_empresa_cliente"]:
    raise ValueError("El ZIP tiene cantidades distintas de distancias y trayectorias empresa-cliente.")
if conteos_zip["distancias_empresa_cliente"] == 0:
    raise ValueError("El ZIP no contiene registros empresa-cliente E2C.")

print("ZIP leído directamente desde:", zip_path)
print("Registros dedicados verificados dentro del ZIP:", conteos_zip)
print("Datos extraídos en:", BASE_DATOS)
print("Caché local copiada en:", DIR_CACHE)
print("Las trayectorias C2C, E2E y E2C se leerán primero desde los archivos dedicados incluidos en el ZIP.")
print("Los arcos ABC desde nodos NE_E reutilizarán E2C para costo, distancia y HTML.")
print("No se consultará la API para pares C2C, E2E o E2C que ya estén completos en el ZIP.")


ZIP de datos detectado por contenido en /content: Datos_Biometano.zip
ZIP leído directamente desde: /content/Datos_Biometano.zip
Registros dedicados verificados dentro del ZIP: {'distancias_cliente_cliente': 56, 'trayectorias_cliente_cliente': 56, 'distancias_empresa_empresa': 225, 'trayectorias_empresa_empresa': 225, 'distancias_empresa_cliente': 120, 'trayectorias_empresa_cliente': 120}
Datos extraídos en: /content/biometano_modelo_pdf/Datos_Biometano_Corredores_Bidireccionales_COMPLETO
Caché local copiada en: /content/cache_google_maps_biometano
Las trayectorias C2C, E2E y E2C se leerán primero desde los archivos dedicados incluidos en el ZIP.
Los arcos ABC desde nodos NE_E reutilizarán E2C para costo, distancia y HTML.
No se consultará la API para pares C2C, E2E o E2C que ya estén completos en el ZIP.


In [ ]:
# ============================================================
# GASODUCTOS CONVENCIONALES - VALIDACION ZIP_ONLY
# ============================================================
# No se consulta IDE Energia, Google ni ninguna fuente externa para definir
# gasoductos convencionales. Tampoco se reconstruyen ni reemplazan sus distancias.
# La fuente de verdad es el ZIP extraido en DIR_INPUT.

IDS_GASODUCTOS_ESPERADOS = [
    'G_NC','G_CS','G_NS','G_LA','G_LP','G_SBQ','G_LSR','G_RJC','G_PA'
]

gas0=pd.read_csv(DIR_INPUT/'gasoductos.csv')
_ids_zip_gas=set(gas0['id'].astype(str))
_ids_esperados=set(IDS_GASODUCTOS_ESPERADOS)
if _ids_zip_gas != _ids_esperados:
    raise ValueError(
        'gasoductos.csv del ZIP no contiene exactamente los 9 gasoductos esperados. '
        f'faltan={sorted(_ids_esperados-_ids_zip_gas)} extras={sorted(_ids_zip_gas-_ids_esperados)}'
    )

_cols_gas_zip={
    'id','nombre','punto_a_nombre','punto_a_zona','punto_a_lat','punto_a_lon',
    'punto_b_nombre','punto_b_zona','punto_b_lat','punto_b_lon',
    'distancia_red_estimada_km','capacidad_efectiva_nm3',
    'costo_fijo_usd_dia','costo_variable_usd_km_nm3'
}
_faltan_cols_gas=sorted(_cols_gas_zip-set(gas0.columns))
if _faltan_cols_gas:
    raise ValueError('gasoductos.csv del ZIP no trae columnas obligatorias: '+str(_faltan_cols_gas))
for _col in ['punto_a_lat','punto_a_lon','punto_b_lat','punto_b_lon',
             'distancia_red_estimada_km','capacidad_efectiva_nm3',
             'costo_fijo_usd_dia','costo_variable_usd_km_nm3']:
    _vals=pd.to_numeric(gas0[_col],errors='coerce')
    if _vals.isna().any():
        raise ValueError(f'gasoductos.csv del ZIP tiene valores no numericos en {_col}.')
    if _col in {'distancia_red_estimada_km','capacidad_efectiva_nm3'} and (_vals<=0).any():
        raise ValueError(f'gasoductos.csv del ZIP tiene valores no positivos en {_col}.')

_dist_gg_path=DIR_INPUT/'distancias_entre_gasoductos_36.csv'
if not _dist_gg_path.exists():
    raise FileNotFoundError(
        'Falta distancias_entre_gasoductos_36.csv dentro del ZIP. '
        'No se generara ni reemplazara automaticamente.'
    )
_dist_gg_zip=pd.read_csv(_dist_gg_path)
_cols_dist_zip={'gasoducto_1_id','gasoducto_2_id','distancia_km'}
_faltan_cols_dist=sorted(_cols_dist_zip-set(_dist_gg_zip.columns))
if _faltan_cols_dist:
    raise ValueError('distancias_entre_gasoductos_36.csv del ZIP no trae columnas: '+str(_faltan_cols_dist))
_pares_zip=set()
for _r in _dist_gg_zip.itertuples(index=False):
    _a=str(getattr(_r,'gasoducto_1_id')); _b=str(getattr(_r,'gasoducto_2_id'))
    if _a==_b:
        raise ValueError('distancias_entre_gasoductos_36.csv trae un par del mismo gasoducto.')
    if _a not in _ids_esperados or _b not in _ids_esperados:
        raise ValueError(f'distancias_entre_gasoductos_36.csv referencia gasoducto desconocido: {_a}, {_b}')
    _d=float(getattr(_r,'distancia_km'))
    if not np.isfinite(_d) or _d < 0:
        raise ValueError(f'distancia invalida en distancias_entre_gasoductos_36.csv para {_a}-{_b}: {_d}')
    _pares_zip.add(tuple(sorted((_a,_b))))
_ids_orden=[g for g in IDS_GASODUCTOS_ESPERADOS]
_pares_esperados={tuple(sorted((_ids_orden[i],_ids_orden[j]))) for i in range(len(_ids_orden)) for j in range(i+1,len(_ids_orden))}
if _pares_zip != _pares_esperados or len(_dist_gg_zip)!=36:
    raise ValueError(
        'distancias_entre_gasoductos_36.csv del ZIP debe traer exactamente los 36 pares unicos.'
    )

print('Gasoductos convencionales validados en modo ZIP_ONLY.')
print('No se consulto ninguna fuente externa ni se reconstruyeron/reemplazaron distancias de gasoductos.')
print({'gasoductos':len(gas0),'pares_gasoducto_gasoducto_zip':len(_dist_gg_zip)})


Gasoductos convencionales validados en modo ZIP_ONLY.
No se consulto ninguna fuente externa ni se reconstruyeron/reemplazaron distancias de gasoductos.
{'gasoductos': 9, 'pares_gasoducto_gasoducto_zip': 36}


#3

In [ ]:
# ============================================================
# 3. CARGA Y VALIDACIÓN ESTRICTA DE LOS DATOS
# ============================================================
empresas = pd.read_csv(DIR_INPUT / "empresas.csv")
clientes = pd.read_csv(DIR_INPUT / "clientes.csv")
sitios = pd.read_csv(DIR_INPUT / "sitios_candidatos.csv")
tipos = pd.read_csv(DIR_INPUT / "tipos_plantas.csv")
vehiculos = pd.read_csv(DIR_INPUT / "vehiculos.csv")

# ============================================================
# CAPACIDADES DE CAMIONES DEFINIDAS MANUALMENTE
# Sobrescriben vehiculos.csv sin modificar el ZIP
# ============================================================

# Capacidades manuales#

if Capacidad_Plantas_Alterada == True:
  for codigo, capacidad in CAPACIDAD_PLANTAS_MANUAL.items():
      tipos.loc[
          tipos["codigo_pdf"].astype(str).eq(codigo),
         "capacidad_max_nm3"
      ] = capacidad

if Capacidad_Camiones_Alterada == True:
  for codigo, capacidad in CAPACIDAD_CAMIONES_MANUAL.items():
      vehiculos.loc[
          vehiculos["codigo_pdf"].astype(str).eq(codigo),
          "capacidad_max_m3_viaje"
      ] = capacidad

gasoductos = pd.read_csv(DIR_INPUT / "gasoductos.csv")
distancias_entre_gasoductos_36 = pd.read_csv(DIR_INPUT / "distancias_entre_gasoductos_36.csv")
arcos_candidatos = pd.read_csv(DIR_INPUT / "arcos_candidatos_google.csv")
parametros_generales = pd.read_csv(DIR_INPUT / "parametros_generales.csv")
fuentes_costos = pd.read_csv(DIR_INPUT / "FUENTES_COSTOS_VARIABLES_TRANSFORMACION.csv")
factores_conversion = pd.read_csv(DIR_INPUT / "factores_conversion.csv")

required = {
    'empresas': (empresas, {
        'id', 'nombre', 'lat', 'lon', 'capacidad_max_biogas_nm3',
        'costo_activacion_biogas_usd', COL_PRECIO_BIOGAS_CRUDO,
        COL_OFERTA_EXCEDENTE_BIOGAS
    }),
    'clientes': (clientes, {
        'id', 'nombre', 'lat', 'lon', 'demanda_max_nm3',
        'precio_venta_usd_nm3', COL_DEMANDA_NO_CUBIERTA
    }),
    'sitios': (sitios, {'id','nombre','lat','lon','region'}),
    'tipos': (tipos, {'codigo_pdf','tipo_instalacion','capacidad_max_nm3','costo_variable_usd_nm3','costo_fijo_usd','formato_entrada','formato_salida'}),
    'factores_conversion': (factores_conversion, {'codigo_pdf','tipo_instalacion','formato_entrada','formato_salida','factor_conversion_salida_por_entrada','aplica_capacidad_y_costo_variable_sobre'}),
    'vehiculos': (vehiculos, {'codigo_pdf','capacidad_max_m3_viaje','costo_fijo_viaje_usd','costo_km_usd','unidad_capacidad'}),
    'gasoductos': (gasoductos, {
        'id','nombre','punto_a_nombre','punto_a_zona','punto_a_lat','punto_a_lon',
        'punto_b_nombre','punto_b_zona','punto_b_lat','punto_b_lon',
        'capacidad_punto_a_nm3','capacidad_punto_b_nm3','capacidad_efectiva_nm3',
        'distancia_red_estimada_km','costo_fijo_usd_dia','costo_variable_usd_km_nm3',
        'bidireccional','independiente_de_otros_corredores'
    }),
}
for name,(df,cols) in required.items():
    faltan = cols - set(df.columns)
    if faltan:
        raise ValueError(f"{name}: faltan columnas {sorted(faltan)}")
    if 'id' in cols and df['id'].duplicated().any():
        raise ValueError(f"{name}: hay identificadores duplicados")


import itertools

# Las 36 distancias son la única tabla de longitudes usada para transportar GasBC
# ENTRE los nueve gasoductos convencionales. La tabla guarda cada par una sola vez;
# el MILP la expande internamente a ambos sentidos.
_cols_gg={'gasoducto_1_id','gasoducto_2_id','distancia_km'}
if not _cols_gg.issubset(distancias_entre_gasoductos_36.columns):
    raise ValueError(f'distancias_entre_gasoductos_36.csv: faltan columnas {sorted(_cols_gg-set(distancias_entre_gasoductos_36.columns))}')
if len(distancias_entre_gasoductos_36)!=36:
    raise ValueError(f'distancias_entre_gasoductos_36.csv debe contener exactamente 36 pares, no {len(distancias_entre_gasoductos_36)}')
_gids=set(gasoductos['id'].astype(str))
_pares_gg=set()
for _r in distancias_entre_gasoductos_36.itertuples(index=False):
    _a=str(_r.gasoducto_1_id); _b=str(_r.gasoducto_2_id); _d=float(_r.distancia_km)
    if _a==_b or _a not in _gids or _b not in _gids or not np.isfinite(_d) or _d<0:
        raise ValueError(('distancia_gasoducto_gasoducto_invalida',_a,_b,_d))
    _pares_gg.add(tuple(sorted((_a,_b))))
if len(_pares_gg)!=36:
    raise ValueError(f'Las 36 filas deben representar los 36 pares únicos; encontrados {len(_pares_gg)}')
if _pares_gg != set(itertools.combinations(sorted(_gids),2)):
    raise ValueError('La tabla de 36 distancias no cubre exactamente todos los pares de los 9 gasoductos.')


# Conversión de material: única fuente de verdad = factores_conversion.csv del ZIP.
factores_conversion['factor_conversion_salida_por_entrada'] = pd.to_numeric(
    factores_conversion['factor_conversion_salida_por_entrada'], errors='raise'
)
FACTORES_CONVERSION = (
    factores_conversion.set_index('codigo_pdf')['factor_conversion_salida_por_entrada']
    .astype(float).to_dict()
)
print("Factores de Conversión:", FACTORES_CONVERSION)

# Empresas, sitios y clientes activos deben coincidir exactamente con los JSON entregados por el usuario.
EMPRESAS_JSON_PATH = BASE_DATOS / "Datos_JSON_Usuario" / "codigo_plantas_biogas_usuario.json"
SITIOS_JSON_PATH = BASE_DATOS / "Datos_JSON_Usuario" / "codigo_puntos_estrategicos_usuario.json"
CLIENTES_JSON_PATH = BASE_DATOS / "Datos_JSON_Usuario" / "codigo_clientes_usuario.json"

empresas_json = json.loads(EMPRESAS_JSON_PATH.read_text(encoding="utf-8"))
sitios_json = json.loads(SITIOS_JSON_PATH.read_text(encoding="utf-8"))
clientes_json = json.loads(CLIENTES_JSON_PATH.read_text(encoding="utf-8"))

campos_empresas_json = {'numero','nombre_instalacion','comuna','latitud','longitud','produccion_diaria_m3'}

campos_sitios_json = {'numero','tipo_nodo','nombre_sector','latitud','longitud','formato_principal','justificacion_estrategica'}
# La justificación estratégica es texto descriptivo y puede tener una redacción distinta
# entre sitios_candidatos.csv y el JSON sin modificar ningún dato físico del sitio.
# Se conserva la validación estricta de nombre, tipo, coordenadas y formato principal.

campos_clientes_json = {'empresa_instalacion','modalidad_suministro','latitud','longitud','justificacion'}

# ------------------------------------------------------------------
# EXCLUSIÓN DE EMPRESAS CON MAYOR CAPACIDAD DE OFERTA
# ------------------------------------------------------------------
# La validación anterior siempre se hace sobre las 15 empresas originales.
# La exclusión se aplica recién aquí, antes de construir los conjuntos y
# variables del MILP. El ranking usa la capacidad ORIGINAL del ZIP.
empresas_completas_originales = empresas.copy()
empresas_completas_originales['capacidad_max_biogas_nm3'] = pd.to_numeric(
    empresas_completas_originales['capacidad_max_biogas_nm3'], errors='raise'
)

ranking_empresas_oferta = empresas_completas_originales.sort_values(
    by=['capacidad_max_biogas_nm3', 'id'],
    ascending=[False, True],
    kind='mergesort',
).reset_index(drop=True)

if NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR is False:
    numero_empresas_a_excluir = 0
else:
    numero_empresas_a_excluir = int(NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR)

if numero_empresas_a_excluir >= len(empresas_completas_originales):
    raise ValueError(
        'NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR debe dejar al menos una empresa disponible. '
        f'Hay {len(empresas_completas_originales)} empresas; usa False o un entero entre 1 y '
        f'{len(empresas_completas_originales)-1}.'
    )

empresas_excluidas_mayor_oferta = ranking_empresas_oferta.head(
    numero_empresas_a_excluir
).copy()
empresas_excluidas_mayor_oferta = empresas_excluidas_mayor_oferta.assign(
    motivo_exclusion='Mayor capacidad original de oferta',
    posicion_ranking_oferta=np.arange(1, numero_empresas_a_excluir + 1, dtype=int),
)
ids_empresas_excluidas_mayor_oferta = set(
    empresas_excluidas_mayor_oferta['id'].astype(str)
)

empresas = empresas_completas_originales.loc[
    ~empresas_completas_originales['id'].astype(str).isin(ids_empresas_excluidas_mayor_oferta)
].copy().reset_index(drop=True)

print(
    'Empresas excluidas por mayor oferta:',
    'ninguna' if not ids_empresas_excluidas_mayor_oferta
    else ', '.join(
        f"{r.id} · {r.nombre} ({float(r.capacidad_max_biogas_nm3):,.0f} Nm³/día)"
        for r in empresas_excluidas_mayor_oferta.itertuples(index=False)
    )
)
print(f'Empresas disponibles para el MILP: {len(empresas)} de {len(empresas_completas_originales)}')

if NUMERO_EMPRESAS_A_UTILIZAR is not False and NUMERO_EMPRESAS_A_UTILIZAR > len(empresas):
    raise ValueError(
        'NUMERO_EMPRESAS_A_UTILIZAR no puede superar el número de empresas disponibles '
        f'después de aplicar exclusiones. Solicitadas: {NUMERO_EMPRESAS_A_UTILIZAR}; '
        f'disponibles: {len(empresas)}.'
    )

if NUMERO_CLIENTES_A_UTILIZAR is not False and NUMERO_CLIENTES_A_UTILIZAR > len(clientes):
    raise ValueError(
        'NUMERO_CLIENTES_A_UTILIZAR no puede superar el número de clientes disponibles. '
        f'Solicitados: {NUMERO_CLIENTES_A_UTILIZAR}; disponibles: {len(clientes)}.'
    )

# El CSV conserva la matriz vial completa de las 15 empresas originales.
# Después de excluir empresas, también deben retirarse inmediatamente sus arcos
# ABIO; de lo contrario la validación de caché intentaría buscar coordenadas de
# empresas que ya no existen en E_COORD (por ejemplo, E1) y produciría KeyError.
_ids_empresas_disponibles = set(empresas['id'].astype(str))
_mascara_abio_excluido = (
    arcos_candidatos['tipo_arco'].astype(str).eq('ABIO')
    & ~arcos_candidatos['origen_id'].astype(str).isin(_ids_empresas_disponibles)
)
arcos_abio_empresas_excluidas = arcos_candidatos.loc[_mascara_abio_excluido].copy()
arcos_candidatos = arcos_candidatos.loc[~_mascara_abio_excluido].copy().reset_index(drop=True)

print(f'Arcos ABIO retirados por exclusión de empresas: {len(arcos_abio_empresas_excluidas):,}')

# ------------------------------------------------------------------
# NODOS FÍSICOS CANDIDATOS Y RELACIONES LOCALES EXPLÍCITAS
# ------------------------------------------------------------------
# REGLA ESTRUCTURAL:
#   1) ninguna empresa cambia jamás de coordenadas;
#   2) todos los sitios estratégicos originales permanecen disponibles;
#   3) la ubicación original de cada empresa se agrega como un sitio candidato
#      adicional de plantas, sin modificar empresas.csv ni sitios_candidatos.csv;
#   4) solamente empresa -> su propio nodo NE_E es una transferencia local.
SITIOS_ORIGINALES_IDS = sitios['id'].astype(str).tolist()

# Guardado inmutable de las coordenadas leídas desde empresas.csv, antes de
# crear cualquier nodo derivado o aplicar la formulación MILP.
COORDENADAS_EMPRESAS_ORIGINALES = {
    str(r.id): (float(r.lat), float(r.lon))
    for r in empresas_completas_originales.itertuples(index=False)
}

# Cada empresa posee un nodo físico propio e inequívoco. Este diccionario no
# asocia empresas con P1, P5 ni con ningún otro sitio estratégico existente.
EMPRESA_A_NODO_COMPLETO = {
    empresa_id: f'NE_{empresa_id}'
    for empresa_id in empresas_completas_originales['id'].astype(str)
}

# Solo las empresas proveedoras no excluidas originan biogás en el MILP. Las
# ubicaciones físicas de todas las empresas, incluso una empresa excluida como
# proveedora, siguen siendo sitios candidatos de instalación.
EMPRESA_A_NODO = {
    empresa_id: EMPRESA_A_NODO_COMPLETO[empresa_id]
    for empresa_id in sorted(_ids_empresas_disponibles)
}

# Cada cliente conserva su nodo propio y puede alojar BG, L, RL, C y RC cuando
# recibe biogás crudo por vehículo o gasoducto corto desde una o varias empresas.
CLIENTE_A_NODO = {j: f'NC_{j}' for j in clientes['id'].astype(str)}

# Los sitios estratégicos originales se conservan sin cambiar coordenadas,
# identificadores ni datos. Solo se agregan columnas de clasificación usadas
# por el modelo y el reporte.
sitios = sitios.copy()
sitios['clase_nodo'] = 'SITIO_ESTRATEGICO'
sitios['referencia_local'] = ''
sitios['tecnologias_permitidas'] = 'BG,L,RL,C,RC'

# Nuevos sitios candidatos derivados de las ubicaciones fijas de las empresas.
# No se altera ninguna fila de empresas.csv: se copia su posición exacta a un
# nodo adicional NE_Ei donde el MILP puede instalar cualquiera de las plantas.
filas_nodos_empresas = []
for r in empresas_completas_originales.itertuples(index=False):
    empresa_id = str(r.id)
    nodo_id = EMPRESA_A_NODO_COMPLETO[empresa_id]
    fila = {col: np.nan for col in sitios.columns}
    region_empresa = getattr(r, 'comuna_region', None)
    if pd.isna(region_empresa) or not str(region_empresa).strip():
        region_empresa = 'Ubicación física de empresa'
    fila.update({
        'id': nodo_id,
        'nombre': f'Ubicación fija de la empresa {empresa_id}: {r.nombre}',
        'lat': float(r.lat),
        'lon': float(r.lon),
        'region': str(region_empresa),
        'ubicacion': str(r.nombre),
        'todas_tecnologias_permitidas': True,
        'fuente': 'empresas.csv; nodo físico candidato derivado sin modificar la empresa',
        'numero': 2000 + int(re.sub(r'\D', '', empresa_id) or 0),
        'tipo_nodo': 'Ubicación fija de empresa',
        'nombre_sector': str(r.nombre),
        'latitud': float(r.lat),
        'longitud': float(r.lon),
        'formato_principal': 'BG / L / C',
        'justificacion_estrategica': (
            'Permite instalar productoras en la ubicación original e inmutable '
            'de la empresa, sin reemplazar los sitios candidatos anteriores.'
        ),
        'origen_registro': 'Derivado de empresas.csv',
        'clase_nodo': 'UBICACION_FIJA_EMPRESA',
        'referencia_local': empresa_id,
        'tecnologias_permitidas': 'BG,L,C',
    })
    filas_nodos_empresas.append(fila)

nodos_empresas = pd.DataFrame(filas_nodos_empresas)
NODOS_EMPRESA_IDS = nodos_empresas['id'].astype(str).tolist()

NODO_EMPRESA_A_EMPRESA = {
    nodo_id: empresa_id
    for empresa_id, nodo_id in EMPRESA_A_NODO_COMPLETO.items()
}

# Nodos de clientes: se habilitan BG, L, RL, C y RC. La planta BG sólo podrá
# recibir biogás desde variables físicas explícitas, por vehículo o gasoducto corto.
filas_nodos_clientes = []
for r in clientes.itertuples(index=False):
    fila = {col: np.nan for col in sitios.columns}
    nodo_id = CLIENTE_A_NODO[str(r.id)]
    fila.update({
        'id': nodo_id,
        'nombre': f'Ubicación exacta del cliente {r.id}: {r.nombre}',
        'lat': float(r.lat),
        'lon': float(r.lon),
        'region': str(r.comuna_region) if pd.notna(r.comuna_region) else 'Nodo físico de cliente',
        'ubicacion': str(r.nombre),
        'todas_tecnologias_permitidas': False,
        'fuente': 'clientes.csv; nodo físico candidato derivado explícitamente',
        'numero': 1000 + int(re.sub(r'\D', '', str(r.id)) or 0),
        'tipo_nodo': 'Ubicación de cliente',
        'nombre_sector': str(r.nombre),
        'latitud': float(r.lat),
        'longitud': float(r.lon),
        'formato_principal': 'BG / L / C',
        'justificacion_estrategica': 'Permite concentrar biogás crudo por vehículo o gasoducto corto, instalar BG y convertir localmente a BioGNL/BioGNC o entregar GasBC.',
        'origen_registro': 'Derivado de clientes.csv',
        'clase_nodo': 'UBICACION_CLIENTE',
        'referencia_local': str(r.id),
        'tecnologias_permitidas': ','.join(TECNOLOGIAS_EN_NODO_CLIENTE),
    })
    filas_nodos_clientes.append(fila)

sitios = pd.concat(
    [sitios, nodos_empresas, pd.DataFrame(filas_nodos_clientes)],
    ignore_index=True,
)

# Comprobación fuerte: ni la tabla de empresas ni los nodos NE_E pueden haber
# cambiado la posición original de una empresa.
_sitios_coord_idx = sitios.set_index('id')
_empresas_coord_idx = empresas_completas_originales.set_index('id')
for empresa_id, coordenadas in COORDENADAS_EMPRESAS_ORIGINALES.items():
    nodo_id = EMPRESA_A_NODO_COMPLETO[empresa_id]

# Sitios usados para generar la red vial productiva tradicional. Incluye los
# sitios originales y las ubicaciones de empresa. Los nodos de cliente se conectan
# mediante aliases E2C/C2C explícitos del ZIP, sin inventar distancias ni trayectorias.
SITIOS_PRODUCTIVOS_IDS = SITIOS_ORIGINALES_IDS + NODOS_EMPRESA_IDS

TECNOLOGIAS_PERMITIDAS_POR_SITIO = {
    str(s): {'BG','L','RL','C','RC'} for s in SITIOS_PRODUCTIVOS_IDS
}
for cliente_id, nodo_id in CLIENTE_A_NODO.items():
    TECNOLOGIAS_PERMITIDAS_POR_SITIO[nodo_id] = set(TECNOLOGIAS_EN_NODO_CLIENTE)

# ------------------------------------------------------------------
# CANDIDATOS DINÁMICOS: GASODUCTO CORTO DE BIOGÁS CRUDO
# ------------------------------------------------------------------
# Se calculan aquí, después de leer empresas, clientes y sitios y antes de
# construir el MILP. No se añade ninguna fila a gasoductos.csv,
# arcos_candidatos_google.csv, archivos de caché ni a ningún archivo del ZIP.
#
# Tipos de destino admitidos:
#   CLIENTE: empresa -> nodo NC_Cj -> BG -> cadena posterior o entrega local.
#   UBICACION_POSIBLE: empresa -> sitio P# -> BG -> cadena posterior.
#   UBICACION_EMPRESA: empresa -> nodo NE_Ek -> BG concentradora -> cadena posterior.
#   ENTRADA_GASODUCTO se incorpora más abajo, una vez creados los nodos NGI.
def _distancia_recta_km(lat1, lon1, lat2, lon2):
    radio_tierra_km = 6371.0088
    phi1 = math.radians(float(lat1))
    phi2 = math.radians(float(lat2))
    dphi = math.radians(float(lat2) - float(lat1))
    dlambda = math.radians(float(lon2) - float(lon1))
    h = (
        math.sin(dphi / 2.0) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    )
    return 2.0 * radio_tierra_km * math.asin(
        min(1.0, math.sqrt(max(0.0, h)))
    )

_COLUMNAS_GASODUCTO_BIOGAS_CORTO = [
    'id', 'nombre', 'empresa_id', 'empresa',
    'destino_tipo', 'destino_id', 'destino_nombre', 'nodo_destino_id',
    # Campos específicos conservados para facilitar auditoría y exportación.
    'cliente_id', 'cliente', 'nodo_cliente_id',
    'ubicacion_id', 'ubicacion',
    'punto_a_lat', 'punto_a_lon', 'punto_b_lat', 'punto_b_lon',
    'distancia_recta_km', 'costo_fijo_usd_dia',
    'costo_variable_usd_nm3_km', 'capacidad_max_nm3_dia',
    'largo_maximo_km', 'route_key', 'geometria',
    'empresa_disponible_en_milp',
]

_filas_gasoductos_biogas_cortos = []

def _registrar_gasoducto_biogas_corto(
    *, empresa_row, destino_tipo, destino_id, destino_nombre,
    nodo_destino_id, destino_lat, destino_lon,
    cliente_id=None, cliente=None, nodo_cliente_id=None,
    ubicacion_id=None, ubicacion=None,
):
    empresa_id = str(empresa_row.id)
    distancia_km = _distancia_recta_km(
        empresa_row.lat, empresa_row.lon, destino_lat, destino_lon
    )
    if not (
        HABILITAR_GASODUCTOS_BIOGAS_CORTOS
        and distancia_km < float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM)
    ):
        return

    if destino_tipo == 'CLIENTE':
        pipe_id = f'GBCORTO_{empresa_id}_{destino_id}'
        tipo_texto = 'cliente'
    elif destino_tipo == 'UBICACION_POSIBLE':
        pipe_id = f'GBCORTO_SITIO_{empresa_id}_{destino_id}'
        tipo_texto = 'ubicación posible'
    elif destino_tipo == 'UBICACION_EMPRESA':
        pipe_id = f'GBCORTO_EMPRESA_{empresa_id}_{destino_id}'
        tipo_texto = 'ubicación de empresa'
    else:
        raise ValueError(f'Tipo de destino de gasoducto corto no reconocido: {destino_tipo}')

    _filas_gasoductos_biogas_cortos.append({
        'id': pipe_id,
        'nombre': (
            f'Gasoducto corto de biogás crudo {empresa_id} → '
            f'{tipo_texto} {destino_id}'
        ),
        'empresa_id': empresa_id,
        'empresa': str(empresa_row.nombre),
        'destino_tipo': str(destino_tipo),
        'destino_id': str(destino_id),
        'destino_nombre': str(destino_nombre),
        'nodo_destino_id': str(nodo_destino_id),
        'cliente_id': None if cliente_id is None else str(cliente_id),
        'cliente': None if cliente is None else str(cliente),
        'nodo_cliente_id': None if nodo_cliente_id is None else str(nodo_cliente_id),
        'ubicacion_id': None if ubicacion_id is None else str(ubicacion_id),
        'ubicacion': None if ubicacion is None else str(ubicacion),
        'punto_a_lat': float(empresa_row.lat),
        'punto_a_lon': float(empresa_row.lon),
        'punto_b_lat': float(destino_lat),
        'punto_b_lon': float(destino_lon),
        'distancia_recta_km': float(distancia_km),
        'costo_fijo_usd_dia': float(
            COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA
        ),
        'costo_variable_usd_nm3_km': float(
            COSTO_VARIABLE_GASODUCTO_BIOGAS_CORTO_USD_NM3_KM
        ),
        'capacidad_max_nm3_dia': float(
            CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA
        ),
        'largo_maximo_km': float(
            LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM
        ),
        'route_key': f'SHORTBIO|{pipe_id}',
        'geometria': 'LINEA_RECTA_CALCULADA_EN_IPYNB',
        'empresa_disponible_en_milp': empresa_id in _ids_empresas_disponibles,
    })

# 1) Empresa -> cliente. El nodo NC_Cj representa una planta potencial situada
# exactamente en las coordenadas físicas del cliente.
for empresa_row in empresas_completas_originales.itertuples(index=False):
    for cliente_row in clientes.itertuples(index=False):
        cliente_id = str(cliente_row.id)
        _registrar_gasoducto_biogas_corto(
            empresa_row=empresa_row,
            destino_tipo='CLIENTE',
            destino_id=cliente_id,
            destino_nombre=str(cliente_row.nombre),
            nodo_destino_id=CLIENTE_A_NODO[cliente_id],
            destino_lat=cliente_row.lat,
            destino_lon=cliente_row.lon,
            cliente_id=cliente_id,
            cliente=str(cliente_row.nombre),
            nodo_cliente_id=CLIENTE_A_NODO[cliente_id],
        )

# 2) Empresa -> ubicación posible P#. Se conservan todas las ubicaciones
# originales del ZIP.
_sitios_posibles_gasoducto = sitios.loc[
    sitios['id'].astype(str).isin(SITIOS_ORIGINALES_IDS)
].copy()
for empresa_row in empresas_completas_originales.itertuples(index=False):
    for sitio_row in _sitios_posibles_gasoducto.itertuples(index=False):
        sitio_id = str(sitio_row.id)
        _registrar_gasoducto_biogas_corto(
            empresa_row=empresa_row,
            destino_tipo='UBICACION_POSIBLE',
            destino_id=sitio_id,
            destino_nombre=str(sitio_row.nombre),
            nodo_destino_id=sitio_id,
            destino_lat=sitio_row.lat,
            destino_lon=sitio_row.lon,
            ubicacion_id=sitio_id,
            ubicacion=str(sitio_row.nombre),
        )

# 3) Empresa -> ubicación física de cualquier empresa NE_Ek. Esto permite que
# varias proveedoras concentren biogás crudo en una BG instalada en una empresa
# distinta. La transferencia de una empresa a su propio nodo sigue disponible
# localmente mediante X y, aunque el candidato de tubería exista a distancia cero,
# su costo fijo evita que sea necesario en el óptimo.
for empresa_row in empresas_completas_originales.itertuples(index=False):
    for sitio_row in nodos_empresas.itertuples(index=False):
        sitio_id = str(sitio_row.id)
        _registrar_gasoducto_biogas_corto(
            empresa_row=empresa_row,
            destino_tipo='UBICACION_EMPRESA',
            destino_id=sitio_id,
            destino_nombre=str(sitio_row.nombre),
            nodo_destino_id=sitio_id,
            destino_lat=sitio_row.lat,
            destino_lon=sitio_row.lon,
            ubicacion_id=sitio_id,
            ubicacion=str(sitio_row.nombre),
        )

gasoductos_biogas_cortos_todos = pd.DataFrame(
    _filas_gasoductos_biogas_cortos,
    columns=_COLUMNAS_GASODUCTO_BIOGAS_CORTO,
)
gasoductos_biogas_cortos = gasoductos_biogas_cortos_todos.loc[
    gasoductos_biogas_cortos_todos['empresa_disponible_en_milp'].astype(bool)
].copy().reset_index(drop=True)

if len(gasoductos_biogas_cortos_todos):
    pass

_conteo_candidatos_tipo = (
    gasoductos_biogas_cortos_todos['destino_tipo'].value_counts().to_dict()
    if len(gasoductos_biogas_cortos_todos) else {}
)
print('Gasoductos cortos calculados en memoria:', len(gasoductos_biogas_cortos_todos))
print(' - empresa -> cliente:', int(_conteo_candidatos_tipo.get('CLIENTE', 0)))
print(' - empresa -> ubicación posible:', int(_conteo_candidatos_tipo.get('UBICACION_POSIBLE', 0)))
print(' - empresa -> ubicación de empresa:', int(_conteo_candidatos_tipo.get('UBICACION_EMPRESA', 0)))
print('Gasoductos cortos disponibles para el MILP:', len(gasoductos_biogas_cortos))
if len(gasoductos_biogas_cortos_todos):
    display(gasoductos_biogas_cortos_todos[[
        'id', 'empresa_id', 'empresa', 'destino_tipo', 'destino_id',
        'destino_nombre', 'nodo_destino_id', 'distancia_recta_km',
        'empresa_disponible_en_milp',
    ]])


# -----------------------------------------------------------------------------
# NUEVO: gasoductos cortos de BIOGÁS CRUDO cliente/HUB -> cliente.
# Son arcos dirigidos. El cliente de origen no necesita procesar el biogás en BG:
# puede recibir biogás crudo desde una o varias empresas/otros clientes y reenviar
# una parte o todo el flujo. La conversión BG se cobra y se activa únicamente en
# el nodo donde el biogás efectivamente se procesa.
# -----------------------------------------------------------------------------
_COLUMNAS_GASODUCTO_BIOGAS_C2C = [
    'id','nombre','cliente_origen_id','cliente_origen','nodo_origen_id',
    'cliente_destino_id','cliente_destino','nodo_destino_id',
    'punto_a_lat','punto_a_lon','punto_b_lat','punto_b_lon',
    'distancia_recta_km','costo_fijo_usd_dia','costo_variable_usd_nm3_km',
    'capacidad_max_nm3_dia','largo_maximo_km','route_key','geometria',
    'producto_transportado',
]
_filas_biogas_c2c=[]
for cliente_origen in clientes.itertuples(index=False):
    jo=str(cliente_origen.id)
    for cliente_destino in clientes.itertuples(index=False):
        jd=str(cliente_destino.id)
        if jo==jd:
            continue
        distancia_km=_distancia_recta_km(
            cliente_origen.lat,cliente_origen.lon,
            cliente_destino.lat,cliente_destino.lon,
        )
        if not (
            HABILITAR_RELEVO_BIOGAS_CRUDO_CLIENTE_CLIENTE
            and HABILITAR_GASODUCTOS_BIOGAS_CORTOS
            and distancia_km < float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM)
        ):
            continue
        h=f'GBCORTO_C2C_{jo}_{jd}'
        _filas_biogas_c2c.append({
            'id':h,
            'nombre':f'Gasoducto corto de biogás crudo cliente/HUB {jo} → cliente {jd}',
            'cliente_origen_id':jo,
            'cliente_origen':str(cliente_origen.nombre),
            'nodo_origen_id':str(CLIENTE_A_NODO[jo]),
            'cliente_destino_id':jd,
            'cliente_destino':str(cliente_destino.nombre),
            'nodo_destino_id':str(CLIENTE_A_NODO[jd]),
            'punto_a_lat':float(cliente_origen.lat),'punto_a_lon':float(cliente_origen.lon),
            'punto_b_lat':float(cliente_destino.lat),'punto_b_lon':float(cliente_destino.lon),
            'distancia_recta_km':float(distancia_km),
            'costo_fijo_usd_dia':float(COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA),
            'costo_variable_usd_nm3_km':float(COSTO_VARIABLE_GASODUCTO_BIOGAS_CORTO_USD_NM3_KM),
            'capacidad_max_nm3_dia':float(CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA),
            'largo_maximo_km':float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM),
            'route_key':f'SHORTBIOC2C|{h}',
            'geometria':'LINEA_RECTA_CALCULADA_EN_IPYNB',
            'producto_transportado':'Biogás crudo',
        })

gasoductos_biogas_cortos_cliente_cliente=pd.DataFrame(
    _filas_biogas_c2c,columns=_COLUMNAS_GASODUCTO_BIOGAS_C2C
)
if len(gasoductos_biogas_cortos_cliente_cliente):
    pass
print('Gasoductos cortos biogás crudo cliente/HUB -> cliente:',len(gasoductos_biogas_cortos_cliente_cliente))

# Únicas transferencias locales declaradas:
#   empresa -> su propio nodo fijo NE_E;
#   nodo de cliente NC_C -> su propio cliente.
# Ninguna empresa queda asociada gratuitamente a P1, P5 ni a otro lugar.
DECLARED_LOCAL_ROUTE_KEYS = {
    f'ABIO|{empresa_id}|{nodo_id}'
    for empresa_id, nodo_id in EMPRESA_A_NODO.items()
}
DECLARED_LOCAL_ROUTE_KEYS.update(
    f'ABC|{nodo_id}|{cliente_id}'
    for cliente_id, nodo_id in CLIENTE_A_NODO.items()
)

# Se agregan únicamente los arcos necesarios para que los nodos NE_E participen
# en la misma formulación que los sitios originales. Las filas y arcos previos
# del ZIP permanecen intactos.
ROUTE_ALIAS_SOURCE = {}
nuevos_arcos = []
_route_keys_existentes = set(arcos_candidatos['route_key'].astype(str))
# Fuente f?sica ampliada: cuando una empresa se excluye como proveedora, sus
# arcos ABIO salen del MILP, pero siguen siendo rutas viales v?lidas del ZIP para
# la ubicaci?n NE_Ei. Se usan s?lo como cache/alias, sin reactivar a la empresa.
_route_keys_fuente_fisica_abio = set(_route_keys_existentes)
if len(arcos_abio_empresas_excluidas):
    _route_keys_fuente_fisica_abio.update(
        arcos_abio_empresas_excluidas['route_key'].astype(str)
    )



# Capacidad efectiva compartida por ambos sentidos de cada corredor.
# Se construyen las direcciones antes de generar los nuevos arcos AIG.
gasoductos['capacidad_efectiva_nm3'] = gasoductos[['capacidad_punto_a_nm3','capacidad_punto_b_nm3']].min(axis=1)

direction_rows=[]
for r in gasoductos.itertuples(index=False):
    direction_rows += [
        {'direccion_id':f'{r.id}_AB','gasoducto_id':r.id,'sentido':f'{r.punto_a_zona}-{r.punto_b_zona}',
         'entrada_nombre':r.punto_a_nombre,'entrada_zona':r.punto_a_zona,'entrada_lat':float(r.punto_a_lat),'entrada_lon':float(r.punto_a_lon),
         'salida_nombre':r.punto_b_nombre,'salida_zona':r.punto_b_zona,'salida_lat':float(r.punto_b_lat),'salida_lon':float(r.punto_b_lon)},
        {'direccion_id':f'{r.id}_BA','gasoducto_id':r.id,'sentido':f'{r.punto_b_zona}-{r.punto_a_zona}',
         'entrada_nombre':r.punto_b_nombre,'entrada_zona':r.punto_b_zona,'entrada_lat':float(r.punto_b_lat),'entrada_lon':float(r.punto_b_lon),
         'salida_nombre':r.punto_a_nombre,'salida_zona':r.punto_a_zona,'salida_lat':float(r.punto_a_lat),'salida_lon':float(r.punto_a_lon)},
    ]
direcciones_gasoductos=pd.DataFrame(direction_rows)

# Los archivos del ZIP contienen fuentes explícitas Google EIG/UIG/GJC.
# El MILP conserva sus claves AIG/AGC, pero éstas son aliases y nunca generan
# una segunda consulta ni una línea recta artificial.
for sitio_id in sorted(SITIOS_ORIGINALES_IDS):
    for direccion_id in direcciones_gasoductos['direccion_id'].astype(str):
        alias=f'AIG|{sitio_id}|{direccion_id}'; source=f'UIG|{sitio_id}|{direccion_id}'
        if alias not in _route_keys_existentes or source not in _route_keys_existentes:
            raise KeyError(f'Falta alias/fuente de ubicación-gasoducto: {alias} -> {source}')
        ROUTE_ALIAS_SOURCE[alias]=source
for direccion_id in direcciones_gasoductos['direccion_id'].astype(str):
    for cliente_id in clientes['id'].astype(str):
        alias=f'AGC|{direccion_id}|{cliente_id}'; source=f'GJC|{direccion_id}|{cliente_id}'
        if alias not in _route_keys_existentes or source not in _route_keys_existentes:
            raise KeyError(f'Falta alias/fuente de gasoducto-cliente: {alias} -> {source}')
        ROUTE_ALIAS_SOURCE[alias]=source


def _agregar_arco_si_falta(tipo, origen, destino, descripcion, modo='DRIVE', estado='PENDIENTE_CACHE', regla=''):
    route_key = f'{tipo}|{origen}|{destino}'
    if route_key in _route_keys_existentes:
        return route_key
    nuevos_arcos.append({
        'tipo_arco': tipo,
        'origen_id': str(origen),
        'destino_id': str(destino),
        'descripcion': descripcion,
        'modo_google': modo,
        'route_key': route_key,
        'estado_inicial': estado,
        'regla': regla,
    })
    _route_keys_existentes.add(route_key)
    return route_key


# Para reutilizar rutas ya existentes sin mover empresas, se identifica únicamente
# un sitio original con coordenadas EXACTAMENTE iguales a cada empresa. Este mapa
# se usa como fuente de distancia/polilínea; nunca reemplaza el nodo NE_E ni cambia
# la posición o la identidad económica de la empresa.
_sitios_originales_idx = sitios.loc[
    sitios['id'].astype(str).isin(SITIOS_ORIGINALES_IDS)
].copy()
EMPRESA_A_SITIO_ORIGINAL_EQUIVALENTE = {}
SITIO_ORIGINAL_A_EMPRESA_REPRESENTATIVA = {}
for empresa_id, (lat_e, lon_e) in COORDENADAS_EMPRESAS_ORIGINALES.items():
    candidatos = _sitios_originales_idx.loc[
        _sitios_originales_idx['lat'].astype(float).eq(float(lat_e))
        & _sitios_originales_idx['lon'].astype(float).eq(float(lon_e)),
        'id',
    ].astype(str).sort_values().tolist()
    if candidatos:
        EMPRESA_A_SITIO_ORIGINAL_EQUIVALENTE[empresa_id] = candidatos[0]
        SITIO_ORIGINAL_A_EMPRESA_REPRESENTATIVA.setdefault(candidatos[0], empresa_id)

print(
    'Empresas con sitio original exactamente coincidente para reutilizar rutas:',
    f'{len(EMPRESA_A_SITIO_ORIGINAL_EQUIVALENTE)}/{len(EMPRESA_A_NODO_COMPLETO)}',
)

# Un arco original empresa -> sitio ubicado en exactamente la misma coordenada
# se vincula al registro E2E de la propia empresa. Sigue siendo un arco distinto
# de empresa -> su NE_E y, por tanto, no se declara local dentro del MILP.
for empresa_id, sitio_equivalente in EMPRESA_A_SITIO_ORIGINAL_EQUIVALENTE.items():
    route_key = f'ABIO|{empresa_id}|{sitio_equivalente}'
    if route_key in _route_keys_existentes:
        ROUTE_ALIAS_SOURCE[route_key] = f'E2E|{empresa_id}|{empresa_id}'

# Las plantas ubicadas en empresas también pueden alimentar entradas de
# gasoducto. Cada AIG|NE_E|dirección reutiliza exactamente la ruta física
# EIG|empresa|dirección que Google Routes guarda dentro del ZIP.
for nodo_empresa in NODOS_EMPRESA_IDS:
    empresa_destino = NODO_EMPRESA_A_EMPRESA[nodo_empresa]
    for direccion_id in direcciones_gasoductos['direccion_id'].astype(str):
        alias_key = _agregar_arco_si_falta(
            'AIG', nodo_empresa, direccion_id,
            'Productora ubicada en empresa a entrada de gasoducto',
            estado='CACHE_EIG',
            regla=f'Reutiliza EIG|{empresa_destino}|{direccion_id}.',
        )
        source_key=f'EIG|{empresa_destino}|{direccion_id}'
        if source_key not in _route_keys_existentes:
            raise KeyError(f'Falta ruta fuente empresa-gasoducto: {source_key}')
        ROUTE_ALIAS_SOURCE[alias_key] = source_key

# Biogás: cada empresa disponible puede procesar localmente en su propio NE_E o
# transportarlo con costo real hacia el NE_E de otra empresa. Cada par no local
# usa como fuente explícita E2E|empresa_origen|empresa_destino, que se obtiene de
# Google Routes API y se guarda dentro del ZIP definitivo.
for empresa_origen in sorted(_ids_empresas_disponibles):
    for nodo_destino in NODOS_EMPRESA_IDS:
        empresa_destino = NODO_EMPRESA_A_EMPRESA[nodo_destino]
        es_local = empresa_origen == empresa_destino
        alias_key = _agregar_arco_si_falta(
            'ABIO', empresa_origen, nodo_destino,
            'Empresa a planta BG instalada en ubicación fija de empresa',
            modo='LOCAL' if es_local else 'DRIVE',
            estado='TRANSFERENCIA_INTERNA' if es_local else 'PENDIENTE_CACHE',
            regla=(
                'Misma empresa y mismo nodo NE_E: 0 km, 0 viajes, 0 camiones y 0 USD.'
                if es_local else
                'Empresa distinta: usa E2E explícito y requiere viajes, capacidad y costo vial.'
            ),
        )
        if not es_local:
            ROUTE_ALIAS_SOURCE[alias_key] = f'E2E|{empresa_origen}|{empresa_destino}'

# Entrega directa de GasBC desde una planta ubicada en empresa hacia un cliente.
# Cada arco del MILP ABC|NE_Ei|Cj reutiliza EXCLUSIVAMENTE el registro dedicado
# E2C|Ei|Cj del ZIP. Por tanto, Gurobi evalúa la distancia y el costo reales y el
# HTML dibuja la trayectoria E2C cuando la alternativa directa resulta óptima.
for nodo_empresa in NODOS_EMPRESA_IDS:
    empresa_id = NODO_EMPRESA_A_EMPRESA[nodo_empresa]
    for cliente_id in clientes['id'].astype(str):
        alias_key = _agregar_arco_si_falta(
            'ABC', nodo_empresa, cliente_id,
            'GasBC directo desde planta BG ubicada en empresa hacia cliente',
            modo='DRIVE',
            estado='CACHE_E2C',
            regla=f'Usa distancia y polilínea E2C|{empresa_id}|{cliente_id} del ZIP.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = f'E2C|{empresa_id}|{cliente_id}'

# Auditoría estructural: deben existir exactamente empresa x cliente aliases E2C.
_E2C_ALIAS_ESPERADOS = {
    f'ABC|{nodo_empresa}|{cliente_id}': f'E2C|{empresa_id}|{cliente_id}'
    for empresa_id, nodo_empresa in EMPRESA_A_NODO_COMPLETO.items()
    for cliente_id in clientes['id'].astype(str)
}
for alias_key, source_key in _E2C_ALIAS_ESPERADOS.items():
    pass
print('Arcos directos GasBC empresa-cliente enlazados a E2C:', len(_E2C_ALIAS_ESPERADOS))

# GasBC intermedio y productos transformados entre cualquier par de sitios
# productivos: sitios estratégicos anteriores + ubicaciones fijas de empresas.
for origen in SITIOS_PRODUCTIVOS_IDS:
    for destino in SITIOS_PRODUCTIVOS_IDS:
        if str(origen) == str(destino):
            continue
        alias_key = _agregar_arco_si_falta(
            'ASITE', origen, destino,
            'Transporte entre sitios productivos candidatos',
        )

        origen_es_empresa = str(origen) in NODO_EMPRESA_A_EMPRESA
        destino_es_empresa = str(destino) in NODO_EMPRESA_A_EMPRESA

        if origen_es_empresa and destino_es_empresa:
            empresa_origen = NODO_EMPRESA_A_EMPRESA[str(origen)]
            empresa_destino = NODO_EMPRESA_A_EMPRESA[str(destino)]
            ROUTE_ALIAS_SOURCE[alias_key] = f'E2E|{empresa_origen}|{empresa_destino}'

        elif origen_es_empresa and str(destino) in SITIOS_ORIGINALES_IDS:
            empresa_origen = NODO_EMPRESA_A_EMPRESA[str(origen)]
            source_key = f'ABIO|{empresa_origen}|{destino}'
            if source_key in _route_keys_fuente_fisica_abio:
                ROUTE_ALIAS_SOURCE[alias_key] = source_key

        elif str(origen) in SITIOS_ORIGINALES_IDS and destino_es_empresa:
            empresa_destino = NODO_EMPRESA_A_EMPRESA[str(destino)]
            source_key_abio = f'ABIO|{empresa_destino}|{origen}'
            if source_key_abio in _route_keys_fuente_fisica_abio:
                ROUTE_ALIAS_SOURCE[alias_key] = source_key_abio
            else:
                sitio_destino = EMPRESA_A_SITIO_ORIGINAL_EQUIVALENTE.get(empresa_destino)
                if sitio_destino and str(origen) != str(sitio_destino):
                    source_key = f'ASITE|{origen}|{sitio_destino}'
                    if source_key in _route_keys_existentes:
                        ROUTE_ALIAS_SOURCE[alias_key] = source_key
                elif sitio_destino and str(origen) == str(sitio_destino):
                    # Mismas coordenadas, pero nodos distintos: se conserva como arco
                    # no local del MILP y se reutiliza un E2E expl?cito de distancia cero.
                    empresa_origen_rep = SITIO_ORIGINAL_A_EMPRESA_REPRESENTATIVA.get(str(origen))
                    if empresa_origen_rep:
                        ROUTE_ALIAS_SOURCE[alias_key] = (
                            f'E2E|{empresa_origen_rep}|{empresa_destino}'
                        )

# Los trayectos desde cualquier sitio productivo hacia el nodo NC_Cj reutilizan
# exactamente la ruta del mismo origen al cliente j. El viewer/HTML no se toca.
for p in SITIOS_PRODUCTIVOS_IDS:
    for j, nodo_cliente in CLIENTE_A_NODO.items():
        source_key = f'ABC|{p}|{j}'
        alias_key = _agregar_arco_si_falta(
            'ASITE', p, nodo_cliente,
            'Productora L/C a productora RL/RC en nodo de cliente',
            estado='CACHE_REUTILIZADA',
            regla=f'Reutiliza la geometría y distancia de {source_key}; no consulta API.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = source_key

# Las plantas L/C instaladas en un nodo de cliente también pueden despachar
# BioGNL/BioGNC hacia las plantas RL/RC de otros clientes. Se reutilizan exactamente
# las trayectorias C2C dedicadas del ZIP, sin crear líneas rectas ni consultar API.
for cliente_origen, nodo_origen in CLIENTE_A_NODO.items():
    for cliente_destino, nodo_destino in CLIENTE_A_NODO.items():
        if str(cliente_origen) == str(cliente_destino):
            continue
        source_key = f'C2C|{cliente_origen}|{cliente_destino}'
        # C2C es una ruta fuente de caché, no un arco candidato del MILP.
        # Su existencia se valida después de la precarga C2C, igual que antes.
        alias_key = _agregar_arco_si_falta(
            'ASITE', nodo_origen, nodo_destino,
            'Transporte desde L/C en un cliente hacia RL/RC en otro cliente',
            estado='CACHE_C2C',
            regla=f'Reutiliza {source_key}; no consulta API.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = source_key

# Biogás crudo por vehículo: cada empresa disponible puede llevarlo directamente
# al nodo BG de cualquier cliente. La distancia y la polilínea se reutilizan de
# E2C|empresa|cliente; no se consulta ni se inventa ninguna ruta nueva.
for empresa_origen in sorted(_ids_empresas_disponibles):
    for cliente_id, nodo_cliente in CLIENTE_A_NODO.items():
        alias_key = _agregar_arco_si_falta(
            'ABIO', empresa_origen, nodo_cliente,
            'Biogás crudo en vehículo desde empresa a BG en cliente',
            modo='DRIVE',
            estado='CACHE_E2C',
            regla=f'Reutiliza E2C|{empresa_origen}|{cliente_id}.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = f'E2C|{empresa_origen}|{cliente_id}'

# Reconversión final local en el cliente, idéntica a la formulación anterior.
for j, nodo_cliente in CLIENTE_A_NODO.items():
    _agregar_arco_si_falta(
        'ABC', nodo_cliente, j,
        'Productora RL/RC a cliente dentro del mismo nodo físico',
        modo='LOCAL',
        estado='TRANSFERENCIA_INTERNA',
        regla='Mismo nodo físico declarado: 0 km, 0 viajes, 0 camiones y 0 USD.',
    )

if nuevos_arcos:
    arcos_candidatos = pd.concat(
        [arcos_candidatos, pd.DataFrame(nuevos_arcos)],
        ignore_index=True,
    )
arcos_candidatos.to_csv(DIR_INPUT / 'arcos_candidatos_google.csv', index=False)
direcciones_gasoductos.to_csv(DIR_INPUT / 'direcciones_gasoductos.csv', index=False)

# Validaciones anti-teletransportación antes de construir el MILP.
for empresa_id, nodo_local in EMPRESA_A_NODO.items():
    for destino in SITIOS_PRODUCTIVOS_IDS:
        key = f'ABIO|{empresa_id}|{destino}'
        if str(destino) != str(nodo_local):
            pass

print('Nodos físicos candidatos:', len(sitios))
print(' - sitios estratégicos originales conservados:', len(SITIOS_ORIGINALES_IDS))
print(' - ubicaciones fijas de empresas agregadas:', len(NODOS_EMPRESA_IDS))
print(' - nodos explícitos de cliente:', len(CLIENTE_A_NODO))
print('Relaciones locales empresa-su propio nodo:', len(EMPRESA_A_NODO))
print('Relaciones locales nodo-cliente:', len(CLIENTE_A_NODO))

expected_clients={
    'C1':(300000,0.50),'C2':(50000,0.68),'C3':(100000,1.02),'C4':(150000,1.04),
    'C5':(130000,1.06),'C6':(30000,0.86),'C7':(25000,0.98),'C8':(20000,0.82),
}
_client_idx=clientes.set_index('id')
#for cid,(dem,price) in expected_clients.items():

# ------------------------------------------------------------------
# PREPARACION DE PARAMETROS DEL PDF
# ------------------------------------------------------------------
# Los datos se mantienen en sus unidades originales. Las funciones de precio
# se discretizan mas adelante, cuando ya existen las variables del MILP.

empresas = empresas.copy()
clientes = clientes.copy()
gasoductos = gasoductos.copy()

for col in [
    'capacidad_max_biogas_nm3',
    'costo_activacion_biogas_usd',
    COL_PRECIO_BIOGAS_CRUDO,
    COL_OFERTA_EXCEDENTE_BIOGAS,
]:
    empresas[col] = pd.to_numeric(empresas[col], errors='coerce').fillna(0.0).astype(float)

for col in ['demanda_max_nm3', 'precio_venta_usd_nm3', COL_DEMANDA_NO_CUBIERTA]:
    clientes[col] = pd.to_numeric(clientes[col], errors='coerce').fillna(0.0).astype(float)

for col in ['costo_construccion_usd','costo_activacion_usd','costo_fijo_usd_dia','costo_variable_usd_nm3_km','costo_variable_usd_km_nm3','capacidad_max_nm3','capacidad_efectiva_nm3']:
    if col in gasoductos.columns:
        gasoductos[col] = pd.to_numeric(gasoductos[col], errors='coerce').fillna(0.0).astype(float)

empresas['capacidad_max_biogas_nm3'] = empresas['capacidad_max_biogas_nm3'].clip(lower=0.0)
clientes['demanda_max_nm3'] = clientes['demanda_max_nm3'].clip(lower=0.0)


if ANULAR_COSTO_FIJO_ACTIVACION_EMPRESAS:
    empresas['costo_activacion_biogas_usd'] = 0.0

if ANULAR_COSTO_FIJO_ACTIVACION_GASODUCTOS:
    if 'costo_fijo_usd_dia' in gasoductos.columns:
        gasoductos['costo_fijo_usd_dia'] = 0.0
    # Importante: no anular los costos fijos de gasoductos cortos dinamicos.
    # COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA y
    # COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA se cobran por binaria de activacion.

empresas[COL_OFERTA_EXCEDENTE_BIOGAS] = (
    empresas[COL_OFERTA_EXCEDENTE_BIOGAS]
    .clip(lower=0.0)
    .where(
        empresas[COL_OFERTA_EXCEDENTE_BIOGAS] <= empresas['capacidad_max_biogas_nm3'],
        empresas['capacidad_max_biogas_nm3'],
    )
)
clientes[COL_DEMANDA_NO_CUBIERTA] = (
    clientes[COL_DEMANDA_NO_CUBIERTA]
    .clip(lower=0.0)
    .where(
        clientes[COL_DEMANDA_NO_CUBIERTA] <= clientes['demanda_max_nm3'],
        clientes['demanda_max_nm3'],
    )
)

precios_biogas_cero = empresas.loc[empresas[COL_PRECIO_BIOGAS_CRUDO].abs() <= EPS, 'id'].astype(str).tolist()
if precios_biogas_cero:
    print(
        "ADVERTENCIA: hay empresas con precio base de biogas crudo igual a 0 en "
        f"{COL_PRECIO_BIOGAS_CRUDO}: {precios_biogas_cero}. "
        "El notebook lo respeta porque viene del ZIP, pero conviene reemplazarlo si se dispone del precio de mercado."
    )

cap_pos_after = int((empresas['capacidad_max_biogas_nm3'] > EPS).sum())
dem_pos_after = int((clientes['demanda_max_nm3'] > EPS).sum())
if cap_pos_after == 0:
    raise ValueError('Todas las capacidades de empresas son cero.')
if dem_pos_after == 0:
    raise ValueError('Todas las demandas de clientes son cero.')

print("Parametros de precio del PDF:")
print(f"  Columna precio biogas crudo: {COL_PRECIO_BIOGAS_CRUDO}")
print(f"  Columna excedente proveedor: {COL_OFERTA_EXCEDENTE_BIOGAS}")
print(f"  Columna demanda no cubierta cliente: {COL_DEMANDA_NO_CUBIERTA}")
print(f"  alpha compra: {100 * ALPHA_COMPRA_PORCENTAJE:.4f}% del precio base")
print(f"  beta venta: {100 * BETA_VENTA_PORCENTAJE:.4f}% del precio base")
print(f"  tramos de precio: {N_TRAMOS_PRECIO}")
print(f"  costo fijo activacion empresas anulado: {ANULAR_COSTO_FIJO_ACTIVACION_EMPRESAS}")
print(f"  costo fijo activacion gasoductos convencionales anulado: {ANULAR_COSTO_FIJO_ACTIVACION_GASODUCTOS}")
print("  costo fijo gasoductos cortos activo: biogas crudo="
      f"{COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA} USD/dia, "
      f"GasBC={COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA} USD/dia")

print("Validación correcta")
print({
    'empresas_disponibles_milp': len(empresas),
    'empresas_excluidas_mayor_oferta': len(empresas_excluidas_mayor_oferta),
    'clientes': len(clientes), 'nodos_fisicos_candidatos': len(sitios),
    'tipos_planta': len(tipos), 'corredores_gasoducto': len(gasoductos), 'gasoductos_biogas_cortos_dinamicos': len(gasoductos_biogas_cortos), 'gasoductos_cortos_a_clientes': int((gasoductos_biogas_cortos['destino_tipo']=='CLIENTE').sum()) if len(gasoductos_biogas_cortos) else 0, 'gasoductos_cortos_a_ubicaciones': int((gasoductos_biogas_cortos['destino_tipo']=='UBICACION_POSIBLE').sum()) if len(gasoductos_biogas_cortos) else 0,
    'direcciones_gasoducto':len(direcciones_gasoductos), 'arcos_candidatos': len(arcos_candidatos),
})
display(tipos[['codigo_pdf','tipo_instalacion','formato_entrada','formato_salida','capacidad_max_nm3','costo_variable_usd_nm3','costo_fijo_usd']])
display(factores_conversion)
display(clientes[['id','nombre','demanda_max_nm3',COL_DEMANDA_NO_CUBIERTA,'precio_venta_usd_nm3']])
display(gasoductos[['id','nombre','punto_a_zona','punto_b_zona','capacidad_efectiva_nm3','distancia_red_estimada_km','costo_fijo_usd_dia','costo_variable_usd_km_nm3','bidireccional']])
display(vehiculos[['codigo_pdf','nombre','capacidad_max_m3_viaje','unidad_capacidad','costo_km_usd']])

# ============================================================
# 3C. ADAPTACIÓN GENERAL A 9 GASODUCTOS / 18 EXTREMOS
# ============================================================
# Esta sección no depende de que antes existieran tres corredores. Recorre
# dinámicamente todas las filas de direcciones_gasoductos (AB y BA por cada
# gasoducto) y agrega:
#   A) un nodo BG exactamente en cada entrada direccional;
#   B) vehículos y gasoductos cortos de biogás crudo Empresa -> Entrada;
#   C) gasoductos cortos de GasBC BG-en-Empresa -> Entrada;
#   D) gasoductos cortos de GasBC desde cualquier BG -> múltiples clientes;
#   E) conversión local L/C en la entrada para continuar como BioGNL/BioGNC.
# Los tramos cortos se representan como líneas rectas. EIG/UIG/GJC siguen siendo
# rutas viales Google y se reconstruyen en la celda de precarga.

ENTRADA_GASODUCTO_A_NODO = {
    str(d): f'NGI_{d}' for d in direcciones_gasoductos['direccion_id'].astype(str)
}
NODO_ENTRADA_A_DIRECCION = {v:k for k,v in ENTRADA_GASODUCTO_A_NODO.items()}

_filas_nodos_entrada = []
for r in direcciones_gasoductos.itertuples(index=False):
    d = str(r.direccion_id)
    nodo_id = ENTRADA_GASODUCTO_A_NODO[d]
    fila = {col: np.nan for col in sitios.columns}
    fila.update({
        'id': nodo_id,
        'nombre': f'Entrada de gasoducto {d}: {r.entrada_nombre}',
        'lat': float(r.entrada_lat),
        'lon': float(r.entrada_lon),
        'region': str(r.entrada_zona),
        'ubicacion': str(r.entrada_nombre),
        'todas_tecnologias_permitidas': False,
        'fuente': 'direcciones_gasoductos.csv; extremo físico de entrada',
        'numero': 3000 + len(_filas_nodos_entrada) + 1,
        'tipo_nodo': 'Entrada de gasoducto',
        'nombre_sector': str(r.entrada_nombre),
        'latitud': float(r.entrada_lat),
        'longitud': float(r.entrada_lon),
        'formato_principal': 'BG / L / RL / C / RC',
        'justificacion_estrategica': (
            'Recibe biogás crudo o BioGNL/BioGNC desde una o varias empresas; puede '
            'producir o reconvertir GasBC, repartirlo localmente por gasoductos cortos '
            'e inyectar el remanente al corredor convencional.'
        ),
        'origen_registro': 'Derivado de direcciones_gasoductos.csv',
        'clase_nodo': 'ENTRADA_GASODUCTO',
        'referencia_local': d,
        'tecnologias_permitidas': 'BG,L,RL,C,RC',
    })
    _filas_nodos_entrada.append(fila)

nodos_entrada_gasoductos = pd.DataFrame(_filas_nodos_entrada)
NODOS_ENTRADA_GASODUCTO_IDS = nodos_entrada_gasoductos['id'].astype(str).tolist()
if set(NODOS_ENTRADA_GASODUCTO_IDS) & set(sitios['id'].astype(str)):
    raise AssertionError('Los identificadores NGI de entrada de gasoducto ya existen.')
sitios = pd.concat([sitios, nodos_entrada_gasoductos], ignore_index=True)
for nodo_id in NODOS_ENTRADA_GASODUCTO_IDS:
    TECNOLOGIAS_PERMITIDAS_POR_SITIO[nodo_id] = {'BG','L','RL','C','RC'}

# Transferencia local de GasBC disponible en NGI_d -> dirección d. Puede provenir de BG/RL/RC y no usa carretera ni Google.
_nuevos_arcos_entrada = []
_existentes_entrada = set(arcos_candidatos['route_key'].astype(str))
for d,nodo_id in ENTRADA_GASODUCTO_A_NODO.items():
    rk = f'AIG|{nodo_id}|{d}'
    DECLARED_LOCAL_ROUTE_KEYS.add(rk)
    if rk not in _existentes_entrada:
        _nuevos_arcos_entrada.append({
            'tipo_arco':'AIG','origen_id':nodo_id,'destino_id':d,
            'descripcion':'GasBC en la entrada física del gasoducto a inyección local (BG/RL/RC)',
            'modo_google':'LOCAL','route_key':rk,
            'estado_inicial':'TRANSFERENCIA_INTERNA',
            'regla':'Mismo punto físico: 0 km, 0 viajes, 0 camiones y 0 USD.',
        })
        _existentes_entrada.add(rk)
if _nuevos_arcos_entrada:
    arcos_candidatos = pd.concat(
        [arcos_candidatos, pd.DataFrame(_nuevos_arcos_entrada)],
        ignore_index=True,
    )
arcos_candidatos.to_csv(DIR_INPUT / 'arcos_candidatos_google.csv', index=False)

# Biogás crudo por vehículo: empresa -> BG situada exactamente en una entrada de
# gasoducto. Cada arco reutiliza la ruta EIG dedicada del ZIP.
for empresa_origen in sorted(_ids_empresas_disponibles):
    for d, nodo_entrada in ENTRADA_GASODUCTO_A_NODO.items():
        alias_key = _agregar_arco_si_falta(
            'ABIO', empresa_origen, nodo_entrada,
            'Biogás crudo en vehículo desde empresa a BG en entrada de gasoducto',
            modo='DRIVE',
            estado='CACHE_EIG',
            regla=f'Reutiliza EIG|{empresa_origen}|{d}.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = f'EIG|{empresa_origen}|{d}'

# Tramos entre paradas para los recorridos de hasta dos BG. Los sitios P# usan
# UIG, los nodos de empresa usan EIG y una entrada hacia un cliente usa GJC del
# sentido opuesto, cuya salida física coincide exactamente con dicha entrada.
_DIRECCION_OPUESTA = {}
for gasoducto_id, grupo in direcciones_gasoductos.groupby('gasoducto_id'):
    direcciones = sorted(grupo['direccion_id'].astype(str).tolist())
    if len(direcciones) != 2:
        raise AssertionError((gasoducto_id, direcciones))
    _DIRECCION_OPUESTA[direcciones[0]] = direcciones[1]
    _DIRECCION_OPUESTA[direcciones[1]] = direcciones[0]

# La salida física de una dirección coincide exactamente con la entrada de la
# dirección opuesta del mismo corredor. Se conserva este mapeo únicamente como
# referencia geométrica para reportes y trayectorias.
SALIDA_GASODUCTO_A_NODO = {
    str(d): ENTRADA_GASODUCTO_A_NODO[_DIRECCION_OPUESTA[str(d)]]
    for d in direcciones_gasoductos['direccion_id'].astype(str)
}
NODO_SALIDA_A_DIRECCION = {v:k for k,v in SALIDA_GASODUCTO_A_NODO.items()}
for d, nodo_salida in SALIDA_GASODUCTO_A_NODO.items():
    d_opuesta=_DIRECCION_OPUESTA[d]

for d, nodo_entrada in ENTRADA_GASODUCTO_A_NODO.items():
    for sitio_id in SITIOS_ORIGINALES_IDS:
        alias_key = _agregar_arco_si_falta(
            'ASITE', sitio_id, nodo_entrada,
            'Tramo vial entre ubicación posible y BG en entrada de gasoducto',
            estado='CACHE_UIG',
            regla=f'Reutiliza UIG|{sitio_id}|{d}.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = f'UIG|{sitio_id}|{d}'
    for empresa_destino, nodo_empresa in EMPRESA_A_NODO_COMPLETO.items():
        alias_key = _agregar_arco_si_falta(
            'ASITE', nodo_empresa, nodo_entrada,
            'Tramo vial entre ubicación de empresa y BG en entrada de gasoducto',
            estado='CACHE_EIG',
            regla=f'Reutiliza EIG|{empresa_destino}|{d}.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = f'EIG|{empresa_destino}|{d}'
    direccion_opuesta = _DIRECCION_OPUESTA[str(d)]
    for cliente_id, nodo_cliente in CLIENTE_A_NODO.items():
        alias_key = _agregar_arco_si_falta(
            'ASITE', nodo_entrada, nodo_cliente,
            'Tramo vial desde planta en entrada de gasoducto a nodo de cliente',
            estado='CACHE_GJC',
            regla=f'Reutiliza GJC|{direccion_opuesta}|{cliente_id}; la salida opuesta coincide con la entrada.',
        )
        ROUTE_ALIAS_SOURCE[alias_key] = f'GJC|{direccion_opuesta}|{cliente_id}'

# -----------------------------------------------------------------------------
# GENERALIZACIÓN DE RUTAS VIALES ENTRE NODOS P/E/C/NGI
# -----------------------------------------------------------------------------
# Para L/C -> BioGNL/BioGNC -> vehículo -> RL/RC no se crean teletransportes ni
# nuevas distancias artificiales. Si ya existe una ruta ASITE dirigida a->b y
# falta b->a, se habilita el sentido inverso reutilizando EXACTAMENTE la misma
# trayectoria física/caché subyacente. Esto es suficiente para que una L/C en
# cliente pueda despachar hacia P, ubicación de empresa o NGI, preservando todas
# las rutas previas y evitando nuevas consultas Google cuando el camino físico ya
# estaba disponible en el sentido contrario.
def _resolver_fuente_alias_route_key(route_key):
    key=str(route_key); vistos=set()
    while key in ROUTE_ALIAS_SOURCE:
        if key in vistos:
            raise RuntimeError(f'Ciclo de aliases detectado en {route_key}')
        vistos.add(key)
        key=str(ROUTE_ALIAS_SOURCE[key])
    return key

_asite_snapshot = sorted(
    rk for rk in set(_route_keys_existentes)
    if str(rk).startswith('ASITE|')
)
_asite_reversos_agregados = 0
for rk in _asite_snapshot:
    _, a, b = str(rk).split('|', 2)
    rev = f'ASITE|{b}|{a}'
    if rev in _route_keys_existentes:
        continue
    alias_key = _agregar_arco_si_falta(
        'ASITE', b, a,
        'Transporte inverso entre nodos productivos P/E/C/NGI',
        estado='CACHE_REVERSA_REUTILIZADA',
        regla=f'Reutiliza la misma trayectoria física de {rk} en sentido inverso; no consulta API.',
    )
    ROUTE_ALIAS_SOURCE[alias_key] = _resolver_fuente_alias_route_key(rk)
    _asite_reversos_agregados += 1
print('Rutas ASITE inversas P/E/C/NGI agregadas sin nueva consulta API:', _asite_reversos_agregados)

# Los aliases anteriores se crean después de la primera consolidación de
# `nuevos_arcos`; incorporar aquí solamente los arcos que todavía no están en
# `arcos_candidatos`, evitando duplicar los ya materializados.
_keys_arcos_existentes_3c = set(arcos_candidatos['route_key'].astype(str))
_arcos_nuevos_3c = [
    fila for fila in nuevos_arcos
    if str(fila['route_key']) not in _keys_arcos_existentes_3c
]
if _arcos_nuevos_3c:
    arcos_candidatos = pd.concat(
        [arcos_candidatos, pd.DataFrame(_arcos_nuevos_3c)],
        ignore_index=True,
    )
# Persistir también los aliases ABIO/ASITE creados para entradas de gasoducto.
arcos_candidatos.to_csv(DIR_INPUT / 'arcos_candidatos_google.csv', index=False)

# Empresa -> entrada de gasoducto mediante gasoducto corto de BIOGÁS CRUDO.
_filas_corto_entrada = []
for empresa_row in empresas_completas_originales.itertuples(index=False):
    empresa_id = str(empresa_row.id)
    for drow in direcciones_gasoductos.itertuples(index=False):
        d = str(drow.direccion_id)
        nodo_id = ENTRADA_GASODUCTO_A_NODO[d]
        distancia_km = _distancia_recta_km(
            empresa_row.lat, empresa_row.lon, drow.entrada_lat, drow.entrada_lon
        )
        if not (
            HABILITAR_GASODUCTOS_BIOGAS_CORTOS
            and distancia_km < float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM)
        ):
            continue
        pipe_id = f'GBCORTO_ENTRADA_{empresa_id}_{d}'
        _filas_corto_entrada.append({
            'id':pipe_id,
            'nombre':f'Gasoducto corto de biogás crudo {empresa_id} → entrada {d}',
            'empresa_id':empresa_id,
            'empresa':str(empresa_row.nombre),
            'destino_tipo':'ENTRADA_GASODUCTO',
            'destino_id':d,
            'destino_nombre':str(drow.entrada_nombre),
            'nodo_destino_id':nodo_id,
            'cliente_id':None,'cliente':None,'nodo_cliente_id':None,
            'ubicacion_id':None,'ubicacion':None,
            'direccion_id':d,'gasoducto_id':str(drow.gasoducto_id),
            'punto_a_lat':float(empresa_row.lat),'punto_a_lon':float(empresa_row.lon),
            'punto_b_lat':float(drow.entrada_lat),'punto_b_lon':float(drow.entrada_lon),
            'distancia_recta_km':float(distancia_km),
            'costo_fijo_usd_dia':float(COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA),
            'costo_variable_usd_nm3_km':float(COSTO_VARIABLE_GASODUCTO_BIOGAS_CORTO_USD_NM3_KM),
            'capacidad_max_nm3_dia':float(CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA),
            'largo_maximo_km':float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM),
            'route_key':f'SHORTBIO|{pipe_id}',
            'geometria':'LINEA_RECTA_CALCULADA_EN_IPYNB',
            'empresa_disponible_en_milp':empresa_id in _ids_empresas_disponibles,
        })

if _filas_corto_entrada:
    _df_corto_entrada = pd.DataFrame(_filas_corto_entrada)
    gasoductos_biogas_cortos_todos = pd.concat(
        [gasoductos_biogas_cortos_todos, _df_corto_entrada],
        ignore_index=True, sort=False,
    )
    gasoductos_biogas_cortos = gasoductos_biogas_cortos_todos.loc[
        gasoductos_biogas_cortos_todos['empresa_disponible_en_milp'].astype(bool)
    ].copy().reset_index(drop=True)

# BG instalada en la misma empresa -> entrada del corredor mediante gasoducto
# corto de GASBC. Esta alternativa NO crea una BG en la entrada: el biogás crudo
# se procesa en NE_E, sale como GasBC y llega listo para ingresar al corredor.
_filas_gasbc_empresa_entrada = []
for empresa_row in empresas_completas_originales.itertuples(index=False):
    empresa_id = str(empresa_row.id)
    nodo_origen_id = str(EMPRESA_A_NODO_COMPLETO[empresa_id])
    if 'BG' not in TECNOLOGIAS_PERMITIDAS_POR_SITIO[nodo_origen_id]:
        raise AssertionError(f'El nodo propio {nodo_origen_id} no permite BG.')
    for drow in direcciones_gasoductos.itertuples(index=False):
        d = str(drow.direccion_id)
        distancia_km = _distancia_recta_km(
            empresa_row.lat, empresa_row.lon, drow.entrada_lat, drow.entrada_lon
        )
        if not (
            HABILITAR_GASODUCTOS_GASBC_CORTOS_EMPRESA_ENTRADA
            and distancia_km < float(LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM)
        ):
            continue
        h = f'GECORTO_{empresa_id}_{d}'
        _filas_gasbc_empresa_entrada.append({
            'id':h,
            'nombre':f'Gasoducto corto de GasBC BG@{empresa_id} → entrada {d}',
            'empresa_id':empresa_id,
            'empresa':str(empresa_row.nombre),
            'nodo_origen_id':nodo_origen_id,
            'direccion_id':d,
            'gasoducto_id':str(drow.gasoducto_id),
            'entrada_nombre':str(drow.entrada_nombre),
            'punto_a_lat':float(empresa_row.lat),'punto_a_lon':float(empresa_row.lon),
            'punto_b_lat':float(drow.entrada_lat),'punto_b_lon':float(drow.entrada_lon),
            'distancia_recta_km':float(distancia_km),
            'costo_fijo_usd_dia':float(COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA),
            'costo_variable_usd_nm3_km':float(COSTO_VARIABLE_GASODUCTO_GASBC_CORTO_USD_NM3_KM),
            'capacidad_max_nm3_dia':float(CAPACIDAD_GASODUCTO_GASBC_CORTO_NM3_DIA),
            'largo_maximo_km':float(LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM),
            'route_key':f'SHORTGASIN|{h}',
            'geometria':'LINEA_RECTA_CALCULADA_EN_IPYNB',
            'empresa_disponible_en_milp':empresa_id in _ids_empresas_disponibles,
            'producto_transportado':'GasBC',
            'proceso_origen':'Biogás crudo -> BG en la misma empresa -> GasBC',
        })

gasoductos_gasbc_cortos_empresa_entrada_todos = pd.DataFrame(_filas_gasbc_empresa_entrada)
gasoductos_gasbc_cortos_empresa_entrada = (
    gasoductos_gasbc_cortos_empresa_entrada_todos.loc[
        gasoductos_gasbc_cortos_empresa_entrada_todos['empresa_disponible_en_milp'].astype(bool)
    ].copy().reset_index(drop=True)
    if len(gasoductos_gasbc_cortos_empresa_entrada_todos)
    else pd.DataFrame(columns=[
        'id','nombre','empresa_id','empresa','nodo_origen_id','direccion_id','gasoducto_id',
        'entrada_nombre','punto_a_lat','punto_a_lon','punto_b_lat','punto_b_lon',
        'distancia_recta_km','costo_fijo_usd_dia','costo_variable_usd_nm3_km',
        'capacidad_max_nm3_dia','largo_maximo_km','route_key','geometria',
        'empresa_disponible_en_milp','producto_transportado','proceso_origen',
    ])
)
if len(gasoductos_gasbc_cortos_empresa_entrada_todos):
    pass


# GasBC disponible en cualquier nodo físico permitido -> cliente, ubicación P#
# o extremo NGI de un gasoducto convencional mediante gasoducto corto. El GasBC
# puede proceder de una BG o de una reconversión RL/RC realizada en el nodo de
# origen. En particular se habilita P/E/C -> gasoducto corto GasBC -> NGI, para
# que ese flujo se mezcle con el GasBC extraído del corredor y ambos puedan usar
# la misma planta L/C instalada exactamente en el extremo del gasoducto.
_COLUMNAS_GASBC_EMPRESA_DESTINO = [
    'id','nombre','origen_tipo','origen_id','origen_nombre',
    'empresa_id','empresa','nodo_origen_id',
    'destino_tipo','destino_id','destino_nombre','nodo_destino_id',
    'cliente_id','cliente','ubicacion_id','ubicacion',
    'punto_a_lat','punto_a_lon','punto_b_lat','punto_b_lon',
    'distancia_recta_km','costo_fijo_usd_dia','costo_variable_usd_nm3_km',
    'capacidad_max_nm3_dia','largo_maximo_km','route_key','geometria',
    'empresa_disponible_en_milp','producto_transportado','proceso_origen',
]
_filas_gasbc_empresa_destino = []


def _clasificar_origen_gasbc(nodo_origen_id):
    nodo_origen_id = str(nodo_origen_id)
    if nodo_origen_id in NODO_EMPRESA_A_EMPRESA:
        empresa_id = str(NODO_EMPRESA_A_EMPRESA[nodo_origen_id])
        return 'UBICACION_EMPRESA', empresa_id, str(empresas_completas_originales.set_index('id').loc[empresa_id,'nombre'])
    if nodo_origen_id in set(str(x) for x in CLIENTE_A_NODO.values()):
        cliente_id = next(j for j,n in CLIENTE_A_NODO.items() if str(n)==nodo_origen_id)
        return 'CLIENTE', None, str(clientes.set_index('id').loc[cliente_id,'nombre'])
    if nodo_origen_id in set(str(x) for x in NODOS_ENTRADA_GASODUCTO_IDS):
        return 'ENTRADA_GASODUCTO', None, str(sitios.set_index('id').loc[nodo_origen_id,'nombre'])
    return 'UBICACION_POSIBLE', None, str(sitios.set_index('id').loc[nodo_origen_id,'nombre'])


def _registrar_gasbc_origen_destino(
    *, origen_row, destino_tipo, destino_id, destino_nombre,
    nodo_destino_id, destino_lat, destino_lon,
    cliente_id=None, cliente=None, ubicacion_id=None, ubicacion=None,
    permitir_distancia_cero=True,
):
    nodo_origen_id = str(origen_row.id)
    nodo_destino_id = str(nodo_destino_id)
    # Nunca se crea un arco de un nodo a sí mismo. Dos IDs lógicos diferentes que
    # comparten coordenadas sí pueden conectarse con 0 km y conservan el costo fijo
    # de activación del gasoducto corto, tal como en los casos previos NGI.
    if nodo_origen_id == nodo_destino_id:
        return
    origen_tipo, empresa_id, _ = _clasificar_origen_gasbc(nodo_origen_id)
    origen_nombre = str(origen_row.nombre)
    distancia_km = _distancia_recta_km(
        origen_row.lat, origen_row.lon, destino_lat, destino_lon
    )
    distancia_valida = (
        distancia_km < float(LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM)
        and (distancia_km > 1e-9 or bool(permitir_distancia_cero))
    )
    if not (HABILITAR_GASODUCTOS_GASBC_CORTOS_EMPRESA_DESTINO and distancia_valida):
        return

    if destino_tipo == 'CLIENTE':
        h = f'GDCORTO_BG_{nodo_origen_id}_{destino_id}'
        destino_texto = f'cliente {destino_id}'
    elif destino_tipo == 'UBICACION_POSIBLE':
        # Se mantienen los IDs históricos empresa->P y nodo->P para no romper
        # trazabilidad ni visualización de resultados anteriores.
        if origen_tipo == 'UBICACION_EMPRESA' and empresa_id in _ids_empresas_disponibles:
            h = f'GDCORTO_SITIO_{empresa_id}_{destino_id}'
        else:
            h = f'GDCORTO_SITIO_RLRC_{nodo_origen_id}_{destino_id}'
        destino_texto = f'ubicación {destino_id}'
    elif destino_tipo == 'UBICACION_EMPRESA':
        h = f'GDCORTO_EMPRESADEST_{nodo_origen_id}_{destino_id}'
        destino_texto = f'ubicación de empresa {destino_id}'
    elif destino_tipo == 'ENTRADA_GASODUCTO':
        h = f'GDCORTO_NGI_{nodo_origen_id}_{destino_id}'
        destino_texto = f'extremo de gasoducto {destino_id}'
    else:
        raise ValueError(f'Destino GasBC directo no reconocido: {destino_tipo}')

    _filas_gasbc_empresa_destino.append({
        'id':h,
        'nombre':f'Gasoducto corto directo de GasBC {nodo_origen_id} → {destino_texto}',
        'origen_tipo':origen_tipo,
        'origen_id':nodo_origen_id,
        'origen_nombre':origen_nombre,
        'empresa_id':empresa_id,
        'empresa':None if empresa_id is None else str(empresas_completas_originales.set_index('id').loc[empresa_id,'nombre']),
        'nodo_origen_id':nodo_origen_id,
        'destino_tipo':str(destino_tipo),
        'destino_id':str(destino_id),
        'destino_nombre':str(destino_nombre),
        'nodo_destino_id':nodo_destino_id,
        'cliente_id':None if cliente_id is None else str(cliente_id),
        'cliente':None if cliente is None else str(cliente),
        'ubicacion_id':None if ubicacion_id is None else str(ubicacion_id),
        'ubicacion':None if ubicacion is None else str(ubicacion),
        'punto_a_lat':float(origen_row.lat),
        'punto_a_lon':float(origen_row.lon),
        'punto_b_lat':float(destino_lat),
        'punto_b_lon':float(destino_lon),
        'distancia_recta_km':float(distancia_km),
        'costo_fijo_usd_dia':float(COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA),
        'costo_variable_usd_nm3_km':float(COSTO_VARIABLE_GASODUCTO_GASBC_CORTO_USD_NM3_KM),
        'capacidad_max_nm3_dia':float(CAPACIDAD_GASODUCTO_GASBC_CORTO_NM3_DIA),
        'largo_maximo_km':float(LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM),
        'route_key':f'SHORTGASDIRECT|{h}',
        'geometria':'LINEA_RECTA_CALCULADA_EN_IPYNB',
        'empresa_disponible_en_milp':True,
        'producto_transportado':'GasBC',
        'proceso_origen':(
            'Pool nodal GasBC: puede provenir de BG local, RL/RC local o GasBC '
            'recibido previamente por otro gasoducto corto/convencional.'
        ),
    })

# -----------------------------------------------------------------------------
# RED GENERAL DE GASBC CORTO: P/E/C/NGI -> P/E/C/NGI
# -----------------------------------------------------------------------------
# Los clientes siguen siendo los únicos destinos donde QGCD puede representar
# venta. P/E/NGI son nodos de tránsito: todo el flujo recibido entra al mismo pool
# GasBC del nodo y desde allí puede venderse, alimentar L/C, inyectarse a corredor
# o salir por otro gasoducto corto.
_sitios_idx_gasbc = sitios.set_index('id')
_nodos_cliente_set = set(str(x) for x in CLIENTE_A_NODO.values())
_nodos_empresa_set = set(str(x) for x in NODOS_EMPRESA_IDS)
_nodos_p_set = set(str(x) for x in SITIOS_ORIGINALES_IDS)
_nodos_ngi_set = set(str(x) for x in NODOS_ENTRADA_GASODUCTO_IDS)
# P/E/C son los nodos no-convencionales que pueden alimentar un NGI mediante
# gasoducto corto. Se define explícitamente para que las validaciones no dependan
# de una variable implícita/antigua.
_nodos_fuente_pec = _nodos_p_set | _nodos_empresa_set | _nodos_cliente_set
_nodos_red_gasbc = _nodos_fuente_pec | _nodos_ngi_set
_sitios_fuente_gasbc = sitios.loc[sitios['id'].astype(str).isin(_nodos_red_gasbc)].copy()

for origen_row in _sitios_fuente_gasbc.itertuples(index=False):
    # 1) cualquier nodo -> cliente
    for cliente_row in clientes.itertuples(index=False):
        j = str(cliente_row.id)
        _registrar_gasbc_origen_destino(
            origen_row=origen_row,
            destino_tipo='CLIENTE',
            destino_id=j,
            destino_nombre=str(cliente_row.nombre),
            nodo_destino_id=CLIENTE_A_NODO[j],
            destino_lat=cliente_row.lat,
            destino_lon=cliente_row.lon,
            cliente_id=j,
            cliente=str(cliente_row.nombre),
        )

    # 2) cualquier nodo -> ubicación P
    for p in sorted(_nodos_p_set):
        destino_row = _sitios_idx_gasbc.loc[p]
        _registrar_gasbc_origen_destino(
            origen_row=origen_row,
            destino_tipo='UBICACION_POSIBLE',
            destino_id=p,
            destino_nombre=str(destino_row['nombre']),
            nodo_destino_id=p,
            destino_lat=float(destino_row['lat']),
            destino_lon=float(destino_row['lon']),
            ubicacion_id=p,
            ubicacion=str(destino_row['nombre']),
        )

    # 3) cualquier nodo -> ubicación física de empresa
    for nodo_e in sorted(_nodos_empresa_set):
        destino_row = _sitios_idx_gasbc.loc[nodo_e]
        _registrar_gasbc_origen_destino(
            origen_row=origen_row,
            destino_tipo='UBICACION_EMPRESA',
            destino_id=nodo_e,
            destino_nombre=str(destino_row['nombre']),
            nodo_destino_id=nodo_e,
            destino_lat=float(destino_row['lat']),
            destino_lon=float(destino_row['lon']),
            ubicacion_id=nodo_e,
            ubicacion=str(destino_row['nombre']),
        )

    # 4) cualquier nodo -> NGI, incluyendo NGI -> NGI por gasoducto corto
    # de GasBC. Se evita sólo el autobucle lógico NGI_d -> NGI_d.
    for nodo_ngi in sorted(_nodos_ngi_set):
        if str(origen_row.id) == str(nodo_ngi):
            continue
        destino_row = _sitios_idx_gasbc.loc[nodo_ngi]
        _registrar_gasbc_origen_destino(
            origen_row=origen_row,
            destino_tipo='ENTRADA_GASODUCTO',
            destino_id=nodo_ngi,
            destino_nombre=str(destino_row['nombre']),
            nodo_destino_id=nodo_ngi,
            destino_lat=float(destino_row['lat']),
            destino_lon=float(destino_row['lon']),
            ubicacion_id=nodo_ngi,
            ubicacion=str(destino_row['nombre']),
        )

gasoductos_gasbc_cortos_empresa_destino_todos = pd.DataFrame(
    _filas_gasbc_empresa_destino,
    columns=_COLUMNAS_GASBC_EMPRESA_DESTINO,
)
gasoductos_gasbc_cortos_empresa_destino = (
    gasoductos_gasbc_cortos_empresa_destino_todos.loc[
        gasoductos_gasbc_cortos_empresa_destino_todos['empresa_disponible_en_milp'].astype(bool)
    ].copy().reset_index(drop=True)
    if len(gasoductos_gasbc_cortos_empresa_destino_todos)
    else pd.DataFrame(columns=_COLUMNAS_GASBC_EMPRESA_DESTINO)
)
if len(gasoductos_gasbc_cortos_empresa_destino_todos):
    _ngi_rows=gasoductos_gasbc_cortos_empresa_destino_todos.loc[
        gasoductos_gasbc_cortos_empresa_destino_todos['destino_tipo'].astype(str)=='ENTRADA_GASODUCTO'
    ]

# Salida del corredor -> cliente, ubicación P# O ubicación física de empresa mediante
# gasoducto corto de GasBC. En un cliente puede ser venta final (QGCP) o tránsito
# (QGCP_HUB). En una ubicación de empresa NE_E sólo se admite tránsito: el GasBC
# entra al balance nodal y desde allí puede repartirse a múltiples clientes usando
# los gasoductos cortos directos GDCORTO ya existentes. Se conserva la alternativa
# de 0 km cuando la salida convencional coincide físicamente con la empresa; en ese
# caso la distancia variable es cero, pero la activación del gasoducto corto sigue
# pagando su costo fijo exactamente como cualquier otro candidato.
_COLUMNAS_GASBC_SALIDA_DESTINO = [
    'id','nombre','direccion_id','gasoducto_id',
    'destino_tipo','destino_id','destino_nombre','nodo_destino_id',
    'cliente_id','cliente','empresa_id','empresa',
    'punto_a_lat','punto_a_lon','punto_b_lat','punto_b_lon',
    'distancia_recta_km','costo_fijo_usd_dia','costo_variable_usd_nm3_km',
    'capacidad_max_nm3_dia','largo_maximo_km','route_key','geometria',
]
_filas_gasbc_salida_cliente = []


def _registrar_gasbc_salida_destino(
    *, drow, destino_tipo, destino_id, destino_nombre, nodo_destino_id,
    destino_lat, destino_lon, cliente_id=None, cliente=None,
    empresa_id=None, empresa=None,
):
    d = str(drow.direccion_id)
    distancia_km = _distancia_recta_km(
        drow.salida_lat, drow.salida_lon, destino_lat, destino_lon
    )
    if not (
        HABILITAR_GASODUCTOS_GASBC_CORTOS_SALIDA_CLIENTE
        and distancia_km < float(LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM)
    ):
        return
    if destino_tipo == 'CLIENTE':
        h = f'GSCORTO_{d}_{destino_id}'
        destino_texto = f'cliente {destino_id}'
    elif destino_tipo == 'UBICACION_EMPRESA':
        h = f'GSCORTO_{d}_EMP_{destino_id}'
        destino_texto = f'ubicación empresa {destino_id}'
    elif destino_tipo == 'UBICACION_POSIBLE':
        h = f'GSCORTO_{d}_P_{destino_id}'
        destino_texto = f'ubicación posible {destino_id}'
    else:
        raise ValueError(f'Destino de salida convencional no reconocido: {destino_tipo}')
    _filas_gasbc_salida_cliente.append({
        'id':h,
        'nombre':f'Gasoducto corto de GasBC salida {d} → {destino_texto}',
        'direccion_id':d,
        'gasoducto_id':str(drow.gasoducto_id),
        'destino_tipo':str(destino_tipo),
        'destino_id':str(destino_id),
        'destino_nombre':str(destino_nombre),
        'nodo_destino_id':str(nodo_destino_id),
        'cliente_id':None if cliente_id is None else str(cliente_id),
        'cliente':None if cliente is None else str(cliente),
        'empresa_id':None if empresa_id is None else str(empresa_id),
        'empresa':None if empresa is None else str(empresa),
        'punto_a_lat':float(drow.salida_lat),'punto_a_lon':float(drow.salida_lon),
        'punto_b_lat':float(destino_lat),'punto_b_lon':float(destino_lon),
        'distancia_recta_km':float(distancia_km),
        'costo_fijo_usd_dia':float(COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA),
        'costo_variable_usd_nm3_km':float(COSTO_VARIABLE_GASODUCTO_GASBC_CORTO_USD_NM3_KM),
        'capacidad_max_nm3_dia':float(CAPACIDAD_GASODUCTO_GASBC_CORTO_NM3_DIA),
        'largo_maximo_km':float(LARGO_MAXIMO_GASODUCTO_GASBC_CORTO_KM),
        'route_key':f'SHORTGAS|{h}',
        'geometria':'LINEA_RECTA_CALCULADA_EN_IPYNB',
    })


for drow in direcciones_gasoductos.itertuples(index=False):
    # Salida convencional -> cliente.
    for crow in clientes.itertuples(index=False):
        j = str(crow.id)
        _registrar_gasbc_salida_destino(
            drow=drow,
            destino_tipo='CLIENTE',
            destino_id=j,
            destino_nombre=str(crow.nombre),
            nodo_destino_id=CLIENTE_A_NODO[j],
            destino_lat=crow.lat,
            destino_lon=crow.lon,
            cliente_id=j,
            cliente=str(crow.nombre),
        )

    # NUEVO: salida convencional -> ubicación posible P#. Esto hace que el extremo
    # físico NGI del gasoducto convencional pueda usarse como ubicación logística:
    # Gasoducto convencional -> NGI/salida -> gasoducto corto GasBC -> P#.
    # El flujo llega a P# como tránsito (QGCP_HUB), entra a balance_BG[P#] y desde
    # allí conserva TODAS las posibilidades existentes: L/C, venta local cuando
    # corresponda, inyección posterior o nuevos gasoductos cortos hacia P/E/C/NGI.
    for p_dest in sorted(SITIOS_ORIGINALES_IDS):
        prow = sitios.set_index('id').loc[str(p_dest)]
        _registrar_gasbc_salida_destino(
            drow=drow,
            destino_tipo='UBICACION_POSIBLE',
            destino_id=str(p_dest),
            destino_nombre=str(prow['nombre']),
            nodo_destino_id=str(p_dest),
            destino_lat=float(prow['lat']),
            destino_lon=float(prow['lon']),
        )

    # NUEVO: salida convencional -> ubicación física de empresa NE_E. La empresa
    # se utiliza sólo como localización/HUB; no se exige que sea proveedora activa.
    if HABILITAR_REENVIO_GASBC_EN_UBICACIONES_EMPRESA:
        for erow in empresas_completas_originales.itertuples(index=False):
            i_dest = str(erow.id)
            nodo_dest = str(EMPRESA_A_NODO_COMPLETO[i_dest])
            _registrar_gasbc_salida_destino(
                drow=drow,
                destino_tipo='UBICACION_EMPRESA',
                destino_id=i_dest,
                destino_nombre=str(erow.nombre),
                nodo_destino_id=nodo_dest,
                destino_lat=erow.lat,
                destino_lon=erow.lon,
                empresa_id=i_dest,
                empresa=str(erow.nombre),
            )


gasoductos_gasbc_cortos_salida_cliente = pd.DataFrame(
    _filas_gasbc_salida_cliente,
    columns=_COLUMNAS_GASBC_SALIDA_DESTINO,
)
if len(gasoductos_gasbc_cortos_salida_cliente):
    pass

print('Adaptación dinámica de gasoductos completada:', {
    'gasoductos':len(gasoductos),
    'direcciones_AB_BA':len(direcciones_gasoductos),
    'nodos_BG_en_entradas':len(NODOS_ENTRADA_GASODUCTO_IDS),
    'cortos_biogas_empresa_entrada':len(_filas_corto_entrada),
    'cortos_GasBC_BG_empresa_entrada':len(gasoductos_gasbc_cortos_empresa_entrada),
    'cortos_GasBC_BG_empresa_destino_directo':len(gasoductos_gasbc_cortos_empresa_destino),
    'cortos_GasBC_salida_cliente_P_o_empresa_HUB':len(gasoductos_gasbc_cortos_salida_cliente),
    'rutas_google_gasoducto_gasoducto':0,
})


Factores de Conversión: {'BG': 0.6, 'L': 0.00166666, 'RL': 600.0, 'C': 0.0033, 'RC': 303.03}
Empresas excluidas por mayor oferta: ninguna
Empresas disponibles para el MILP: 15 de 15
Arcos ABIO retirados por exclusión de empresas: 0
Gasoductos cortos calculados en memoria: 24
 - empresa -> cliente: 1
 - empresa -> ubicación posible: 2
 - empresa -> ubicación de empresa: 21
Gasoductos cortos disponibles para el MILP: 24


,id,empresa_id,empresa,destino_tipo,destino_id,destino_nombre,nodo_destino_id,distancia_recta_km,empresa_disponible_en_milp
0,GBCORTO_E7_C8,E7,Cervecera CCU Chile Ltda.,CLIENTE,C8,"CCU (Planta Cervecera, Quilicura)",NC_C8,0.000000,True
1,GBCORTO_SITIO_E3_P21,E3,Central de Generación El Trebal,UBICACION_POSIBLE,P21,Nodo intermedio Ñuble,P21,7.328104,True
2,GBCORTO_SITIO_E7_P12,E7,Cervecera CCU Chile Ltda.,UBICACION_POSIBLE,P12,Nodo intermedio Coquimbo central,P12,8.358690,True
3,GBCORTO_EMPRESA_E1_NE_E1,E1,Agrícola Santa Lucía Limitada,UBICACION_EMPRESA,NE_E1,Ubicación fija de la empresa E1: Agrícola Sant...,NE_E1,0.000000,True
4,GBCORTO_EMPRESA_E1_NE_E14,E1,Agrícola Santa Lucía Limitada,UBICACION_EMPRESA,NE_E14,Ubicación fija de la empresa E14: Bio Energía ...,NE_E14,0.242549,True
5,GBCORTO_EMPRESA_E2_NE_E2,E2,Central ERNC Santa Marta y Planta de Quema Cen...,UBICACION_EMPRESA,NE_E2,Ubicación fija de la empresa E2: Central ERNC ...,NE_E2,0.000000,True
6,GBCORTO_EMPRESA_E3_NE_E3,E3,Central de Generación El Trebal,UBICACION_EMPRESA,NE_E3,Ubicación fija de la empresa E3: Central de Ge...,NE_E3,0.000000,True
7,GBCORTO_EMPRESA_E4_NE_E4,E4,RRSS Cemarc (Bioenergía Los Pinos),UBICACION_EMPRESA,NE_E4,Ubicación fija de la empresa E4: RRSS Cemarc (...,NE_E4,0.000000,True
8,GBCORTO_EMPRESA_E5_NE_E5,E5,Agrosuper - Biodigestor Pocillas,UBICACION_EMPRESA,NE_E5,Ubicación fija de la empresa E5: Agrosuper - B...,NE_E5,0.000000,True
9,GBCORTO_EMPRESA_E5_NE_E10,E5,Agrosuper - Biodigestor Pocillas,UBICACION_EMPRESA,NE_E10,Ubicación fija de la empresa E10: Agrosuper - ...,NE_E10,5.903062,True


Gasoductos cortos biogás crudo cliente/HUB -> cliente: 0
Empresas con sitio original exactamente coincidente para reutilizar rutas: 0/15
Arcos directos GasBC empresa-cliente enlazados a E2C: 120
Nodos físicos candidatos: 48
 - sitios estratégicos originales conservados: 25
 - ubicaciones fijas de empresas agregadas: 15
 - nodos explícitos de cliente: 8
Relaciones locales empresa-su propio nodo: 15
Relaciones locales nodo-cliente: 8
Parametros de precio del PDF:
  Columna precio biogas crudo: precio_mercado_biogas_crudo_usd_nm3
  Columna excedente proveedor: oferta_excedente_biogas_crudo_nm3_dia
  Columna demanda no cubierta cliente: demanda_no_cubrida_nm3_dia
  alpha compra: 25.0000% del precio base
  beta venta: 25.0000% del precio base
  tramos de precio: 8
  costo fijo activacion empresas anulado: True
  costo fijo activacion gasoductos convencionales anulado: True
  costo fijo gasoductos cortos activo: biogas crudo=500.0 USD/dia, GasBC=625.0 USD/dia
Validación correcta
{'empresas_d

,codigo_pdf,tipo_instalacion,formato_entrada,formato_salida,capacidad_max_nm3,costo_variable_usd_nm3,costo_fijo_usd
0,BG,Biogas_a_GasBC,Biogas_crudo,GasBC,9999999.0,0.026627,2020.60
1,L,BioGNL,GasBC,BioGNL,9999999.0,0.138463,5599.22
2,RL,GNL_a_GasBC,BioGNL,GasBC,60000.0,0.016480,1419.29
3,C,BioGNC,GasBC,BioGNC,50000.0,0.026627,3527.34
4,RC,GNC_a_GasBC,BioGNC,GasBC,50000.0,0.003000,235.17


,codigo_pdf,tipo_instalacion,formato_entrada,formato_salida,factor_conversion_salida_por_entrada,interpretacion,aplica_capacidad_y_costo_variable_sobre,escenario
0,BG,Biogas_a_GasBC,Biogas_crudo,GasBC,0.600000,1 m3 de biogás crudo produce 0.600 Nm3 de GasBC.,entrada_procesada,valores_definidos_usuario_2026_07_30
1,L,BioGNL,GasBC,BioGNL,0.001667,1 Nm3 de GasBC produce 0.00166666 m3 de BioGNL.,entrada_procesada,valores_definidos_usuario_2026_07_30
2,RL,GNL_a_GasBC,BioGNL,GasBC,600.000000,1 m3 de BioGNL produce 600 Nm3 de GasBC.,entrada_procesada,valores_definidos_usuario_2026_07_30
3,C,BioGNC,GasBC,BioGNC,0.003300,1 Nm3 de GasBC produce 0.0033 m3 de BioGNC.,entrada_procesada,valores_definidos_usuario_2026_07_30
4,RC,GNC_a_GasBC,BioGNC,GasBC,303.030000,1 m3 de BioGNC produce 303.03 Nm3 de GasBC.,entrada_procesada,valores_definidos_usuario_2026_07_30


,id,nombre,demanda_max_nm3,demanda_no_cubrida_nm3_dia,precio_venta_usd_nm3
0,C1,Metrogas S.A. (City Gate San Bernardo),27400.0,27400.0,0.51
1,C2,Empresas Carozzi S.A. (Complejo Industrial Nos),50000.0,5000.0,0.68
2,C3,Codelco - División Chuquicamata (Mina Rajo Abi...,100000.0,5000.0,1.02
3,C4,Minera Escondida - BHP (Antofagasta),150000.0,7500.0,1.04
4,C5,Compañía Minera Doña Inés de Collahuasi (Regió...,130000.0,6500.0,1.06
5,C6,Celulosa Arauco y Constitución (Complejo Horco...,30000.0,500.0,0.86
6,C7,"Cermaq Chile S.A. (Planta Chinquihue, Puerto M...",25000.0,1730.0,0.98
7,C8,"CCU (Planta Cervecera, Quilicura)",20000.0,2000.0,0.82


,id,nombre,punto_a_zona,punto_b_zona,capacidad_efectiva_nm3,distancia_red_estimada_km,costo_fijo_usd_dia,costo_variable_usd_km_nm3,bidireccional
0,G_NC,Corredor gasoducto Norte-Centro,Norte,Centro,8000000.0,1188.00,0.0,0.00035,True
1,G_CS,Corredor gasoducto Centro-Sur,Centro,Sur,9700000.0,479.00,0.0,0.00035,True
2,G_NS,Corredor gasoducto Norte-Sur,Norte,Sur,8000000.0,1664.00,0.0,0.00035,True
3,G_LA,Lateral Arauco,Biobío,Arauco,500000.0,26.00,0.0,0.00035,True
4,G_LP,Línea Principal — Gasoducto del Pacífico,Ñuble,Biobío,6000000.0,244.20,0.0,0.00035,True
5,G_SBQ,Gasoducto San Bernardo–Quillota 24 Pulg. (Rato...,Región Metropolitana,Valparaíso,7100000.0,111.19,0.0,0.00035,True
6,G_LSR,Lateral Santa Rosa,Región Metropolitana,Región Metropolitana,8500000.0,3.75,0.0,0.00035,True
7,G_RJC,Ramal Jama–Crucero,Antofagasta,Antofagasta,127200.0,0.02,0.0,0.00035,True
8,G_PA,Cabo Negro–Gasco,Magallanes,Magallanes,2000000.0,2.80,0.0,0.00035,True


,codigo_pdf,nombre,capacidad_max_m3_viaje,unidad_capacidad,costo_km_usd
0,BIO,Biogas,30.0,m3 del formato transportado/viaje,2.20
1,GNL,GNL,40.0,m3 del formato transportado/viaje,0.93
2,GNC,GNC,30.0,m3 del formato transportado/viaje,0.98
3,BC,GasBC_Camion,40.0,m3 del formato transportado/viaje,2.20


Rutas ASITE inversas P/E/C/NGI agregadas sin nueva consulta API: 1184
Adaptación dinámica de gasoductos completada: {'gasoductos': 9, 'direcciones_AB_BA': 18, 'nodos_BG_en_entradas': 18, 'cortos_biogas_empresa_entrada': 1, 'cortos_GasBC_BG_empresa_entrada': 1, 'cortos_GasBC_BG_empresa_destino_directo': 44, 'cortos_GasBC_salida_cliente_P_o_empresa_HUB': 7, 'rutas_google_gasoducto_gasoducto': 0}


In [ ]:
# ============================================================
# 3B. PRECARGA ÚNICA GOOGLE ROUTES API
#     Conserva la precarga cliente -> cliente y completa únicamente los arcos
#     nuevos necesarios para las ubicaciones fijas de empresas.
# ============================================================
import getpass
import requests
from datetime import timedelta

CLIENT_MATRIX_PATH = DIR_CACHE / "google_client_to_client_matrix.jsonl"
CLIENT_POLYLINE_PATH = DIR_CACHE / "google_client_to_client_polylines.json"
CLIENT_MANIFEST_PATH = DIR_CACHE / "google_client_to_client_manifest.json"
COMPANY_MATRIX_PATH = DIR_CACHE / "google_company_to_company_matrix.jsonl"
COMPANY_POLYLINE_PATH = DIR_CACHE / "google_company_to_company_polylines.json"
COMPANY_MANIFEST_PATH = DIR_CACHE / "google_company_to_company_manifest.json"
COMPANY_CLIENT_MATRIX_PATH = DIR_CACHE / "google_company_to_client_matrix.jsonl"
COMPANY_CLIENT_POLYLINE_PATH = DIR_CACHE / "google_company_to_client_polylines.json"
COMPANY_CLIENT_MANIFEST_PATH = DIR_CACHE / "google_company_to_client_manifest.json"
EIG_MATRIX_PATH = DIR_CACHE / "google_company_to_gas_pipeline_matrix.jsonl"
EIG_POLYLINE_PATH = DIR_CACHE / "google_company_to_gas_pipeline_polylines.json"
EIG_MANIFEST_PATH = DIR_CACHE / "google_company_to_gas_pipeline_manifest.json"
UIG_MATRIX_PATH = DIR_CACHE / "google_site_to_gas_pipeline_matrix.jsonl"
UIG_POLYLINE_PATH = DIR_CACHE / "google_site_to_gas_pipeline_polylines.json"
UIG_MANIFEST_PATH = DIR_CACHE / "google_site_to_gas_pipeline_manifest.json"
GJC_MATRIX_PATH = DIR_CACHE / "google_gas_pipeline_to_client_matrix.jsonl"
GJC_POLYLINE_PATH = DIR_CACHE / "google_gas_pipeline_to_client_polylines.json"
GJC_MANIFEST_PATH = DIR_CACHE / "google_gas_pipeline_to_client_manifest.json"

MATRIX_ENDPOINT = "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix"
ROUTES_ENDPOINT = "https://routes.googleapis.com/directions/v2:computeRoutes"


def _duration_seconds(value):
    text = str(value or "0s").strip()
    if text.endswith("s"):
        text = text[:-1]
    try:
        return float(text or 0.0)
    except Exception:
        return 0.0


def _coord_signature_clientes(a, b):
    ca = clientes.set_index("id").loc[a]
    cb = clientes.set_index("id").loc[b]
    return "|".join(
        f"{float(v):.7f}"
        for v in (ca.lat, ca.lon, cb.lat, cb.lon)
    )


def _load_jsonl_latest(path):
    latest = {}
    if not path.exists():
        return latest
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        rec = json.loads(line)
        key = str(rec.get("route_key", "")).strip()
        if key:
            latest[key] = rec
    return latest


def _write_jsonl(path, records):
    path.write_text(
        "".join(json.dumps(records[k], ensure_ascii=False) + "\n" for k in sorted(records)),
        encoding="utf-8",
    )


client_ids = clientes["id"].astype(str).tolist()
client_pairs = [(a, b) for a in client_ids for b in client_ids if a != b]
client_keys = {f"C2C|{a}|{b}" for a, b in client_pairs}

matrix_all = _load_jsonl_latest(MATRIX_CACHE_PATH)
poly_all = json.loads(POLYLINE_CACHE_PATH.read_text(encoding="utf-8") or "{}")

# El ZIP posee archivos dedicados para cliente -> cliente. Se fusionan antes de
# comprobar faltantes; de este modo el notebook nunca ignora esas trayectorias
# ni solicita nuevamente la API cuando ya están disponibles localmente.
client_matrix_seed = _load_jsonl_latest(CLIENT_MATRIX_SEED_PATH)
client_poly_seed = json.loads(
    CLIENT_POLYLINE_SEED_PATH.read_text(encoding="utf-8") or "{}"
) if CLIENT_POLYLINE_SEED_PATH.exists() else {}
if not isinstance(client_poly_seed, dict):
    raise ValueError(f"{CLIENT_POLYLINE_SEED_PATH.name} debe contener un objeto JSON.")

matrix_all.update(client_matrix_seed)
poly_all.update(client_poly_seed)
print(
    "Registros cliente-cliente leídos directamente del ZIP:",
    {"distancias": len(client_matrix_seed), "trayectorias": len(client_poly_seed)},
)

company_matrix_seed = _load_jsonl_latest(COMPANY_MATRIX_SEED_PATH)
company_poly_seed = json.loads(
    COMPANY_POLYLINE_SEED_PATH.read_text(encoding="utf-8") or "{}"
) if COMPANY_POLYLINE_SEED_PATH.exists() else {}
if not isinstance(company_poly_seed, dict):
    raise ValueError(f"{COMPANY_POLYLINE_SEED_PATH.name} debe contener un objeto JSON.")
matrix_all.update(company_matrix_seed)
poly_all.update(company_poly_seed)
print(
    "Registros empresa-empresa leídos directamente del ZIP:",
    {"distancias": len(company_matrix_seed), "trayectorias": len(company_poly_seed)},
)

company_client_matrix_seed = _load_jsonl_latest(COMPANY_CLIENT_MATRIX_SEED_PATH)
company_client_poly_seed = json.loads(
    COMPANY_CLIENT_POLYLINE_SEED_PATH.read_text(encoding="utf-8") or "{}"
) if COMPANY_CLIENT_POLYLINE_SEED_PATH.exists() else {}
if not isinstance(company_client_poly_seed, dict):
    raise ValueError(f"{COMPANY_CLIENT_POLYLINE_SEED_PATH.name} debe contener un objeto JSON.")
matrix_all.update(company_client_matrix_seed)
poly_all.update(company_client_poly_seed)
print(
    "Registros empresa-cliente E2C leídos directamente del ZIP:",
    {"distancias": len(company_client_matrix_seed), "trayectorias": len(company_client_poly_seed)},
)

# Las tres cachés dedicadas de los gasoductos también se fusionan ANTES de
# comprobar faltantes. Así, al reutilizar un ZIP actualizado, EIG/UIG/GJC no se
# vuelven a consultar aunque la caché general haya sido reconstruida por separado.
for _familia, _matrix_path, _poly_path in [
    ('EIG', EIG_MATRIX_PATH, EIG_POLYLINE_PATH),
    ('UIG', UIG_MATRIX_PATH, UIG_POLYLINE_PATH),
    ('GJC', GJC_MATRIX_PATH, GJC_POLYLINE_PATH),
]:
    _matrix_seed = _load_jsonl_latest(_matrix_path)
    _poly_seed = json.loads(
        _poly_path.read_text(encoding='utf-8') or '{}'
    ) if _poly_path.exists() else {}
    if not isinstance(_poly_seed, dict):
        raise ValueError(f'{_poly_path.name} debe contener un objeto JSON.')
    matrix_all.update(_matrix_seed)
    poly_all.update(_poly_seed)
    print(
        f'Registros {_familia} leídos directamente del ZIP:',
        {'distancias': len(_matrix_seed), 'trayectorias': len(_poly_seed)},
    )

# Cuando cambian los puntos A/B de los nueve gasoductos se fuerzan únicamente
# las tres familias solicitadas. Las rutas C2C, E2E y E2C se conservan.
if RECONSTRUIR_RUTAS_GASODUCTOS_DESDE_CERO:
    _prefijos_refresco = ('EIG|','UIG|','GJC|')
    _eliminadas_m = [k for k in list(matrix_all) if str(k).startswith(_prefijos_refresco)]
    _eliminadas_p = [k for k in list(poly_all) if str(k).startswith(_prefijos_refresco)]
    for k in _eliminadas_m:
        matrix_all.pop(k, None)
    for k in _eliminadas_p:
        poly_all.pop(k, None)
    print('Reconstrucción forzada EIG/UIG/GJC:', {
        'distancias_eliminadas':len(_eliminadas_m),
        'trayectorias_eliminadas':len(_eliminadas_p),
    })

# La caché general también queda integrada para que todas las celdas posteriores
# utilicen una única fuente, incluida la generación del HTML.
_write_jsonl(MATRIX_CACHE_PATH, matrix_all)
POLYLINE_CACHE_PATH.write_text(
    json.dumps(poly_all, ensure_ascii=False, indent=2), encoding="utf-8"
)

missing_matrix = []
missing_poly = []
for a, b in client_pairs:
    rk = f"C2C|{a}|{b}"
    sig = _coord_signature_clientes(a, b)
    rec_m = matrix_all.get(rk)
    rec_p = poly_all.get(rk)
    if not (rec_m and rec_m.get("coordinate_signature") == sig and rec_m.get("condition") == "ROUTE_EXISTS" and rec_m.get("distance_meters") is not None):
        missing_matrix.append((a, b))
    if not (rec_p and rec_p.get("coordinate_signature") == sig and str(rec_p.get("encoded_polyline", "")).strip()):
        missing_poly.append((a, b))

print(f"Pares cliente-cliente dirigidos: {len(client_pairs)}")
print(f"Distancias faltantes: {len(missing_matrix)}; trayectorias faltantes: {len(missing_poly)}")
if not missing_matrix and not missing_poly:
    print("OK: todas las trayectorias cliente-cliente provienen del ZIP; no se usará Google Routes API.")

api_calls_matrix = 0
api_calls_routes = 0
consulta_realizada = False

if missing_matrix or missing_poly:
    if not CONSULTAR_API_CLIENTE_CLIENTE_SI_FALTA:
        raise RuntimeError(
            "Faltan datos cliente-cliente y CONSULTAR_API_CLIENTE_CLIENTE_SI_FALTA=False."
        )

    api_key = getpass.getpass("Google Maps Platform API Key (no se guardará): ").strip()
    if not api_key:
        raise ValueError("Se requiere una API key para completar la precarga cliente-cliente.")

    now = datetime.now(timezone.utc)
    expires = now + timedelta(days=30)
    fetched_iso = now.isoformat()
    expires_iso = expires.isoformat()

    # Una sola solicitud matricial para todos los pares de clientes.
    if missing_matrix:
        locations = [
            {
                "waypoint": {
                    "location": {
                        "latLng": {
                            "latitude": float(clientes.set_index("id").loc[j, "lat"]),
                            "longitude": float(clientes.set_index("id").loc[j, "lon"]),
                        }
                    }
                }
            }
            for j in client_ids
        ]
        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": api_key,
            "X-Goog-FieldMask": "originIndex,destinationIndex,status,condition,distanceMeters,duration",
        }
        body = {
            "origins": locations,
            "destinations": locations,
            "travelMode": "DRIVE",
            "routingPreference": "TRAFFIC_UNAWARE",
        }
        resp = requests.post(MATRIX_ENDPOINT, headers=headers, json=body, timeout=180)
        if not resp.ok:
            raise RuntimeError(f"Compute Route Matrix falló: HTTP {resp.status_code}: {resp.text[:1500]}")
        elements = resp.json()
        if isinstance(elements, dict):
            elements = elements.get("elements", elements.get("routeMatrix", []))
        if not isinstance(elements, list):
            raise RuntimeError("Respuesta inesperada de Compute Route Matrix.")
        api_calls_matrix = 1
        for element in elements:
            oi = int(element.get("originIndex", -1))
            di = int(element.get("destinationIndex", -1))
            if oi < 0 or di < 0 or oi == di:
                continue
            a, b = client_ids[oi], client_ids[di]
            rk = f"C2C|{a}|{b}"
            status = element.get("status") or {}
            status_code = status.get("code", 0) if isinstance(status, dict) else 0
            condition = element.get("condition")
            distance = element.get("distanceMeters")
            duration = element.get("duration")
            matrix_all[rk] = {
                "route_key": rk,
                "coordinate_signature": _coord_signature_clientes(a, b),
                "condition": condition,
                "distance_meters": distance,
                "duration_seconds": _duration_seconds(duration),
                "duration": duration,
                "status_code": status_code,
                "status_message": status.get("message", "") if isinstance(status, dict) else "",
                "fetched_at_utc": fetched_iso,
                "expires_at_utc": expires_iso,
                "api_product": "Google Routes API Compute Route Matrix - cliente a cliente",
            }

    # Compute Route Matrix no entrega polilíneas. Por eso se consulta computeRoutes
    # únicamente para los pares dirigidos cuya geometría aún falta.
    for pos, (a, b) in enumerate(missing_poly, 1):
        ca = clientes.set_index("id").loc[a]
        cb = clientes.set_index("id").loc[b]
        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": api_key,
            "X-Goog-FieldMask": "routes.distanceMeters,routes.duration,routes.polyline.encodedPolyline",
        }
        body = {
            "origin": {"location": {"latLng": {"latitude": float(ca.lat), "longitude": float(ca.lon)}}},
            "destination": {"location": {"latLng": {"latitude": float(cb.lat), "longitude": float(cb.lon)}}},
            "travelMode": "DRIVE",
            "routingPreference": "TRAFFIC_UNAWARE",
            "polylineQuality": "HIGH_QUALITY",
            "polylineEncoding": "ENCODED_POLYLINE",
        }
        resp = requests.post(ROUTES_ENDPOINT, headers=headers, json=body, timeout=180)
        if not resp.ok:
            raise RuntimeError(f"Compute Routes {a}->{b} falló: HTTP {resp.status_code}: {resp.text[:1500]}")
        payload = resp.json()
        routes = payload.get("routes") or []
        if not routes:
            raise RuntimeError(f"Google no devolvió trayectoria para {a}->{b}.")
        route = routes[0]
        encoded = ((route.get("polyline") or {}).get("encodedPolyline") or "").strip()
        if not encoded:
            raise RuntimeError(f"La trayectoria {a}->{b} no contiene encodedPolyline.")
        rk = f"C2C|{a}|{b}"
        poly_all[rk] = {
            "route_key": rk,
            "coordinate_signature": _coord_signature_clientes(a, b),
            "encoded_polyline": encoded,
            "distance_meters": route.get("distanceMeters"),
            "duration": route.get("duration"),
            "fetched_at_utc": fetched_iso,
            "expires_at_utc": expires_iso,
            "api_product": "Google Routes API Compute Routes - trayectoria cliente a cliente",
            "status": "ROUTE_EXISTS",
        }
        api_calls_routes += 1
        if pos % 10 == 0 or pos == len(missing_poly):
            print(f"Trayectorias cliente-cliente obtenidas: {pos}/{len(missing_poly)}")

    consulta_realizada = True
    _write_jsonl(MATRIX_CACHE_PATH, matrix_all)
    POLYLINE_CACHE_PATH.write_text(json.dumps(poly_all, ensure_ascii=False, indent=2), encoding="utf-8")

# -----------------------------------------------------------------------------
# PRECARGA EXPLÍCITA EMPRESA -> EMPRESA
# -----------------------------------------------------------------------------
# Se guardan 15 x 15 registros dirigidos E2E. La diagonal y los pares de
# coordenadas idénticas se construyen determinísticamente a distancia cero. Los
# demás pares faltantes se consultan con Compute Routes, que entrega en una sola
# respuesta la distancia, duración y polilínea de ese par.
_company_idx = empresas_completas_originales.set_index('id')
company_ids = empresas_completas_originales['id'].astype(str).tolist()
company_pairs_all = [(a, b) for a in company_ids for b in company_ids]
company_keys = {f'E2E|{a}|{b}' for a, b in company_pairs_all}


def _coord_signature_empresas(a, b):
    ca = _company_idx.loc[a]
    cb = _company_idx.loc[b]
    return '|'.join(
        f'{float(v):.7f}'
        for v in (ca.lat, ca.lon, cb.lat, cb.lon)
    )


def _encode_polyline_google(points):
    """Codifica una secuencia [(lat, lon), ...] con Encoded Polyline Algorithm."""
    resultado = []
    lat_anterior = 0
    lon_anterior = 0

    def _encode_delta(delta):
        valor = ~(delta << 1) if delta < 0 else (delta << 1)
        chars = []
        while valor >= 0x20:
            chars.append(chr((0x20 | (valor & 0x1F)) + 63))
            valor >>= 5
        chars.append(chr(valor + 63))
        return ''.join(chars)

    for lat, lon in points:
        lat_i = int(round(float(lat) * 1e5))
        lon_i = int(round(float(lon) * 1e5))
        resultado.append(_encode_delta(lat_i - lat_anterior))
        resultado.append(_encode_delta(lon_i - lon_anterior))
        lat_anterior, lon_anterior = lat_i, lon_i
    return ''.join(resultado)


def _company_matrix_valida(rec, a, b):
    return bool(
        rec
        and rec.get('coordinate_signature') == _coord_signature_empresas(a, b)
        and rec.get('condition') == 'ROUTE_EXISTS'
        and rec.get('distance_meters') is not None
    )


def _company_poly_valida(rec, a, b):
    return bool(
        rec
        and rec.get('coordinate_signature') == _coord_signature_empresas(a, b)
        and str(rec.get('encoded_polyline', '')).strip()
    )


now_e2e = datetime.now(timezone.utc)
expires_e2e = now_e2e + timedelta(days=30)
fetched_e2e_iso = now_e2e.isoformat()
expires_e2e_iso = expires_e2e.isoformat()

# La misma coordenada no se consulta a Google. Se guarda igualmente una
# trayectoria explícita y el modelo sigue distinguiendo empresas diferentes.
pares_cero_generados = 0
for a, b in company_pairs_all:
    ca = (float(_company_idx.loc[a, 'lat']), float(_company_idx.loc[a, 'lon']))
    cb = (float(_company_idx.loc[b, 'lat']), float(_company_idx.loc[b, 'lon']))
    if ca != cb:
        continue
    rk = f'E2E|{a}|{b}'
    sig = _coord_signature_empresas(a, b)
    encoded = _encode_polyline_google([ca, cb])
    matrix_all[rk] = {
        'route_key': rk,
        'coordinate_signature': sig,
        'condition': 'ROUTE_EXISTS',
        'distance_meters': 0.0,
        'duration_seconds': 0.0,
        'duration': '0s',
        'status_code': 0,
        'status_message': '',
        'fetched_at_utc': fetched_e2e_iso,
        'expires_at_utc': expires_e2e_iso,
        'api_product': (
            'Registro determinístico empresa-empresa: coordenadas idénticas'
        ),
        'same_company': bool(a == b),
        'same_coordinates': True,
    }
    poly_all[rk] = {
        'route_key': rk,
        'coordinate_signature': sig,
        'encoded_polyline': encoded,
        'distance_meters': 0.0,
        'duration': '0s',
        'fetched_at_utc': fetched_e2e_iso,
        'expires_at_utc': expires_e2e_iso,
        'api_product': (
            'Polilínea determinística empresa-empresa: coordenadas idénticas'
        ),
        'status': 'ROUTE_EXISTS',
        'same_company': bool(a == b),
        'same_coordinates': True,
    }
    pares_cero_generados += 1

faltantes_empresa_empresa = []
for a, b in company_pairs_all:
    if a == b:
        continue
    rk = f'E2E|{a}|{b}'
    if not (
        _company_matrix_valida(matrix_all.get(rk), a, b)
        and _company_poly_valida(poly_all.get(rk), a, b)
    ):
        faltantes_empresa_empresa.append((a, b))

print('Pares empresa-empresa dirigidos totales:', len(company_pairs_all))
print('Pares empresa-empresa no diagonales:', len(company_pairs_all) - len(company_ids))
print('Pares empresa-empresa con coordenadas idénticas normalizados:', pares_cero_generados)
print('Pares empresa-empresa que requieren Google Routes API:', len(faltantes_empresa_empresa))

api_calls_routes_empresas = 0
consulta_realizada_empresas = False

if faltantes_empresa_empresa:
    if not CONSULTAR_API_EMPRESA_EMPRESA_SI_FALTA:
        raise RuntimeError(
            'Faltan distancias/trayectorias empresa-empresa y '
            'CONSULTAR_API_EMPRESA_EMPRESA_SI_FALTA=False. Primeros pares:\n'
            + '\n'.join(f'{a}->{b}' for a, b in faltantes_empresa_empresa[:30])
        )

    if not ('api_key' in globals() and str(api_key).strip()):
        api_key = getpass.getpass(
            'Google Maps Platform API Key para completar empresa -> empresa '
            '(no se guardará): '
        ).strip()
    if not api_key:
        raise ValueError(
            'Se requiere una API key para completar las rutas empresa-empresa.'
        )

    def _compute_route_con_reintentos(a, b, max_intentos=4):
        ca = _company_idx.loc[a]
        cb = _company_idx.loc[b]
        headers = {
            'Content-Type': 'application/json',
            'X-Goog-Api-Key': api_key,
            'X-Goog-FieldMask': (
                'routes.distanceMeters,routes.duration,'
                'routes.polyline.encodedPolyline'
            ),
        }
        body = {
            'origin': {
                'location': {
                    'latLng': {
                        'latitude': float(ca.lat),
                        'longitude': float(ca.lon),
                    }
                }
            },
            'destination': {
                'location': {
                    'latLng': {
                        'latitude': float(cb.lat),
                        'longitude': float(cb.lon),
                    }
                }
            },
            'travelMode': 'DRIVE',
            'routingPreference': 'TRAFFIC_UNAWARE',
            'polylineQuality': 'HIGH_QUALITY',
            'polylineEncoding': 'ENCODED_POLYLINE',
        }
        ultimo_error = None
        for intento in range(1, max_intentos + 1):
            resp = requests.post(
                ROUTES_ENDPOINT, headers=headers, json=body, timeout=180
            )
            if resp.ok:
                return resp
            ultimo_error = (
                f'HTTP {resp.status_code}: {resp.text[:1500]}'
            )
            if resp.status_code not in {429, 500, 502, 503, 504}:
                break
            time.sleep(min(8.0, 0.75 * (2 ** (intento - 1))))
        raise RuntimeError(
            f'Compute Routes empresa-empresa {a}->{b} falló: {ultimo_error}'
        )

    for posicion, (a, b) in enumerate(faltantes_empresa_empresa, 1):
        resp = _compute_route_con_reintentos(a, b)
        payload = resp.json()
        routes = payload.get('routes') or []
        if not routes:
            raise RuntimeError(f'Google no devolvió ruta para empresa {a}->{b}.')
        route = routes[0]
        encoded = ((route.get('polyline') or {}).get('encodedPolyline') or '').strip()
        distance_meters = route.get('distanceMeters')
        duration = route.get('duration')
        if not encoded or distance_meters is None:
            raise RuntimeError(
                f'Google devolvió una ruta empresa-empresa incompleta para {a}->{b}.'
            )

        rk = f'E2E|{a}|{b}'
        sig = _coord_signature_empresas(a, b)
        matrix_all[rk] = {
            'route_key': rk,
            'coordinate_signature': sig,
            'condition': 'ROUTE_EXISTS',
            'distance_meters': distance_meters,
            'duration_seconds': _duration_seconds(duration),
            'duration': duration,
            'status_code': 0,
            'status_message': '',
            'fetched_at_utc': fetched_e2e_iso,
            'expires_at_utc': expires_e2e_iso,
            'api_product': 'Google Routes API Compute Routes - empresa a empresa',
            'origin_company_id': a,
            'destination_company_id': b,
            'same_company': False,
            'same_coordinates': False,
        }
        poly_all[rk] = {
            'route_key': rk,
            'coordinate_signature': sig,
            'encoded_polyline': encoded,
            'distance_meters': distance_meters,
            'duration': duration,
            'fetched_at_utc': fetched_e2e_iso,
            'expires_at_utc': expires_e2e_iso,
            'api_product': 'Google Routes API Compute Routes - empresa a empresa',
            'status': 'ROUTE_EXISTS',
            'origin_company_id': a,
            'destination_company_id': b,
            'same_company': False,
            'same_coordinates': False,
        }
        api_calls_routes_empresas += 1
        consulta_realizada = True
        consulta_realizada_empresas = True

        # Guardado incremental para no perder pares si Colab se interrumpe.
        if posicion % 10 == 0 or posicion == len(faltantes_empresa_empresa):
            company_matrix_tmp = {
                k: v for k, v in matrix_all.items() if k in company_keys
            }
            company_poly_tmp = {
                k: v for k, v in poly_all.items() if k in company_keys
            }
            _write_jsonl(COMPANY_MATRIX_PATH, company_matrix_tmp)
            COMPANY_POLYLINE_PATH.write_text(
                json.dumps(company_poly_tmp, ensure_ascii=False, indent=2),
                encoding='utf-8',
            )
            _write_jsonl(MATRIX_CACHE_PATH, matrix_all)
            POLYLINE_CACHE_PATH.write_text(
                json.dumps(poly_all, ensure_ascii=False, indent=2),
                encoding='utf-8',
            )
            print(
                'Rutas empresa-empresa obtenidas:',
                f'{posicion}/{len(faltantes_empresa_empresa)}',
            )
        time.sleep(0.05)

restantes_empresa_empresa = []
for a, b in company_pairs_all:
    rk = f'E2E|{a}|{b}'
    if not (
        _company_matrix_valida(matrix_all.get(rk), a, b)
        and _company_poly_valida(poly_all.get(rk), a, b)
    ):
        restantes_empresa_empresa.append((a, b))
if restantes_empresa_empresa:
    raise RuntimeError(
        'La precarga empresa-empresa quedó incompleta. Primeros pares:\n'
        + '\n'.join(f'{a}->{b}' for a, b in restantes_empresa_empresa[:30])
    )

company_matrix_records = {
    k: v for k, v in matrix_all.items() if k in company_keys
}
company_poly_records = {
    k: v for k, v in poly_all.items() if k in company_keys
}
_write_jsonl(COMPANY_MATRIX_PATH, company_matrix_records)
COMPANY_POLYLINE_PATH.write_text(
    json.dumps(company_poly_records, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
COMPANY_MANIFEST_PATH.write_text(
    json.dumps(
        {
            'tipo': 'precarga_empresa_empresa',
            'empresas': len(company_ids),
            'pares_dirigidos_totales_incluida_diagonal': len(company_pairs_all),
            'pares_dirigidos_no_diagonales': len(company_pairs_all) - len(company_ids),
            'registros_distancia': len(company_matrix_records),
            'registros_trayectoria': len(company_poly_records),
            'pares_cero_por_coordenadas_identicas': pares_cero_generados,
            'solicitudes_compute_routes': api_calls_routes_empresas,
            'consulta_realizada_en_esta_ejecucion': consulta_realizada_empresas,
            'route_key_schema': 'E2E|empresa_origen|empresa_destino',
            'archivos': [
                COMPANY_MATRIX_PATH.name,
                COMPANY_POLYLINE_PATH.name,
            ],
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)

# Las cachés generales también incluyen E2E para que los alias ABIO/ASITE usados
# por el MILP y por el HTML se materialicen sin una consulta posterior.
_write_jsonl(MATRIX_CACHE_PATH, matrix_all)
POLYLINE_CACHE_PATH.write_text(
    json.dumps(poly_all, ensure_ascii=False, indent=2), encoding='utf-8'
)

# -----------------------------------------------------------------------------
# VALIDACIÓN EXPLÍCITA EMPRESA -> CLIENTE (E2C)
# -----------------------------------------------------------------------------
# Estos 15 x 8 pares son la fuente exacta de los arcos ABC|NE_Ei|Cj. No se
# aproximan con líneas rectas ni se sustituyen por rutas desde puntos estratégicos.
_company_client_idx_e = empresas_completas_originales.set_index('id')
_company_client_idx_j = clientes.set_index('id')
company_client_pairs = [
    (e, j)
    for e in empresas_completas_originales['id'].astype(str)
    for j in clientes['id'].astype(str)
]
company_client_keys = {f'E2C|{e}|{j}' for e, j in company_client_pairs}


def _coord_signature_empresa_cliente(e, j):
    ce = _company_client_idx_e.loc[e]
    cj = _company_client_idx_j.loc[j]
    return '|'.join(
        f'{float(v):.7f}'
        for v in (ce.lat, ce.lon, cj.lat, cj.lon)
    )


def _e2c_matrix_valida(rec, e, j):
    return bool(
        rec
        and rec.get('coordinate_signature') == _coord_signature_empresa_cliente(e, j)
        and rec.get('condition') == 'ROUTE_EXISTS'
        and rec.get('distance_meters') is not None
    )


def _e2c_poly_valida(rec, e, j):
    return bool(
        rec
        and rec.get('coordinate_signature') == _coord_signature_empresa_cliente(e, j)
        and str(rec.get('encoded_polyline', '')).strip()
    )


faltantes_empresa_cliente = [
    (e, j)
    for e, j in company_client_pairs
    if not (
        _e2c_matrix_valida(matrix_all.get(f'E2C|{e}|{j}'), e, j)
        and _e2c_poly_valida(poly_all.get(f'E2C|{e}|{j}'), e, j)
    )
]
if faltantes_empresa_cliente and EXIGIR_EMPRESA_CLIENTE_E2C_COMPLETO:
    raise RuntimeError(
        'El ZIP E2C está incompleto o no coincide con las coordenadas actuales. '
        'Primeros pares faltantes/inválidos:\n'
        + '\n'.join(f'{e}->{j}' for e, j in faltantes_empresa_cliente[:30])
    )
if faltantes_empresa_cliente:
    raise RuntimeError(
        'No se permite construir los arcos directos empresa-cliente sin E2C completo.'
    )

company_client_matrix_records = {
    k: matrix_all[k] for k in sorted(company_client_keys)
}
company_client_poly_records = {
    k: poly_all[k] for k in sorted(company_client_keys)
}
_write_jsonl(COMPANY_CLIENT_MATRIX_PATH, company_client_matrix_records)
COMPANY_CLIENT_POLYLINE_PATH.write_text(
    json.dumps(company_client_poly_records, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
COMPANY_CLIENT_MANIFEST_PATH.write_text(
    json.dumps(
        {
            'tipo': 'precarga_empresa_cliente',
            'empresas': len(empresas_completas_originales),
            'clientes': len(clientes),
            'pares_dirigidos_totales': len(company_client_pairs),
            'registros_distancia': len(company_client_matrix_records),
            'registros_trayectoria': len(company_client_poly_records),
            'consulta_realizada_en_esta_ejecucion': False,
            'route_key_schema': 'E2C|empresa_origen|cliente_destino',
            'uso_en_modelo': 'ABC|NE_Ei|Cj mediante ROUTE_ALIAS_SOURCE',
            'uso_en_html': 'polilínea E2C clonada al alias ABC seleccionado',
            'archivos': [
                COMPANY_CLIENT_MATRIX_PATH.name,
                COMPANY_CLIENT_POLYLINE_PATH.name,
            ],
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)
print('Pares empresa-cliente E2C validados para GasBC directo:', len(company_client_pairs))

# -----------------------------------------------------------------------------
# PRECARGA DE ARCOS NUEVOS PARA NODOS DE EMPRESA
# -----------------------------------------------------------------------------
# La formulación agrega NE_Ei como sitios candidatos, pero nunca modifica las
# coordenadas de las empresas. Los arcos originales permanecen en la caché; aquí
# solo se consultan las rutas nuevas no locales y no cubiertas por alias exactos.
_E_COORD_PRE = {
    str(r.id): (float(r.lat), float(r.lon))
    for r in empresas_completas_originales.itertuples(index=False)
}
_J_COORD_PRE = {
    str(r.id): (float(r.lat), float(r.lon))
    for r in clientes.itertuples(index=False)
}
_S_COORD_PRE = {
    str(r.id): (float(r.lat), float(r.lon))
    for r in sitios.itertuples(index=False)
}
_GI_COORD_PRE = {
    str(r.direccion_id): (float(r.entrada_lat), float(r.entrada_lon))
    for r in direcciones_gasoductos.itertuples(index=False)
}
_GO_COORD_PRE = {
    str(r.direccion_id): (float(r.salida_lat), float(r.salida_lon))
    for r in direcciones_gasoductos.itertuples(index=False)
}


def _coords_route_key_general(rk):
    typ, a, b = str(rk).split('|', 2)
    if typ == 'ABIO':
        return _E_COORD_PRE[a], _S_COORD_PRE[b]
    if typ == 'ABC':
        return _S_COORD_PRE[a], _J_COORD_PRE[b]
    if typ == 'ASITE':
        return _S_COORD_PRE[a], _S_COORD_PRE[b]
    if typ == 'AIG':
        return _S_COORD_PRE[a], _GI_COORD_PRE[b]
    if typ == 'AGC':
        return _GO_COORD_PRE[a], _J_COORD_PRE[b]
    if typ == 'C2C':
        return _J_COORD_PRE[a], _J_COORD_PRE[b]
    if typ == 'E2E':
        return _E_COORD_PRE[a], _E_COORD_PRE[b]
    if typ == 'EIG':
        return _E_COORD_PRE[a], _GI_COORD_PRE[b]
    if typ == 'UIG':
        return _S_COORD_PRE[a], _GI_COORD_PRE[b]
    if typ == 'GJC':
        return _GO_COORD_PRE[a], _J_COORD_PRE[b]
    if typ == 'E2C':
        return _E_COORD_PRE[a], _J_COORD_PRE[b]
    raise KeyError(rk)


def _coord_signature_general(rk):
    origen, destino = _coords_route_key_general(rk)
    return '|'.join(f'{float(v):.7f}' for v in (*origen, *destino))


def _matrix_general_valida(rec, rk):
    return bool(
        rec
        and rec.get('coordinate_signature') == _coord_signature_general(rk)
        and rec.get('condition') == 'ROUTE_EXISTS'
        and rec.get('distance_meters') is not None
    )


def _poly_general_valida(rec, rk):
    return bool(
        rec
        and rec.get('coordinate_signature') == _coord_signature_general(rk)
        and str(rec.get('encoded_polyline', '')).strip()
    )


candidate_keys_modelo = set(arcos_candidatos['route_key'].astype(str))
keys_sin_consulta = set(DECLARED_LOCAL_ROUTE_KEYS) | set(ROUTE_ALIAS_SOURCE)
keys_consultables = sorted(candidate_keys_modelo - keys_sin_consulta)

# Auditoría: Google sólo puede consultar las familias solicitadas o las familias
# viales originales. No existe ninguna clave gasoducto -> gasoducto.
if CONSULTAR_API_GASODUCTO_GASODUCTO:
    raise ValueError('CONSULTAR_API_GASODUCTO_GASODUCTO debe permanecer False.')
if any(str(k).startswith(('GG|','G2G|','GAS2GAS|')) for k in candidate_keys_modelo):
    raise AssertionError('Se detectó una ruta gasoducto-gasoducto prohibida.')

faltantes_nuevos = [
    rk for rk in keys_consultables
    if not (
        _matrix_general_valida(matrix_all.get(rk), rk)
        and _poly_general_valida(poly_all.get(rk), rk)
    )
]

# IMPORTANTE: los interruptores se revisan únicamente para familias que
# REALMENTE siguen faltando. Antes se recorrían todas las claves consultables y
# se lanzaba el error aunque EIG/UIG/GJC ya estuvieran completas en el ZIP.
_tipos_faltantes = {str(rk).split('|', 1)[0] for rk in faltantes_nuevos}
if 'EIG' in _tipos_faltantes and not CONSULTAR_API_EMPRESA_GASODUCTO_SI_FALTA:
    _ejemplos = [rk for rk in faltantes_nuevos if rk.startswith('EIG|')][:10]
    raise RuntimeError(
        'Faltan rutas EIG reales y su consulta está deshabilitada. Primeros registros:\n'
        + '\n'.join(_ejemplos)
    )
if 'UIG' in _tipos_faltantes and not CONSULTAR_API_UBICACION_GASODUCTO_SI_FALTA:
    _ejemplos = [rk for rk in faltantes_nuevos if rk.startswith('UIG|')][:10]
    raise RuntimeError(
        'Faltan rutas UIG reales y su consulta está deshabilitada. Primeros registros:\n'
        + '\n'.join(_ejemplos)
    )
if 'GJC' in _tipos_faltantes and not CONSULTAR_API_GASODUCTO_CLIENTE_SI_FALTA:
    _ejemplos = [rk for rk in faltantes_nuevos if rk.startswith('GJC|')][:10]
    raise RuntimeError(
        'Faltan rutas GJC reales y su consulta está deshabilitada. Primeros registros:\n'
        + '\n'.join(_ejemplos)
    )

# La lista debe estar dominada por rutas que incluyen al menos un NE_E. Si la
# caché original también tiene un registro inválido, se repara en la misma pasada.
print('Arcos viales nuevos o inválidos que requieren precarga:', len(faltantes_nuevos))
api_calls_routes_nodos_empresa = 0

if faltantes_nuevos:
    # Cada familia se autoriza con su propio interruptor. Ya no se exige el
    # interruptor genérico de nodos de empresa para EIG/UIG/GJC.
    _permiso_por_tipo = {
        'EIG':bool(CONSULTAR_API_EMPRESA_GASODUCTO_SI_FALTA),
        'UIG':bool(CONSULTAR_API_UBICACION_GASODUCTO_SI_FALTA),
        'GJC':bool(CONSULTAR_API_GASODUCTO_CLIENTE_SI_FALTA),
    }
    _bloqueadas = [
        rk for rk in faltantes_nuevos
        if not _permiso_por_tipo.get(
            str(rk).split('|',1)[0],
            bool(CONSULTAR_API_NODOS_EMPRESA_SI_FALTA),
        )
    ]
    if _bloqueadas:
        raise RuntimeError(
            'Hay rutas viales faltantes cuya familia tiene la consulta API deshabilitada. '
            'Primeros registros:\n' + '\n'.join(_bloqueadas[:30])
        )

    # Se solicita una sola API key para toda la ejecución. No se guarda en el
    # notebook, en los resultados ni en el ZIP descargado.
    if not ('api_key' in globals() and str(api_key).strip()):
        api_key = getpass.getpass(
            'Google Maps Platform API Key para completar EIG/UIG/GJC y otros arcos viales faltantes '
            '(no se guardará): '
        ).strip()
    if not api_key:
        raise ValueError('Se requiere una API key para completar los nuevos arcos viales.')

    now_general = datetime.now(timezone.utc)
    expires_general = now_general + timedelta(days=30)
    fetched_general_iso = now_general.isoformat()
    expires_general_iso = expires_general.isoformat()

    for posicion, rk in enumerate(faltantes_nuevos, 1):
        origen, destino = _coords_route_key_general(rk)
        headers = {
            'Content-Type': 'application/json',
            'X-Goog-Api-Key': api_key,
            'X-Goog-FieldMask': (
                'routes.distanceMeters,routes.duration,'
                'routes.polyline.encodedPolyline'
            ),
        }
        body = {
            'origin': {
                'location': {
                    'latLng': {
                        'latitude': float(origen[0]),
                        'longitude': float(origen[1]),
                    }
                }
            },
            'destination': {
                'location': {
                    'latLng': {
                        'latitude': float(destino[0]),
                        'longitude': float(destino[1]),
                    }
                }
            },
            'travelMode': 'DRIVE',
            'routingPreference': 'TRAFFIC_UNAWARE',
            'polylineQuality': 'HIGH_QUALITY',
            'polylineEncoding': 'ENCODED_POLYLINE',
        }
        resp = requests.post(ROUTES_ENDPOINT, headers=headers, json=body, timeout=180)
        if not resp.ok:
            raise RuntimeError(
                f'Compute Routes falló para {rk}: HTTP {resp.status_code}: '
                f'{resp.text[:1500]}'
            )
        payload = resp.json()
        routes = payload.get('routes') or []
        if not routes:
            raise RuntimeError(f'Google no devolvió una ruta vial para {rk}.')
        route = routes[0]
        encoded = ((route.get('polyline') or {}).get('encodedPolyline') or '').strip()
        distance_meters = route.get('distanceMeters')
        duration = route.get('duration')
        if not encoded or distance_meters is None:
            raise RuntimeError(f'Ruta incompleta devuelta por Google para {rk}.')

        signature = _coord_signature_general(rk)
        matrix_all[rk] = {
            'route_key': rk,
            'coordinate_signature': signature,
            'condition': 'ROUTE_EXISTS',
            'distance_meters': distance_meters,
            'duration_seconds': _duration_seconds(duration),
            'duration': duration,
            'status_code': 0,
            'status_message': '',
            'fetched_at_utc': fetched_general_iso,
            'expires_at_utc': expires_general_iso,
            'api_product': 'Google Routes API Compute Routes - rutas gasoducto y nodos' ,
        }
        poly_all[rk] = {
            'route_key': rk,
            'coordinate_signature': signature,
            'encoded_polyline': encoded,
            'distance_meters': distance_meters,
            'duration': duration,
            'fetched_at_utc': fetched_general_iso,
            'expires_at_utc': expires_general_iso,
            'api_product': 'Google Routes API Compute Routes - rutas gasoducto y nodos' ,
            'status': 'ROUTE_EXISTS',
        }
        api_calls_routes_nodos_empresa += 1
        consulta_realizada = True

        # Persistencia incremental para no perder las rutas ya obtenidas si la
        # sesión de Colab se interrumpe durante una precarga extensa.
        if posicion % 25 == 0 or posicion == len(faltantes_nuevos):
            _write_jsonl(MATRIX_CACHE_PATH, matrix_all)
            POLYLINE_CACHE_PATH.write_text(
                json.dumps(poly_all, ensure_ascii=False, indent=2),
                encoding='utf-8',
            )
            print(
                'Rutas de nodos de empresa obtenidas:',
                f'{posicion}/{len(faltantes_nuevos)}',
            )

# Verificación inmediata. Los alias se materializan en la celda siguiente desde
# sus rutas fuente; los arcos locales se normalizan allí a distancia cero.
restantes_nuevos = [
    rk for rk in keys_consultables
    if not (
        _matrix_general_valida(matrix_all.get(rk), rk)
        and _poly_general_valida(poly_all.get(rk), rk)
    )
]
if restantes_nuevos:
    raise RuntimeError(
        'La precarga de nodos de empresa quedó incompleta. Primeros registros:\n'
        + '\n'.join(restantes_nuevos[:30])
    )

# Archivos dedicados para los tres grupos nuevos solicitados.
def _exportar_familia_gasoducto(prefijo, matrix_path, poly_path, manifest_path, descripcion):
    keys=sorted(k for k in candidate_keys_modelo if str(k).startswith(prefijo+'|'))
    mrec={k:matrix_all[k] for k in keys if k in matrix_all}
    prec={k:poly_all[k] for k in keys if k in poly_all}
    faltan=[k for k in keys if k not in mrec or k not in prec]
    if faltan:
        raise RuntimeError(f'La familia {prefijo} quedó incompleta: {faltan[:20]}')
    _write_jsonl(matrix_path,mrec)
    poly_path.write_text(json.dumps(prec,ensure_ascii=False,indent=2),encoding='utf-8')
    manifest_path.write_text(json.dumps({'tipo':descripcion,'route_key_schema':prefijo+'|origen|destino','registros_distancia':len(mrec),'registros_trayectoria':len(prec),'consulta_google':True,'gasoducto_a_gasoducto':False,'updated_at_utc':datetime.now(timezone.utc).isoformat()},ensure_ascii=False,indent=2),encoding='utf-8')
    return len(keys)

conteo_eig=_exportar_familia_gasoducto('EIG',EIG_MATRIX_PATH,EIG_POLYLINE_PATH,EIG_MANIFEST_PATH,'empresa_a_entrada_gasoducto')
conteo_uig=_exportar_familia_gasoducto('UIG',UIG_MATRIX_PATH,UIG_POLYLINE_PATH,UIG_MANIFEST_PATH,'ubicacion_a_entrada_gasoducto')
conteo_gjc=_exportar_familia_gasoducto('GJC',GJC_MATRIX_PATH,GJC_POLYLINE_PATH,GJC_MANIFEST_PATH,'salida_gasoducto_a_cliente')
print({'EIG':conteo_eig,'UIG':conteo_uig,'GJC':conteo_gjc,'gasoducto_gasoducto':0})

# Archivos dedicados, además de su integración en las cachés generales.
client_matrix_records = {k: v for k, v in matrix_all.items() if k in client_keys}
client_poly_records = {k: v for k, v in poly_all.items() if k in client_keys}
_write_jsonl(CLIENT_MATRIX_PATH, client_matrix_records)
CLIENT_POLYLINE_PATH.write_text(
    json.dumps(client_poly_records, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
CLIENT_MANIFEST_PATH.write_text(
    json.dumps(
        {
            'tipo': 'precarga_cliente_cliente',
            'pares_dirigidos': len(client_pairs),
            'solicitudes_compute_route_matrix': api_calls_matrix,
            'solicitudes_compute_routes_polilineas': api_calls_routes,
            'consulta_realizada_en_esta_ejecucion': bool(
                api_calls_matrix or api_calls_routes
            ),
            'archivos': [CLIENT_MATRIX_PATH.name, CLIENT_POLYLINE_PATH.name],
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)

# Actualiza el manifiesto general sin borrar sus campos originales.
try:
    manifest_general = json.loads(
        MANIFEST_CACHE_PATH.read_text(encoding='utf-8') or '{}'
    )
except Exception:
    manifest_general = {}
manifest_general.update({
    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    'api_key_saved': False,
    'company_to_company': {
        'route_key_schema': 'E2E|empresa_origen|empresa_destino',
        'companies': len(company_ids),
        'directed_pairs_including_diagonal': len(company_pairs_all),
        'matrix_records': len(company_matrix_records),
        'polyline_records': len(company_poly_records),
        'dedicated_files': [
            COMPANY_MATRIX_PATH.name,
            COMPANY_POLYLINE_PATH.name,
            COMPANY_MANIFEST_PATH.name,
        ],
    },
    'company_to_client': {
        'route_key_schema': 'E2C|empresa_origen|cliente_destino',
        'companies': len(empresas_completas_originales),
        'clients': len(clientes),
        'directed_pairs': len(company_client_pairs),
        'matrix_records': len(company_client_matrix_records),
        'polyline_records': len(company_client_poly_records),
        'model_alias_schema': 'ABC|NE_Ei|Cj',
        'dedicated_files': [
            COMPANY_CLIENT_MATRIX_PATH.name,
            COMPANY_CLIENT_POLYLINE_PATH.name,
            COMPANY_CLIENT_MANIFEST_PATH.name,
        ],
    },
    'client_to_client': {
        'directed_pairs': len(client_pairs),
        'matrix_records': len(client_matrix_records),
        'polyline_records': len(client_poly_records),
        'dedicated_files': [
            CLIENT_MATRIX_PATH.name,
            CLIENT_POLYLINE_PATH.name,
            CLIENT_MANIFEST_PATH.name,
        ],
    },
    'note_definitive_zip': (
        'Incluye rutas explícitas E2E, E2C y C2C. Los arcos directos de GasBC '
        'ABC|NE_Ei|Cj usan E2C para costos y para el trazado del HTML.'
    ),
})
MANIFEST_CACHE_PATH.write_text(
    json.dumps(manifest_general, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

# Copia inmediata dentro del árbol extraído y crea el ZIP definitivo reutilizable.
DIR_API_SEED.mkdir(parents=True, exist_ok=True)
archivos_api_actualizados = [
    MATRIX_CACHE_PATH,
    POLYLINE_CACHE_PATH,
    MANIFEST_CACHE_PATH,
    CLIENT_MATRIX_PATH,
    CLIENT_POLYLINE_PATH,
    CLIENT_MANIFEST_PATH,
    COMPANY_MATRIX_PATH,
    COMPANY_POLYLINE_PATH,
    COMPANY_MANIFEST_PATH,
    COMPANY_CLIENT_MATRIX_PATH,
    COMPANY_CLIENT_POLYLINE_PATH,
    COMPANY_CLIENT_MANIFEST_PATH,
    EIG_MATRIX_PATH, EIG_POLYLINE_PATH, EIG_MANIFEST_PATH,
    UIG_MATRIX_PATH, UIG_POLYLINE_PATH, UIG_MANIFEST_PATH,
    GJC_MATRIX_PATH, GJC_POLYLINE_PATH, GJC_MANIFEST_PATH,
]
for p in archivos_api_actualizados:
    if not p.exists():
        raise FileNotFoundError(f'No se generó el archivo obligatorio {p.name}.')
    shutil.copy2(p, DIR_API_SEED / p.name)

ZIP_DATOS_DEFINITIVO = CONTENT / NOMBRE_ZIP_DATOS_DEFINITIVO
staging_definitivo = CONTENT / '_staging_zip_definitivo_biometano'
if staging_definitivo.exists():
    shutil.rmtree(staging_definitivo)
shutil.copytree(BASE_DATOS, staging_definitivo / CARPETA_RAIZ_DATOS)
if ZIP_DATOS_DEFINITIVO.exists():
    ZIP_DATOS_DEFINITIVO.unlink()
with zipfile.ZipFile(ZIP_DATOS_DEFINITIVO, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in staging_definitivo.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(staging_definitivo))
shutil.rmtree(staging_definitivo)

print('ZIP definitivo completo creado:', ZIP_DATOS_DEFINITIVO)
print({
    'matrix_calls_clientes': api_calls_matrix,
    'route_calls_clientes': api_calls_routes,
    'route_calls_empresa_empresa': api_calls_routes_empresas,
    'route_calls_otros_nodos_empresa': api_calls_routes_nodos_empresa,
    'pairs_clientes': len(client_pairs),
    'pairs_empresa_empresa_incluida_diagonal': len(company_pairs_all),
    'company_matrix_records': len(company_matrix_records),
    'company_polyline_records': len(company_poly_records),
    'pairs_empresa_cliente': len(company_client_pairs),
    'company_client_matrix_records': len(company_client_matrix_records),
    'company_client_polyline_records': len(company_client_poly_records),
})

if EN_COLAB and DESCARGAR_ZIP_DEFINITIVO_COMPLETO:
    print('Descargando automáticamente el ZIP definitivo completo...')
    files.download(str(ZIP_DATOS_DEFINITIVO))


Registros cliente-cliente leídos directamente del ZIP: {'distancias': 56, 'trayectorias': 56}
Registros empresa-empresa leídos directamente del ZIP: {'distancias': 225, 'trayectorias': 225}
Registros empresa-cliente E2C leídos directamente del ZIP: {'distancias': 120, 'trayectorias': 120}
Registros EIG leídos directamente del ZIP: {'distancias': 0, 'trayectorias': 0}
Registros UIG leídos directamente del ZIP: {'distancias': 0, 'trayectorias': 0}
Registros GJC leídos directamente del ZIP: {'distancias': 0, 'trayectorias': 0}
Pares cliente-cliente dirigidos: 56
Distancias faltantes: 0; trayectorias faltantes: 0
OK: todas las trayectorias cliente-cliente provienen del ZIP; no se usará Google Routes API.
Pares empresa-empresa dirigidos totales: 225
Pares empresa-empresa no diagonales: 210
Pares empresa-empresa con coordenadas idénticas normalizados: 15
Pares empresa-empresa que requieren Google Routes API: 0
Pares empresa-cliente E2C validados para GasBC directo: 120
Arcos viales nuevos o 

# 4

Los costos variables de `tipos_plantas.csv` se aplican al volumen de entrada procesado por cada tecnología.

Los factores se leen exclusivamente desde `factores_conversion.csv`. En esta versión no se limita el factor a valores menores o iguales a uno, porque una reconversión desde m³ físicos de BioGNL o BioGNC hacia Nm³ de GasBC puede tener factores 600 o 303,03.

Las capacidades de camión se expresan en m³ del formato transportado por viaje. Cada viaje corresponde a ida y vuelta y el costo vial se calcula como `2 × costo_km × distancia_ida`.


In [ ]:
# ============================================================
# 4. LECTURA Y VALIDACIÓN DE LA CACHÉ LOCAL COMPLETA
#    Integra C2C, E2E, E2C, EIG, UIG y GJC; materializa aliases del MILP.
# ============================================================
def utcnow():
    return datetime.now(timezone.utc)


def parse_iso(s):
    if not s:
        return None
    try:
        return datetime.fromisoformat(str(s).replace("Z", "+00:00"))
    except Exception:
        return None


def load_matrix_cache(path=MATRIX_CACHE_PATH):
    latest = {}
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL inválido en {path.name}, línea {line_no}: {exc}") from exc
            key = str(rec.get("route_key", "")).strip()
            if key:
                latest[key] = rec
    return latest


def load_polyline_cache(path=POLYLINE_CACHE_PATH):
    data = json.loads(path.read_text(encoding="utf-8") or "{}")
    if not isinstance(data, dict):
        raise ValueError(f"{path.name} debe contener un objeto JSON.")
    return data


def save_polyline_cache(poly_cache, path=POLYLINE_CACHE_PATH):
    path.write_text(json.dumps(poly_cache, ensure_ascii=False, indent=2), encoding="utf-8")


def node_tables():
    # EIG y E2E pueden referirse a cualquiera de las 15 empresas originales,
    # incluso cuando alguna fue excluida como proveedora del MILP. Su ubicación
    # física sigue disponible como sitio candidato, por lo que la validación de
    # caché debe conservar las coordenadas del conjunto original completo.
    empresas_para_coordenadas = (
        empresas_completas_originales
        if "empresas_completas_originales" in globals()
        else empresas
    )
    E_coord = {str(r.id): (float(r.lat), float(r.lon)) for r in empresas_para_coordenadas.itertuples()}
    J_coord = {str(r.id): (float(r.lat), float(r.lon)) for r in clientes.itertuples()}
    S_coord = {str(r.id): (float(r.lat), float(r.lon)) for r in sitios.itertuples()}
    GI_coord = {str(r.direccion_id): (float(r.entrada_lat), float(r.entrada_lon)) for r in direcciones_gasoductos.itertuples()}
    GO_coord = {str(r.direccion_id): (float(r.salida_lat), float(r.salida_lon)) for r in direcciones_gasoductos.itertuples()}
    return E_coord, J_coord, S_coord, GI_coord, GO_coord


E_COORD, J_COORD, S_COORD, GI_COORD, GO_COORD = node_tables()


def coord_for_route_key(rk):
    typ, a, b = str(rk).split("|", 2)
    if typ == "ABIO": return E_COORD[a], S_COORD[b]
    if typ == "ABC": return S_COORD[a], J_COORD[b]
    if typ == "ASITE": return S_COORD[a], S_COORD[b]
    if typ == "AIG": return S_COORD[a], GI_COORD[b]
    if typ == "AGC": return GO_COORD[a], J_COORD[b]
    if typ == "C2C": return J_COORD[a], J_COORD[b]
    if typ == "E2E": return E_COORD[a], E_COORD[b]
    if typ == "E2C": return E_COORD[a], J_COORD[b]
    # Familias fuente consultadas por Google Routes para los gasoductos.
    if typ == "EIG": return E_COORD[a], GI_COORD[b]
    if typ == "UIG": return S_COORD[a], GI_COORD[b]
    if typ == "GJC": return GO_COORD[a], J_COORD[b]
    raise KeyError(f"Tipo de route_key no soportado: {rk}")


def coordinate_signature(rk):
    o, d = coord_for_route_key(rk)
    return "|".join(f"{float(v):.7f}" for v in (*o, *d))


def signature_ok(rec, rk):
    return bool(rec and rec.get("coordinate_signature") == coordinate_signature(rk))


def haversine_metros(coord_a, coord_b):
    lat1, lon1 = map(math.radians, map(float, coord_a)); lat2, lon2 = map(math.radians, map(float, coord_b))
    dlat = lat2-lat1; dlon = lon2-lon1
    h = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return 2.0*6_371_008.8*math.asin(min(1.0, math.sqrt(max(0.0, h))))


def route_same_location(rk):
    return str(rk) in DECLARED_LOCAL_ROUTE_KEYS


def matrix_record_usable(rec, rk):
    return route_same_location(rk) or bool(signature_ok(rec, rk) and rec.get("condition") == "ROUTE_EXISTS" and rec.get("distance_meters") is not None)


def poly_valid(rec, rk):
    return bool(signature_ok(rec, rk) and str(rec.get("encoded_polyline", "")).strip())


def encode_polyline(points):
    output=[]; prev_lat=0; prev_lon=0
    for lat,lon in points:
        ilat=int(round(float(lat)*1e5)); ilon=int(round(float(lon)*1e5))
        for delta in (ilat-prev_lat, ilon-prev_lon):
            value=~(delta<<1) if delta<0 else (delta<<1)
            while value>=0x20:
                output.append(chr((0x20|(value&0x1F))+63)); value >>= 5
            output.append(chr(value+63))
        prev_lat,prev_lon=ilat,ilon
    return "".join(output)


_abio_origenes=set(arcos_candidatos.loc[arcos_candidatos['tipo_arco'].astype(str).eq('ABIO'),'origen_id'].astype(str))
_empresas_abio_sin_coordenadas=sorted(_abio_origenes-set(E_COORD))
if _empresas_abio_sin_coordenadas:
    raise RuntimeError('Hay arcos ABIO de empresas excluidas o inexistentes: '+', '.join(_empresas_abio_sin_coordenadas))

candidate_keys=set(arcos_candidatos['route_key'].astype(str))
client_to_client_keys={f'C2C|{a}|{b}' for a in J_COORD for b in J_COORD if a!=b}
all_required_keys=candidate_keys|client_to_client_keys
cache=load_matrix_cache(); poly_cache=load_polyline_cache()

for alias_key,source_key in ROUTE_ALIAS_SOURCE.items():
    source_matrix=cache.get(source_key)
    if not source_matrix:
        raise RuntimeError(f'No existe en caché la ruta fuente {source_key} para crear {alias_key}.')
    alias_matrix=dict(source_matrix); alias_matrix['route_key']=alias_key; alias_matrix['coordinate_signature']=coordinate_signature(alias_key); alias_matrix['cache_alias_source']=source_key
    cache[alias_key]=alias_matrix
    source_poly=poly_cache.get(source_key)
    if source_poly:
        alias_poly=dict(source_poly); alias_poly['route_key']=alias_key; alias_poly['coordinate_signature']=coordinate_signature(alias_key); alias_poly['cache_alias_source']=source_key
        poly_cache[alias_key]=alias_poly

faltantes_matriz=sorted(rk for rk in all_required_keys if not matrix_record_usable(cache.get(rk),rk))
if faltantes_matriz:
    raise RuntimeError(f'La caché no contiene {len(faltantes_matriz)} distancias válidas. Primeros registros:\n'+'\n'.join(faltantes_matriz[:30]))

arc_meta={str(r.route_key):(str(r.tipo_arco),str(r.origen_id),str(r.destino_id)) for r in arcos_candidatos.itertuples()}
records=[]; normalizados_cero=[]; misma_ubicacion=[]
for rk in sorted(all_required_keys):
    rec=cache.get(rk,{})
    distancia=rec.get('distance_meters'); condicion=rec.get('condition'); duracion=rec.get('duration_seconds')
    es_local=route_same_location(rk)
    if es_local:
        distancia=0.0; duracion=0.0; condicion='ROUTE_EXISTS'; misma_ubicacion.append(rk); normalizados_cero.append(rk)
    elif condicion=='ROUTE_EXISTS' and distancia is None and float(duracion or 0.0)==0.0:
        distancia=0.0; normalizados_cero.append(rk)
    if rk in arc_meta:
        tipo,orig,dest=arc_meta[rk]
    else:
        _,orig,dest=rk.split('|',2); tipo='C2C'
    records.append({
        'route_key':rk,'tipo_arco':tipo,'origen_id':orig,'destino_id':dest,
        'condition':condicion,'distance_meters':distancia,'duration_seconds':duracion,
        'fetched_at_utc':rec.get('fetched_at_utc'),'expires_at_utc':rec.get('expires_at_utc'),
        'cache_valida_coordenadas':True,'normalizado_local_distancia_cero':rk in normalizados_cero,
        'misma_ubicacion':es_local,'distancia_geodesica_m':haversine_metros(*coord_for_route_key(rk)),
        'status_message':'Transferencia interna por mismo nodo físico declarado' if es_local else rec.get('status_message',''),
    })

distancias=pd.DataFrame(records)
distancias['ruta_valida']=(distancias['condition']=='ROUTE_EXISTS') & distancias['distance_meters'].notna()
distancias.to_csv(DIR_RESULTADOS/'Matriz_Rutas_Google_Normalizada.csv',index=False)
all_valid_road_keys=set(distancias.loc[distancias['ruta_valida'],'route_key'])

for rk in sorted(all_valid_road_keys):
    fila=distancias.set_index('route_key').loc[rk]
    es_local=bool(fila['misma_ubicacion'])
    if not es_local and poly_valid(poly_cache.get(rk),rk):
        continue
    if float(fila['distance_meters'])==0.0:
        origin,destination=coord_for_route_key(rk); matrix_rec=cache.get(rk,{})
        poly_cache[rk]={
            'route_key':rk,'coordinate_signature':coordinate_signature(rk),
            'fetched_at_utc':matrix_rec.get('fetched_at_utc'),'expires_at_utc':matrix_rec.get('expires_at_utc'),
            'api_product':'Representación local de tramo de distancia cero',
            'encoded_polyline':encode_polyline([origin,destination]),'distance_meters':0,'duration':'0s','status':'LOCAL_ZERO_LENGTH',
        }
save_polyline_cache(poly_cache)

faltantes_polilinea=sorted(rk for rk in all_valid_road_keys if not poly_valid(poly_cache.get(rk),rk))
if faltantes_polilinea:
    raise RuntimeError(f'Faltan {len(faltantes_polilinea)} trayectorias en caché. Primeros registros:\n'+'\n'.join(faltantes_polilinea[:30]))

expiry_values=[parse_iso(rec.get('expires_at_utc')) for rec in cache.values()]+[parse_iso(rec.get('expires_at_utc')) for rec in poly_cache.values()]
expiry_values=[x for x in expiry_values if x is not None]
earliest_expiry=min(expiry_values) if expiry_values else None
if earliest_expiry and earliest_expiry<=utcnow():
    print('ADVERTENCIA: la vigencia temporal declarada de la caché ya pasó:',earliest_expiry.isoformat())

print('LECTURA LOCAL COMPLETA')
print(f' - Arcos originales validados: {len(candidate_keys):,}')
print(f' - Arcos cliente-cliente validados: {len(client_to_client_keys):,}')
print(f' - Fuentes E2C empresa-cliente disponibles: {len(company_client_keys):,}')
print(f' - Aliases ABC empresa-cliente materializados desde E2C: {len(_E2C_ALIAS_ESPERADOS):,}')
print(f' - Rutas viales utilizables: {len(all_valid_road_keys):,}')
print(f' - Transferencias locales declaradas sin camión: {len(set(misma_ubicacion)):,}')
print(distancias.groupby(['tipo_arco','ruta_valida']).size().unstack(fill_value=0))

LECTURA LOCAL COMPLETA
 - Arcos originales validados: 7,048
 - Arcos cliente-cliente validados: 56
 - Fuentes E2C empresa-cliente disponibles: 120
 - Aliases ABC empresa-cliente materializados desde E2C: 120
 - Rutas viales utilizables: 7,104
 - Transferencias locales declaradas sin camión: 41
ruta_valida  True
tipo_arco        
ABC           328
ABIO          990
AGC           144
AIG           738
ASITE        3984
C2C            56
EIG           270
GJC           144
UIG           450


# 5

In [ ]:
# ============================================================
# 5. CONJUNTOS DISPERSOS, COSTOS VIALES Y RED DE REPARTO
# ============================================================
valid=distancias.loc[distancias['ruta_valida']].copy(); valid['distance_km']=valid['distance_meters']/1000.0
DIST=valid.set_index('route_key')['distance_km'].to_dict()
LOCAL_ROAD_KEYS=set(valid.loc[valid['misma_ubicacion'].astype(bool),'route_key'].astype(str))

def road_arc_requires_truck(route_key): return str(route_key) not in LOCAL_ROAD_KEYS

I=empresas['id'].astype(str).tolist(); J=clientes['id'].astype(str).tolist(); S=sitios['id'].astype(str).tolist(); _IDS_EMPRESAS_MILP=set(I)
valid_abio=valid.loc[valid.tipo_arco.eq('ABIO') & valid['origen_id'].astype(str).isin(_IDS_EMPRESAS_MILP)]
ABIO=[tuple(x) for x in valid_abio[['origen_id','destino_id']].itertuples(index=False,name=None)]
ABC=[tuple(x) for x in valid.loc[valid.tipo_arco=='ABC',['origen_id','destino_id']].itertuples(index=False,name=None)]
ASITE=[tuple(x) for x in valid.loc[valid.tipo_arco=='ASITE',['origen_id','destino_id']].itertuples(index=False,name=None)]
AIG=[tuple(x) for x in valid.loc[valid.tipo_arco=='AIG',['origen_id','destino_id']].itertuples(index=False,name=None)]
AGC=[tuple(x) for x in valid.loc[valid.tipo_arco=='AGC',['origen_id','destino_id']].itertuples(index=False,name=None)]
C2C=[tuple(x) for x in valid.loc[valid.tipo_arco=='C2C',['origen_id','destino_id']].itertuples(index=False,name=None)]

G=gasoductos['id'].tolist(); D=direcciones_gasoductos['direccion_id'].tolist(); T=['BG','L','RL','C','RC']
E=empresas.set_index('id'); CL=clientes.set_index('id'); PL=tipos.set_index('codigo_pdf'); VEH=vehiculos.set_index('codigo_pdf'); GA=gasoductos.set_index('id'); GD=direcciones_gasoductos.set_index('direccion_id'); DG=GD['gasoducto_id'].to_dict()

# Red convencional entre los 9 gasoductos. El CSV contiene 36 pares únicos;
# se expande a 72 arcos dirigidos con la misma distancia en ambos sentidos.
DIST_GG={}
for _r in distancias_entre_gasoductos_36.itertuples(index=False):
    _a=str(_r.gasoducto_1_id); _b=str(_r.gasoducto_2_id); _d=float(_r.distancia_km)
    DIST_GG[(_a,_b)]=_d
    DIST_GG[(_b,_a)]=_d
GG_DIR=[(_a,_b) for _a in G for _b in G if _a!=_b]
# Los nueve gasoductos usan el mismo costo unitario en los datos actuales.
COSTO_GG_USD_KM_NM3=float(GA.iloc[0]['costo_variable_usd_km_nm3'])
DIST_INTRA_GASODUCTO_KM={g:float(GA.loc[g,'distancia_red_estimada_km']) for g in G}
for _g,_dist in DIST_INTRA_GASODUCTO_KM.items():
    if not np.isfinite(_dist) or _dist<=0:
        raise ValueError(f'Distancia interna A/B invalida en gasoductos.csv del ZIP para {_g}: {_dist}')
print('Distancias internas A/B leidas desde gasoductos.csv del ZIP:',DIST_INTRA_GASODUCTO_KM)
GG_out=defaultdict(list); GG_in=defaultdict(list)
for _a,_b in GG_DIR:
    GG_out[_a].append(_b); GG_in[_b].append(_a)

# Gasoductos cortos de biogás crudo calculados en memoria.
H_CORTO=gasoductos_biogas_cortos['id'].astype(str).tolist()
GCORTO=gasoductos_biogas_cortos.set_index('id')
H_CORTO_by_i=defaultdict(list)
H_CORTO_by_s=defaultdict(list)
H_CORTO_by_j=defaultdict(list)
H_CORTO_by_tipo=defaultdict(list)
H_CORTO_CLIENTE=[]
H_CORTO_UBICACION=[]
H_CORTO_ENTRADA_GASODUCTO=[]
GASODUCTO_CORTO_BY_COMPANY_BG={}
for h in H_CORTO:
    i=str(GCORTO.loc[h,'empresa_id'])
    s=str(GCORTO.loc[h,'nodo_destino_id'])
    destino_tipo=str(GCORTO.loc[h,'destino_tipo'])
    H_CORTO_by_i[i].append(h)
    H_CORTO_by_s[s].append(h)
    H_CORTO_by_tipo[destino_tipo].append(h)
    if destino_tipo=='CLIENTE':
        j=str(GCORTO.loc[h,'cliente_id'])
        H_CORTO_by_j[j].append(h)
        H_CORTO_CLIENTE.append(h)
    elif destino_tipo=='UBICACION_POSIBLE':
        H_CORTO_UBICACION.append(h)
    elif destino_tipo=='UBICACION_EMPRESA':
        H_CORTO_UBICACION.append(h)
    elif destino_tipo=='ENTRADA_GASODUCTO':
        d=str(GCORTO.loc[h,'direccion_id'])
        H_CORTO_ENTRADA_GASODUCTO.append(h)
    else:
        raise AssertionError((h,destino_tipo))
    pair=(i,s)
    if pair in GASODUCTO_CORTO_BY_COMPANY_BG:
        raise AssertionError(f'Dos gasoductos cortos apuntan al mismo par empresa-sitio: {pair}')
    GASODUCTO_CORTO_BY_COMPANY_BG[pair]=h


# Gasoductos cortos de biogás crudo desde una ubicación de empresa usada como HUB.
# Se reutiliza exactamente la geometría/costo del candidato físico empresa -> destino,
# pero el flujo QBCP_HUB NO representa producción de esa empresa: sólo tránsito del
# biogás que ya estaba disponible en su nodo NE_E. Se excluye el arco NE_E -> NE_E.
_HUB_EMPRESA_df = gasoductos_biogas_cortos_todos.copy()
_HUB_EMPRESA_df['nodo_origen_id'] = _HUB_EMPRESA_df['empresa_id'].astype(str).map(EMPRESA_A_NODO_COMPLETO)
_HUB_EMPRESA_df = _HUB_EMPRESA_df.loc[
    HABILITAR_RELEVO_BIOGAS_CRUDO_EN_EMPRESAS
    & (_HUB_EMPRESA_df['nodo_origen_id'].astype(str) != _HUB_EMPRESA_df['nodo_destino_id'].astype(str))
].copy().reset_index(drop=True)
gasoductos_biogas_cortos_hub_empresa = _HUB_EMPRESA_df
H_EHUB = gasoductos_biogas_cortos_hub_empresa['id'].astype(str).tolist()
EHUB = (
    gasoductos_biogas_cortos_hub_empresa.set_index('id')
    if len(gasoductos_biogas_cortos_hub_empresa) else pd.DataFrame()
)
H_EHUB_SET=set(H_EHUB); H_CORTO_SET=set(H_CORTO)
H_EHUB_SHARED=[h for h in H_EHUB if h in H_CORTO_SET]
H_EHUB_SOLO=[h for h in H_EHUB if h not in H_CORTO_SET]
H_EHUB_by_origen_s=defaultdict(list); H_EHUB_by_destino_s=defaultdict(list)
for h in H_EHUB:
    so=str(EHUB.loc[h,'nodo_origen_id']); sd=str(EHUB.loc[h,'nodo_destino_id'])
    H_EHUB_by_origen_s[so].append(h); H_EHUB_by_destino_s[sd].append(h)
    if h in H_CORTO_SET:
        pass
print('Gasoductos cortos disponibles para relevo de biogás en ubicación de empresa:',len(H_EHUB),
      '| compartidos con producción:',len(H_EHUB_SHARED),'| sólo HUB:',len(H_EHUB_SOLO))


# Gasoductos cortos de biogás crudo cliente/HUB -> cliente.
H_BIOC2C=(
    gasoductos_biogas_cortos_cliente_cliente['id'].astype(str).tolist()
    if len(gasoductos_biogas_cortos_cliente_cliente) else []
)
BIOC2C=(
    gasoductos_biogas_cortos_cliente_cliente.set_index('id')
    if len(gasoductos_biogas_cortos_cliente_cliente) else pd.DataFrame()
)
H_BIOC2C_by_origen_s=defaultdict(list)
H_BIOC2C_by_destino_s=defaultdict(list)
H_BIOC2C_by_origen_j=defaultdict(list)
H_BIOC2C_by_destino_j=defaultdict(list)
for h in H_BIOC2C:
    so=str(BIOC2C.loc[h,'nodo_origen_id']); sd=str(BIOC2C.loc[h,'nodo_destino_id'])
    jo=str(BIOC2C.loc[h,'cliente_origen_id']); jd=str(BIOC2C.loc[h,'cliente_destino_id'])
    H_BIOC2C_by_origen_s[so].append(h); H_BIOC2C_by_destino_s[sd].append(h)
    H_BIOC2C_by_origen_j[jo].append(h); H_BIOC2C_by_destino_j[jd].append(h)


# Gasoductos cortos de BIOGÁS CRUDO desde P#, CLIENTE o NGI usados como HUB.
# P# conserva la conectividad histórica hacia cualquier nodo. Además, un cliente
# o un NGI que tenga biogás crudo disponible puede reenviarlo a P#, ubicación de
# empresa, cliente u otro NGI. Cliente->cliente permanece en H_BIOC2C para no
# duplicar tuberías físicas.
_FILAS_PHUB=[]
_sitios_idx_phub=sitios.set_index('id')
_cliente_nodo_set=set(str(x) for x in CLIENTE_A_NODO.values())
_empresa_nodo_set=set(str(x) for x in NODOS_EMPRESA_IDS)
_entrada_nodo_set=set(str(x) for x in NODOS_ENTRADA_GASODUCTO_IDS)
_original_p_set=set(str(x) for x in SITIOS_ORIGINALES_IDS)
_origenes_relevo_biogas = set(_original_p_set) | set(_cliente_nodo_set) | set(_entrada_nodo_set)
if HABILITAR_RELEVO_BIOGAS_CRUDO_EN_UBICACIONES_P and HABILITAR_GASODUCTOS_BIOGAS_CORTOS:
    for so in sorted(_origenes_relevo_biogas):
        origen=_sitios_idx_phub.loc[so]
        origen_es_cliente = so in _cliente_nodo_set
        origen_es_ngi = so in _entrada_nodo_set
        for sd in S:
            sd=str(sd)
            if sd==so:
                continue
            # C->C ya está representado por H_BIOC2C, con su activación/capacidad
            # propia; no se crea una segunda variable para el mismo par.
            if origen_es_cliente and sd in _cliente_nodo_set:
                continue
            destino=_sitios_idx_phub.loc[sd]
            distancia_km=_distancia_recta_km(origen.lat,origen.lon,destino.lat,destino.lon)
            if not (distancia_km < float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM)):
                continue
            if sd in _original_p_set:
                destino_tipo='UBICACION_POSIBLE'
            elif sd in _empresa_nodo_set:
                destino_tipo='UBICACION_EMPRESA'
            elif sd in _cliente_nodo_set:
                destino_tipo='CLIENTE'
            elif sd in _entrada_nodo_set:
                destino_tipo='ENTRADA_GASODUCTO'
            else:
                destino_tipo='OTRO_NODO'
            if origen_es_cliente:
                h=f'GBCORTO_CHUB_{so}_{sd}'
                origen_tipo='CLIENTE_HUB'
            elif origen_es_ngi:
                h=f'GBCORTO_NGIHUB_{so}_{sd}'
                origen_tipo='ENTRADA_GASODUCTO_HUB'
            else:
                h=f'GBCORTO_PHUB_{so}_{sd}'
                origen_tipo='UBICACION_POSIBLE_HUB'
            _FILAS_PHUB.append({
                'id':h,'nombre':f'Gasoducto corto de biogás crudo HUB {so} → {sd}',
                'origen_tipo':origen_tipo,'origen_id':so,
                'origen_nombre':str(origen.nombre),'nodo_origen_id':so,
                'destino_tipo':destino_tipo,'destino_id':sd,
                'destino_nombre':str(destino.nombre),'nodo_destino_id':sd,
                'punto_a_lat':float(origen.lat),'punto_a_lon':float(origen.lon),
                'punto_b_lat':float(destino.lat),'punto_b_lon':float(destino.lon),
                'distancia_recta_km':float(distancia_km),
                'costo_fijo_usd_dia':float(COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA),
                'costo_variable_usd_nm3_km':float(COSTO_VARIABLE_GASODUCTO_BIOGAS_CORTO_USD_NM3_KM),
                'capacidad_max_nm3_dia':float(CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA),
                'largo_maximo_km':float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM),
                'route_key':f'SHORTBIOPHUB|{h}','geometria':'LINEA_RECTA_CALCULADA_EN_IPYNB',
                'producto_transportado':'Biogás crudo',
            })
gasoductos_biogas_cortos_hub_p=pd.DataFrame(_FILAS_PHUB)
H_PHUB=(gasoductos_biogas_cortos_hub_p['id'].astype(str).tolist() if len(gasoductos_biogas_cortos_hub_p) else [])
PHUB=(gasoductos_biogas_cortos_hub_p.set_index('id') if len(gasoductos_biogas_cortos_hub_p) else pd.DataFrame())
H_PHUB_by_origen_s=defaultdict(list); H_PHUB_by_destino_s=defaultdict(list)
for h in H_PHUB:
    so=str(PHUB.loc[h,'nodo_origen_id']); sd=str(PHUB.loc[h,'nodo_destino_id'])
    H_PHUB_by_origen_s[so].append(h); H_PHUB_by_destino_s[sd].append(h)
print('Gasoductos cortos biogás crudo HUB P#/cliente/NGI -> otro nodo:',len(H_PHUB))

ABC_by_s=defaultdict(list); ABC_by_j=defaultdict(list); ABIO_by_i=defaultdict(list); ABIO_by_s=defaultdict(list)
ASITE_out=defaultdict(list); ASITE_in=defaultdict(list); AIG_by_s=defaultdict(list); AIG_by_d=defaultdict(list)
AGC_by_d=defaultdict(list); AGC_by_j=defaultdict(list); D_by_g=defaultdict(list)
for i,s in ABIO: ABIO_by_i[i].append(s); ABIO_by_s[s].append(i)
for s,j in ABC: ABC_by_s[s].append(j); ABC_by_j[j].append(s)
for p,q in ASITE: ASITE_out[p].append(q); ASITE_in[q].append(p)
for s,d in AIG: AIG_by_s[s].append(d); AIG_by_d[d].append(s)
for d,j in AGC: AGC_by_d[d].append(j); AGC_by_j[j].append(d)
for d in D: D_by_g[DG[d]].append(d)

# ============================================================
# CONTINUIDAD FÍSICA DE LAS 36 CONEXIONES DIRECTAS
# ============================================================
# Los 9 gasoductos siguen siendo los únicos elementos de la red convencional y
# YGG[g1,g2] sigue siendo UNA conexión directa, sin gasoductos intermedios.
#
# Los nombres AB/BA NO forman una red adicional: se conservan únicamente como
# metadatos geométricos para identificar cuál extremo físico coincide con el
# punto de referencia usado al calcular cada una de las 36 distancias. Esto
# impide que el MILP inyecte en un extremo y, sin costo, extraiga desde otro.

def _direccion_en_referencia(g, lat, lon, lado):
    g=str(g); lado=str(lado)
    if lado not in {'entrada','salida'}:
        raise ValueError(lado)
    candidatos=[]
    for d in D_by_g[g]:
        la=float(GD.loc[d,f'{lado}_lat']); lo=float(GD.loc[d,f'{lado}_lon'])
        dist=float(_distancia_recta_km(float(lat),float(lon),la,lo))
        candidatos.append((dist,str(d)))
    if not candidatos:
        raise AssertionError((g,lado,'sin_direcciones'))
    dist,d=min(candidatos,key=lambda x:x[0])
    # Las referencias del CSV fueron construidas desde estos mismos extremos;
    # una diferencia mayor a 50 m indicaría datos incompatibles.
    if dist>0.05:
        raise AssertionError((g,lado,'referencia_no_coincide',dist,d,lat,lon))
    return d

GG_ORIGIN_D={}   # (g1,g2) -> dirección cuya ENTRADA coincide con referencia de g1
GG_DEST_D={}     # (g1,g2) -> dirección cuya SALIDA coincide con referencia de g2
GG_REF_COORD={}  # (g1,g2) -> ((lat1,lon1),(lat2,lon2)) exactamente como en CSV
for _r in distancias_entre_gasoductos_36.itertuples(index=False):
    _g1=str(_r.gasoducto_1_id); _g2=str(_r.gasoducto_2_id)
    _p1=(float(_r.referencia_1_lat),float(_r.referencia_1_lon))
    _p2=(float(_r.referencia_2_lat),float(_r.referencia_2_lon))
    for _a,_b,_pa,_pb in ((_g1,_g2,_p1,_p2),(_g2,_g1,_p2,_p1)):
        GG_ORIGIN_D[(_a,_b)]=_direccion_en_referencia(_a,_pa[0],_pa[1],'entrada')
        GG_DEST_D[(_a,_b)]=_direccion_en_referencia(_b,_pb[0],_pb[1],'salida')
        GG_REF_COORD[(_a,_b)]=(_pa,_pb)

GG_BY_ORIGIN_D=defaultdict(list)
GG_BY_DEST_D=defaultdict(list)
for _g1,_g2 in GG_DIR:
    GG_BY_ORIGIN_D[GG_ORIGIN_D[(_g1,_g2)]].append(_g2)
    GG_BY_DEST_D[GG_DEST_D[(_g1,_g2)]].append(_g1)

# Permanecer en el mismo gasoducto sólo puede conectar entrada y salida que
# correspondan al MISMO punto físico. La dirección opuesta tiene exactamente
# esa salida física.
SAME_POINT_OUT_D={str(d):str(_DIRECCION_OPUESTA[str(d)]) for d in D}
SAME_POINT_IN_D={v:k for k,v in SAME_POINT_OUT_D.items()}
for _din,_dout in SAME_POINT_OUT_D.items():
    _gap=_distancia_recta_km(
        float(GD.loc[_din,'entrada_lat']),float(GD.loc[_din,'entrada_lon']),
        float(GD.loc[_dout,'salida_lat']),float(GD.loc[_dout,'salida_lon']),
    )
    if _gap>0.05:
        raise AssertionError((_din,_dout,'entrada_salida_mismo_punto_no_coinciden',_gap))

# Referencias geométricas entre direcciones y nodos físicos de entrada/salida.
D_SALIDA_by_s=defaultdict(list)
D_ENTRADA_by_s=defaultdict(list)
for d in D:
    nodo_salida=str(SALIDA_GASODUCTO_A_NODO[str(d)])
    nodo_entrada=str(ENTRADA_GASODUCTO_A_NODO[str(d)])
    # La inyección RL/RC usa exclusivamente la transferencia AIG local de 0 km.
    D_SALIDA_by_s[nodo_salida].append(str(d))
    D_ENTRADA_by_s[nodo_entrada].append(str(d))

# Gasoductos cortos de GasBC desde una planta BG ubicada en la empresa hasta
# la entrada de una dirección del corredor principal.
H_GECORTO=(
    gasoductos_gasbc_cortos_empresa_entrada['id'].astype(str).tolist()
    if len(gasoductos_gasbc_cortos_empresa_entrada) else []
)
GECORTO=(
    gasoductos_gasbc_cortos_empresa_entrada.set_index('id')
    if len(gasoductos_gasbc_cortos_empresa_entrada) else pd.DataFrame()
)
H_GECORTO_by_s=defaultdict(list); H_GECORTO_by_d=defaultdict(list)
H_GECORTO_by_g=defaultdict(list); H_GECORTO_by_i=defaultdict(list)
for h in H_GECORTO:
    i=str(GECORTO.loc[h,'empresa_id']); s=str(GECORTO.loc[h,'nodo_origen_id'])
    d=str(GECORTO.loc[h,'direccion_id']); g=str(GECORTO.loc[h,'gasoducto_id'])
    H_GECORTO_by_s[s].append(h); H_GECORTO_by_d[d].append(h)
    H_GECORTO_by_g[g].append(h); H_GECORTO_by_i[i].append(h)


# Gasoductos cortos DIRECTOS de GasBC desde BG en empresa hacia ubicación/cliente.
# No contienen direccion_id ni gasoducto_id convencional.
H_GDCORTO=(
    gasoductos_gasbc_cortos_empresa_destino['id'].astype(str).tolist()
    if len(gasoductos_gasbc_cortos_empresa_destino) else []
)
GDCORTO=(
    gasoductos_gasbc_cortos_empresa_destino.set_index('id')
    if len(gasoductos_gasbc_cortos_empresa_destino) else pd.DataFrame()
)
H_GDCORTO_by_i=defaultdict(list)
H_GDCORTO_by_origen_s=defaultdict(list)
H_GDCORTO_by_destino_s=defaultdict(list)
H_GDCORTO_by_j=defaultdict(list)
H_GDCORTO_CLIENTE=[]
H_GDCORTO_UBICACION=[]
H_GDCORTO_EMPRESA=[]
H_GDCORTO_ENTRADA_GASODUCTO=[]
for h in H_GDCORTO:
    s0=str(GDCORTO.loc[h,'nodo_origen_id'])
    sd=str(GDCORTO.loc[h,'nodo_destino_id'])
    tipo=str(GDCORTO.loc[h,'destino_tipo'])
    empresa_val=GDCORTO.loc[h,'empresa_id']
    if not pd.isna(empresa_val) and str(empresa_val) in I:
        i=str(empresa_val)
        H_GDCORTO_by_i[i].append(h)
    H_GDCORTO_by_origen_s[s0].append(h)
    H_GDCORTO_by_destino_s[sd].append(h)
    if tipo=='CLIENTE':
        j=str(GDCORTO.loc[h,'cliente_id'])
        H_GDCORTO_by_j[j].append(h)
        H_GDCORTO_CLIENTE.append(h)
    elif tipo=='UBICACION_POSIBLE':
        H_GDCORTO_UBICACION.append(h)
    elif tipo=='UBICACION_EMPRESA':
        H_GDCORTO_EMPRESA.append(h)
    elif tipo=='ENTRADA_GASODUCTO':
        H_GDCORTO_ENTRADA_GASODUCTO.append(h)
    else:
        raise AssertionError((h,tipo))

# Gasoductos cortos de GasBC desde la salida de una dirección convencional.
# El destino puede ser un cliente, una ubicación P# o una ubicación física de empresa NE_E.
# Los arcos a P/E son exclusivamente de tránsito/HUB y alimentan balance_BG[destino].
H_GSCORTO=(
    gasoductos_gasbc_cortos_salida_cliente['id'].astype(str).tolist()
    if len(gasoductos_gasbc_cortos_salida_cliente) else []
)
GSCORTO=(
    gasoductos_gasbc_cortos_salida_cliente.set_index('id')
    if len(gasoductos_gasbc_cortos_salida_cliente) else pd.DataFrame()
)
H_GSCORTO_by_d=defaultdict(list)
H_GSCORTO_by_j=defaultdict(list)
H_GSCORTO_by_g=defaultdict(list)
H_GSCORTO_by_destino_s=defaultdict(list)
H_GSCORTO_CLIENTE=[]
H_GSCORTO_UBICACION=[]
H_GSCORTO_EMPRESA=[]
for h in H_GSCORTO:
    d=str(GSCORTO.loc[h,'direccion_id'])
    g=str(GSCORTO.loc[h,'gasoducto_id'])
    tipo=str(GSCORTO.loc[h,'destino_tipo'])
    sd=str(GSCORTO.loc[h,'nodo_destino_id'])
    H_GSCORTO_by_d[d].append(h)
    H_GSCORTO_by_g[g].append(h)
    H_GSCORTO_by_destino_s[sd].append(h)
    if tipo=='CLIENTE':
        j=str(GSCORTO.loc[h,'cliente_id'])
        H_GSCORTO_by_j[j].append(h)
        H_GSCORTO_CLIENTE.append(h)
    elif tipo=='UBICACION_POSIBLE':
        H_GSCORTO_UBICACION.append(h)
    elif tipo=='UBICACION_EMPRESA':
        i_dest=str(GSCORTO.loc[h,'empresa_id'])
        H_GSCORTO_EMPRESA.append(h)
    else:
        raise AssertionError((h,tipo))

# Validación de la nueva cadena empresa(s) -> gasoducto corto de biogás ->
# BG en entrada -> inyección local al corredor.
for h in H_CORTO_ENTRADA_GASODUCTO:
    d=str(GCORTO.loc[h,'direccion_id']); s=str(GCORTO.loc[h,'nodo_destino_id'])

# GasBC intermedio desde una planta BG hacia una planta L o C.
# Se conservan las transferencias locales p=q. Los pares no locales quedan
# disponibles s?lo por alternativas de gasoducto; el uso vial se bloquea en el MILP.
_ASITE_SET=set(ASITE)
GBL=[]  # GasBC producido en BG[p] y procesado en L[q].
GBC=[]  # GasBC producido en BG[p] y procesado en C[q].
for p in S:
    if 'BG' not in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(p)]:
        continue
    for q in S:
        conexion_disponible=(str(p)==str(q)) or ((p,q) in _ASITE_SET)
        if not conexion_disponible:
            continue
        permitidas=TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(q)]
        if 'L' in permitidas:
            GBL.append((p,q))
        if 'C' in permitidas:
            GBC.append((p,q))

GBL=sorted(set(GBL)); GBC=sorted(set(GBC))
GBL_SET=set(GBL); GBC_SET=set(GBC)
GBL_by_p=defaultdict(list); GBL_by_q=defaultdict(list)
GBC_by_p=defaultdict(list); GBC_by_q=defaultdict(list)
for p,q in GBL: GBL_by_p[p].append(q); GBL_by_q[q].append(p)
for p,q in GBC: GBC_by_p[p].append(q); GBC_by_q[q].append(p)

# Cada salida física de gasoducto es también el NGI de la dirección opuesta y
# debe poder alojar L/C localmente. Esto garantiza la cadena corredor -> L/C
# sin un arco ASITE ni un gasoducto corto artificial.
for d in D:
    _sout=str(SALIDA_GASODUCTO_A_NODO[str(d)])

# Validación de convergencia: P/E/C/NGI -> GasBC corto -> NGI -> L/C local.
# También se admite NGI -> gasoducto corto GasBC -> NGI, conservando masa en
# el pool GasBC nodal; sólo se evita el autobucle lógico.
for h in H_GDCORTO_ENTRADA_GASODUCTO:
    _so=str(GDCORTO.loc[h,'nodo_origen_id'])
    _sd=str(GDCORTO.loc[h,'nodo_destino_id'])

# Validación estructural de la cadena solicitada:
# empresa A -> empresa B -> BG[NE_B] -> GasBC -> L[NE_B] o C[NE_B].
# El primer tramo es vial para A != B; BG -> L/C es local p=q.
for empresa_origen in I:
    nodo_propio = EMPRESA_A_NODO[empresa_origen]
    for empresa_destino, nodo_destino in EMPRESA_A_NODO_COMPLETO.items():
        ruta_abio = f'ABIO|{empresa_origen}|{nodo_destino}'
        if empresa_origen == empresa_destino:
            pass
        else:
            pass

def trip_cost(vehicle_code,route_key):
    if not road_arc_requires_truck(route_key): return 0.0
    v=VEH.loc[vehicle_code]; d=float(DIST[route_key])
    return float(v.costo_fijo_viaje_usd)+2.0*float(v.costo_km_usd)*d

# Orígenes físicos desde los cuales sale GasBC final en camión.
ORIGENES_REPARTO=[]; ORIGEN_INFO={}; CLIENTES_REPARTO={}; ORIGIN_CLIENT_ROUTE_KEY={}
for s in S:
    code=f'SITE:{s}'; ORIGENES_REPARTO.append(code); ORIGEN_INFO[code]={'tipo':'SITE','id':s,'nombre':str(sitios.set_index('id').loc[s,'nombre'])}
    clientes_validos=[]
    for j in ABC_by_s[s]:
        rk=f'ABC|{s}|{j}'; ORIGIN_CLIENT_ROUTE_KEY[(code,j)]=rk
        if road_arc_requires_truck(rk): clientes_validos.append(j)
    CLIENTES_REPARTO[code]=sorted(clientes_validos)
for d in D:
    code=f'PIPE:{d}'; ORIGENES_REPARTO.append(code); ORIGEN_INFO[code]={'tipo':'PIPE','id':d,'nombre':str(GD.loc[d,'salida_nombre'])}
    clientes_validos=[]
    for j in AGC_by_d[d]:
        rk=f'AGC|{d}|{j}'; ORIGIN_CLIENT_ROUTE_KEY[(code,j)]=rk
        if road_arc_requires_truck(rk): clientes_validos.append(j)
    CLIENTES_REPARTO[code]=sorted(clientes_validos)

# Cada ubicación de empresa participa como origen de reparto directo de GasBC.
# La clave de optimización es ABC, pero su distancia/polilínea proviene de E2C.
for empresa_id, nodo_empresa in EMPRESA_A_NODO_COMPLETO.items():
    origin_code = f'SITE:{nodo_empresa}'
    for cliente_id in J:
        alias_key = f'ABC|{nodo_empresa}|{cliente_id}'
print('Orígenes empresa habilitados para GasBC directo:', len(EMPRESA_A_NODO_COMPLETO))


def delivery_flow_refs(origin_code,j):
    info=ORIGEN_INFO[origin_code]
    if info['tipo']=='SITE':
        s=info['id']; return [('ZD',s,j),('ZL',s,j),('ZC',s,j)]
    d=info['id']; return [('ZG',d,j),('ZG_HUB',d,j)]


def origin_client_route_key(origin_code,j): return ORIGIN_CLIENT_ROUTE_KEY[(origin_code,j)]

def c2c_route_key(j,k): return f'C2C|{j}|{k}'

# Relación entre una parada física y el cliente en cuya ubicación se encuentra.
# Además de NC_Cj, el ZIP contiene puntos estratégicos de consumo (P21...P25)
# prácticamente coincidentes con clientes. Se reconocen por coordenadas, sin
# inventar trayectorias: el tramo entre dos ubicaciones de clientes siempre usa
# la clave C2C incluida en el ZIP.
def _haversine_km_coords(lat1,lon1,lat2,lon2):
    p1,p2=math.radians(float(lat1)),math.radians(float(lat2))
    dphi=math.radians(float(lat2)-float(lat1))
    dlambda=math.radians(float(lon2)-float(lon1))
    h=math.sin(dphi/2.0)**2+math.cos(p1)*math.cos(p2)*math.sin(dlambda/2.0)**2
    return 2.0*6371.0088*math.asin(min(1.0,math.sqrt(max(0.0,h))))

CLIENTES_ASOCIADOS_POR_PARADA=defaultdict(set)
for cliente_id,nodo_id in CLIENTE_A_NODO.items():
    CLIENTES_ASOCIADOS_POR_PARADA[str(nodo_id)].add(str(cliente_id))
for cliente_id in J:
    CLIENTES_ASOCIADOS_POR_PARADA[str(cliente_id)].add(str(cliente_id))

_sitios_idx=sitios.set_index('id')
_clientes_idx=clientes.set_index('id')
UMBRAL_ASOCIACION_SITIO_CLIENTE_KM=1.0
for sitio_id in SITIOS_ORIGINALES_IDS:
    sr=_sitios_idx.loc[str(sitio_id)]
    candidatos=[]
    for cliente_id in J:
        cr=_clientes_idx.loc[str(cliente_id)]
        d=_haversine_km_coords(sr.lat,sr.lon,cr.lat,cr.lon)
        if d<=UMBRAL_ASOCIACION_SITIO_CLIENTE_KM:
            candidatos.append((d,str(cliente_id)))
    if candidatos:
        # Se toma el cliente físicamente más próximo. Esto asocia, por ejemplo,
        # P21-C3, P22-C4, P23-C1, P24-C6 y P25-C7 con los datos actuales.
        _,cliente_cercano=min(candidatos)
        CLIENTES_ASOCIADOS_POR_PARADA[str(sitio_id)].add(cliente_cercano)

NODO_A_CLIENTE={
    parada:next(iter(clientes_asociados))
    for parada,clientes_asociados in CLIENTES_ASOCIADOS_POR_PARADA.items()
    if len(clientes_asociados)==1
}

def cliente_unico_de_parada(parada):
    asociados=CLIENTES_ASOCIADOS_POR_PARADA.get(str(parada),set())
    return next(iter(asociados)) if len(asociados)==1 else None

def paradas_representan_mismo_cliente(a,b):
    ca=cliente_unico_de_parada(a); cb=cliente_unico_de_parada(b)
    return ca is not None and cb is not None and ca==cb

def ruta_entre_nodos_de_clientes(a,b):
    """Clave C2C del ZIP cuando las dos paradas representan clientes distintos."""
    cliente_a=cliente_unico_de_parada(a)
    cliente_b=cliente_unico_de_parada(b)
    if cliente_a is None or cliente_b is None or cliente_a==cliente_b:
        return None
    key=c2c_route_key(cliente_a,cliente_b)
    return key if key in DIST else None

def ruta_entre_destinos_de_planta(a,b):
    """
    Conecta dos destinos usando exclusivamente trayectorias precargadas del ZIP.
    Si ambos puntos están en ubicaciones de clientes distintos, usa primero C2C;
    en los demás casos usa el arco sitio-a-sitio ASITE.
    """
    key_c2c=ruta_entre_nodos_de_clientes(a,b)
    if key_c2c is not None:
        return key_c2c
    key_asite=f'ASITE|{a}|{b}'
    return key_asite if key_asite in DIST else None


# -----------------------------------------------------------------------------
# VIAJES CANDIDATOS DIRECTOS Y COMPARTIDOS
#   Origen -> A -> Origen
#   Origen -> A -> B -> Origen, con A != B.
#
# Para cada pareja no ordenada {A,B} se conserva el orden de menor distancia.
# Los viajes son enteros y las cantidades transformadas/entregadas son
# continuas y no negativas. Una parada puede procesar cero; el balance QSEG
# conserva explícitamente la carga que continúa o regresa al origen.
#
# No se impone un número máximo artificial de viajes. Las cotas de flujo usadas
# para fortalecer numéricamente el modelo provienen exclusivamente de oferta,
# demanda y conversiones del ZIP.
# -----------------------------------------------------------------------------
TRANSPORT_NETWORKS={}
TRANSPORT_ROUTE_CANDIDATES={}
TRANSPORT_ROUTE_IDS_BY_NETWORK_STOP=defaultdict(list)
_TRANSPORT_ROUTE_COUNTER=0

# Cotas físicas globales derivadas de los datos de entrada. Se usan únicamente
# para fortalecer el MILP y nunca limitan por debajo de la oferta/demanda posible.
_TOTAL_BIOGAS_MAX=float(E['capacidad_max_biogas_nm3'].astype(float).sum())
_TOTAL_DEMAND_MAX=float(CL['demanda_max_nm3'].astype(float).sum())
_TOTAL_GASBC_MAX=_TOTAL_BIOGAS_MAX*float(FACTORES_CONVERSION['BG'])
_TOTAL_BIOGNL_MAX=_TOTAL_GASBC_MAX*float(FACTORES_CONVERSION['L'])
_TOTAL_BIOGNC_MAX=_TOTAL_GASBC_MAX*float(FACTORES_CONVERSION['C'])

def _max_flow_route_from_zip(mode,origin_code,sequence):
    """Cota superior válida en las unidades del producto transportado."""
    mode=str(mode)
    if mode=='BIO':
        value=float(E.loc[str(origin_code),'capacidad_max_biogas_nm3'])
    elif mode=='BC_PROCESO':
        value=_TOTAL_GASBC_MAX
    elif mode=='GNL':
        value=_TOTAL_BIOGNL_MAX
    elif mode=='GNC':
        value=_TOTAL_BIOGNC_MAX
    elif mode=='BC':
        # Las rutas SITE siguen siendo entregas finales. Una ruta PIPE puede descargar
        # GasBC en un cliente/HUB para reenviarlo a otros clientes; por eso su cota
        # válida debe cubrir la demanda total potencial aguas abajo, no sólo la demanda
        # de la parada intermedia.
        info=ORIGEN_INFO.get(str(origin_code),{})
        if (
            info.get('tipo')=='PIPE'
            and PERMITIR_VEHICULO_GASBC_SALIDA_CONVENCIONAL_A_CLIENTE_HUB
        ):
            value=_TOTAL_DEMAND_MAX
        else:
            value=sum(float(CL.loc[str(j),'demanda_max_nm3']) for j in sequence)
    else:
        raise KeyError(f'Modo de transporte desconocido: {mode}')
    return max(0.0,float(value))

def _new_transport_route_id():
    global _TRANSPORT_ROUTE_COUNTER
    _TRANSPORT_ROUTE_COUNTER += 1
    return f'TR{_TRANSPORT_ROUTE_COUNTER:06d}'

def _register_simple_transport_network(
    network_id, mode, origin_code, origin_name, vehicle_code,
    stop_ids, delivery_refs, outbound_key_builder, between_key_builder,
):
    stop_ids=sorted({
        str(stop) for stop in stop_ids
        if outbound_key_builder(str(stop)) in DIST
        and road_arc_requires_truck(outbound_key_builder(str(stop)))
    })
    if not stop_ids:
        return

    network_id=str(network_id)
    net={
        'network_id':network_id,
        'mode':str(mode),
        'origin_code':str(origin_code),
        'origin_name':str(origin_name),
        'vehicle':str(vehicle_code),
        'stops':stop_ids,
        'delivery_refs':{str(k):[tuple(ref) for ref in v] for k,v in delivery_refs.items()},
    }
    TRANSPORT_NETWORKS[network_id]=net

    fixed=float(VEH.loc[vehicle_code,'costo_fijo_viaje_usd'])
    km=float(VEH.loc[vehicle_code,'costo_km_usd'])

    def add_candidate(sequence,segment_keys,segment_reverse):
        sequence=tuple(map(str,sequence))
        max_paradas=(
            int(MAXIMO_PARADAS_BIOGAS_CRUDO_VEHICULO)
            if str(mode)=='BIO' else 2
        )
        longitudes_permitidas={2} if FORZAR_EXACTAMENTE_DOS_PARADAS else set(range(1,max_paradas+1))
        if len(sequence) not in longitudes_permitidas:
            raise AssertionError(
                f'Cantidad de paradas no permitida para la ruta: {sequence}'
            )
        if len(sequence)==2 and sequence[0]==sequence[1]:
            raise AssertionError(f'Los dos destinos deben ser distintos: {sequence}')
        rid=_new_transport_route_id()
        distance=sum(float(DIST[k]) for k in segment_keys)
        max_flow_from_zip=_max_flow_route_from_zip(mode,origin_code,sequence)
        candidate={
            'route_id':rid,
            'network_id':network_id,
            'mode':str(mode),
            'origin_code':str(origin_code),
            'origin_name':str(origin_name),
            'vehicle':str(vehicle_code),
            'stops':sequence,
            'segment_keys':tuple(map(str,segment_keys)),
            'segment_reverse':tuple(bool(x) for x in segment_reverse),
            'segment_distances_km':tuple(float(DIST[k]) for k in segment_keys),
            'distance_km':float(distance),
            'cost_per_vehicle_usd':float(fixed+km*distance),
            'max_flow_from_zip':float(max_flow_from_zip),
            'delivery_refs':{
                str(stop):list(net['delivery_refs'][str(stop)])
                for stop in sequence
            },
            'clientes_asociados_paradas':tuple(
                cliente_unico_de_parada(stop) for stop in sequence
            ),
            'tramo_entre_paradas_desde_zip':str(segment_keys[1]) if len(sequence)==2 else None,
        }
        TRANSPORT_ROUTE_CANDIDATES[rid]=candidate
        for stop in sequence:
            TRANSPORT_ROUTE_IDS_BY_NETWORK_STOP[(network_id,str(stop))].append(rid)

    # Los viajes directos sólo existen cuando la prueba forzada está apagada.
    if not FORZAR_EXACTAMENTE_DOS_PARADAS:
        for a in stop_ids:
            out_key=outbound_key_builder(a)
            add_candidate((a,),(out_key,out_key),(False,True))

    if PERMITIR_RUTAS_COMPARTIDAS:
        # Viajes con exactamente dos paradas. Se conserva un solo orden por pareja.
        for ia,a in enumerate(stop_ids):
            for b in stop_ids[ia+1:]:
                # Dos IDs físicos no cuentan como dos destinos reales si ambos
                # representan al mismo cliente (por ejemplo P22 y NC_C4).
                if str(mode) in {'GNL','GNC','BC'} and paradas_representan_mismo_cliente(a,b):
                    continue
                candidates=[]
                key_ab=between_key_builder(a,b)
                if key_ab in DIST:
                    candidates.append((
                        float(DIST[outbound_key_builder(a)])+float(DIST[key_ab])+float(DIST[outbound_key_builder(b)]),
                        (a,b),
                        (outbound_key_builder(a),key_ab,outbound_key_builder(b)),
                    ))
                key_ba=between_key_builder(b,a)
                if key_ba in DIST:
                    candidates.append((
                        float(DIST[outbound_key_builder(b)])+float(DIST[key_ba])+float(DIST[outbound_key_builder(a)]),
                        (b,a),
                        (outbound_key_builder(b),key_ba,outbound_key_builder(a)),
                    ))
                if not candidates:
                    continue

                _,sequence,keys=min(candidates,key=lambda z:(z[0],z[1]))

                # SIN PODA ECONÓMICA DE RUTAS MULTICLIENTE:
                # toda pareja físicamente factible de dos destinos se conserva como
                # candidata. El MILP/Gurobi decide si conviene usar una gira compartida
                # o viajes directos separados. Esto aplica especialmente a BioGNL/GNL
                # y BioGNC/GNC, donde la decisión global puede depender de plantas,
                # capacidades, conversiones y red de GasBC aguas abajo.

                add_candidate(sequence,keys,(False,False,True))

# El sistema de transporte vehicular para biog?s crudo y GasBC se retira por
# completo. No se registran redes BIO ni BCPROC; los flujos viales no locales
# equivalentes se fijan a cero directamente en el MILP. Las transferencias
# locales y las alternativas por gasoducto permanecen intactas.
if PERMITIR_VEHICULOS_BIOGAS_CRUDO:
    raise RuntimeError('El transporte vehicular de biog?s crudo fue retirado de este notebook.')
if PERMITIR_VEHICULOS_GASBC:
    raise RuntimeError('El transporte vehicular de GasBC fue retirado de este notebook.')


# BioGNL/BioGNC: cada vehículo mantiene un único producto durante la gira.
# GNL sólo visita plantas RL; GNC sólo visita plantas RC.
for p in S:
    stops_gnl=[
        q for q in ASITE_out[p]
        if str(q)!=str(p)
        and 'RL' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(q)]
        and f'ASITE|{p}|{q}' in DIST
        and road_arc_requires_truck(f'ASITE|{p}|{q}')
    ]
    stops_gnc=[
        q for q in ASITE_out[p]
        if str(q)!=str(p)
        and 'RC' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(q)]
        and f'ASITE|{p}|{q}' in DIST
        and road_arc_requires_truck(f'ASITE|{p}|{q}')
    ]
    if stops_gnl:
        _register_simple_transport_network(
            f'GNL:{p}','GNL',p,str(sitios.set_index('id').loc[p,'nombre']),'GNL',
            stops_gnl,{q:[('YL',p,q)] for q in stops_gnl},
            lambda q,p=p:f'ASITE|{p}|{q}',
            ruta_entre_destinos_de_planta,
        )
    if stops_gnc:
        _register_simple_transport_network(
            f'GNC:{p}','GNC',p,str(sitios.set_index('id').loc[p,'nombre']),'GNC',
            stops_gnc,{q:[('YC',p,q)] for q in stops_gnc},
            lambda q,p=p:f'ASITE|{p}|{q}',
            ruta_entre_destinos_de_planta,
        )

# GasBC final por veh?culo retirado: no se registran redes BC. La venta o
# tr?nsito de GasBC queda limitada a transferencias locales, gasoductos cortos y
# gasoductos convencionales.

# Comprobación automática de la corrección Metrogas-Carozzi.
_requisitos_c1_c2={
    'ASITE|P1|NC_C1','ASITE|P1|NC_C2',
    'C2C|C1|C2','C2C|C2|C1',
}
if PERMITIR_RUTAS_COMPARTIDAS and _requisitos_c1_c2.issubset(DIST):
    _giras_c1_c2=[
        route for route in TRANSPORT_ROUTE_CANDIDATES.values()
        if route['network_id']=='GNC:P1'
        and len(route['stops'])==2
        and set(route['stops'])=={'NC_C1','NC_C2'}
    ]
    # En modo forzado no se elimina por comparación con viajes directos.
    if _giras_c1_c2:
        _gira_prueba=_giras_c1_c2[0]
        print(
            'Gira GNC Metrogas-Carozzi habilitada:',
            ' -> '.join(_gira_prueba['stops']),
            f"{_gira_prueba['distance_km']:.3f} km",
            f"US$ {_gira_prueba['cost_per_vehicle_usd']:.2f} por vehículo",
        )

# Auditoría estructural: en la prueba forzada toda ruta NTR tiene exactamente
# dos destinos distintos y nunca existe Origen -> A -> Origen.
if FORZAR_EXACTAMENTE_DOS_PARADAS:
    pass
else:
    pass
if not PERMITIR_RUTAS_COMPARTIDAS:
    pass
if FORZAR_EXACTAMENTE_DOS_PARADAS:
    pass
# Cuando las dos paradas son ubicaciones de clientes distintos, el segmento
# central debe ser exactamente una trayectoria C2C leída del ZIP.
for route in TRANSPORT_ROUTE_CANDIDATES.values():
    if len(route['stops'])==2 and route['mode'] in {'GNL','GNC','BC'}:
        expected_c2c=ruta_entre_nodos_de_clientes(*route['stops'])
        if expected_c2c is not None:
            pass
# Auditoría de generalización vehicular: si existe ASITE desde un nodo con L/C
# hacia P/E/C/NGI compatible, el destino debe aparecer como parada candidata de
# la red GNL/GNC. El límite de una o dos paradas permanece en el generador común.
for _p in S:
    for _q in ASITE_out[_p]:
        if _p == _q:
            continue
        if 'L' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(_p)] and 'RL' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(_q)]:
            pass
        if 'C' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(_p)] and 'RC' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(_q)]:
            pass

for route in TRANSPORT_ROUTE_CANDIDATES.values():
    if route['mode']=='BIO':
        pass
    elif route['mode']=='GNL':
        pass
    elif route['mode']=='GNC':
        pass
    elif route['mode']=='BC_PROCESO':
        pass

print({
    'ABIO':len(ABIO),'ABC':len(ABC),'GBL':len(GBL),'GBC':len(GBC),'AGNL=AGNC':len(ASITE),'AIG':len(AIG),'AGC':len(AGC),'C2C':len(C2C),'gasoductos_biogas_cortos':len(H_CORTO),'gasoductos_biogas_cliente_cliente':len(H_BIOC2C),'gasoductos_GasBC_empresa_entrada':len(H_GECORTO),'gasoductos_GasBC_empresa_destino_directo':len(H_GDCORTO),
    'redes_transporte_simple':len(TRANSPORT_NETWORKS),
    'viajes_candidatos_totales':len(TRANSPORT_ROUTE_CANDIDATES),
    'viajes_candidatos_directos':sum(len(r['stops'])==1 for r in TRANSPORT_ROUTE_CANDIDATES.values()),
    'origenes_reparto':len(ORIGENES_REPARTO),'empresas_disponibles_milp':len(I),
    'transferencias_locales_sin_camion':len(LOCAL_ROAD_KEYS),'rutas_viales_validas':len(valid),
})


Distancias internas A/B leidas desde gasoductos.csv del ZIP: {'G_NC': 1188.0, 'G_CS': 479.0, 'G_NS': 1664.0, 'G_LA': 26.0, 'G_LP': 244.2, 'G_SBQ': 111.19, 'G_LSR': 3.75, 'G_RJC': 0.02, 'G_PA': 2.8}
Gasoductos cortos disponibles para relevo de biogás en ubicación de empresa: 10 | compartidos con producción: 10 | sólo HUB: 0
Gasoductos cortos biogás crudo HUB P#/cliente/NGI -> otro nodo: 34
Orígenes empresa habilitados para GasBC directo: 15
Gira GNC Metrogas-Carozzi habilitada: NC_C2 -> NC_C1 3432.680 km US$ 3364.03 por vehículo
{'ABIO': 990, 'ABC': 328, 'GBL': 4050, 'GBC': 4050, 'AGNL=AGNC': 3984, 'AIG': 738, 'AGC': 144, 'C2C': 56, 'gasoductos_biogas_cortos': 25, 'gasoductos_biogas_cliente_cliente': 0, 'gasoductos_GasBC_empresa_entrada': 1, 'gasoductos_GasBC_empresa_destino_directo': 44, 'redes_transporte_simple': 132, 'viajes_candidatos_totales': 233568, 'viajes_candidatos_directos': 7968, 'origenes_reparto': 84, 'empresas_disponibles_milp': 15, 'transferencias_locales_sin_camion': 41

# 6

Ampliación: BioGNL/BioGNC multicliente → GasBC → gasoductos cortos

Esta versión conserva el reparto vehicular existente de BioGNL/BioGNC (viaje directo o gira de hasta dos paradas). En cada parada con planta **RL** o **RC**, el producto reconvertido a **GasBC** puede: (1) venderse localmente como antes, o (2) alimentar uno o varios **gasoductos cortos de GasBC hacia otros clientes**. El flujo de cada gasoducto corto se descompone internamente en `QGCD_BG`, `QGCD_RL` y `QGCD_RC`, evitando doble contabilización y manteniendo el mismo costo, capacidad y activación del `QGCD` total.


Ampliación adicional: clientes como HUB de biogás crudo y GasBC

- **Biogás crudo:** un cliente puede recibirlo, no procesarlo todavía y reenviarlo por uno o varios gasoductos cortos hacia otros clientes. Una ubicación de empresa también puede actuar como HUB y reenviar el mismo biogás hacia otro nodo físico, incluida una entrada de gasoducto convencional, sin atribuirlo a la producción de la empresa intermedia. El balance `balance_biogas_crudo_*` impide duplicación y la planta BG sólo se activa donde el flujo se procesa realmente.
- **GasBC:** cada gasoducto corto hacia cliente separa venta final (`QGCD`) de tránsito. El tránsito entra al pool GasBC del cliente y puede volver a salir hacia otros clientes sin contabilizar una venta intermedia.
- **Gasoducto convencional → cliente/HUB:** la salida convencional puede llegar por carretera (`ZG_HUB`) o por gasoducto corto (`QGCP_HUB`) y después continuar como GasBC hacia otro cliente.
- **Gasoducto convencional → ubicación de empresa/HUB:** la salida convencional también puede llegar por gasoducto corto a `NE_Ei`; esa ubicación funciona como nodo físico de GasBC y puede repartir el flujo a múltiples clientes sin exigir producción de la empresa intermedia.
- **RL/RC con segundo salto:** GasBC producido por RL/RC puede usar un primer gasoducto corto como tránsito (`QGCD_HUB_RL`/`QGCD_HUB_RC`) y desde el cliente intermedio continuar por otro gasoducto corto.
- **RL/RC → gasoducto convencional:** una RL/RC ubicada exactamente en el nodo de entrada de una dirección puede inyectar su GasBC al corredor. El flujo total `YG` se descompone en `YG_BASE + YG_RL + YG_RC`, conservando masa, capacidad y costo del corredor sin duplicación.


### Ampliación: redes P/E/C/NGI generalizadas
GasBC corto admite tránsito entre P, ubicaciones de empresa, clientes y NGI, incluido NGI→NGI; biogás crudo desde clientes puede reenviarse a P/E/NGI; y L/C en cliente puede despachar BioGNL/BioGNC por vehículo a RL/RC en P/E/NGI, manteniendo máximo dos destinos.


In [ ]:
# ============================================================
# 6. CONSTRUCCIÓN DEL MILP CON REPARTO MULTI-CLIENTE
# ============================================================
class MILPBuilder:
    def __init__(self):
        self.names=[]; self.c=[]; self.integrality=[]; self.lb=[]; self.ub=[]; self.idx={}
        self.rows=[]; self.row_lb=[]; self.row_ub=[]; self.row_names=[]
        self.indicators=[]
    def add_var(self,name,c=0.0,integrality=0,lb=0.0,ub=np.inf):
        if name in self.idx: raise KeyError(name)
        k=len(self.names); self.idx[name]=k; self.names.append(name); self.c.append(float(c)); self.integrality.append(int(integrality)); self.lb.append(float(lb)); self.ub.append(float(ub)); return k
    def add_con(self,coefs,lb=-np.inf,ub=np.inf,name=''):
        clean={int(k):float(v) for k,v in coefs.items() if abs(v)>0}; self.rows.append(clean); self.row_lb.append(float(lb)); self.row_ub.append(float(ub)); self.row_names.append(name)
    def add_indicator(self,bin_idx,bin_value,coefs,sense='=',rhs=0.0,name=''):
        clean={int(k):float(v) for k,v in coefs.items() if abs(v)>0}
        if sense not in {'=','<','>'}:
            raise ValueError(f'Sentido indicador no válido: {sense}')
        self.indicators.append((int(bin_idx),bool(bin_value),clean,sense,float(rhs),str(name)))
    def solve(self,options=None):
        options=dict(options or {})
        rr=[]; cc=[]; dd=[]
        for r,coefs in enumerate(self.rows):
            for k,v in coefs.items():
                rr.append(r); cc.append(k); dd.append(v)

        A=coo_matrix(
            (dd,(rr,cc)),
            shape=(len(self.rows),len(self.names)),
        ).tocsr()

        c=np.asarray(self.c,dtype=float)
        integrality=np.asarray(self.integrality,dtype=np.uint8)
        lb=np.asarray(self.lb,dtype=float)
        ub=np.asarray(self.ub,dtype=float)
        row_lb=np.asarray(self.row_lb,dtype=float)
        row_ub=np.asarray(self.row_ub,dtype=float)

        directos=sum(
            len(route['stops'])==1
            for route in TRANSPORT_ROUTE_CANDIDATES.values()
        )
        compartidos=sum(
            len(route['stops'])==2
            for route in TRANSPORT_ROUTE_CANDIDATES.values()
        )
        print(
            'Resolviendo una sola optimización MILP desde cero con Gurobi:',
            f'{len(self.names):,} variables,',
            f'{len(self.rows):,} restricciones originales,',
            f'{directos:,} viajes directos y',
            f'{compartidos:,} viajes de exactamente dos destinos disponibles.',
        )

        # Gurobi representa cada fila con un único sentido. Una restricción
        # acotada lb <= A x <= ub se carga como dos filas equivalentes; las
        # igualdades y restricciones de un solo lado se cargan una sola vez.
        finite_lb=np.isfinite(row_lb)
        finite_ub=np.isfinite(row_ub)
        equality=finite_lb & finite_ub & (row_lb==row_ub)
        lower=finite_lb & ~equality
        upper=finite_ub & ~equality

        n_eq=int(equality.sum())
        n_lb=int(lower.sum())
        n_ub=int(upper.sum())
        print(
            'Filas entregadas a Gurobi:',
            f'{n_eq+n_lb+n_ub:,}',
            f'(igualdades={n_eq:,}, cotas inferiores={n_lb:,}, cotas superiores={n_ub:,}).',
        )

        env=None
        model=None
        try:
            env=crear_entorno_gurobi()
            model=gp.Model('Biometano_MILP_Profesional',env=env)

            for param,value in options.items():
                if value is not None:
                    model.setParam(param,value)

            if GUROBI_GUARDAR_LOG:
                model.setParam('LogFile',str(DIR_RESULTADOS/'gurobi_solver.log'))

            g_lb=np.where(np.isneginf(lb),-GRB.INFINITY,lb)
            g_ub=np.where(np.isposinf(ub), GRB.INFINITY,ub)
            is_binary=(integrality!=0) & (lb==0.0) & (ub==1.0)
            vtype=np.where(
                integrality==0,
                GRB.CONTINUOUS,
                np.where(is_binary,GRB.BINARY,GRB.INTEGER),
            )

            # API matricial dispersa: evita crear cientos de miles de objetos
            # Python uno por uno y entrega directamente la matriz CSR al solver.
            gx=model.addMVar(
                shape=len(self.names),
                lb=g_lb,
                ub=g_ub,
                obj=c,
                vtype=vtype,
                name='x',
            )
            model.ModelSense=GRB.MINIMIZE

            if n_eq:
                model.addMConstr(A[equality],gx,'=',row_lb[equality],name='eq')
            if n_lb:
                model.addMConstr(A[lower],gx,'>',row_lb[lower],name='lb')
            if n_ub:
                model.addMConstr(A[upper],gx,'<',row_ub[upper],name='ub')

            # Restricciones indicadoras nativas: evitan un Big-M adicional y
            # expresan directamente que una planta inactiva no puede procesar
            # ninguna cantidad física.
            sense_map={'=':GRB.EQUAL,'<':GRB.LESS_EQUAL,'>':GRB.GREATER_EQUAL}
            for bin_idx,bin_value,coefs,sense,rhs,name in self.indicators:
                vars_indicator=[gx[k].item() for k in coefs]
                expr=gp.LinExpr(list(coefs.values()),vars_indicator)
                model.addGenConstrIndicator(
                    gx[bin_idx].item(),bin_value,expr,sense_map[sense],rhs,name=name
                )

            model.update()

            if GUROBI_EXPORTAR_MODELO_MPS:
                model.write(str(DIR_RESULTADOS/'Modelo_Biometano_Gurobi.mps'))

            progreso={'numero':0,'ultimo_beneficio':None}

            def _callback_gurobi(cb_model,where):
                if where != GRB.Callback.MIPSOL:
                    return
                try:
                    objetivo_min=float(cb_model.cbGet(GRB.Callback.MIPSOL_OBJ))
                    if not math.isfinite(objetivo_min):
                        return
                    beneficio_actual=-objetivo_min
                    beneficio_anterior=progreso['ultimo_beneficio']
                    tolerancia=max(1e-7,1e-10*max(1.0,abs(beneficio_actual)))
                    if (
                        beneficio_anterior is not None
                        and abs(beneficio_actual-beneficio_anterior)<=tolerancia
                    ):
                        return
                    progreso['numero']+=1
                    progreso['ultimo_beneficio']=beneficio_actual
                    nodos=float(cb_model.cbGet(GRB.Callback.MIPSOL_NODCNT))
                    tiempo=float(cb_model.cbGet(GRB.Callback.RUNTIME))
                    print(
                        f"[Gurobi] Solución {progreso['numero']}: "
                        f"beneficio = US$ {beneficio_actual:,.2f} | "
                        f"nodos = {nodos:,.0f} | tiempo = {tiempo:,.2f} s",
                        flush=True,
                    )
                except Exception as exc:
                    # Un fallo de impresión nunca debe detener el optimizador.
                    print(f'Aviso en callback Gurobi: {exc}',flush=True)

            # Única llamada profesional al optimizador. Gurobi administra de
            # forma nativa presolve, relajación, heurísticas, cortes,
            # branch-and-cut, paralelismo y prueba del gap.
            model.optimize(_callback_gurobi)

            status=int(model.Status)
            sol_count=int(model.SolCount)

            status_map={}
            for const_name,text in [
                ('LOADED','modelo cargado'),
                ('OPTIMAL','solución óptima encontrada'),
                ('INFEASIBLE','modelo infactible'),
                ('INF_OR_UNBD','modelo infactible o no acotado'),
                ('UNBOUNDED','modelo no acotado'),
                ('CUTOFF','límite objetivo alcanzado'),
                ('ITERATION_LIMIT','límite de iteraciones alcanzado'),
                ('NODE_LIMIT','límite de nodos alcanzado'),
                ('TIME_LIMIT','límite de tiempo alcanzado'),
                ('SOLUTION_LIMIT','límite de soluciones alcanzado'),
                ('INTERRUPTED','optimización interrumpida'),
                ('NUMERIC','dificultad numérica'),
                ('SUBOPTIMAL','solución subóptima disponible'),
                ('USER_OBJ_LIMIT','límite objetivo del usuario alcanzado'),
                ('WORK_LIMIT','límite de trabajo alcanzado'),
                ('MEM_LIMIT','límite de memoria alcanzado'),
            ]:
                code=getattr(GRB,const_name,None)
                if code is not None:
                    status_map[int(code)]=text

            message=(
                f"Gurobi status {status}: {status_map.get(status,'estado no clasificado')}"
            )

            if (
                sol_count==0
                and GUROBI_CALCULAR_IIS_SI_INFACTIBLE
                and status in {
                    int(getattr(GRB,'INFEASIBLE',-999)),
                    int(getattr(GRB,'INF_OR_UNBD',-998)),
                }
            ):
                print('Calculando IIS de Gurobi para diagnóstico...')
                model.computeIIS()
                model.write(str(DIR_RESULTADOS/'Modelo_Biometano_IIS.ilp'))

            x_value=np.asarray(gx.X,dtype=float) if sol_count>0 else None
            try:
                mip_gap=float(model.MIPGap) if sol_count>0 else None
            except Exception:
                mip_gap=None
            try:
                objective=float(model.ObjVal) if sol_count>0 else None
            except Exception:
                objective=None
            try:
                objective_bound=float(model.ObjBound)
            except Exception:
                objective_bound=None

            return SimpleNamespace(
                x=x_value,
                status=status,
                message=message,
                success=(status==int(getattr(GRB,'OPTIMAL',2))),
                fun=objective,
                mip_gap=mip_gap,
                mip_node_count=float(model.NodeCount),
                mip_dual_bound=objective_bound,
                runtime=float(model.Runtime),
                sol_count=sol_count,
                solver='Gurobi Optimizer',
                api='gurobipy matrix API',
            )
        except gp.GurobiError as exc:
            raise RuntimeError(
                f"Gurobi error {getattr(exc,'errno','?')}: {exc}. "
                "Verifica que la licencia académica completa esté activa y "
                "que GUROBI_LICENSE_MODE coincida con tu configuración."
            ) from exc
        finally:
            if model is not None:
                model.dispose()
            if env is not None:
                env.dispose()

m=MILPBuilder()
def V(prefix,*args): return m.idx[(prefix,)+tuple(args)]
def HAS(prefix,*args): return (prefix,)+tuple(args) in m.idx
def ADD(co,prefix,*args,coef=1.0):
    idx=m.idx.get((prefix,)+tuple(args))
    if idx is not None:
        co[idx]=co.get(idx,0.0)+float(coef)
def av(prefix,*args,**kw): return m.add_var((prefix,)+tuple(args),**kw)
K={t:float(FACTORES_CONVERSION[t]) for t in T}

# ============================================================
# FUNCIONES DE PRECIO DISCRETIZADAS
# ============================================================
# La cantidad de compra depende solo del proveedor i; la cantidad de venta
# depende solo del cliente j. Los tramos cargan costo/ingreso incremental
# como coeficientes constantes de la funcion objetivo, manteniendo MILP.

def _breakpoints_precio(qmax, umbral):
    qmax = max(0.0, float(qmax))
    umbral = min(max(0.0, float(umbral)), qmax)
    puntos = [0.0]
    if umbral > EPS:
        puntos.append(umbral)
    resto = qmax - umbral
    if resto > EPS:
        for frac in FRACCIONES_TRAMOS_PRECIO:
            puntos.append(umbral + resto * float(frac))
    puntos = sorted({round(p, 8) for p in puntos if p >= -EPS and p <= qmax + EPS})
    if not puntos or puntos[0] > EPS:
        puntos.insert(0, 0.0)
    if qmax > EPS and abs(puntos[-1] - qmax) > 1e-7:
        puntos.append(qmax)
    return puntos


def _precio_compra_proveedor(i, q):
    q = max(0.0, float(q))
    qmax = max(EPS, float(E.loc[i, 'capacidad_max_biogas_nm3']))
    excedente = min(max(0.0, float(E.loc[i, COL_OFERTA_EXCEDENTE_BIOGAS])), qmax)
    p0 = float(E.loc[i, COL_PRECIO_BIOGAS_CRUDO])
    rango = qmax - excedente
    if rango <= EPS:
        return p0
    return p0 * (1.0 + ALPHA_COMPRA_PORCENTAJE * max(0.0, q - excedente) / rango)


def _precio_venta_cliente(j, q):
    q = max(0.0, float(q))
    dmax = max(EPS, float(CL.loc[j, 'demanda_max_nm3']))
    dnc = min(max(0.0, float(CL.loc[j, COL_DEMANDA_NO_CUBIERTA])), dmax)
    p0 = float(CL.loc[j, 'precio_venta_usd_nm3'])
    rango = dmax - dnc
    if rango <= EPS:
        return max(PRECIO_VENTA_MIN_USD_NM3, p0)
    return max(PRECIO_VENTA_MIN_USD_NM3, p0 * (1.0 - BETA_VENTA_PORCENTAJE * max(0.0, q - dnc) / rango))


def _costo_total_compra(i, q):
    return float(q) * _precio_compra_proveedor(i, q)


def _ingreso_total_venta(j, q):
    return float(q) * _precio_venta_cliente(j, q)


def _construir_tramos_precio(owner_id, puntos, funcion_total):
    tramos = []
    for k, (lo, hi) in enumerate(zip(puntos[:-1], puntos[1:])):
        ancho = float(hi - lo)
        if ancho <= EPS:
            continue
        total_lo = float(funcion_total(owner_id, lo))
        total_hi = float(funcion_total(owner_id, hi))
        coef = (total_hi - total_lo) / ancho
        tramos.append({
            'k': k,
            'q_min': float(lo),
            'q_max': float(hi),
            'ancho': ancho,
            'coef': float(coef),
        })
    return tramos


TRAMOS_COMPRA = {
    i: _construir_tramos_precio(
        i,
        _breakpoints_precio(E.loc[i, 'capacidad_max_biogas_nm3'], E.loc[i, COL_OFERTA_EXCEDENTE_BIOGAS]),
        _costo_total_compra,
    )
    for i in I
}

TRAMOS_VENTA = {
    j: _construir_tramos_precio(
        j,
        _breakpoints_precio(CL.loc[j, 'demanda_max_nm3'], CL.loc[j, COL_DEMANDA_NO_CUBIERTA]),
        _ingreso_total_venta,
    )
    for j in J
}

print(
    "Tramos de precio creados:",
    f"compra={sum(len(v) for v in TRAMOS_COMPRA.values())}",
    f"venta={sum(len(v) for v in TRAMOS_VENTA.values())}",
)


# Flujos y economía de producción.
# X y QBCP contabilizan compra/transporte de biogás crudo; el costo variable BG
# se cobra únicamente en QBIO_BG, es decir, donde el biogás realmente se procesa.
for i,s in ABIO:
    rk=f'ABIO|{i}|{s}'
    if PERMITIR_VEHICULOS_BIOGAS_CRUDO or not road_arc_requires_truck(rk):
        av('X',i,s)
for s in S: av('QBIO_BG',s,c=float(PL.loc['BG','costo_variable_usd_nm3']))
# QBCP[h] transporta biogás crudo desde una empresa hasta un nodo de destino:
# cliente, ubicación P#, ubicación de empresa o entrada de gasoducto. Su coeficiente incluye compra de
# biogás y costo variable del gasoducto por Nm³ y kilómetro. El procesamiento
# BG se cobra después, únicamente sobre QBIO_BG en el nodo que realmente procesa.
# El costo fijo se asigna a BCP[h].
for h in H_CORTO:
    i=str(GCORTO.loc[h,'empresa_id'])
    av(
        'QBCP',h,
        c=(
            float(GCORTO.loc[h,'costo_variable_usd_nm3_km'])
            *float(GCORTO.loc[h,'distancia_recta_km'])
        ),
    )
# Relevo de biogás crudo desde una ubicación de empresa/HUB. El costo de compra
# ya fue pagado por la empresa que originó el flujo; aquí sólo se cobra el segundo
# gasoducto corto. Puede compartir tubería con la producción propia de la empresa.
for h in H_EHUB:
    av(
        'QBCP_HUB',h,
        c=float(EHUB.loc[h,'costo_variable_usd_nm3_km'])*float(EHUB.loc[h,'distancia_recta_km']),
    )

# Relevo de biogás crudo entre clientes/HUBs. El costo de compra ya se pagó en
# la entrada original desde una empresa y el costo BG se paga sólo al procesarlo.
for h in H_BIOC2C:
    av(
        'QBC2C',h,
        c=float(BIOC2C.loc[h,'costo_variable_usd_nm3_km'])*float(BIOC2C.loc[h,'distancia_recta_km']),
    )
for h in H_PHUB:
    av(
        'QBCP_P',h,
        c=float(PHUB.loc[h,'costo_variable_usd_nm3_km'])*float(PHUB.loc[h,'distancia_recta_km']),
    )

for s in S:
    av('QL',s,c=float(PL.loc['L','costo_variable_usd_nm3'])); av('QC',s,c=float(PL.loc['C','costo_variable_usd_nm3']))
# Origen y destino quedan explícitos: varias BG pueden abastecer una misma L/C.
for p,q in GBL:
    rk=f'ASITE|{p}|{q}'
    if PERMITIR_VEHICULOS_GASBC or str(p)==str(q) or (rk in DIST and not road_arc_requires_truck(rk)):
        av('GBL',p,q)
for p,q in GBC:
    rk=f'ASITE|{p}|{q}'
    if PERMITIR_VEHICULOS_GASBC or str(p)==str(q) or (rk in DIST and not road_arc_requires_truck(rk)):
        av('GBC',p,q)
for p,q in ASITE:
    av('YL',p,q,c=float(PL.loc['RL','costo_variable_usd_nm3'])); av('YC',p,q,c=float(PL.loc['RC','costo_variable_usd_nm3']))
for s in S:
    av('YL0',s,c=float(PL.loc['RL','costo_variable_usd_nm3'])); av('YC0',s,c=float(PL.loc['RC','costo_variable_usd_nm3']))
# Transferencia local en nodos NGI: GasBC generado por RL/RC en el mismo nodo
# puede alimentar L/C local sin activar ni consumir capacidad del gasoducto convencional.
NGI_LOCAL_TRANSFER_SITES=sorted(str(s) for s in S if str(s) in set(map(str,NODOS_ENTRADA_GASODUCTO_IDS)))
NGI_LOCAL_TRANSFER_SET=set(NGI_LOCAL_TRANSFER_SITES)
for s in NGI_LOCAL_TRANSFER_SITES:
    av('QRL_L_LOCAL',s); av('QRL_C_LOCAL',s)
    av('QRC_L_LOCAL',s); av('QRC_C_LOCAL',s)
# YG[s,d] es el flujo TOTAL de GasBC que se inyecta al gasoducto.
# Los accesos AIG viales de GasBC no se crean si el modo vehicular esta apagado.
for s,d in AIG:
    rk=f'AIG|{s}|{d}'
    if (not PERMITIR_VEHICULOS_GASBC) and road_arc_requires_truck(rk):
        continue
    g=DG[d]
    av('YG',s,d)
    av('YG_BASE',s,d)
    av('YG_RL',s,d)
    av('YG_RC',s,d)
    m.add_con(
        {V('YG',s,d):1,V('YG_BASE',s,d):-1,V('YG_RL',s,d):-1,V('YG_RC',s,d):-1},
        lb=0,ub=0,name=f'descomposicion_fuente_entrada_corredor_{s}_{d}',
    )
    if (
        (not HABILITAR_INYECCION_RL_RC_GASODUCTO_CONVENCIONAL)
        or str(s)!=str(ENTRADA_GASODUCTO_A_NODO[str(d)])
    ):
        m.add_con({V('YG_RL',s,d):1},lb=0,ub=0,name=f'prohibir_RL_no_local_a_corredor_{s}_{d}')
        m.add_con({V('YG_RC',s,d):1},lb=0,ub=0,name=f'prohibir_RC_no_local_a_corredor_{s}_{d}')
# QGCE[h] es GasBC ya producido por BG en el nodo propio de la empresa.
# Aquí sólo se cobra el gasoducto CORTO hasta el acceso. El recorrido por la red
# convencional se cobra después en YGG usando las 36 distancias gasoducto-gasoducto.
for h in H_GECORTO:
    d_h=str(GECORTO.loc[h,'direccion_id']); g_h=DG[d_h]
    av(
        'QGCE',h,
        c=(
            float(GECORTO.loc[h,'costo_variable_usd_nm3_km'])
            *float(GECORTO.loc[h,'distancia_recta_km'])
        ),
    )

# Flujo DIRECTO entre gasoductos convencionales. Existen 72 variables dirigidas,
# pero sólo 36 números de distancia porque D(g1,g2)=D(g2,g1).
# IMPORTANTE: YGG[g1,g2] representa un único transporte físico g1 -> g2.
# No puede utilizar otro gasoducto como nodo intermedio.
for g1,g2 in GG_DIR:
    av('YGG',g1,g2,c=COSTO_GG_USD_KM_NM3*float(DIST_GG[(g1,g2)]))
# Flujo interno A/B del mismo gasoducto convencional.
# YGINTRA[d] entra por la entrada de d y sale por la salida de d; su distancia
# viene exclusivamente de gasoductos.csv del ZIP. YGGS se conserva para el caso
# historico de permanencia en el mismo punto fisico, sin romper trazas previas.
for d in D:
    _g_intra=str(DG[d])
    av('YGINTRA',d,c=float(GA.loc[_g_intra,'costo_variable_usd_km_nm3'])*float(DIST_INTRA_GASODUCTO_KM[_g_intra]))
    av('YGGS',d)
    # Extracción LOCAL desde la salida física del corredor hacia una planta L/C
    # instalada exactamente en ese extremo. No existe carretera ni gasoducto
    # corto intermedio: el GasBC pasa directamente del corredor al proceso.
    av('QG_L_DIRECT',d)
    av('QG_C_DIRECT',d)
for s,j in ABC:
    rk=f'ABC|{s}|{j}'
    if PERMITIR_VEHICULOS_GASBC or not road_arc_requires_truck(rk):
        av('ZD',s,j); av('ZL',s,j); av('ZC',s,j)
for d,j in AGC:
    rk=f'AGC|{d}|{j}'
    if PERMITIR_VEHICULOS_GASBC:
        av('ZG',d,j)
        av('ZG_HUB',d,j)
        if not HABILITAR_REENVIO_GASBC_EN_CLIENTES:
            m.add_con({V('ZG_HUB',d,j):1},lb=0,ub=0,name=f'prohibir_ZG_hub_GasBC_{d}_{j}')
    elif HABILITAR_REENVIO_GASBC_EN_CLIENTES and not road_arc_requires_truck(rk):
        av('ZG_HUB',d,j)

VARIABLES_VIALES_PROHIBIDAS_OMITIDAS={
    'X_ABIO':sum(1 for i,s in ABIO if (not PERMITIR_VEHICULOS_BIOGAS_CRUDO) and road_arc_requires_truck(f'ABIO|{i}|{s}')),
    'GBL':sum(1 for p,q in GBL if (not PERMITIR_VEHICULOS_GASBC) and str(p)!=str(q) and f'ASITE|{p}|{q}' in DIST and road_arc_requires_truck(f'ASITE|{p}|{q}')),
    'GBC':sum(1 for p,q in GBC if (not PERMITIR_VEHICULOS_GASBC) and str(p)!=str(q) and f'ASITE|{p}|{q}' in DIST and road_arc_requires_truck(f'ASITE|{p}|{q}')),
    'ZD_ZL_ZC':3*sum(1 for s,j in ABC if (not PERMITIR_VEHICULOS_GASBC) and road_arc_requires_truck(f'ABC|{s}|{j}')),
    'ZG':len(AGC) if not PERMITIR_VEHICULOS_GASBC else 0,
    'ZG_HUB_vial':sum(1 for d,j in AGC if (not PERMITIR_VEHICULOS_GASBC) and road_arc_requires_truck(f'AGC|{d}|{j}')),
    'YG_AIG':sum(1 for s,d in AIG if (not PERMITIR_VEHICULOS_GASBC) and road_arc_requires_truck(f'AIG|{s}|{d}')),
}
for h in H_GSCORTO:
    tipo_dest=str(GSCORTO.loc[h,'destino_tipo'])
    costo_variable=(
        float(GSCORTO.loc[h,'costo_variable_usd_nm3_km'])
        *float(GSCORTO.loc[h,'distancia_recta_km'])
    )
    # QGCP es venta final únicamente cuando el destino es CLIENTE.
    if tipo_dest=='CLIENTE':
        j=str(GSCORTO.loc[h,'cliente_id'])
        av('QGCP',h,c=costo_variable)
    elif tipo_dest in {'UBICACION_POSIBLE','UBICACION_EMPRESA'}:
        av('QGCP',h,c=costo_variable)
        # P# y ubicaciones de empresa son HUBs, no clientes: nunca existe ingreso
        # en este primer tramo. El GasBC se deposita mediante QGCP_HUB en el pool
        # nodal y el ingreso sólo aparece cuando una ruta posterior llega a cliente.
        m.add_con({V('QGCP',h):1},lb=0,ub=0,name=f'prohibir_venta_en_hub_salida_{h}')
    else:
        raise AssertionError((h,tipo_dest))

    # QGCP_HUB transporta el mismo GasBC sin venta y lo entrega al pool nodal del
    # destino, desde donde puede continuar por uno o varios gasoductos cortos.
    av('QGCP_HUB',h,c=costo_variable)
    if tipo_dest=='CLIENTE' and not HABILITAR_REENVIO_GASBC_EN_CLIENTES:
        m.add_con({V('QGCP_HUB',h):1},lb=0,ub=0,name=f'prohibir_QGCP_hub_GasBC_{h}')
    if tipo_dest=='UBICACION_EMPRESA' and not HABILITAR_REENVIO_GASBC_EN_UBICACIONES_EMPRESA:
        m.add_con({V('QGCP_HUB',h):1},lb=0,ub=0,name=f'prohibir_QGCP_hub_empresa_GasBC_{h}')
# QGCD transporta GasBC directamente desde una BG instalada en cualquier nodo
# permitido. Si termina en un cliente incorpora el ingreso por venta; si termina en una ubicación entra al
# balance GasBC del nodo y el ingreso se contabiliza aguas abajo.
for h in H_GDCORTO:
    costo=(
        float(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])
        *float(GDCORTO.loc[h,'distancia_recta_km'])
    )
    # El ingreso por venta se contabiliza con QSELL_SEG del cliente.

    # QGCD es el flujo TOTAL del gasoducto corto y conserva exactamente su
    # costo/ingreso original. Se descompone por procedencia para que un mismo
    # gasoducto pueda transportar GasBC producido por BG o GasBC obtenido tras
    # reconvertir BioGNL/BioGNC en una planta RL/RC del nodo de origen.
    av('QGCD',h,c=costo)
    # Flujo físico de tránsito: llega a un cliente/HUB pero NO se vende allí.
    # Puede salir nuevamente como GasBC por otro gasoducto corto.
    av('QGCD_HUB',h,c=(
        float(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])*float(GDCORTO.loc[h,'distancia_recta_km'])
    ))
    # Primer salto de tránsito originado directamente en una reconversión RL/RC.
    # En el cliente destino entra al mismo pool GasBC que QGCD_HUB y desde allí
    # puede efectuar el segundo salto por otro gasoducto corto.
    av('QGCD_HUB_RL',h,c=(
        float(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])*float(GDCORTO.loc[h,'distancia_recta_km'])
    ))
    av('QGCD_HUB_RC',h,c=(
        float(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])*float(GDCORTO.loc[h,'distancia_recta_km'])
    ))
    av('QGCD_BG',h)
    av('QGCD_RL',h)
    av('QGCD_RC',h)
    m.add_con(
        {
            V('QGCD',h):1,
            V('QGCD_BG',h):-1,
            V('QGCD_RL',h):-1,
            V('QGCD_RC',h):-1,
        },
        lb=0,ub=0,
        name=f'descomposicion_fuente_gasoducto_corto_GasBC_{h}',
    )

    s0=str(GDCORTO.loc[h,'nodo_origen_id'])
    destino_tipo=str(GDCORTO.loc[h,'destino_tipo'])
    es_cliente=destino_tipo=='CLIENTE'
    es_ubicacion_p=destino_tipo=='UBICACION_POSIBLE'
    es_ubicacion_empresa=destino_tipo=='UBICACION_EMPRESA'
    es_entrada_gasoducto=destino_tipo=='ENTRADA_GASODUCTO'
    if (not es_cliente) or (not HABILITAR_REENVIO_GASBC_EN_CLIENTES):
        m.add_con({V('QGCD_HUB',h):1},lb=0,ub=0,name=f'prohibir_reenvio_GasBC_{h}')
        m.add_con({V('QGCD_HUB_RL',h):1},lb=0,ub=0,name=f'prohibir_reenvio_GasBC_RL_{h}')
        m.add_con({V('QGCD_HUB_RC',h):1},lb=0,ub=0,name=f'prohibir_reenvio_GasBC_RC_{h}')
    destino_admite_rlrc=es_cliente or es_ubicacion_p or es_ubicacion_empresa or es_entrada_gasoducto
    if (not destino_admite_rlrc) or ('RL' not in TECNOLOGIAS_PERMITIDAS_POR_SITIO[s0]):
        m.add_con({V('QGCD_RL',h):1},lb=0,ub=0,name=f'prohibir_fuente_RL_gasoducto_corto_{h}')
        m.add_con({V('QGCD_HUB_RL',h):1},lb=0,ub=0,name=f'prohibir_fuente_RL_hub_gasoducto_corto_{h}')
    if (not destino_admite_rlrc) or ('RC' not in TECNOLOGIAS_PERMITIDAS_POR_SITIO[s0]):
        m.add_con({V('QGCD_RC',h):1},lb=0,ub=0,name=f'prohibir_fuente_RC_gasoducto_corto_{h}')
        m.add_con({V('QGCD_HUB_RC',h):1},lb=0,ub=0,name=f'prohibir_fuente_RC_hub_gasoducto_corto_{h}')
    # El componente QGCD_BG representa el pool GasBC nodal (BG local o tránsito
    # recibido previamente). Por eso C/P/E/NGI pueden reenviar GasBC hacia otro
    # P/E/C/NGI sin exigir una nueva conversión RL/RC en el nodo de origen.

RUTAS_ENTREGA_CANDIDATAS=[]
for s,j in ABC:
    RUTAS_ENTREGA_CANDIDATAS += [(f'ZD|{s}|{j}',('ZD',s,j),j,'GasBC directo'),(f'ZL|{s}|{j}',('ZL',s,j),j,'BioGNL a GasBC'),(f'ZC|{s}|{j}',('ZC',s,j),j,'BioGNC a GasBC')]
for d,j in AGC: RUTAS_ENTREGA_CANDIDATAS.append((f'ZG|{d}|{j}',('ZG',d,j),j,'Gasoducto a cliente por carretera'))
for h in H_GSCORTO_CLIENTE:
    j=str(GSCORTO.loc[h,'cliente_id'])
    RUTAS_ENTREGA_CANDIDATAS.append((f'QGCP|{h}',('QGCP',h),j,'Gasoducto corto salida-cliente'))
for h in H_GDCORTO_CLIENTE:
    j=str(GDCORTO.loc[h,'cliente_id'])
    RUTAS_ENTREGA_CANDIDATAS.append((f'QGCD|{h}',('QGCD',h),j,'GasBC por gasoducto corto nodo-cliente'))
RUTAS_ENTREGA_POR_CLIENTE=defaultdict(list)
for ruta_id,flow_ref,j,familia in RUTAS_ENTREGA_CANDIDATAS: RUTAS_ENTREGA_POR_CLIENTE[j].append(ruta_id)

for i in I: av('BE',i,integrality=1,ub=1)
# BJ[j]=1 activa al cliente para las restricciones globales, sin exigir venta mínima.
for j in J: av('BJ',j,integrality=1,ub=1)
for g in G: av('BG',g,integrality=1,ub=1)
def _costo_fijo_gasoducto_corto(df, h):
    valor = float(df.loc[h, 'costo_fijo_usd_dia'])
    return valor if math.isfinite(valor) else 0.0

# Costos fijos de activacion de gasoductos cortos: se cobran una sola vez
# cuando la binaria fisica del gasoducto queda activa, independiente del flujo.
for h in H_CORTO:
    av('BCP',h,c=_costo_fijo_gasoducto_corto(GCORTO,h),integrality=1,ub=1)
for h in H_BIOC2C:
    av('BCP2C',h,c=_costo_fijo_gasoducto_corto(BIOC2C,h),integrality=1,ub=1)
for h in H_PHUB:
    av('BCP_P',h,c=_costo_fijo_gasoducto_corto(PHUB,h),integrality=1,ub=1)
# Si el gasoducto fisico ya existe en H_CORTO, QBCP y QBCP_HUB comparten BCP.
# Solo los HUB ubicados en empresas no proveedoras necesitan una binaria propia.
for h in H_EHUB_SOLO:
    av('BCP_HUB',h,c=_costo_fijo_gasoducto_corto(EHUB,h),integrality=1,ub=1)

for h in H_GECORTO:
    av('BGCE',h,c=_costo_fijo_gasoducto_corto(GECORTO,h),integrality=1,ub=1)
for h in H_GSCORTO:
    av('BGCP',h,c=_costo_fijo_gasoducto_corto(GSCORTO,h),integrality=1,ub=1)
for h in H_GDCORTO:
    av('BGCD',h,c=_costo_fijo_gasoducto_corto(GDCORTO,h),integrality=1,ub=1)

# Porcentajes comunes opcionales. Estas variables son continuas y mantienen la
# formulación como MILP; las igualdades condicionales se linealizan con las
# binarias BE y BJ, sin productos entre variables.
if compra_porcentaje: av('PORC_COMPRA',lb=0.0,ub=1.0)
if venta_porcentaje: av('PORC_VENTA',lb=0.0,ub=1.0)


# Variables de tramos economicos. Sus cotas son los anchos de cada tramo.
for i, tramos in TRAMOS_COMPRA.items():
    for k, tramo in enumerate(tramos):
        av('QBUY_SEG', i, k, c=float(tramo['coef']), lb=0.0, ub=float(tramo['ancho']))

for j, tramos in TRAMOS_VENTA.items():
    for k, tramo in enumerate(tramos):
        av('QSELL_SEG', j, k, c=-float(tramo['coef']), lb=0.0, ub=float(tramo['ancho']))

# Cardinalidad GLOBAL de empresas proveedoras.
# Esta es una única suma sobre todas las empresas de la solución; no existe
# ninguna cardinalidad por cliente. No se exige flujo mínimo si BE queda activa.
if NUMERO_EMPRESAS_A_UTILIZAR is not False:
    m.add_con(
        {V('BE',i):1 for i in I},
        lb=float(NUMERO_EMPRESAS_A_UTILIZAR),
        ub=float(NUMERO_EMPRESAS_A_UTILIZAR),
        name='numero_global_empresas_utilizadas_fijado',
    )

# Cardinalidad GLOBAL de clientes atendidos. Es una única suma sobre BJ y no
# existe ninguna exigencia de número de clientes por empresa, planta o ruta.
if NUMERO_CLIENTES_A_UTILIZAR is not False:
    m.add_con(
        {V('BJ',j):1 for j in J},
        lb=float(NUMERO_CLIENTES_A_UTILIZAR),
        ub=float(NUMERO_CLIENTES_A_UTILIZAR),
        name='numero_global_clientes_utilizados_fijado',
    )
for t in T:
    for s in S:
        av('NP',t,s,c=float(PL.loc[t,'costo_fijo_usd']),integrality=1)
        # AP=1 significa que la tecnología procesa flujo físico en ese sitio.
        # Es binaria y queda enlazada a NP para que cualquier flujo cobre al
        # menos una unidad completa del costo fijo de la planta.
        av('AP',t,s,integrality=1,lb=0.0,ub=1.0)
for t in T:
    for s in S:
        if t not in TECNOLOGIAS_PERMITIDAS_POR_SITIO[str(s)]: m.add_con({V('NP',t,s):1},lb=0,ub=0,name=f'tecnologia_no_permitida_{t}_{s}')
if NUMERO_TOTAL_PLANTAS is not False:
    m.add_con({V('NP',t,s):1 for t in T for s in S},lb=float(NUMERO_TOTAL_PLANTAS),ub=float(NUMERO_TOTAL_PLANTAS),name='numero_total_plantas_fijado')

# Variables de transporte vehicular.
# En esta versi?n s?lo se construyen rutas para BioGNL/BioGNC. Las antiguas
# familias BIO, BCPROC y BC no se registran como viajes; los flujos viales no
# locales de biog?s crudo y GasBC se bloquean abajo mediante restricciones
# directas sobre sus variables f?sicas.

# NTR[r]      : número entero no negativo de viajes completos.
# QTR[r,s]    : cantidad transformada/procesada/entregada en la parada s.
# QSEG[r,k]   : carga que permanece en los vehículos durante el tramo k.
#
# Para m paradas existen m+1 tramos: salida, tramos entre paradas y retorno.
# La carga de retorno recibe una penalización numéricamente despreciable para
# impedir circulación inútil de producto, pero puede ser positiva.
for route_id,route in TRANSPORT_ROUTE_CANDIDATES.items():
    max_flow=float(route['max_flow_from_zip'])
    stops=list(route['stops'])
    av(
        'NTR',route_id,
        c=float(route['cost_per_vehicle_usd']),
        integrality=1,
    )
    if ACTIVAR_BINARIA_GIRA_TRANSPORTE:
        av('BTR',route_id,integrality=1,ub=1)
        m.add_con(
            {V('NTR',route_id):1,V('BTR',route_id):-1},
            lb=0,
            name=f'gira_activa_exige_al_menos_un_viaje_{route_id}',
        )
    if str(route['mode']) in {'BIO','BC_PROCESO','BC'}:
        raise AssertionError(
            f'Ruta vehicular {route_id} de modo {route["mode"]} no debe existir: '
            'biog?s crudo y GasBC no se transportan en veh?culos.'
        )

    # Un vehiculo GNL/GNC solo puede parar en un nodo que tenga activa la
    # planta de reconversion correspondiente: RL para BioGNL y RC para BioGNC.
    tipo_reconversion_por_modo = {'GNL': 'RL', 'GNC': 'RC'}
    tipo_req = tipo_reconversion_por_modo.get(str(route['mode']))
    if tipo_req is not None:
        for stop in stops:
            m.add_con(
                {V('BTR', route_id): 1, V('AP', tipo_req, stop): -1},
                ub=0,
                name=f'vehiculo_{route["mode"]}_solo_para_con_{tipo_req}_activa_{route_id}_{stop}',
            )
    for stop in stops:
        av('QTR',route_id,stop)
    for segment_index in range(len(stops)+1):
        return_penalty=(
            float(PENALIZACION_RETORNO_REMANENTE_USD_UNIDAD)
            if segment_index==len(stops) else 0.0
        )
        av('QSEG',route_id,segment_index,c=return_penalty)
    if ACTIVAR_BINARIA_GIRA_TRANSPORTE:
        m.add_indicator(
            V('BTR',route_id),False,
            {V('NTR',route_id):1},
            sense='=',rhs=0.0,
            name=f'gira_inactiva_sin_viajes_{route_id}',
        )
        for stop in stops:
            m.add_indicator(
                V('BTR',route_id),False,
                {V('QTR',route_id,stop):1},
                sense='=',rhs=0.0,
                name=f'gira_inactiva_sin_descarga_{route_id}_{stop}',
            )
        for segment_index in range(len(stops)+1):
            m.add_indicator(
                V('BTR',route_id),False,
                {V('QSEG',route_id,segment_index):1},
                sense='=',rhs=0.0,
                name=f'gira_inactiva_sin_carga_{route_id}_{segment_index}',
            )

# Traslado GasBC -> entrada de gasoducto. En el modelo original era un viaje
# directo NIG. Para que en esta prueba sea imposible cualquier ciclo
# Origen -> A -> Origen, no se crean NIG viales cuando se fuerzan dos paradas.
# Las transferencias AIG locales, que no utilizan vehículo, siguen disponibles.
trip_defs_all=[]
for s,d in AIG:
    trip_defs_all.append(('NIG',(s,d),'BC',f'AIG|{s}|{d}'))
trip_defs=[
    item for item in trip_defs_all
    if road_arc_requires_truck(item[3])
    and not FORZAR_EXACTAMENTE_DOS_PARADAS
    and PERMITIR_VEHICULOS_GASBC
]
for prefix,args,veh,rk in trip_defs:
    av(prefix,*args,c=trip_cost(veh,rk),integrality=1)
    # NIG es el traslado vial de GasBC desde una planta hacia la entrada del
    # gasoducto convencional. Se conserva la variable, pero se fija a cero para
    # que ese GasBC sólo pueda llegar por transferencia local o gasoducto corto.
    if str(veh) == 'BC' and not PERMITIR_VEHICULOS_GASBC:
        m.add_con(
            {V(prefix,*args):1},lb=0,ub=0,
            name=f'prohibir_vehiculo_GasBC_{prefix}_{"_".join(map(str,args))}',
        )

# Auditoría estructural: las cardinalidades sólo pueden aparecer como una suma
# global por tipo de participante; nunca se crean cardinalidades por ruta, planta
# o pareja empresa-cliente.
if NUMERO_EMPRESAS_A_UTILIZAR is not False:
    _restricciones_cardinalidad_empresas = [
        nombre for nombre in m.row_names if 'empresas_utilizadas_fijado' in nombre
    ]
if NUMERO_CLIENTES_A_UTILIZAR is not False:
    _restricciones_cardinalidad_clientes = [
        nombre for nombre in m.row_names if 'clientes_utilizados_fijado' in nombre
    ]

# Empresas y balances tecnológicos.
def _coeficientes_suministro_empresa(i):
    co={}
    for s in ABIO_by_i[i]:
        ADD(co,'X',i,s)
    for h in H_CORTO_by_i[i]:
        co[V('QBCP',h)]=1
    return co


# Vincula la cantidad total comprada a cada proveedor con sus tramos de precio.
for i in I:
    co = _coeficientes_suministro_empresa(i)
    for k, _tramo in enumerate(TRAMOS_COMPRA.get(i, [])):
        co[V('QBUY_SEG', i, k)] = co.get(V('QBUY_SEG', i, k), 0.0) - 1.0
    m.add_con(co, lb=0.0, ub=0.0, name=f'compra_precio_segmentada_{i}')


for i in I:
    cap=float(E.loc[i,'capacidad_max_biogas_nm3'])
    if cap<=EPS:
        # Una empresa sin capacidad disponible no puede contarse como utilizada.
        m.add_con({V('BE',i):1},lb=0,ub=0,name=f'empresa_inhabilitada_{i}')
    co_max=_coeficientes_suministro_empresa(i); co_max[V('BE',i)]=-cap
    m.add_con(co_max,ub=0,name=f'empresa_max_{i}')

    if compra_porcentaje and cap>EPS:
        # Si BE_i=1: suministro total = cap_i * PORC_COMPRA.
        co_pct_sup=_coeficientes_suministro_empresa(i)
        co_pct_sup[V('PORC_COMPRA')]=-cap
        co_pct_sup[V('BE',i)]=cap
        m.add_con(co_pct_sup,ub=cap,name=f'compra_porcentaje_comun_sup_{i}')

        co_pct_inf=_coeficientes_suministro_empresa(i)
        co_pct_inf[V('PORC_COMPRA')]=-cap
        co_pct_inf[V('BE',i)]=-cap
        m.add_con(co_pct_inf,lb=-cap,name=f'compra_porcentaje_comun_inf_{i}')

# Activacion y capacidad de cada gasoducto corto. Q>0 obliga B=1 por la cota
# superior. No se impone flujo minimo si B=1: con costos fijos positivos,
# el objetivo empuja B=0 cuando no hay flujo; la cota inferior podria crear ciclos.
for h in H_CORTO:
    cap_h=float(GCORTO.loc[h,'capacidad_max_nm3_dia'])
    co_cap={V('QBCP',h):1,V('BCP',h):-cap_h}
    if h in H_EHUB_SET:
        co_cap[V('QBCP_HUB',h)]=1
    m.add_con(co_cap,ub=0,name=f'capacidad_gasoducto_biogas_corto_{h}')

# Candidatos físicos de relevo que parten desde una empresa no proveedora: no
# tienen QBCP de producción propia, pero sí pueden transportar flujo recibido.
for h in H_EHUB_SOLO:
    cap_h=float(EHUB.loc[h,'capacidad_max_nm3_dia'])
    m.add_con({V('QBCP_HUB',h):1,V('BCP_HUB',h):-cap_h},ub=0,name=f'capacidad_gasoducto_biogas_hub_empresa_{h}')

# Activación/capacidad de gasoductos cortos de biogás crudo cliente/HUB -> cliente.
for h in H_BIOC2C:
    cap_h=float(BIOC2C.loc[h,'capacidad_max_nm3_dia'])
    m.add_con({V('QBC2C',h):1,V('BCP2C',h):-cap_h},ub=0,name=f'capacidad_gasoducto_biogas_c2c_{h}')
for h in H_PHUB:
    cap_h=float(PHUB.loc[h,'capacidad_max_nm3_dia'])
    m.add_con({V('QBCP_P',h):1,V('BCP_P',h):-cap_h},ub=0,name=f'capacidad_gasoducto_biogas_phub_{h}')

# El biogás crudo recibido por gasoducto corto entra primero al balance de biogás
# crudo, igual que el recibido por carretera. Por tanto puede continuar como GasBC
# directo, BioGNL, BioGNC o ingresar a un corredor principal. No se impone una
# salida ZD artificial. En los nodos NGI sólo está permitida BG y existe un AIG
# local hacia la dirección correspondiente, de modo que allí la cadena física es
# empresa(s) -> gasoducto corto -> BG -> corredor.
for h in H_CORTO_ENTRADA_GASODUCTO:
    s_entrada=str(GCORTO.loc[h,'nodo_destino_id'])
    d_entrada=str(GCORTO.loc[h,'direccion_id'])

# Activación y capacidad de los gasoductos cortos de GasBC BG@empresa -> entrada.
for h in H_GECORTO:
    cap_h=float(GECORTO.loc[h,'capacidad_max_nm3_dia'])
    m.add_con(
        {V('QGCE',h):1,V('BGCE',h):-cap_h},ub=0,
        name=f'capacidad_gasoducto_corto_gasbc_empresa_entrada_{h}',
    )

# La red general de GasBC corto se rige únicamente por conservación de masa en
# balance_BG/balance_RL/balance_RC, activación y capacidad física de cada tubería.
# Se elimina la cota legacy que vinculaba empresa->P exclusivamente al biogás
# propio de esa empresa, porque impedía usar una ubicación de empresa como HUB
# para GasBC recibido previamente desde P/C/NGI. H_GECORTO mantiene sus balances
# y capacidades originales por separado.

# Activación y capacidad de los gasoductos cortos de GasBC salida -> cliente/empresa HUB.
for h in H_GSCORTO:
    cap_h=float(GSCORTO.loc[h,'capacidad_max_nm3_dia'])
    m.add_con(
        {V('QGCP',h):1,V('QGCP_HUB',h):1,V('BGCP',h):-cap_h},ub=0,
        name=f'capacidad_gasoducto_corto_salida_cliente_{h}',
    )
    # Refuerzo numérico equivalente al Big-M anterior: si BGCP=0, el flujo
    # físico debe ser exactamente cero. No cambia la lógica del MILP; evita
    # residuos Q>0 causados por IntFeasTol multiplicada por una capacidad grande.
    m.add_indicator(
        V('BGCP',h),False,
        {V('QGCP',h):1,V('QGCP_HUB',h):1},
        sense='=',rhs=0.0,
        name=f'inactivo_sin_flujo_gasoducto_corto_salida_cliente_{h}',
    )
# Activación y capacidad de GasBC directo empresa -> ubicación/cliente.
for h in H_GDCORTO:
    cap_h=float(GDCORTO.loc[h,'capacidad_max_nm3_dia'])
    m.add_con(
        {V('QGCD',h):1,V('QGCD_HUB',h):1,V('QGCD_HUB_RL',h):1,V('QGCD_HUB_RC',h):1,V('BGCD',h):-cap_h},ub=0,
        name=f'capacidad_gasoducto_corto_gasbc_directo_{h}',
    )

if compra_porcentaje:
    # Sin empresas activas, el porcentaje común queda fijado en cero.
    co={V('PORC_COMPRA'):1}
    for i in I: co[V('BE',i)]=co.get(V('BE',i),0)-1
    m.add_con(co,ub=0,name='porcentaje_compra_cero_sin_empresas')

# Balance de BIOGÁS CRUDO antes de BG. Esto evita duplicar masa: todo biogás
# recibido se procesa exactamente una vez o continúa por un gasoducto C2C.
for s in S:
    raw={V('QBIO_BG',s):-1}
    for i in ABIO_by_s[s]: ADD(raw,'X',i,s)
    for h in H_CORTO_by_s[s]: raw[V('QBCP',h)]=raw.get(V('QBCP',h),0)+1
    # QBCP_HUB conserva masa en nodos de empresa: entra al destino y sale del NE_E
    # de origen sin contarse como producción de la empresa intermedia.
    for h in H_EHUB_by_destino_s[s]: raw[V('QBCP_HUB',h)]=raw.get(V('QBCP_HUB',h),0)+1
    for h in H_EHUB_by_origen_s[s]: raw[V('QBCP_HUB',h)]=raw.get(V('QBCP_HUB',h),0)-1
    for h in H_BIOC2C_by_destino_s[s]: raw[V('QBC2C',h)]=raw.get(V('QBC2C',h),0)+1
    for h in H_BIOC2C_by_origen_s[s]: raw[V('QBC2C',h)]=raw.get(V('QBC2C',h),0)-1
    for h in H_PHUB_by_destino_s[s]: raw[V('QBCP_P',h)]=raw.get(V('QBCP_P',h),0)+1
    for h in H_PHUB_by_origen_s[s]: raw[V('QBCP_P',h)]=raw.get(V('QBCP_P',h),0)-1
    m.add_con(raw,lb=0,ub=0,name=f'balance_biogas_crudo_{s}')

CLIENTES_HUB_by_s=defaultdict(list)
for _j,_s in CLIENTE_A_NODO.items():
    CLIENTES_HUB_by_s[str(_s)].append(str(_j))

for s in S:
    co={}
    co[V('QBIO_BG',s)]=co.get(V('QBIO_BG',s),0)+K['BG']
    # GasBC que llega como tránsito entra al pool nodal. QGCD hacia cliente sigue
    # siendo venta final; QGCD_HUB es la fracción que NO se vende y puede reenviarse.
    for h in H_GDCORTO_by_destino_s[s]:
        if str(GDCORTO.loc[h,'destino_tipo']) in {'UBICACION_POSIBLE','UBICACION_EMPRESA','ENTRADA_GASODUCTO'}:
            # En P# o NGI el gasoducto corto no vende: deposita TODO el GasBC
            # (BG/RL/RC) en el pool nodal. En NGI este pool comparte exactamente
            # las mismas entradas L/C que el GasBC extraído del corredor.
            co[V('QGCD',h)]=co.get(V('QGCD',h),0)+1
        elif str(GDCORTO.loc[h,'destino_tipo'])=='CLIENTE':
            co[V('QGCD_HUB',h)]=co.get(V('QGCD_HUB',h),0)+1
            co[V('QGCD_HUB_RL',h)]=co.get(V('QGCD_HUB_RL',h),0)+1
            co[V('QGCD_HUB_RC',h)]=co.get(V('QGCD_HUB_RC',h),0)+1
    # GasBC que llega desde la salida del gasoducto convencional entra al mismo
    # pool nodal. Por carretera sólo existen clientes/HUB; por gasoducto corto el
    # destino puede ser cliente o ubicación física de empresa NE_E.
    for j_hub in CLIENTES_HUB_by_s[s]:
        for d in AGC_by_j[j_hub]:
            ADD(co,'ZG_HUB',d,j_hub)
    for h in H_GSCORTO_by_destino_s[s]:
        co[V('QGCP_HUB',h)]=co.get(V('QGCP_HUB',h),0)+1
    for j in ABC_by_s[s]: ADD(co,'ZD',s,j,coef=-1)
    # El GasBC puede salir desde BG[s] hacia cualquier L/C conectada, incluida
    # la transferencia local s->s sin camión.
    for q in GBL_by_p[s]: ADD(co,'GBL',s,q,coef=-1)
    for q in GBC_by_p[s]: ADD(co,'GBC',s,q,coef=-1)
    # Sólo la componente base del flujo al corredor se descuenta de este pool.
    # Las componentes provenientes de RL/RC se descuentan en sus balances propios.
    for d in AIG_by_s[s]: ADD(co,'YG_BASE',s,d,coef=-1)
    # GasBC que sale desde BG/pool nodal hacia corredor o por gasoductos cortos. Sólo la
    # componente QGCD_BG se descuenta de este balance; las componentes QGCD_RL y
    # QGCD_RC se descuentan de sus balances de reconversión respectivos.
    for h in H_GECORTO_by_s[s]: co[V('QGCE',h)]=co.get(V('QGCE',h),0)-1
    for h in H_GDCORTO_by_origen_s[s]:
        co[V('QGCD_BG',h)]=co.get(V('QGCD_BG',h),0)-1
        co[V('QGCD_HUB',h)]=co.get(V('QGCD_HUB',h),0)-1
    m.add_con(co,lb=0,ub=0,name=f'balance_BG_{s}')

# Entrada total de cada planta L/C. Además del GasBC recibido desde una BG,
# L/C puede consumir directamente GasBC extraído del gasoducto convencional si
# la planta está EXACTAMENTE en el nodo físico de salida de la dirección.
for q in S:
    q_key=str(q)
    entradaL={V('QL',q):1}
    for p in GBL_by_q[q]:
        ADD(entradaL,'GBL',p,q,coef=-1)
    for d in D_SALIDA_by_s[q]:
        entradaL[V('QG_L_DIRECT',d)]=entradaL.get(V('QG_L_DIRECT',d),0)-1
    if q_key in NGI_LOCAL_TRANSFER_SET:
        entradaL[V('QRL_L_LOCAL',q_key)]=entradaL.get(V('QRL_L_LOCAL',q_key),0)-1
        entradaL[V('QRC_L_LOCAL',q_key)]=entradaL.get(V('QRC_L_LOCAL',q_key),0)-1
    m.add_con(entradaL,lb=0,ub=0,name=f'entrada_L_desde_BG_o_salida_gasoducto_{q}')

    entradaC={V('QC',q):1}
    for p in GBC_by_q[q]:
        ADD(entradaC,'GBC',p,q,coef=-1)
    for d in D_SALIDA_by_s[q]:
        entradaC[V('QG_C_DIRECT',d)]=entradaC.get(V('QG_C_DIRECT',d),0)-1
    if q_key in NGI_LOCAL_TRANSFER_SET:
        entradaC[V('QRL_C_LOCAL',q_key)]=entradaC.get(V('QRL_C_LOCAL',q_key),0)-1
        entradaC[V('QRC_C_LOCAL',q_key)]=entradaC.get(V('QRC_C_LOCAL',q_key),0)-1
    m.add_con(entradaC,lb=0,ub=0,name=f'entrada_C_desde_BG_o_salida_gasoducto_{q}')

# Conversión en L/C y distribución posterior de BioGNL/BioGNC.
for s in S:
    coL={V('QL',s):K['L'],V('YL0',s):-1}; coC={V('QC',s):K['C'],V('YC0',s):-1}
    for q in ASITE_out[s]: coL[V('YL',s,q)]=coL.get(V('YL',s,q),0)-1; coC[V('YC',s,q)]=coC.get(V('YC',s,q),0)-1
    m.add_con(coL,lb=0,ub=0,name=f'balance_L_{s}'); m.add_con(coC,lb=0,ub=0,name=f'balance_C_{s}')
for q in S:
    q_key=str(q)
    coL={V('YL0',q):K['RL']}; coC={V('YC0',q):K['RC']}
    for p in ASITE_in[q]:
        coL[V('YL',p,q)]=coL.get(V('YL',p,q),0)+K['RL']
        coC[V('YC',p,q)]=coC.get(V('YC',p,q),0)+K['RC']
    # El GasBC reconvertido puede venderse localmente como antes o salir por uno
    # o varios gasoductos cortos hacia otros clientes. Al usar variables fuente
    # separadas no se duplica el flujo y se conserva la identidad RL vs RC.
    for j in ABC_by_s[q]:
        ADD(coL,'ZL',q,j,coef=-1)
        ADD(coC,'ZC',q,j,coef=-1)
    for h in H_GDCORTO_by_origen_s[q]:
        destino_tipo=str(GDCORTO.loc[h,'destino_tipo'])
        if destino_tipo in {'CLIENTE','UBICACION_POSIBLE','UBICACION_EMPRESA','ENTRADA_GASODUCTO'}:
            coL[V('QGCD_RL',h)]=coL.get(V('QGCD_RL',h),0)-1
            coC[V('QGCD_RC',h)]=coC.get(V('QGCD_RC',h),0)-1
        if destino_tipo=='CLIENTE':
            coL[V('QGCD_HUB_RL',h)]=coL.get(V('QGCD_HUB_RL',h),0)-1
            coC[V('QGCD_HUB_RC',h)]=coC.get(V('QGCD_HUB_RC',h),0)-1
    # NUEVO: si q coincide con la entrada física de una dirección convencional,
    # el GasBC producido por RL/RC puede inyectarse directamente al corredor.
    # YG_RL/YG_RC forman parte de YG total, por lo que capacidad y costo del
    # gasoducto siguen contabilizándose exactamente una vez.
    if q_key in NGI_LOCAL_TRANSFER_SET:
        coL[V('QRL_L_LOCAL',q_key)]=coL.get(V('QRL_L_LOCAL',q_key),0)-1
        coL[V('QRL_C_LOCAL',q_key)]=coL.get(V('QRL_C_LOCAL',q_key),0)-1
        coC[V('QRC_L_LOCAL',q_key)]=coC.get(V('QRC_L_LOCAL',q_key),0)-1
        coC[V('QRC_C_LOCAL',q_key)]=coC.get(V('QRC_C_LOCAL',q_key),0)-1
    if HABILITAR_INYECCION_RL_RC_GASODUCTO_CONVENCIONAL:
        for d in AIG_by_s[q]:
            if str(q)==str(ENTRADA_GASODUCTO_A_NODO[str(d)]):
                ADD(coL,'YG_RL',q,d,coef=-1)
                ADD(coC,'YG_RC',q,d,coef=-1)
    m.add_con(coL,lb=0,ub=0,name=f'balance_RL_{q}')
    m.add_con(coC,lb=0,ub=0,name=f'balance_RC_{q}')
# Transporte convencional DIRECTO entre los 9 gasoductos, con continuidad
# física estricta en los puntos usados por las 36 distancias.
#
# YGG[g1,g2] sigue siendo UNA sola conexión directa y nunca puede utilizar un
# tercer gasoducto como tránsito. La diferencia es que ahora sólo puede consumir
# GasBC inyectado exactamente en el punto de g1 usado para D[g1,g2], y sólo puede
# alimentar extracciones situadas exactamente en el punto de g2 usado por esa
# misma distancia. Se elimina así cualquier salto gratis entre extremos.

# 1) Balance en cada punto físico de INYECCIÓN.
for d in D:
    g=str(DG[d])
    local_in={}
    for sitio in AIG_by_d[d]:
        ADD(local_in,'YG',sitio,d)
    for h in H_GECORTO_by_d[d]:
        local_in[V('QGCE',h)]=local_in.get(V('QGCE',h),0)+1

    co=dict(local_in)
    # Recorrido longitudinal por la misma direccion del gasoducto: entrada -> salida.
    co[V('YGINTRA',d)]=co.get(V('YGINTRA',d),0)-1
    # Permanecer en el mismo gasoducto sólo conserva el mismo punto físico.
    co[V('YGGS',d)]=co.get(V('YGGS',d),0)-1
    # Una transferencia g->h sólo puede salir desde el extremo de referencia
    # almacenado en la fila correspondiente de las 36 distancias.
    for h in GG_BY_ORIGIN_D[d]:
        co[V('YGG',g,h)]=co.get(V('YGG',g,h),0)-1
    m.add_con(co,lb=0,ub=0,name=f'balance_punto_inyeccion_convencional_{d}')
    # Si no hay recorrido físico por el gasoducto, la permanencia YGGS no puede
    # absorber GasBC recién producido por RL/RC. Ese caso se maneja con las
    # transferencias locales NGI anteriores y no consume capacidad convencional.
    stay_no_rlrc={V('YGGS',d):1}
    for sitio in AIG_by_d[d]:
        ADD(stay_no_rlrc,'YG_BASE',sitio,d,coef=-1)
    for h in H_GECORTO_by_d[d]:
        stay_no_rlrc[V('QGCE',h)]=stay_no_rlrc.get(V('QGCE',h),0)-1
    m.add_con(stay_no_rlrc,ub=0,name=f'permanencia_mismo_punto_sin_RL_RC_{d}')

# 2) Balance en cada punto físico de EXTRACCIÓN.
for d in D:
    g=str(DG[d])
    local_out={}
    for j in AGC_by_d[d]:
        ADD(local_out,'ZG',d,j)
        ADD(local_out,'ZG_HUB',d,j)
    for h in H_GSCORTO_by_d[d]:
        local_out[V('QGCP',h)]=local_out.get(V('QGCP',h),0)+1
        local_out[V('QGCP_HUB',h)]=local_out.get(V('QGCP_HUB',h),0)+1
    # Consumo directo en L/C situada exactamente en la salida física. Estas
    # variables alimentan QL/QC en SALIDA_GASODUCTO_A_NODO[d].
    local_out[V('QG_L_DIRECT',d)]=local_out.get(V('QG_L_DIRECT',d),0)+1
    local_out[V('QG_C_DIRECT',d)]=local_out.get(V('QG_C_DIRECT',d),0)+1

    co={idx:-coef for idx,coef in local_out.items()}
    # El recorrido interno A/B de esta direccion llega a su propia salida.
    co[V('YGINTRA',d)]=co.get(V('YGINTRA',d),0)+1
    # El flujo local que entró por el mismo punto se identifica por la dirección
    # opuesta, cuya entrada coincide exactamente con esta salida.
    d_in=SAME_POINT_IN_D[d]
    co[V('YGGS',d_in)]=co.get(V('YGGS',d_in),0)+1
    # Sólo recibe las transferencias cuyo punto final de referencia coincide con d.
    for h in GG_BY_DEST_D[d]:
        co[V('YGG',h,g)]=co.get(V('YGG',h,g),0)+1
    m.add_con(co,lb=0,ub=0,name=f'balance_punto_extraccion_convencional_{d}')

# 3) Capacidad/activación del gasoducto físico. Se conserva exactamente la misma
# capacidad global: suma de inyecciones locales + transferencias directas recibidas.
for g in G:
    cap={}
    for d in D_by_g[g]:
        for sitio in AIG_by_d[d]:
            ADD(cap,'YG',sitio,d)
        for h in H_GECORTO_by_d[d]:
            cap[V('QGCE',h)]=cap.get(V('QGCE',h),0)+1
    for h in GG_in[g]:
        cap[V('YGG',h,g)]=cap.get(V('YGG',h,g),0)+1
    cap[V('BG',g)]=-float(GA.loc[g,'capacidad_efectiva_nm3'])
    m.add_con(cap,ub=0,name=f'cap_corredor_{g}')

# Capacidades de plantas y activación física obligatoria.
# Para cada tecnología se usa exactamente su flujo de entrada físico. Si AP=0,
# la restricción indicadora fuerza ese flujo a cero. Por contraposición, todo
# flujo positivo obliga AP=1; como NP>=AP, se paga al menos una unidad completa
# del costo fijo de la planta.
for s in S:
    flujo_planta={
        'BG':{V('QBIO_BG',s):1},
        'L':{V('QL',s):1},
        'RL':{V('YL0',s):1,**{V('YL',p,s):1 for p in ASITE_in[s]}},
        'C':{V('QC',s):1},
        'RC':{V('YC0',s):1,**{V('YC',p,s):1 for p in ASITE_in[s]}},
    }
    for t,co_flujo in flujo_planta.items():
        m.add_con(
            {V('NP',t,s):1,V('AP',t,s):-1},
            lb=0,
            name=f'activacion_fisica_paga_costo_fijo_{t}_{s}',
        )
        if co_flujo:
            m.add_indicator(
                V('AP',t,s),False,co_flujo,sense='=',rhs=0.0,
                name=f'planta_inactiva_sin_flujo_{t}_{s}',
            )
        else:
            # Una tecnología sin ningún arco físico de entrada en el sitio no
            # puede declararse en uso.
            m.add_con(
                {V('AP',t,s):1},lb=0,ub=0,
                name=f'planta_sin_entrada_inactiva_{t}_{s}',
            )
        co_cap=dict(co_flujo)
        co_cap[V('NP',t,s)]=-float(PL.loc[t,'capacidad_max_nm3'])
        m.add_con(co_cap,ub=0,name=f'cap_{t}_{s}')

# Capacidades de transporte.
def cap_trip(flow_idx,trip_idx,veh,name):
    m.add_con(
        {flow_idx:1,trip_idx:-float(VEH.loc[veh,'capacidad_max_m3_viaje'])},
        ub=0,
        name=name,
    )

def cap_trip_if_needed(rk,flow_ref,trip_ref,veh,name):
    if road_arc_requires_truck(rk):
        cap_trip(V(*flow_ref),V(*trip_ref),veh,name)

# Traslado hacia entradas de gasoducto. Los AIG locales no usan veh?culo y no
# se modifican; todo AIG vial de GasBC queda prohibido.
for s,d in AIG:
    rk=f'AIG|{s}|{d}'
    if road_arc_requires_truck(rk) and HAS('YG',s,d):
        if (not PERMITIR_VEHICULOS_GASBC) or FORZAR_EXACTAMENTE_DOS_PARADAS:
            m.add_con(
                {V('YG',s,d):1},lb=0,ub=0,
                name=f'prohibir_viaje_directo_IG_{s}_{d}',
            )
        else:
            cap_trip(
                V('YG',s,d),V('NIG',s,d),'BC',f'trip_IG_{s}_{d}'
            )

# Balance físico explícito de la carga.
#
# QSEG[r,k] es la carga agregada de todos los vehículos de la ruta durante el
# tramo k. En cada parada:
#
#     carga_que_llega = cantidad_transformada_o_entregada + remanente_que_sale
#
# De esta forma, el remanente continúa a la siguiente parada y, después de la
# última, regresa al origen. QTR es no negativa, pero puede valer exactamente 0.
for route_id,route in TRANSPORT_ROUTE_CANDIDATES.items():
    cap=float(VEH.loc[route['vehicle'],'capacidad_max_m3_viaje'])
    n_idx=V('NTR',route_id)
    stops=list(route['stops'])
    if len(stops) not in {1,2}:
        raise AssertionError(f'Ruta con más de dos paradas: {route_id}')

    # Capacidad del conjunto de vehículos en todos los tramos, incluido retorno.
    for segment_index in range(len(stops)+1):
        m.add_con(
            {
                V('QSEG',route_id,segment_index):1,
                n_idx:-cap,
            },
            ub=0,
            name=f'capacidad_tramo_{route_id}_{segment_index}',
        )

    # Conservación exacta en cada parada.
    for stop_position,stop in enumerate(stops):
        m.add_con(
            {
                V('QSEG',route_id,stop_position):1,
                V('QTR',route_id,stop):-1,
                V('QSEG',route_id,stop_position+1):-1,
            },
            lb=0,
            ub=0,
            name=f'balance_carga_parada_{route_id}_{stop_position}_{stop}',
        )

    descarga_minima = float(DESCARGA_MINIMA_POR_PARADA_POR_VIAJE or 0.0)
    if len(stops)==2 and descarga_minima>0:
        for stop in stops:
            m.add_con(
                {V('QTR',route_id,stop):1,n_idx:-descarga_minima},
                lb=0,
                name=f'descarga_minima_parada_compartida_{route_id}_{stop}',
            )

    if not PERMITIR_RETORNO_CON_REMANENTE:
        m.add_con(
            {V('QSEG',route_id,len(stops)):1},
            lb=0,
            ub=0,
            name=f'retorno_obligatoriamente_vacio_{route_id}',
        )

# Saturación condicional de directos antes de usar rutas compartidas.
# Si una ruta compartida activa entrega a un cliente j y existe un viaje directo
# desde el mismo origen/red hacia ese cliente, ese directo debe entregar
# exactamente capacidad*NTR. Si no existe directo hacia j, no se agrega nada.
_directos_por_red_cliente = defaultdict(list)
_compartidas_por_red_cliente = defaultdict(list)

for _rid, _route in TRANSPORT_ROUTE_CANDIDATES.items():
    _stops = list(_route['stops'])
    if len(_stops) == 1:
        _cliente = cliente_unico_de_parada(_stops[0])
        if _cliente is not None:
            _directos_por_red_cliente[
                (str(_route['network_id']), str(_cliente))
            ].append(_rid)
    elif len(_stops) == 2:
        for _stop in _stops:
            _cliente = cliente_unico_de_parada(_stop)
            if _cliente is not None:
                _compartidas_por_red_cliente[
                    (str(_route['network_id']), str(_cliente))
                ].append(_rid)

if not ACTIVAR_BINARIA_GIRA_TRANSPORTE and _compartidas_por_red_cliente:
    raise RuntimeError(
        'La restricción de directos llenos requiere BTR; '
        'activa ACTIVAR_BINARIA_GIRA_TRANSPORTE para imponerla con indicadores.'
    )

for (_network_id, _cliente), _shared_ids in _compartidas_por_red_cliente.items():
    _direct_ids = _directos_por_red_cliente.get((_network_id, _cliente), [])
    if not _direct_ids:
        continue
    for _shared_id in _shared_ids:
        for _direct_id in _direct_ids:
            _direct_route = TRANSPORT_ROUTE_CANDIDATES[_direct_id]
            _direct_stop = str(_direct_route['stops'][0])
            _cap = float(VEH.loc[_direct_route['vehicle'], 'capacidad_max_m3_viaje'])
            m.add_indicator(
                V('BTR', _shared_id), True,
                {
                    V('QTR', _direct_id, _direct_stop): 1,
                    V('NTR', _direct_id): -_cap,
                },
                sense='=',
                rhs=0.0,
                name=(
                    'compartida_exige_directo_lleno_'
                    f'{_network_id}_{_cliente}_{_direct_id}_{_shared_id}'
                ),
            )

# Cobertura exacta: cada flujo físico no local debe ser transportado mediante
# rutas de exactamente dos destinos. Los flujos locales permanecen sin camión.
# Si una red dispone de un solo destino, ese flujo queda forzado a cero porque
# no puede formar una gira válida A-B bajo esta prueba.
for network_id,net in TRANSPORT_NETWORKS.items():
    for stop in net['stops']:
        co={}
        for route_id in TRANSPORT_ROUTE_IDS_BY_NETWORK_STOP[(network_id,stop)]:
            co[V('QTR',route_id,stop)]=co.get(V('QTR',route_id,stop),0)+1
        for ref in net['delivery_refs'][stop]:
            co[V(*ref)]=co.get(V(*ref),0)-1
        m.add_con(
            co,lb=0,ub=0,
            name=f'cobertura_transporte_{network_id}_{stop}',
        )


def _coeficientes_venta_cliente(j):
    co = {}
    for s in ABC_by_j[j]:
        for z in ['ZD', 'ZL', 'ZC']:
            ADD(co,z,s,j)
    for d in AGC_by_j[j]:
        ADD(co,'ZG',d,j)
    for h in H_GSCORTO_by_j[j]:
        co[V('QGCP', h)] = co.get(V('QGCP', h), 0.0) + 1.0
    for h in H_GDCORTO_by_j[j]:
        co[V('QGCD', h)] = co.get(V('QGCD', h), 0.0) + 1.0
    return co

# Demanda maxima y activacion binaria GLOBAL de clientes.
for j in J:
    dem=float(CL.loc[j,'demanda_max_nm3'])
    co=_coeficientes_venta_cliente(j)
    co_precio = dict(co)
    for k, _tramo in enumerate(TRAMOS_VENTA.get(j, [])):
        co_precio[V('QSELL_SEG', j, k)] = co_precio.get(V('QSELL_SEG', j, k), 0.0) - 1.0
    m.add_con(co_precio, lb=0.0, ub=0.0, name=f'venta_precio_segmentada_{j}')

    if dem<=EPS:
        # Un cliente sin demanda disponible no puede contarse como utilizado.
        m.add_con(co,lb=0,ub=0,name=f'demanda_nula_{j}')
        m.add_con({V('BJ',j):1},lb=0,ub=0,name=f'cliente_inhabilitado_{j}')
        continue

    # Venta_j <= demanda_j * BJ_j. No se impone venta mínima si BJ_j=1.
    co_max=dict(co); co_max[V('BJ',j)]=-dem
    m.add_con(co_max,ub=0,name=f'demanda_activacion_{j}')

    if venta_porcentaje:
        # Si BJ_j=1: Venta_j = demanda_j * PORC_VENTA.
        # Si BJ_j=0: demanda_activacion fuerza Venta_j=0 y estas cotas se relajan.
        co_pct_sup=dict(co)
        co_pct_sup[V('PORC_VENTA')]=-dem
        co_pct_sup[V('BJ',j)]=dem
        m.add_con(co_pct_sup,ub=dem,name=f'venta_porcentaje_comun_sup_{j}')

        co_pct_inf=dict(co)
        co_pct_inf[V('PORC_VENTA')]=-dem
        co_pct_inf[V('BJ',j)]=-dem
        m.add_con(co_pct_inf,lb=-dem,name=f'venta_porcentaje_comun_inf_{j}')

if venta_porcentaje:
    # Sin clientes activados, el porcentaje común queda fijado en cero.
    co={V('PORC_VENTA'):1}
    for j in J: co[V('BJ',j)]=co.get(V('BJ',j),0)-1
    m.add_con(co,ub=0,name='porcentaje_venta_cero_sin_clientes')

print(
    'Número de empresas:',
    'decisión libre' if NUMERO_EMPRESAS_A_UTILIZAR is False
    else f'exactamente {NUMERO_EMPRESAS_A_UTILIZAR} activada(s) por binaria',
)
print(
    'Número de clientes:',
    'decisión libre' if NUMERO_CLIENTES_A_UTILIZAR is False
    else f'exactamente {NUMERO_CLIENTES_A_UTILIZAR} activado(s) por binaria',
)
print('Compra porcentual común:', 'activada' if compra_porcentaje else 'desactivada')
print('Venta porcentual común:', 'activada' if venta_porcentaje else 'desactivada')
# Optimización auxiliar de auditorías (NO modifica el MILP):
# las búsquedas exactas sobre una lista son O(n); sobre un set son O(1).
# m.row_names se conserva intacta y este set se usa únicamente para validar.
_row_names_set = set(m.row_names)

# Auditoría estructural de las nuevas redes HUB.

# Auditoría de la ampliación: las tuberías a cliente o P# pueden usar RL/RC
# cuando la tecnología existe en el nodo de origen.
for h in H_GDCORTO_UBICACION + H_GDCORTO_EMPRESA + H_GDCORTO_ENTRADA_GASODUCTO:
    s0=str(GDCORTO.loc[h,'nodo_origen_id'])
    if 'RL' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[s0]:
        pass
    if 'RC' in TECNOLOGIAS_PERMITIDAS_POR_SITIO[s0]:
        pass
    # El pool GasBC nodal puede alimentar la tubería aunque el GasBC haya llegado
    # desde otro nodo; ya no se prohíbe QGCD_BG para C/P/NGI -> P/E/NGI.
    if str(GDCORTO.loc[h,'destino_tipo'])=='ENTRADA_GASODUCTO':
        pass

print(
    f'Variables: {len(m.names):,}; restricciones: {len(m.rows):,}; '
    f'viajes candidatos={len(TRANSPORT_ROUTE_CANDIDATES):,}'
)
_integrality_arr = np.asarray(m.integrality, dtype=np.uint8)
_integrality_counts = np.bincount(_integrality_arr, minlength=2)
print(pd.Series({
    'continuas': int(_integrality_counts[0]),
    'enteras/binarias': int(_integrality_counts[1]),
}))
for prefix,args,veh,rk in trip_defs_all:
    if not road_arc_requires_truck(rk) or FORZAR_EXACTAMENTE_DOS_PARADAS:
        pass
if FORZAR_EXACTAMENTE_DOS_PARADAS:
    pass
# NP, NIG y NTR permanecen sin cota superior artificial. BCP sí es binaria.
if not PERMITIR_VEHICULOS_BIOGAS_CRUDO:
    pass
if not PERMITIR_VEHICULOS_GASBC:
    pass
_n_balance_carga_parada = 0
_n_capacidad_tramo = 0
for _row_name in m.row_names:
    _row_name_str = str(_row_name)
    if 'balance_carga_parada_' in _row_name_str:
        _n_balance_carga_parada += 1
    if 'capacidad_tramo_' in _row_name_str:
        _n_capacidad_tramo += 1

_expected_balance_carga = 0
_expected_capacidad_tramo = 0
for _route in TRANSPORT_ROUTE_CANDIDATES.values():
    _nstops = len(_route['stops'])
    _expected_balance_carga += _nstops
    _expected_capacidad_tramo += _nstops + 1

for route_id,route in TRANSPORT_ROUTE_CANDIDATES.items():
    if FORZAR_EXACTAMENTE_DOS_PARADAS:
        pass
    else:
        pass
    # m.idx está indexado por los nombres-tupla de las variables. V(...) devuelve
    # el índice entero, por lo que la auditoría debe comprobar las tuplas y no los
    # índices contra las claves del diccionario.
if FORZAR_EXACTAMENTE_DOS_PARADAS:
    pass


Tramos de precio creados: compra=78 venta=57
Número de empresas: exactamente 5 activada(s) por binaria
Número de clientes: decisión libre
Compra porcentual común: activada
Venta porcentual común: desactivada
Variables: 1,629,128; restricciones: 2,087,793; viajes candidatos=233,568
continuas           1161189
enteras/binarias     467939
dtype: int64


# 7 (Algoritmo)

In [ ]:
# ============================================================
# 7. RESOLUCIÓN Y AUDITORÍA ECONÓMICA
# ============================================================
# Gurobi minimiza. economic_c representa costo - ingreso, por lo que
# Pi = -(economic_c @ x). No existe ningún término de diversificación.
economic_c=np.array(m.c,dtype=float)
m.c=economic_c.tolist()
resultado=m.solve(SOLVER_OPTIONS)
print("Optimización de beneficio económico:", resultado.message)
if resultado.x is None:
    raise RuntimeError(
        "El solver no produjo una solución para maximizar el beneficio económico. "
        f"status={resultado.status}: {resultado.message}"
    )

x=np.asarray(resultado.x,dtype=float)
obj_economico=float(economic_c @ x)
if (
    obj_economico > 0
    and NUMERO_TOTAL_PLANTAS is False
    and NUMERO_EMPRESAS_A_UTILIZAR is False
    and NUMERO_CLIENTES_A_UTILIZAR is False
):
    # Cuando no hay cardinalidades fijadas, la solución nula es factible y
    # domina cualquier operación con beneficio negativo.
    x=np.zeros(len(m.names),dtype=float)
    obj_economico=0.0
    print('La solución encontrada tenía beneficio negativo; se usa la solución nula factible.')

PREFIJOS_FLUJO_FISICO={
    'X','QBCP','QBCP_HUB','QBIO_BG','QBC2C','QBCP_P','QL','QC','GBL','GBC','YL','YC','YL0','YC0','YG','QGCE','QGCD','QGCD_HUB',
    'QGCD_HUB_RL','QGCD_HUB_RC','QGCD_BG','QGCD_RL','QGCD_RC','ZD','ZL','ZC','ZG','ZG_HUB','QGCP','QGCP_HUB',
    'QG_L_DIRECT','QG_C_DIRECT','QRL_L_LOCAL','QRL_C_LOCAL','QRC_L_LOCAL','QRC_C_LOCAL','YGINTRA','YGG','QTR','QSEG',
}
def _umbral_resultado(name):
    return TOL_FLUJO_CERO_NM3 if name[0] in PREFIJOS_FLUJO_FISICO else EPS

# x se conserva intacto para auditoría económica. sol es la interpretación
# física/exportable y convierte únicamente residuos numéricos en cero.
sol={
    name:float(x[k])
    for k,name in enumerate(m.names)
    if float(x[k]) > _umbral_resultado(name)
}
beneficio=-float(economic_c @ x)

# Auditoria estructural posterior de activaciones desactivada.
# El costo fijo en la binaria deberia eliminar activaciones sin flujo salvo degeneracion numerica.

# Conteos reales para auditoria y para comprobar la cardinalidad solicitada.
def _x_by_name(prefix,*args):
    idx=m.idx.get((prefix,)+tuple(args))
    return 0.0 if idx is None else float(x[idx])

flujo_empresa={
    i:(
        sum(_x_by_name('X',i,s) for s in ABIO_by_i[i])
        +sum(_x_by_name('QBCP',h) for h in H_CORTO_by_i[i])
    )
    for i in I
}
flujo_cliente={
    j:sum(_x_by_name(z,s,j) for s in ABC_by_j[j] for z in ['ZD','ZL','ZC'])
      +sum(_x_by_name('ZG',d,j) for d in AGC_by_j[j])
      +sum(_x_by_name('QGCP',h) for h in H_GSCORTO_by_j[j])
      +sum(_x_by_name('QGCD',h) for h in H_GDCORTO_by_j[j])
    for j in J
}
flujo_ruta={ruta_id:_x_by_name(*flow_ref) for ruta_id,flow_ref,_,_ in RUTAS_ENTREGA_CANDIDATAS}
N_E=sum(v>TOL_FLUJO_CERO_NM3 for v in flujo_empresa.values())
N_C=sum(v>TOL_FLUJO_CERO_NM3 for v in flujo_cliente.values())
N_R=sum(v>TOL_FLUJO_CERO_NM3 for v in flujo_ruta.values())
numero_empresas_activadas_binaria=int(round(sum(x[V('BE',i)] for i in I)))
numero_clientes_activados_binaria=int(round(sum(x[V('BJ',j)] for j in J)))
porcentaje_compra_comun=(float(x[V('PORC_COMPRA')]) if compra_porcentaje else None)
porcentaje_venta_comun=(float(x[V('PORC_VENTA')]) if venta_porcentaje else None)

# Sin flujo mínimo, las binarias activadas pueden superar a los participantes con flujo positivo.
if NUMERO_EMPRESAS_A_UTILIZAR is not False:
    pass
if NUMERO_CLIENTES_A_UTILIZAR is not False:
    pass
# Verificación numérica de las igualdades porcentuales sobre participantes activos.
if compra_porcentaje:
    for i in I:
        cap=float(E.loc[i,'capacidad_max_biogas_nm3'])
        if flujo_empresa[i]>TOL_FLUJO_CERO_NM3:
            esperado=cap*porcentaje_compra_comun
            tol=max(1e-4,1e-7*max(1.0,cap,abs(esperado)))
if venta_porcentaje:
    for j in J:
        dem=float(CL.loc[j,'demanda_max_nm3'])
        if flujo_cliente[j]>TOL_FLUJO_CERO_NM3:
            esperado=dem*porcentaje_venta_comun
            tol=max(1e-4,1e-7*max(1.0,dem,abs(esperado)))

print(
    json.dumps(
        {
            'beneficio_usd': float(beneficio),
            'numero_empresas_activadas': int(numero_empresas_activadas_binaria),
            'numero_empresas_con_flujo_positivo': int(N_E),
            'numero_clientes_activados': int(numero_clientes_activados_binaria),
            'numero_clientes_con_flujo_positivo': int(N_C),
            'porcentaje_compra_comun': porcentaje_compra_comun,
            'porcentaje_venta_comun': porcentaje_venta_comun,
            'numero_rutas_con_flujo_positivo': int(N_R),
        },
        ensure_ascii=False,
        indent=2,
    )
)

# ------------------------------------------------------------------
# Auditoría económica robusta.
#
# IMPORTANTE: para reconstruir ingresos y costos se usa directamente el vector
# completo x. El diccionario sol sólo conserva valores > EPS y es apropiado para
# tablas de resultados, pero no para una identidad contable exacta: residuos
# numéricos muy pequeños multiplicados por costos grandes pueden hacer fallar el
# ------------------------------------------------------------------
def _xval(prefix,*args):
    idx=m.idx.get((prefix,)+tuple(args))
    return 0.0 if idx is None else float(x[idx])

def _entero_solver(v,tol=TOL_ENTERO_RESULTADO):
    r=round(float(v))
    return float(r) if abs(float(v)-r)<=tol else float(v)

revenue_reconstructed=0.0
for j, tramos in TRAMOS_VENTA.items():
    for k, tramo in enumerate(tramos):
        revenue_reconstructed += float(tramo['coef']) * _xval('QSELL_SEG', j, k)

company_var=0.0
for i, tramos in TRAMOS_COMPRA.items():
    for k, tramo in enumerate(tramos):
        company_var += float(tramo['coef']) * _xval('QBUY_SEG', i, k)

compra_segmentada_por_empresa = {
    i: sum(_xval('QBUY_SEG', i, k) for k, _tramo in enumerate(TRAMOS_COMPRA.get(i, [])))
    for i in I
}
costo_compra_segmentado_por_empresa = {
    i: sum(float(tramo['coef']) * _xval('QBUY_SEG', i, k) for k, tramo in enumerate(TRAMOS_COMPRA.get(i, [])))
    for i in I
}
venta_segmentada_por_cliente = {
    j: sum(_xval('QSELL_SEG', j, k) for k, _tramo in enumerate(TRAMOS_VENTA.get(j, [])))
    for j in J
}
ingreso_segmentado_por_cliente = {
    j: sum(float(tramo['coef']) * _xval('QSELL_SEG', j, k) for k, tramo in enumerate(TRAMOS_VENTA.get(j, [])))
    for j in J
}
precio_promedio_compra_empresa = {
    i: (costo_compra_segmentado_por_empresa[i] / compra_segmentada_por_empresa[i]
        if compra_segmentada_por_empresa[i] > EPS else float(E.loc[i, COL_PRECIO_BIOGAS_CRUDO]))
    for i in I
}
precio_promedio_venta_cliente = {
    j: (ingreso_segmentado_por_cliente[j] / venta_segmentada_por_cliente[j]
        if venta_segmentada_por_cliente[j] > EPS else float(CL.loc[j, 'precio_venta_usd_nm3']))
    for j in J
}

plant_var=(
    float(PL.loc['BG','costo_variable_usd_nm3'])*sum(_xval('QBIO_BG',s) for s in S)
    +float(PL.loc['L','costo_variable_usd_nm3'])*sum(_xval('QL',s) for s in S)
    +float(PL.loc['RL','costo_variable_usd_nm3'])*(sum(_xval('YL0',s) for s in S)+sum(_xval('YL',p,q) for p,q in ASITE))
    +float(PL.loc['C','costo_variable_usd_nm3'])*sum(_xval('QC',s) for s in S)
    +float(PL.loc['RC','costo_variable_usd_nm3'])*(sum(_xval('YC0',s) for s in S)+sum(_xval('YC',p,q) for p,q in ASITE))
)
company_fixed=sum(float(E.loc[i,'costo_activacion_biogas_usd'])*_xval('BE',i) for i in I)
plant_fixed=sum(float(PL.loc[t,'costo_fijo_usd'])*_xval('NP',t,s) for t in T for s in S)
pipe_fixed_corredores=sum(float(GA.loc[g,'costo_fijo_usd_dia'])*_xval('BG',g) for g in G)
pipe_variable_corredores=(
    sum(
        COSTO_GG_USD_KM_NM3*float(DIST_GG[(g1,g2)])*_xval('YGG',g1,g2)
        for g1,g2 in GG_DIR
    )
    +sum(
        float(GA.loc[str(DG[d]),'costo_variable_usd_km_nm3'])
        *float(DIST_INTRA_GASODUCTO_KM[str(DG[d])])
        *_xval('YGINTRA',d)
        for d in D
    )
)
pipe_fixed_biogas_corto=sum(
    float(GCORTO.loc[h,'costo_fijo_usd_dia'])*_xval('BCP',h)
    for h in H_CORTO
)
pipe_variable_biogas_corto=sum(
    float(GCORTO.loc[h,'costo_variable_usd_nm3_km'])
    *float(GCORTO.loc[h,'distancia_recta_km'])
    *_xval('QBCP',h)
    for h in H_CORTO
)
pipe_fixed_biogas_hub_empresa=sum(
    float(EHUB.loc[h,'costo_fijo_usd_dia'])*_xval('BCP_HUB',h) for h in H_EHUB_SOLO
)
pipe_variable_biogas_hub_empresa=sum(
    float(EHUB.loc[h,'costo_variable_usd_nm3_km'])*float(EHUB.loc[h,'distancia_recta_km'])*_xval('QBCP_HUB',h)
    for h in H_EHUB
)
pipe_fixed_biogas_c2c=sum(
    float(BIOC2C.loc[h,'costo_fijo_usd_dia'])*_xval('BCP2C',h) for h in H_BIOC2C
)
pipe_variable_biogas_c2c=sum(
    float(BIOC2C.loc[h,'costo_variable_usd_nm3_km'])*float(BIOC2C.loc[h,'distancia_recta_km'])*_xval('QBC2C',h)
    for h in H_BIOC2C
)
pipe_fixed_biogas_phub=sum(float(PHUB.loc[h,'costo_fijo_usd_dia'])*_xval('BCP_P',h) for h in H_PHUB)
pipe_variable_biogas_phub=sum(
    float(PHUB.loc[h,'costo_variable_usd_nm3_km'])*float(PHUB.loc[h,'distancia_recta_km'])*_xval('QBCP_P',h)
    for h in H_PHUB
)
pipe_fixed_gasbc_entrada=sum(
    float(GECORTO.loc[h,'costo_fijo_usd_dia'])*_xval('BGCE',h)
    for h in H_GECORTO
)
pipe_variable_gasbc_entrada=sum(
    float(GECORTO.loc[h,'costo_variable_usd_nm3_km'])
    *float(GECORTO.loc[h,'distancia_recta_km'])*_xval('QGCE',h)
    for h in H_GECORTO
)
pipe_fixed_gasbc_corto=sum(
    float(GSCORTO.loc[h,'costo_fijo_usd_dia'])*_xval('BGCP',h)
    for h in H_GSCORTO
)
pipe_variable_gasbc_corto=sum(
    float(GSCORTO.loc[h,'costo_variable_usd_nm3_km'])
    *float(GSCORTO.loc[h,'distancia_recta_km'])*(_xval('QGCP',h)+_xval('QGCP_HUB',h))
    for h in H_GSCORTO
)
pipe_fixed_gasbc_directo=sum(
    float(GDCORTO.loc[h,'costo_fijo_usd_dia'])*_xval('BGCD',h)
    for h in H_GDCORTO
)
pipe_variable_gasbc_directo=sum(
    float(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])
    *float(GDCORTO.loc[h,'distancia_recta_km'])*(
        _xval('QGCD',h)+_xval('QGCD_HUB',h)+_xval('QGCD_HUB_RL',h)+_xval('QGCD_HUB_RC',h)
    )
    for h in H_GDCORTO
)
pipe_fixed=pipe_fixed_corredores+pipe_fixed_biogas_corto+pipe_fixed_biogas_hub_empresa+pipe_fixed_biogas_c2c+pipe_fixed_biogas_phub+pipe_fixed_gasbc_entrada+pipe_fixed_gasbc_corto+pipe_fixed_gasbc_directo
pipe_variable=pipe_variable_corredores+pipe_variable_biogas_corto+pipe_variable_biogas_hub_empresa+pipe_variable_biogas_c2c+pipe_variable_biogas_phub+pipe_variable_gasbc_entrada+pipe_variable_gasbc_corto+pipe_variable_gasbc_directo
TRANSPORT_PREFIXES={'NIG','NTR'}
transport=float(sum(economic_c[k]*x[k] for k,n in enumerate(m.names) if n[0] in TRANSPORT_PREFIXES))
total_cost_reconstructed=float(company_var+plant_var+company_fixed+plant_fixed+pipe_fixed+pipe_variable+transport)

# Descomposición directa del mismo vector de coeficientes que usa Gurobi.
# Esta es la referencia contable exacta para los KPIs globales.
revenue_objective=float(sum(-economic_c[k]*x[k] for k in range(len(x)) if economic_c[k] < 0.0))
total_cost_objective=float(sum(economic_c[k]*x[k] for k in range(len(x)) if economic_c[k] > 0.0))
revenue=revenue_objective
total_cost=total_cost_objective

auditoria_ingresos_gap=float(revenue_reconstructed-revenue_objective)
auditoria_costos_gap=float(total_cost_reconstructed-total_cost_objective)
auditoria_beneficio_gap=float(beneficio-(revenue-total_cost))

# Diagnóstico por familia de variable, útil si en el futuro se añade un término
# económico nuevo al MILP y se olvida incorporarlo al desglose manual.
economic_by_prefix=defaultdict(float)
for k,name in enumerate(m.names):
    if economic_c[k] != 0.0 and x[k] != 0.0:
        economic_by_prefix[name[0]] += float(economic_c[k]*x[k])

AUDIT_TOL=max(
    1e-4,
    1e-9*max(1.0,abs(beneficio),abs(revenue),abs(total_cost)),
)
if abs(auditoria_ingresos_gap)>AUDIT_TOL or abs(auditoria_costos_gap)>AUDIT_TOL:
    print(
        'Advertencia de auditoría del desglose manual:',
        json.dumps(
            {
                'gap_ingresos_usd':auditoria_ingresos_gap,
                'gap_costos_usd':auditoria_costos_gap,
                'contribucion_costo_menos_ingreso_por_prefijo':dict(sorted(economic_by_prefix.items())),
            },
            ensure_ascii=False,
            indent=2,
            default=float,
        )
    )


served_by_client=[]
for j in J:
    served=float(flujo_cliente[j])
    dem=float(CL.loc[j,'demanda_max_nm3'])
    served_by_client.append({
        'cliente_id':j,
        'cliente':CL.loc[j,'nombre'],
        'activado_binario':int(round(x[V('BJ',j)])),
        'demanda_nm3':dem,
        'servido_nm3':served,
        'porcentaje_satisfecho':100*served/dem if dem else 0,
        'precio_usd_nm3':float(precio_promedio_venta_cliente.get(j, CL.loc[j,'precio_venta_usd_nm3'])),
    })
df_clientes_resultado=pd.DataFrame(served_by_client)
total_served=df_clientes_resultado['servido_nm3'].sum(); total_demand=df_clientes_resultado['demanda_nm3'].sum()

numero_total_plantas_activas=sum(sol.get(('NP',t,s),0.0) for t in T for s in S)
if NUMERO_TOTAL_PLANTAS is not False:
    pass

kpis={
    'estado_solver':str(resultado.message),'status':int(resultado.status),'mip_gap':getattr(resultado,'mip_gap',None),
    'funcion_objetivo':'maximizar beneficio económico total: ingresos - costos',
    'valor_funcion_objetivo_usd':beneficio,
    'modo_numero_empresas_a_utilizar':'libre' if NUMERO_EMPRESAS_A_UTILIZAR is False else 'fijado_exacto',
    'numero_empresas_solicitado':None if NUMERO_EMPRESAS_A_UTILIZAR is False else int(NUMERO_EMPRESAS_A_UTILIZAR),
    'numero_empresas_activadas_binaria':numero_empresas_activadas_binaria,
    'numero_empresas_utilizadas_flujo_positivo':N_E,'numero_empresas_maximo':len(I),
    'compra_porcentaje_igual_activada':bool(compra_porcentaje),
    'porcentaje_compra_comun_fraccion':porcentaje_compra_comun,
    'porcentaje_compra_comun_pct':(
        None if porcentaje_compra_comun is None else 100*porcentaje_compra_comun
    ),
    'numero_empresas_originales':int(len(empresas_completas_originales)),
    'numero_empresas_excluidas_mayor_oferta':int(len(empresas_excluidas_mayor_oferta)),
    'parametro_empresas_mayor_oferta_a_excluir':(
        None if NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR is False
        else int(NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR)
    ),
    'ids_empresas_excluidas_mayor_oferta':sorted(ids_empresas_excluidas_mayor_oferta),
    'modo_numero_clientes_a_utilizar':'libre' if NUMERO_CLIENTES_A_UTILIZAR is False else 'fijado_exacto_global',
    'numero_clientes_solicitado':None if NUMERO_CLIENTES_A_UTILIZAR is False else int(NUMERO_CLIENTES_A_UTILIZAR),
    'numero_clientes_activados_binaria':numero_clientes_activados_binaria,
    'numero_clientes_atendidos_flujo_positivo':N_C,'numero_clientes_maximo':len(J),
    'venta_porcentaje_igual_activada':bool(venta_porcentaje),
    'porcentaje_venta_comun_fraccion':porcentaje_venta_comun,
    'porcentaje_venta_comun_pct':(
        None if porcentaje_venta_comun is None else 100*porcentaje_venta_comun
    ),
    'numero_rutas_entrega_con_flujo_positivo':N_R,'numero_rutas_entrega_maximo':len(RUTAS_ENTREGA_CANDIDATAS),
    'beneficio_usd':beneficio,'ingresos_usd':revenue,'costos_totales_usd':total_cost,
    'ingresos_reconstruidos_desglose_usd':revenue_reconstructed,
    'costos_reconstruidos_desglose_usd':total_cost_reconstructed,
    'diferencia_ingresos_desglose_vs_objetivo_usd':auditoria_ingresos_gap,
    'diferencia_costos_desglose_vs_objetivo_usd':auditoria_costos_gap,
    'diferencia_beneficio_contable_usd':auditoria_beneficio_gap,
    'tolerancia_auditoria_economica_usd':AUDIT_TOL,
    'costo_empresa_variable_usd':company_var,'costo_procesamiento_variable_usd':plant_var,
    'costo_empresas_fijo_usd':company_fixed,'costo_plantas_fijo_usd':plant_fixed,
    'costo_fijo_corredores_gasoducto_usd':pipe_fixed,'costo_variable_corredores_gasoducto_usd':pipe_variable,
    'costo_corredores_gasoducto_usd':pipe_fixed+pipe_variable,'costo_transporte_vial_usd':transport,
    'demanda_total_nm3':total_demand,'demanda_satisfecha_nm3':total_served,
    'porcentaje_demanda_satisfecha':100*total_served/total_demand if total_demand else 0,
    'clientes_100pct':int((df_clientes_resultado.porcentaje_satisfecho>=99.999).sum()),
    'modo_numero_total_plantas':'libre' if NUMERO_TOTAL_PLANTAS is False else 'fijado_exacto',
    'numero_total_plantas_solicitado':None if NUMERO_TOTAL_PLANTAS is False else int(NUMERO_TOTAL_PLANTAS),
    'numero_total_plantas_activas':int(round(numero_total_plantas_activas)),
    'criterio_transferencia_local':CRITERIO_TRANSFERENCIA_LOCAL,
    'transferencias_locales_declaradas_sin_viajes_sin_camiones':int(len(LOCAL_ROAD_KEYS)),
    'nodos_fisicos_candidatos':int(len(S)),
    'nodos_cliente_candidatos_RL_RC':int(len(CLIENTE_A_NODO)),
    'relaciones_empresa_nodo_local':dict(EMPRESA_A_NODO),
    'relaciones_cliente_nodo_local':dict(CLIENTE_A_NODO),
    'factores_conversion':dict(FACTORES_CONVERSION),
    'rendimiento_ruta_directa':K['BG'],
    'rendimiento_ruta_biognl':K['BG']*K['L']*K['RL'],
    'rendimiento_ruta_biognc':K['BG']*K['C']*K['RC'],
    'modelo_reparto':'rutas directas y compartidas con balance QSEG; cada parada procesa QTR>=0, el remanente continúa y la carga final puede retornar al origen',
    'flujos_GasBC_BG_a_L_positivos':int(sum(_xval('GBL',p,q)>EPS for p,q in GBL)),
    'flujos_GasBC_BG_a_C_positivos':int(sum(_xval('GBC',p,q)>EPS for p,q in GBC)),
    'gasoductos_cortos_GasBC_directos_con_flujo':int(sum((
        _xval('QGCD',h)+_xval('QGCD_HUB',h)+_xval('QGCD_HUB_RL',h)+_xval('QGCD_HUB_RC',h)
    )>EPS for h in H_GDCORTO)),
    'gasoductos_cortos_biogas_cliente_cliente_con_flujo':int(sum(_xval('QBC2C',h)>EPS for h in H_BIOC2C)),
    'gasoductos_cortos_biogas_P_hub_con_flujo':int(sum(_xval('QBCP_P',h)>EPS for h in H_PHUB)),
    'flujo_biogas_crudo_reenviado_desde_P_nm3':float(sum(_xval('QBCP_P',h) for h in H_PHUB)),
    'flujo_GasBC_reenviado_por_hubs_nm3':float(
        sum(_xval('QGCD_HUB',h)+_xval('QGCD_HUB_RL',h)+_xval('QGCD_HUB_RC',h) for h in H_GDCORTO)
        +sum(_xval('ZG_HUB',d,j) for d,j in AGC)
        +sum(_xval('QGCP_HUB',h) for h in H_GSCORTO)
    ),
}

# ------------------------------------------------------------------
# Auditoría de rutas y balance físico de carga QSEG.
# También se construyen mapas de atribución para repartir el costo de cada viaje
# entre los flujos físicos que realmente transporta.
# ------------------------------------------------------------------
transport_cost_by_flow_ref=defaultdict(float)
transport_trips_by_flow_ref=defaultdict(float)
transport_distance_by_flow_ref=defaultdict(float)
transport_segment_keys_by_flow_ref=defaultdict(list)
transport_tour_ids_by_flow_ref=defaultdict(list)
# Trayectorias físicas exactas por flujo económico. Cada entrada conserva sólo
# el prefijo de la gira que realmente lleva producto hasta la parada de ese flujo.
# Esto evita fusionar paradas posteriores/retornos con la ruta de otro cliente.
transport_paths_by_flow_ref=defaultdict(list)

transport_route_rows=[]
transport_arc_rows=[]
transport_decision_rows=[]

_OPERACION_POR_MODO={
    'BIO':'biogas_entregado_a_BG_para_transformacion',
    'BC_PROCESO':'GasBC_entregado_a_planta_L_o_C',
    'GNL':'BioGNL_transformado_a_GasBC_en_RL',
    'GNC':'BioGNC_transformado_a_GasBC_en_RC',
    'BC':'GasBC_entregado_al_cliente',
}

for route_id,route in TRANSPORT_ROUTE_CANDIDATES.items():
    stops=list(map(str,route['stops']))
    n=_entero_solver(_xval('NTR',route_id))
    deliveries={
        stop:_xval('QTR',route_id,stop)
        for stop in stops
    }
    loads=[
        _xval('QSEG',route_id,segment_index)
        for segment_index in range(len(stops)+1)
    ]
    total_processed=sum(deliveries.values())
    if n<=EPS and total_processed<=TOL_FLUJO_CERO_NM3 and max(loads,default=0.0)<=TOL_FLUJO_CERO_NM3:
        continue

    cap=float(VEH.loc[route['vehicle'],'capacidad_max_m3_viaje'])
    tol_balance=max(
        1e-6,
        1e-7*max(1.0,cap*max(1.0,n),total_processed,max(loads,default=0.0)),
    )

    if n<=EPS and (total_processed>tol_balance or max(loads,default=0.0)>tol_balance):
        raise AssertionError(
            f'Ruta {route_id} tiene carga o transformación sin viajes NTR.'
        )

    # Capacidad y conservación exacta: llega = transforma/entrega + remanente.
    for segment_index,load in enumerate(loads):
        if load < -tol_balance or load > cap*n+tol_balance:
            raise AssertionError(
                f'Carga fuera de capacidad en QSEG[{route_id},{segment_index}]: '
                f'{load}; capacidad agregada={cap*n}.'
            )
    for stop_position,stop in enumerate(stops):
        incoming=loads[stop_position]
        processed=deliveries[stop]
        outgoing=loads[stop_position+1]
        if processed < -tol_balance:
            raise AssertionError(
                f'QTR[{route_id},{stop}] debe ser mayor o igual que cero.'
            )
        if abs(incoming-processed-outgoing)>tol_balance:
            raise AssertionError(
                f'Balance de carga incumplido en {route_id}, parada {stop}: '
                f'llega={incoming}, procesa={processed}, sale={outgoing}.'
            )

    route_cost=float(route['cost_per_vehicle_usd'])*n
    route_distance=float(route['distance_km'])*n

    # Atribución económica sólo a las cantidades realmente procesadas/entregadas.
    for stop,q_stop in deliveries.items():
        if q_stop<=TOL_FLUJO_CERO_NM3 or total_processed<=TOL_FLUJO_CERO_NM3:
            continue
        refs=[tuple(ref) for ref in route['delivery_refs'][stop]]
        ref_values={ref:max(0.0,_xval(*ref)) for ref in refs}
        ref_total=sum(ref_values.values())
        if ref_total<=TOL_FLUJO_CERO_NM3:
            continue
        stop_share=q_stop/total_processed
        for ref,ref_value in ref_values.items():
            if ref_value<=TOL_FLUJO_CERO_NM3:
                continue
            ref_share=ref_value/ref_total
            share=stop_share*ref_share
            transport_cost_by_flow_ref[ref]+=route_cost*share
            transport_trips_by_flow_ref[ref]+=n*share
            transport_distance_by_flow_ref[ref]+=route_distance*share
            # La ruta económica asociada a esta parada usa sólo el trayecto
            # origen -> ... -> parada actual. No incluye segmentos posteriores ni
            # el retorno al origen, porque esos tramos no llevaron producto a este
            # cliente/planta.
            stop_position = stops.index(str(stop))
            prefix_keys = list(route['segment_keys'][:stop_position+1])
            prefix_reverse = list(route.get('segment_reverse', [False]*len(route['segment_keys']))[:stop_position+1])
            for key in prefix_keys:
                if key not in transport_segment_keys_by_flow_ref[ref]:
                    transport_segment_keys_by_flow_ref[ref].append(key)
            if route_id not in transport_tour_ids_by_flow_ref[ref]:
                transport_tour_ids_by_flow_ref[ref].append(route_id)
            transport_paths_by_flow_ref[ref].append({
                'transport_route_id':str(route_id),
                'mode':str(route['mode']),
                'vehicle':str(route['vehicle']),
                'target_stop':str(stop),
                'stop_position':int(stop_position),
                'attributed_quantity':float(q_stop*ref_share),
                'attributed_share':float(share),
                'segment_keys':prefix_keys,
                'segment_reverse':prefix_reverse,
            })

    carga_retorno=loads[-1]
    capacidad_agregada=cap*n
    transport_route_rows.append({
        'transport_route_id':route_id,
        'network_id':route['network_id'],
        'modo':route['mode'],
        'origen_reparto':route['origin_code'],
        'origen_nombre':route['origin_name'],
        'vehiculo':route['vehicle'],
        'secuencia_paradas':' -> '.join(stops),
        'paradas_ids':stops,
        'numero_paradas':len(stops),
        'numero_viajes':n,
        'capacidad_agregada_vehiculos':capacidad_agregada,
        'carga_total':total_processed,
        'carga_total_transformada_o_entregada':total_processed,
        'carga_salida_origen_calculada':loads[0],
        'carga_salida_origen_QSEG':loads[0],
        'carga_retorno_origen_QSEG':carga_retorno,
        'ocupacion_salida_porcentaje':(
            100.0*loads[0]/capacidad_agregada if capacidad_agregada>EPS else 0.0
        ),
        'descarga_minima_por_parada_requerida':float(DESCARGA_MINIMA_POR_PARADA_POR_VIAJE or 0.0),
        'todas_paradas_transformacion_no_negativa':all(
            q>=-tol_balance for q in deliveries.values()
        ),
        'ambas_paradas_con_descarga_positiva':(
            len(stops)==2 and all(q>TOL_FLUJO_CERO_NM3 for q in deliveries.values())
        ),
        'distancia_por_viaje_km':float(route['distance_km']),
        'costo_por_viaje_usd':float(route['cost_per_vehicle_usd']),
        'distancia_total_km':route_distance,
        'costo_total_usd':route_cost,
        'regreso_vacio':bool(carga_retorno<=EPS),
        'retorno_con_remanente':bool(carga_retorno>EPS),
    })

    nodes=[('ORIGEN',stops[0])]
    if len(stops)==2:
        nodes.append((stops[0],stops[1]))
    nodes.append((stops[-1],'ORIGEN'))

    vehicle=route['vehicle']
    fixed=float(VEH.loc[vehicle,'costo_fijo_viaje_usd'])
    km=float(VEH.loc[vehicle,'costo_km_usd'])
    for idx,(edge,key,reverse,distance,load) in enumerate(zip(
        nodes,
        route['segment_keys'],
        route['segment_reverse'],
        route['segment_distances_km'],
        loads,
    )):
        segment_cost=n*(km*float(distance)+(fixed if idx==0 else 0.0))
        transport_arc_rows.append({
            'transport_route_id':route_id,
            'network_id':route['network_id'],
            'modo':route['mode'],
            'origen_reparto':route['origin_code'],
            'segmento_indice':idx,
            'tipo_tramo':'SALIDA' if idx==0 else ('RETORNO' if idx==len(nodes)-1 else 'CONTINUA'),
            'desde':edge[0],
            'hacia':edge[1],
            'route_key':key,
            'polilinea_invertida':bool(reverse),
            'numero_camiones':n,
            'carga_transportada':load,
            'carga_promedio_por_camion':load/n if n>EPS else 0.0,
            'origen_carga_tramo':f'QSEG[{route_id},{idx}]',
            'distancia_km':float(distance),
            'costo_usd':segment_cost,
        })

    for stop_position,(stop,q_stop) in enumerate(deliveries.items()):
        incoming=loads[stop_position]
        outgoing=loads[stop_position+1]
        next_destination=(
            stops[stop_position+1]
            if stop_position+1<len(stops)
            else str(route['origin_code'])
        )
        transport_decision_rows.append({
            'transport_route_id':route_id,
            'network_id':route['network_id'],
            'modo':route['mode'],
            'origen_reparto':route['origin_code'],
            'numero_orden_parada':stop_position+1,
            'parada_id':stop,
            'operacion_en_parada':_OPERACION_POR_MODO.get(route['mode'],'transformacion_o_entrega'),
            'carga_entrada_parada':incoming,
            'entrega_o_descarga':q_stop,
            'cantidad_transformada_o_entregada':q_stop,
            'remanente_despues_parada':outgoing,
            'destino_del_remanente':next_destination,
            'accion_del_remanente':(
                'CONTINUAR_A_SIGUIENTE_PARADA'
                if stop_position+1<len(stops)
                else 'REGRESAR_AL_ORIGEN'
            ),
            'descarga_minima_requerida':float(DESCARGA_MINIMA_POR_PARADA_POR_VIAJE or 0.0),
            'transformacion_mayor_o_igual_a_cero':bool(q_stop>=-tol_balance),
            'parada_real_con_descarga_positiva':bool(q_stop>TOL_FLUJO_CERO_NM3),
            'variable_descarga_milp':f'QTR[{route_id},{stop}]',
            'variable_carga_entrada_milp':f'QSEG[{route_id},{stop_position}]',
            'variable_remanente_salida_milp':f'QSEG[{route_id},{stop_position+1}]',
            'numero_camiones':n,
            'regreso_vacio':bool(carga_retorno<=EPS),
            'carga_retorno_origen':carga_retorno,
        })

df_viajes_transporte=pd.DataFrame(transport_route_rows)
df_arcos_transporte=pd.DataFrame(transport_arc_rows)
df_decisiones_transporte=pd.DataFrame(transport_decision_rows)

# Alias conservados para no romper exportaciones y partes del visualizador.
df_reparto_arcos=df_arcos_transporte.copy()
if len(df_reparto_arcos):
    df_reparto_arcos['carga_promedio_por_camion_nm3']=(
        df_reparto_arcos['carga_transportada']
        /df_reparto_arcos['numero_camiones'].replace(0,np.nan)
    )
df_decisiones_reparto=df_decisiones_transporte.copy()
df_giras_reparto=df_viajes_transporte.copy()
if len(df_giras_reparto):
    df_giras_reparto['secuencia_clientes']=df_giras_reparto['secuencia_paradas']
    df_giras_reparto['numero_clientes_por_gira']=df_giras_reparto['numero_paradas']

kpis['numero_redes_transporte_simple']=len(TRANSPORT_NETWORKS)
kpis['numero_viajes_candidatos_totales']=len(TRANSPORT_ROUTE_CANDIDATES)
kpis['numero_viajes_candidatos_directos']=int(
    sum(len(route['stops'])==1 for route in TRANSPORT_ROUTE_CANDIDATES.values())
)
kpis['numero_viajes_candidatos_compartidos']=int(
    sum(len(route['stops'])==2 for route in TRANSPORT_ROUTE_CANDIDATES.values())
)
kpis['numero_viajes_candidatos_utilizados']=int(
    sum(_xval('NTR',rid)>EPS for rid in TRANSPORT_ROUTE_CANDIDATES)
)
kpis['numero_vehiculos_totales_en_viajes']=float(
    sum(_xval('NTR',rid) for rid in TRANSPORT_ROUTE_CANDIDATES)
)
kpis['permitir_rutas_compartidas']=bool(PERMITIR_RUTAS_COMPARTIDAS)
kpis['forzar_exactamente_dos_paradas']=bool(FORZAR_EXACTAMENTE_DOS_PARADAS)
kpis['permitir_retorno_con_remanente']=bool(PERMITIR_RETORNO_CON_REMANENTE)
kpis['minimo_paradas_por_ruta_fijado_estructuralmente']=2 if FORZAR_EXACTAMENTE_DOS_PARADAS else 1
kpis['maximo_paradas_por_ruta_fijado_estructuralmente']=2
if FORZAR_EXACTAMENTE_DOS_PARADAS:
    pass
if len(df_viajes_transporte):
    pass
if len(df_decisiones_transporte):
    pass
kpis['numero_variables_carga_por_tramo_QSEG']=int(
    sum(len(route['stops'])+1 for route in TRANSPORT_ROUTE_CANDIDATES.values())
)
kpis['numero_maximo_viajes_impuesto']=None
kpis['transformacion_minima_por_parada_impuesta']=float(DESCARGA_MINIMA_POR_PARADA_POR_VIAJE or 0.0)
kpis['rutas_activas_con_dos_descargas_reales']=int(
    df_viajes_transporte['ambas_paradas_con_descarga_positiva'].sum()
) if len(df_viajes_transporte) else 0
kpis['rutas_activas_con_retorno_remanente']=int(
    df_viajes_transporte['retorno_con_remanente'].sum()
) if len(df_viajes_transporte) else 0
kpis['carga_total_retornada_al_origen']=float(
    df_viajes_transporte['carga_retorno_origen_QSEG'].sum()
) if len(df_viajes_transporte) else 0.0

print(json.dumps(kpis,ensure_ascii=False,indent=2,default=str))
# La identidad beneficio = ingresos - costos ya fue verificada arriba usando
# exactamente los coeficientes económicos entregados a Gurobi.


Resolviendo una sola optimización MILP desde cero con Gurobi: 1,629,128 variables, 2,087,793 restricciones originales, 7,968 viajes directos y 225,600 viajes de exactamente dos destinos disponibles.
Filas entregadas a Gurobi: 2,087,793 (igualdades=701,469, cotas inferiores=233,913, cotas superiores=1,152,411).
Gurobi WLS Access ID: ··········
Gurobi WLS Secret: ··········
Gurobi WLS License ID: ··········
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2851354
Academic license 2851354 - for non-commercial use only - registered to ar___@alumnos.uai.cl
Set parameter TimeLimit to value 3000
Set parameter MIPGap to value 1e-07
Set parameter Presolve to value -1
Set parameter Cuts to value -1
Set parameter MIPFocus to value 0
Set parameter Threads to value 0
Set parameter OutputFlag to value 1
Set parameter FeasibilityTol to value 1e-09
Set parameter IntFeasTol to value 1e-09
Set parameter NumericFocus to value 2
Set parameter LogFile to value "/content/Re

 # 8

In [ ]:
# ============================================================
# 8. TABLAS DE RESULTADOS Y TRAZABILIDAD CON CONVERSIONES
# ============================================================
import copy
def value(prefix,*args): return sol.get((prefix,)+tuple(args),0.0)

def company_supply_value(i):
    return (
        sum(value('X',i,s) for s in ABIO_by_i[i])
        +sum(value('QBCP',h) for h in H_CORTO_by_i[i])
    )

def plant_input(t,s):
    if t=='BG': return value('QBIO_BG',s)
    if t=='L': return value('QL',s)
    if t=='RL': return value('YL0',s)+sum(value('YL',p,s) for p in ASITE_in[s])
    if t=='C': return value('QC',s)
    if t=='RC': return value('YC0',s)+sum(value('YC',p,s) for p in ASITE_in[s])
    raise KeyError(t)

def plant_output(t,s):
    return FACTORES_CONVERSION[t]*plant_input(t,s)


def cantidad_tramo_compra_empresa(i):
    return sum(value('QBUY_SEG', i, k) for k, _tramo in enumerate(TRAMOS_COMPRA.get(i, [])))


def costo_tramo_compra_empresa(i):
    return sum(float(tramo['coef']) * value('QBUY_SEG', i, k) for k, tramo in enumerate(TRAMOS_COMPRA.get(i, [])))


def cantidad_tramo_venta_cliente(j):
    return sum(value('QSELL_SEG', j, k) for k, _tramo in enumerate(TRAMOS_VENTA.get(j, [])))


def ingreso_tramo_venta_cliente(j):
    return sum(float(tramo['coef']) * value('QSELL_SEG', j, k) for k, tramo in enumerate(TRAMOS_VENTA.get(j, [])))


compra_segmentada_por_empresa = {i: cantidad_tramo_compra_empresa(i) for i in I}
costo_compra_segmentado_por_empresa = {i: costo_tramo_compra_empresa(i) for i in I}
venta_segmentada_por_cliente = {j: cantidad_tramo_venta_cliente(j) for j in J}
ingreso_segmentado_por_cliente = {j: ingreso_tramo_venta_cliente(j) for j in J}

precio_promedio_compra_empresa = {
    i: (costo_compra_segmentado_por_empresa[i] / compra_segmentada_por_empresa[i]
        if compra_segmentada_por_empresa[i] > EPS else float(E.loc[i, COL_PRECIO_BIOGAS_CRUDO]))
    for i in I
}
precio_promedio_venta_cliente = {
    j: (ingreso_segmentado_por_cliente[j] / venta_segmentada_por_cliente[j]
        if venta_segmentada_por_cliente[j] > EPS else float(CL.loc[j, 'precio_venta_usd_nm3']))
    for j in J
}

df_tramos_compra_resultado = pd.DataFrame([
    {
        'empresa_id': i,
        'tramo': k,
        'q_min_nm3': tramo['q_min'],
        'q_max_nm3': tramo['q_max'],
        'ancho_nm3': tramo['ancho'],
        'costo_incremental_usd_nm3': tramo['coef'],
        'cantidad_usada_nm3': value('QBUY_SEG', i, k),
    }
    for i, tramos in TRAMOS_COMPRA.items()
    for k, tramo in enumerate(tramos)
])

df_tramos_venta_resultado = pd.DataFrame([
    {
        'cliente_id': j,
        'tramo': k,
        'q_min_nm3': tramo['q_min'],
        'q_max_nm3': tramo['q_max'],
        'ancho_nm3': tramo['ancho'],
        'ingreso_incremental_usd_nm3': tramo['coef'],
        'cantidad_usada_nm3': value('QSELL_SEG', j, k),
    }
    for j, tramos in TRAMOS_VENTA.items()
    for k, tramo in enumerate(tramos)
])

df_empresas_resultado=pd.DataFrame([
    {'empresa_id':i,'empresa':E.loc[i,'nombre'],
     'utilizada':int(company_supply_value(i)>EPS),
     'binaria_activacion_economica':round(value('BE',i)),
     'suministro_biogas_crudo_nm3':company_supply_value(i),
     'suministro_nm3':company_supply_value(i),
     'precio_promedio_compra_biogas_usd_nm3':precio_promedio_compra_empresa.get(i, float(E.loc[i, COL_PRECIO_BIOGAS_CRUDO])),
     'costo_compra_biogas_segmentado_usd':costo_compra_segmentado_por_empresa.get(i, 0.0),
     'capacidad_nm3':float(E.loc[i,'capacidad_max_biogas_nm3']),
     'porcentaje_capacidad_comprado':(
         100*company_supply_value(i)
         /float(E.loc[i,'capacidad_max_biogas_nm3'])
         if float(E.loc[i,'capacidad_max_biogas_nm3'])>EPS else 0.0
     )}
    for i in I
])
df_plantas_resultado=pd.DataFrame([
    {'tipo':t,'sitio_id':s,'sitio':sitios.set_index('id').loc[s,'nombre'],'unidades_activas':round(value('NP',t,s)),
     'factor_conversion_salida_por_entrada':float(FACTORES_CONVERSION[t]),
     'entrada_procesada_nm3':plant_input(t,s),'salida_producida_nm3':plant_output(t,s),
     'capacidad_por_unidad_nm3_dia':float(PL.loc[t,'capacidad_max_nm3']),
     'capacidad_total_entrada_nm3_dia':value('NP',t,s)*float(PL.loc[t,'capacidad_max_nm3'])}
    for t in T for s in S if value('NP',t,s)>EPS or plant_input(t,s)>EPS
])

# Trazabilidad explícita del GasBC enviado desde cada BG hacia plantas L/C.
df_gasbc_bg_a_l_c=pd.DataFrame([
    {
        'tipo_destino':tipo,
        'bg_origen_id':p,
        'bg_origen':sitios.set_index('id').loc[p,'nombre'],
        'planta_destino_id':q,
        'planta_destino':sitios.set_index('id').loc[q,'nombre'],
        'gasbc_transportado_nm3':value(prefijo,p,q),
        'transferencia_local_sin_camion':bool(str(p)==str(q) or not road_arc_requires_truck(f'ASITE|{p}|{q}')),
    }
    for tipo,prefijo,arcos in [('L','GBL',GBL),('C','GBC',GBC)]
    for p,q in arcos
    if value(prefijo,p,q)>EPS
])

# GasBC extraído directamente del corredor hacia L/C instalada exactamente en
# su salida física. No existe tramo corto, carretera ni vehículo antes de L/C.
df_gasbc_gasoducto_a_l_c=pd.DataFrame([
    {
        'tipo_destino':tipo,
        'direccion_salida_id':str(d),
        'gasoducto_id':str(DG[d]),
        'gasoducto':str(GA.loc[str(DG[d]),'nombre']),
        'nodo_salida_id':str(SALIDA_GASODUCTO_A_NODO[str(d)]),
        'nodo_salida':str(sitios.set_index('id').loc[str(SALIDA_GASODUCTO_A_NODO[str(d)]),'nombre']),
        'gasbc_directo_a_planta_nm3':value(prefijo,d),
        'transferencia_local_sin_camion':True,
        'gasoducto_corto_intermedio':False,
    }
    for tipo,prefijo in [('L','QG_L_DIRECT'),('C','QG_C_DIRECT')]
    for d in D
    if value(prefijo,d)>TOL_FLUJO_CERO_NM3
])

df_gasoductos_resultado=pd.DataFrame([
    {
        'gasoducto_id':g,'gasoducto':GA.loc[g,'nombre'],'activado':round(value('BG',g)),
        'flujo_total_nm3':(
            sum(value('YG',s,d) for d in D_by_g[g] for s in AIG_by_d[d])
            +sum(value('QGCE',h) for h in H_GECORTO_by_g[g])
            +sum(value('YGG',h,g) for h in GG_in[g])
        ),
        'flujo_entrada_desde_otros_gasoductos_nm3':sum(value('YGG',h,g) for h in GG_in[g]),
        'flujo_salida_hacia_otros_gasoductos_nm3':sum(value('YGG',g,h) for h in GG_out[g]),
        'flujo_entrada_pool_base_nm3':sum(value('YG_BASE',s,d) for d in D_by_g[g] for s in AIG_by_d[d]),
        'flujo_entrada_desde_RL_nm3':sum(value('YG_RL',s,d) for d in D_by_g[g] for s in AIG_by_d[d]),
        'flujo_entrada_desde_RC_nm3':sum(value('YG_RC',s,d) for d in D_by_g[g] for s in AIG_by_d[d]),
        'flujo_entrada_por_gc_corto_nm3':sum(value('QGCE',h) for h in H_GECORTO_by_g[g]),
        'flujo_recorrido_interno_AB_BA_nm3':sum(value('YGINTRA',d) for d in D_by_g[g]),
        'flujo_recorrido_interno_AB_nm3':value('YGINTRA',f'{g}_AB'),
        'flujo_recorrido_interno_BA_nm3':value('YGINTRA',f'{g}_BA'),
        'flujo_salida_directa_a_L_nm3':sum(value('QG_L_DIRECT',d) for d in D_by_g[g]),
        'flujo_salida_directa_a_C_nm3':sum(value('QG_C_DIRECT',d) for d in D_by_g[g]),
        'flujo_salida_directa_a_L_C_nm3':sum(
            value('QG_L_DIRECT',d)+value('QG_C_DIRECT',d) for d in D_by_g[g]
        ),
        'flujo_AB_nm3':(
            sum(value('YG',s,f'{g}_AB') for s in AIG_by_d[f'{g}_AB'])
            +sum(value('QGCE',h) for h in H_GECORTO_by_d[f'{g}_AB'])
        ),
        'flujo_BA_nm3':(
            sum(value('YG',s,f'{g}_BA') for s in AIG_by_d[f'{g}_BA'])
            +sum(value('QGCE',h) for h in H_GECORTO_by_d[f'{g}_BA'])
        ),
        'capacidad_efectiva_nm3':float(GA.loc[g,'capacidad_efectiva_nm3']),
        'punto_A':GA.loc[g,'punto_a_nombre'],'zona_A':GA.loc[g,'punto_a_zona'],
        'punto_B':GA.loc[g,'punto_b_nombre'],'zona_B':GA.loc[g,'punto_b_zona'],
        'distancia_red_estimada_km':float(GA.loc[g,'distancia_red_estimada_km']),
        'costo_fijo_usd_dia':float(GA.loc[g,'costo_fijo_usd_dia']),
        'costo_variable_usd_km_nm3':float(GA.loc[g,'costo_variable_usd_km_nm3']),
        'costo_variable_total_usd':(
            sum(
                COSTO_GG_USD_KM_NM3*float(DIST_GG[(g,h)])*value('YGG',g,h)
                for h in GG_out[g]
            )
            +sum(
                float(GA.loc[g,'costo_variable_usd_km_nm3'])
                *float(DIST_INTRA_GASODUCTO_KM[g])
                *value('YGINTRA',d)
                for d in D_by_g[g]
            )
        ),
        'costo_total_gasoducto_usd':(
            float(GA.loc[g,'costo_fijo_usd_dia'])*round(value('BG',g))
            +sum(COSTO_GG_USD_KM_NM3*float(DIST_GG[(g,h)])*value('YGG',g,h) for h in GG_out[g])
            +sum(float(GA.loc[g,'costo_variable_usd_km_nm3'])*float(DIST_INTRA_GASODUCTO_KM[g])*value('YGINTRA',d) for d in D_by_g[g])
        )
    }
    for g in G
])

df_transferencias_gasoducto_gasoducto=pd.DataFrame([
    {
        'gasoducto_origen_id':g1,
        'gasoducto_origen':str(GA.loc[g1,'nombre']),
        'gasoducto_destino_id':g2,
        'gasoducto_destino':str(GA.loc[g2,'nombre']),
        'distancia_km':float(DIST_GG[(g1,g2)]),
        'flujo_nm3':value('YGG',g1,g2),
        'costo_variable_usd_km_nm3':float(COSTO_GG_USD_KM_NM3),
        'costo_variable_total_usd':float(COSTO_GG_USD_KM_NM3)*float(DIST_GG[(g1,g2)])*value('YGG',g1,g2),
    }
    for g1,g2 in GG_DIR if value('YGG',g1,g2)>TOL_FLUJO_CERO_NM3
])

df_recorridos_internos_gasoducto=pd.DataFrame([
    {
        'direccion_id':str(d),
        'gasoducto_id':str(DG[d]),
        'gasoducto':str(GA.loc[str(DG[d]),'nombre']),
        'sentido':str(GD.loc[d,'sentido']),
        'entrada_nombre':str(GD.loc[d,'entrada_nombre']),
        'salida_nombre':str(GD.loc[d,'salida_nombre']),
        'distancia_red_estimada_km':float(DIST_INTRA_GASODUCTO_KM[str(DG[d])]),
        'flujo_nm3':value('YGINTRA',d),
        'costo_variable_usd_km_nm3':float(GA.loc[str(DG[d]),'costo_variable_usd_km_nm3']),
        'costo_variable_total_usd':float(GA.loc[str(DG[d]),'costo_variable_usd_km_nm3'])*float(DIST_INTRA_GASODUCTO_KM[str(DG[d])])*value('YGINTRA',d),
    }
    for d in D if value('YGINTRA',d)>TOL_FLUJO_CERO_NM3
])

df_gasoductos_biogas_cortos_resultado=pd.DataFrame([
    {
        'gasoducto_id':h,
        'gasoducto':str(GCORTO.loc[h,'nombre']),
        'tipo_producto':'Biogás crudo',
        'empresa_id':str(GCORTO.loc[h,'empresa_id']),
        'destino_tipo':str(GCORTO.loc[h,'destino_tipo']),
        'destino_id':str(GCORTO.loc[h,'destino_id']),
        'destino':str(GCORTO.loc[h,'destino_nombre']),
        'nodo_destino_id':str(GCORTO.loc[h,'nodo_destino_id']),
        'cliente_id':(
            None if pd.isna(GCORTO.loc[h,'cliente_id'])
            else str(GCORTO.loc[h,'cliente_id'])
        ),
        'ubicacion_id':(
            None if pd.isna(GCORTO.loc[h,'ubicacion_id'])
            else str(GCORTO.loc[h,'ubicacion_id'])
        ),
        'proceso_posterior':(
            'En cliente/HUB: puede procesarse en BG o reenviarse todavía como biogás crudo hacia otros clientes'
            if str(GCORTO.loc[h,'destino_tipo'])=='CLIENTE'
            else (
                'BG en entrada del gasoducto e inyección local al corredor principal'
                if str(GCORTO.loc[h,'destino_tipo'])=='ENTRADA_GASODUCTO'
                else (
                    'BG en ubicación de empresa; puede concentrar varias empresas y continuar como GasBC, BioGNL, BioGNC o gasoducto'
                    if str(GCORTO.loc[h,'destino_tipo'])=='UBICACION_EMPRESA'
                    else 'BG en ubicación posible; puede continuar como GasBC, BioGNL, BioGNC o gasoducto'
                )
            )
        ),
        'activado':round(value('BCP',h)),
        'flujo_produccion_nm3':value('QBCP',h),
        'flujo_transito_hub_nm3':value('QBCP_HUB',h) if h in H_EHUB_SET else 0.0,
        'flujo_total_nm3':value('QBCP',h)+(value('QBCP_HUB',h) if h in H_EHUB_SET else 0.0),
        'capacidad_efectiva_nm3':float(GCORTO.loc[h,'capacidad_max_nm3_dia']),
        'distancia_recta_km':float(GCORTO.loc[h,'distancia_recta_km']),
        'costo_fijo_usd_dia':float(GCORTO.loc[h,'costo_fijo_usd_dia']),
        'costo_variable_usd_km_nm3':float(GCORTO.loc[h,'costo_variable_usd_nm3_km']),
        'costo_variable_total_usd':(
            float(GCORTO.loc[h,'costo_variable_usd_nm3_km'])
            *float(GCORTO.loc[h,'distancia_recta_km'])
            *(value('QBCP',h)+(value('QBCP_HUB',h) if h in H_EHUB_SET else 0.0))
        ),
        'costo_total_gasoducto_usd':(
            float(GCORTO.loc[h,'costo_fijo_usd_dia'])*round(value('BCP',h))
            +float(GCORTO.loc[h,'costo_variable_usd_nm3_km'])
             *float(GCORTO.loc[h,'distancia_recta_km'])*(value('QBCP',h)+(value('QBCP_HUB',h) if h in H_EHUB_SET else 0.0))
        ),
        'geometria':'Línea recta calculada en el notebook',
    }
    for h in H_CORTO
])

df_gasoductos_biogas_hub_empresa_solo_resultado=pd.DataFrame([
    {
        'gasoducto_id':h,'gasoducto':str(EHUB.loc[h,'nombre']),'tipo_producto':'Biogás crudo',
        'origen_tipo':'UBICACION_EMPRESA_HUB','empresa_hub_id':str(EHUB.loc[h,'empresa_id']),
        'nodo_origen_id':str(EHUB.loc[h,'nodo_origen_id']),
        'destino_tipo':str(EHUB.loc[h,'destino_tipo']),'destino_id':str(EHUB.loc[h,'destino_id']),
        'destino':str(EHUB.loc[h,'destino_nombre']),'nodo_destino_id':str(EHUB.loc[h,'nodo_destino_id']),
        'activado':round(value('BCP_HUB',h)),'flujo_total_nm3':value('QBCP_HUB',h),
        'capacidad_efectiva_nm3':float(EHUB.loc[h,'capacidad_max_nm3_dia']),
        'distancia_recta_km':float(EHUB.loc[h,'distancia_recta_km']),
        'costo_fijo_usd_dia':float(EHUB.loc[h,'costo_fijo_usd_dia']),
        'costo_variable_usd_nm3_km':float(EHUB.loc[h,'costo_variable_usd_nm3_km']),
        'costo_variable_total_usd':float(EHUB.loc[h,'costo_variable_usd_nm3_km'])*float(EHUB.loc[h,'distancia_recta_km'])*value('QBCP_HUB',h),
        'costo_total_gasoducto_usd':float(EHUB.loc[h,'costo_fijo_usd_dia'])*round(value('BCP_HUB',h))+float(EHUB.loc[h,'costo_variable_usd_nm3_km'])*float(EHUB.loc[h,'distancia_recta_km'])*value('QBCP_HUB',h),
        'geometria':'Línea recta calculada en el notebook',
        'proceso_posterior':'Biogás crudo en tránsito; puede procesarse en BG o continuar por otro HUB permitido',
    } for h in H_EHUB_SOLO
])

df_gasoductos_biogas_cortos_cliente_cliente_resultado=pd.DataFrame([
    {
        'gasoducto_id':h,'gasoducto':str(BIOC2C.loc[h,'nombre']),'tipo_producto':'Biogás crudo',
        'origen_tipo':'CLIENTE_HUB','cliente_origen_id':str(BIOC2C.loc[h,'cliente_origen_id']),
        'cliente_origen':str(BIOC2C.loc[h,'cliente_origen']),'nodo_origen_id':str(BIOC2C.loc[h,'nodo_origen_id']),
        'destino_tipo':'CLIENTE','cliente_destino_id':str(BIOC2C.loc[h,'cliente_destino_id']),
        'cliente_destino':str(BIOC2C.loc[h,'cliente_destino']),'nodo_destino_id':str(BIOC2C.loc[h,'nodo_destino_id']),
        'activado':round(value('BCP2C',h)),'flujo_total_nm3':value('QBC2C',h),
        'capacidad_efectiva_nm3':float(BIOC2C.loc[h,'capacidad_max_nm3_dia']),
        'distancia_recta_km':float(BIOC2C.loc[h,'distancia_recta_km']),
        'costo_fijo_usd_dia':float(BIOC2C.loc[h,'costo_fijo_usd_dia']),
        'costo_variable_usd_nm3_km':float(BIOC2C.loc[h,'costo_variable_usd_nm3_km']),
        'punto_a_lat':float(BIOC2C.loc[h,'punto_a_lat']),'punto_a_lon':float(BIOC2C.loc[h,'punto_a_lon']),
        'punto_b_lat':float(BIOC2C.loc[h,'punto_b_lat']),'punto_b_lon':float(BIOC2C.loc[h,'punto_b_lon']),
        'costo_variable_total_usd':float(BIOC2C.loc[h,'costo_variable_usd_nm3_km'])*float(BIOC2C.loc[h,'distancia_recta_km'])*value('QBC2C',h),
        'costo_total_gasoducto_usd':float(BIOC2C.loc[h,'costo_fijo_usd_dia'])*round(value('BCP2C',h))+float(BIOC2C.loc[h,'costo_variable_usd_nm3_km'])*float(BIOC2C.loc[h,'distancia_recta_km'])*value('QBC2C',h),
        'geometria':'Línea recta calculada en el notebook',
        'proceso_posterior':'Puede volver a reenviarse como biogás crudo o procesarse en BG en el cliente de destino',
    } for h in H_BIOC2C
])


df_gasoductos_biogas_cortos_hub_p_resultado=pd.DataFrame([
    {
        'gasoducto_corto_id':h,'gasoducto':str(PHUB.loc[h,'nombre']),'tipo_producto':'Biogás crudo',
        'origen_tipo':str(PHUB.loc[h,'origen_tipo']),'origen_id':str(PHUB.loc[h,'origen_id']),
        'origen':str(PHUB.loc[h,'origen_nombre']),'nodo_origen_id':str(PHUB.loc[h,'nodo_origen_id']),
        'destino_tipo':str(PHUB.loc[h,'destino_tipo']),'destino_id':str(PHUB.loc[h,'destino_id']),
        'destino':str(PHUB.loc[h,'destino_nombre']),'nodo_destino_id':str(PHUB.loc[h,'nodo_destino_id']),
        'activado':round(value('BCP_P',h)),'flujo_total_nm3':value('QBCP_P',h),
        'capacidad_efectiva_nm3':float(PHUB.loc[h,'capacidad_max_nm3_dia']),
        'distancia_recta_km':float(PHUB.loc[h,'distancia_recta_km']),
        'costo_fijo_usd_dia':float(PHUB.loc[h,'costo_fijo_usd_dia']),
        'costo_variable_usd_nm3_km':float(PHUB.loc[h,'costo_variable_usd_nm3_km']),
        'punto_a_lat':float(PHUB.loc[h,'punto_a_lat']),'punto_a_lon':float(PHUB.loc[h,'punto_a_lon']),
        'punto_b_lat':float(PHUB.loc[h,'punto_b_lat']),'punto_b_lon':float(PHUB.loc[h,'punto_b_lon']),
        'costo_variable_total_usd':float(PHUB.loc[h,'costo_variable_usd_nm3_km'])*float(PHUB.loc[h,'distancia_recta_km'])*value('QBCP_P',h),
        'costo_total_gasoducto_usd':float(PHUB.loc[h,'costo_fijo_usd_dia'])*round(value('BCP_P',h))+float(PHUB.loc[h,'costo_variable_usd_nm3_km'])*float(PHUB.loc[h,'distancia_recta_km'])*value('QBCP_P',h),
        'geometria':'Línea recta calculada en el notebook',
        'proceso_posterior':'Biogás crudo en tránsito; puede seguir crudo por otro gasoducto corto o procesarse en BG en el nodo destino',
    } for h in H_PHUB
])

df_gasoductos_gasbc_cortos_empresa_entrada_resultado=pd.DataFrame([
    {
        'gasoducto_corto_id':h,
        'gasoducto_id':str(GECORTO.loc[h,'gasoducto_id']),
        'direccion_id':str(GECORTO.loc[h,'direccion_id']),
        'empresa_id':str(GECORTO.loc[h,'empresa_id']),
        'empresa':str(GECORTO.loc[h,'empresa']),
        'nodo_origen_id':str(GECORTO.loc[h,'nodo_origen_id']),
        'tipo_producto':'GasBC',
        'proceso_origen':'BG en la misma empresa -> GasBC -> entrada del corredor',
        'activado':round(value('BGCE',h)),
        'flujo_total_nm3':value('QGCE',h),
        'capacidad_efectiva_nm3':float(GECORTO.loc[h,'capacidad_max_nm3_dia']),
        'distancia_recta_km':float(GECORTO.loc[h,'distancia_recta_km']),
        'costo_fijo_usd_dia':float(GECORTO.loc[h,'costo_fijo_usd_dia']),
        'costo_variable_usd_nm3_km':float(GECORTO.loc[h,'costo_variable_usd_nm3_km']),
        'punto_a_lat':float(GECORTO.loc[h,'punto_a_lat']),
        'punto_a_lon':float(GECORTO.loc[h,'punto_a_lon']),
        'punto_b_lat':float(GECORTO.loc[h,'punto_b_lat']),
        'punto_b_lon':float(GECORTO.loc[h,'punto_b_lon']),
        'costo_variable_total_usd':(
            float(GECORTO.loc[h,'costo_variable_usd_nm3_km'])
            *float(GECORTO.loc[h,'distancia_recta_km'])*value('QGCE',h)
        ),
        'costo_total_gasoducto_usd':(
            float(GECORTO.loc[h,'costo_fijo_usd_dia'])*round(value('BGCE',h))
            +float(GECORTO.loc[h,'costo_variable_usd_nm3_km'])
             *float(GECORTO.loc[h,'distancia_recta_km'])*value('QGCE',h)
        ),
        'geometria':'Línea recta calculada en el notebook',
    } for h in H_GECORTO
])

df_gasoductos_gasbc_cortos_salida_cliente_resultado=pd.DataFrame([
    {
        'gasoducto_corto_id':h,
        'gasoducto_id':str(GSCORTO.loc[h,'gasoducto_id']),
        'direccion_id':str(GSCORTO.loc[h,'direccion_id']),
        'destino_tipo':str(GSCORTO.loc[h,'destino_tipo']),
        'destino_id':str(GSCORTO.loc[h,'destino_id']),
        'destino':str(GSCORTO.loc[h,'destino_nombre']),
        'nodo_destino_id':str(GSCORTO.loc[h,'nodo_destino_id']),
        'cliente_id':None if pd.isna(GSCORTO.loc[h,'cliente_id']) else str(GSCORTO.loc[h,'cliente_id']),
        'cliente':None if pd.isna(GSCORTO.loc[h,'cliente']) else str(GSCORTO.loc[h,'cliente']),
        'empresa_id':None if pd.isna(GSCORTO.loc[h,'empresa_id']) else str(GSCORTO.loc[h,'empresa_id']),
        'empresa':None if pd.isna(GSCORTO.loc[h,'empresa']) else str(GSCORTO.loc[h,'empresa']),
        'activado':round(value('BGCP',h)),
        'flujo_total_nm3':value('QGCP',h)+value('QGCP_HUB',h),
        'flujo_venta_final_nm3':value('QGCP',h),
        'flujo_reenvio_hub_nm3':value('QGCP_HUB',h),
        'capacidad_efectiva_nm3':float(GSCORTO.loc[h,'capacidad_max_nm3_dia']),
        'distancia_recta_km':float(GSCORTO.loc[h,'distancia_recta_km']),
        'costo_fijo_usd_dia':float(GSCORTO.loc[h,'costo_fijo_usd_dia']),
        'costo_variable_usd_nm3_km':float(GSCORTO.loc[h,'costo_variable_usd_nm3_km']),
        'punto_a_lat':float(GSCORTO.loc[h,'punto_a_lat']),
        'punto_a_lon':float(GSCORTO.loc[h,'punto_a_lon']),
        'punto_b_lat':float(GSCORTO.loc[h,'punto_b_lat']),
        'punto_b_lon':float(GSCORTO.loc[h,'punto_b_lon']),
        'costo_variable_total_usd':(
            float(GSCORTO.loc[h,'costo_variable_usd_nm3_km'])
            *float(GSCORTO.loc[h,'distancia_recta_km'])*(value('QGCP',h)+value('QGCP_HUB',h))
        ),
        'geometria':'Línea recta calculada en el notebook',
    } for h in H_GSCORTO
])


df_gasoductos_gasbc_cortos_empresa_destino_resultado=pd.DataFrame([
    {
        'gasoducto_corto_id':h,
        'origen_tipo':str(GDCORTO.loc[h,'origen_tipo']),
        'origen_id':str(GDCORTO.loc[h,'origen_id']),
        'origen':str(GDCORTO.loc[h,'origen_nombre']),
        'empresa_id':None if pd.isna(GDCORTO.loc[h,'empresa_id']) else str(GDCORTO.loc[h,'empresa_id']),
        'empresa':None if pd.isna(GDCORTO.loc[h,'empresa']) else str(GDCORTO.loc[h,'empresa']),
        'nodo_origen_id':str(GDCORTO.loc[h,'nodo_origen_id']),
        'destino_tipo':str(GDCORTO.loc[h,'destino_tipo']),
        'destino_id':str(GDCORTO.loc[h,'destino_id']),
        'destino':str(GDCORTO.loc[h,'destino_nombre']),
        'nodo_destino_id':str(GDCORTO.loc[h,'nodo_destino_id']),
        'cliente_id':None if pd.isna(GDCORTO.loc[h,'cliente_id']) else str(GDCORTO.loc[h,'cliente_id']),
        'ubicacion_id':None if pd.isna(GDCORTO.loc[h,'ubicacion_id']) else str(GDCORTO.loc[h,'ubicacion_id']),
        'tipo_producto':'GasBC',
        'proceso_origen':'GasBC desde BG o reconversión RL/RC en nodo de origen -> gasoducto corto -> destino',
        'usa_gasoducto_convencional':False,
        'activado':round(value('BGCD',h)),
        'flujo_total_nm3':(value('QGCD',h)+value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h)),
        'flujo_venta_final_nm3':(
            value('QGCD',h) if str(GDCORTO.loc[h,'destino_tipo'])=='CLIENTE' else 0.0
        ),
        'flujo_reenvio_hub_nm3':(
            value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h)
            if str(GDCORTO.loc[h,'destino_tipo'])=='CLIENTE'
            else value('QGCD',h)
        ),
        'flujo_reenvio_hub_desde_pool_BG_o_transito_nm3':value('QGCD_HUB',h),
        'flujo_reenvio_hub_desde_RL_nm3':value('QGCD_HUB_RL',h),
        'flujo_reenvio_hub_desde_RC_nm3':value('QGCD_HUB_RC',h),
        'flujo_desde_BG_nm3':value('QGCD_BG',h),
        'flujo_desde_RL_BioGNL_nm3':value('QGCD_RL',h),
        'flujo_desde_RC_BioGNC_nm3':value('QGCD_RC',h),
        'capacidad_efectiva_nm3':float(GDCORTO.loc[h,'capacidad_max_nm3_dia']),
        'distancia_recta_km':float(GDCORTO.loc[h,'distancia_recta_km']),
        'costo_fijo_usd_dia':float(GDCORTO.loc[h,'costo_fijo_usd_dia']),
        'costo_variable_usd_nm3_km':float(GDCORTO.loc[h,'costo_variable_usd_nm3_km']),
        'punto_a_lat':float(GDCORTO.loc[h,'punto_a_lat']),
        'punto_a_lon':float(GDCORTO.loc[h,'punto_a_lon']),
        'punto_b_lat':float(GDCORTO.loc[h,'punto_b_lat']),
        'punto_b_lon':float(GDCORTO.loc[h,'punto_b_lon']),
        'costo_variable_total_usd':(
            float(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])
            *float(GDCORTO.loc[h,'distancia_recta_km'])*(value('QGCD',h)+value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h))
        ),
        'costo_total_gasoducto_usd':(
            float(GDCORTO.loc[h,'costo_fijo_usd_dia'])*round(value('BGCD',h))
            +float(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])
             *float(GDCORTO.loc[h,'distancia_recta_km'])*(value('QGCD',h)+value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h))
        ),
        'geometria':'Línea recta calculada en el notebook',
    } for h in H_GDCORTO
])

# Descomposición dispersa posterior al MILP.
# Los parámetros NUMERO_EMPRESAS_A_UTILIZAR y NUMERO_CLIENTES_A_UTILIZAR sólo
# controlan cardinalidades TOTALES y globales. Las restricciones opcionales
# compra_porcentaje y venta_porcentaje fijan proporciones comunes únicamente
# entre participantes activados. Después de BG el producto es homogéneo y el MILP agregado no
# conserva identidad empresarial aguas abajo. Para la trazabilidad del reporte,
# se construyen primero las rutas físicas agregadas y luego se asigna a cada una
# el suministro empresarial mediante una regla dispersa de "mejor ajuste".
#
# Esta regla:
# - conserva exactamente el suministro total de cada empresa en cada sitio BG;
# - conserva exactamente la cantidad entregada por cada ruta y cliente;
# - evita el antiguo reparto proporcional que hacía aparecer a todas las
#   empresas del sitio BG en todos sus clientes;
# - no impone ninguna cantidad de empresas por cliente.
def positive(v):
    v=float(v)
    return v if v>TOL_FLUJO_CERO_NM3 else 0.0

def add_base_route(base_routes, rec):
    q=positive(rec.get('cantidad_nm3',0.0))
    raw=positive(rec.get('biogas_crudo_entrada_BG_nm3',0.0))
    if q<=TOL_FLUJO_CERO_NM3 or raw<=TOL_FLUJO_CERO_NM3:
        return
    rec=dict(rec)
    rec['cantidad_nm3']=q
    rec['gasbc_final_entregado_nm3']=q
    rec['biogas_crudo_entrada_BG_nm3']=raw
    base_routes.append(rec)

base_routes=[]
KBG=float(K['BG']); KL=float(K['L']); KRL=float(K['RL']); KC=float(K['C']); KRC=float(K['RC'])

# 1) GasBC directo: BG -> cliente.
for bg in S:
    for j in sorted(ABC_by_s[bg]):
        q=positive(value('ZD',bg,j))
        add_base_route(base_routes,{
            'bg_site':bg,'familia':'GasBC','cliente_id':j,'final_site':bg,
            'biogas_crudo_entrada_BG_nm3':q/KBG,
            'gasbc_salida_BG_nm3':q,
            'cantidad_nm3':q,
        })

# 1B) GasBC por gasoducto corto desde la componente BG -> cliente.
# Las componentes procedentes de RL/RC se reconstruyen en las familias BioGNL
# y BioGNC para conservar toda su trazabilidad de conversión y del vehículo.
for h in H_GDCORTO_CLIENTE:
    q=positive(value('QGCD_BG',h))
    bg=str(GDCORTO.loc[h,'nodo_origen_id'])
    j=str(GDCORTO.loc[h,'cliente_id'])
    add_base_route(base_routes,{
        'bg_site':bg,'bg_production_site':bg,'familia':'GasBC','cliente_id':j,'final_site':bg,
        'biogas_crudo_entrada_BG_nm3':q/KBG,
        'gasbc_salida_BG_nm3':q,'cantidad_nm3':q,
        'gasoducto_gasbc_directo_id':str(h),
        'gasoducto_gasbc_directo_destino_tipo':'CLIENTE',
        'gasoducto_gasbc_directo_destino_id':j,
        'usa_gasoducto_gasbc_directo':True,
        'fuente_gasbc_gasoducto_corto':'BG',
    })

# Fuentes de GasBC que alimentan una dirección del corredor. Se reutiliza tanto
# para las entregas directas como para las nuevas conversiones L/C en la salida.
def _gasbc_fuentes_bg_de_LC(tipo, sitio_conversion):
    """Fuentes BG que alimentan L/C; se usa para trazar RL/RC -> corredor."""
    tipo=str(tipo); sitio_conversion=str(sitio_conversion)
    fuentes=[]
    if tipo=='L':
        for bg in sorted(GBL_by_q[sitio_conversion]):
            q=positive(value('GBL',bg,sitio_conversion))
            if q>TOL_FLUJO_CERO_NM3:
                fuentes.append({'bg':str(bg),'q':q})
    elif tipo=='C':
        for bg in sorted(GBC_by_q[sitio_conversion]):
            q=positive(value('GBC',bg,sitio_conversion))
            if q>TOL_FLUJO_CERO_NM3:
                fuentes.append({'bg':str(bg),'q':q})
    else:
        raise KeyError(tipo)
    return fuentes


def _expandir_fuente_reconversion_corredor(tipo, sitio_reconversion, q_pipe):
    """Descompone YG_RL/YG_RC según L/C y BG aguas arriba, sólo para reporte."""
    tipo=str(tipo); sitio_reconversion=str(sitio_reconversion); q_pipe=positive(q_pipe)
    if q_pipe<=TOL_FLUJO_CERO_NM3:
        return []
    if tipo=='GNL':
        conv_code='L'; k_conv=KL; k_reconv=KRL
        entradas=[]
        q0=positive(value('YL0',sitio_reconversion))
        if q0>TOL_FLUJO_CERO_NM3:
            entradas.append((sitio_reconversion,q0))
        for sitio_conv in sorted(ASITE_in[sitio_reconversion]):
            q=positive(value('YL',sitio_conv,sitio_reconversion))
            if q>TOL_FLUJO_CERO_NM3:
                entradas.append((str(sitio_conv),q))
        modo='RECONVERSION_GNL'
    elif tipo=='GNC':
        conv_code='C'; k_conv=KC; k_reconv=KRC
        entradas=[]
        q0=positive(value('YC0',sitio_reconversion))
        if q0>TOL_FLUJO_CERO_NM3:
            entradas.append((sitio_reconversion,q0))
        for sitio_conv in sorted(ASITE_in[sitio_reconversion]):
            q=positive(value('YC',sitio_conv,sitio_reconversion))
            if q>TOL_FLUJO_CERO_NM3:
                entradas.append((str(sitio_conv),q))
        modo='RECONVERSION_GNC'
    else:
        raise KeyError(tipo)

    total_producto=sum(q for _,q in entradas)
    if total_producto<=TOL_FLUJO_CERO_NM3:
        raise AssertionError((tipo,sitio_reconversion,'inyeccion_sin_entrada_reconversion',q_pipe))

    fuentes=[]
    for sitio_conv,q_producto in entradas:
        q_pipe_conv=q_pipe*q_producto/total_producto
        bg_sources=_gasbc_fuentes_bg_de_LC(conv_code,sitio_conv)
        total_bg=sum(src['q'] for src in bg_sources)
        if total_bg<=TOL_FLUJO_CERO_NM3:
            raise AssertionError((tipo,sitio_reconversion,sitio_conv,'conversion_sin_fuente_BG',q_pipe_conv))
        for src in bg_sources:
            q_part=q_pipe_conv*src['q']/total_bg
            fuentes.append({
                'bg':str(src['bg']),
                'q':q_part,
                'modo_entrada':modo,
                'raw_per_pipe_gasbc':1.0/(KBG*k_conv*k_reconv),
                'gasoducto_corto_entrada_gasbc_id':None,
                'pre_conversion_tipo':tipo,
                'pre_conversion_site':str(sitio_conv),
                'pre_reconversion_site':str(sitio_reconversion),
                'pre_gasbc_entrada_conversion_por_pipe':1.0/(k_conv*k_reconv),
                'pre_producto_entrada_reconversion_por_pipe':1.0/k_reconv,
            })
    total_expandido=sum(x['q'] for x in fuentes)
    tol=max(1e-5,1e-8*max(1.0,q_pipe,total_expandido))
    return fuentes


def _fuentes_entrada_corredor(d):
    """Fuentes físicas de GasBC que alimentan la dirección d, incluida RL/RC local."""
    d=str(d); fuentes=[]
    for sitio in sorted(AIG_by_d[d]):
        q_base=positive(value('YG_BASE',sitio,d))
        if q_base>TOL_FLUJO_CERO_NM3:
            fuentes.append({
                'bg':str(sitio),'q':q_base,'modo_entrada':'AIG',
                'raw_per_pipe_gasbc':1.0/KBG,
                'gasoducto_corto_entrada_gasbc_id':None,
                'gasoducto_inyeccion_id':str(DG[d]),
                'direccion_inyeccion_id':d,
            })
        q_rl=positive(value('YG_RL',sitio,d))
        if q_rl>TOL_FLUJO_CERO_NM3:
            for _src in _expandir_fuente_reconversion_corredor('GNL',sitio,q_rl):
                _src['gasoducto_inyeccion_id']=str(DG[d]); _src['direccion_inyeccion_id']=d
                fuentes.append(_src)
        q_rc=positive(value('YG_RC',sitio,d))
        if q_rc>TOL_FLUJO_CERO_NM3:
            for _src in _expandir_fuente_reconversion_corredor('GNC',sitio,q_rc):
                _src['gasoducto_inyeccion_id']=str(DG[d]); _src['direccion_inyeccion_id']=d
                fuentes.append(_src)
    for h_entrada in sorted(H_GECORTO_by_d[d]):
        q_in=positive(value('QGCE',h_entrada))
        if q_in>TOL_FLUJO_CERO_NM3:
            fuentes.append({
                'bg':str(GECORTO.loc[h_entrada,'nodo_origen_id']),
                'q':q_in,'modo_entrada':'GASODUCTO_CORTO_GASBC',
                'raw_per_pipe_gasbc':1.0/KBG,
                'gasoducto_corto_entrada_gasbc_id':str(h_entrada),
                'gasoducto_inyeccion_id':str(DG[d]),
                'direccion_inyeccion_id':d,
            })
    return fuentes


def _clean_optional_id(value_obj):
    if value_obj is None:
        return ''
    if isinstance(value_obj, (list, tuple, set)):
        for item in value_obj:
            cleaned = _clean_optional_id(item)
            if cleaned:
                return cleaned
        return ''
    try:
        if pd.isna(value_obj):
            return ''
    except Exception:
        pass
    value_str = str(value_obj).strip()
    return '' if value_str.lower() in {'', 'none', 'nan', 'nat', 'null'} else value_str


def _campos_fuente_entrada_corredor(src, q_pipe):
    """Campos de trazabilidad de la cadena previa a la entrada del corredor."""
    q_pipe=float(q_pipe)
    raw=q_pipe*float(src.get('raw_per_pipe_gasbc',1.0/KBG))
    out={
        'bg_site':str(src['bg']),
        'biogas_crudo_entrada_BG_nm3':raw,
        'gasbc_salida_BG_nm3':raw*KBG,
        'gasbc_entrada_corredor_nm3':q_pipe,
        'entrada_corredor_modo':str(src.get('modo_entrada',src.get('entrada_corredor_modo','AIG'))),
        'gasoducto_corto_entrada_gasbc_id':src.get('gasoducto_corto_entrada_gasbc_id'),
        'gasoducto_inyeccion_id':src.get('gasoducto_inyeccion_id'),
        'direccion_inyeccion_id':src.get('direccion_inyeccion_id'),
        'red_gasoductos_ids':list(src.get('red_gasoductos_ids') or ([src.get('gasoducto_inyeccion_id')] if src.get('gasoducto_inyeccion_id') else [])),
        'red_gasoductos_arcos':list(src.get('red_gasoductos_arcos') or []),
        'red_gasoductos_distancia_km':float(src.get('red_gasoductos_distancia_km',0.0) or 0.0),
        'red_gasoductos_recorrido_interno_direccion':_clean_optional_id(src.get('red_gasoductos_recorrido_interno_direccion')) or None,
    }
    tipo=src.get('pre_conversion_tipo')
    if tipo=='GNL':
        out.update({
            'pre_l_site':str(src['pre_conversion_site']),
            'pre_rl_site':str(src['pre_reconversion_site']),
            'gasbc_entrada_L_pre_nm3':q_pipe*float(src['pre_gasbc_entrada_conversion_por_pipe']),
            'biognl_salida_L_pre_nm3':q_pipe*float(src['pre_producto_entrada_reconversion_por_pipe']),
            'biognl_entrada_RL_pre_nm3':q_pipe*float(src['pre_producto_entrada_reconversion_por_pipe']),
            'gasbc_salida_RL_pre_nm3':q_pipe,
        })
    elif tipo=='GNC':
        out.update({
            'pre_c_site':str(src['pre_conversion_site']),
            'pre_rc_site':str(src['pre_reconversion_site']),
            'gasbc_entrada_C_pre_nm3':q_pipe*float(src['pre_gasbc_entrada_conversion_por_pipe']),
            'biognc_salida_C_pre_nm3':q_pipe*float(src['pre_producto_entrada_reconversion_por_pipe']),
            'biognc_entrada_RC_pre_nm3':q_pipe*float(src['pre_producto_entrada_reconversion_por_pipe']),
            'gasbc_salida_RC_pre_nm3':q_pipe,
        })
    return out


def _campos_fuente_reconversion_local_ngi(src, q_local):
    """Campos de trazabilidad para RL/RC@NGI -> L/C@NGI sin usar corredor."""
    q_local=float(q_local)
    out=_campos_fuente_entrada_corredor(src,q_local)
    modo=str(src.get('modo_entrada',out.get('entrada_corredor_modo','RECONVERSION')))
    out.update({
        'gasbc_transferencia_local_ngi_nm3':q_local,
        'entrada_corredor_modo':f'{modo}_LOCAL_NGI',
        'gasoducto_corto_entrada_gasbc_id':None,
        'gasoducto_inyeccion_id':None,
        'direccion_inyeccion_id':None,
        'red_gasoductos_ids':[],
        'red_gasoductos_arcos':[],
        'red_gasoductos_distancia_km':0.0,
        'red_gasoductos_recorrido_interno_direccion':None,
        'salida_corredor_modo':'TRANSFERENCIA_LOCAL_NGI',
        'conversion_en_salida_gasoducto':False,
        'transferencia_local_ngi':True,
        'nodo_transferencia_local_ngi_id':str(src.get('nodo_transferencia_local_ngi_id',src.get('pre_reconversion_site',''))),
    })
    out.pop('gasbc_entrada_corredor_nm3',None)
    return out


# ------------------------------------------------------------------
# Descomposición exacta del transporte DIRECTO entre los 9 gasoductos.
#
# La asignación se hace por PUNTO FÍSICO de entrada/salida. YGG[g1,g2] continúa
# siendo una única conexión directa, pero una fuente sólo puede usarla si entró
# por la referencia de g1 usada para esa distancia; al llegar, sólo se entrega a
# salidas que nacen en la referencia de g2 correspondiente.
# ------------------------------------------------------------------
def _demanda_extraccion_direccion(d):
    d=str(d)
    return (
        sum(positive(value('ZG',d,j))+positive(value('ZG_HUB',d,j)) for j in AGC_by_d[d])
        +sum(positive(value('QGCP',h))+positive(value('QGCP_HUB',h)) for h in H_GSCORTO_by_d[d])
        +positive(value('QG_L_DIRECT',d))
        +positive(value('QG_C_DIRECT',d))
    )


def _descomponer_red_gasoductos():
    tol=max(1e-6,float(TOL_FLUJO_CERO_NM3))
    tokens_by_entry_d=defaultdict(list)
    _tid=0
    for d in D:
        for src in _fuentes_entrada_corredor(d):
            q=positive(src.get('q',0.0))
            if q<=tol:
                continue
            _tid+=1
            tokens_by_entry_d[str(d)].append({
                'token_id':f'SRCGG{_tid}','q':q,'src':copy.deepcopy(src)
            })

    pair_res={(g1,g2):positive(value('YGG',g1,g2)) for g1,g2 in GG_DIR}
    intra_res={str(d):positive(value('YGINTRA',d)) for d in D}
    stay_res={str(d):positive(value('YGGS',d)) for d in D}

    # Auditoría exacta en cada punto de inyección.
    for d in D:
        d=str(d); g=str(DG[d])
        supplied=sum(t['q'] for t in tokens_by_entry_d[d])
        assigned=intra_res[d]+stay_res[d]+sum(pair_res[(g,h)] for h in GG_BY_ORIGIN_D[d])
        _tol=max(1e-4,1e-8*max(1.0,supplied,assigned))
        if abs(supplied-assigned)>_tol:
            raise AssertionError(('asignacion_directa_punto_origen_post',d,supplied,assigned))

    # Auditoría exacta en cada punto de extracción.
    for d_out in D:
        d_out=str(d_out); g=str(DG[d_out])
        demanded=positive(_demanda_extraccion_direccion(d_out))
        d_in=SAME_POINT_IN_D[d_out]
        received=intra_res[d_out]+stay_res[d_in]+sum(pair_res[(h,g)] for h in GG_BY_DEST_D[d_out])
        _tol=max(1e-4,1e-8*max(1.0,demanded,received))
        if abs(demanded-received)>_tol:
            raise AssertionError(('asignacion_directa_punto_destino_post',d_out,demanded,received))

    allocations=defaultdict(list)  # clave = dirección física de salida
    for d_in in D:
        d_in=str(d_in); g1=str(DG[d_in])
        tok_list=tokens_by_entry_d[d_in]
        for tok in tok_list:
            rem=float(tok['q'])
            while rem>tol:
                # Primero se asigna el recorrido longitudinal interno A/B si existe.
                if intra_res[d_in]>tol:
                    kind='INTRA'; g2=g1; d_out=d_in
                    available=intra_res[d_in]
                # Luego puede permanecer en el mismo punto fisico historico.
                elif stay_res[d_in]>tol:
                    kind='STAY'; g2=g1; d_out=SAME_POINT_OUT_D[d_in]
                    available=stay_res[d_in]
                else:
                    g2=next((h for h in GG_BY_ORIGIN_D[d_in] if pair_res[(g1,h)]>tol),None)
                    if g2 is None:
                        raise AssertionError(('sin_destino_directo_para_fuente',tok['token_id'],d_in,rem))
                    kind='PAIR'; d_out=GG_DEST_D[(g1,g2)]; available=pair_res[(g1,g2)]
                part=min(rem,available)
                rem-=part
                if kind=='INTRA':
                    intra_res[d_in]-=part
                elif kind=='STAY':
                    stay_res[d_in]-=part
                else:
                    pair_res[(g1,g2)]-=part

                src=copy.deepcopy(tok['src']); src['q']=part
                src['gasoducto_salida_red_id']=str(g2)
                src['direccion_salida_red_id']=str(d_out)
                if kind=='INTRA':
                    src['red_gasoductos_ids']=[str(g1)]
                    src['red_gasoductos_arcos']=[]
                    src['red_gasoductos_distancia_km']=float(DIST_INTRA_GASODUCTO_KM[str(g1)])
                    src['red_gasoductos_recorrido_interno_direccion']=str(d_in)
                elif kind=='STAY':
                    src['red_gasoductos_ids']=[str(g1)]
                    src['red_gasoductos_arcos']=[]
                    src['red_gasoductos_distancia_km']=0.0
                    src['red_gasoductos_recorrido_interno_direccion']=None
                else:
                    src['red_gasoductos_ids']=[str(g1),str(g2)]
                    src['red_gasoductos_arcos']=[(str(g1),str(g2))]
                    src['red_gasoductos_distancia_km']=float(DIST_GG[(str(g1),str(g2))])
                    src['red_gasoductos_recorrido_interno_direccion']=None
                allocations[str(d_out)].append(src)

    rem_pairs=sum(max(0.0,v) for v in pair_res.values())
    rem_intra=sum(max(0.0,v) for v in intra_res.values())
    rem_stay=sum(max(0.0,v) for v in stay_res.values())
    rem_total=rem_pairs+rem_intra+rem_stay
    _tol=max(1e-4,1e-8*max(1.0,rem_total))
    if rem_total>_tol:
        raise AssertionError(('transferencias_directas_no_descompuestas',rem_pairs,rem_intra,rem_stay))
    return allocations


FUENTES_SALIDA_RED_GASODUCTO=_descomponer_red_gasoductos()

def _fuentes_salida_red_gasoducto(d):
    # Se conserva el nombre de la función por compatibilidad, pero la clave es
    # ahora la DIRECCIÓN DE SALIDA física, no sólo el id del gasoducto.
    return [copy.deepcopy(x) for x in FUENTES_SALIDA_RED_GASODUCTO.get(str(d),[])]


def _fuentes_gasbc_para_conversion(tipo, sitio_conversion):
    """GasBC que entra a L/C desde BG o directamente desde una salida de corredor."""
    tipo=str(tipo); sitio_conversion=str(sitio_conversion)
    if tipo not in {'L','C'}:
        raise KeyError(tipo)
    fuentes=[]
    if tipo=='L':
        for bg in sorted(GBL_by_q[sitio_conversion]):
            q=positive(value('GBL',bg,sitio_conversion))
            if q>TOL_FLUJO_CERO_NM3:
                fuentes.append({'origen':'BG','bg':str(bg),'q':q})
        pref_direct='QG_L_DIRECT'
    else:
        for bg in sorted(GBC_by_q[sitio_conversion]):
            q=positive(value('GBC',bg,sitio_conversion))
            if q>TOL_FLUJO_CERO_NM3:
                fuentes.append({'origen':'BG','bg':str(bg),'q':q})
        pref_direct='QG_C_DIRECT'

    if sitio_conversion in set(globals().get('NGI_LOCAL_TRANSFER_SET', set())):
        _local_specs=(
            (('QRL_L_LOCAL','GNL'),('QRC_L_LOCAL','GNC'))
            if tipo=='L'
            else (('QRL_C_LOCAL','GNL'),('QRC_C_LOCAL','GNC'))
        )
        for pref_local,tipo_fuente in _local_specs:
            q_local=positive(value(pref_local,sitio_conversion))
            if q_local<=TOL_FLUJO_CERO_NM3:
                continue
            for _src in _expandir_fuente_reconversion_corredor(tipo_fuente,sitio_conversion,q_local):
                _src.update({
                    'origen':'RECONVERSION_LOCAL_NGI',
                    'local_transfer_var':pref_local,
                    'local_transfer_fuente_tipo':tipo_fuente,
                    'local_transfer_destino_conversion_tipo':tipo,
                    'nodo_transferencia_local_ngi_id':sitio_conversion,
                })
                fuentes.append(_src)

    # Si sitio_conversion es la salida física de una o más direcciones, repartir
    # la extracción directa entre las fuentes que realmente alimentaron ese punto.
    # La misma descomposición proporcional se usa para las otras salidas del
    # corredor, por lo que toda la trazabilidad conserva masa exactamente.
    for d in sorted(D_SALIDA_by_s[sitio_conversion]):
        q_direct=positive(value(pref_direct,d))
        if q_direct<=TOL_FLUJO_CERO_NM3:
            continue
        fuentes_pipe=_fuentes_salida_red_gasoducto(d)
        total_pipe=sum(positive(src.get('q',0.0)) for src in fuentes_pipe)
        if total_pipe<=TOL_FLUJO_CERO_NM3:
            raise AssertionError((tipo,sitio_conversion,d,'conversion_salida_gasoducto_sin_fuente',q_direct))
        for src0 in fuentes_pipe:
            qsrc=positive(src0.get('q',0.0))
            if qsrc<=TOL_FLUJO_CERO_NM3:
                continue
            part=q_direct*qsrc/total_pipe
            src=copy.deepcopy(src0)
            src.update({
                'origen':'GASODUCTO',
                'q':part,
                'gasoducto_id':str(DG[d]),
                'direccion_id':str(d),
                'direccion_salida_red_id':str(d),
                'nodo_conversion_salida_id':sitio_conversion,
                'entrada_corredor_modo':str(src0.get('modo_entrada','AIG')),
            })
            fuentes.append(src)
    return fuentes


# 2) BioGNL: BG -> L -> vehículo GNL -> RL -> (cliente local o gasoducto corto -> otro cliente).
def _salidas_post_reconversion(tipo, sitio):
    """Devuelve salidas finales de una RL/RC sin mezclar sus procedencias."""
    sitio=str(sitio)
    out=[]
    if tipo=='GNL':
        for j in sorted(ABC_by_s[sitio]):
            q=positive(value('ZL',sitio,j))
            if q>TOL_FLUJO_CERO_NM3:
                out.append({'cliente_id':str(j),'q':q,'h_directo':None})
        for h in sorted(H_GDCORTO_by_origen_s[sitio]):
            if str(GDCORTO.loc[h,'destino_tipo'])!='CLIENTE':
                continue
            q=positive(value('QGCD_RL',h))
            if q>TOL_FLUJO_CERO_NM3:
                out.append({
                    'cliente_id':str(GDCORTO.loc[h,'cliente_id']),
                    'q':q,'h_directo':str(h),
                })
    elif tipo=='GNC':
        for j in sorted(ABC_by_s[sitio]):
            q=positive(value('ZC',sitio,j))
            if q>TOL_FLUJO_CERO_NM3:
                out.append({'cliente_id':str(j),'q':q,'h_directo':None})
        for h in sorted(H_GDCORTO_by_origen_s[sitio]):
            if str(GDCORTO.loc[h,'destino_tipo'])!='CLIENTE':
                continue
            q=positive(value('QGCD_RC',h))
            if q>TOL_FLUJO_CERO_NM3:
                out.append({
                    'cliente_id':str(GDCORTO.loc[h,'cliente_id']),
                    'q':q,'h_directo':str(h),
                })
    else:
        raise KeyError(tipo)
    return out

for rl in S:
    source_inputs=[]
    local=positive(value('YL0',rl))
    if local>TOL_FLUJO_CERO_NM3:
        source_inputs.append((rl,local))
    for l in sorted(ASITE_in[rl]):
        q=positive(value('YL',l,rl))
        if q>TOL_FLUJO_CERO_NM3:
            source_inputs.append((l,q))
    total_input=sum(q for _,q in source_inputs)
    if total_input<=EPS:
        continue
    for salida in _salidas_post_reconversion('GNL',rl):
        j=str(salida['cliente_id'])
        q_final_total=positive(salida['q'])
        h_directo=salida['h_directo']
        for l,biognl_input_path in source_inputs:
            q_path=q_final_total*biognl_input_path/total_input
            gasbc_sources=_fuentes_gasbc_para_conversion('L',l)
            total_gasbc_l=sum(src['q'] for src in gasbc_sources)
            if total_gasbc_l<=EPS:
                continue
            tol_l=max(1e-5,1e-8*max(1.0,total_gasbc_l,positive(value('QL',l))))
            for src in gasbc_sources:
                q=q_path*src['q']/total_gasbc_l
                gasbc_l=q/(KL*KRL)
                biognl=gasbc_l*KL
                if src['origen']=='GASODUCTO':
                    rec=_campos_fuente_entrada_corredor(src,gasbc_l)
                elif src['origen']=='RECONVERSION_LOCAL_NGI':
                    rec=_campos_fuente_reconversion_local_ngi(src,gasbc_l)
                else:
                    raw=gasbc_l/KBG
                    rec={
                        'bg_site':src['bg'],
                        'biogas_crudo_entrada_BG_nm3':raw,
                        'gasbc_salida_BG_nm3':gasbc_l,
                    }
                rec.update({
                    'familia':'BioGNL','cliente_id':j,'final_site':rl,
                    'l_site':l,'rl_site':rl,
                    'gasbc_entrada_L_nm3':gasbc_l,
                    'biognl_salida_L_nm3':biognl,
                    'biognl_entrada_RL_nm3':biognl,
                    'gasbc_salida_RL_nm3':q,
                    'cantidad_nm3':q,
                })
                if h_directo is not None:
                    rec.update({
                        'gasoducto_gasbc_directo_id':str(h_directo),
                        'gasoducto_gasbc_directo_destino_tipo':'CLIENTE',
                        'gasoducto_gasbc_directo_destino_id':j,
                        'usa_gasoducto_gasbc_directo':True,
                        'post_reconversion_gasoducto_corto':True,
                        'fuente_gasbc_gasoducto_corto':'RL_BioGNL',
                    })
                if src['origen']=='GASODUCTO':
                    rec.update({
                        'gasoducto_id':src['gasoducto_id'],
                        'direccion_id':src['direccion_id'],
                        'entrada_corredor_modo':src['entrada_corredor_modo'],
                        'gasoducto_corto_entrada_gasbc_id':src.get('gasoducto_corto_entrada_gasbc_id'),
                        'salida_corredor_modo':'CONVERSION_GNL',
                        'conversion_en_salida_gasoducto':True,
                    })
                    rec.update({k:v for k,v in _campos_fuente_entrada_corredor(src,gasbc_l).items() if k.startswith('pre_') or k.endswith('_pre_nm3') or k=='gasbc_entrada_corredor_nm3'})
                add_base_route(base_routes,rec)

# 3) BioGNC: BG -> C -> vehículo GNC -> RC -> (cliente local o gasoducto corto -> otro cliente).
for rc in S:
    source_inputs=[]
    local=positive(value('YC0',rc))
    if local>TOL_FLUJO_CERO_NM3:
        source_inputs.append((rc,local))
    for c in sorted(ASITE_in[rc]):
        q=positive(value('YC',c,rc))
        if q>TOL_FLUJO_CERO_NM3:
            source_inputs.append((c,q))
    total_input=sum(q for _,q in source_inputs)
    if total_input<=EPS:
        continue
    for salida in _salidas_post_reconversion('GNC',rc):
        j=str(salida['cliente_id'])
        q_final_total=positive(salida['q'])
        h_directo=salida['h_directo']
        for c,biognc_input_path in source_inputs:
            q_path=q_final_total*biognc_input_path/total_input
            gasbc_sources=_fuentes_gasbc_para_conversion('C',c)
            total_gasbc_c=sum(src['q'] for src in gasbc_sources)
            if total_gasbc_c<=EPS:
                continue
            tol_c=max(1e-5,1e-8*max(1.0,total_gasbc_c,positive(value('QC',c))))
            for src in gasbc_sources:
                q=q_path*src['q']/total_gasbc_c
                gasbc_c=q/(KC*KRC)
                biognc=gasbc_c*KC
                if src['origen']=='GASODUCTO':
                    rec=_campos_fuente_entrada_corredor(src,gasbc_c)
                elif src['origen']=='RECONVERSION_LOCAL_NGI':
                    rec=_campos_fuente_reconversion_local_ngi(src,gasbc_c)
                else:
                    raw=gasbc_c/KBG
                    rec={
                        'bg_site':src['bg'],
                        'biogas_crudo_entrada_BG_nm3':raw,
                        'gasbc_salida_BG_nm3':gasbc_c,
                    }
                rec.update({
                    'familia':'BioGNC','cliente_id':j,'final_site':rc,
                    'c_site':c,'rc_site':rc,
                    'gasbc_entrada_C_nm3':gasbc_c,
                    'biognc_salida_C_nm3':biognc,
                    'biognc_entrada_RC_nm3':biognc,
                    'gasbc_salida_RC_nm3':q,
                    'cantidad_nm3':q,
                })
                if h_directo is not None:
                    rec.update({
                        'gasoducto_gasbc_directo_id':str(h_directo),
                        'gasoducto_gasbc_directo_destino_tipo':'CLIENTE',
                        'gasoducto_gasbc_directo_destino_id':j,
                        'usa_gasoducto_gasbc_directo':True,
                        'post_reconversion_gasoducto_corto':True,
                        'fuente_gasbc_gasoducto_corto':'RC_BioGNC',
                    })
                if src['origen']=='GASODUCTO':
                    rec.update({
                        'gasoducto_id':src['gasoducto_id'],
                        'direccion_id':src['direccion_id'],
                        'entrada_corredor_modo':src['entrada_corredor_modo'],
                        'gasoducto_corto_entrada_gasbc_id':src.get('gasoducto_corto_entrada_gasbc_id'),
                        'salida_corredor_modo':'CONVERSION_GNC',
                        'conversion_en_salida_gasoducto':True,
                    })
                    rec.update({k:v for k,v in _campos_fuente_entrada_corredor(src,gasbc_c).items() if k.startswith('pre_') or k.endswith('_pre_nm3') or k=='gasbc_entrada_corredor_nm3'})
                add_base_route(base_routes,rec)

# 4) Gasoducto: BG -> entrada de dirección -> corredor -> cliente.
# La entrada puede ser vial/local YG o un gasoducto corto de GasBC desde una BG
# situada en la misma empresa. Ambos modos alimentan el mismo balance direccional.
for d in D:
    source_inputs=_fuentes_salida_red_gasoducto(d)
    total_input=sum(src['q'] for src in source_inputs)
    if total_input<=EPS:
        continue
    for j in sorted(AGC_by_d[d]):
        q_final_total=positive(value('ZG',d,j))
        for src in source_inputs:
            q=q_final_total*src['q']/total_input
            rec=_campos_fuente_entrada_corredor(src,q)
            rec.update({
                'familia':'Gasoducto','cliente_id':j,'final_site':None,
                'gasoducto_id':DG[d],'direccion_id':d,
                'cantidad_nm3':q,
                'salida_corredor_modo':'CARRETERA',
            })
            add_base_route(base_routes,rec)
    for h in sorted(H_GSCORTO_by_d[d]):
        if str(GSCORTO.loc[h,'destino_tipo'])!='CLIENTE':
            continue
        j=str(GSCORTO.loc[h,'cliente_id'])
        q_final_total=positive(value('QGCP',h))
        for src in source_inputs:
            q=q_final_total*src['q']/total_input
            rec=_campos_fuente_entrada_corredor(src,q)
            rec.update({
                'familia':'Gasoducto','cliente_id':j,'final_site':None,
                'gasoducto_id':DG[d],'direccion_id':d,
                'cantidad_nm3':q,
                'salida_corredor_modo':'GASODUCTO_CORTO',
                'gasoducto_corto_salida_cliente_id':h,
                'usa_gasoducto_corto_salida_cliente':True,
            })
            add_base_route(base_routes,rec)

# Campos de cantidades que deben escalarse cuando una ruta agregada necesita ser
# dividida entre dos empresas por falta de capacidad suficiente en una sola.
ROUTE_FLOW_FIELDS={
    'biogas_crudo_entrada_BG_nm3','gasbc_salida_BG_nm3',
    'gasbc_entrada_L_nm3','biognl_salida_L_nm3','biognl_entrada_RL_nm3',
    'gasbc_salida_RL_nm3','gasbc_entrada_C_nm3','biognc_salida_C_nm3',
    'biognc_entrada_RC_nm3','gasbc_salida_RC_nm3','cantidad_nm3',
    'gasbc_entrada_corredor_nm3','gasbc_transferencia_local_ngi_nm3',
    'gasbc_entrada_L_pre_nm3','biognl_salida_L_pre_nm3','biognl_entrada_RL_pre_nm3','gasbc_salida_RL_pre_nm3',
    'gasbc_entrada_C_pre_nm3','biognc_salida_C_pre_nm3','biognc_entrada_RC_pre_nm3','gasbc_salida_RC_pre_nm3',
    'gasbc_final_entregado_nm3',
}

def split_route_for_source(rec, source_key, raw_chunk):
    """Divide una ruta conservando empresa, fuente inicial y relevos de biogás crudo."""
    modo=source_key[0]
    empresa_id=source_key[1]
    h=source_key[2] if len(source_key)>2 else None
    raw_source_site=(source_key[3] if len(source_key)>3 else None)
    raw_relay_path=(tuple(source_key[4]) if len(source_key)>4 and source_key[4] is not None else tuple())
    raw_total=positive(rec['biogas_crudo_entrada_BG_nm3'])
    if raw_total<=EPS or raw_chunk<=EPS:
        return None
    factor=float(raw_chunk)/raw_total
    out=dict(rec)
    out['empresa_id']=str(empresa_id)
    for field in ROUTE_FLOW_FIELDS:
        if field in out:
            out[field]=float(out[field])*factor
    out['biogas_crudo_entrada_BG_nm3']=float(raw_chunk)
    out['cantidad_nm3']=positive(out.get('cantidad_nm3',0.0))
    out['gasbc_final_entregado_nm3']=out['cantidad_nm3']
    # `raw_source_site` es el nodo al que llegó originalmente X[i,s] antes de
    # cualquier relevo de biogás crudo. `bg_site` sigue siendo el nodo donde BG
    # procesa físicamente el gas. Se guardan los relevos para costos y polilíneas.
    out['raw_source_site']=str(raw_source_site) if raw_source_site not in (None,'','None','nan') else str(out['bg_site'])
    out['raw_relay_path']=[f'{kind}|{hid}' for kind,hid in raw_relay_path]
    out['gasoductos_biogas_relevo_empresa_ids']=[str(hid) for kind,hid in raw_relay_path if kind=='EHUB']
    out['gasoductos_biogas_relevo_cliente_ids']=[str(hid) for kind,hid in raw_relay_path if kind=='C2C']
    out['gasoductos_biogas_relevo_P_ids']=[str(hid) for kind,hid in raw_relay_path if kind=='PHUB']

    if modo=='PIPE':
        h=str(h)
        destino_tipo=str(GCORTO.loc[h,'destino_tipo'])
        out['bg_production_site']=str(out['bg_site'])
        out['tipo_fuente_bg']='PIPE_BIOGAS'
        out['gasoducto_biogas_corto_id']=h
        out['usa_gasoducto_biogas_corto']=True
        out['gasoducto_destino_tipo']=destino_tipo
        out['gasoducto_destino_id']=str(GCORTO.loc[h,'destino_id'])
        out['gasoducto_nodo_destino_id']=str(GCORTO.loc[h,'nodo_destino_id'])
        out['tipo_transporte_biogas_crudo']=(
            'Gasoducto corto empresa-cliente'
            if destino_tipo=='CLIENTE'
            else (
                'Gasoducto corto empresa-entrada de gasoducto'
                if destino_tipo=='ENTRADA_GASODUCTO'
                else (
                    'Gasoducto corto empresa-ubicación de empresa'
                    if destino_tipo=='UBICACION_EMPRESA'
                    else 'Gasoducto corto empresa-ubicación posible'
                )
            )
        )
    elif modo=='ROAD':
        out['bg_production_site']=str(out['bg_site'])
        out['tipo_fuente_bg']='ROAD_OR_LOCAL'
        out['gasoducto_biogas_corto_id']=None
        out['usa_gasoducto_biogas_corto']=False
        out['gasoducto_destino_tipo']=None
        out['gasoducto_destino_id']=None
        out['gasoducto_nodo_destino_id']=None
        out['tipo_transporte_biogas_crudo']='Transporte vial o transferencia local existente'
    elif modo=='GASBC_DIRECT':
        h=str(h)
        out['bg_production_site']=str(GDCORTO.loc[h,'nodo_origen_id'])
        out['raw_source_site']=str(GDCORTO.loc[h,'nodo_origen_id'])
        out['tipo_fuente_bg']='GASBC_DIRECT'
        out['gasoducto_biogas_corto_id']=None
        out['usa_gasoducto_biogas_corto']=False
        out['gasoducto_gasbc_directo_id']=h
        out['usa_gasoducto_gasbc_directo']=True
        out['gasoducto_gasbc_directo_destino_tipo']=str(GDCORTO.loc[h,'destino_tipo'])
        out['gasoducto_gasbc_directo_destino_id']=str(GDCORTO.loc[h,'destino_id'])
        out['gasoducto_gasbc_directo_nodo_destino_id']=str(GDCORTO.loc[h,'nodo_destino_id'])
        out['tipo_transporte_biogas_crudo']='Procesamiento BG en nodo de origen; GasBC directo por gasoducto corto'
    else:
        raise ValueError(source_key)
    return out

# Asignación dispersa por sitio BG y por modo físico de llegada. Los flujos de
# gasoducto corto se asignan con prioridad determinista, sin restringir la
# familia logística posterior a la planta BG.
_hub_logistico_activo=(
    # Los relevos de BIOGÁS CRUDO QBCP_HUB/QBC2C/QBCP_P se descomponen exactamente
    # arriba y, por tanto, NO requieren colapsar la identidad a MIXTO_HUB.
    any((value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h))>TOL_FLUJO_CERO_NM3 for h in H_GDCORTO)
    # Todo GDCORTO cuyo destino NO es cliente es tránsito entre pools GasBC. Esto
    # incluye P#, ubicaciones de empresa y NGI y cubre fuentes BG, RL y RC mediante QGCD.
    or any(
        str(GDCORTO.loc[h,'destino_tipo'])!='CLIENTE'
        and value('QGCD',h)>TOL_FLUJO_CERO_NM3
        for h in H_GDCORTO
    )
    or any(value('ZG_HUB',d,j)>TOL_FLUJO_CERO_NM3 for d,j in AGC)
    or any(value('QGCP_HUB',h)>TOL_FLUJO_CERO_NM3 for h in H_GSCORTO)
)

routes=[]
base_by_bg=defaultdict(list)
for rec in base_routes:
    base_by_bg[str(rec['bg_site'])].append(rec)

# -------------------------------------------------------------------------
# Descomposición EXACTA del balance de biogás crudo cuando existe relevo HUB.
# QBCP_HUB y QBC2C son flujos de tránsito: no crean nueva producción. El código
# anterior no los incorporaba al inventario de fuentes de cada BG y por eso podía
# dejar, por ejemplo, X[E3,NE_E3] sin consumir mientras el mismo volumen reaparecía
# como necesidad en NC_C2. Aquí se descompone el flujo físico ya resuelto por el
# MILP en caminos fuente -> (relevos) -> BG, conservando la empresa originaria.
# -------------------------------------------------------------------------
def _decompose_raw_biogas_sources_to_bg():
    tol_net=max(1e-5,float(TOL_FLUJO_CERO_NM3))
    sink_remaining={str(s):positive(value('QBIO_BG',s)) for s in S}
    relay={}
    adjacency=defaultdict(list)

    for h in sorted(H_EHUB):
        q=positive(value('QBCP_HUB',h))
        if q<=tol_net:
            continue
        u=str(EHUB.loc[h,'nodo_origen_id']); v=str(EHUB.loc[h,'nodo_destino_id'])
        ek=('EHUB',str(h))
        relay[ek]={'u':u,'v':v,'q':q}
        adjacency[u].append(ek)

    for h in sorted(H_BIOC2C):
        q=positive(value('QBC2C',h))
        if q<=tol_net:
            continue
        u=str(BIOC2C.loc[h,'nodo_origen_id']); v=str(BIOC2C.loc[h,'nodo_destino_id'])
        ek=('C2C',str(h))
        relay[ek]={'u':u,'v':v,'q':q}
        adjacency[u].append(ek)
    for h in sorted(H_PHUB):
        q=positive(value('QBCP_P',h))
        if q<=tol_net:
            continue
        u=str(PHUB.loc[h,'nodo_origen_id']); v=str(PHUB.loc[h,'nodo_destino_id'])
        ek=('PHUB',str(h))
        relay[ek]={'u':u,'v':v,'q':q}
        adjacency[u].append(ek)

    # Fuentes donde la compra se paga realmente: carretera/local X y primer
    # gasoducto corto QBCP. Los relevos posteriores sólo transportan esa masa.
    sources=[]
    for i,s in sorted(ABIO):
        q=positive(value('X',i,s))
        if q>tol_net:
            sources.append((('ROAD',str(i),None,str(s),tuple()),str(s),q))
    for h in sorted(H_CORTO):
        q=positive(value('QBCP',h))
        if q>tol_net:
            s=str(GCORTO.loc[h,'nodo_destino_id'])
            i=str(GCORTO.loc[h,'empresa_id'])
            sources.append((('PIPE',i,str(h),s,tuple()),s,q))

    # Determinismo: primero fuentes por tubería y después carretera; dentro de
    # cada grupo se ordena por empresa/nodo. Esto no cambia la factibilidad, sólo
    # hace reproducible la atribución cuando dos fuentes se mezclan en un HUB.
    sources.sort(key=lambda z:(0 if z[0][0]=='PIPE' else 1,str(z[0][1]),str(z[1]),str(z[0][2])))
    processed=defaultdict(list)

    def _find_sink_path(start):
        start=str(start)
        queue=[start]; parent={start:None}; parent_edge={}
        head=0
        while head<len(queue):
            u=queue[head]; head+=1
            if sink_remaining.get(u,0.0)>tol_net:
                path=[]; cur=u
                while parent[cur] is not None:
                    path.append(parent_edge[cur]); cur=parent[cur]
                path.reverse()
                return u,path
            for ek in sorted(adjacency.get(u,[]),key=str):
                ed=relay[ek]
                if ed['q']<=tol_net:
                    continue
                v=str(ed['v'])
                if v in parent:
                    continue
                parent[v]=u; parent_edge[v]=ek; queue.append(v)
        return None,None

    for source_key,start,source_q in sources:
        left=float(source_q)
        guard=0
        while left>tol_net:
            guard+=1
            if guard>10000:
                raise RuntimeError(('descomposicion_biogas_hub_sin_convergencia',source_key,left))
            target,path=_find_sink_path(start)
            if target is None:
                raise AssertionError((
                    'biogas_hub_sin_camino_a_BG',source_key,start,left,
                    {k:v['q'] for k,v in relay.items() if v['q']>tol_net},
                    {k:v for k,v in sink_remaining.items() if v>tol_net},
                ))
            take=min(left,sink_remaining[target])
            for ek in path:
                take=min(take,relay[ek]['q'])
            if take<=tol_net:
                raise AssertionError(('biogas_hub_chunk_nulo',source_key,start,target,path,left))
            mode,empresa_id,h0,source_site,_=source_key
            traced_key=(mode,empresa_id,h0,source_site,tuple(path))
            processed[str(target)].append((traced_key,float(take)))
            left=max(0.0,left-take)
            sink_remaining[target]=max(0.0,sink_remaining[target]-take)
            for ek in path:
                relay[ek]['q']=max(0.0,relay[ek]['q']-take)

    residual_sinks={k:v for k,v in sink_remaining.items() if v>tol_net}
    residual_relays={k:v['q'] for k,v in relay.items() if v['q']>tol_net}
    cycle_residuals={}
    if not residual_sinks and residual_relays:
        node_balance=defaultdict(float)
        for ek,q in residual_relays.items():
            ed=relay[ek]
            node_balance[str(ed['u'])]-=float(q)
            node_balance[str(ed['v'])]+=float(q)
        residual_node_balance={
            k:v for k,v in node_balance.items()
            if abs(v)>max(tol_net,1e-8*max(1.0,abs(v)))
        }
        # Residual relays with zero node imbalance are closed circulations. They
        # do not feed any BG sink and therefore cannot be assigned to a source.
        if not residual_node_balance:
            cycle_residuals={k:float(v) for k,v in residual_relays.items()}
            for ek in cycle_residuals:
                relay[ek]['q']=0.0
            residual_relays={}
    if residual_sinks or residual_relays:
        residual_node_balance=defaultdict(float)
        for ek,q in residual_relays.items():
            ed=relay[ek]
            residual_node_balance[str(ed['u'])]-=float(q)
            residual_node_balance[str(ed['v'])]+=float(q)
        raise AssertionError((
            'descomposicion_biogas_hub_residual',
            residual_sinks,
            residual_relays,
            {k:v for k,v in residual_node_balance.items() if abs(v)>tol_net},
        ))
    return processed,cycle_residuals

raw_processed_by_bg,raw_biogas_relay_cycle_residuals=_decompose_raw_biogas_sources_to_bg()
_raw_biogas_relay_cycle_residuals_by_kind_h=defaultdict(float)
for _relay_key,_relay_q in raw_biogas_relay_cycle_residuals.items():
    _raw_biogas_relay_cycle_residuals_by_kind_h[(str(_relay_key[0]),str(_relay_key[1]))]+=float(_relay_q)
_raw_hub_biogas_activo=(
    any(value('QBCP_HUB',h)>TOL_FLUJO_CERO_NM3 for h in H_EHUB)
    or any(value('QBC2C',h)>TOL_FLUJO_CERO_NM3 for h in H_BIOC2C)
    or any(value('QBCP_P',h)>TOL_FLUJO_CERO_NM3 for h in H_PHUB)
)

# -------------------------------------------------------------------------
# Descomposición EXACTA del GasBC que atraviesa HUBs.
#
# El balance BG del MILP permite que GasBC producido aguas arriba llegue a otro
# nodo mediante QGCD/QGCD_HUB, una reconversión RL/RC seguida de QGCD_HUB_RL/RC hacia cliente o QGCD_RL/RC hacia P#/NGI,
# o una dirección de gasoducto convencional seguida de ZG_HUB/QGCP_HUB. El
# postproceso antiguo sólo comparaba la producción BG local con las rutas finales
# del nodo; por eso aparecían diferencias como P11 = 767.6717 Nm3 y se intentaba
# ocultarlas con MIXTO_HUB. Aquí cada flujo HUB se convierte en un arco de
# transferencia GasBC nodo->nodo y se propaga la procedencia empresarial real.
# -------------------------------------------------------------------------

def _usage_add(usage,key,input_q,output_q):
    key=str(key); input_q=float(input_q); output_q=float(output_q)
    rec=usage.setdefault(key,{'input':0.0,'output':0.0})
    rec['input']+=input_q; rec['output']+=output_q


def _hub_path_record(path_id,origin,dest,input_q,output_q,*,producer_usage=None,
                     transport_usage=None,segment_keys=None,pipe_distance_km=0.0,event=None):
    input_q=positive(input_q); output_q=positive(output_q)
    if input_q<=TOL_FLUJO_CERO_NM3 and output_q<=TOL_FLUJO_CERO_NM3:
        return None
    if input_q<=TOL_FLUJO_CERO_NM3 or output_q<=TOL_FLUJO_CERO_NM3:
        raise AssertionError(('hub_path_cantidad_invalida',path_id,input_q,output_q))
    return {
        'path_id':str(path_id),'origin':str(origin),'dest':str(dest),
        'input_q':float(input_q),'output_q':float(output_q),
        'ratio':float(output_q)/float(input_q),
        'producer_usage':copy.deepcopy(producer_usage or {}),
        'transport_usage':copy.deepcopy(transport_usage or []),
        'segment_keys':list(dict.fromkeys(map(str,segment_keys or []))),
        'pipe_distance_km':float(pipe_distance_km),
        'event':copy.deepcopy(event or {'tipo':'HUB_GASBC','path_id':str(path_id)}),
    }


def _build_hub_gasbc_paths():
    paths=[]
    def add(*args,**kwargs):
        p=_hub_path_record(*args,**kwargs)
        if p is not None: paths.append(p)

    # A) GasBC directo entre nodos. Hacia ubicación posible QGCD_BG es tránsito;
    # hacia cliente QGCD_HUB es tránsito sin venta.
    for h in sorted(H_GDCORTO):
        so=str(GDCORTO.loc[h,'nodo_origen_id']); sd=str(GDCORTO.loc[h,'nodo_destino_id'])
        tipo=str(GDCORTO.loc[h,'destino_tipo'])
        if tipo!='CLIENTE':
            q=positive(value('QGCD_BG',h))
            if q>TOL_FLUJO_CERO_NM3:
                usage={}; _usage_add(usage,f'SHORTGASDIRECT|{h}',q,q)
                add(f'GASBC_SITE|{h}',so,sd,q,q,
                    producer_usage=usage,segment_keys=[f'SHORTGASDIRECT|{h}'],
                    pipe_distance_km=float(GDCORTO.loc[h,'distancia_recta_km']),
                    event={'tipo':'GASBC_CORTO_A_UBICACION','gasoducto_corto_id':str(h),
                           'origen':so,'destino':sd,'gasbc_entrada_nm3':q,'gasbc_salida_nm3':q})
        else:
            q=positive(value('QGCD_HUB',h))
            if q>TOL_FLUJO_CERO_NM3:
                usage={}; _usage_add(usage,f'SHORTGASDIRECT|{h}',q,q)
                add(f'GASBC_HUB|{h}',so,sd,q,q,
                    producer_usage=usage,segment_keys=[f'SHORTGASDIRECT|{h}'],
                    pipe_distance_km=float(GDCORTO.loc[h,'distancia_recta_km']),
                    event={'tipo':'GASBC_CORTO_HUB','gasoducto_corto_id':str(h),
                           'origen':so,'destino':sd,'gasbc_entrada_nm3':q,'gasbc_salida_nm3':q})

    # B) BG -> L -> BioGNL -> RL -> GasBC -> gasoducto corto HUB.
    for rl in S:
        gnl_inputs=[]
        q0=positive(value('YL0',rl))
        if q0>TOL_FLUJO_CERO_NM3: gnl_inputs.append((str(rl),q0))
        for l in sorted(ASITE_in[rl]):
            q=positive(value('YL',l,rl))
            if q>TOL_FLUJO_CERO_NM3: gnl_inputs.append((str(l),q))
        total_gnl=sum(q for _,q in gnl_inputs)
        if total_gnl<=TOL_FLUJO_CERO_NM3: continue
        for h in sorted(H_GDCORTO_by_origen_s[rl]):
            _tipo_dest=str(GDCORTO.loc[h,'destino_tipo'])
            if _tipo_dest=='CLIENTE':
                # En cliente sólo la fracción *_HUB continúa aguas abajo.
                q_hub=positive(value('QGCD_HUB_RL',h))
            elif _tipo_dest in {'UBICACION_POSIBLE','UBICACION_EMPRESA','ENTRADA_GASODUCTO'}:
                # En P# o NGI toda la salida RL por este gasoducto corto entra al
                # balance GasBC del nodo y por definición es tránsito.
                q_hub=positive(value('QGCD_RL',h))
            else:
                continue
            if q_hub<=TOL_FLUJO_CERO_NM3: continue
            dest=str(GDCORTO.loc[h,'nodo_destino_id'])
            for l,gnl_in in gnl_inputs:
                q_out_l=q_hub*gnl_in/total_gnl
                gasbc_l=q_out_l/(KL*KRL)
                gas_sources=[(str(p),positive(value('GBL',p,l))) for p in sorted(GBL_by_q[l])]
                total_gas=sum(q for _,q in gas_sources)
                tol_l=max(1e-5,1e-8*max(1.0,total_gas,positive(value('QL',l))))
                if abs(total_gas-positive(value('QL',l)))>tol_l:
                    raise AssertionError(('hub_gnl_fuentes_L',l,total_gas,value('QL',l)))
                if total_gas<=TOL_FLUJO_CERO_NM3: continue
                for p,qsrc in gas_sources:
                    if qsrc<=TOL_FLUJO_CERO_NM3: continue
                    inp=gasbc_l*qsrc/total_gas
                    outq=q_out_l*qsrc/total_gas
                    gnl_q=inp*KL
                    usage={}
                    _usage_add(usage,f'PLANT|L|{l}',inp,gnl_q)
                    _usage_add(usage,f'PLANT|RL|{rl}',gnl_q,outq)
                    _usage_add(usage,f'SHORTGASDIRECT|{h}',outq,outq)
                    transports=[]
                    if p!=l: transports.append({'kind':'FLOW_REF','ref':('GBL',p,l),'q':inp})
                    if l!=str(rl): transports.append({'kind':'FLOW_REF','ref':('YL',l,str(rl)),'q':gnl_q})
                    add(f'GNL_HUB|{h}|{l}|{p}',p,dest,inp,outq,
                        producer_usage=usage,transport_usage=transports,
                        segment_keys=[f'SHORTGASDIRECT|{h}'],
                        pipe_distance_km=float(GDCORTO.loc[h,'distancia_recta_km']),
                        event={'tipo':'BIOGNL_A_GASBC_HUB','l_site':l,'rl_site':str(rl),
                               'gasoducto_corto_id':str(h),'origen_bg':p,'destino':dest,
                               'gasbc_entrada_L_nm3':inp,'biognl_nm3':gnl_q,'gasbc_salida_RL_nm3':outq})

    # C) BG -> C -> BioGNC -> RC -> GasBC -> gasoducto corto HUB.
    for rc in S:
        gnc_inputs=[]
        q0=positive(value('YC0',rc))
        if q0>TOL_FLUJO_CERO_NM3: gnc_inputs.append((str(rc),q0))
        for c in sorted(ASITE_in[rc]):
            q=positive(value('YC',c,rc))
            if q>TOL_FLUJO_CERO_NM3: gnc_inputs.append((str(c),q))
        total_gnc=sum(q for _,q in gnc_inputs)
        if total_gnc<=TOL_FLUJO_CERO_NM3: continue
        for h in sorted(H_GDCORTO_by_origen_s[rc]):
            _tipo_dest=str(GDCORTO.loc[h,'destino_tipo'])
            if _tipo_dest=='CLIENTE':
                # En cliente sólo la fracción *_HUB continúa aguas abajo.
                q_hub=positive(value('QGCD_HUB_RC',h))
            elif _tipo_dest in {'UBICACION_POSIBLE','UBICACION_EMPRESA','ENTRADA_GASODUCTO'}:
                # En P# o NGI toda la salida RC por este gasoducto corto entra al
                # balance GasBC del nodo y por definición es tránsito.
                q_hub=positive(value('QGCD_RC',h))
            else:
                continue
            if q_hub<=TOL_FLUJO_CERO_NM3: continue
            dest=str(GDCORTO.loc[h,'nodo_destino_id'])
            for c,gnc_in in gnc_inputs:
                q_out_c=q_hub*gnc_in/total_gnc
                gasbc_c=q_out_c/(KC*KRC)
                gas_sources=[(str(p),positive(value('GBC',p,c))) for p in sorted(GBC_by_q[c])]
                total_gas=sum(q for _,q in gas_sources)
                tol_c=max(1e-5,1e-8*max(1.0,total_gas,positive(value('QC',c))))
                if abs(total_gas-positive(value('QC',c)))>tol_c:
                    raise AssertionError(('hub_gnc_fuentes_C',c,total_gas,value('QC',c)))
                if total_gas<=TOL_FLUJO_CERO_NM3: continue
                for p,qsrc in gas_sources:
                    if qsrc<=TOL_FLUJO_CERO_NM3: continue
                    inp=gasbc_c*qsrc/total_gas
                    outq=q_out_c*qsrc/total_gas
                    gnc_q=inp*KC
                    usage={}
                    _usage_add(usage,f'PLANT|C|{c}',inp,gnc_q)
                    _usage_add(usage,f'PLANT|RC|{rc}',gnc_q,outq)
                    _usage_add(usage,f'SHORTGASDIRECT|{h}',outq,outq)
                    transports=[]
                    if p!=c: transports.append({'kind':'FLOW_REF','ref':('GBC',p,c),'q':inp})
                    if c!=str(rc): transports.append({'kind':'FLOW_REF','ref':('YC',c,str(rc)),'q':gnc_q})
                    add(f'GNC_HUB|{h}|{c}|{p}',p,dest,inp,outq,
                        producer_usage=usage,transport_usage=transports,
                        segment_keys=[f'SHORTGASDIRECT|{h}'],
                        pipe_distance_km=float(GDCORTO.loc[h,'distancia_recta_km']),
                        event={'tipo':'BIOGNC_A_GASBC_HUB','c_site':c,'rc_site':str(rc),
                               'gasoducto_corto_id':str(h),'origen_bg':p,'destino':dest,
                               'gasbc_entrada_C_nm3':inp,'biognc_nm3':gnc_q,'gasbc_salida_RC_nm3':outq})

    # D) GasBC por corredor convencional y posterior llegada a cliente/HUB.
    for d in D:
        sources=_fuentes_salida_red_gasoducto(d)
        total_in=sum(positive(src['q']) for src in sources)
        if total_in<=TOL_FLUJO_CERO_NM3: continue
        hub_outputs=[]
        for j in sorted(AGC_by_d[d]):
            q=positive(value('ZG_HUB',d,j))
            if q>TOL_FLUJO_CERO_NM3:
                hub_outputs.append({
                    'modo':'ROAD','dest_node':str(CLIENTE_A_NODO[str(j)]),
                    'destino_tipo':'CLIENTE','destino_id':str(j),
                    'cliente_id':str(j),'empresa_id':None,'h_out':None,'q':q,
                })
        for h_out in sorted(H_GSCORTO_by_d[d]):
            q=positive(value('QGCP_HUB',h_out))
            if q>TOL_FLUJO_CERO_NM3:
                row=GSCORTO.loc[h_out]
                hub_outputs.append({
                    'modo':'PIPE','dest_node':str(row['nodo_destino_id']),
                    'destino_tipo':str(row['destino_tipo']),
                    'destino_id':str(row['destino_id']),
                    'cliente_id':None if pd.isna(row['cliente_id']) else str(row['cliente_id']),
                    'empresa_id':None if pd.isna(row['empresa_id']) else str(row['empresa_id']),
                    'h_out':str(h_out),'q':q,
                })
        g=str(DG[d])
        for hub_out in hub_outputs:
            out_mode=str(hub_out['modo'])
            dest=str(hub_out['dest_node'])
            h_out=hub_out['h_out']
            q_hub=positive(hub_out['q'])
            destino_tipo=str(hub_out['destino_tipo'])
            destino_id=str(hub_out['destino_id'])
            j=hub_out.get('cliente_id')
            empresa_hub=hub_out.get('empresa_id')
            for src in sources:
                qsrc=positive(src['q'])
                if qsrc<=TOL_FLUJO_CERO_NM3: continue
                part=q_hub*qsrc/total_in
                p=str(src['bg'])
                usage={}; transports=[]; segkeys=[]; pipe_dist=0.0
                h_in=src.get('gasoducto_corto_entrada_gasbc_id')
                pre_tipo=str(src.get('pre_conversion_tipo') or '')
                # Cantidad que debe consumirse realmente del pool GasBC del BG
                # de origen. Si el GasBC pasó por L/RL o C/RC antes de entrar al
                # gasoducto convencional, `part` es la cantidad POST-reconversión
                # y por tanto debe deshacerse el rendimiento de esa cadena.
                # Usar `part` como input del path perdía exactamente el factor
                # KL*KRL o KC*KRC y dejaba tokens fantasma en el BG de origen.
                path_input_q=part
                if pre_tipo=='GNL':
                    pre_l=str(src['pre_conversion_site']); pre_rl=str(src['pre_reconversion_site'])
                    gasbc_l=part*float(src['pre_gasbc_entrada_conversion_por_pipe'])
                    biognl=part*float(src['pre_producto_entrada_reconversion_por_pipe'])
                    path_input_q=gasbc_l
                    _usage_add(usage,f'PLANT|L|{pre_l}',gasbc_l,biognl)
                    _usage_add(usage,f'PLANT|RL|{pre_rl}',biognl,part)
                    if p!=pre_l:
                        transports.append({'kind':'FLOW_REF','ref':('GBL',p,pre_l),'q':gasbc_l})
                    if pre_l!=pre_rl:
                        transports.append({'kind':'FLOW_REF','ref':('YL',pre_l,pre_rl),'q':biognl})
                elif pre_tipo=='GNC':
                    pre_c=str(src['pre_conversion_site']); pre_rc=str(src['pre_reconversion_site'])
                    gasbc_c=part*float(src['pre_gasbc_entrada_conversion_por_pipe'])
                    biognc=part*float(src['pre_producto_entrada_reconversion_por_pipe'])
                    path_input_q=gasbc_c
                    _usage_add(usage,f'PLANT|C|{pre_c}',gasbc_c,biognc)
                    _usage_add(usage,f'PLANT|RC|{pre_rc}',biognc,part)
                    if p!=pre_c:
                        transports.append({'kind':'FLOW_REF','ref':('GBC',p,pre_c),'q':gasbc_c})
                    if pre_c!=pre_rc:
                        transports.append({'kind':'FLOW_REF','ref':('YC',pre_c,pre_rc),'q':biognc})
                elif h_in not in (None,'','None','nan'):
                    h_in=str(h_in); _usage_add(usage,f'SHORTGASIN|{h_in}',part,part)
                    segkeys.append(f'SHORTGASIN|{h_in}')
                    pipe_dist+=float(GECORTO.loc[h_in,'distancia_recta_km'])
                else:
                    d_inj=str(src.get('direccion_inyeccion_id') or d)
                    transports.append({'kind':'DIRECT_YG','bg':p,'d':d_inj,'q':part})
                    segkeys.append(f'AIG|{p}|{d_inj}')
                _path_ids=list(src.get('red_gasoductos_ids') or [g])
                _path_edges=list(src.get('red_gasoductos_arcos') or [])
                _path_intra=_clean_optional_id(src.get('red_gasoductos_recorrido_interno_direccion'))
                for _gp in _path_ids:
                    _usage_add(usage,f'PIPE|{_gp}',part,part)
                    segkeys.append(f'PIPE|{_gp}')
                if _path_intra:
                    _usage_add(usage,f'PIPEINTRA|{_path_intra}',part,part)
                    segkeys.append(f'PIPEINTRA|{_path_intra}')
                for _ga,_gb in _path_edges:
                    _usage_add(usage,f'PIPEPAIR|{_ga}|{_gb}',part,part)
                pipe_dist+=float(src.get('red_gasoductos_distancia_km',0.0) or 0.0)
                if out_mode=='PIPE':
                    _usage_add(usage,f'SHORTGAS|{h_out}',part,part)
                    segkeys.append(f'SHORTGAS|{h_out}')
                    pipe_dist+=float(GSCORTO.loc[h_out,'distancia_recta_km'])
                else:
                    transports.append({'kind':'FLOW_REF','ref':('ZG_HUB',str(d),str(j)),'q':part})
                add(f'PIPE_HUB|{d}|{out_mode}|{destino_tipo}|{destino_id}|{p}|{h_out or "ROAD"}',p,dest,path_input_q,part,
                    producer_usage=usage,transport_usage=transports,segment_keys=segkeys,
                    pipe_distance_km=pipe_dist,
                    event={'tipo':'GASODUCTO_CONVENCIONAL_A_HUB','direccion_id':str(d),
                           'gasoducto_id':g,'origen_bg':p,'destino':dest,
                           'destino_hub_tipo':destino_tipo,'destino_hub_id':destino_id,
                           'cliente_hub':j,'empresa_hub':empresa_hub,
                           'modo_salida':out_mode,'gasoducto_corto_salida_id':h_out,
                           'entrada_corredor_modo':str(src.get('modo_entrada','AIG')),
                           'direccion_inyeccion_id':src.get('direccion_inyeccion_id'),
                           'gasoducto_inyeccion_id':src.get('gasoducto_inyeccion_id'),
                           'red_gasoductos_ids':list(src.get('red_gasoductos_ids') or [g]),
                           'red_gasoductos_arcos':list(src.get('red_gasoductos_arcos') or []),
                           'red_gasoductos_distancia_km':float(src.get('red_gasoductos_distancia_km',0.0) or 0.0),
                           'red_gasoductos_recorrido_interno_direccion':_clean_optional_id(src.get('red_gasoductos_recorrido_interno_direccion')) or None,
                           'pre_conversion_tipo':src.get('pre_conversion_tipo'),
                           'pre_conversion_site':src.get('pre_conversion_site'),
                           'pre_reconversion_site':src.get('pre_reconversion_site'),
                           'gasbc_nm3':part})
    return paths


hub_gasbc_paths=_build_hub_gasbc_paths()
hub_paths_by_origin=defaultdict(list)
hub_paths_by_dest=defaultdict(list)
for _p in hub_gasbc_paths:
    hub_paths_by_origin[_p['origin']].append(_p)
    hub_paths_by_dest[_p['dest']].append(_p)
for _s in hub_paths_by_origin:
    hub_paths_by_origin[_s].sort(key=lambda p:p['path_id'])


def _token_scale(tok,factor,*,new_qty=None):
    factor=float(factor)
    out=copy.deepcopy(tok)
    out['raw_q']=float(tok['raw_q'])*factor
    out['qty']=float(tok['qty'])*factor if new_qty is None else float(new_qty)
    usage={}
    for pk,u in (tok.get('hub_producer_usage') or {}).items():
        usage[str(pk)]={'input':float(u.get('input',0.0))*factor,
                        'output':float(u.get('output',0.0))*factor}
    out['hub_producer_usage']=usage
    tu=[]
    for u in (tok.get('hub_transport_usage') or []):
        z=copy.deepcopy(u)
        if 'q' in z: z['q']=float(z['q'])*factor
        tu.append(z)
    out['hub_transport_usage']=tu
    ev=[]
    for e in (tok.get('hub_chain_events') or []):
        z=copy.deepcopy(e)
        for k,v in list(z.items()):
            if k.endswith('_nm3') and isinstance(v,(int,float,np.number)):
                z[k]=float(v)*factor
        ev.append(z)
    out['hub_chain_events']=ev
    # La distancia/las claves son propiedades del camino, no bases de prorrateo.
    out['hub_segment_keys_static']=list(tok.get('hub_segment_keys_static') or [])
    out['hub_pipe_distance_km']=float(tok.get('hub_pipe_distance_km',0.0))
    return out


def _take_from_tokens(tokens,amount,tol):
    amount=float(amount)
    if amount<=tol: return None,0.0
    active=[(idx,t) for idx,t in enumerate(tokens) if float(t.get('qty',0.0))>tol]
    if not active: return None,0.0
    sufficient=[(idx,t) for idx,t in active if float(t['qty'])>=amount-tol]
    if sufficient:
        idx,t=min(sufficient,key=lambda it:(float(it[1]['qty'])-amount,str(it[1].get('empresa_id')),str(it[1].get('raw_source_site'))))
    else:
        idx,t=max(active,key=lambda it:float(it[1]['qty']))
    tokens.pop(idx)
    tq=float(t['qty']); take=min(amount,tq)
    if take>=tq-tol:
        part=t; take=tq
    else:
        f=take/tq
        part=_token_scale(t,f,new_qty=take)
        rem=_token_scale(t,1.0-f,new_qty=tq-take)
        tokens.append(rem)
    return part,take


def _append_hub_path_to_token(tok,path,input_chunk):
    scale=float(input_chunk)/float(path['input_q'])
    out=copy.deepcopy(tok)
    for pk,u in path.get('producer_usage',{}).items():
        rec=out.setdefault('hub_producer_usage',{}).setdefault(str(pk),{'input':0.0,'output':0.0})
        rec['input']+=float(u.get('input',0.0))*scale
        rec['output']+=float(u.get('output',0.0))*scale
    for u in path.get('transport_usage',[]):
        z=copy.deepcopy(u)
        if 'q' in z: z['q']=float(z['q'])*scale
        out.setdefault('hub_transport_usage',[]).append(z)
    for key in path.get('segment_keys',[]):
        if str(key) not in out.setdefault('hub_segment_keys_static',[]):
            out['hub_segment_keys_static'].append(str(key))
    out['hub_pipe_distance_km']=float(out.get('hub_pipe_distance_km',0.0))+float(path.get('pipe_distance_km',0.0))
    ev=copy.deepcopy(path.get('event',{}))
    for k,v in list(ev.items()):
        if k.endswith('_nm3') and isinstance(v,(int,float,np.number)):
            ev[k]=float(v)*scale
    ev['path_id']=str(path['path_id'])
    out.setdefault('hub_chain_events',[]).append(ev)
    out['qty']=float(path['output_q'])*scale
    return out


def _route_from_token(rec,tok,gasbc_chunk):
    gasbc_total=positive(rec.get('gasbc_salida_BG_nm3',positive(rec['biogas_crudo_entrada_BG_nm3'])*KBG))
    if gasbc_total<=TOL_FLUJO_CERO_NM3: return None
    factor=float(gasbc_chunk)/gasbc_total
    out=dict(rec)
    for field in ROUTE_FLOW_FIELDS:
        if field in out:
            out[field]=float(out[field])*factor
    # La ruta final comienza en out['bg_site'], pero la planta BG que realmente
    # produjo este GasBC puede estar aguas arriba de uno o varios HUBs.
    out['empresa_id']=str(tok['empresa_id'])
    out['bg_production_site']=str(tok['bg_production_site'])
    out['raw_source_site']=str(tok['raw_source_site'])
    out['biogas_crudo_entrada_BG_nm3']=float(tok['raw_q'])
    out['gasbc_salida_BG_nm3']=float(tok['raw_q'])*KBG
    out['gasbc_entrada_ruta_actual_nm3']=float(gasbc_chunk)
    out['gasbc_perdido_antes_ruta_actual_nm3']=max(0.0,float(tok['raw_q'])*KBG-float(gasbc_chunk))
    out['cantidad_nm3']=positive(out.get('cantidad_nm3',0.0))
    out['gasbc_final_entregado_nm3']=out['cantidad_nm3']
    out['hub_producer_usage']=copy.deepcopy(tok.get('hub_producer_usage') or {})
    out['hub_transport_usage']=copy.deepcopy(tok.get('hub_transport_usage') or [])
    out['hub_segment_keys_static']=list(tok.get('hub_segment_keys_static') or [])
    out['hub_pipe_distance_km']=float(tok.get('hub_pipe_distance_km',0.0))
    out['hub_chain_events']=copy.deepcopy(tok.get('hub_chain_events') or [])
    out['usa_hub_gasbc']=bool(out['hub_chain_events'])
    out['hub_chain_text']=' → '.join(str(e.get('tipo','HUB')) for e in out['hub_chain_events'])

    mode=str(tok['source_mode']); h0=tok.get('source_h')
    relay_path=tuple(tok.get('raw_relay_path') or ())
    out['raw_relay_path']=[f'{kind}|{hid}' for kind,hid in relay_path]
    out['gasoductos_biogas_relevo_empresa_ids']=[str(hid) for kind,hid in relay_path if kind=='EHUB']
    out['gasoductos_biogas_relevo_cliente_ids']=[str(hid) for kind,hid in relay_path if kind=='C2C']
    out['gasoductos_biogas_relevo_P_ids']=[str(hid) for kind,hid in relay_path if kind=='PHUB']
    if mode=='PIPE':
        h0=str(h0)
        destino_tipo=str(GCORTO.loc[h0,'destino_tipo'])
        out['tipo_fuente_bg']='PIPE_BIOGAS'
        out['gasoducto_biogas_corto_id']=h0
        out['usa_gasoducto_biogas_corto']=True
        out['gasoducto_destino_tipo']=destino_tipo
        out['gasoducto_destino_id']=str(GCORTO.loc[h0,'destino_id'])
        out['gasoducto_nodo_destino_id']=str(GCORTO.loc[h0,'nodo_destino_id'])
        out['tipo_transporte_biogas_crudo']=(
            'Gasoducto corto empresa-cliente' if destino_tipo=='CLIENTE' else
            'Gasoducto corto empresa-entrada de gasoducto' if destino_tipo=='ENTRADA_GASODUCTO' else
            'Gasoducto corto empresa-ubicación de empresa' if destino_tipo=='UBICACION_EMPRESA' else
            'Gasoducto corto empresa-ubicación posible'
        )
    elif mode=='ROAD':
        out['tipo_fuente_bg']='ROAD_OR_LOCAL'
        out['gasoducto_biogas_corto_id']=None
        out['usa_gasoducto_biogas_corto']=False
        out['gasoducto_destino_tipo']=None
        out['gasoducto_destino_id']=None
        out['gasoducto_nodo_destino_id']=None
        out['tipo_transporte_biogas_crudo']='Transporte vial o transferencia local existente'
    else:
        raise AssertionError(('fuente_raw_no_reconocida',mode,tok))
    return out


# Tokens iniciales: cada uno conserva empresa, transporte inicial y relevos RAW.
tokens_by_bg=defaultdict(list)
for _bg,_items in raw_processed_by_bg.items():
    for _source_key,_raw_q in _items:
        if _raw_q<=TOL_FLUJO_CERO_NM3: continue
        _mode,_empresa,_h0,_raw_site,_relay_path=_source_key
        tokens_by_bg[str(_bg)].append({
            'qty':float(_raw_q)*KBG,'raw_q':float(_raw_q),
            'empresa_id':str(_empresa),'source_mode':str(_mode),'source_h':_h0,
            'raw_source_site':str(_raw_site),'raw_relay_path':tuple(_relay_path or ()),
            'bg_production_site':str(_bg),'hub_producer_usage':{},
            'hub_transport_usage':[],'hub_segment_keys_static':[],
            'hub_pipe_distance_km':0.0,'hub_chain_events':[],
        })

# Demanda terminal de GasBC de las rutas finales ya reconstruidas aguas abajo.
route_needs_by_bg=defaultdict(list)
for _bg,_recs in base_by_bg.items():
    for _idx,_rec in enumerate(sorted(_recs,key=lambda r:(str(r.get('cliente_id','')),str(r.get('familia','')),-positive(r.get('cantidad_nm3',0.0))))):
        _q=positive(_rec.get('gasbc_salida_BG_nm3',positive(_rec['biogas_crudo_entrada_BG_nm3'])*KBG))
        if _q>TOL_FLUJO_CERO_NM3:
            route_needs_by_bg[str(_bg)].append({'rec':_rec,'remaining':float(_q),'order':_idx})

# Auditoría de balance por nodo ANTES de atribuir empresas. Esto reemplaza el
# antigua comprobación supply_total==demand_total que ignoraba entradas/salidas HUB.
for _s in S:
    _s=str(_s)
    local=positive(value('QBIO_BG',_s))*KBG
    incoming=sum(float(p['output_q']) for p in hub_paths_by_dest.get(_s,[]))
    terminal=sum(float(r['remaining']) for r in route_needs_by_bg.get(_s,[]))
    outgoing=sum(float(p['input_q']) for p in hub_paths_by_origin.get(_s,[]))
    lhs=local+incoming; rhs=terminal+outgoing
    _tol=max(1e-5,1e-8*max(1.0,lhs,rhs))
    if abs(lhs-rhs)>_tol:
        raise AssertionError((_s,'balance_postproceso_GasBC_HUB',lhs,rhs,
                              {'BG_local':local,'HUB_entrada':incoming,
                               'rutas_finales':terminal,'HUB_salida':outgoing}))

# Residuales de los caminos HUB para permitir cadenas de varios saltos.
for _p in hub_gasbc_paths:
    _p['_remaining_input']=float(_p['input_q'])

routes=[]
_max_iter=max(1000,20*(len(S)+len(hub_gasbc_paths)+len(base_routes)+1))
for _iter in range(_max_iter):
    progress=False
    # 1) Prioridad a los relevos: evita bloquear un ciclo por consumir primero
    #    todo el inventario local en una venta terminal.
    for _s in sorted(map(str,S)):
        toks=tokens_by_bg[_s]
        for _p in hub_paths_by_origin.get(_s,[]):
            while _p['_remaining_input']>TOL_FLUJO_CERO_NM3:
                _part,_take=_take_from_tokens(toks,_p['_remaining_input'],TOL_FLUJO_CERO_NM3)
                if _part is None or _take<=TOL_FLUJO_CERO_NM3: break
                _p['_remaining_input']=max(0.0,_p['_remaining_input']-_take)
                _outtok=_append_hub_path_to_token(_part,_p,_take)
                tokens_by_bg[str(_p['dest'])].append(_outtok)
                progress=True
    # 2) Con el remanente del pool se cubren las rutas que terminan en ventas.
    for _s in sorted(map(str,S)):
        toks=tokens_by_bg[_s]
        for _rn in route_needs_by_bg.get(_s,[]):
            while _rn['remaining']>TOL_FLUJO_CERO_NM3:
                _part,_take=_take_from_tokens(toks,_rn['remaining'],TOL_FLUJO_CERO_NM3)
                if _part is None or _take<=TOL_FLUJO_CERO_NM3: break
                _rn['remaining']=max(0.0,_rn['remaining']-_take)
                _route=_route_from_token(_rn['rec'],_part,_take)
                if _route is not None: routes.append(_route)
                progress=True
    if not progress:
        break

_residual_paths={p['path_id']:p['_remaining_input'] for p in hub_gasbc_paths if p['_remaining_input']>TOL_FLUJO_CERO_NM3}
_residual_routes={
    (s,idx,str(rn['rec'].get('familia')),str(rn['rec'].get('cliente_id'))):rn['remaining']
    for s,arr in route_needs_by_bg.items() for idx,rn in enumerate(arr)
    if rn['remaining']>TOL_FLUJO_CERO_NM3
}
_residual_tokens={s:sum(float(t.get('qty',0.0)) for t in arr) for s,arr in tokens_by_bg.items()
                  if sum(float(t.get('qty',0.0)) for t in arr)>TOL_FLUJO_CERO_NM3}
if _residual_paths or _residual_routes or _residual_tokens:
    raise AssertionError(('descomposicion_GasBC_HUB_incompleta',
                          {'paths':_residual_paths,'routes':_residual_routes,'tokens':_residual_tokens}))

# Conservación de la procedencia empresarial: ningún HUB crea o destruye la base
# de biogás crudo atribuida a cada proveedor; las pérdidas de proceso se reflejan
# sólo en la cantidad de GasBC corriente del token.
_expected_raw_company={str(i):(
    sum(positive(value('X',i,s)) for s in ABIO_by_i[i])
    +sum(positive(value('QBCP',h)) for h in H_CORTO_by_i[i])
) for i in I}
_observed_raw_company=defaultdict(float)
for _r in routes:
    _observed_raw_company[str(_r['empresa_id'])]+=positive(_r.get('biogas_crudo_entrada_BG_nm3',0.0))
for _i,_expected in _expected_raw_company.items():
    _observed=float(_observed_raw_company.get(_i,0.0))
    _tol=max(1e-4,1e-8*max(1.0,_expected,_observed))
    if abs(_observed-_expected)>_tol:
        raise AssertionError((_i,'trazabilidad_empresa_HUB',_observed,_expected))

# Metadatos legibles de cada ruta. Los nombres de las plantas se leen
# desde tipos_plantas.csv para que el HTML nunca sustituya el tipo real por
# etiquetas genéricas en lugar del nombre real del tipo de planta.
SITE_NAME=sitios.set_index('id')['nombre'].to_dict(); E_NAME=E['nombre'].to_dict(); J_NAME=CL['nombre'].to_dict(); G_NAME=GA['nombre'].to_dict()
G_CORTO_NAME={h:str(GCORTO.loc[h,'nombre']) for h in H_CORTO}
PLANT_TYPE_NAME={t:str(PL.loc[t,'tipo_instalacion']) for t in T}
for n,r in enumerate(routes,1):
    r['route_id']=f'R{n:04d}'
    r['empresa']=E_NAME[r['empresa_id']]; r['cliente']=J_NAME[r['cliente_id']]; r['bg_nombre']=SITE_NAME[r['bg_site']]
    r['bg_production_site']=str(r.get('bg_production_site') or r['bg_site'])
    r['bg_production_nombre']=SITE_NAME[r['bg_production_site']]
    r['bg_tipo_planta']=PLANT_TYPE_NAME['BG']
    h_directo=r.get('gasoducto_gasbc_directo_id')
    if h_directo is not None and str(h_directo) not in {'','None','nan'}:
        h_directo=str(h_directo)
        r['gasoducto_gasbc_directo']=str(GDCORTO.loc[h_directo,'nombre'])
        r['distancia_gasoducto_gasbc_directo_km']=float(GDCORTO.loc[h_directo,'distancia_recta_km'])
        r['gasoducto_gasbc_directo_destino_tipo']=str(GDCORTO.loc[h_directo,'destino_tipo'])
        r['gasoducto_gasbc_directo_destino_id']=str(GDCORTO.loc[h_directo,'destino_id'])
    # Cualquier familia puede haber atravesado el corredor antes de llegar a L/C.
    # En ese caso se conservan los metadatos del gasoducto también para BioGNL/BioGNC.
    if r.get('gasoducto_id') not in (None,'','None','nan'):
        d=str(r['direccion_id']); g=str(r['gasoducto_id'])
        r['gasoducto']=G_NAME[g]
        r['sentido']=GD.loc[d,'sentido']
        r['entrada_gasoducto']=GD.loc[d,'entrada_nombre']
        r['salida_gasoducto']=GD.loc[d,'salida_nombre']
        r['nodo_conversion_salida_id']=str(SALIDA_GASODUCTO_A_NODO[d])
        h_entrada=r.get('gasoducto_corto_entrada_gasbc_id')
        if h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
            h_entrada=str(h_entrada)
            r['gasoducto_corto_entrada_gasbc']=str(GECORTO.loc[h_entrada,'nombre'])
            r['distancia_gasoducto_corto_entrada_gasbc_km']=float(GECORTO.loc[h_entrada,'distancia_recta_km'])
            r['usa_gasoducto_corto_entrada_gasbc']=True

    if r['familia'] in {'GasBC'}:
        r['origen_reparto']=f"SITE:{r['bg_site']}"
    elif r['familia']=='BioGNL':
        r['origen_reparto']=f"SITE:{r['rl_site']}"
    elif r['familia']=='BioGNC':
        r['origen_reparto']=f"SITE:{r['rc_site']}"
    elif r['familia']=='Gasoducto':
        r['origen_reparto']=f"PIPE:{r['direccion_id']}"
    if r.get('usa_gasoducto_biogas_corto'):
        h=str(r['gasoducto_biogas_corto_id'])
        r['gasoducto_biogas_corto']=G_CORTO_NAME[h]
        r['distancia_gasoducto_biogas_corto_km']=float(GCORTO.loc[h,'distancia_recta_km'])
        r['gasoducto_destino_tipo']=str(GCORTO.loc[h,'destino_tipo'])
        r['gasoducto_destino_id']=str(GCORTO.loc[h,'destino_id'])
        r['gasoducto_destino_nombre']=str(GCORTO.loc[h,'destino_nombre'])
    if r.get('pre_l_site') not in (None,'','None','nan'):
        r['pre_transformacion_origen']=SITE_NAME[str(r['pre_l_site'])]
        r['pre_transformacion_destino']=SITE_NAME[str(r['pre_rl_site'])]
        r['pre_conversion_producto']='BioGNL'
    elif r.get('pre_c_site') not in (None,'','None','nan'):
        r['pre_transformacion_origen']=SITE_NAME[str(r['pre_c_site'])]
        r['pre_transformacion_destino']=SITE_NAME[str(r['pre_rc_site'])]
        r['pre_conversion_producto']='BioGNC'
    if r['familia']=='BioGNL':
        r['transformacion_origen']=SITE_NAME[r['l_site']]; r['transformacion_destino']=SITE_NAME[r['rl_site']]
        r['l_tipo_planta']=PLANT_TYPE_NAME['L']; r['rl_tipo_planta']=PLANT_TYPE_NAME['RL']
    elif r['familia']=='BioGNC':
        r['transformacion_origen']=SITE_NAME[r['c_site']]; r['transformacion_destino']=SITE_NAME[r['rc_site']]
        r['c_tipo_planta']=PLANT_TYPE_NAME['C']; r['rc_tipo_planta']=PLANT_TYPE_NAME['RC']
    elif r['familia']=='Gasoducto':
        d=r['direccion_id']; r['gasoducto']=G_NAME[r['gasoducto_id']]
        r['sentido']=GD.loc[d,'sentido']; r['entrada_gasoducto']=GD.loc[d,'entrada_nombre']; r['salida_gasoducto']=GD.loc[d,'salida_nombre']
        h_entrada=r.get('gasoducto_corto_entrada_gasbc_id')
        if h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
            h_entrada=str(h_entrada)
            r['gasoducto_corto_entrada_gasbc']=str(GECORTO.loc[h_entrada,'nombre'])
            r['distancia_gasoducto_corto_entrada_gasbc_km']=float(GECORTO.loc[h_entrada,'distancia_recta_km'])
            r['usa_gasoducto_corto_entrada_gasbc']=True
        h_salida=r.get('gasoducto_corto_salida_cliente_id')
        if h_salida is not None and str(h_salida) not in {'','None','nan'}:
            h_salida=str(h_salida)
            r['gasoducto_corto_salida_cliente']=str(GSCORTO.loc[h_salida,'nombre'])
            r['distancia_gasoducto_corto_salida_cliente_km']=float(GSCORTO.loc[h_salida,'distancia_recta_km'])

df_rutas=pd.DataFrame(routes)
if len(df_rutas):
    cols=['route_id']+[c for c in df_rutas.columns if c!='route_id']
    df_rutas=df_rutas[cols]

# Validaciones de conservación de la descomposición.
route_total=float(df_rutas['cantidad_nm3'].sum()) if len(df_rutas) else 0.0
served_total=(sum(value('ZD',s,j)+value('ZL',s,j)+value('ZC',s,j) for s,j in ABC)+sum(value('ZG',d,j) for d,j in AGC)+sum(value('QGCP',h) for h in H_GSCORTO)+sum(value('QGCD',h) for h in H_GDCORTO_CLIENTE))
# Auditoría separada por modo de llegada: el HTML no puede convertir una mezcla
# carretera+gasoducto en una única ruta visual incorrecta.
if len(df_rutas):
    _short_col=df_rutas.get('gasoducto_biogas_corto_id',pd.Series(index=df_rutas.index,dtype=object))
    _mask_pipe=_short_col.notna() & ~_short_col.astype(str).isin({'','None','nan'})
    raw_pipe_by_h=(
        df_rutas.loc[_mask_pipe]
        .groupby(_short_col.loc[_mask_pipe].astype(str))['biogas_crudo_entrada_BG_nm3']
        .sum().to_dict()
    )
    _raw_source_site_col=df_rutas.get(
        'raw_source_site',
        df_rutas.get('bg_production_site',df_rutas.get('bg_site'))
    ).astype(str)
    _audit_road=df_rutas.loc[~_mask_pipe].copy()
    _audit_road['_raw_source_site_audit']=_raw_source_site_col.loc[_audit_road.index]
    raw_road_by_pair=(
        _audit_road
        .groupby(['empresa_id','_raw_source_site_audit'])['biogas_crudo_entrada_BG_nm3']
        .sum().to_dict()
    )
else:
    raw_pipe_by_h={}
    raw_road_by_pair={}

for h in ([] if _hub_logistico_activo else H_CORTO):
    expected=positive(value('QBCP',h))
    observed=float(raw_pipe_by_h.get(str(h),0.0))
for i,s in ([] if _hub_logistico_activo else ABIO):
    expected=positive(value('X',i,s))
    observed=float(raw_road_by_pair.get((i,s),0.0))

# Auditoría adicional de los relevos de biogás crudo. Cada Nm³ que atraviesa
# QBCP_HUB/QBC2C debe aparecer exactamente en las rutas finales que heredan ese
# tramo físico. Se omite sólo cuando existe pooling aguas abajo de GasBC.
if len(df_rutas) and not _hub_logistico_activo:
    _relay_emp_by_h=defaultdict(float); _relay_c2c_by_h=defaultdict(float); _relay_p_by_h=defaultdict(float)
    for _,_rr in df_rutas.iterrows():
        _q=max(0.0,positive(_rr.get('biogas_crudo_entrada_BG_nm3',0.0)))
        for _h in (_rr.get('gasoductos_biogas_relevo_empresa_ids',[]) or []):
            _relay_emp_by_h[str(_h)]+=_q
        for _h in (_rr.get('gasoductos_biogas_relevo_cliente_ids',[]) or []):
            _relay_c2c_by_h[str(_h)]+=_q
        for _h in (_rr.get('gasoductos_biogas_relevo_P_ids',[]) or []):
            _relay_p_by_h[str(_h)]+=_q
    for h in H_EHUB:
        expected=max(0.0,positive(value('QBCP_HUB',h))-float(_raw_biogas_relay_cycle_residuals_by_kind_h.get(('EHUB',str(h)),0.0)))
        observed=float(_relay_emp_by_h.get(str(h),0.0))
    for h in H_BIOC2C:
        expected=max(0.0,positive(value('QBC2C',h))-float(_raw_biogas_relay_cycle_residuals_by_kind_h.get(('C2C',str(h)),0.0)))
        observed=float(_relay_c2c_by_h.get(str(h),0.0))
    for h in H_PHUB:
        expected=max(0.0,positive(value('QBCP_P',h))-float(_raw_biogas_relay_cycle_residuals_by_kind_h.get(('PHUB',str(h)),0.0)))
        observed=float(_relay_p_by_h.get(str(h),0.0))

if len(df_rutas):
    _short_gas_in_col=df_rutas.get(
        'gasoducto_corto_entrada_gasbc_id',
        pd.Series(index=df_rutas.index,dtype=object),
    )
    _mask_short_gas_in=_short_gas_in_col.notna() & ~_short_gas_in_col.astype(str).isin({'','None','nan'})
    gas_in_by_h=(
        df_rutas.loc[_mask_short_gas_in]
        .groupby(_short_gas_in_col.loc[_mask_short_gas_in].astype(str))['gasbc_salida_BG_nm3']
        .sum().to_dict()
    )
else:
    gas_in_by_h={}
for h in H_GECORTO:
    expected=positive(value('QGCE',h))
    observed=float(gas_in_by_h.get(str(h),0.0))



# Helper local: esta función también se redefine más adelante en la sección de
# polilíneas, pero debe existir aquí porque la auditoría de QGCD se ejecuta antes.
def _list_ids(value_obj):
    if value_obj is None:
        return []
    if isinstance(value_obj,(list,tuple,set)):
        return [str(x) for x in value_obj if str(x) not in {'','None','nan'}]
    try:
        if pd.isna(value_obj):
            return []
    except Exception:
        pass
    return [str(value_obj)]


# Auditoría robusta de QGCD. `QGCD` es el flujo físico total del gasoducto corto
# y por formulación se descompone exactamente por procedencia: BG, RL o RC.
# Esta identidad es la comprobación cuantitativa correcta; una ruta que continúa
# por P/HUB no debe exigirse como una venta terminal en `df_rutas`.
traced_shortgas_ids=set()
if len(df_rutas):
    for _,_row in df_rutas.iterrows():
        _h_direct=getattr(_row,'gasoducto_gasbc_directo_id',None)
        if _h_direct is not None and str(_h_direct) not in {'','None','nan'}:
            traced_shortgas_ids.add(str(_h_direct))
        for _key in _list_ids(getattr(_row,'hub_segment_keys_static',None)):
            _key=str(_key)
            if _key.startswith('SHORTGASDIRECT|'):
                traced_shortgas_ids.add(_key.split('|',1)[1])
        _usage=getattr(_row,'hub_producer_usage',{})
        if isinstance(_usage,dict):
            for _key in _usage:
                _key=str(_key)
                if _key.startswith('SHORTGASDIRECT|'):
                    traced_shortgas_ids.add(_key.split('|',1)[1])

for h in H_GDCORTO:
    expected=positive(value('QGCD',h))
    observed_components=(
        positive(value('QGCD_BG',h))
        +positive(value('QGCD_RL',h))
        +positive(value('QGCD_RC',h))
    )
    tol=max(1e-4,1e-8*max(1.0,expected,observed_components))
    if expected>TOL_FLUJO_CERO_NM3:
        pass

print(f"Rutas descompuestas: {len(df_rutas)}")
print("Rendimientos acumulados:", {
    'GasBC/Gasoducto':K['BG'],
    'BioGNL':K['BG']*K['L']*K['RL'],
    'BioGNC':K['BG']*K['C']*K['RC'],
})
display(df_rutas.head(30))

Rutas descompuestas: 14
Rendimientos acumulados: {'GasBC/Gasoducto': 0.6, 'BioGNL': 0.5999976, 'BioGNC': 0.5999994}


,route_id,bg_site,familia,cliente_id,final_site,biogas_crudo_entrada_BG_nm3,gasbc_salida_BG_nm3,cantidad_nm3,gasbc_final_entregado_nm3,empresa_id,...,c_tipo_planta,rc_tipo_planta,l_site,rl_site,gasbc_entrada_L_nm3,biognl_salida_L_nm3,biognl_entrada_RL_nm3,gasbc_salida_RL_nm3,l_tipo_planta,rl_tipo_planta
0,R0001,NC_C2,GasBC,C2,NC_C2,30303.030303,18181.818182,18181.800000,18181.800000,E1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,R0002,NC_C2,GasBC,C2,NC_C2,24617.516502,14770.509901,14770.495131,14770.495131,E2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,R0003,NC_C2,GasBC,C2,NC_C2,5302.122021,3181.273212,3181.260487,3181.260487,E6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,R0004,NE_E1,BioGNC,C1,NC_C1,30303.030303,18181.818182,18181.800000,18181.800000,E1,...,BioGNC,GNC_a_GasBC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R0005,NE_E1,BioGNC,C6,NC_C6,30303.030303,18181.818182,18181.800000,18181.800000,E1,...,BioGNC,GNC_a_GasBC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,R0006,NE_E1,BioGNC,C7,NC_C7,30303.030303,18181.818182,18181.800000,18181.800000,E1,...,BioGNC,GNC_a_GasBC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,R0007,NE_E1,BioGNC,C8,NC_C8,30303.030303,18181.818182,18181.800000,18181.800000,E1,...,BioGNC,GNC_a_GasBC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,R0008,NE_E14,BioGNL,C3,NC_C3,120000.480002,72000.288001,72000.000000,72000.000000,E1,...,NaN,NaN,NE_E14,NC_C3,72000.288001,120.000000,120.000000,72000.000000,BioGNL,GNL_a_GasBC
8,R0009,NE_E14,BioGNL,C4,NC_C4,160000.640003,96000.384002,96000.000000,96000.000000,E1,...,NaN,NaN,NE_E14,NC_C4,96000.384002,160.000000,160.000000,96000.000000,BioGNL,GNL_a_GasBC
9,R0010,NE_E14,BioGNL,C5,NC_C5,93975.756036,56385.453622,56385.228080,56385.228080,E1,...,NaN,NaN,NE_E14,NC_C5,56385.453622,93.975380,93.975380,56385.228080,BioGNL,GNL_a_GasBC


# 9

In [ ]:
# ============================================================
# 9. POLILÍNEAS DE LA SOLUCIÓN Y VIAJES DIRECTOS/COMPARTIDOS
# ============================================================
def _segments_for_flow_ref(ref):
    return list(transport_segment_keys_by_flow_ref.get(tuple(ref),[]))

def _tour_ids_for_flow_ref(ref):
    return list(transport_tour_ids_by_flow_ref.get(tuple(ref),[]))

def _paths_for_flow_ref(ref):
    return [dict(x) for x in transport_paths_by_flow_ref.get(tuple(ref),[])]


def _conversion_uses_pipeline(row):
    return (
        str(getattr(row,'familia','')) in {'BioGNL','BioGNC'}
        and getattr(row,'gasoducto_id',None) is not None
        and str(getattr(row,'gasoducto_id',None)) not in {'','None','nan'}
    )

def _entry_reconversion_mode(row):
    return str(getattr(row,'entrada_corredor_modo',''))


def _uses_entry_reconversion(row):
    return _entry_reconversion_mode(row).startswith('RECONVERSION_')


def _uses_local_ngi_transfer(row):
    try:
        q_local=_finite(getattr(row,'gasbc_transferencia_local_ngi_nm3',0.0))
    except Exception:
        q_local=0.0
    flag=getattr(row,'transferencia_local_ngi',False)
    try:
        if pd.isna(flag):
            flag=False
    except Exception:
        pass
    return bool(flag) or q_local>TOL_FLUJO_CERO_NM3


def _pre_entry_transport_refs(row):
    refs=[]
    modo=_entry_reconversion_mode(row)
    if modo.startswith('RECONVERSION_GNL'):
        pre_l=str(getattr(row,'pre_l_site'))
        pre_rl=str(getattr(row,'pre_rl_site'))
        if str(row.bg_site)!=pre_l:
            refs.append(('GBL',str(row.bg_site),pre_l))
        if pre_l!=pre_rl:
            refs.append(('YL',pre_l,pre_rl))
    elif modo.startswith('RECONVERSION_GNC'):
        pre_c=str(getattr(row,'pre_c_site'))
        pre_rc=str(getattr(row,'pre_rc_site'))
        if str(row.bg_site)!=pre_c:
            refs.append(('GBC',str(row.bg_site),pre_c))
        if pre_c!=pre_rc:
            refs.append(('YC',pre_c,pre_rc))
    return refs

def _list_ids(value_obj):
    if value_obj is None:
        return []
    if isinstance(value_obj,(list,tuple,set)):
        return [str(x) for x in value_obj if str(x) not in {'','None','nan'}]
    try:
        if pd.isna(value_obj):
            return []
    except Exception:
        pass
    return [str(value_obj)]



def _clean_optional_id(value_obj):
    if value_obj is None:
        return ''
    if isinstance(value_obj, (list, tuple, set)):
        for item in value_obj:
            cleaned = _clean_optional_id(item)
            if cleaned:
                return cleaned
        return ''
    try:
        if pd.isna(value_obj):
            return ''
    except Exception:
        pass
    value_str = str(value_obj).strip()
    return '' if value_str.lower() in {'', 'none', 'nan', 'nat', 'null'} else value_str


def _raw_relay_segment_keys(row):
    keys=[]
    for h in _list_ids(getattr(row,'gasoductos_biogas_relevo_empresa_ids',None)):
        keys.append(f'SHORTBIO|{h}' if h in H_CORTO_SET else f'SHORTBIOHUBEMP|{h}')
    for h in _list_ids(getattr(row,'gasoductos_biogas_relevo_cliente_ids',None)):
        keys.append(f'SHORTBIOC2C|{h}')
    return keys

def _hub_transport_usage_list(row):
    value_obj=getattr(row,'hub_transport_usage',None)
    return list(value_obj) if isinstance(value_obj,(list,tuple)) else []


def _hub_static_segment_keys(row):
    value_obj=getattr(row,'hub_segment_keys_static',None)
    if not isinstance(value_obj,(list,tuple,set)):
        return []
    return [str(x) for x in value_obj if str(x) not in {'','None','nan'}]


def _collect_route_transport(row, getter):
    values=[]
    def add(items):
        for item in items:
            if item not in values:
                values.append(item)

    # Viajes físicos anteriores a la ruta final debidos a relevos GasBC HUB.
    for _hu in _hub_transport_usage_list(row):
        if not isinstance(_hu,dict):
            continue
        if str(_hu.get('kind',''))=='FLOW_REF':
            _ref=tuple(_hu.get('ref',()))
            if _ref:
                add(getter(_ref))

    h_direct=getattr(row,'gasoducto_gasbc_directo_id',None)
    h=getattr(row,'gasoducto_biogas_corto_id',None)
    direct_to_client=(
        h_direct is not None and str(h_direct) not in {'','None','nan'}
        and str(GDCORTO.loc[str(h_direct),'destino_tipo'])=='CLIENTE'
    )
    # Para un gasoducto corto BG->cliente se conserva también el recorrido previo
    # del biogás crudo hasta la BG concentradora. Un SHORTGASDIRECT hacia una
    # ubicación, en cambio, ya representa la fuente que llegó como GasBC.
    if direct_to_client or h_direct is None or str(h_direct) in {'','None','nan'}:
        if h is None or str(h) in {'','None','nan'}:
            add(getter(('X',row.empresa_id,getattr(row,'raw_source_site',getattr(row,'bg_production_site',row.bg_site)))))
    for _pre_ref in _pre_entry_transport_refs(row):
        add(getter(_pre_ref))
    if row.familia=='BioGNL' and not _conversion_uses_pipeline(row) and not _uses_local_ngi_transfer(row) and row.bg_site!=row.l_site:
        add(getter(('GBL',row.bg_site,row.l_site)))
    elif row.familia=='BioGNC' and not _conversion_uses_pipeline(row) and not _uses_local_ngi_transfer(row) and row.bg_site!=row.c_site:
        add(getter(('GBC',row.bg_site,row.c_site)))

    if row.familia=='BioGNL' and row.l_site!=row.rl_site:
        add(getter(('YL',row.l_site,row.rl_site)))
    elif row.familia=='BioGNC' and row.c_site!=row.rc_site:
        add(getter(('YC',row.c_site,row.rc_site)))

    if row.familia=='GasBC':
        h_direct=getattr(row,'gasoducto_gasbc_directo_id',None)
        direct_to_client=(
            h_direct is not None and str(h_direct) not in {'','None','nan'}
            and str(GDCORTO.loc[str(h_direct),'destino_tipo'])=='CLIENTE'
        )
        if not direct_to_client:
            add(getter(('ZD',row.bg_site,row.cliente_id)))
    elif row.familia=='BioGNL':
        if not direct_to_client:
            add(getter(('ZL',row.rl_site,row.cliente_id)))
    elif row.familia=='BioGNC':
        if not direct_to_client:
            add(getter(('ZC',row.rc_site,row.cliente_id)))
    elif row.familia=='Gasoducto':
        h_salida=getattr(row,'gasoducto_corto_salida_cliente_id',None)
        if h_salida is None or str(h_salida) in {'','None','nan'}:
            add(getter(('ZG',row.direccion_id,row.cliente_id)))
    return values

def route_transport_tour_ids(row):
    return _collect_route_transport(row,_tour_ids_for_flow_ref)

def route_transport_flow_refs(row):
    # Orden aguas-arriba -> aguas-abajo, igual que `_collect_route_transport`.
    return _collect_route_transport(row,lambda ref:[tuple(ref)])

def route_segments(row):
    seg=[]

    def add_keys(keys):
        for key in keys:
            if key not in seg:
                seg.append(key)

    # Transporte de biogás crudo desde la empresa hasta la planta BG.
    h_direct=getattr(row,'gasoducto_gasbc_directo_id',None)
    h=getattr(row,'gasoducto_biogas_corto_id',None)
    direct_to_client=(
        h_direct is not None and str(h_direct) not in {'','None','nan'}
        and str(GDCORTO.loc[str(h_direct),'destino_tipo'])=='CLIENTE'
    )
    if h is not None and str(h) not in {'','None','nan'}:
        add_keys([f'SHORTBIO|{h}'])
    elif direct_to_client or h_direct is None or str(h_direct) in {'','None','nan'}:
        add_keys(_segments_for_flow_ref(('X',row.empresa_id,getattr(row,'raw_source_site',getattr(row,'bg_production_site',row.bg_site)))))
    else:
        # GasBC que llegó desde otra BG a una ubicación posible.
        add_keys([f'SHORTGASDIRECT|{h_direct}'])

    # Relevos de biogás crudo entre ubicaciones de empresa/clientes antes de BG.
    add_keys(_raw_relay_segment_keys(row))

    # Relevos/transformaciones de GasBC que ocurrieron antes de que el producto
    # llegara al nodo bg_site desde el cual parte la ruta final actual. Si una
    # reconversión vehicular alimenta el corredor, ese viaje ocurre antes del
    # gasoducto convencional y debe aparecer antes en el HTML.
    for _hu in _hub_transport_usage_list(row):
        if not isinstance(_hu,dict):
            continue
        if str(_hu.get('kind',''))=='FLOW_REF':
            _ref=tuple(_hu.get('ref',()))
            if _ref:
                add_keys(_segments_for_flow_ref(_ref))
        elif str(_hu.get('kind',''))=='DIRECT_YG':
            add_keys([f"AIG|{str(_hu.get('bg'))}|{str(_hu.get('d'))}"])
    add_keys(_hub_static_segment_keys(row))

    # Etapas previas a la entrada cuando el GasBC se obtiene mediante
    # L/C remota -> vehículo GNL/GNC -> RL/RC en el extremo del corredor.
    for _pre_ref in _pre_entry_transport_refs(row):
        add_keys(_segments_for_flow_ref(_pre_ref))

    # Transporte intermedio de GasBC desde BG hacia L/C. Si L/C está en la
    # salida del corredor, el tramo intermedio es entrada al gasoducto + corredor;
    # la conversión GasBC->BioGNL/BioGNC ocurre directamente en ese nodo de salida.
    if _conversion_uses_pipeline(row):
        h_entrada=getattr(row,'gasoducto_corto_entrada_gasbc_id',None)
        if _uses_entry_reconversion(row):
            # RL/RC está exactamente en NGI_d: no existe tramo vial de GasBC.
            pass
        elif h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
            add_keys([f'SHORTGASIN|{h_entrada}'])
        else:
            add_keys([f"AIG|{row.bg_site}|{getattr(row,'direccion_inyeccion_id',row.direccion_id)}"])
        _intra_d=_clean_optional_id(getattr(row,'red_gasoductos_recorrido_interno_direccion',''))
        if _intra_d:
            add_keys([f'PIPEINTRA|{_intra_d}'])
        _gg_edges=list(getattr(row,'red_gasoductos_arcos',None) or [])
        add_keys([f'PIPEPAIR|{_a}|{_b}' for _a,_b in _gg_edges])
    elif row.familia=='BioGNL' and not _uses_local_ngi_transfer(row) and row.bg_site!=row.l_site:
        add_keys(_segments_for_flow_ref(('GBL',row.bg_site,row.l_site)))
    elif row.familia=='BioGNC' and not _uses_local_ngi_transfer(row) and row.bg_site!=row.c_site:
        add_keys(_segments_for_flow_ref(('GBC',row.bg_site,row.c_site)))

    # Transporte posterior de BioGNL/BioGNC.
    if row.familia=='BioGNL' and row.l_site!=row.rl_site:
        add_keys(_segments_for_flow_ref(('YL',row.l_site,row.rl_site)))
    elif row.familia=='BioGNC' and row.c_site!=row.rc_site:
        add_keys(_segments_for_flow_ref(('YC',row.c_site,row.rc_site)))
    elif row.familia=='Gasoducto':
        h_entrada=getattr(row,'gasoducto_corto_entrada_gasbc_id',None)
        if _uses_entry_reconversion(row):
            # RL/RC está exactamente en NGI_d: no existe tramo vial de GasBC.
            pass
        elif h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
            add_keys([f'SHORTGASIN|{h_entrada}'])
        else:
            add_keys([f"AIG|{row.bg_site}|{getattr(row,'direccion_inyeccion_id',row.direccion_id)}"])
        _intra_d=_clean_optional_id(getattr(row,'red_gasoductos_recorrido_interno_direccion',''))
        if _intra_d:
            add_keys([f'PIPEINTRA|{_intra_d}'])
        _gg_edges=list(getattr(row,'red_gasoductos_arcos',None) or [])
        add_keys([f'PIPEPAIR|{_a}|{_b}' for _a,_b in _gg_edges])

    # Transporte final de GasBC al cliente.
    if row.familia=='GasBC':
        direct_to_client=(
            h_direct is not None and str(h_direct) not in {'','None','nan'}
            and str(GDCORTO.loc[str(h_direct),'destino_tipo'])=='CLIENTE'
        )
        if direct_to_client:
            add_keys([f'SHORTGASDIRECT|{h_direct}'])
        else:
            add_keys(_segments_for_flow_ref(('ZD',row.bg_site,row.cliente_id)))
    elif row.familia=='BioGNL':
        if direct_to_client:
            add_keys([f'SHORTGASDIRECT|{h_direct}'])
        else:
            add_keys(_segments_for_flow_ref(('ZL',row.rl_site,row.cliente_id)))
    elif row.familia=='BioGNC':
        if direct_to_client:
            add_keys([f'SHORTGASDIRECT|{h_direct}'])
        else:
            add_keys(_segments_for_flow_ref(('ZC',row.rc_site,row.cliente_id)))
    elif row.familia=='Gasoducto':
        h_salida=getattr(row,'gasoducto_corto_salida_cliente_id',None)
        if h_salida is not None and str(h_salida) not in {'','None','nan'}:
            add_keys([f'SHORTGAS|{h_salida}'])
        else:
            add_keys(_segments_for_flow_ref(('ZG',row.direccion_id,row.cliente_id)))

    return seg

def gasbc_intermediate_segments(row):
    if _conversion_uses_pipeline(row):
        seg=[]
        h_entrada=getattr(row,'gasoducto_corto_entrada_gasbc_id',None)
        if _uses_entry_reconversion(row):
            modo=_entry_reconversion_mode(row)
            if modo.startswith('RECONVERSION_GNL') and str(row.bg_site)!=str(row.pre_l_site):
                seg.extend(_segments_for_flow_ref(('GBL',str(row.bg_site),str(row.pre_l_site))))
            elif modo.startswith('RECONVERSION_GNC') and str(row.bg_site)!=str(row.pre_c_site):
                seg.extend(_segments_for_flow_ref(('GBC',str(row.bg_site),str(row.pre_c_site))))
        elif h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
            seg.append(f'SHORTGASIN|{h_entrada}')
        else:
            seg.append(f"AIG|{row.bg_site}|{getattr(row,'direccion_inyeccion_id',row.direccion_id)}")
        _intra_d=_clean_optional_id(getattr(row,'red_gasoductos_recorrido_interno_direccion',''))
        if _intra_d:
            seg.append(f'PIPEINTRA|{_intra_d}')
        _gg_edges=list(getattr(row,'red_gasoductos_arcos',None) or [])
        seg.extend([f'PIPEPAIR|{_a}|{_b}' for _a,_b in _gg_edges])
        return seg
    if _uses_local_ngi_transfer(row):
        return []
    if row.familia=='BioGNL' and row.bg_site!=row.l_site:
        return _segments_for_flow_ref(('GBL',row.bg_site,row.l_site))
    if row.familia=='BioGNC' and row.bg_site!=row.c_site:
        return _segments_for_flow_ref(('GBC',row.bg_site,row.c_site))
    return []

active_road_keys=set()
def pre_entry_product_segments(row):
    modo=_entry_reconversion_mode(row)
    if modo.startswith('RECONVERSION_GNL') and str(row.pre_l_site)!=str(row.pre_rl_site):
        return _segments_for_flow_ref(('YL',str(row.pre_l_site),str(row.pre_rl_site)))
    if modo.startswith('RECONVERSION_GNC') and str(row.pre_c_site)!=str(row.pre_rc_site):
        return _segments_for_flow_ref(('YC',str(row.pre_c_site),str(row.pre_rc_site)))
    return []


if len(df_rutas):
    df_rutas['segmentos_gasbc_intermedio']=df_rutas.apply(gasbc_intermediate_segments,axis=1)
    df_rutas['segmentos_pre_reconversion']=df_rutas.apply(pre_entry_product_segments,axis=1)
    df_rutas['segmentos']=df_rutas.apply(route_segments,axis=1)
    df_rutas['transport_tour_ids']=df_rutas.apply(route_transport_tour_ids,axis=1)
    df_rutas['giras_reparto_asociadas']=df_rutas['transport_tour_ids'].map(
        lambda ids:'; '.join(map(str,ids)) if ids else 'Sin gira vial'
    )
    for segs in df_rutas['segmentos']:
        active_road_keys.update(s for s in segs if not (s.startswith('PIPE|') or s.startswith('SHORTBIO|') or s.startswith('SHORTBIOHUBEMP|') or s.startswith('SHORTBIOC2C|') or s.startswith('SHORTGASIN|') or s.startswith('SHORTGAS|') or s.startswith('SHORTGASDIRECT|')))
else:
    df_rutas['segmentos']=pd.Series(dtype=object)
    df_rutas['transport_tour_ids']=pd.Series(dtype=object)

# Giras reales ordenadas. El HTML las dibuja directamente, en lugar de inferir
# el recorrido del vehículo desde una descomposición contable por cliente.
transport_tours_payload=[]
_mode_segment_type={'BIO':'Biogas','BC_PROCESO':'GasBC','GNL':'BioGNL','GNC':'BioGNC','BC':'GasBC'}
if len(df_viajes_transporte):
    for _,tour_row in df_viajes_transporte.iterrows():
        tour_id=str(tour_row['transport_route_id'])
        arcs=(
            df_arcos_transporte.loc[
                df_arcos_transporte['transport_route_id'].astype(str).eq(tour_id)
            ]
            .sort_values('segmento_indice')
        )
        segment_type=_mode_segment_type.get(str(tour_row['modo']),'GasBC')
        decisions=(
            df_decisiones_transporte.loc[
                df_decisiones_transporte['transport_route_id'].astype(str).eq(tour_id)
            ]
            .sort_values('numero_orden_parada')
        )
        stop_decisions=[]
        for _,decision in decisions.iterrows():
            stop_decisions.append({
                'stop_number':int(decision['numero_orden_parada']),
                'stop_id':str(decision['parada_id']),
                'operation':str(decision['operacion_en_parada']),
                'incoming_load':float(decision['carga_entrada_parada']),
                'processed_load':float(decision['cantidad_transformada_o_entregada']),
                'remaining_load':float(decision['remanente_despues_parada']),
                'remaining_destination':str(decision['destino_del_remanente']),
                'remaining_action':str(decision['accion_del_remanente']),
            })
        segments=[]
        for _,arc in arcs.iterrows():
            key=str(arc['route_key'])
            segments.append({
                'key':key,
                'reverse':bool(arc['polilinea_invertida']),
                'segment_index':int(arc['segmento_indice']),
                'leg_type':str(arc['tipo_tramo']),
                'from':str(arc['desde']),
                'to':str(arc['hacia']),
                'distance_km':float(arc['distancia_km']),
                'load_nm3':float(arc['carga_transportada']),
                'segment_type':segment_type,
                'segment_label':f"{segment_type} · {arc['tipo_tramo']}",
            })
            if not key.startswith('PIPE|'):
                active_road_keys.add(key)
        stops=list(tour_row['paradas_ids']) if isinstance(tour_row['paradas_ids'],(list,tuple)) else [
            x.strip() for x in str(tour_row['secuencia_paradas']).split('->') if x.strip()
        ]
        associated_clients=[cliente_unico_de_parada(stop) for stop in stops]
        transport_tours_payload.append({
            'transport_route_id':tour_id,
            'network_id':str(tour_row['network_id']),
            'mode':str(tour_row['modo']),
            'vehicle':str(tour_row['vehiculo']),
            'origin_code':str(tour_row['origen_reparto']),
            'origin_name':str(tour_row['origen_nombre']),
            'stops':list(map(str,stops)),
            'associated_clients':associated_clients,
            'sequence':str(tour_row['origen_reparto'])+' → '+' → '.join(map(str,stops))+' → '+str(tour_row['origen_reparto']),
            'number_of_vehicles':float(tour_row['numero_viajes']),
            'distance_km':float(tour_row['distancia_por_viaje_km']),
            'cost_usd':float(tour_row['costo_por_viaje_usd']),
            'origin_load':float(tour_row['carga_salida_origen_QSEG']),
            'return_load':float(tour_row['carga_retorno_origen_QSEG']),
            'returns_empty':bool(tour_row['regreso_vacio']),
            'stop_decisions':stop_decisions,
            'segments':segments,
            'polyline_source':'Archivos_Datos_API del ZIP',
        })

poly_cache=load_polyline_cache()
for tour in transport_tours_payload:
    for seg in tour.get('segments', []):
        rec_poly = poly_cache.get(str(seg.get('key')), {})
        seg['polyline_source_key'] = str(rec_poly.get('cache_alias_source') or seg.get('key'))
        seg['uses_e2c'] = seg['polyline_source_key'].startswith('E2C|')
failed=[rk for rk in sorted(active_road_keys) if not (str(rk).startswith(('PIPEPAIR|','PIPEINTRA|','SHORTBIO|','SHORTBIOPHUB|','SHORTBIOHUBEMP|','SHORTBIOC2C|','SHORTGASIN|','SHORTGAS|','SHORTGASDIRECT|')) or poly_valid(poly_cache.get(rk),rk))]
if failed:
    raise RuntimeError('La solución usa segmentos sin polilínea local:\n'+'\n'.join(failed))
used_c2c=sorted(k for k in active_road_keys if str(k).startswith('C2C|'))
used_e2c_aliases=sorted(
    k for k in active_road_keys
    if str(poly_cache.get(k, {}).get('cache_alias_source', '')).startswith('E2C|')
)
print(f'Segmentos viales únicos de la solución: {len(active_road_keys):,}')
print(f'Trayectorias cliente-cliente C2C incluidas en las giras reales: {len(used_c2c):,}')
if used_c2c:
    print('Ejemplos C2C tomados del ZIP:',used_c2c[:10])
print(f'Entregas directas empresa-cliente que usan E2C en la solución: {len(used_e2c_aliases):,}')
if used_e2c_aliases:
    print('Aliases ABC con fuente E2C:', [
        (k, poly_cache[k].get('cache_alias_source')) for k in used_e2c_aliases[:10]
    ])


Segmentos viales únicos de la solución: 23
Trayectorias cliente-cliente C2C incluidas en las giras reales: 0
Entregas directas empresa-cliente que usan E2C en la solución: 1
Aliases ABC con fuente E2C: [('ASITE|NC_C4|NE_E3', 'E2C|E3|C4')]


# 10


- Auditoría `QGCD` compatible con tramos `SHORTGASDIRECT` intermedios/HUB.
- Cada cliente recibe sólo el prefijo físico de la gira que realmente lo abasteció.
- Si una fila económica usa más de una gira, el HTML la separa en rutas visuales independientes.
- BioGNL/BioGNC se determinan por el `mode` real de la gira; un `C2C` ya no se fuerza a GasBC.
- Colores: BioGNL violeta, BioGNC naranjo, GasBC verde, Biometano Gaseoso verde azulado, Gasoducto Convencional celeste y BioGás Crudo azul fuerte.


In [ ]:
# ============================================================
# 10. HTML GOOGLE MAPS CON VISUALIZADOR COMPLETO
#     - DATOS: parámetros, tablas y gráficos de entrada
#     - ESTADÍSTICAS: tarjetas numéricas, barplots, scatterplots y boxplots
# ============================================================

def clean_records(df, cols=None):
    if cols is not None:
        df=df[cols]
    return json.loads(df.replace({np.nan:None}).to_json(orient='records',force_ascii=False))

def escape_script_json(obj):
    return json.dumps(obj,ensure_ascii=False,default=str).replace('</','<\\/')

def _finite(v, default=0.0):
    try:
        x=float(v)
        return x if math.isfinite(x) else float(default)
    except Exception:
        return float(default)


def _clean_optional_id(value_obj):
    if value_obj is None:
        return ''
    if isinstance(value_obj, (list, tuple, set)):
        for item in value_obj:
            cleaned = _clean_optional_id(item)
            if cleaned:
                return cleaned
        return ''
    try:
        if pd.isna(value_obj):
            return ''
    except Exception:
        pass
    value_str = str(value_obj).strip()
    return '' if value_str.lower() in {'', 'none', 'nan', 'nat', 'null'} else value_str

def _safe_div(a,b):
    return _finite(a)/_finite(b) if abs(_finite(b))>EPS else 0.0

def _haversine_km(lat1,lon1,lat2,lon2):
    r=6371.0088
    p1,p2=math.radians(float(lat1)),math.radians(float(lat2))
    dp=math.radians(float(lat2)-float(lat1)); dl=math.radians(float(lon2)-float(lon1))
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*r*math.asin(min(1.0,math.sqrt(a)))

SITE_NAME=sitios.set_index('id')['nombre'].to_dict()
SITE_REGION=sitios.set_index('id')['region'].to_dict()
# Nombres exactos de los tipos de planta leídos exclusivamente desde el ZIP.
# Esto evita etiquetas genéricas y mantiene el visualizador sincronizado con
# `tipos_plantas.csv`.
PLANT_TYPE_NAME={t:str(PL.loc[t,'tipo_instalacion']) for t in T}
TYPE_NAME={t:f"{t} · {PLANT_TYPE_NAME[t]}" for t in T}
PALETTE=['#1565C0','#C62828','#2E7D32','#7B2CBF','#F57C00','#00838F','#AD1457','#5D4037','#455A64','#6A1B9A','#0277BD','#558B2F','#EF6C00','#4527A0','#00695C','#9E9D24','#D84315','#283593','#00897B','#8E24AA','#3949AB','#43A047','#FB8C00','#00ACC1']
company_colors={i:PALETTE[n%len(PALETTE)] for n,i in enumerate(I)}
client_colors={j:PALETTE[(n+6)%len(PALETTE)] for n,j in enumerate(J)}
type_colors={'BG':'#16A34A','L':'#7C3AED','RL':'#6D28D9','C':'#F97316','RC':'#EA580C','Gasoducto':'#0284C7'}

# -----------------------------------------------------------------------------
# Cargas, capacidades y costos directos de cada planta/gasoducto activo.
# -----------------------------------------------------------------------------
plant_load={}
for s in S:
    plant_load[('BG',s)]=value('QBIO_BG',s)
    plant_load[('L',s)]=value('QL',s)
    plant_load[('RL',s)]=value('YL0',s)+sum(value('YL',p,s) for p in ASITE_in[s])
    plant_load[('C',s)]=value('QC',s)
    plant_load[('RC',s)]=value('YC0',s)+sum(value('YC',p,s) for p in ASITE_in[s])
plant_units={(t,s):value('NP',t,s) for t in T for s in S}
pipe_flow={
    g:(
        sum(value('YG',s,d) for d in D_by_g[g] for s in AIG_by_d[d])
        +sum(value('QGCE',h) for h in H_GECORTO_by_g[g])
        +sum(value('YGG',h,g) for h in GG_in[g])
    )
    for g in G
}
short_biogas_pipe_flow={h:value('QBCP',h)+(value('QBCP_HUB',h) if h in H_EHUB_SET else 0.0) for h in H_CORTO}
short_biogas_hub_empresa_solo_flow={h:value('QBCP_HUB',h) for h in H_EHUB_SOLO}
short_biogas_c2c_pipe_flow={h:value('QBC2C',h) for h in H_BIOC2C}
short_biogas_p_hub_pipe_flow={h:value('QBCP_P',h) for h in H_PHUB}
short_gasbc_inlet_pipe_flow={h:value('QGCE',h) for h in H_GECORTO}
short_gasbc_direct_pipe_flow={h:(value('QGCD',h)+value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h)) for h in H_GDCORTO}
short_gasbc_pipe_flow={h:value('QGCP',h)+value('QGCP_HUB',h) for h in H_GSCORTO}

def _gdcorto_origen_label(h):
    return f"{GDCORTO.loc[h,'nodo_origen_id']} · {GDCORTO.loc[h,'origen_nombre']}"

producer_meta={}
for t in T:
    for s in S:
        load=_finite(plant_load[(t,s)])
        units=_finite(plant_units[(t,s)])
        if load>EPS or units>EPS:
            key=f'PLANT|{t}|{s}'
            capacity=units*_finite(PL.loc[t,'capacidad_max_nm3'])
            producer_meta[key]={
                'key':key,'kind':'Planta','type':t,'site':s,
                'label':f"{TYPE_NAME[t]} · {s} · {SITE_NAME[s]}",
                'short_label':f"{t} · {s}",
                'lat':_finite(sitios.set_index('id').loc[s,'lat']),
                'lon':_finite(sitios.set_index('id').loc[s,'lon']),
                'load':load,'units':units,'capacity':capacity,
                'fixed_cost':units*_finite(PL.loc[t,'costo_fijo_usd']),
                'variable_cost':load*_finite(PL.loc[t,'costo_variable_usd_nm3']),
                'variable_rate':_finite(PL.loc[t,'costo_variable_usd_nm3']),
                'color':type_colors[t],
            }
for g in G:
    flow=_finite(pipe_flow[g]); active=_finite(value('BG',g))
    if flow>EPS or active>EPS:
        key=f'PIPE|{g}'
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto','site':g,
            'label':f"Gasoducto · {g} · {GA.loc[g,'nombre']}",
            'short_label':f"Gasoducto · {g}",
            'lat':(_finite(GA.loc[g,'punto_a_lat'])+_finite(GA.loc[g,'punto_b_lat']))/2,'lon':(_finite(GA.loc[g,'punto_a_lon'])+_finite(GA.loc[g,'punto_b_lon']))/2,
            'load':flow,'units':active,'capacity':_finite(GA.loc[g,'capacidad_efectiva_nm3']),
            'fixed_cost':active*_finite(GA.loc[g,'costo_fijo_usd_dia']),
            # El costo variable longitudinal A/B se registra en PIPEINTRA|d;
            # PIPE conserva la atribucion fija del activo fisico.
            'variable_cost':0.0,
            'variable_rate':0.0,
            'color':type_colors['Gasoducto'],
        }
for d in D:
    flow=_finite(value('YGINTRA',d))
    if flow>EPS:
        g=str(DG[d])
        key=f'PIPEINTRA|{d}'
        dist=_finite(DIST_INTRA_GASODUCTO_KM[g])
        rate=_finite(GA.loc[g,'costo_variable_usd_km_nm3'])*dist
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Transferencia Gasoducto','subtype':'gasoducto_interno',
            'site':f'{g}:{d}',
            'label':f"Gasoducto convencional interno · {d}",
            'short_label':f"{d} interno",
            'lat':(_finite(GD.loc[d,'entrada_lat'])+_finite(GD.loc[d,'salida_lat']))/2,
            'lon':(_finite(GD.loc[d,'entrada_lon'])+_finite(GD.loc[d,'salida_lon']))/2,
            'load':flow,'units':1.0,'capacity':flow,
            'fixed_cost':0.0,'variable_cost':flow*rate,'variable_rate':rate,
            'color':type_colors['Gasoducto'],
        }
for g1,g2 in GG_DIR:
    flow=_finite(value('YGG',g1,g2))
    if flow>EPS:
        key=f'PIPEPAIR|{g1}|{g2}'
        dist=_finite(DIST_GG[(g1,g2)])
        rate=_finite(COSTO_GG_USD_KM_NM3)*dist
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Transferencia Gasoducto','subtype':'gasoducto_gasoducto',
            'site':f'{g1}->{g2}',
            'label':f"Gasoducto convencional · {g1} → {g2}",
            'short_label':f"{g1} → {g2}",
            'lat':(_finite(GG_REF_COORD[(g1,g2)][0][0])+_finite(GG_REF_COORD[(g1,g2)][1][0]))/2,
            'lon':(_finite(GG_REF_COORD[(g1,g2)][0][1])+_finite(GG_REF_COORD[(g1,g2)][1][1]))/2,
            'load':flow,'units':1.0,'capacity':flow,
            'fixed_cost':0.0,'variable_cost':flow*rate,'variable_rate':rate,
            'color':type_colors['Gasoducto'],
        }
for h in H_CORTO:
    flow=_finite(short_biogas_pipe_flow[h]); active=_finite(value('BCP',h))
    if flow>EPS or active>EPS:
        key=f'SHORTBIO|{h}'
        destino_tipo=str(GCORTO.loc[h,'destino_tipo'])
        destino_texto=(
            'cliente' if destino_tipo=='CLIENTE' else (
                'entrada de gasoducto' if destino_tipo=='ENTRADA_GASODUCTO'
                else (
                    'ubicación de empresa' if destino_tipo=='UBICACION_EMPRESA'
                    else 'ubicación posible'
                )
            )
        )
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto Biogás Crudo','site':h,
            'subtype':'biogas_crudo_corto',
            'destination_type':destino_tipo,
            'destination_id':str(GCORTO.loc[h,'destino_id']),
            'label':f"Gasoducto corto de biogás crudo · {GCORTO.loc[h,'empresa_id']} → {destino_texto} {GCORTO.loc[h,'destino_id']}",
            'short_label':f"Gasoducto corto · {GCORTO.loc[h,'empresa_id']} → {GCORTO.loc[h,'destino_id']}",
            'lat':(_finite(GCORTO.loc[h,'punto_a_lat'])+_finite(GCORTO.loc[h,'punto_b_lat']))/2,
            'lon':(_finite(GCORTO.loc[h,'punto_a_lon'])+_finite(GCORTO.loc[h,'punto_b_lon']))/2,
            'load':flow,'units':active,
            'capacity':_finite(GCORTO.loc[h,'capacidad_max_nm3_dia']),
            'fixed_cost':active*_finite(GCORTO.loc[h,'costo_fijo_usd_dia']),
            'variable_cost':(
                flow*_finite(GCORTO.loc[h,'costo_variable_usd_nm3_km'])
                *_finite(GCORTO.loc[h,'distancia_recta_km'])
            ),
            'variable_rate':(
                _finite(GCORTO.loc[h,'costo_variable_usd_nm3_km'])
                *_finite(GCORTO.loc[h,'distancia_recta_km'])
            ),
            'color':'#0E7490',
        }
for h in H_BIOC2C:
    flow=_finite(short_biogas_c2c_pipe_flow[h]); active=_finite(value('BCP2C',h))
    if flow>EPS or active>EPS:
        key=f'SHORTBIOC2C|{h}'
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto Biogás Crudo HUB','site':h,
            'subtype':'biogas_crudo_cliente_cliente','destination_type':'CLIENTE',
            'destination_id':str(BIOC2C.loc[h,'cliente_destino_id']),
            'label':f"Gasoducto corto biogás crudo HUB · cliente {BIOC2C.loc[h,'cliente_origen_id']} → cliente {BIOC2C.loc[h,'cliente_destino_id']}",
            'short_label':f"Biogás HUB · {BIOC2C.loc[h,'cliente_origen_id']} → {BIOC2C.loc[h,'cliente_destino_id']}",
            'lat':(_finite(BIOC2C.loc[h,'punto_a_lat'])+_finite(BIOC2C.loc[h,'punto_b_lat']))/2,
            'lon':(_finite(BIOC2C.loc[h,'punto_a_lon'])+_finite(BIOC2C.loc[h,'punto_b_lon']))/2,
            'load':flow,'units':active,'capacity':_finite(BIOC2C.loc[h,'capacidad_max_nm3_dia']),
            'fixed_cost':active*_finite(BIOC2C.loc[h,'costo_fijo_usd_dia']),
            'variable_cost':flow*_finite(BIOC2C.loc[h,'costo_variable_usd_nm3_km'])*_finite(BIOC2C.loc[h,'distancia_recta_km']),
            'variable_rate':_finite(BIOC2C.loc[h,'costo_variable_usd_nm3_km'])*_finite(BIOC2C.loc[h,'distancia_recta_km']),
            'color':'#0F766E',
        }

for h in H_PHUB:
    flow=_finite(short_biogas_p_hub_pipe_flow[h]); active=_finite(value('BCP_P',h))
    if flow>EPS or active>EPS:
        key=f'SHORTBIOPHUB|{h}'
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto Biogás Crudo HUB P','site':h,
            'subtype':'biogas_crudo_hub_nodal','destination_type':str(PHUB.loc[h,'destino_tipo']),
            'destination_id':str(PHUB.loc[h,'destino_id']),
            'label':f"Gasoducto corto biogás crudo HUB · {PHUB.loc[h,'origen_id']} → {PHUB.loc[h,'destino_id']}",
            'short_label':f"Biogás HUB · {PHUB.loc[h,'origen_id']} → {PHUB.loc[h,'destino_id']}",
            'lat':(_finite(PHUB.loc[h,'punto_a_lat'])+_finite(PHUB.loc[h,'punto_b_lat']))/2,
            'lon':(_finite(PHUB.loc[h,'punto_a_lon'])+_finite(PHUB.loc[h,'punto_b_lon']))/2,
            'load':flow,'units':active,'capacity':_finite(PHUB.loc[h,'capacidad_max_nm3_dia']),
            'fixed_cost':active*_finite(PHUB.loc[h,'costo_fijo_usd_dia']),
            'variable_cost':flow*_finite(PHUB.loc[h,'costo_variable_usd_nm3_km'])*_finite(PHUB.loc[h,'distancia_recta_km']),
            'variable_rate':_finite(PHUB.loc[h,'costo_variable_usd_nm3_km'])*_finite(PHUB.loc[h,'distancia_recta_km']),
            'color':'#0F766E',
        }

for h in H_EHUB_SOLO:
    flow=_finite(short_biogas_hub_empresa_solo_flow[h]); active=_finite(value('BCP_HUB',h))
    if flow>EPS or active>EPS:
        key=f'SHORTBIOHUBEMP|{h}'
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto Biogás Crudo','site':h,
            'subtype':'biogas_crudo_hub_empresa',
            'destination_type':str(EHUB.loc[h,'destino_tipo']),'destination_id':str(EHUB.loc[h,'destino_id']),
            'label':f"Gasoducto corto HUB biogás crudo · {EHUB.loc[h,'empresa_id']} → {EHUB.loc[h,'destino_id']}",
            'short_label':f"HUB empresa · {EHUB.loc[h,'empresa_id']} → {EHUB.loc[h,'destino_id']}",
            'lat':(_finite(EHUB.loc[h,'punto_a_lat'])+_finite(EHUB.loc[h,'punto_b_lat']))/2,
            'lon':(_finite(EHUB.loc[h,'punto_a_lon'])+_finite(EHUB.loc[h,'punto_b_lon']))/2,
            'load':flow,'units':active,'capacity':_finite(EHUB.loc[h,'capacidad_max_nm3_dia']),
            'fixed_cost':active*_finite(EHUB.loc[h,'costo_fijo_usd_dia']),
            'variable_cost':flow*_finite(EHUB.loc[h,'costo_variable_usd_nm3_km'])*_finite(EHUB.loc[h,'distancia_recta_km']),
            'variable_rate':_finite(EHUB.loc[h,'costo_variable_usd_nm3_km'])*_finite(EHUB.loc[h,'distancia_recta_km']),
            'color':type_colors['Gasoducto'],
        }
for h in H_GECORTO:
    flow=_finite(short_gasbc_inlet_pipe_flow[h]); active=_finite(value('BGCE',h))
    if flow>EPS or active>EPS:
        key=f'SHORTGASIN|{h}'
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto Corto GasBC Entrada','site':h,
            'subtype':'gasbc_corto_entrada',
            'label':f"Gasoducto corto de GasBC · BG@{GECORTO.loc[h,'empresa_id']} → entrada {GECORTO.loc[h,'direccion_id']}",
            'short_label':f"Corto GasBC entrada · {GECORTO.loc[h,'empresa_id']} → {GECORTO.loc[h,'direccion_id']}",
            'lat':(_finite(GECORTO.loc[h,'punto_a_lat'])+_finite(GECORTO.loc[h,'punto_b_lat']))/2,
            'lon':(_finite(GECORTO.loc[h,'punto_a_lon'])+_finite(GECORTO.loc[h,'punto_b_lon']))/2,
            'load':flow,'units':active,'capacity':_finite(GECORTO.loc[h,'capacidad_max_nm3_dia']),
            'fixed_cost':active*_finite(GECORTO.loc[h,'costo_fijo_usd_dia']),
            'variable_cost':flow*_finite(GECORTO.loc[h,'costo_variable_usd_nm3_km'])*_finite(GECORTO.loc[h,'distancia_recta_km']),
            'variable_rate':_finite(GECORTO.loc[h,'costo_variable_usd_nm3_km'])*_finite(GECORTO.loc[h,'distancia_recta_km']),
            'color':'#2563EB',
        }

for h in H_GDCORTO:
    flow=_finite(short_gasbc_direct_pipe_flow[h]); active=_finite(value('BGCD',h))
    if flow>EPS or active>EPS:
        key=f'SHORTGASDIRECT|{h}'
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto Corto GasBC Directo','site':h,
            'subtype':'gasbc_corto_directo',
            'destination_type':str(GDCORTO.loc[h,'destino_tipo']),
            'destination_id':str(GDCORTO.loc[h,'destino_id']),
            'label':f"Gasoducto corto directo de GasBC · {_gdcorto_origen_label(h)} → {GDCORTO.loc[h,'destino_tipo']} {GDCORTO.loc[h,'destino_id']}",
            'short_label':f"Corto GasBC directo · {GDCORTO.loc[h,'nodo_origen_id']} → {GDCORTO.loc[h,'destino_id']}",
            'lat':(_finite(GDCORTO.loc[h,'punto_a_lat'])+_finite(GDCORTO.loc[h,'punto_b_lat']))/2,
            'lon':(_finite(GDCORTO.loc[h,'punto_a_lon'])+_finite(GDCORTO.loc[h,'punto_b_lon']))/2,
            'load':flow,'units':active,'capacity':_finite(GDCORTO.loc[h,'capacidad_max_nm3_dia']),
            'fixed_cost':active*_finite(GDCORTO.loc[h,'costo_fijo_usd_dia']),
            'variable_cost':flow*_finite(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])*_finite(GDCORTO.loc[h,'distancia_recta_km']),
            'variable_rate':_finite(GDCORTO.loc[h,'costo_variable_usd_nm3_km'])*_finite(GDCORTO.loc[h,'distancia_recta_km']),
            'color':'#2563EB',
        }
def _gscorto_destino_label(h):
    tipo=str(GSCORTO.loc[h,'destino_tipo'])
    destino_id=str(GSCORTO.loc[h,'destino_id'])
    if tipo=='CLIENTE':
        return f'cliente {destino_id}'
    if tipo=='UBICACION_POSIBLE':
        return f'ubicación posible {destino_id}'
    if tipo=='UBICACION_EMPRESA':
        return f'ubicación empresa {destino_id}'
    return f'{tipo} {destino_id}'

for h in H_GSCORTO:
    flow=_finite(short_gasbc_pipe_flow[h]); active=_finite(value('BGCP',h))
    if flow>EPS or active>EPS:
        key=f'SHORTGAS|{h}'
        producer_meta[key]={
            'key':key,'kind':'Gasoducto','type':'Gasoducto Corto GasBC Salida','site':h,
            'subtype':'gasbc_corto_salida',
            'label':f"Gasoducto corto de GasBC · salida {GSCORTO.loc[h,'direccion_id']} → {_gscorto_destino_label(h)}",
            'short_label':f"Corto GasBC salida · {GSCORTO.loc[h,'direccion_id']} → {_gscorto_destino_label(h)}",
            'lat':(_finite(GSCORTO.loc[h,'punto_a_lat'])+_finite(GSCORTO.loc[h,'punto_b_lat']))/2,
            'lon':(_finite(GSCORTO.loc[h,'punto_a_lon'])+_finite(GSCORTO.loc[h,'punto_b_lon']))/2,
            'load':flow,'units':active,'capacity':_finite(GSCORTO.loc[h,'capacidad_max_nm3_dia']),
            'fixed_cost':active*_finite(GSCORTO.loc[h,'costo_fijo_usd_dia']),
            'variable_cost':flow*_finite(GSCORTO.loc[h,'costo_variable_usd_nm3_km'])*_finite(GSCORTO.loc[h,'distancia_recta_km']),
            'variable_rate':_finite(GSCORTO.loc[h,'costo_variable_usd_nm3_km'])*_finite(GSCORTO.loc[h,'distancia_recta_km']),
            'color':'#1D4ED8',
        }
producer_keys=sorted(producer_meta,key=lambda k:(producer_meta[k]['kind']=='Gasoducto',producer_meta[k]['type'],producer_meta[k]['site']))
producer_colors={k:PALETTE[(n+3)%len(PALETTE)] if producer_meta[k]['kind']=='Planta' else '#212121' for n,k in enumerate(producer_keys)}

# -----------------------------------------------------------------------------
# Economía contable por ruta reconstruida.
#
# BENEFICIO POR RUTA = INGRESOS POR VENTA - COSTOS VARIABLES
#                      - COSTOS FIJOS PRORRATEADOS.
#
# Cada costo fijo se distribuye entre el universo completo de rutas que usa el
# mismo activo. La fracción de una ruta es:
#
#        flujo de salida del activo usado por la ruta
#   ----------------------------------------------------
#       flujo de salida total de todas sus rutas
#
# Para empresas se usa biogás crudo efectivamente suministrado. Para cada planta
# se usa el producto que sale de esa planta (GasBC, BioGNL o BioGNC, según tipo).
# El costo fijo efectivo de empresa usa el costo original de los datos.
# -----------------------------------------------------------------------------
def _flow_transport_alloc(flow_ref,q):
    """Atribuye a una ruta final la parte del transporte usada por un flujo."""
    ref=tuple(flow_ref)
    total=max(0.0,_finite(value(*ref)))
    share=_safe_div(q,total)
    return (
        _finite(transport_cost_by_flow_ref.get(ref,0.0))*share,
        _finite(transport_trips_by_flow_ref.get(ref,0.0))*share,
        _finite(transport_distance_by_flow_ref.get(ref,0.0))*share,
    )

def _direct_trip_alloc(prefix,args,flow_prefix,flow_args,vehicle,route_key,q):
    """Atribución para el traslado directo GasBC -> entrada de gasoducto."""
    total=_finite(value(flow_prefix,*flow_args))
    trips=_finite(value(prefix,*args))
    share=_safe_div(q,total)
    return (
        _finite(trip_cost(vehicle,route_key))*trips*share,
        trips*share,
        2.0*_finite(DIST.get(route_key,0.0))*trips*share,
    )

# Biogás crudo total que sale de cada empresa hacia todas sus rutas y clientes.
# El costo fijo de la empresa se distribuye solo para reportar rutas; dentro de
# cada sistema independiente se contabiliza una única vez, sin duplicarlo.
_company_raw_total=defaultdict(float)
if len(df_rutas):
    for _,_rr_fixed in df_rutas.iterrows():
        _i_fixed=str(_rr_fixed['empresa_id'])
        _raw_fixed=max(0.0,_finite(
            _rr_fixed.get(
                'biogas_crudo_entrada_BG_nm3',
                _rr_fixed.get('cantidad_nm3',0.0),
            )
        ))
        _company_raw_total[_i_fixed]+=_raw_fixed

def _fixed_company_alloc(i,raw_q):
    fixed_effective=(
        _finite(E.loc[i,'costo_activacion_biogas_usd'])
        *_finite(value('BE',i))
    )
    raw_q=max(0.0,_finite(raw_q))
    company_raw=_finite(_company_raw_total.get(i,0.0))
    if company_raw>EPS:
        return fixed_effective*_safe_div(raw_q,company_raw)
    supply=(
        sum(max(0.0,_finite(value('X',i,s))) for s in ABIO_by_i[i])
        +sum(max(0.0,_finite(value('QBCP',h))) for h in H_CORTO_by_i[i])
    )
    return fixed_effective*_safe_div(raw_q,supply)

def _route_producer_keys(rec):
    fam=str(rec['familia']); bg=str(rec['bg_site'])
    bg_production=str(rec.get('bg_production_site') or bg)
    keys=[f'PLANT|BG|{bg_production}']
    short_h=rec.get('gasoducto_biogas_corto_id')
    if short_h is not None and str(short_h) not in {'','None','nan'}:
        keys.append(f'SHORTBIO|{short_h}')
    for _h in (rec.get('gasoductos_biogas_relevo_empresa_ids',[]) or []):
        _h=str(_h)
        keys.append(f'SHORTBIO|{_h}' if _h in H_CORTO_SET else f'SHORTBIOHUBEMP|{_h}')
    for _h in (rec.get('gasoductos_biogas_relevo_cliente_ids',[]) or []):
        keys.append(f'SHORTBIOC2C|{str(_h)}')
    for _h in (rec.get('gasoductos_biogas_relevo_P_ids',[]) or []):
        keys.append(f'SHORTBIOPHUB|{str(_h)}')
    direct_h=rec.get('gasoducto_gasbc_directo_id')
    if direct_h is not None and str(direct_h) not in {'','None','nan'}:
        keys.append(f'SHORTGASDIRECT|{direct_h}')
    if rec.get('pre_l_site') not in (None,'','None','nan'):
        keys += [f"PLANT|L|{rec['pre_l_site']}",f"PLANT|RL|{rec['pre_rl_site']}"]
    elif rec.get('pre_c_site') not in (None,'','None','nan'):
        keys += [f"PLANT|C|{rec['pre_c_site']}",f"PLANT|RC|{rec['pre_rc_site']}"]
    if fam=='BioGNL':
        keys += [f"PLANT|L|{rec['l_site']}",f"PLANT|RL|{rec['rl_site']}"]
    elif fam=='BioGNC':
        keys += [f"PLANT|C|{rec['c_site']}",f"PLANT|RC|{rec['rc_site']}"]

    _hub_usage=rec.get('hub_producer_usage',{})
    if isinstance(_hub_usage,dict):
        keys += [str(k) for k in _hub_usage.keys()]

    # El corredor puede ser un activo intermedio tanto de una ruta Gasoducto
    # directa como de una ruta BioGNL/BioGNC cuya L/C está en la salida.
    gas_id=rec.get('gasoducto_id')
    if gas_id is not None and str(gas_id) not in {'','None','nan'}:
        h_entrada=rec.get('gasoducto_corto_entrada_gasbc_id')
        if h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
            keys.append(f'SHORTGASIN|{h_entrada}')
        _path_ids=list(rec.get('red_gasoductos_ids') or [str(gas_id)])
        _path_edges=list(rec.get('red_gasoductos_arcos') or [])
        _path_intra=_clean_optional_id(rec.get('red_gasoductos_recorrido_interno_direccion'))
        # PIPE mantiene sólo la atribución económica/fija de los gasoductos usados.
        keys += [f'PIPE|{_g}' for _g in _path_ids]
        # PIPEINTRA representa el recorrido longitudinal A/B dentro del mismo gasoducto.
        if _path_intra:
            keys.append(f'PIPEINTRA|{_path_intra}')
        # PIPEPAIR representa la única conexión convencional DIRECTA origen->destino.
        keys += [f'PIPEPAIR|{_a}|{_b}' for _a,_b in _path_edges]

    if fam=='Gasoducto':
        h_salida=rec.get('gasoducto_corto_salida_cliente_id')
        if h_salida is not None and str(h_salida) not in {'','None','nan'}:
            keys.append(f'SHORTGAS|{h_salida}')
    return list(dict.fromkeys(k for k in keys if k in producer_meta))

def _route_input_for_producer_base(rec,pk):
    # Base física para el costo VARIABLE de transformación/transporte.
    pm=producer_meta[pk]
    if pm['kind']=='Gasoducto':
        if pm.get('subtype') in {'biogas_crudo_corto','biogas_crudo_hub_empresa','biogas_crudo_cliente_cliente','biogas_crudo_hub_p'}:
            return _finite(rec.get('biogas_crudo_entrada_BG_nm3',0.0))
        if pm.get('subtype')=='gasbc_corto_directo':
            if bool(rec.get('post_reconversion_gasoducto_corto',False)):
                return _finite(rec.get('cantidad_nm3',0.0))
            return _finite(rec.get('gasbc_entrada_ruta_actual_nm3',rec.get('gasbc_salida_BG_nm3',rec.get('cantidad_nm3',0.0))))
        if pm.get('subtype') in {'gasoducto_gasoducto','gasoducto_interno'}:
            return _finite(rec.get('gasbc_entrada_corredor_nm3',rec.get('cantidad_nm3',0.0)))
        if pm.get('type')=='Gasoducto':
            return _finite(rec.get('gasbc_entrada_corredor_nm3',rec.get('cantidad_nm3',0.0)))
        return _finite(rec.get('cantidad_nm3',0.0))
    t=pm['type']; site=str(pm['site'])
    if t=='BG':
        return max(0.0,_finite(rec.get('biogas_crudo_entrada_BG_nm3',0.0)))
    if t=='L':
        if str(rec.get('pre_l_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_entrada_L_pre_nm3',0.0)))
        if str(rec.get('l_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_entrada_L_nm3',0.0)))
    if t=='RL':
        if str(rec.get('pre_rl_site'))==site:
            return max(0.0,_finite(rec.get('biognl_entrada_RL_pre_nm3',0.0)))
        if str(rec.get('rl_site'))==site:
            return max(0.0,_finite(rec.get('biognl_entrada_RL_nm3',0.0)))
    if t=='C':
        if str(rec.get('pre_c_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_entrada_C_pre_nm3',0.0)))
        if str(rec.get('c_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_entrada_C_nm3',0.0)))
    if t=='RC':
        if str(rec.get('pre_rc_site'))==site:
            return max(0.0,_finite(rec.get('biognc_entrada_RC_pre_nm3',0.0)))
        if str(rec.get('rc_site'))==site:
            return max(0.0,_finite(rec.get('biognc_entrada_RC_nm3',0.0)))
    return 0.0


def _route_output_for_producer_base(rec,pk):
    # Base para repartir costo FIJO: producto que sale del activo por esta ruta.
    pm=producer_meta[pk]
    if pm['kind']=='Gasoducto':
        if pm.get('subtype') in {'biogas_crudo_corto','biogas_crudo_hub_empresa','biogas_crudo_cliente_cliente','biogas_crudo_hub_p'}:
            return max(0.0,_finite(rec.get('biogas_crudo_entrada_BG_nm3',0.0)))
        if pm.get('subtype')=='gasbc_corto_directo':
            if bool(rec.get('post_reconversion_gasoducto_corto',False)):
                return max(0.0,_finite(rec.get('cantidad_nm3',0.0)))
            return max(0.0,_finite(rec.get('gasbc_entrada_ruta_actual_nm3',rec.get('gasbc_salida_BG_nm3',rec.get('cantidad_nm3',0.0)))))
        if pm.get('subtype') in {'gasoducto_gasoducto','gasoducto_interno'}:
            return max(0.0,_finite(rec.get('gasbc_entrada_corredor_nm3',rec.get('cantidad_nm3',0.0))))
        if pm.get('type')=='Gasoducto':
            return max(0.0,_finite(rec.get('gasbc_entrada_corredor_nm3',rec.get('cantidad_nm3',0.0))))
        return max(0.0,_finite(rec.get('cantidad_nm3',0.0)))
    t=pm['type']; site=str(pm['site'])
    if t=='BG':
        return max(0.0,_finite(rec.get('gasbc_salida_BG_nm3',0.0)))
    if t=='L':
        if str(rec.get('pre_l_site'))==site:
            return max(0.0,_finite(rec.get('biognl_salida_L_pre_nm3',0.0)))
        if str(rec.get('l_site'))==site:
            return max(0.0,_finite(rec.get('biognl_salida_L_nm3',0.0)))
    if t=='RL':
        if str(rec.get('pre_rl_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_salida_RL_pre_nm3',0.0)))
        if str(rec.get('rl_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_salida_RL_nm3',0.0)))
    if t=='C':
        if str(rec.get('pre_c_site'))==site:
            return max(0.0,_finite(rec.get('biognc_salida_C_pre_nm3',0.0)))
        if str(rec.get('c_site'))==site:
            return max(0.0,_finite(rec.get('biognc_salida_C_nm3',0.0)))
    if t=='RC':
        if str(rec.get('pre_rc_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_salida_RC_pre_nm3',0.0)))
        if str(rec.get('rc_site'))==site:
            return max(0.0,_finite(rec.get('gasbc_salida_RC_nm3',0.0)))
    return 0.0

def _hub_usage_value(rec,pk,field):
    usage=rec.get('hub_producer_usage',{})
    if not isinstance(usage,dict): return 0.0
    row=usage.get(str(pk),{})
    if not isinstance(row,dict): return 0.0
    return max(0.0,_finite(row.get(field,0.0)))


def _route_input_for_producer(rec,pk):
    # Si el activo quedó registrado explícitamente en hub_producer_usage, esa
    # cantidad es la fuente de verdad para ESTA ruta. No se suma nuevamente la
    # representación base/canónica del mismo activo, porque eso duplicaba el flujo
    # de corredores y gasoductos cortos heredados (p. ej. 30.000 -> 60.000).
    hub=_hub_usage_value(rec,pk,'input')
    if hub>EPS:
        return hub
    return _route_input_for_producer_base(rec,pk)


def _route_output_for_producer(rec,pk):
    # Mismo criterio para la salida usada en el prorrateo contable de costos fijos.
    hub=_hub_usage_value(rec,pk,'output')
    if hub>EPS:
        return hub
    return _route_output_for_producer_base(rec,pk)


# Denominador del prorrateo fijo: salida total de TODAS las rutas que usan cada
# productora/gasoducto, incluyendo rutas hacia clientes diferentes.
_producer_route_output_total=defaultdict(float)
if len(df_rutas):
    for _,_rr_prod in df_rutas.iterrows():
        _rec_prod=_rr_prod.to_dict()
        for _pk_prod in _route_producer_keys(_rec_prod):
            _producer_route_output_total[_pk_prod]+=_route_output_for_producer(_rec_prod,_pk_prod)

def _entry_reconversion_transport_alloc(rec):
    modo=str(rec.get('entrada_corredor_modo',''))
    total_cost=total_trips=total_dist=0.0
    if modo.startswith('RECONVERSION_GNL'):
        pre_l=str(rec['pre_l_site']); pre_rl=str(rec['pre_rl_site']); bg=str(rec['bg_site'])
        if bg!=pre_l:
            c,t,d=_flow_transport_alloc(('GBL',bg,pre_l),_finite(rec.get('gasbc_entrada_L_pre_nm3',0.0)))
            total_cost+=c; total_trips+=t; total_dist+=d
        if pre_l!=pre_rl:
            c,t,d=_flow_transport_alloc(('YL',pre_l,pre_rl),_finite(rec.get('biognl_entrada_RL_pre_nm3',0.0)))
            total_cost+=c; total_trips+=t; total_dist+=d
    elif modo.startswith('RECONVERSION_GNC'):
        pre_c=str(rec['pre_c_site']); pre_rc=str(rec['pre_rc_site']); bg=str(rec['bg_site'])
        if bg!=pre_c:
            c,t,d=_flow_transport_alloc(('GBC',bg,pre_c),_finite(rec.get('gasbc_entrada_C_pre_nm3',0.0)))
            total_cost+=c; total_trips+=t; total_dist+=d
        if pre_c!=pre_rc:
            c,t,d=_flow_transport_alloc(('YC',pre_c,pre_rc),_finite(rec.get('biognc_entrada_RC_pre_nm3',0.0)))
            total_cost+=c; total_trips+=t; total_dist+=d
    return total_cost,total_trips,total_dist


route_rows=[]
if len(df_rutas):
    for _,row in df_rutas.iterrows():
        rec=row.to_dict(); q=_finite(rec['cantidad_nm3'])
        i=str(rec['empresa_id']); j=str(rec['cliente_id']); bg=str(rec['bg_site']); fam=str(rec['familia'])
        bg_production=str(rec.get('bg_production_site') or bg)
        revenue_route=q*_finite(precio_promedio_venta_cliente.get(j, CL.loc[j,'precio_venta_usd_nm3']))
        raw_q=_finite(rec.get('biogas_crudo_entrada_BG_nm3',q))
        company_var_route=raw_q*_finite(precio_promedio_compra_empresa.get(i, E.loc[i,COL_PRECIO_BIOGAS_CRUDO]))
        company_fixed_route=_fixed_company_alloc(i,raw_q)
        short_h=rec.get('gasoducto_biogas_corto_id')
        direct_h=rec.get('gasoducto_gasbc_directo_id')
        if (
            fam=='GasBC'
            and direct_h is not None and str(direct_h) not in {'','None','nan'}
        ):
            direct_h=str(direct_h)
            transport_route,round_trips_route,road_distance=0.0,0.0,0.0
            internal_pipe_distance=_finite(GDCORTO.loc[direct_h,'distancia_recta_km'])
        elif short_h is not None and str(short_h) not in {'','None','nan'}:
            short_h=str(short_h)
            transport_route,round_trips_route,road_distance=0.0,0.0,0.0
            internal_pipe_distance=_finite(GCORTO.loc[short_h,'distancia_recta_km'])
        else:
            _raw_source_site=str(rec.get('raw_source_site') or bg_production)
            transport_route,round_trips_route,road_distance=_flow_transport_alloc(
                ('X',i,_raw_source_site),raw_q
            )
            internal_pipe_distance=0.0

        # Distancia física de todos los relevos de biogás crudo heredados por la ruta.
        for _h in (rec.get('gasoductos_biogas_relevo_empresa_ids',[]) or []):
            _h=str(_h)
            _tbl=GCORTO if _h in H_CORTO_SET else EHUB
            internal_pipe_distance+=_finite(_tbl.loc[_h,'distancia_recta_km'])
        for _h in (rec.get('gasoductos_biogas_relevo_cliente_ids',[]) or []):
            internal_pipe_distance+=_finite(BIOC2C.loc[str(_h),'distancia_recta_km'])
        for _h in (rec.get('gasoductos_biogas_relevo_P_ids',[]) or []):
            internal_pipe_distance+=_finite(PHUB.loc[str(_h),'distancia_recta_km'])

        # Tramos previos de GasBC que atravesaron uno o varios HUBs antes de
        # comenzar la ruta final actual. Sus costos ya existen en las variables
        # del MILP; aquí sólo se atribuye la fracción correspondiente a esta ruta.
        _hub_transport=rec.get('hub_transport_usage',[])
        if isinstance(_hub_transport,(list,tuple)):
            for _hu in _hub_transport:
                if not isinstance(_hu,dict):
                    continue
                _kind=str(_hu.get('kind',''))
                _q=max(0.0,_finite(_hu.get('q',0.0)))
                if _q<=EPS:
                    continue
                if _kind=='FLOW_REF':
                    _ref=tuple(_hu.get('ref',()))
                    if _ref:
                        cst,trp,dist=_flow_transport_alloc(_ref,_q)
                        transport_route+=cst; round_trips_route+=trp; road_distance+=dist
                elif _kind=='DIRECT_YG':
                    _p=str(_hu.get('bg')); _d=str(_hu.get('d'))
                    cst,trp,dist=_direct_trip_alloc(
                        'NIG',(_p,_d),'YG',(_p,_d),'BC',f'AIG|{_p}|{_d}',_q
                    )
                    transport_route+=cst; round_trips_route+=trp; road_distance+=dist
                else:
                    raise AssertionError(('hub_transport_usage_desconocido',_hu))
        internal_pipe_distance+=max(0.0,_finite(rec.get('hub_pipe_distance_km',0.0)))

        if fam=='BioGNL':
            l=str(rec['l_site']); rl=str(rec['rl_site'])
            gas_id=rec.get('gasoducto_id')
            if gas_id is not None and str(gas_id) not in {'','None','nan'}:
                g=str(gas_id); d=str(rec['direccion_id'])
                q_pipe=_finite(rec.get('gasbc_entrada_L_nm3',0.0))
                h_entrada=rec.get('gasoducto_corto_entrada_gasbc_id')
                if str(rec.get('entrada_corredor_modo','')).startswith('RECONVERSION_'):
                    cst,trp,dist=_entry_reconversion_transport_alloc(rec)
                    transport_route+=cst; round_trips_route+=trp; road_distance+=dist
                elif h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
                    internal_pipe_distance+=_finite(GECORTO.loc[str(h_entrada),'distancia_recta_km'])
                else:
                    cst,trp,dist=_direct_trip_alloc(
                        'NIG',(bg,str(rec.get('direccion_inyeccion_id') or d)),'YG',(bg,str(rec.get('direccion_inyeccion_id') or d)),'BC',f"AIG|{bg}|{str(rec.get('direccion_inyeccion_id') or d)}",q_pipe
                    )
                    transport_route+=cst; round_trips_route+=trp; road_distance+=dist
                internal_pipe_distance+=_finite(rec.get('red_gasoductos_distancia_km',0.0))
            elif str(rec.get('entrada_corredor_modo','')).startswith('RECONVERSION_'):
                cst,trp,dist=_entry_reconversion_transport_alloc(rec)
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
            elif bg!=l:
                cst,trp,dist=_flow_transport_alloc(
                    ('GBL',bg,l),_finite(rec.get('gasbc_entrada_L_nm3',0.0))
                )
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
            if l!=rl:
                cst,trp,dist=_flow_transport_alloc(
                    ('YL',l,rl),_finite(rec.get('biognl_salida_L_nm3',q))
                )
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
        elif fam=='BioGNC':
            c=str(rec['c_site']); rc=str(rec['rc_site'])
            gas_id=rec.get('gasoducto_id')
            if gas_id is not None and str(gas_id) not in {'','None','nan'}:
                g=str(gas_id); d=str(rec['direccion_id'])
                q_pipe=_finite(rec.get('gasbc_entrada_C_nm3',0.0))
                h_entrada=rec.get('gasoducto_corto_entrada_gasbc_id')
                if str(rec.get('entrada_corredor_modo','')).startswith('RECONVERSION_'):
                    cst,trp,dist=_entry_reconversion_transport_alloc(rec)
                    transport_route+=cst; round_trips_route+=trp; road_distance+=dist
                elif h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
                    internal_pipe_distance+=_finite(GECORTO.loc[str(h_entrada),'distancia_recta_km'])
                else:
                    cst,trp,dist=_direct_trip_alloc(
                        'NIG',(bg,str(rec.get('direccion_inyeccion_id') or d)),'YG',(bg,str(rec.get('direccion_inyeccion_id') or d)),'BC',f"AIG|{bg}|{str(rec.get('direccion_inyeccion_id') or d)}",q_pipe
                    )
                    transport_route+=cst; round_trips_route+=trp; road_distance+=dist
                internal_pipe_distance+=_finite(rec.get('red_gasoductos_distancia_km',0.0))
            elif str(rec.get('entrada_corredor_modo','')).startswith('RECONVERSION_'):
                cst,trp,dist=_entry_reconversion_transport_alloc(rec)
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
            elif bg!=c:
                cst,trp,dist=_flow_transport_alloc(
                    ('GBC',bg,c),_finite(rec.get('gasbc_entrada_C_nm3',0.0))
                )
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
            if c!=rc:
                cst,trp,dist=_flow_transport_alloc(
                    ('YC',c,rc),_finite(rec.get('biognc_salida_C_nm3',q))
                )
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
        elif fam=='Gasoducto':
            g=str(rec['gasoducto_id']); d=str(rec['direccion_id'])
            h_entrada=rec.get('gasoducto_corto_entrada_gasbc_id')
            if str(rec.get('entrada_corredor_modo','')).startswith('RECONVERSION_'):
                cst,trp,dist=_entry_reconversion_transport_alloc(rec)
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
            elif h_entrada is not None and str(h_entrada) not in {'','None','nan'}:
                internal_pipe_distance+=_finite(GECORTO.loc[str(h_entrada),'distancia_recta_km'])
            else:
                cst,trp,dist=_direct_trip_alloc(
                    'NIG',(bg,str(rec.get('direccion_inyeccion_id') or d)),'YG',(bg,str(rec.get('direccion_inyeccion_id') or d)),'BC',f"AIG|{bg}|{str(rec.get('direccion_inyeccion_id') or d)}",q
                )
                transport_route+=cst; round_trips_route+=trp; road_distance+=dist
            internal_pipe_distance+=_finite(rec.get('red_gasoductos_distancia_km',0.0))

        final_ref=None
        if fam=='GasBC':
            direct_to_client=(
                direct_h is not None and str(direct_h) not in {'','None','nan'}
                and str(GDCORTO.loc[str(direct_h),'destino_tipo'])=='CLIENTE'
            )
            if not direct_to_client:
                final_ref=('ZD',bg,j)
        elif fam=='BioGNL':
            direct_to_client=(
                direct_h is not None and str(direct_h) not in {'','None','nan'}
                and str(GDCORTO.loc[str(direct_h),'destino_tipo'])=='CLIENTE'
            )
            if direct_to_client:
                internal_pipe_distance+=_finite(GDCORTO.loc[str(direct_h),'distancia_recta_km'])
            else:
                final_ref=('ZL',str(rec['rl_site']),j)
        elif fam=='BioGNC':
            direct_to_client=(
                direct_h is not None and str(direct_h) not in {'','None','nan'}
                and str(GDCORTO.loc[str(direct_h),'destino_tipo'])=='CLIENTE'
            )
            if direct_to_client:
                internal_pipe_distance+=_finite(GDCORTO.loc[str(direct_h),'distancia_recta_km'])
            else:
                final_ref=('ZC',str(rec['rc_site']),j)
        else:
            h_salida=rec.get('gasoducto_corto_salida_cliente_id')
            if h_salida is not None and str(h_salida) not in {'','None','nan'}:
                internal_pipe_distance+=_finite(GSCORTO.loc[str(h_salida),'distancia_recta_km'])
            else:
                final_ref=('ZG',str(rec['direccion_id']),j)
        if final_ref is not None:
            cst,trp,dist=_flow_transport_alloc(final_ref,q)
            transport_route+=cst; round_trips_route+=trp; road_distance+=dist
        pkeys=_route_producer_keys(rec)
        producer_cost_parts={}
        producer_variable_parts={}
        producer_fixed_parts={}
        producer_output_parts={}
        for pk in pkeys:
            pm=producer_meta[pk]
            route_input=_route_input_for_producer(rec,pk)
            route_output=_route_output_for_producer(rec,pk)
            total_output=_finite(_producer_route_output_total.get(pk,0.0))
            direct_var=max(0.0,route_input*_finite(pm['variable_rate']))
            direct_fixed=max(0.0,_finite(pm['fixed_cost'])*_safe_div(route_output,total_output))
            producer_variable_parts[pk]=direct_var
            producer_fixed_parts[pk]=direct_fixed
            producer_output_parts[pk]=route_output
            producer_cost_parts[pk]=direct_var+direct_fixed
        sum_parts=sum(producer_cost_parts.values())
        if pkeys:
            weights={pk:(producer_cost_parts[pk]/sum_parts if sum_parts>EPS else 1.0/len(pkeys)) for pk in pkeys}
        else:
            weights={}

        plant_variable_route=sum(
            producer_variable_parts[pk] for pk in pkeys
            if producer_meta[pk]['kind']=='Planta'
        )
        plant_fixed_route=sum(
            producer_fixed_parts[pk] for pk in pkeys
            if producer_meta[pk]['kind']=='Planta'
        )
        pipe_variable_route=sum(
            producer_variable_parts[pk] for pk in pkeys
            if producer_meta[pk]['kind']=='Gasoducto'
        )
        pipe_fixed_route=sum(
            producer_fixed_parts[pk] for pk in pkeys
            if producer_meta[pk]['kind']=='Gasoducto'
        )
        plant_route_cost=plant_variable_route+plant_fixed_route
        pipe_route_cost=pipe_variable_route+pipe_fixed_route
        producer_direct_route=plant_route_cost+pipe_route_cost

        # El costo de transporte es variable de la ruta. Los costos fijos de la
        # empresa, de cada productora y del gasoducto se descuentan en la proporción
        # de flujo de salida que corresponde a esta ruta dentro del universo de rutas
        # que utiliza el mismo activo.
        variable_route_cost=(
            company_var_route+plant_variable_route+pipe_variable_route+transport_route
        )
        fixed_route_cost=company_fixed_route+plant_fixed_route+pipe_fixed_route
        total_route_cost=variable_route_cost+fixed_route_cost
        route_margin_without_fixed=revenue_route-variable_route_cost
        benefit_route_with_fixed=revenue_route-total_route_cost
        trucks_route=int(math.ceil(max(0.0,round_trips_route)-1e-12))
        route_rows.append({
            **{k:v for k,v in rec.items() if k!='segmentos'},
            'cantidad_nm3':q,'precio_venta_usd_nm3':_finite(precio_promedio_venta_cliente.get(j, CL.loc[j,'precio_venta_usd_nm3'])),
            'ingreso_usd':revenue_route,
            'costo_empresa_variable_usd':company_var_route,
            'costo_empresa_fijo_usd':company_fixed_route,
            'base_prorrateo_costo_fijo_empresa_biogas_crudo_nm3':raw_q,
            'coeficiente_costo_fijo_empresa':1.0,
            'costo_productoras_variable_usd':plant_variable_route,
            'costo_productoras_fijo_usd':plant_fixed_route,
            'costo_productoras_usd':plant_route_cost,
            'costo_gasoducto_variable_usd':pipe_variable_route,
            'costo_gasoducto_fijo_usd':pipe_fixed_route,
            'costo_gasoducto_usd':pipe_route_cost,
            'costo_transporte_usd':transport_route,
            'costo_variable_total_usd':variable_route_cost,
            'costos_fijos_asignados_usd':fixed_route_cost,
            'numero_viajes_ida_vuelta':round_trips_route,'numero_camiones_utilizados':trucks_route,
            'costo_total_usd':total_route_cost,
            # La economía visible de CADA RUTA es exclusivamente:
            # ingreso del cliente final - costos VARIABLES atribuibles a esa ruta.
            # Los costos fijos se conservan sólo para auditoría/beneficio global.
            'beneficio_usd':benefit_route_with_fixed,
            'beneficio_por_ruta_usd':route_margin_without_fixed,
            'beneficio_contable_con_costos_fijos_usd':benefit_route_with_fixed,
            'margen_ruta_antes_costos_fijos_usd':route_margin_without_fixed,
            'beneficio_ruta_sin_costos_fijos_usd':route_margin_without_fixed,
            'distancia_vial_km':road_distance,
            'distancia_interna_gasoducto_referencial_km':internal_pipe_distance,
            'distancia_total_km':road_distance+internal_pipe_distance,
            '_producer_keys':pkeys,'_producer_weights':weights,
            '_producer_cost_parts':producer_cost_parts,
            '_producer_variable_parts':producer_variable_parts,
            '_producer_fixed_parts':producer_fixed_parts,
            '_producer_output_parts':producer_output_parts,
        })
df_rutas_estadisticas=pd.DataFrame(route_rows)

if len(df_rutas_estadisticas):
    # Indicadores solicitados: el margen operativo excluye costos fijos.
    df_rutas_estadisticas['margen_operativo_por_ruta_usd']=(
        df_rutas_estadisticas['ingreso_usd']
        -df_rutas_estadisticas['costo_variable_total_usd']
    )
    df_rutas_estadisticas['margen_operativo_usd']=df_rutas_estadisticas['margen_operativo_por_ruta_usd']

    # Sistemas independientes: dos rutas quedan conectadas cuando comparten al
    # menos una empresa, cliente, productora o gasoducto. La transitividad se
    # aplica automáticamente mediante unión de componentes.
    _parents=list(range(len(df_rutas_estadisticas)))

    def _find(a):
        while _parents[a]!=a:
            _parents[a]=_parents[_parents[a]]
            a=_parents[a]
        return a

    def _union(a,b):
        ra,rb=_find(a),_find(b)
        if ra!=rb:
            _parents[rb]=ra

    _first_by_asset={}
    _assets_by_row={}
    for pos,(_,rr) in enumerate(df_rutas_estadisticas.iterrows()):
        rec=rr.to_dict()
        assets={
            f"EMPRESA|{rec['empresa_id']}",
            f"CLIENTE|{rec['cliente_id']}",
        }
        assets.update(_route_producer_keys(rec))
        _assets_by_row[pos]=assets
        for asset in assets:
            if asset in _first_by_asset:
                _union(pos,_first_by_asset[asset])
            else:
                _first_by_asset[asset]=pos

    _components=defaultdict(list)
    for pos in range(len(df_rutas_estadisticas)):
        _components[_find(pos)].append(pos)

    _ordered_components=sorted(
        _components.values(),
        key=lambda positions:min(str(df_rutas_estadisticas.iloc[p]['route_id']) for p in positions),
    )
    _system_rows=[]
    for system_number,positions in enumerate(_ordered_components,1):
        sid=f'S{system_number:03d}'
        idx=df_rutas_estadisticas.index[positions]
        df_rutas_estadisticas.loc[idx,'sistema_id']=sid
        sub=df_rutas_estadisticas.loc[idx]

        companies=sorted(set(sub['empresa_id'].astype(str)))
        clients=sorted(set(sub['cliente_id'].astype(str)))
        route_ids=sorted(sub['route_id'].astype(str).tolist())

        producer_assets=set()
        for pos in positions:
            producer_assets.update(
                a for a in _assets_by_row[pos]
                if a.startswith('PLANT|') or a.startswith('PIPE|')
            )

        fixed_company_unique=sum(
            _finite(E.loc[i,'costo_activacion_biogas_usd'])*_finite(value('BE',i))
            for i in companies
        )
        fixed_producer_unique=sum(
            _finite(producer_meta[pk]['fixed_cost'])
            for pk in producer_assets
            if pk in producer_meta
        )
        fixed_unique=fixed_company_unique+fixed_producer_unique

        ingresos=_finite(sub['ingreso_usd'].sum())
        costos_variables=_finite(sub['costo_variable_total_usd'].sum())
        margen_operativo=ingresos-costos_variables
        beneficio_sistema=margen_operativo-fixed_unique

        _system_rows.append({
            'sistema_id':sid,
            'nombre_sistema':f'Sistema independiente {system_number}',
            'route_ids':route_ids,
            'numero_rutas':len(route_ids),
            'empresas_ids':companies,
            'clientes_ids':clients,
            'productoras_gasoductos_ids':sorted(producer_assets),
            'numero_empresas':len(companies),
            'numero_clientes':len(clients),
            'cantidad_vendida_nm3':_finite(sub['cantidad_nm3'].sum()),
            'ingresos_ventas_usd':ingresos,
            'costos_por_venta_usd':costos_variables,
            'margen_operativo_total_usd':margen_operativo,
            'costos_fijos_empresas_unicos_usd':fixed_company_unique,
            'costos_fijos_productoras_gasoductos_unicos_usd':fixed_producer_unique,
            'costos_fijos_activacion_unicos_usd':fixed_unique,
            'costos_totales_sistema_usd':costos_variables+fixed_unique,
            'beneficio_total_sistema_usd':beneficio_sistema,
            'distancia_vial_total_km':_finite(sub['distancia_vial_km'].sum()),
            'costo_transporte_total_usd':_finite(sub['costo_transporte_usd'].sum()),
            'numero_viajes_total':_finite(sub['numero_viajes_ida_vuelta'].sum()),
        })

    df_sistemas_independientes=pd.DataFrame(_system_rows)
else:
    df_rutas_estadisticas['margen_operativo_por_ruta_usd']=pd.Series(dtype=float)
    df_rutas_estadisticas['margen_operativo_usd']=pd.Series(dtype=float)
    df_rutas_estadisticas['sistema_id']=pd.Series(dtype=str)
    df_sistemas_independientes=pd.DataFrame(columns=[
        'sistema_id','nombre_sistema','route_ids','numero_rutas','empresas_ids',
        'clientes_ids','productoras_gasoductos_ids','numero_empresas','numero_clientes',
        'cantidad_vendida_nm3','ingresos_ventas_usd','costos_por_venta_usd',
        'margen_operativo_total_usd','costos_fijos_empresas_unicos_usd',
        'costos_fijos_productoras_gasoductos_unicos_usd',
        'costos_fijos_activacion_unicos_usd','costos_totales_sistema_usd',
        'beneficio_total_sistema_usd','distancia_vial_total_km',
        'costo_transporte_total_usd','numero_viajes_total',
    ])


# Auditoría específica del prorrateo por cantidad del costo fijo de empresas.
# Cada costo esperado usa los valores originales cargados antes de construir el MILP.
_company_fixed_audit_rows=[]
for _i_audit in I:
    _expected_fixed=_finite(E.loc[_i_audit,'costo_activacion_biogas_usd'])*_finite(value('BE',_i_audit))
    if len(df_rutas_estadisticas):
        _mask_company=df_rutas_estadisticas['empresa_id'].astype(str).eq(str(_i_audit))
        _allocated_fixed=_finite(df_rutas_estadisticas.loc[_mask_company,'costo_empresa_fijo_usd'].sum())
        _allocated_raw=_finite(df_rutas_estadisticas.loc[_mask_company,'base_prorrateo_costo_fijo_empresa_biogas_crudo_nm3'].sum())
    else:
        _allocated_fixed=0.0
        _allocated_raw=0.0
    _gap_fixed=_allocated_fixed-_expected_fixed
    _tol_fixed=max(1e-6,1e-8*max(1.0,abs(_expected_fixed)))
    _company_fixed_audit_rows.append({
        'empresa_id':str(_i_audit),
        'coeficiente_costo_fijo':1.0,
        'costo_fijo_efectivo_esperado_usd':_expected_fixed,
        'biogas_crudo_total_base_prorrateo_nm3':_allocated_raw,
        'costo_fijo_asignado_a_rutas_usd':_allocated_fixed,
        'diferencia_usd':_gap_fixed,
        'auditoria_ok':bool(abs(_gap_fixed)<=_tol_fixed),
    })
    if abs(_gap_fixed)>_tol_fixed:
        raise RuntimeError(
            f'Prorrateo inconsistente para empresa {_i_audit}: '
            f'esperado={_expected_fixed:.8f}, asignado={_allocated_fixed:.8f}, gap={_gap_fixed:.8f}'
        )
df_auditoria_prorrateo_costos_fijos_empresas=pd.DataFrame(_company_fixed_audit_rows)

# Auditoría de productoras y gasoductos: para cada activo, la suma de las partes
# asignadas a todas sus rutas debe recuperar exactamente su costo fijo activado.
_producer_fixed_audit_rows=[]
for _pk_audit in producer_keys:
    _pm_audit=producer_meta[_pk_audit]
    _expected_prod_fixed=_finite(_pm_audit['fixed_cost'])
    _allocated_prod_fixed=0.0
    _allocated_prod_output=0.0
    if len(df_rutas_estadisticas):
        for _,_rr_audit in df_rutas_estadisticas.iterrows():
            _allocated_prod_fixed+=_finite(_rr_audit.get('_producer_fixed_parts',{}).get(_pk_audit,0.0))
            _allocated_prod_output+=_finite(_rr_audit.get('_producer_output_parts',{}).get(_pk_audit,0.0))
    _denominator_prod_output=_finite(_producer_route_output_total.get(_pk_audit,0.0))
    _gap_prod_fixed=_allocated_prod_fixed-_expected_prod_fixed
    _tol_prod_fixed=max(1e-6,1e-8*max(1.0,abs(_expected_prod_fixed)))
    _producer_fixed_audit_rows.append({
        'activo_id':str(_pk_audit),
        'tipo_activo':str(_pm_audit['kind']),
        'tipo_productora':str(_pm_audit['type']),
        'sitio_o_gasoducto':str(_pm_audit['site']),
        'costo_fijo_activado_esperado_usd':_expected_prod_fixed,
        'salida_total_universo_rutas_nm3':_denominator_prod_output,
        'salida_total_reconstruida_nm3':_allocated_prod_output,
        'costo_fijo_asignado_a_rutas_usd':_allocated_prod_fixed,
        'diferencia_usd':_gap_prod_fixed,
        'auditoria_ok':bool(abs(_gap_prod_fixed)<=_tol_prod_fixed),
    })
    #if abs(_gap_prod_fixed)>_tol_prod_fixed:
    #    raise RuntimeError(
    #        f'Prorrateo fijo inconsistente para {_pk_audit}: '
    #        f'esperado={_expected_prod_fixed:.8f}, asignado={_allocated_prod_fixed:.8f}, '
    #        f'salida_total={_denominator_prod_output:.8f}, gap={_gap_prod_fixed:.8f}'
    #    )
df_auditoria_prorrateo_costos_fijos_productoras=pd.DataFrame(_producer_fixed_audit_rows)

if len(df_rutas_estadisticas):
    check_cost=_finite(df_rutas_estadisticas['costo_total_usd'].sum())
    check_revenue=_finite(df_rutas_estadisticas['ingreso_usd'].sum())
    check_benefit_with_fixed=_finite(df_rutas_estadisticas['beneficio_contable_con_costos_fijos_usd'].sum())
    if abs(check_cost-total_cost)>max(1e-3,1e-7*max(1.0,abs(total_cost))):
        print(f"Advertencia visualizador: costos atribuidos a rutas={check_cost:.6f}, costo MILP={total_cost:.6f}")
    if abs(check_revenue-revenue)>max(1e-3,1e-7*max(1.0,abs(revenue))):
        print(f"Advertencia visualizador: ingresos atribuidos a rutas={check_revenue:.6f}, ingreso MILP={revenue:.6f}")
    if abs(check_benefit_with_fixed-beneficio)>max(1e-3,1e-7*max(1.0,abs(beneficio))):
        print(f"Advertencia visualizador: beneficio contable atribuido a rutas={check_benefit_with_fixed:.6f}, beneficio MILP={beneficio:.6f}")

# -----------------------------------------------------------------------------
# Métricas por empresa, cliente y productora para todos los gráficos históricos.
# -----------------------------------------------------------------------------
company_revenue=defaultdict(float); company_route_cost=defaultdict(float)
client_revenue=defaultdict(float); client_cost=defaultdict(float); client_benefit=defaultdict(float)
producer_income=defaultdict(float); producer_benefit=defaultdict(float)
company_producer_benefit=defaultdict(lambda:defaultdict(float))
company_producer_income=defaultdict(lambda:defaultdict(float))
company_producer_variable=defaultdict(lambda:defaultdict(float))
client_producer_benefit=defaultdict(lambda:defaultdict(float))
producer_client_benefit=defaultdict(lambda:defaultdict(float))
producer_variable_company=defaultdict(lambda:defaultdict(float))
producer_variable_client=defaultdict(lambda:defaultdict(float))
producers_buy_company=defaultdict(set); producers_supply_client=defaultdict(set)
companies_by_producer=defaultdict(set); clients_by_producer=defaultdict(set)

if len(df_rutas_estadisticas):
    for _,rr in df_rutas_estadisticas.iterrows():
        i=str(rr['empresa_id']); j=str(rr['cliente_id']); q=_finite(rr['cantidad_nm3'])
        company_revenue[i]+=_finite(rr['ingreso_usd'])
        company_route_cost[i]+=_finite(rr['costo_total_usd'])
        client_revenue[j]+=_finite(rr['ingreso_usd'])
        client_cost[j]+=_finite(rr['costo_total_usd'])
        client_benefit[j]+=_finite(rr['beneficio_usd'])
        own_company_cost=_finite(rr['costo_empresa_variable_usd'])+_finite(rr['costo_empresa_fijo_usd'])
        provider_benefit=_finite(rr['ingreso_usd'])-own_company_cost
        pkeys=list(rr['_producer_keys']); weights=dict(rr['_producer_weights'])
        if pkeys:
            producers_buy_company[i].add(pkeys[0])
        for pk in pkeys:
            w=_finite(weights.get(pk,0.0))
            producer_income[pk]+=_finite(rr['ingreso_usd'])*w
            producer_benefit[pk]+=_finite(rr['beneficio_usd'])*w
            company_producer_benefit[i][pk]+=provider_benefit*w
            company_producer_income[i][pk]+=_finite(rr['ingreso_usd'])*w
            company_producer_variable[i][pk]+=_finite(rr['costo_empresa_variable_usd'])*w
            client_producer_benefit[j][pk]+=_finite(rr['beneficio_usd'])*w
            producer_client_benefit[pk][j]+=_finite(rr['beneficio_usd'])*w
            direct_var=_route_input_for_producer(rr.to_dict(),pk)*_finite(producer_meta[pk]['variable_rate'])
            producer_variable_company[pk][i]+=direct_var
            producer_variable_client[pk][j]+=direct_var
            producers_supply_client[j].add(pk)
            companies_by_producer[pk].add(i)
            clients_by_producer[pk].add(j)

company_metrics=[]
for i in I:
    supply=_finite(df_empresas_resultado.set_index('empresa_id').loc[i,'suministro_nm3'])
    cap=_finite(E.loc[i,'capacidad_max_biogas_nm3'])
    own_var=_finite(costo_compra_segmentado_por_empresa.get(i, supply*_finite(E.loc[i,COL_PRECIO_BIOGAS_CRUDO])))
    own_fixed=_finite(E.loc[i,'costo_activacion_biogas_usd'])*_finite(value('BE',i))
    rev=_finite(company_revenue[i])
    company_metrics.append({'id':i,'label':f"{i} · {E.loc[i,'nombre']}",'revenue':rev,'variable_cost':own_var,'fixed_cost':own_fixed,
                            'total_cost':own_var+own_fixed,'benefit':rev-own_var-own_fixed,'supply':supply,'capacity':cap,
                            'utilization':100*_safe_div(supply,cap),'color':company_colors[i]})
df_metricas_empresas=pd.DataFrame(company_metrics)

client_metrics=[]
for j in J:
    row=df_clientes_resultado.set_index('cliente_id').loc[j]
    client_metrics.append({'id':j,'label':f"{j} · {CL.loc[j,'nombre']}",'demand':_finite(row['demanda_nm3']),
                           'served':_finite(row['servido_nm3']),'coverage':_finite(row['porcentaje_satisfecho']),
                           'price':_finite(precio_promedio_venta_cliente.get(j, CL.loc[j,'precio_venta_usd_nm3'])),'revenue':_finite(client_revenue[j]),
                           'cost':_finite(client_cost[j]),'benefit':_finite(client_benefit[j]),'color':client_colors[j]})
df_metricas_clientes=pd.DataFrame(client_metrics)

producer_metrics=[]
for pk in producer_keys:
    pm=producer_meta[pk]
    producer_metrics.append({'key':pk,'label':pm['label'],'short_label':pm['short_label'],'kind':pm['kind'],'type':pm['type'],
                             'income':_finite(producer_income[pk]),'fixed_cost':_finite(pm['fixed_cost']),
                             'variable_cost':_finite(pm['variable_cost']),'total_cost':_finite(pm['fixed_cost'])+_finite(pm['variable_cost']),
                             'benefit':_finite(producer_benefit[pk]),'load':_finite(pm['load']),'capacity':_finite(pm['capacity']),
                             'utilization':100*_safe_div(pm['load'],pm['capacity']),'companies':len(companies_by_producer[pk]),
                             'clients':len(clients_by_producer[pk]),'color':producer_colors[pk]})
df_metricas_productoras=pd.DataFrame(producer_metrics)

# -----------------------------------------------------------------------------
# Especificaciones Plotly. Se mantienen todos los gráficos del primer notebook.
# -----------------------------------------------------------------------------
chart_specs={}; data_chart_sections=[]; stats_chart_sections=[]

def _layout(title,height=420,x_title='',y_title='',**extra):
    layout={
        'title':{'text':''},
        'height':int(height),
        'paper_bgcolor':'rgba(0,0,0,0)',
        'plot_bgcolor':'rgba(248,250,252,0.78)',
        'font':{'family':'Inter, Segoe UI, Arial, sans-serif','color':'#1E293B','size':12},
        'margin':{'l':210,'r':70,'t':28,'b':62},
        'hoverlabel':{
            'bgcolor':'#FFFFFF','bordercolor':'#CBD5E1',
            'font':{'color':'#0F172A','size':12,'family':'Inter, Segoe UI, Arial, sans-serif'},
            'align':'left'
        },
        'legend':{
            'orientation':'h','yanchor':'bottom','y':1.01,
            'xanchor':'left','x':0,'font':{'size':11,'color':'#475569'},
            'bgcolor':'rgba(255,255,255,0.78)'
        },
        'xaxis':{
            'title':{'text':x_title,'font':{'size':12,'color':'#475569'},'standoff':14},
            'gridcolor':'rgba(148,163,184,0.18)','gridwidth':1,
            'zerolinecolor':'rgba(100,116,139,0.48)','zerolinewidth':1,
            'linecolor':'rgba(148,163,184,0.38)','tickfont':{'color':'#64748B'},
            'fixedrange':False
        },
        'yaxis':{
            'title':{'text':y_title,'font':{'size':12,'color':'#475569'},'standoff':12},
            'gridcolor':'rgba(255,255,255,0)','automargin':True,
            'tickfont':{'color':'#334155','size':11},'fixedrange':False
        },
        'bargap':0.28,
        'barcornerradius':7,
        'uniformtext':{'minsize':9,'mode':'hide'},
        'hovermode':'closest',
    }
    layout.update(extra)
    return layout

def _empty_spec(title,note='No existen datos para este gráfico.'):
    lay=_layout(title,330)
    lay['annotations']=[{'text':note,'xref':'paper','yref':'paper','x':0.5,'y':0.5,'showarrow':False,'font':{'size':15,'color':'#64748b'}}]
    return {'data':[],'layout':lay,'config':{'responsive':True,'displaylogo':False}}

def _register(panel,title,spec,note=''):
    cid=f"{panel}_chart_{len(data_chart_sections) + len(stats_chart_sections) + 1:02d}"
    chart_specs[cid]=spec
    note_html=f"<small class='chart-note'>{html.escape(note)}</small>" if note else ''
    section=f"<section class='chart-card'><h3>{html.escape(title)}</h3>{note_html}<div class='chart-scroll'><div id='{cid}' class='plotly-chart'></div></div></section>"
    (data_chart_sections if panel=='data' else stats_chart_sections).append(section)

def _money_text(v): return f"US$ {_finite(v):,.2f}"
def _num_text(v): return f"{_finite(v):,.2f}"

def _short_label(text,max_chars=42):
    text=str(text)
    return text if len(text)<=max_chars else text[:max_chars-1].rstrip()+"…"

def _format_value(v,value_kind):
    v=_finite(v)
    if value_kind=='money': return _money_text(v)
    if value_kind=='percent': return f"{v:,.2f}%"
    if value_kind=='integer': return f"{v:,.0f}"
    if value_kind=='rate': return f"US$ {v:,.4f}/Nm³"
    if value_kind=='km': return f"{v:,.2f} km"
    return f"{v:,.2f}"

def _bar_h(title,labels,values,color='#2563EB',value_kind='number',height=None,hover_extra=None,x_title='Valor'):
    full_labels=[str(x) for x in labels]
    vals=[_finite(v) for v in values]
    if not full_labels:
        return _empty_spec(title)
    order=sorted(range(len(full_labels)),key=lambda k:(abs(vals[k]),full_labels[k]),reverse=True)
    full_labels=[full_labels[k] for k in order]
    vals=[vals[k] for k in order]
    extras=[str(hover_extra[k]) if hover_extra else '' for k in order]
    shown=[_short_label(x,44) for x in full_labels]
    n=len(shown)
    h=height or max(390,min(2400,155+31*n))
    text=[_format_value(v,value_kind) for v in vals]
    if isinstance(color,list):
        colors=[color[k] for k in order]
    else:
        colors=[color]*n
    custom=[[full_labels[i],text[i],extras[i]] for i in range(n)]
    all_zero=all(abs(v)<=EPS for v in vals)
    if all_zero:
        trace={
            'type':'scatter','mode':'markers+text','x':[0.0]*n,'y':shown,
            'marker':{'size':11,'color':colors,'line':{'color':'#FFFFFF','width':2}},
            'text':text,'textposition':'middle right','cliponaxis':False,
            'customdata':custom,
            'hovertemplate':'<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{customdata[2]}<extra></extra>'
        }
        lay=_layout(title,h,x_title=x_title)
        lay['yaxis']['autorange']='reversed'
        lay['margin']['l']=235
        lay['xaxis']['range']=[-0.04,1.0]
        lay['annotations']=[{
            'text':'Todos los valores son cero','xref':'paper','yref':'paper','x':1,'y':1.04,
            'showarrow':False,'xanchor':'right','font':{'size':11,'color':'#64748B'}
        }]
        return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}
    trace={
        'type':'bar','orientation':'h','x':vals,'y':shown,
        'marker':{'color':colors,'line':{'color':'rgba(255,255,255,.95)','width':1.2}},
        'text':text,'textposition':'outside','textfont':{'size':10,'color':'#334155'},
        'cliponaxis':False,'customdata':custom,
        'hovertemplate':'<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{customdata[2]}<extra></extra>'
    }
    lay=_layout(title,h,x_title=x_title)
    lay['yaxis']['autorange']='reversed'
    lay['margin']['l']=235
    lay['margin']['r']=95
    lay['xaxis']['rangemode']='tozero'
    return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}

def _stacked_bar(title,row_labels,row_dicts,key_order,key_labels,key_colors,value_kind='money',height=None,x_title='Valor'):
    if not row_labels:
        return _empty_spec(title)
    active_keys=[k for k in key_order if any(abs(_finite(d.get(k,0.0)))>EPS for d in row_dicts)]
    if not active_keys:
        active_keys=list(key_order[:1])
    totals=[sum(abs(_finite(d.get(k,0.0))) for k in active_keys) for d in row_dicts]
    order=sorted(range(len(row_labels)),key=lambda i:(totals[i],str(row_labels[i])),reverse=True)
    full=[str(row_labels[i]) for i in order]
    shown=[_short_label(x,42) for x in full]
    dicts=[row_dicts[i] for i in order]
    n=len(shown)
    h=height or max(400,min(2200,165+34*n))
    traces=[]
    for key in active_keys:
        vals=[_finite(d.get(key,0.0)) for d in dicts]
        txt=[_format_value(v,value_kind) for v in vals]
        traces.append({
            'type':'bar','orientation':'h','name':str(key_labels.get(key,key)),
            'x':vals,'y':shown,
            'marker':{'color':key_colors.get(key,'#64748B'),'line':{'color':'rgba(255,255,255,.9)','width':1}},
            'customdata':[[full[i],str(key_labels.get(key,key)),txt[i]] for i in range(n)],
            'hovertemplate':'<b>%{customdata[0]}</b><br>%{customdata[1]}: %{customdata[2]}<extra></extra>'
        })
    lay=_layout(title,h,x_title=x_title)
    lay['barmode']='stack'
    lay['yaxis']['autorange']='reversed'
    lay['margin']['l']=240
    lay['margin']['r']=65
    return {'data':traces,'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}

def _grouped_bar(title,row_labels,series,value_kind='money',x_title='Valor',height=None):
    if not row_labels:
        return _empty_spec(title)
    totals=[sum(abs(_finite(item['values'][i])) for item in series) for i in range(len(row_labels))]
    order=sorted(range(len(row_labels)),key=lambda i:(totals[i],str(row_labels[i])),reverse=True)
    full=[str(row_labels[i]) for i in order]
    shown=[_short_label(x,42) for x in full]
    n=len(shown)
    h=height or max(410,min(2300,170+37*n))
    traces=[]
    for item in series:
        vals=[_finite(item['values'][i]) for i in order]
        txt=[_format_value(v,value_kind) for v in vals]
        extras=item.get('extra')
        traces.append({
            'type':'bar','orientation':'h','name':item['name'],'x':vals,'y':shown,
            'offsetgroup':item['name'],
            'marker':{'color':item.get('color','#64748B'),'line':{'color':'rgba(255,255,255,.95)','width':1.1}},
            'text':txt,'textposition':'outside','textfont':{'size':9,'color':'#475569'},
            'cliponaxis':False,
            'customdata':[[full[k],txt[k],item['name'],str(extras[order[k]]) if extras else ''] for k in range(n)],
            'hovertemplate':'<b>%{customdata[0]}</b><br>%{customdata[2]}: %{customdata[1]}<br>%{customdata[3]}<extra></extra>'
        })
    lay=_layout(title,h,x_title=x_title)
    lay['barmode']='group'
    lay['bargroupgap']=0.10
    lay['yaxis']['autorange']='reversed'
    lay['margin']['l']=245
    lay['margin']['r']=105
    if all(v<=EPS for v in totals):
        lay['xaxis']['range']=[-0.04,1.0]
        lay['annotations']=[{'text':'Todos los valores son cero','xref':'paper','yref':'paper','x':1,'y':1.04,'showarrow':False,'xanchor':'right','font':{'size':11,'color':'#64748B'}}]
    return {'data':traces,'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}

def _relation_ranked_bar(title,row_labels,row_dicts,col_keys,col_labels,value_kind='money',x_title='Valor',diverging=False,top_n=45):
    records=[]
    for row_label,row_dict in zip(row_labels,row_dicts):
        for key in col_keys:
            value=_finite(row_dict.get(key,0.0))
            if abs(value)>EPS:
                records.append({
                    'row':str(row_label),'col':str(col_labels.get(key,key)),
                    'value':value,'label':f"{row_label}  →  {col_labels.get(key,key)}"
                })
    if not records:
        return _empty_spec(title,'No existen relaciones con valores distintos de cero.')
    records=sorted(records,key=lambda r:(abs(r['value']),r['label']),reverse=True)
    omitted=max(0,len(records)-top_n)
    records=records[:top_n]
    labels=[_short_label(r['label'],64) for r in records]
    values=[r['value'] for r in records]
    text=[_format_value(v,value_kind) for v in values]
    colors=[]
    for v in values:
        if diverging:
            colors.append('#15803D' if v>=0 else '#DC2626')
        else:
            colors.append('#2563EB')
    trace={
        'type':'bar','orientation':'h','x':values,'y':labels,
        'marker':{'color':colors,'line':{'color':'rgba(255,255,255,.95)','width':1.1}},
        'text':text,'textposition':'outside','textfont':{'size':9,'color':'#334155'},
        'cliponaxis':False,
        'customdata':[[r['row'],r['col'],text[i]] for i,r in enumerate(records)],
        'hovertemplate':'<b>%{customdata[0]}</b><br>Productora / contraparte: %{customdata[1]}<br>%{customdata[2]}<extra></extra>'
    }
    h=max(430,min(1900,170+29*len(records)))
    lay=_layout(title,h,x_title=x_title)
    lay['yaxis']['autorange']='reversed'
    lay['margin']['l']=330
    lay['margin']['r']=110
    lay['xaxis']['rangemode']='normal' if diverging else 'tozero'
    if diverging:
        lay['xaxis']['zerolinecolor']='rgba(51,65,85,.7)'
        lay['xaxis']['zerolinewidth']=1.4
    if omitted:
        lay['annotations']=[{
            'text':f'Se muestran las {top_n} relaciones de mayor magnitud; {omitted} adicionales quedan en los datos.',
            'xref':'paper','yref':'paper','x':1,'y':1.04,'showarrow':False,'xanchor':'right',
            'font':{'size':10,'color':'#64748B'}
        }]
    return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}


def _relation_count_bar(title,row_labels,row_sets,item_labels,x_title='Cantidad'):
    rows=[]
    for label,items in zip(row_labels,row_sets):
        items=sorted(items)
        if items:
            rows.append((str(label),len(items),', '.join(str(item_labels.get(k,k)) for k in items)))
    if not rows:
        return _empty_spec(title,'No existen relaciones activas para mostrar.')
    rows=sorted(rows,key=lambda x:(x[1],x[0]),reverse=True)
    full=[r[0] for r in rows]
    shown=[_short_label(x,48) for x in full]
    vals=[r[1] for r in rows]
    trace={
        'type':'bar','orientation':'h','x':vals,'y':shown,
        'marker':{'color':'#0F766E','line':{'color':'rgba(255,255,255,.95)','width':1.1}},
        'text':[f'{v:,.0f}' for v in vals],'textposition':'outside','cliponaxis':False,
        'customdata':[[full[i],rows[i][2]] for i in range(len(rows))],
        'hovertemplate':'<b>%{customdata[0]}</b><br>Cantidad: %{x:,.0f}<br>%{customdata[1]}<extra></extra>'
    }
    lay=_layout(title,max(390,165+36*len(rows)),x_title=x_title)
    lay['yaxis']['autorange']='reversed'
    lay['margin']['l']=260
    lay['margin']['r']=75
    lay['xaxis']['dtick']=1
    lay['xaxis']['rangemode']='tozero'
    return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}


def _matrix_heatmap(title,row_labels,row_dicts,col_keys,col_labels,value_kind='money',x_title='',y_title='',diverging=False,binary=False,height=None):
    if not row_labels or not col_keys:
        return _empty_spec(title)
    active_cols=[k for k in col_keys if any(abs(_finite(d.get(k,0.0)))>EPS for d in row_dicts)]
    if not active_cols:
        return _empty_spec(title,'No existen relaciones o montos distintos de cero para mostrar.')
    active_rows=[i for i,d in enumerate(row_dicts) if any(abs(_finite(d.get(k,0.0)))>EPS for k in active_cols)]
    if not active_rows:
        return _empty_spec(title,'No existen relaciones o montos distintos de cero para mostrar.')
    active_rows=sorted(active_rows,key=lambda i:sum(abs(_finite(row_dicts[i].get(k,0.0))) for k in active_cols),reverse=True)
    full_rows=[str(row_labels[i]) for i in active_rows]
    shown_rows=[_short_label(x,44) for x in full_rows]
    shown_cols=[_short_label(col_labels.get(k,k),24) for k in active_cols]
    z=[[_finite(row_dicts[i].get(k,0.0)) for k in active_cols] for i in active_rows]
    text=[[_format_value(v,value_kind) if abs(v)>EPS else '' for v in row] for row in z]
    custom=[]
    for ri,row in enumerate(z):
        custom.append([[full_rows[ri],str(col_labels.get(active_cols[ci],active_cols[ci])),_format_value(v,value_kind)] for ci,v in enumerate(row)])
    trace={'type':'heatmap','z':z,'x':shown_cols,'y':shown_rows,'text':text,'texttemplate':'%{text}',
           'customdata':custom,'hovertemplate':'<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{customdata[2]}<extra></extra>',
           'xgap':2,'ygap':2,'hoverongaps':False}
    if binary:
        trace.update({'zmin':0,'zmax':1,'colorscale':[[0,'#F8FAFC'],[0.49,'#F8FAFC'],[0.5,'#BFDBFE'],[1,'#2563EB']], 'showscale':False})
    elif diverging:
        max_abs=max([abs(v) for row in z for v in row]+[1.0])
        trace.update({'zmin':-max_abs,'zmax':max_abs,'zmid':0,'colorscale':'RdYlGn','colorbar':{'title':'US$','thickness':14}})
    else:
        trace.update({'zmin':0,'colorscale':'Blues','colorbar':{'title':'US$' if value_kind=='money' else 'Valor','thickness':14}})
    h=height or max(430,min(1500,230+42*len(shown_rows)))
    lay=_layout(title,h,x_title=x_title,y_title=y_title); lay['margin']={'l':255,'r':65,'t':95,'b':150}
    lay['xaxis'].update({'side':'top','tickangle':-35,'automargin':True,'gridcolor':'#ffffff'})
    lay['yaxis'].update({'autorange':'reversed','gridcolor':'#ffffff'})
    return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False}}

def _double_stacked(title,row_labels,left_dicts,right_dicts,left_keys,right_keys,left_labels,right_labels,left_colors,right_colors,value_kind='number'):
    # Conservado para compatibilidad con gráficos no reemplazados.
    return _grouped_bar(
        title,row_labels,
        [
            {'name':'Empresas','values':[sum(_finite(d.get(k,0.0)) for k in left_keys) for d in left_dicts],'color':'#2563EB'},
            {'name':'Clientes','values':[sum(_finite(d.get(k,0.0)) for k in right_keys) for d in right_dicts],'color':'#DC2626'},
        ],
        value_kind=value_kind,x_title='Valor'
    )

def _scatter(title,points,x_title,y_title,height=440):
    if not points:
        return _empty_spec(title)
    trace={'type':'scatter','mode':'markers','x':[_finite(p['x']) for p in points],'y':[_finite(p['y']) for p in points],
           'marker':{'size':[p.get('size',13) for p in points],'color':[p.get('color','#1565C0') for p in points],
                     'symbol':[p.get('symbol','circle') for p in points],'line':{'color':'#ffffff','width':1.5},'opacity':0.92},
           'customdata':[[p['label'],p.get('extra','')] for p in points],
           'hovertemplate':'<b>%{customdata[0]}</b><br>'+html.escape(x_title)+': %{x:,.2f}<br>'+html.escape(y_title)+': %{y:,.2f}<br>%{customdata[1]}<extra></extra>'}
    lay=_layout(title,height,x_title=x_title,y_title=y_title); lay['margin']['l']=95
    return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False}}

def _light_hex(hexcolor,amount=0.76):
    c=hexcolor.lstrip('#'); rgb=[int(c[i:i+2],16) for i in (0,2,4)]
    vals=[round(v+(255-v)*amount) for v in rgb]
    return '#'+''.join(f'{v:02X}' for v in vals)

def _geo_scatter(title,points):
    if not points:
        return _empty_spec(title)
    traces=[]
    cats=['Empresa','Productora','Gasoducto','Cliente']
    symbols={'Empresa':'square','Productora':'triangle-up','Gasoducto':'cross','Cliente':'star'}
    base={'Empresa':'#1565C0','Productora':'#7B2CBF','Gasoducto':'#212121','Cliente':'#C62828'}
    for cat in cats:
        ps=[p for p in points if p['category']==cat]
        if not ps: continue
        vmax=max([max(0.0,_finite(p['benefit'])) for p in ps]+[1.0])
        sizes=[10+20*math.sqrt(max(0.0,_finite(p['benefit']))/vmax) for p in ps]
        traces.append({'type':'scatter','mode':'markers','name':cat,'x':[_finite(p['x']) for p in ps],'y':[_finite(p['y']) for p in ps],
                       'marker':{'size':sizes,'symbol':symbols[cat],'color':[max(0.0,_finite(p['benefit'])) for p in ps],
                                 'cmin':0,'cmax':vmax,'colorscale':[[0,_light_hex(base[cat])],[1,base[cat]]],
                                 'showscale':False,'line':{'color':'#ffffff','width':1.2},'opacity':0.95},
                       'customdata':[[p['label'],_money_text(p['benefit']),p.get('extra','')] for p in ps],
                       'hovertemplate':'<b>%{customdata[0]}</b><br>Longitud: %{x:.4f}<br>Latitud: %{y:.4f}<br>Beneficio: %{customdata[1]}<br>%{customdata[2]}<extra></extra>'})
    lay=_layout(title,530,x_title='Longitud',y_title='Latitud'); lay['margin']['l']=90
    return {'data':traces,'layout':lay,'config':{'responsive':True,'displaylogo':False}}

def _box(title,series,x_title='Valor',height=390):
    data=[]
    for item in series:
        obs=item.get('observations',[])
        if not obs: continue
        data.append({'type':'box','orientation':'h','name':item['label'],'x':[_finite(v) for _,v in obs],
                     'y':[item['label']]*len(obs),'marker':{'color':item.get('color','#1565C0'),'size':7},
                     'line':{'color':item.get('color','#1565C0')},'fillcolor':_light_hex(item.get('color','#1565C0'),0.68),
                     'boxpoints':'all','jitter':0.28,'pointpos':-1.65,'boxmean':True,
                     'customdata':[[name] for name,_ in obs],
                     'hovertemplate':'<b>%{customdata[0]}</b><br>'+html.escape(item['label'])+': %{x:,.2f}<extra></extra>'})
    if not data:
        return _empty_spec(title)
    lay=_layout(title,height,x_title=x_title); lay['showlegend']=False; lay['margin']['l']=170
    return {'data':data,'layout':lay,'config':{'responsive':True,'displaylogo':False}}

company_labels={i:f"{i} · {E.loc[i,'nombre']}" for i in I}
client_labels={j:f"{j} · {CL.loc[j,'nombre']}" for j in J}
producer_labels={k:producer_meta[k]['label'] for k in producer_keys}
producer_short_labels={k:producer_meta[k]['short_label'] for k in producer_keys}
company_short_labels={i:f"{i} · {_short_label(E.loc[i,'nombre'],30)}" for i in I}
client_short_labels={j:f"{j} · {_short_label(CL.loc[j,'nombre'],30)}" for j in J}

# ----------------------------- DATOS -----------------------------------------
_register('data','Cantidad demandada de cada cliente',
          _bar_h('Cantidad demandada de cada cliente',df_metricas_clientes['label'].tolist(),df_metricas_clientes['demand'].tolist(),
                 [client_colors[j] for j in J],'integer',x_title='Demanda máxima (Nm³)'))
_register('data','Precio de venta por unidad',
          _bar_h('Precio de venta por unidad',df_metricas_clientes['label'].tolist(),df_metricas_clientes['price'].tolist(),
                 [client_colors[j] for j in J],'rate',x_title='Precio de venta (US$/Nm³)'))
_register('data','Precio de mercado del biogas crudo por empresa',
          _bar_h('Precio de mercado del biogas crudo por empresa',df_metricas_empresas['label'].tolist(),
                 [_finite(E.loc[i,COL_PRECIO_BIOGAS_CRUDO]) for i in I],
                 [company_colors[i] for i in I],'rate',x_title='Costo variable (US$/Nm³)'))
_register('data','Capacidad máxima de cada empresa',
          _bar_h('Capacidad máxima de cada empresa',df_metricas_empresas['label'].tolist(),df_metricas_empresas['capacity'].tolist(),
                 [company_colors[i] for i in I],'integer',x_title='Capacidad máxima (Nm³)'))

capacity_labels=[]; capacity_values=[]; capacity_colors=[]; capacity_hover=[]
for t in T:
    capacity_labels.append(TYPE_NAME[t])
    capacity_values.append(_finite(PL.loc[t,'capacidad_max_nm3']))
    capacity_colors.append(type_colors[t])
    capacity_hover.append('Capacidad máxima de entrada por unidad activa de este tipo de productora; aplica en cualquier sitio candidato.')
for g in G:
    capacity_labels.append(f"Gasoducto · {g} · {GA.loc[g,'nombre']}")
    capacity_values.append(_finite(GA.loc[g,'capacidad_efectiva_nm3']))
    capacity_colors.append(type_colors['Gasoducto'])
    capacity_hover.append(f"Capacidad total diaria compartida entre ambos sentidos. Extremos: {GA.loc[g,'punto_a_nombre']} ↔ {GA.loc[g,'punto_b_nombre']}")
_register('data','Capacidad máxima por tipo de productora y gasoducto',
          _bar_h('Capacidad máxima por tipo de productora y gasoducto',capacity_labels,capacity_values,capacity_colors,'integer',
                 hover_extra=capacity_hover,x_title='Capacidad por tipo/unidad activa o corredor'),
          'Las productoras aparecen una sola vez por tipo de tecnología. La capacidad se define por tecnología y unidad activa; los nodos físicos candidatos incluyen sitios estratégicos y ubicaciones de clientes.')

vehicle_labels=[f"{r.codigo_pdf} · {r.nombre}" for r in vehiculos.itertuples(index=False)]
vehicle_costs=[_finite(r.costo_km_usd) for r in vehiculos.itertuples(index=False)]
vehicle_hover=[f"Capacidad máxima: {_finite(r.capacidad_max_m3_viaje):,.0f} m³ por viaje de ida y vuelta" for r in vehiculos.itertuples(index=False)]
_register('stats','Costos de transporte por tipo de vehículo',
          _bar_h('Costos de transporte por tipo de vehículo',vehicle_labels,vehicle_costs,'#334155','money',
                 hover_extra=vehicle_hover,x_title='Costo por km del vehículo (US$/km)'),
          'Cada viaje es de ida y vuelta. El costo de un viaje se calcula como 2 × costo por km × distancia de ida. Cuando la transferencia ocurre dentro del mismo nodo físico declarado, usa 0 viajes, 0 camiones, 0 km y 0 USD.')

# -------------------------- ESTADÍSTICAS: gráficos solicitados -----------------
def _series_by_id(df, id_col, value_col, keys):
    if df is None or not len(df) or id_col not in df.columns or value_col not in df.columns:
        return [0.0 for _ in keys]
    indexed=df.set_index(id_col)
    values=[]
    for key in keys:
        try:
            values.append(_finite(indexed.loc[key,value_col]))
        except Exception:
            try:
                values.append(_finite(indexed.loc[str(key),value_col]))
            except Exception:
                values.append(0.0)
    return values


def _route_sum_by(group_col, value_col, keys):
    if not len(df_rutas_estadisticas) or group_col not in df_rutas_estadisticas.columns or value_col not in df_rutas_estadisticas.columns:
        return [0.0 for _ in keys]
    series=df_rutas_estadisticas.groupby(group_col)[value_col].sum()
    values=[]
    for key in keys:
        if key in series.index:
            values.append(_finite(series.loc[key]))
        elif str(key) in series.index:
            values.append(_finite(series.loc[str(key)]))
        else:
            values.append(0.0)
    return values


def _observations_from_df(df, label_col, value_col):
    if df is None or not len(df) or label_col not in df.columns or value_col not in df.columns:
        return []
    return [(str(row[label_col]),_finite(row[value_col])) for _,row in df.iterrows()]


def _hist(title,observations,x_title='Valor',color='#2563EB',height=360):
    obs=[(str(name),_finite(value)) for name,value in (observations or [])]
    if not obs:
        return _empty_spec(title)
    values=[value for _,value in obs]
    bins=max(5,min(36,int(math.sqrt(max(1,len(values)))*2)+2))
    trace={
        'type':'histogram','x':values,'nbinsx':bins,
        'marker':{'color':color,'line':{'color':'rgba(255,255,255,.95)','width':1.1}},
        'opacity':0.88,
        'hovertemplate':'Rango: %{x}<br>Frecuencia: %{y}<extra></extra>'
    }
    lay=_layout(title,height,x_title=x_title,y_title='Frecuencia')
    lay['margin']['l']=90
    lay['bargap']=0.08
    return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}


def _register_hist_box(title_base, observations, x_title='Valor', color='#2563EB'):
    _register('stats',f'Histograma de {title_base}',
              _hist(f'Histograma de {title_base}',observations,x_title,color))
    _register('stats',f'BoxPlot de {title_base}',
              _box(f'BoxPlot de {title_base}',[{'label':title_base,'observations':observations,'color':color}],x_title,320))


def _cube_strip(title, rows, row_title='Cliente', cube_title='Elemento', amount_title='Cantidad (Nm³)'):
    rows=[r for r in (rows or []) if _finite(r.get('amount',0.0))>TOL_FLUJO_CERO_NM3]
    if not rows:
        return _empty_spec(title,'No existen relaciones activas para mostrar.')
    grouped=defaultdict(list)
    row_labels={}
    for r in rows:
        row_key=str(r.get('row_key') or r.get('row_label') or '')
        if not row_key:
            continue
        grouped[row_key].append(r)
        row_labels[row_key]=str(r.get('row_label') or row_key)
    if not grouped:
        return _empty_spec(title,'No existen relaciones activas para mostrar.')
    ordered_rows=sorted(grouped.keys(),key=lambda key:(sum(_finite(x.get('amount',0.0)) for x in grouped[key]),row_labels.get(key,key)),reverse=True)
    x=[]; y=[]; colors=[]; symbols=[]; custom=[]
    max_cubes=1
    for row_key in ordered_rows:
        entries=sorted(grouped[row_key],key=lambda r:(_finite(r.get('amount',0.0)),str(r.get('cube_label') or r.get('cube_key'))),reverse=True)
        max_cubes=max(max_cubes,len(entries))
        for pos,r in enumerate(entries,1):
            amount=_finite(r.get('amount',0.0))
            x.append(pos)
            y.append(_short_label(row_labels.get(row_key,row_key),48))
            colors.append(r.get('color','#2563EB'))
            symbols.append(r.get('symbol','square'))
            custom.append([
                row_labels.get(row_key,row_key),
                str(r.get('cube_label') or r.get('cube_key') or ''),
                _num_text(amount),
                str(r.get('extra','')),
            ])
    trace={
        'type':'scatter','mode':'markers','x':x,'y':y,
        'marker':{'symbol':symbols,'size':18,'color':colors,'line':{'color':'#FFFFFF','width':1.4},'opacity':0.94},
        'customdata':custom,
        'hovertemplate':'<b>%{customdata[0]}</b><br>'+html.escape(cube_title)+': %{customdata[1]}<br>'+html.escape(amount_title)+': %{customdata[2]}<br>%{customdata[3]}<extra></extra>'
    }
    lay=_layout(title,max(390,min(1900,170+36*len(ordered_rows))),x_title='Cubitos usados',y_title=row_title)
    lay['margin']['l']=255
    lay['xaxis'].update({'tickmode':'linear','dtick':1,'range':[0,max_cubes+1],'gridcolor':'rgba(148,163,184,0.15)'})
    lay['yaxis']['autorange']='reversed'
    lay['showlegend']=False
    return {'data':[trace],'layout':lay,'config':{'responsive':True,'displaylogo':False,'scrollZoom':False}}


_factores_conversion=globals().get('FACTORES_CONVERSION',{})
_factor_bg=max(EPS,_finite(_factores_conversion.get('BG',1.0)))
_factor_rl=max(EPS,_finite(_factores_conversion.get('RL',600.0)))
_factor_rc=max(EPS,_finite(_factores_conversion.get('RC',303.03)))


def _producer_gasbc_eq(pk):
    pm=producer_meta.get(pk,{})
    t=str(pm.get('type',''))
    load=max(0.0,_finite(pm.get('load',0.0)))
    if t=='BG':
        return load*_factor_bg
    if t in {'L','C'}:
        return load
    if t=='RL':
        return load*_factor_rl
    if t=='RC':
        return load*_factor_rc
    return load


def _producer_biogas_eq(pk):
    return _safe_div(_producer_gasbc_eq(pk),_factor_bg)


def _route_output_gasbc_eq(rec, pk):
    pm=producer_meta.get(pk,{})
    q=max(0.0,_finite(_route_output_for_producer(rec,pk)))
    t=str(pm.get('type',''))
    if t in {'L'}:
        return q*_factor_rl
    if t in {'C'}:
        return q*_factor_rc
    return q


def _producer_detail(pk):
    pm=producer_meta.get(pk,{})
    return (
        f"Tipo: {html.escape(str(TYPE_NAME.get(str(pm.get('type','')),pm.get('type',''))))}<br>"
        f"Biometano gaseoso eq.: {_num_text(_producer_gasbc_eq(pk))} Nm³<br>"
        f"Biogás crudo eq.: {_num_text(_producer_biogas_eq(pk))} Nm³<br>"
        f"Costo total: {_money_text(_finite(pm.get('fixed_cost',0.0))+_finite(pm.get('variable_cost',0.0)))}"
    )


_plant_keys_stats=[k for k in producer_keys if producer_meta[k]['kind']=='Planta']
_plant_labels_stats=[producer_meta[k]['label'] for k in _plant_keys_stats]
_plant_rows_stats=[producer_short_labels[k] for k in _plant_keys_stats]
_plant_colors_stats=[producer_meta[k].get('color',producer_colors.get(k,'#7B2CBF')) for k in _plant_keys_stats]
_plant_hover_stats=[_producer_detail(k) for k in _plant_keys_stats]


def _add_pipe_row(rows,key,label,flow,capacity,color,product):
    flow=max(0.0,_finite(flow))
    if flow<=TOL_FLUJO_CERO_NM3:
        return
    capacity=max(0.0,_finite(capacity))
    rows.append({
        'key':str(key),'label':str(label),'flow':flow,'capacity':capacity,
        'utilization':100*_safe_div(flow,capacity),'color':color,'product':product,
    })


_short_biogas_rows=[]
for h in H_CORTO:
    _add_pipe_row(_short_biogas_rows,f'SHORTBIO|{h}',producer_meta.get(f'SHORTBIO|{h}',{}).get('label',f'Gasoducto corto de biogás crudo · {h}'),
                  short_biogas_pipe_flow[h],GCORTO.loc[h,'capacidad_max_nm3_dia'],'#0E7490','Biogás Crudo')
for h in H_EHUB_SOLO:
    _add_pipe_row(_short_biogas_rows,f'SHORTBIOHUBEMP|{h}',producer_meta.get(f'SHORTBIOHUBEMP|{h}',{}).get('label',f'Gasoducto corto HUB biogás crudo · {h}'),
                  short_biogas_hub_empresa_solo_flow[h],EHUB.loc[h,'capacidad_max_nm3_dia'],'#0F766E','Biogás Crudo')
for h in H_BIOC2C:
    _add_pipe_row(_short_biogas_rows,f'SHORTBIOC2C|{h}',producer_meta.get(f'SHORTBIOC2C|{h}',{}).get('label',f'Gasoducto corto biogás crudo HUB · {h}'),
                  short_biogas_c2c_pipe_flow[h],BIOC2C.loc[h,'capacidad_max_nm3_dia'],'#0F766E','Biogás Crudo')
for h in H_PHUB:
    _add_pipe_row(_short_biogas_rows,f'SHORTBIOPHUB|{h}',producer_meta.get(f'SHORTBIOPHUB|{h}',{}).get('label',f'Gasoducto corto biogás crudo HUB P · {h}'),
                  short_biogas_p_hub_pipe_flow[h],PHUB.loc[h,'capacidad_max_nm3_dia'],'#0F766E','Biogás Crudo')

_short_gasbc_rows=[]
for h in H_GECORTO:
    _add_pipe_row(_short_gasbc_rows,f'SHORTGASIN|{h}',producer_meta.get(f'SHORTGASIN|{h}',{}).get('label',f'Gasoducto corto de GasBC a entrada · {h}'),
                  short_gasbc_inlet_pipe_flow[h],GECORTO.loc[h,'capacidad_max_nm3_dia'],'#2563EB','Biometano Gaseoso')
for h in H_GDCORTO:
    _add_pipe_row(_short_gasbc_rows,f'SHORTGASDIRECT|{h}',producer_meta.get(f'SHORTGASDIRECT|{h}',{}).get('label',f'Gasoducto corto directo de GasBC · {h}'),
                  short_gasbc_direct_pipe_flow[h],GDCORTO.loc[h,'capacidad_max_nm3_dia'],'#1D4ED8','Biometano Gaseoso')
for h in H_GSCORTO:
    _add_pipe_row(_short_gasbc_rows,f'SHORTGAS|{h}',producer_meta.get(f'SHORTGAS|{h}',{}).get('label',f'Gasoducto corto de GasBC salida · {h}'),
                  short_gasbc_pipe_flow[h],GSCORTO.loc[h,'capacidad_max_nm3_dia'],'#1E40AF','Biometano Gaseoso')

_short_pipe_rows=_short_biogas_rows+_short_gasbc_rows
_conventional_pipe_rows=[]
for g in G:
    _add_pipe_row(_conventional_pipe_rows,f'PIPE|{g}',f"{g} · {GA.loc[g,'nombre']}",pipe_flow[g],GA.loc[g,'capacidad_efectiva_nm3'],type_colors['Gasoducto'],'Gasoducto convencional')



_installed_asset_labels=[]; _installed_asset_values=[]; _installed_asset_colors=[]; _installed_asset_hover=[]
for _t in T:
    _installed_asset_labels.append(f"Productora · {TYPE_NAME.get(str(_t),str(_t))}")
    _installed_asset_values.append(int(round(sum(max(0.0,_finite(value('NP',_t,_s))) for _s in S))))
    _installed_asset_colors.append(type_colors.get(_t,'#7B2CBF'))
    _installed_asset_hover.append('Unidades productoras activas instaladas de este tipo.')


def _count_active_short_assets(_items,_flow_map,_active_check=None):
    total=0
    for _h in _items:
        installed=_finite(_flow_map.get(_h,0.0))>TOL_FLUJO_CERO_NM3
        if _active_check is not None:
            try:
                installed=installed or bool(_active_check(_h))
            except Exception:
                installed=installed
        if installed:
            total+=1
    return total


def _add_installed_short_asset(_label,_items,_flow_map,_color,_extra,_active_check=None):
    _installed_asset_labels.append(_label)
    _installed_asset_values.append(_count_active_short_assets(_items,_flow_map,_active_check))
    _installed_asset_colors.append(_color)
    _installed_asset_hover.append(_extra)


_add_installed_short_asset('Gasoducto corto · Biogás crudo empresa-destino',H_CORTO,short_biogas_pipe_flow,'#0E7490','Gasoductos cortos de biogás crudo instalados o con flujo positivo.',lambda _h: _finite(value('BCP',_h))>EPS or (_h in H_EHUB_SET and _finite(value('BCP_HUB',_h))>EPS))
_add_installed_short_asset('Gasoducto corto · Biogás crudo HUB empresa',H_EHUB_SOLO,short_biogas_hub_empresa_solo_flow,'#0F766E','Gasoductos cortos HUB de biogás crudo desde empresa instalados o con flujo positivo.',lambda _h: _finite(value('BCP_HUB',_h))>EPS)
_add_installed_short_asset('Gasoducto corto · Biogás crudo HUB cliente-cliente',H_BIOC2C,short_biogas_c2c_pipe_flow,'#0F766E','Gasoductos cortos HUB cliente-cliente instalados o con flujo positivo.',lambda _h: _finite(value('BCP2C',_h))>EPS)
_add_installed_short_asset('Gasoducto corto · Biogás crudo HUB productora',H_PHUB,short_biogas_p_hub_pipe_flow,'#0F766E','Gasoductos cortos HUB nodales de biogás crudo instalados o con flujo positivo.',lambda _h: _finite(value('BCP_P',_h))>EPS)
_add_installed_short_asset('Gasoducto corto · GasBC entrada corredor',H_GECORTO,short_gasbc_inlet_pipe_flow,'#2563EB','Gasoductos cortos de GasBC hacia entrada de corredor instalados o con flujo positivo.',lambda _h: _finite(value('BGCE',_h))>EPS)
_add_installed_short_asset('Gasoducto corto · GasBC directo',H_GDCORTO,short_gasbc_direct_pipe_flow,'#1D4ED8','Gasoductos cortos directos de GasBC instalados o con flujo positivo.',lambda _h: _finite(value('BGCD',_h))>EPS)
_add_installed_short_asset('Gasoducto corto · GasBC salida corredor',H_GSCORTO,short_gasbc_pipe_flow,'#1E40AF','Gasoductos cortos de GasBC desde salida de corredor instalados o con flujo positivo.',lambda _h: _finite(value('BGCP',_h))>EPS)
_register('stats','Cantidad de productoras y gasoductos cortos instalados por tipo',
          _bar_h('Cantidad de productoras y gasoductos cortos instalados por tipo',_installed_asset_labels,_installed_asset_values,
                 _installed_asset_colors,'integer',hover_extra=_installed_asset_hover,x_title='Cantidad instalada'),
          'Cuenta unidades productoras activas por tecnologia y gasoductos cortos instalados por familia.')

_client_margin_values=_route_sum_by('cliente_id','margen_operativo_por_ruta_usd',J)
_client_transport_values=_route_sum_by('cliente_id','costo_transporte_usd',J)
_company_margin_values=_route_sum_by('empresa_id','margen_operativo_por_ruta_usd',I)
_company_transport_values=_route_sum_by('empresa_id','costo_transporte_usd',I)

_register('stats','Cantidad de demanda satisfecha a cada cliente',
          _bar_h('Cantidad de demanda satisfecha a cada cliente',df_metricas_clientes['label'].tolist(),df_metricas_clientes['served'].tolist(),
                 [client_colors[j] for j in J],'integer',x_title='Cantidad suministrada (Nm³)'))
_register('stats','Porcentaje de demanda satisfecha por cliente',
          _bar_h('Porcentaje de demanda satisfecha por cliente',df_metricas_clientes['label'].tolist(),df_metricas_clientes['coverage'].tolist(),
                 [client_colors[j] for j in J],'percent',x_title='Cobertura (%)'))
_register('stats','Margen Operativo Total por Cliente',
          _bar_h('Margen Operativo Total por Cliente',df_metricas_clientes['label'].tolist(),_client_margin_values,
                 [client_colors[j] for j in J],'money',x_title='Margen operativo total (US$)'))
_register('stats','Costo Variable de Transporte Total por Cliente',
          _bar_h('Costo Variable de Transporte Total por Cliente',df_metricas_clientes['label'].tolist(),_client_transport_values,
                 [client_colors[j] for j in J],'money',x_title='Costo variable de transporte (US$)'))

_register('stats','Porcentaje de Oferta Máxima Comprada a cada empresa',
          _bar_h('Porcentaje de Oferta Máxima Comprada a cada empresa',df_metricas_empresas['label'].tolist(),df_metricas_empresas['utilization'].tolist(),
                 [company_colors[i] for i in I],'percent',x_title='Utilización de la oferta máxima (%)'))
_register('stats','Cantidad de oferta comprada a cada Empresa',
          _bar_h('Cantidad de oferta comprada a cada Empresa',df_metricas_empresas['label'].tolist(),df_metricas_empresas['supply'].tolist(),
                 [company_colors[i] for i in I],'number',x_title='Biogás crudo comprado (Nm³)'))
_register('stats','Margen Operativo Total por Empresa',
          _bar_h('Margen Operativo Total por Empresa',df_metricas_empresas['label'].tolist(),_company_margin_values,
                 [company_colors[i] for i in I],'money',x_title='Margen operativo total (US$)'))
_register('stats','Costo Variable de Transporte Total por Empresa',
          _bar_h('Costo Variable de Transporte Total por Empresa',df_metricas_empresas['label'].tolist(),_company_transport_values,
                 [company_colors[i] for i in I],'money',x_title='Costo variable de transporte (US$)'))

_register('stats','Porcentaje de Capacidad Máxima Utilizada de cada productora',
          _bar_h('Porcentaje de Capacidad Máxima Utilizada de cada productora',_plant_labels_stats,
                 [100*_safe_div(producer_meta[k]['load'],producer_meta[k]['capacity']) for k in _plant_keys_stats],
                 _plant_colors_stats,'percent',hover_extra=_plant_hover_stats,x_title='Utilización (%)'))
_register('stats','Porcentaje de Capacidad Máxima Utilizada de cada Gasoducto Corto',
          _bar_h('Porcentaje de Capacidad Máxima Utilizada de cada Gasoducto Corto',
                 [r['label'] for r in _short_pipe_rows],[r['utilization'] for r in _short_pipe_rows],
                 [r['color'] for r in _short_pipe_rows],'percent',
                 hover_extra=[f"{r['product']}<br>Flujo: {_num_text(r['flow'])} Nm³<br>Capacidad: {_num_text(r['capacity'])} Nm³/día" for r in _short_pipe_rows],
                 x_title='Utilización (%)'))
_register('stats','Porcentaje de Capacidad Máxima Utilizada de cada Gasoducto Convencional',
          _bar_h('Porcentaje de Capacidad Máxima Utilizada de cada Gasoducto Convencional',
                 [r['label'] for r in _conventional_pipe_rows],[r['utilization'] for r in _conventional_pipe_rows],
                 [r['color'] for r in _conventional_pipe_rows],'percent',
                 hover_extra=[f"Flujo: {_num_text(r['flow'])} Nm³<br>Capacidad: {_num_text(r['capacity'])} Nm³/día" for r in _conventional_pipe_rows],
                 x_title='Utilización (%)'))
_register('stats','Cantidad Transportada por Gasoducto de Biometano Gaseoso Utilizado',
          _bar_h('Cantidad Transportada por Gasoducto de Biometano Gaseoso Utilizado',
                 [r['label'] for r in _short_gasbc_rows],[r['flow'] for r in _short_gasbc_rows],
                 [r['color'] for r in _short_gasbc_rows],'number',x_title='Biometano gaseoso transportado (Nm³)'))
_register('stats','Cantidad Transportada por Gasoducto de Biogás Crudo Utilizado',
          _bar_h('Cantidad Transportada por Gasoducto de Biogás Crudo Utilizado',
                 [r['label'] for r in _short_biogas_rows],[r['flow'] for r in _short_biogas_rows],
                 [r['color'] for r in _short_biogas_rows],'number',x_title='Biogás crudo transportado (Nm³)'))
_register('stats','Cantidad Transportada por Gasoducto Convencional Utilizado',
          _bar_h('Cantidad Transportada por Gasoducto Convencional Utilizado',
                 [r['label'] for r in _conventional_pipe_rows],[r['flow'] for r in _conventional_pipe_rows],
                 [r['color'] for r in _conventional_pipe_rows],'number',x_title='Cantidad transportada (Nm³)'))
_register('stats','Cantidad de Biometano Gaseoso Equivalente transformado | utilizado por Productora',
          _bar_h('Cantidad de Biometano Gaseoso Equivalente transformado | utilizado por Productora',
                 _plant_labels_stats,[_producer_gasbc_eq(k) for k in _plant_keys_stats],
                 _plant_colors_stats,'number',hover_extra=_plant_hover_stats,x_title='Biometano gaseoso equivalente (Nm³)'))
_register('stats','Cantidad de Biogás Crudo Equivalente transformado | utilizado por Productora',
          _bar_h('Cantidad de Biogás Crudo Equivalente transformado | utilizado por Productora',
                 _plant_labels_stats,[_producer_biogas_eq(k) for k in _plant_keys_stats],
                 _plant_colors_stats,'number',hover_extra=_plant_hover_stats,x_title='Biogás crudo equivalente (Nm³)'))

_vehicle_tours=defaultdict(float)
for _tour in transport_tours_payload:
    _vehicle_tours[str(_tour.get('vehicle','Vehículo'))]+=_finite(_tour.get('number_of_vehicles',0.0))
_register('stats','Número de Viajes Ida/Vuelta por Tipo de Vehículo',
          _bar_h('Número de Viajes Ida/Vuelta por Tipo de Vehículo',
                 list(_vehicle_tours.keys()),list(_vehicle_tours.values()),
                 '#475569','number',x_title='Viajes ida/vuelta'))

_register('stats','Costos Fijos y Variables por Productora',
          _grouped_bar('Costos Fijos y Variables por Productora',_plant_rows_stats,[
              {'name':'Costo fijo','values':[_finite(producer_meta[k]['fixed_cost']) for k in _plant_keys_stats],'color':'#4F46E5','extra':_plant_hover_stats},
              {'name':'Costo variable','values':[_finite(producer_meta[k]['variable_cost']) for k in _plant_keys_stats],'color':'#F59E0B','extra':_plant_hover_stats},
          ],'money','Costo (US$)'))
_register('stats','Número de Empresas cuyas rutas utilizaron una Productora vs Número de Clientes que la necesitaron',
          _grouped_bar('Número de Empresas cuyas rutas utilizaron una Productora vs Número de Clientes que la necesitaron',_plant_rows_stats,[
              {'name':'Empresas con rutas','values':[len(companies_by_producer[k]) for k in _plant_keys_stats],'color':'#2563EB','extra':_plant_hover_stats},
              {'name':'Clientes que la necesitaron','values':[len(clients_by_producer[k]) for k in _plant_keys_stats],'color':'#0F766E','extra':_plant_hover_stats},
          ],'integer','Cantidad'))


# -------------------------- ESTADÍSTICAS: barras con cubitos -------------------
_company_cubes=[]
if len(df_rutas_estadisticas):
    _tmp=df_rutas_estadisticas.groupby(['cliente_id','empresa_id'])['cantidad_nm3'].sum()
    for (_cliente_id,_empresa_id),_amount in _tmp.items():
        _company_cubes.append({
            'row_key':str(_cliente_id),
            'row_label':client_labels.get(str(_cliente_id),str(_cliente_id)),
            'cube_key':str(_empresa_id),
            'cube_label':company_labels.get(str(_empresa_id),str(_empresa_id)),
            'amount':_finite(_amount),
            'color':company_colors.get(str(_empresa_id),'#2563EB'),
            'extra':'Biometano gaseoso equivalente recibido por el cliente',
        })
_register('stats','Empresas utilizadas por Cliente',
          _cube_strip('Empresas utilizadas por Cliente',_company_cubes,'Cliente','Empresa','Biometano gaseoso equivalente (Nm³)'))

def _client_asset_visual(pk):
    pm=producer_meta.get(pk,{})
    ptype=str(pm.get('type',''))
    subtype=str(pm.get('subtype',''))
    key=str(pk)
    if pm.get('kind')=='Planta':
        return 'Productora','square',pm.get('color',producer_colors.get(pk,'#7B2CBF'))
    if key.startswith('SHORT') or subtype.startswith(('biogas_crudo','gasbc_corto')) or 'Corto' in ptype or 'Biogás Crudo' in ptype:
        return 'Gasoducto corto','triangle-up',pm.get('color','#0E7490')
    return 'Gasoducto convencional','circle',pm.get('color',type_colors.get('Gasoducto','#212121'))


_client_asset_cubes_by_pair=defaultdict(float)
if len(df_rutas_estadisticas):
    for _,_rr in df_rutas_estadisticas.iterrows():
        _cliente_id=str(_rr.get('cliente_id'))
        _rec=_rr.to_dict()
        for _pk in list(_rr.get('_producer_keys',[]) or []):
            if _pk not in producer_meta:
                continue
            _client_asset_cubes_by_pair[(_cliente_id,_pk)]+=_route_output_gasbc_eq(_rec,_pk)
_client_asset_cubes=[]
for (_cliente_id,_pk),_amount in _client_asset_cubes_by_pair.items():
    _asset_group,_asset_symbol,_asset_color=_client_asset_visual(_pk)
    _client_asset_cubes.append({
        'row_key':_cliente_id,
        'row_label':client_labels.get(_cliente_id,_cliente_id),
        'cube_key':_pk,
        'cube_label':f"{_asset_group} · {producer_labels.get(_pk,_pk)}",
        'amount':_amount,
        'color':_asset_color,
        'symbol':_asset_symbol,
        'extra':f"Tipo de elemento: {_asset_group}<br>{_producer_detail(_pk)}",
    })
_register('stats','Productoras, Gasoductos Cortos y Gasoductos Convencionales utilizadas por Cliente',
          _cube_strip('Productoras, Gasoductos Cortos y Gasoductos Convencionales utilizadas por Cliente',
                      _client_asset_cubes,'Cliente','Elemento utilizado','Flujo asociado (Nm³ GasBC eq.)'))

# -------------------------- ESTADÍSTICAS: histogramas y boxplots ---------------
# -------------------------- ESTADÍSTICAS: histogramas y boxplots ---------------
_df_empresas_con_compra_stats=(
    df_metricas_empresas.loc[df_metricas_empresas['supply'].map(_finite)>TOL_FLUJO_CERO_NM3].copy()
    if len(df_metricas_empresas) else df_metricas_empresas
)
_df_clientes_con_venta_stats=(
    df_metricas_clientes.loc[df_metricas_clientes['served'].map(_finite)>TOL_FLUJO_CERO_NM3].copy()
    if len(df_metricas_clientes) else df_metricas_clientes
)
_register_hist_box('Beneficios Diarios de las Empresas',
                   _observations_from_df(_df_empresas_con_compra_stats,'label','benefit'),'Beneficio diario (US$)','#067647')
_register_hist_box('Cantidad Comprada de las Empresas',
                   _observations_from_df(_df_empresas_con_compra_stats,'label','supply'),'Cantidad comprada (Nm³)','#2563EB')
_register_hist_box('Beneficios Diarios de los Clientes',
                   _observations_from_df(_df_clientes_con_venta_stats,'label','benefit'),'Beneficio diario (US$)','#0F766E')
_register_hist_box('Cantidad Vendida de los Cliente',
                   _observations_from_df(_df_clientes_con_venta_stats,'label','served'),'Cantidad vendida (Nm³)','#14B8A6')
_register_hist_box('Distancias de Transporte de las Ruta',
                   [(f"{r.route_id} · {r.empresa} → {r.cliente}",r.distancia_total_km) for r in df_rutas_estadisticas.itertuples(index=False)] if len(df_rutas_estadisticas) else [],
                   'Distancia total (km)','#7B2CBF')
_register_hist_box('Costos Variables Totales de Transporte de las Rutas',
                   [(f"{r.route_id} · {r.empresa} → {r.cliente}",r.costo_transporte_usd) for r in df_rutas_estadisticas.itertuples(index=False)] if len(df_rutas_estadisticas) else [],
                   'Costo variable de transporte (US$)','#C2410C')
_register_hist_box('Margen Operativo de las Rutas',
                   [(f"{r.route_id} · {r.empresa} → {r.cliente}",r.margen_operativo_por_ruta_usd) for r in df_rutas_estadisticas.itertuples(index=False)] if len(df_rutas_estadisticas) else [],
                   'Margen operativo (US$)','#15803D')
_register_hist_box('Cantidad de NM³ de Biometano Gaseoso de los gasoductos cortos de Biometano Gaseoso',
                   [(r['label'],r['flow']) for r in _short_gasbc_rows],
                   'Biometano gaseoso transportado (Nm³)','#1D4ED8')
_register_hist_box('Cantidad de NM³ de Biogás Crudo de los gasoductos de Biogás Crudo',
                   [(r['label'],r['flow']) for r in _short_biogas_rows],
                   'Biogás crudo transportado (Nm³)','#0E7490')


# -------------------------- ESTADÍSTICAS: scatterplots -------------------------
_producer_biogas_points=[
    {'label':producer_labels[k],'x':_producer_biogas_eq(k),'y':_finite(producer_meta[k]['variable_cost']),
     'color':producer_meta[k].get('color',producer_colors.get(k,'#7B2CBF')),'symbol':'square',
     'extra':f"Tipo: {TYPE_NAME.get(str(producer_meta[k].get('type','')),producer_meta[k].get('type',''))}<br>{_producer_detail(k)}"}
    for k in _plant_keys_stats
]
_register('stats','Biogás Crudo Equivalente por Productora vs Costo Variable de Transformación',
          _scatter('Biogás Crudo Equivalente por Productora vs Costo Variable de Transformación',
                   _producer_biogas_points,'Biogás crudo equivalente (Nm³)','Costo variable de transformación (US$)'))

_producer_gasbc_points=[
    {'label':producer_labels[k],'x':_producer_gasbc_eq(k),'y':_finite(producer_meta[k]['variable_cost']),
     'color':producer_meta[k].get('color',producer_colors.get(k,'#7B2CBF')),'symbol':'square',
     'extra':f"Tipo: {TYPE_NAME.get(str(producer_meta[k].get('type','')),producer_meta[k].get('type',''))}<br>{_producer_detail(k)}"}
    for k in _plant_keys_stats
]
_register('stats','Biometano Gaseoso Equivalente por Productora vs Costo Variable Total de Transformación',
          _scatter('Biometano Gaseoso Equivalente por Productora vs Costo Variable Total de Transformación',
                   _producer_gasbc_points,'Biometano gaseoso equivalente (Nm³)','Costo variable total de transformación (US$)'))

_route_count_by_client=(df_rutas_estadisticas.groupby('cliente_id')['route_id'].nunique() if len(df_rutas_estadisticas) else pd.Series(dtype=float))
_client_unique_plants=defaultdict(set)
if len(df_rutas_estadisticas):
    for _,_rr in df_rutas_estadisticas.iterrows():
        _cliente_id=str(_rr.get('cliente_id'))
        for _pk in list(_rr.get('_producer_keys',[]) or []):
            if _pk in producer_meta and producer_meta[_pk].get('kind')=='Planta':
                _client_unique_plants[_cliente_id].add(_pk)

_client_route_points=[]
for _j in J:
    _client_route_points.append({
        'label':client_labels[_j],
        'x':_finite(client_revenue[_j]),
        'y':_finite(_route_count_by_client.get(_j,_route_count_by_client.get(str(_j),0.0))),
        'color':client_colors.get(_j,'#0F766E'),
        'symbol':'circle',
        'extra':f"Demanda satisfecha: {_num_text(_series_by_id(df_metricas_clientes,'id','served',[_j])[0])} Nm³",
    })
_register('stats','Ingresos Totales Asociados a un Cliente vs Número de Rutas Asociada a ese Cliente',
          _scatter('Ingresos Totales Asociados a un Cliente vs Número de Rutas Asociada a ese Cliente',
                   _client_route_points,'Ingresos totales asociados al cliente (US$)','Número de rutas asociadas'))

_client_metric_by_id={str(row.get('id')):row for _,row in df_metricas_clientes.iterrows()} if len(df_metricas_clientes) else {}
_client_price_coverage_points=[]
_client_price_quantity_points=[]
_client_routes_benefit_points=[]
for _j in J:
    _m=_client_metric_by_id.get(str(_j),{})
    _price=_finite(_m.get('price',CL.loc[_j,'precio_venta_usd_nm3']))
    _served=_finite(_m.get('served',0.0))
    _coverage=_finite(_m.get('coverage',0.0))
    _benefit=_finite(_m.get('benefit',0.0))
    _routes=_finite(_route_count_by_client.get(_j,_route_count_by_client.get(str(_j),0.0)))
    _client_price_coverage_points.append({
        'label':client_labels[_j],'x':_price,'y':_coverage,'color':client_colors.get(_j,'#0F766E'),'symbol':'circle',
        'extra':f"Cantidad vendida: {_num_text(_served)} Nm³<br>Rutas: {_routes:,.0f}",
    })
    _client_price_quantity_points.append({
        'label':client_labels[_j],'x':_price,'y':_served,'color':client_colors.get(_j,'#0F766E'),'symbol':'diamond',
        'extra':f"Demanda satisfecha: {_coverage:,.2f}%<br>Rutas: {_routes:,.0f}",
    })
    _client_routes_benefit_points.append({
        'label':client_labels[_j],'x':_routes,'y':_benefit,'color':client_colors.get(_j,'#0F766E'),'symbol':'square',
        'extra':f"Precio de venta: US$ {_price:,.4f}/Nm³<br>Cantidad vendida: {_num_text(_served)} Nm³",
    })
_register('stats','Precio de Venta del Cliente vs Porcentaje de Demanda Satisfecha',
          _scatter('Precio de Venta del Cliente vs Porcentaje de Demanda Satisfecha',
                   _client_price_coverage_points,'Precio de venta del cliente (US$/Nm³)','Demanda satisfecha (%)'))
_register('stats','Precio de Venta del Cliente vs Cantidad Vendida',
          _scatter('Precio de Venta del Cliente vs Cantidad Vendida',
                   _client_price_quantity_points,'Precio de venta del cliente (US$/Nm³)','Cantidad vendida (Nm³)'))
_register('stats','Número de Rutas del Cliente vs Beneficio Total del Cliente',
          _scatter('Número de Rutas del Cliente vs Beneficio Total del Cliente',
                   _client_routes_benefit_points,'Número de rutas del cliente','Beneficio total del cliente (US$)'))

_client_plant_points=[]
for _j in J:
    _client_plant_points.append({
        'label':client_labels[_j],
        'x':_finite(client_revenue[_j]),
        'y':len(_client_unique_plants.get(str(_j),set())),
        'color':client_colors.get(_j,'#0F766E'),
        'symbol':'diamond',
        'extra':'Productoras únicas: '+(', '.join(producer_short_labels[k] for k in sorted(_client_unique_plants.get(str(_j),set()))) or 'ninguna'),
    })
_register('stats','Ingresos Totales de Clientes vs Número de Productoras Únicas usadas para ese Cliente',
          _scatter('Ingresos Totales de Clientes vs Número de Productoras Únicas usadas para ese Cliente',
                   _client_plant_points,'Ingresos totales del cliente (US$)','Número de productoras únicas'))

_route_points=[]
if len(df_rutas_estadisticas):
    for _r in df_rutas_estadisticas.itertuples(index=False):
        _label=f"{_r.route_id} · {_r.empresa} → {_r.cliente}"
        _extra=(
            f"Familia: {_r.familia}<br>Cantidad vendida: {_r.cantidad_nm3:,.2f} Nm³"
            f"<br>Ingresos: {_money_text(_r.ingreso_usd)}"
            f"<br>Costo variable total: {_money_text(_r.costo_variable_total_usd)}"
            f"<br>Costo transporte: {_money_text(_r.costo_transporte_usd)}"
            f"<br>Margen operativo: {_money_text(_r.margen_operativo_por_ruta_usd)}"
        )
        _route_points.append({
            'label':_label,'distance':_finite(_r.distancia_total_km),'quantity':_finite(_r.cantidad_nm3),
            'variable_cost':_finite(_r.costo_variable_total_usd),'transport_cost':_finite(_r.costo_transporte_usd),
            'margin':_finite(_r.margen_operativo_por_ruta_usd),
            'color':company_colors.get(str(_r.empresa_id),'#1565C0'),'extra':_extra,
        })

_register('stats','Margen Operativo vs Distancia Total de la Ruta',
          _scatter('Margen Operativo vs Distancia Total de la Ruta',
                   [{'label':p['label'],'x':p['distance'],'y':p['margin'],'color':p['color'],'symbol':'circle','extra':p['extra']} for p in _route_points],
                   'Distancia total de la ruta (km)','Margen operativo de la ruta (US$)'))
_register('stats','Costo Variable vs Distancia Total de la Ruta',
          _scatter('Costo Variable vs Distancia Total de la Ruta',
                   [{'label':p['label'],'x':p['distance'],'y':p['variable_cost'],'color':p['color'],'symbol':'circle','extra':p['extra']} for p in _route_points],
                   'Distancia total de la ruta (km)','Costo variable total de la ruta (US$)'))
_register('stats','Margen Operativo vs Costos de Transporte total de la Ruta',
          _scatter('Margen Operativo vs Costos de Transporte total de la Ruta',
                   [{'label':p['label'],'x':p['transport_cost'],'y':p['margin'],'color':p['color'],'symbol':'circle','extra':p['extra']} for p in _route_points],
                   'Costo de transporte total de la ruta (US$)','Margen operativo de la ruta (US$)'))
_register('stats','Margen Operativo vs Cantidad Vendida de las Rutas de Biometano Gaseoso',
          _scatter('Margen Operativo vs Cantidad Vendida de las Rutas de Biometano Gaseoso',
                   [{'label':p['label'],'x':p['quantity'],'y':p['margin'],'color':p['color'],'symbol':'circle','extra':p['extra']} for p in _route_points],
                   'Cantidad vendida de la ruta (Nm³ GasBC eq.)','Margen operativo de la ruta (US$)'))
_register('stats','Distancia Total vs Cantidad Vendida de las Rutas',
          _scatter('Distancia Total vs Cantidad Vendida de las Rutas',
                   [{'label':p['label'],'x':p['quantity'],'y':p['distance'],'color':p['color'],'symbol':'circle','extra':p['extra']} for p in _route_points],
                   'Cantidad vendida de la ruta (Nm³)','Distancia total de la ruta (km)'))

# -----------------------------------------------------------------------------
# HTML de ambos paneles. Los números históricos aparecen al inicio de Estadísticas.
# -----------------------------------------------------------------------------
def _table(df,cols=None,max_rows=500):
    d=df if cols is None else df[cols]
    return d.head(max_rows).to_html(index=False,classes='data-table',border=0,escape=True,float_format=lambda x:f'{x:,.4f}')

def make_model_data_html():
    empresas_costos = empresas[['id','nombre',COL_PRECIO_BIOGAS_CRUDO]].copy()
    configuracion_plantas = (
        'Libre: decisión del MILP' if NUMERO_TOTAL_PLANTAS is False
        else f'Exactamente {NUMERO_TOTAL_PLANTAS}'
    )
    return f"""
    <section class='data-hero'>
      <span class='data-kicker'>ENTRADAS DEL MODELO</span>
      <h2>Datos del modelamiento</h2>
      <p>Parámetros cargados directamente desde el ZIP. Los factores de conversión tienen una única fuente de verdad: <code>factores_conversion.csv</code>.</p>
    </section>
    <section class='summary-grid data-summary'>
      <div class='summary-card'><span>Empresas</span><b>{len(empresas)}</b></div>
      <div class='summary-card'><span>Clientes</span><b>{len(clientes)}</b></div>
      <div class='summary-card'><span>Productoras candidatas</span><b>{len(sitios)}</b></div>
      <div class='summary-card'><span>Tecnologías</span><b>{len(tipos)}</b></div>
      <div class='summary-card'><span>Gasoductos existentes</span><b>{len(gasoductos)}</b></div><div class='summary-card'><span>Gasoductos cortos Candidatos<br><small>cliente + ubicación posible</small></span><b>{len(gasoductos_biogas_cortos_todos)}</b></div>
      <div class='summary-card'><span>Número total de plantas</span><b>{configuracion_plantas}</b></div>
    </section>
    <section class='data-block'>
      <div class='section-heading'><span>01</span><div><h2>Gráficos de entrada</h2><p>Escalas, costos y capacidades utilizados por el MILP.</p></div></div>
      {''.join(data_chart_sections)}
    </section>
    <section class='data-block'>
      <div class='section-heading'><span>02</span><div><h2>Parámetros principales</h2><p>Tablas compactas y sin metadatos de estimación energética.</p></div></div>
      <details open><summary>Costos variables de transformación</summary>{_table(fuentes_costos,['codigo_pdf','tipo_instalacion','costo_variable_usd_nm3'])}</details>
      <details open><summary>Precio de mercado del biogas crudo por empresa</summary>{_table(empresas_costos)}</details>
      <details><summary>Factores de Conversión</summary>{_table(factores_conversion,['codigo_pdf','tipo_instalacion','formato_entrada','formato_salida','factor_conversion_salida_por_entrada'])}</details>
      <details><summary>Tipos de planta</summary>{_table(tipos,['codigo_pdf','tipo_instalacion','formato_entrada','formato_salida','capacidad_max_nm3','costo_fijo_usd'])}</details>
      <details><summary>Corredores de gasoducto bidireccionales e independientes</summary>{_table(gasoductos,['id','nombre','punto_a_zona','punto_a_nombre','punto_b_zona','punto_b_nombre','capacidad_efectiva_nm3','distancia_red_estimada_km','costo_fijo_usd_dia','costo_variable_usd_km_nm3'])}</details><details open><summary>Gasoductos cortos de biogás crudo Candidatos</summary>{_table(gasoductos_biogas_cortos_todos,['id','empresa_id','empresa','destino_tipo','destino_id','destino_nombre','nodo_destino_id','distancia_recta_km','costo_fijo_usd_dia','costo_variable_usd_nm3_km','capacidad_max_nm3_dia','empresa_disponible_en_milp']) if len(gasoductos_biogas_cortos_todos) else '<p>No se detectaron pares empresa-cliente ni empresa-ubicación bajo el largo máximo configurado.</p>'}</details><details><summary>Gasoductos cortos de GasBC: BG en empresa → entrada del corredor</summary>{_table(gasoductos_gasbc_cortos_empresa_entrada_todos,['id','empresa_id','empresa','nodo_origen_id','direccion_id','gasoducto_id','entrada_nombre','distancia_recta_km','costo_fijo_usd_dia','costo_variable_usd_nm3_km','capacidad_max_nm3_dia','empresa_disponible_en_milp']) if len(gasoductos_gasbc_cortos_empresa_entrada_todos) else '<p>No existen candidatos bajo el largo máximo configurado.</p>'}</details>
      <details><summary>Gasoductos cortos de GasBC: BG en empresa → ubicación/cliente directamente</summary>{_table(gasoductos_gasbc_cortos_empresa_destino_todos,['id','empresa_id','empresa','nodo_origen_id','destino_tipo','destino_id','destino_nombre','nodo_destino_id','distancia_recta_km','costo_fijo_usd_dia','costo_variable_usd_nm3_km','capacidad_max_nm3_dia','empresa_disponible_en_milp']) if len(gasoductos_gasbc_cortos_empresa_destino_todos) else '<p>No existen candidatos directos bajo el largo máximo configurado.</p>'}</details>
      <details><summary>Empresas</summary>{_table(empresas,['id','nombre','lat','lon','capacidad_max_biogas_nm3',COL_PRECIO_BIOGAS_CRUDO,'costo_activacion_biogas_usd'])}</details>
      <details><summary>Clientes</summary>{_table(clientes,['id','empresa_instalacion','latitud','longitud','demanda_max_nm3','precio_venta_usd_nm3'])}</details>
      <details><summary>Productoras candidatas</summary>{_table(sitios,['id','nombre','lat','lon'])}</details>
    </section>
    """

def make_stats_html():
    total_biogas=_finite(df_metricas_empresas['supply'].sum())
    total_biogas_cost=_finite(df_metricas_empresas['total_cost'].sum())
    provider_benefit=_finite(df_metricas_empresas['benefit'].sum())
    company_id_set={str(i) for i in I}
    supplier_company_ids={
        str(row.get('id'))
        for _,row in df_metricas_empresas.iterrows()
        if _finite(row.get('supply',0.0))>EPS
    }
    active_company_ids=set(supplier_company_ids)

    def _register_company_site(site_id):
        sid=str(site_id or '').strip()
        if not sid or sid in {'None','nan'}:
            return
        if sid.startswith('NE_') and sid[3:] in company_id_set:
            active_company_ids.add(sid[3:])
        try:
            hits=sitios.loc[sitios['id'].astype(str).eq(sid)]
            for _,site_row in hits.iterrows():
                ref=str(site_row.get('referencia_local','')).strip()
                clase=(str(site_row.get('clase_nodo',''))+' '+str(site_row.get('clase_visual',''))).upper()
                if ref in company_id_set and ('EMPRESA' in clase or sid.startswith('NE_')):
                    active_company_ids.add(ref)
        except Exception:
            pass
    active_plant_units={t:int(round(sum(value('NP',t,s) for s in S))) for t in T}
    active_plants=sum(active_plant_units.values())
    active_conventional=[g for g in G if _finite(pipe_flow[g])>TOL_FLUJO_CERO_NM3]
    active_short_biogas=[h for h in H_CORTO if _finite(short_biogas_pipe_flow[h])>TOL_FLUJO_CERO_NM3]
    active_short_biogas_c2c=[h for h in H_BIOC2C if _finite(short_biogas_c2c_pipe_flow[h])>TOL_FLUJO_CERO_NM3]
    active_short_gas_in=[h for h in H_GECORTO if _finite(short_gasbc_inlet_pipe_flow[h])>TOL_FLUJO_CERO_NM3]
    active_short_gas_direct=[h for h in H_GDCORTO if _finite(short_gasbc_direct_pipe_flow[h])>TOL_FLUJO_CERO_NM3]
    active_short_gas_out=[h for h in H_GSCORTO if _finite(short_gasbc_pipe_flow[h])>TOL_FLUJO_CERO_NM3]
    active_short_count=len(active_short_biogas)+len(active_short_biogas_c2c)+len(active_short_gas_in)+len(active_short_gas_direct)+len(active_short_gas_out)

    if len(df_plantas_resultado):
        for _,plant_row in df_plantas_resultado.iterrows():
            active_here=max(
                _finite(plant_row.get('unidades_activas',0.0)),
                _finite(plant_row.get('entrada_procesada_nm3',0.0)),
                _finite(plant_row.get('salida_producida_nm3',0.0)),
            )>EPS
            if active_here:
                _register_company_site(plant_row.get('sitio_id'))

    for h in active_short_biogas:
        try:
            emp=GCORTO.loc[h].get('empresa_id')
            if emp is not None and not pd.isna(emp) and str(emp) in company_id_set:
                active_company_ids.add(str(emp))
            _register_company_site(GCORTO.loc[h].get('nodo_destino_id'))
        except Exception:
            pass
    for h in active_short_gas_in:
        try:
            emp=GECORTO.loc[h].get('empresa_id')
            if emp is not None and not pd.isna(emp) and str(emp) in company_id_set:
                active_company_ids.add(str(emp))
            _register_company_site(GECORTO.loc[h].get('nodo_origen_id'))
        except Exception:
            pass
    for h in active_short_gas_direct:
        try:
            emp=GDCORTO.loc[h].get('empresa_id')
            if emp is not None and not pd.isna(emp) and str(emp) in company_id_set:
                active_company_ids.add(str(emp))
            _register_company_site(GDCORTO.loc[h].get('nodo_origen_id'))
            _register_company_site(GDCORTO.loc[h].get('nodo_destino_id'))
        except Exception:
            pass
    for h in active_short_gas_out:
        try:
            _register_company_site(GSCORTO.loc[h].get('nodo_destino_id'))
        except Exception:
            pass
    for tour in transport_tours_payload:
        if _finite(tour.get('origin_load',0.0))>TOL_FLUJO_CERO_NM3:
            _register_company_site(tour.get('origin_code'))
        for seg in tour.get('segments',[]) or []:
            if _finite(seg.get('load_nm3',0.0))>TOL_FLUJO_CERO_NM3:
                _register_company_site(seg.get('from'))
                _register_company_site(seg.get('to'))

    active_supplier_companies=len(supplier_company_ids)
    active_companies=len(active_company_ids)
    active_company_names='; '.join(
        f"{cid} · {E.loc[cid,'nombre']}" if cid in E.index else cid
        for cid in sorted(active_company_ids,key=lambda x:(len(str(x)),str(x)))
    )
    supplied_clients=int((df_metricas_clientes['served']>EPS).sum())
    conventional_pipe_cards=''.join(
        f"<div class='summary-card'><span>Gasoducto convencional utilizado<br><small>{html.escape(str(GA.loc[g,'nombre']))}</small></span><b>{pipe_flow[g]:,.2f} Nm³</b></div>"
        for g in active_conventional
    ) or "<p>No se utilizó ningún gasoducto convencional.</p>"
    short_pipe_cards=''.join(
        f"<div class='summary-card'><span>Gasoducto corto · Biogás crudo<br><small>{html.escape(str(GCORTO.loc[h,'empresa_id']))} → {html.escape(str(GCORTO.loc[h,'destino_tipo']))} {html.escape(str(GCORTO.loc[h,'destino_id']))}</small></span><b>{short_biogas_pipe_flow[h]:,.2f} Nm³</b></div>"
        for h in active_short_biogas
    )
    short_pipe_cards+=''.join(
        f"<div class='summary-card'><span>Gasoducto corto · Biogás crudo HUB<br><small>cliente {html.escape(str(BIOC2C.loc[h,'cliente_origen_id']))} → cliente {html.escape(str(BIOC2C.loc[h,'cliente_destino_id']))}</small></span><b>{short_biogas_c2c_pipe_flow[h]:,.2f} Nm³</b></div>"
        for h in active_short_biogas_c2c
    )
    short_pipe_cards+=''.join(
        f"<div class='summary-card'><span>Gasoducto corto · GasBC a entrada<br><small>BG@{html.escape(str(GECORTO.loc[h,'empresa_id']))} → {html.escape(str(GECORTO.loc[h,'direccion_id']))}</small></span><b>{short_gasbc_inlet_pipe_flow[h]:,.2f} Nm³</b></div>"
        for h in active_short_gas_in
    )
    short_pipe_cards+=''.join(
        f"<div class='summary-card'><span>Gasoducto corto · GasBC directo<br><small>{html.escape(_gdcorto_origen_label(h))} → {html.escape(str(GDCORTO.loc[h,'destino_tipo']))} {html.escape(str(GDCORTO.loc[h,'destino_id']))}</small></span><b>{short_gasbc_direct_pipe_flow[h]:,.2f} Nm³</b></div>"
        for h in active_short_gas_direct
    )
    short_pipe_cards+=''.join(
        f"<div class='summary-card'><span>Gasoducto corto · GasBC desde salida convencional<br><small>{html.escape(str(GSCORTO.loc[h,'direccion_id']))} → {html.escape(_gscorto_destino_label(h))}</small></span><b>{short_gasbc_pipe_flow[h]:,.2f} Nm³</b></div>"
        for h in active_short_gas_out
    )
    short_pipe_cards=short_pipe_cards or "<p>No se utilizó ningún gasoducto corto.</p>"
    plant_type_cards=''.join(
        f"<div class='summary-card'><span>Número de plantas {html.escape(TYPE_NAME[t])}</span><b>{active_plant_units[t]}</b></div>"
        for t in T
    )
    clients_table=_table(df_clientes_resultado.sort_values('porcentaje_satisfecho',ascending=False))
    plants_table=_table(df_plantas_resultado) if len(df_plantas_resultado) else '<p>No se activaron plantas.</p>'
    companies_table=_table(df_empresas_resultado)
    conventional_pipes_used=df_gasoductos_resultado.loc[
        df_gasoductos_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
    ].copy()
    short_pipe_frames=[]
    if len(df_gasoductos_biogas_cortos_resultado):
        short_pipe_frames.append(df_gasoductos_biogas_cortos_resultado.loc[
            df_gasoductos_biogas_cortos_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
        ].copy())
    if len(df_gasoductos_biogas_cortos_cliente_cliente_resultado):
        short_pipe_frames.append(df_gasoductos_biogas_cortos_cliente_cliente_resultado.loc[
            df_gasoductos_biogas_cortos_cliente_cliente_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
        ].copy())
    if len(df_gasoductos_biogas_hub_empresa_solo_resultado):
        short_pipe_frames.append(df_gasoductos_biogas_hub_empresa_solo_resultado.loc[
            df_gasoductos_biogas_hub_empresa_solo_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
        ].copy())
    if len(df_gasoductos_biogas_cortos_hub_p_resultado):
        short_pipe_frames.append(df_gasoductos_biogas_cortos_hub_p_resultado.loc[
            df_gasoductos_biogas_cortos_hub_p_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
        ])

    if len(df_gasoductos_gasbc_cortos_empresa_entrada_resultado):
        short_pipe_frames.append(df_gasoductos_gasbc_cortos_empresa_entrada_resultado.loc[
            df_gasoductos_gasbc_cortos_empresa_entrada_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
        ].copy())
    if len(df_gasoductos_gasbc_cortos_empresa_destino_resultado):
        short_pipe_frames.append(df_gasoductos_gasbc_cortos_empresa_destino_resultado.loc[
            df_gasoductos_gasbc_cortos_empresa_destino_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
        ])
    if len(df_gasoductos_gasbc_cortos_salida_cliente_resultado):
        short_pipe_frames.append(df_gasoductos_gasbc_cortos_salida_cliente_resultado.loc[
            df_gasoductos_gasbc_cortos_salida_cliente_resultado['flujo_total_nm3'].astype(float)>TOL_FLUJO_CERO_NM3
        ].copy())
    short_pipes_used=pd.concat(short_pipe_frames,ignore_index=True,sort=False) if short_pipe_frames else pd.DataFrame()
    conventional_pipes_table=_table(conventional_pipes_used) if len(conventional_pipes_used) else '<p>No se utilizaron gasoductos convencionales.</p>'
    short_pipes_table=_table(short_pipes_used) if len(short_pipes_used) else '<p>No se utilizaron gasoductos cortos.</p>'
    route_table=_table(df_rutas_estadisticas.drop(columns=['_producer_keys','_producer_weights','_producer_cost_parts','_producer_variable_parts','_producer_fixed_parts'],errors='ignore')) if len(df_rutas_estadisticas) else '<p>No existen rutas activas.</p>'
    return f"""
    <h2>Estadísticas de la solución</h2>
    <section class='summary-report-section'>
      <h2>Corporación</h2>
      <section class='summary-grid'>
        <div class='summary-card'><span>Beneficio de la Corporación</span><b>US$ {beneficio:,.2f}</b></div>
        <div class='summary-card'><span>Ingresos Totales de la Corporación</span><b>US$ {revenue:,.2f}</b></div>
        <div class='summary-card'><span>Costos Totales de la Corporación</span><b>US$ {total_cost:,.2f}</b></div>
        <div class='summary-card'><span>Cantidad de Biogás Comprado</span><b>{total_biogas:,.2f} Nm³</b></div>
        <div class='summary-card'><span>Costo de la Cantidad de Biogás Comprado</span><b>US$ {total_biogas_cost:,.2f}</b></div>
      </section>
    </section>
    <div class='summary-major-gap'></div>
    <section class='summary-report-section'><h2>Empresas Proveedoras</h2><section class='summary-grid'>
      <div class='summary-card'><span>Beneficios Totales Atribuidos</span><b>US$ {provider_benefit:,.2f}</b></div>
      <div class='summary-card'><span>Costo Variable del Biogás</span><b>US$ {company_var:,.2f}</b></div>
      <div class='summary-card'><span>Costo Total del Biogás</span><b>US$ {total_biogas_cost:,.2f}</b></div>
    </section></section>
    <div class='summary-major-gap'></div>
    <section class='summary-report-section'><h2>Gasoductos convencionales utilizados</h2><section class='summary-grid'>{conventional_pipe_cards}</section></section>
    <div class='summary-major-gap'></div>
    <section class='summary-report-section'><h2>Gasoductos cortos utilizados</h2><section class='summary-grid'>{short_pipe_cards}</section></section>
    <div class='summary-major-gap'></div>
    <section class='summary-report-section'><h2>Clientes</h2><section class='summary-grid'>
      <div class='summary-card'><span>Beneficio Total de Clientes</span><b>US$ {beneficio:,.2f}</b></div>
      <div class='summary-card'><span>Ingresos Totales de los Clientes</span><b>US$ {revenue:,.2f}</b></div>
      <div class='summary-card'><span>Costos Totales de los Clientes</span><b>US$ {total_cost:,.2f}</b></div>
    </section></section>
    <div class='summary-major-gap'></div>
    <section class='summary-report-section'><h2>Indicadores Operacionales</h2><section class='summary-grid'>
      <div class='summary-card'><span>Número de Empresas Activas<br><small>{html.escape(active_company_names) if active_company_names else 'Sin empresas activas'}</small></span><b>{active_companies} de {len(I)}</b></div>
      <div class='summary-card'><span>Empresas proveedoras de biogás</span><b>{active_supplier_companies} de {len(I)}</b></div>
      <div class='summary-card'><span>Número de Productoras Abiertas</span><b>{active_plants}</b></div>
      <div class='summary-card'><span>Configuración de plantas</span><b>{'Libre' if NUMERO_TOTAL_PLANTAS is False else f'Exactamente {NUMERO_TOTAL_PLANTAS}'}</b></div>
      {plant_type_cards}
      <div class='summary-card'><span>Gasoductos convencionales utilizados</span><b>{len(active_conventional)}</b></div>
      <div class='summary-card'><span>Gasoductos cortos utilizados</span><b>{active_short_count}</b></div>
      <div class='summary-card'><span>Clientes con demanda completa</span><b>{kpis['clientes_100pct']} de {len(J)}</b></div>
      <div class='summary-card'><span>Clientes con suministro</span><b>{supplied_clients} de {len(J)}</b></div>
      <div class='summary-card'><span>Demanda satisfecha</span><b>{kpis['porcentaje_demanda_satisfecha']:.2f}%</b></div>
      <div class='summary-card'><span>Cantidad entregada</span><b>{total_served:,.2f} Nm³</b></div>
      <div class='summary-card'><span>Rutas óptimas descompuestas</span><b>{len(df_rutas_estadisticas)}</b></div>
    </section></section>
    <div class='section-gap'></div>
    <h2>Gráficos de resultados</h2>
    {''.join(stats_chart_sections)}
    <div class='section-gap'></div>
    <h2>Tablas de resultados</h2>
    <details open><summary>Clientes</summary>{clients_table}</details>
    <details><summary>Vehículos</summary>{_table(vehiculos,['codigo_pdf','nombre','capacidad_max_m3_viaje','unidad_capacidad','costo_km_usd'])}</details>
    <details><summary>Empresas</summary>{companies_table}</details>
    <details><summary>Plantas activas</summary>{plants_table}</details>
    <details open><summary>Gasoductos convencionales utilizados</summary>{conventional_pipes_table}</details>
    <details open><summary>Gasoductos cortos utilizados</summary>{short_pipes_table}</details>
    <details><summary>Economía atribuida por ruta</summary>{route_table}</details>
    <details><summary>Desglose de costos</summary><table class='data-table'>
      <tr><th>Concepto</th><th>US$</th></tr>
      <tr><td>Variable empresas</td><td>{company_var:,.2f}</td></tr><tr><td>Variable procesamiento</td><td>{plant_var:,.2f}</td></tr>
      <tr><td>Transporte vial</td><td>{transport:,.2f}</td></tr><tr><td>Fijo empresas</td><td>{company_fixed:,.2f}</td></tr>
      <tr><td>Fijo plantas</td><td>{plant_fixed:,.2f}</td></tr><tr><td>Fijo gasoductos</td><td>{pipe_fixed:,.2f}</td></tr><tr><td>Variable gasoductos (km × Nm³)</td><td>{pipe_variable:,.2f}</td></tr>
    </table></details>
    """


# -----------------------------------------------------------------------------
# Payload del visor histórico: rutas agrupadas por cliente y geometría local.
# -----------------------------------------------------------------------------
FAMILY_COLORS = {
    "GasBC": "#16A34A",
    "BioGNL": "#7C3AED",
    "BioGNC": "#F97316",
    "Gasoducto": "#38BDF8",
}
SEGMENT_COLORS = {
    "Biogas": "#1D4ED8",
    "GasBC": "#16A34A",
    "BiometanoGaseoso": "#059669",
    "BioGNL": "#7C3AED",
    "BioGNC": "#F97316",
    "Gasoducto": "#38BDF8",
    "GasoductoBiogas": "#1D4ED8",
}
FAMILY_SHORT = {
    "GasBC": "GasBC",
    "BioGNL": "BioGNL",
    "BioGNC": "BioGNC",
    "Gasoducto": "Gasoducto",
}
# Texto compacto que debe aparecer exactamente una vez por ruta en el HTML.
# Se conserva la notación tecnológica de la formulación MILP y no se usan
# descripciones operacionales genéricas en reemplazo del nombre de la planta.
FAMILY_PROCESS = {
    "GasBC": "Empresa → BG → reparto multi-cliente → Cliente(s).",
    "BioGNL": "Empresa → BG → GasBC a planta L → BioGNL → GNL a GasBC (RL) → reparto multi-cliente → Cliente(s).",
    "BioGNC": "Empresa → BG → GasBC a planta C → BioGNC → GNC a GasBC (RC) → reparto multi-cliente → Cliente(s).",
    "Gasoducto": (
        "Empresa → BG → extremo de ingreso del gasoducto → "
        "extremo de salida → reparto multi-cliente → Cliente(s)."
    ),
}

EXPECTED_FAMILY_PROCESS = {
    "GasBC": "Empresa → BG → reparto multi-cliente → Cliente(s).",
    "BioGNL": "Empresa → BG → GasBC a planta L → BioGNL → GNL a GasBC (RL) → reparto multi-cliente → Cliente(s).",
    "BioGNC": "Empresa → BG → GasBC a planta C → BioGNC → GNC a GasBC (RC) → reparto multi-cliente → Cliente(s).",
    "Gasoducto": (
        "Empresa → BG → extremo de ingreso del gasoducto → "
        "extremo de salida → reparto multi-cliente → Cliente(s)."
    ),
}

SHORT_BIOGAS_PIPE_PROCESS_CLIENT = (
    "Empresa → gasoducto corto de biogás crudo → BG en ubicación del cliente "
    "→ entrega local de GasBC al cliente."
)
SHORT_BIOGAS_PIPE_PROCESS_SITE = (
    "Empresa → gasoducto corto de biogás crudo → BG en ubicación posible"
)
SHORT_BIOGAS_PIPE_PROCESS_ENTRY = (
    "Empresa(s) → gasoducto corto de biogás crudo → BG en la entrada del corredor "
    "→ gasoducto principal → salida → cliente."
)
SHORT_GASBC_INLET_PROCESS = (
    "Empresa → BG en la misma empresa → GasBC → gasoducto corto a la entrada "
    "→ gasoducto principal → salida → cliente."
)
SHORT_BIOGAS_TO_ENTRY_AS_LOCATION_PROCESS = (
    "Empresa(s) → gasoducto corto de biogás crudo → BG instalada en una ubicación "
    "de planta coincidente con la entrada de un corredor → GasBC → reparto "
    "multi-cliente → Cliente(s). El gasoducto convencional no se utiliza."
)
SHORT_GASBC_TO_ENTRY_AS_LOCATION_PROCESS = (
    "P/E/C/NGI → GasBC → gasoducto corto → otro nodo P/E/C/NGI → pool GasBC "
    "→ L/C o reparto/cadena posterior."
)
PIPE_NODE_AS_LOCATION_PROCESS = (
    "Empresa → BG instalada en una ubicación logística → GasBC → reparto multi-cliente → Cliente(s)."
)
DIRECT_GASBC_PIPE_PROCESS_CLIENT = (
    "Empresa → BG en la misma empresa → GasBC → gasoducto corto directo → Cliente."
)
DIRECT_GASBC_PIPE_PROCESS_SITE = (
    "Empresa → BG en la misma empresa → GasBC → gasoducto corto directo → ubicación posible"
)
PIPE_TO_GNL_PROCESS = (
    "Empresa → BG → GasBC → entrada de gasoducto convencional → salida → "
    "L (GasBC a BioGNL) → vehículo GNL → RL (BioGNL a GasBC) → cliente."
)
PIPE_TO_GNC_PROCESS = (
    "Empresa → BG → GasBC → entrada de gasoducto convencional → salida → "
    "C (GasBC a BioGNC) → vehículo GNC → RC (BioGNC a GasBC) → cliente."
)

INLET_GNL_TO_PIPE_PREFIX = (
    "Empresa → BG → L (GasBC a BioGNL) → vehículo GNL → RL en entrada del gasoducto "
    "→ GasBC → gasoducto convencional → salida"
)
INLET_GNC_TO_PIPE_PREFIX = (
    "Empresa → BG → C (GasBC a BioGNC) → vehículo GNC → RC en entrada del gasoducto "
    "→ GasBC → gasoducto convencional → salida"
)

def _site_downstream_process(fam, uses_pipe):
    if fam=='GasBC' or (fam=='Gasoducto' and not uses_pipe):
        return " → entrega de GasBC al cliente."
    if fam=='BioGNL':
        return " → L → BioGNL → RL → GasBC → cliente."
    if fam=='BioGNC':
        return " → C → BioGNC → RC → GasBC → cliente."
    if fam=='Gasoducto' and uses_pipe:
        return " → inyección al gasoducto convencional → salida → GasBC → cliente."
    return " → cliente."

def _uses_conventional_pipeline(segment_keys):
    return any(str(key).startswith(('PIPE|','PIPEINTRA|','PIPEPAIR|')) for key in (segment_keys or []))

def _route_process_text(rec, segment_keys=None):
    fam=str(rec.get('familia',''))
    # Cuando se conocen los segmentos exportados, ellos son la fuente de verdad.
    # El fallback por familia conserva compatibilidad con llamadas antiguas.
    uses_pipe=(
        _uses_conventional_pipeline(segment_keys)
        if segment_keys is not None
        else fam=='Gasoducto'
    )
    # Cadena con reconversión vehicular ANTES del corredor.
    entrada_modo=str(rec.get('entrada_corredor_modo',''))
    if uses_pipe and entrada_modo.startswith('RECONVERSION_GNL'):
        base=INLET_GNL_TO_PIPE_PREFIX
        if fam=='BioGNL':
            return base + " → L → BioGNL → vehículo GNL → RL → GasBC → cliente."
        if fam=='BioGNC':
            return base + " → C → BioGNC → vehículo GNC → RC → GasBC → cliente."
        return base + " → GasBC → cliente."
    if uses_pipe and entrada_modo.startswith('RECONVERSION_GNC'):
        base=INLET_GNC_TO_PIPE_PREFIX
        if fam=='BioGNL':
            return base + " → L → BioGNL → vehículo GNL → RL → GasBC → cliente."
        if fam=='BioGNC':
            return base + " → C → BioGNC → vehículo GNC → RC → GasBC → cliente."
        return base + " → GasBC → cliente."
    # Nueva cadena anterior: el corredor es intermedio y la salida alimenta L/C local.
    if uses_pipe and fam=='BioGNL':
        return PIPE_TO_GNL_PROCESS
    if uses_pipe and fam=='BioGNC':
        return PIPE_TO_GNC_PROCESS
    direct_h=rec.get('gasoducto_gasbc_directo_id')
    if direct_h is not None and str(direct_h) not in {'','None','nan'}:
        destino_tipo=str(rec.get('gasoducto_gasbc_directo_destino_tipo') or GDCORTO.loc[str(direct_h),'destino_tipo'])
        if destino_tipo=='CLIENTE':
            if fam=='BioGNL' and bool(rec.get('post_reconversion_gasoducto_corto',False)):
                return "Empresa → BG → L → BioGNL → vehículo GNL → RL → GasBC → gasoducto corto → cliente."
            if fam=='BioGNC' and bool(rec.get('post_reconversion_gasoducto_corto',False)):
                return "Empresa → BG → C → BioGNC → vehículo GNC → RC → GasBC → gasoducto corto → cliente."
        if destino_tipo=='ENTRADA_GASODUCTO':
            return "GasBC → gasoducto corto → extremo de gasoducto (NGI) → pool GasBC → L/C o cadena posterior."
            return DIRECT_GASBC_PIPE_PROCESS_CLIENT
        return DIRECT_GASBC_PIPE_PROCESS_SITE + _site_downstream_process(fam,uses_pipe)
    short_gas_in=rec.get('gasoducto_corto_entrada_gasbc_id')
    if short_gas_in is not None and str(short_gas_in) not in {'','None','nan'}:
        return SHORT_GASBC_INLET_PROCESS if uses_pipe else SHORT_GASBC_TO_ENTRY_AS_LOCATION_PROCESS
    short_h=rec.get('gasoducto_biogas_corto_id')
    if short_h is not None and str(short_h) not in {'','None','nan'}:
        destino_tipo=str(rec.get('gasoducto_destino_tipo') or GCORTO.loc[str(short_h),'destino_tipo'])
        if destino_tipo in {'UBICACION_POSIBLE','UBICACION_EMPRESA'}:
            return SHORT_BIOGAS_PIPE_PROCESS_SITE + _site_downstream_process(fam,uses_pipe)
        if destino_tipo=='ENTRADA_GASODUCTO':
            return SHORT_BIOGAS_PIPE_PROCESS_ENTRY if uses_pipe else SHORT_BIOGAS_TO_ENTRY_AS_LOCATION_PROCESS
        return SHORT_BIOGAS_PIPE_PROCESS_CLIENT
    if fam=='Gasoducto' and not uses_pipe:
        return PIPE_NODE_AS_LOCATION_PROCESS
    return FAMILY_PROCESS.get(fam,'')

def _route_family_label(rec, segment_keys, tour_ids=None):
    packs = _transport_pack_sequence(rec, segment_keys, tour_ids=tour_ids)
    return " → ".join(pack["label"] for pack in packs)

def _route_family_color(rec, segment_keys, tour_ids=None):
    packs = _transport_pack_sequence(rec, segment_keys, tour_ids=tour_ids)
    return packs[0]["color"] if packs else '#334155'

segmentos_por_ruta = (
    df_rutas.set_index("route_id")["segmentos"].to_dict() if len(df_rutas) else {}
)
giras_por_ruta = (
    df_rutas.set_index("route_id")["transport_tour_ids"].to_dict() if len(df_rutas) else {}
)


def _route_stage_labels(rec, segment_keys=None, tour_ids=None):
    texto=_route_family_label(rec,segment_keys or [],tour_ids=tour_ids)
    return [texto] if texto else []


def _route_chain(rec, segment_keys=None, tour_ids=None):
    return _route_family_label(rec,segment_keys or [],tour_ids=tour_ids)


def _tour_segment_priority(seg):
    leg=str(seg.get('leg_type','')).upper()
    load=max(0.0,_finite(seg.get('load_nm3',0.0)))
    try:
        idx=int(seg.get('segment_index',0))
    except Exception:
        idx=0
    reverse=bool(seg.get('reverse',False))
    score=0.0
    if load>TOL_FLUJO_CERO_NM3:
        score+=1000.0+min(load,999.0)
    if leg in {'SALIDA','CONTINUA','DIRECTO','IDA'}:
        score+=100.0
    if leg in {'RETORNO','RETURN'}:
        score-=100.0
    score+=10.0 if not reverse else -10.0
    return score-idx*1e-6


def _tour_mode_maps(tour_ids=None):
    ids={str(x) for x in (tour_ids or [])}
    mode_by_key={}
    reverse_by_key={}
    score_by_key={}
    if not ids:
        return mode_by_key,reverse_by_key
    for tour in transport_tours_payload:
        tid=str(tour.get('transport_route_id',''))
        if tid not in ids:
            continue
        for seg in tour.get('segments',[]) or []:
            key=str(seg.get('key',''))
            if not key:
                continue
            score=_tour_segment_priority(seg)
            if key not in score_by_key or score>score_by_key[key]:
                score_by_key[key]=score
                mode_by_key[key]=str(tour.get('mode',''))
                reverse_by_key[key]=bool(seg.get('reverse',False))
    return mode_by_key,reverse_by_key

def _visual_segment_distance_km(key):
    """Distancia física usada únicamente por el HTML para el tooltip del tramo."""
    key=str(key or '')
    try:
        if key.startswith('SHORTBIO|'):
            return _finite(GCORTO.loc[key.split('|',1)[1],'distancia_recta_km'])
        if key.startswith('SHORTBIOPHUB|'):
            return _finite(PHUB.loc[key.split('|',1)[1],'distancia_recta_km'])
        if key.startswith('SHORTBIOHUBEMP|'):
            return _finite(EHUB.loc[key.split('|',1)[1],'distancia_recta_km'])
        if key.startswith('SHORTBIOC2C|'):
            return _finite(BIOC2C.loc[key.split('|',1)[1],'distancia_recta_km'])
        if key.startswith('SHORTGASIN|'):
            return _finite(GECORTO.loc[key.split('|',1)[1],'distancia_recta_km'])
        if key.startswith('SHORTGASDIRECT|'):
            return _finite(GDCORTO.loc[key.split('|',1)[1],'distancia_recta_km'])
        if key.startswith('SHORTGAS|'):
            return _finite(GSCORTO.loc[key.split('|',1)[1],'distancia_recta_km'])
        if key.startswith('PIPEINTRA|'):
            _d=key.split('|',1)[1]
            return _finite(DIST_INTRA_GASODUCTO_KM[str(DG[_d])])
        if key.startswith('PIPEPAIR|'):
            _p=key.split('|')
            return _finite(DIST_GG[(str(_p[1]),str(_p[2]))])
        if key.startswith('PIPE|'):
            return 0.0
    except Exception:
        return 0.0
    return 0.0


def _visual_segment_quantity_nm3(rec,key):
    """Cantidad atribuible a ESTA ruta; no modifica ningún flujo del MILP."""
    key=str(key or '')
    try:
        if key in producer_meta:
            q=_route_input_for_producer(rec,key)
            if q>EPS:
                return max(0.0,_finite(q))
    except Exception:
        pass
    if key.startswith(('SHORTBIO|','SHORTBIOPHUB|','SHORTBIOHUBEMP|','SHORTBIOC2C|')):
        return max(0.0,_finite(rec.get('biogas_crudo_entrada_BG_nm3',0.0)))
    if key.startswith(('SHORTGASIN|','SHORTGASDIRECT|','SHORTGAS|','PIPE|','PIPEINTRA|','PIPEPAIR|')):
        return max(0.0,_finite(rec.get('gasbc_entrada_corredor_nm3',
                                      rec.get('gasbc_entrada_ruta_actual_nm3',
                                              rec.get('cantidad_nm3',0.0)))))
    return 0.0


def _segment_defs(rec, segment_keys, tour_ids=None):
    """Clasifica cada tramo por el producto REAL transportado en ese tramo."""
    fam = str(rec.get("familia", ""))
    mode_by_key,reverse_by_key=_tour_mode_maps(tour_ids)
    defs=[]
    for key in segment_keys or []:
        key=str(key)
        reverse=bool(reverse_by_key.get(key,False))
        actual_mode=mode_by_key.get(key,'')

        if key.startswith(("SHORTBIO|","SHORTBIOPHUB|","SHORTBIOHUBEMP|","SHORTBIOC2C|")):
            segment_type="GasoductoBiogas"
            segment_label="BioGás Crudo por gasoducto corto"
        elif key.startswith(("SHORTGASDIRECT|","SHORTGASIN|","SHORTGAS|")):
            segment_type="GasBC"
            segment_label="GasBC por gasoducto corto"
        elif key.startswith("PIPEINTRA|"):
            segment_type="Gasoducto"
            segment_label="GasBC por Gasoducto Convencional A/B"
            reverse=False
        elif key.startswith("PIPEPAIR|"):
            segment_type="Gasoducto"
            segment_label="GasBC por Gasoducto Convencional"
            reverse=False
        elif key.startswith("PIPE|"):
            segment_type="Gasoducto"
            segment_label="GasBC por Gasoducto Convencional"
            if key not in reverse_by_key:
                reverse=str(rec.get('direccion_id','')).endswith('_BA')
        else:
            # Para ASITE/AIG/AGC/ABC/C2C/ABIO manda la gira física, no la
            # familia final de la ruta. Así un C2C de una gira GNC sigue siendo
            # BioGNC hasta que efectivamente se descarga/reconvierte.
            mode=actual_mode
            if not mode and key.startswith('ABIO|'):
                mode='BIO'
            if mode=='GNL':
                segment_type="BioGNL"; segment_label="BioGNL en vehículo"
            elif mode=='GNC':
                segment_type="BioGNC"; segment_label="BioGNC en vehículo"
            elif mode=='BIO':
                segment_type="Biogas"; segment_label="BioGás Crudo en vehículo"
            elif mode in {'BC','BC_PROCESO'}:
                segment_type="BiometanoGaseoso"; segment_label="Biometano Gaseoso en vehículo"
            else:
                # Fallback sólo para exportaciones antiguas sin metadata de gira.
                if fam=='BioGNL' and key.startswith('ASITE|'):
                    segment_type="BioGNL"; segment_label="BioGNL en vehículo"
                elif fam=='BioGNC' and key.startswith('ASITE|'):
                    segment_type="BioGNC"; segment_label="BioGNC en vehículo"
                else:
                    segment_type="BiometanoGaseoso"; segment_label="Biometano Gaseoso en vehículo"

        defs.append({
            "key":key,
            "segment_type":segment_type,
            "segment_label":segment_label,
            "color":SEGMENT_COLORS.get(segment_type,"#334155"),
            "reverse":reverse,
            "distance_km":_visual_segment_distance_km(key),
            "quantity_nm3":_visual_segment_quantity_nm3(rec,key),
        })
    return defs


TRANSPORT_PACK_COLORS = {
    "BioGNL": "#7C3AED",                 # violeta
    "BioGNC": "#F97316",                 # naranjo
    "GasBC": "#16A34A",                  # verde: gasoducto corto GasBC
    "Biometano Gaseoso": "#059669",      # verde azulado: vehículo gaseoso
    "Gasoducto Convencional": "#38BDF8", # celeste
    "BioGás Crudo": "#1D4ED8",           # azul fuerte
}

# Blindaje visual: no confundir GNL con GNC.
# BioGNL/GNL = violeta; BioGNC/GNC = naranjo, en familia, segmento y pack.


def _transport_pack_sequence(rec, segment_keys, tour_ids=None):
    """Packs de TRANSPORTE físicos, exactamente en el orden recorrido.

    No incluye plantas ni etapas de procesamiento. Las repeticiones se conservan.
    """
    packs=[]
    for seg in _segment_defs(rec,segment_keys or [],tour_ids=tour_ids):
        key=str(seg.get('key',''))
        seg_type=str(seg.get('segment_type',''))
        if key.startswith(('PIPE|','PIPEPAIR|')):
            label='Gasoducto Convencional'
        elif key.startswith(("SHORTBIO|","SHORTBIOPHUB|","SHORTBIOHUBEMP|","SHORTBIOC2C|")) or seg_type in {'Biogas','GasoductoBiogas'}:
            label='BioGás Crudo'
        elif key.startswith(("SHORTGASDIRECT|","SHORTGASIN|","SHORTGAS|")):
            label='GasBC'
        elif seg_type=='BioGNL':
            label='BioGNL'
        elif seg_type=='BioGNC':
            label='BioGNC'
        elif seg_type=='BiometanoGaseoso':
            label='Biometano Gaseoso'
        else:
            continue
        packs.append({'label':label,'color':TRANSPORT_PACK_COLORS[label],'key':key})
    return packs


def _direction_pipe_parts(direction_id):
    d=str(direction_id or '').strip()
    if d.endswith('_AB'):
        return d[:-3],'AB'
    if d.endswith('_BA'):
        return d[:-3],'BA'
    return None,None


def _pipe_marker_for_direction(direction_id, role):
    """Icono A/B del gasoducto tocado físicamente por una entrada o salida."""
    g,suffix=_direction_pipe_parts(direction_id)
    if not g or suffix not in {'AB','BA'}:
        return None
    role=str(role or '').lower()
    if role=='entrada':
        end='A' if suffix=='AB' else 'B'
    else:
        end='B' if suffix=='AB' else 'A'
    return f'pipe{end}|{g}'


def _pipe_marker_nearest_reference(g,lat,lon):
    """Devuelve pipeA|g o pipeB|g para la referencia física más cercana."""
    try:
        row=gasoductos.set_index('id').loc[str(g)]
        lat=float(lat); lon=float(lon)
        da=(float(row['punto_a_lat'])-lat)**2+(float(row['punto_a_lon'])-lon)**2
        db=(float(row['punto_b_lat'])-lat)**2+(float(row['punto_b_lon'])-lon)**2
        return f"pipe{'A' if da<=db else 'B'}|{g}"
    except Exception:
        return None


def _pipepair_reference_markers(g1,g2):
    """Iconos de los extremos/referencias realmente usados por un PIPEPAIR."""
    g1=str(g1); g2=str(g2)
    try:
        df=distancias_entre_gasoductos_36
        hit=df[(df['gasoducto_1_id'].astype(str)==g1)&(df['gasoducto_2_id'].astype(str)==g2)]
        reverse=False
        if not len(hit):
            hit=df[(df['gasoducto_1_id'].astype(str)==g2)&(df['gasoducto_2_id'].astype(str)==g1)]
            reverse=True
        if not len(hit):
            return []
        row=hit.iloc[0]
        if not reverse:
            refs=[
                _pipe_marker_nearest_reference(g1,row['referencia_1_lat'],row['referencia_1_lon']),
                _pipe_marker_nearest_reference(g2,row['referencia_2_lat'],row['referencia_2_lon']),
            ]
        else:
            refs=[
                _pipe_marker_nearest_reference(g1,row['referencia_2_lat'],row['referencia_2_lon']),
                _pipe_marker_nearest_reference(g2,row['referencia_1_lat'],row['referencia_1_lon']),
            ]
        return [x for x in refs if x]
    except Exception:
        return []


def _route_conventional_events(rec):
    events=[rec]
    for ev in _visual_list(rec.get('hub_chain_events',[])):
        if isinstance(ev,dict):
            events.append(ev)
    return events


def _route_actual_pipepairs(rec):
    """Arcos YGG físicos de la ruta, incluyendo los heredados por HUB."""
    out=[]; seen=set()
    for ev in _route_conventional_events(rec):
        for arc in _visual_list(ev.get('red_gasoductos_arcos',[])):
            if not isinstance(arc,(list,tuple)) or len(arc)<2:
                continue
            pair=(str(arc[0]),str(arc[1]))
            if pair[0] and pair[1] and pair not in seen:
                seen.add(pair); out.append(pair)
    return out


def _route_pipe_node_keys(rec, segment_keys=None):
    """Sólo los iconos de gasoducto convencional realmente tocados por la ruta."""
    keys=set()

    def add(value_):
        if value_ and str(value_) not in {'','None','nan'}:
            keys.add(str(value_))

    # Entradas/salidas explícitas, tanto de la ruta base como de cadenas HUB.
    for ev in _route_conventional_events(rec):
        add(_pipe_marker_for_direction(ev.get('direccion_inyeccion_id'),'entrada'))
        add(_pipe_marker_for_direction(ev.get('direccion_id'),'salida'))

    # Tramos del visualizador: PIPEPAIR usa exactamente sus referencias de conexión.
    for segment_key in (segment_keys or []):
        sk=str(segment_key)
        if sk.startswith('PIPEPAIR|'):
            p=sk.split('|')
            if len(p)>=3:
                for mk in _pipepair_reference_markers(p[1],p[2]):
                    add(mk)
        elif sk.startswith('PIPE|'):
            # Fallback defensivo para exportaciones antiguas: si sólo queda PIPE,
            # conservamos ambos iconos del activo, pero no inventamos movimiento.
            g=sk.split('|',1)[1]
            add(f'pipeA|{g}'); add(f'pipeB|{g}')
        elif sk.startswith('SHORTGASIN|'):
            try:
                d=str(GECORTO.loc[sk.split('|',1)[1],'direccion_id'])
                add(_pipe_marker_for_direction(d,'entrada'))
            except Exception:
                pass
        elif sk.startswith('SHORTGAS|'):
            try:
                d=str(GSCORTO.loc[sk.split('|',1)[1],'direccion_id'])
                add(_pipe_marker_for_direction(d,'salida'))
            except Exception:
                pass
        elif sk.startswith('SHORTBIO|'):
            try:
                row=GCORTO.loc[sk.split('|',1)[1]]
                d=row.get('direccion_id')
                if d is not None and str(d) not in {'','None','nan'}:
                    add(_pipe_marker_for_direction(d,'entrada'))
            except Exception:
                pass
        elif sk.startswith('AIG|'):
            try:
                d=sk.rsplit('|',1)[1]
                add(_pipe_marker_for_direction(d,'entrada'))
            except Exception:
                pass
    return sorted(keys)


def _route_node_keys(rec, segment_keys=None):
    keys = {f"company|{rec.get('empresa_id')}", f"client|{rec.get('cliente_id')}"}
    for delivery in (rec.get('clientes_compartidos', []) or []):
        cid = delivery.get('cliente_id') if isinstance(delivery, dict) else None
        if cid is not None and str(cid) not in {'','None','nan'}:
            keys.add(f"client|{cid}")

    def add_site(value_):
        if value_ is None or str(value_) in {'','None','nan'}:
            return
        keys.add(f"site|{value_}")

    bg_site = rec.get("bg_site")
    if bg_site:
        add_site(bg_site)
    bg_production_site = rec.get("bg_production_site")
    if bg_production_site:
        add_site(bg_production_site)

    # Toda planta que participa efectivamente en la ruta debe tener su icono.
    try:
        for producer_key in _route_producer_keys(rec):
            if str(producer_key).startswith('PLANT|'):
                add_site(str(producer_key).split('|',2)[2])
    except Exception:
        pass

    fam = str(rec.get("familia", ""))
    for key in ['pre_l_site','pre_rl_site','pre_c_site','pre_rc_site']:
        add_site(rec.get(key))
    if fam == "BioGNL":
        for key in ["l_site", "rl_site"]:
            add_site(rec.get(key))
    elif fam == "BioGNC":
        for key in ["c_site", "rc_site"]:
            add_site(rec.get(key))

    # Extremos físicos de gasoductos cortos/relevos usados.
    for segment_key in (segment_keys or []):
        sk=str(segment_key)
        try:
            if sk.startswith('SHORTBIO|'):
                h=sk.split('|',1)[1]
                row=GCORTO.loc[h]
                if 'empresa_id' in row.index:
                    keys.add(f"company|{row['empresa_id']}")
                if 'nodo_origen_id' in row.index:
                    add_site(row['nodo_origen_id'])
                if 'nodo_destino_id' in row.index:
                    add_site(row['nodo_destino_id'])
                if str(row.get('destino_tipo',''))=='CLIENTE' and row.get('destino_id') is not None:
                    keys.add(f"client|{row['destino_id']}")
            elif sk.startswith('SHORTBIOPHUB|'):
                h=sk.split('|',1)[1]; row=PHUB.loc[h]
                add_site(row.get('nodo_origen_id',row.get('origen_id')))
                add_site(row.get('nodo_destino_id',row.get('destino_id')))
            elif sk.startswith('SHORTBIOHUBEMP|'):
                h=sk.split('|',1)[1]; row=EHUB.loc[h]
                if row.get('empresa_id') is not None:
                    keys.add(f"company|{row['empresa_id']}")
                add_site(row.get('nodo_origen_id'))
                add_site(row.get('nodo_destino_id',row.get('destino_id')))
            elif sk.startswith('SHORTBIOC2C|'):
                h=sk.split('|',1)[1]; row=BIOC2C.loc[h]
                if row.get('cliente_origen_id') is not None:
                    keys.add(f"client|{row['cliente_origen_id']}")
                if row.get('cliente_destino_id') is not None:
                    keys.add(f"client|{row['cliente_destino_id']}")
                add_site(row.get('nodo_origen_id'))
                add_site(row.get('nodo_destino_id'))
            elif sk.startswith('SHORTGASDIRECT|'):
                h=sk.split('|',1)[1]; row=GDCORTO.loc[h]
                if row.get('empresa_id') is not None and str(row.get('empresa_id')) not in {'','None','nan'}:
                    keys.add(f"company|{row['empresa_id']}")
                add_site(row.get('nodo_origen_id'))
                add_site(row.get('nodo_destino_id',row.get('destino_id')))
                if str(row.get('destino_tipo',''))=='CLIENTE' and row.get('destino_id') is not None:
                    keys.add(f"client|{row['destino_id']}")
            elif sk.startswith('SHORTGASIN|'):
                h=sk.split('|',1)[1]; row=GECORTO.loc[h]
                if row.get('empresa_id') is not None:
                    keys.add(f"company|{row['empresa_id']}")
                add_site(row.get('nodo_origen_id'))
            elif sk.startswith('SHORTGAS|'):
                h=sk.split('|',1)[1]; row=GSCORTO.loc[h]
                if str(row.get('destino_tipo',''))=='CLIENTE' and not pd.isna(row.get('cliente_id')):
                    keys.add(f"client|{row['cliente_id']}")
                if str(row.get('destino_tipo',''))=='UBICACION_EMPRESA' and not pd.isna(row.get('empresa_id')):
                    keys.add(f"company|{row['empresa_id']}")
                add_site(row.get('nodo_destino_id'))
        except Exception:
            pass

    # RESTAURADO: iconos de los extremos/referencias de gasoducto convencional
    # que la ruta realmente toca. No se muestran gasoductos ajenos a la ruta.
    keys.update(_route_pipe_node_keys(rec,segment_keys))

    shared_client_ids={str(rec.get('cliente_id'))}
    for delivery in (rec.get('clientes_compartidos', []) or []):
        cid = delivery.get('cliente_id') if isinstance(delivery, dict) else None
        if cid is not None and str(cid) not in {'','None','nan'}:
            shared_client_ids.add(str(cid))
    allowed_client_keys={f"client|{cid}" for cid in shared_client_ids}
    keys={
        k for k in keys
        if not str(k).startswith('client|') or str(k) in allowed_client_keys
    }
    return sorted(k for k in keys if k and not str(k).endswith('None'))


def _visual_list(value_):
    if isinstance(value_, np.ndarray):
        return value_.tolist()
    if isinstance(value_, (list, tuple, set)):
        return list(value_)
    return []


def _visual_is_local_nontransport_key(key):
    """Transferencias nodales de 0 km: interacción, no un pack de transporte."""
    key=str(key or '')
    if not key.startswith(('AIG|','ASITE|','ABC|','AGC|','C2C|','ABIO|')):
        return False
    try:
        return not bool(road_arc_requires_truck(key))
    except Exception:
        try:
            return _finite(DIST.get(key,0.0))<=1e-9
        except Exception:
            return False


def _route_visual_segment_keys(rec, base_keys):
    """Secuencia física usada SOLO por el HTML, sin alterar el MILP.

    Reglas visuales:
    - AIG/transferencias locales de 0 km son interacciones nodales, no vehículos.
    - PIPE|g es un activo económico/fijo; el movimiento físico entre gasoductos
      está en PIPEPAIR|g1|g2 (YGG). Por eso PIPE no se dibuja como viaje.
    - Los PIPEPAIR faltantes en rutas HUB se reconstruyen desde los arcos YGG
      registrados en hub_chain_events/red_gasoductos_arcos.
    """
    keys=[]
    def add(key):
        key=str(key or '').strip()
        if key and key not in keys:
            keys.append(key)

    for key in _visual_list(base_keys):
        add(key)

    short_h=rec.get('gasoducto_biogas_corto_id')
    if short_h is not None and str(short_h) not in {'','None','nan'}:
        add(f'SHORTBIO|{short_h}')

    for h in _visual_list(rec.get('gasoductos_biogas_relevo_empresa_ids', [])):
        h=str(h)
        add(f'SHORTBIO|{h}' if h in H_CORTO_SET else f'SHORTBIOHUBEMP|{h}')
    for h in _visual_list(rec.get('gasoductos_biogas_relevo_cliente_ids', [])):
        add(f'SHORTBIOC2C|{h}')
    for h in _visual_list(rec.get('gasoductos_biogas_relevo_P_ids', [])):
        add(f'SHORTBIOPHUB|{h}')

    for raw in _visual_list(rec.get('raw_relay_path', [])):
        raw=str(raw)
        if raw.startswith('PHUB|'):
            add('SHORTBIOPHUB|'+raw.split('|',1)[1])
        elif raw.startswith('C2C|'):
            add('SHORTBIOC2C|'+raw.split('|',1)[1])
        elif raw.startswith('EHUB|'):
            h=raw.split('|',1)[1]
            add(f'SHORTBIO|{h}' if h in H_CORTO_SET else f'SHORTBIOHUBEMP|{h}')

    direct_h=rec.get('gasoducto_gasbc_directo_id')
    if direct_h is not None and str(direct_h) not in {'','None','nan'}:
        add(f'SHORTGASDIRECT|{direct_h}')
    inlet_h=rec.get('gasoducto_corto_entrada_gasbc_id')
    if inlet_h is not None and str(inlet_h) not in {'','None','nan'}:
        add(f'SHORTGASIN|{inlet_h}')
    outlet_h=rec.get('gasoducto_corto_salida_cliente_id')
    if outlet_h is not None and str(outlet_h) not in {'','None','nan'}:
        add(f'SHORTGAS|{outlet_h}')

    for key in _visual_list(rec.get('hub_segment_keys_static', [])):
        add(key)

    actual_pairs=[f'PIPEPAIR|{a}|{b}' for a,b in _route_actual_pipepairs(rec)]
    existing_pairs=[k for k in keys if str(k).startswith('PIPEPAIR|')]
    ordered_pairs=_ordered_unique(existing_pairs+actual_pairs)

    # PIPE|g no representa una trayectoria g completa; es atribución económica.
    # Si había PIPE en la posición del corredor, se reemplaza allí por los YGG
    # reales para no mandar el PIPEPAIR al final de la ruta (caso CCU/C8).
    normalized=[]; inserted_pairs=False
    for key in keys:
        key=str(key)
        if key.startswith('PIPE|'):
            if ordered_pairs and not inserted_pairs:
                normalized.extend([p for p in ordered_pairs if p not in normalized])
                inserted_pairs=True
            continue
        if key.startswith('PIPEPAIR|'):
            if key not in normalized:
                normalized.append(key)
            inserted_pairs=True
            continue
        if _visual_is_local_nontransport_key(key):
            continue
        if key not in normalized:
            normalized.append(key)

    # Fallback para una exportación HUB donde no quedó ningún PIPE económico en
    # base_keys: insertar el YGG justo antes de la salida del corredor.
    if ordered_pairs and not inserted_pairs:
        out=[]; done=False
        for key in normalized:
            if not done and str(key).startswith(('SHORTGAS|','AGC|')):
                out.extend([p for p in ordered_pairs if p not in out]); done=True
            out.append(key)
        if not done:
            out.extend([p for p in ordered_pairs if p not in out])
        normalized=_ordered_unique(out)

    return normalized


# Campos aditivos que se prorratean sólo en la representación visual cuando una
# misma fila económica contiene dos o más giras físicas independientes.
_VISUAL_ADDITIVE_FIELDS={
    'biogas_crudo_entrada_BG_nm3','gasbc_salida_BG_nm3','cantidad_nm3','ingreso_usd',
    'costo_empresa_variable_usd','costo_productoras_variable_usd','costo_gasoducto_variable_usd',
    'costo_transporte_usd','costo_variable_total_usd','costo_total_usd','costos_fijos_asignados_usd',
    'margen_ruta_antes_costos_fijos_usd','margen_operativo_por_ruta_usd','beneficio_por_ruta_usd',
    'numero_viajes_ida_vuelta','numero_camiones_utilizados','distancia_vial_km',
    'distancia_interna_gasoducto_referencial_km','costo_empresa_fijo_usd',
    'costo_productoras_fijo_usd','costo_gasoducto_fijo_usd'
}

def _ordered_unique(values):
    out=[];seen=set()
    for value_ in values or []:
        value_=str(value_)
        if value_ and value_ not in seen:
            seen.add(value_);out.append(value_)
    return out

def _visual_route_variants(row):
    """Separa una ruta económica cuando fue abastecida por giras físicas distintas."""
    source_id=str(row.route_id)
    base_segments=list(segmentos_por_ruta.get(source_id,[]) or [])
    base_tours=_ordered_unique(giras_por_ruta.get(source_id,[]) or [])
    refs=[tuple(x) for x in route_transport_flow_refs(row)]

    split_ref=None;split_paths=[]
    # Elegir el flujo más aguas abajo que tenga varias giras; es el que define
    # cómo llegó físicamente el producto al cliente/RC/RL de esta ruta.
    for ref in reversed(refs):
        paths=[p for p in _paths_for_flow_ref(ref) if float(p.get('attributed_quantity',0.0))>TOL_FLUJO_CERO_NM3]
        by_tour={}
        for p in paths:
            by_tour[str(p.get('transport_route_id'))]=p
        if len(by_tour)>1:
            split_ref=ref;split_paths=list(by_tour.values());break

    if split_ref is None:
        return [{
            'route_id':source_id,'source_route_id':source_id,'fraction':1.0,
            'segments':base_segments,'tour_ids':base_tours,
        }]

    alt_keys=set()
    for p in split_paths:
        alt_keys.update(map(str,p.get('segment_keys',[]) or []))
    total=sum(max(0.0,float(p.get('attributed_quantity',0.0))) for p in split_paths)
    if total<=TOL_FLUJO_CERO_NM3:
        total=float(len(split_paths))

    first_alt_index=next((i for i,k in enumerate(base_segments) if str(k) in alt_keys),len(base_segments))
    static_before=[str(k) for k in base_segments[:first_alt_index] if str(k) not in alt_keys]
    static_after=[str(k) for k in base_segments[first_alt_index:] if str(k) not in alt_keys]
    split_tour_ids={str(p.get('transport_route_id')) for p in split_paths}
    retained_tours=[t for t in base_tours if str(t) not in split_tour_ids]

    variants=[]
    for p in split_paths:
        tid=str(p.get('transport_route_id'))
        q=max(0.0,float(p.get('attributed_quantity',0.0)))
        frac=(q/total) if total>0 else 1.0/len(split_paths)
        segments=_ordered_unique(static_before + list(p.get('segment_keys',[]) or []) + static_after)
        variants.append({
            'route_id':f'{source_id}_{tid}',
            'source_route_id':source_id,
            'fraction':frac,
            'segments':segments,
            'tour_ids':_ordered_unique(retained_tours+[tid]),
        })
    return variants



# -----------------------------------------------------------------------------
# Descargas intermedias por gasoductos de biogás crudo.
#
# Si una misma corriente de una empresa llega a un nodo P, una fracción se
# procesa/vende al cliente local de P y el resto continúa por otro gasoducto,
# la ruta hacia el cliente FINAL conserva un solo cliente económico. La descarga
# local se añade como metadata del punto intermedio, igual que una parada de una
# gira vehicular de dos destinos.
# -----------------------------------------------------------------------------
def _raw_relay_tokens_for_drop(rec):
    vals=_visual_list(rec.get('raw_relay_path',[]))
    out=[]
    for value_ in vals:
        value_=str(value_)
        if '|' not in value_:
            continue
        kind,hid=value_.split('|',1)
        out.append((str(kind),str(hid)))
    return out


def _raw_relay_destination_for_drop(token):
    kind,hid=token
    try:
        if kind=='EHUB':
            return str(EHUB.loc[hid,'nodo_destino_id'])
        if kind=='C2C':
            return str(BIOC2C.loc[hid,'nodo_destino_id'])
        if kind=='PHUB':
            return str(PHUB.loc[hid,'nodo_destino_id'])
    except Exception:
        return None
    return None


def _raw_source_signature_for_drop(rec):
    h=rec.get('gasoducto_biogas_corto_id')
    h=None if h is None or str(h) in {'','None','nan'} else str(h)
    return (
        str(rec.get('empresa_id')),
        str(rec.get('tipo_fuente_bg')),
        h,
        str(rec.get('raw_source_site')),
    )


def _client_node_for_drop(cliente_id):
    try:
        return str(CLIENTE_A_NODO[str(cliente_id)])
    except Exception:
        return None


_site_drop_index=sitios.set_index('id',drop=False) if len(sitios) else pd.DataFrame()


def _route_intermediate_drop_events(rec):
    """Descargas locales que comparten el prefijo físico de la ruta seleccionada."""
    route_id=str(rec.get('route_id',''))
    tokens=_raw_relay_tokens_for_drop(rec)
    start_node=str(rec.get('raw_source_site') or '')
    final_bg=str(rec.get('bg_production_site') or rec.get('bg_site') or '')

    # nodes[k] corresponde al nodo alcanzado después de tokens[:k].
    nodes=[start_node] if start_node else []
    for tok in tokens:
        dest=_raw_relay_destination_for_drop(tok)
        if dest:
            nodes.append(dest)

    source_sig=_raw_source_signature_for_drop(rec)
    events=[]
    seen_nodes=set()
    for k,node in enumerate(nodes):
        node=str(node)
        if not node or node==final_bg or node in seen_nodes:
            continue
        seen_nodes.add(node)
        prefix=tokens[:k]

        raw_drop=0.0
        gasbc_local=0.0
        ingreso_local=0.0
        local_clients=[]
        local_routes=[]
        local_client_amounts=defaultdict(float)

        for _,other in df_rutas_estadisticas.iterrows():
            o=other.to_dict()
            if str(o.get('route_id'))==route_id:
                continue
            if _raw_source_signature_for_drop(o)!=source_sig:
                continue
            if str(o.get('bg_production_site') or o.get('bg_site') or '')!=node:
                continue
            if _client_node_for_drop(o.get('cliente_id'))!=node:
                continue
            if _raw_relay_tokens_for_drop(o)!=prefix:
                continue

            qraw=max(0.0,_finite(o.get('biogas_crudo_entrada_BG_nm3',0.0)))
            qfinal=max(0.0,_finite(o.get('cantidad_nm3',0.0)))
            if qraw<=TOL_FLUJO_CERO_NM3 and qfinal<=TOL_FLUJO_CERO_NM3:
                continue
            raw_drop+=qraw
            gasbc_local+=qfinal
            ingreso_local+=max(0.0,_finite(o.get('ingreso_usd',0.0)))
            _local_client_id=str(o.get('cliente_id'))
            local_clients.append(_local_client_id)
            local_routes.append(str(o.get('route_id')))
            local_client_amounts[_local_client_id]+=qfinal

        if raw_drop<=TOL_FLUJO_CERO_NM3:
            continue

        lat=lon=None
        site_name=SITE_NAME.get(node,node)
        try:
            sr=_site_drop_index.loc[node]
            if isinstance(sr,pd.DataFrame):
                sr=sr.iloc[0]
            lat=_finite(sr['lat'])
            lon=_finite(sr['lon'])
            site_name=str(sr.get('nombre',site_name))
        except Exception:
            pass

        local_client_deliveries=[
            {
                'cliente_id':cid,
                'cliente':str(CL.loc[cid,'nombre']) if cid in CL.index else cid,
                'cantidad_gasbc_nm3':float(amount),
            }
            for cid,amount in sorted(local_client_amounts.items(), key=lambda item:(-item[1], item[0]))
            if amount>TOL_FLUJO_CERO_NM3
        ]

        events.append({
            'site_id':node,
            'site_name':site_name,
            'lat':lat,
            'lon':lon,
            'producto_descargado':'BioGás Crudo',
            'cantidad_dejada_nm3':raw_drop,
            'gasbc_vendido_local_nm3':gasbc_local,
            'ingreso_venta_local_usd':ingreso_local,
            'clientes_locales':_ordered_unique(local_clients),
            'clientes_entregas_gasbc':local_client_deliveries,
            'rutas_locales':_ordered_unique(local_routes),
        })
    return events


_PLANT_INPUT_PRODUCT={
    'BG':'Biogás Crudo',
    'L':'Biometano Gaseoso',
    'RL':'BioGNL',
    'C':'Biometano Gaseoso',
    'RC':'BioGNC',
}
_TOUR_INPUT_PRODUCT={
    'BIO':'Biogás Crudo',
    'BC':'Biometano Gaseoso',
    'BC_PROCESO':'Biometano Gaseoso',
    'GNL':'BioGNL',
    'GNC':'BioGNC',
}
_ACTIVE_PLANT_SITES={
    str(pm.get('site')) for pm in producer_meta.values()
    if pm.get('kind')=='Planta'
}
_SITE_IDS_BY_REFERENCE=defaultdict(list)
for _sid_interaction,_srow_interaction in sitios.set_index('id',drop=False).iterrows():
    _ref_interaction=str(_srow_interaction.get('referencia_local') or '').strip()
    if _ref_interaction:
        _SITE_IDS_BY_REFERENCE[_ref_interaction].append(str(_sid_interaction))


def _interaction_site_ids_from_code(raw_code):
    """Traduce una parada física a iconos de productora, sin revelar otro cliente."""
    code=str(raw_code or '').strip()
    if not code:
        return []
    candidates=[]
    if code.startswith('SITE:'):
        candidates.append(code.split(':',1)[1])
    elif code.startswith('CLIENT:'):
        ref=code.split(':',1)[1]
        candidates.extend(_SITE_IDS_BY_REFERENCE.get(ref,[]))
    elif code.startswith('COMPANY:'):
        ref=code.split(':',1)[1]
        candidates.extend(_SITE_IDS_BY_REFERENCE.get(ref,[]))
    else:
        if code in S:
            candidates.append(code)
        candidates.extend(_SITE_IDS_BY_REFERENCE.get(code,[]))
    return _ordered_unique([s for s in candidates if str(s) in _ACTIVE_PLANT_SITES])


def _append_node_interaction(out,key,entry):
    key=str(key)
    if not key:
        return
    rows=out.setdefault(key,[])
    signature=(
        str(entry.get('kind','')),
        str(entry.get('plant_type','')),
        str(entry.get('product','')),
        round(_finite(entry.get('quantity_nm3',0.0)),8),
        str(entry.get('note','')),
    )
    for old_entry in rows:
        old_signature=(
            str(old_entry.get('kind','')),
            str(old_entry.get('plant_type','')),
            str(old_entry.get('product','')),
            round(_finite(old_entry.get('quantity_nm3',0.0)),8),
            str(old_entry.get('note','')),
        )
        if old_signature==signature:
            return
    rows.append(entry)


def _route_node_interactions(rec,segment_keys,tour_ids,drop_events):
    """Metadata puramente visual para tooltips dependientes de la ruta seleccionada."""
    out={}
    final_client=str(rec.get('cliente_id'))
    company=str(rec.get('empresa_id'))
    raw_q=max(0.0,_finite(rec.get('biogas_crudo_entrada_BG_nm3',0.0)))
    final_q=max(0.0,_finite(rec.get('cantidad_nm3',0.0)))

    _append_node_interaction(out,f'company|{company}',{
        'kind':'company','product':'Biogás Crudo','quantity_nm3':raw_q,
        'note':'Biogás crudo extraído para esta ruta',
    })
    shared_deliveries=list(rec.get('clientes_compartidos', []) or [])
    if not shared_deliveries:
        shared_deliveries=[{
            'cliente_id':final_client,
            'cliente':str(rec.get('cliente','')),
            'cantidad_gasbc_nm3':final_q,
        }]
    for delivery in shared_deliveries:
        cid=str(delivery.get('cliente_id',''))
        if not cid or cid in {'None','nan'}:
            continue
        q_delivery=max(0.0,_finite(delivery.get('cantidad_gasbc_nm3',0.0)))
        note='GasBC entregado al cliente principal' if cid==final_client else 'GasBC entregado a este cliente en la ruta compartida'
        _append_node_interaction(out,f'client|{cid}',{
            'kind':'client','product':'GasBC','quantity_nm3':q_delivery,
            'note':note,
        })

    for pk in _route_producer_keys(rec):
        pm=producer_meta.get(pk,{})
        if pm.get('kind')!='Planta':
            continue
        q=max(0.0,_finite(_route_input_for_producer(rec,pk)))
        if q<=TOL_FLUJO_CERO_NM3:
            continue
        t=str(pm.get('type',''))
        _append_node_interaction(out,f"site|{pm.get('site')}",{
            'kind':'plant',
            'plant_type':TYPE_NAME.get(t,t),
            'product':_PLANT_INPUT_PRODUCT.get(t,'Producto'),
            'quantity_nm3':q,
            'note':'Cantidad que ingresó a la productora en esta ruta',
        })

    for event in drop_events or []:
        q=max(0.0,_finite(event.get('cantidad_dejada_nm3',0.0)))
        if q<=TOL_FLUJO_CERO_NM3:
            continue
        _append_node_interaction(out,f"site|{event.get('site_id')}",{
            'kind':'shared_drop',
            'product':str(event.get('producto_descargado') or 'Biogás Crudo'),
            'quantity_nm3':q,
            'note':'Cantidad dejada aquí para abastecimiento secundario',
        })

    # Las paradas compartidas del reparto se muestran como iconos de CLIENTE,
    # no como ubicaciones/productoras. Evita que una ruta de Carozzi, por ejemplo,
    # muestre una productora lejana sólo porque el mismo camión continuó a Cermaq.

    # Tooltip específico para los iconos de gasoducto convencional de ESTA ruta.
    q_pipe=max(0.0,_finite(rec.get('gasbc_entrada_corredor_nm3',
                                   rec.get('gasbc_entrada_ruta_actual_nm3',
                                           rec.get('cantidad_nm3',0.0)))))
    if q_pipe<=TOL_FLUJO_CERO_NM3:
        q_pipe=final_q
    for pipe_key in _route_pipe_node_keys(rec,segment_keys):
        _append_node_interaction(out,pipe_key,{
            'kind':'pipeline',
            'product':'Biometano Gaseoso',
            'quantity_nm3':q_pipe,
            'note':'Gasoducto convencional utilizado/interactuado por esta ruta',
        })
    return out


def _client_name_for_payload(cliente_id):
    cliente_id=str(cliente_id)
    try:
        return str(CL.loc[cliente_id,'nombre'])
    except Exception:
        return cliente_id


def _add_shared_client_delivery(acc, cliente_id, quantity_nm3, route_id=None):
    cliente_id=str(cliente_id or '').strip()
    if not cliente_id or cliente_id in {'None','nan'}:
        return
    q=max(0.0,_finite(quantity_nm3))
    if q<=TOL_FLUJO_CERO_NM3:
        return
    row=acc.setdefault(cliente_id,{
        'cliente_id':cliente_id,
        'cliente':_client_name_for_payload(cliente_id),
        'cantidad_gasbc_nm3':0.0,
        'route_ids':[],
    })
    row['cantidad_gasbc_nm3']+=q
    if route_id is not None:
        rid=str(route_id)
        if rid and rid not in row['route_ids']:
            row['route_ids'].append(rid)


_TOUR_MAP_BY_ID={str(t.get('transport_route_id')):t for t in transport_tours_payload}


def _client_id_from_stop_code(stop_code):
    try:
        cid=cliente_unico_de_parada(stop_code)
        if cid is not None and str(cid) not in {'','None','nan'}:
            return str(cid)
    except Exception:
        pass
    code=str(stop_code or '').strip()
    if code.startswith('CLIENT:'):
        return code.split(':',1)[1]
    if code.startswith('NC_'):
        return code[3:]
    return None


def _processed_delivery_to_gasbc_nm3(mode, quantity_nm3):
    q=max(0.0,_finite(quantity_nm3))
    mode=str(mode or '').upper()
    factors=globals().get('FACTORES_CONVERSION',{})
    if mode=='GNL':
        return q*float(factors.get('RL',600.0))
    if mode=='GNC':
        return q*float(factors.get('RC',303.03))
    if mode=='BIO':
        return q*float(factors.get('BG',1.0))
    return q


def _tour_client_deliveries_gasbc(tour):
    acc={}
    if not tour:
        return []
    tid=str(tour.get('transport_route_id',''))
    mode=str(tour.get('mode',''))
    for decision in tour.get('stop_decisions',[]) or []:
        cid=_client_id_from_stop_code(decision.get('stop_id'))
        q=_processed_delivery_to_gasbc_nm3(mode,decision.get('processed_load',0.0))
        _add_shared_client_delivery(acc,cid,q,tid)
    deliveries=list(acc.values())
    deliveries.sort(key=lambda row:(-_finite(row.get('cantidad_gasbc_nm3',0.0)),str(row.get('cliente_id'))))
    return deliveries


def _tour_positive_segment_keys(tour_ids):
    keys=[]
    for tid in tour_ids or []:
        tour=_TOUR_MAP_BY_ID.get(str(tid))
        if not tour:
            continue
        segments=sorted(tour.get('segments',[]) or [],key=lambda seg:int(seg.get('segment_index',0) or 0))
        for seg in segments:
            key=str(seg.get('key','')).strip()
            if not key:
                continue
            if _finite(seg.get('load_nm3',0.0))<=TOL_FLUJO_CERO_NM3:
                continue
            if key not in keys:
                keys.append(key)
    return keys


def _tour_segment_key_set(tour_ids):
    keys=set()
    for tid in tour_ids or []:
        tour=_TOUR_MAP_BY_ID.get(str(tid))
        if not tour:
            continue
        for seg in tour.get('segments',[]) or []:
            key=str(seg.get('key','')).strip()
            if key:
                keys.add(key)
    return keys


def _non_tour_visual_segments(segment_keys,tour_ids):
    tour_keys=_tour_segment_key_set(tour_ids)
    if not tour_keys:
        return list(segment_keys or [])
    return [str(k) for k in (segment_keys or []) if str(k) not in tour_keys]


def _shared_client_deliveries_for_route(rec, source_route_id, tour_ids, drop_events):
    acc={}
    final_client=str(rec.get('cliente_id'))
    tour_set={str(t) for t in (tour_ids or []) if str(t) not in {'','None','nan'}}

    # Preferir la verdad operacional del viaje: parada, cliente y carga descargada.
    # Para GNC/GNL se convierte la carga procesada de vuelta a GasBC.
    for tid in tour_set:
        for delivery in _tour_client_deliveries_gasbc(_TOUR_MAP_BY_ID.get(tid)):
            _add_shared_client_delivery(
                acc,
                delivery.get('cliente_id'),
                delivery.get('cantidad_gasbc_nm3',0.0),
                tid,
            )

    if not acc or final_client not in acc:
        _add_shared_client_delivery(acc, final_client, rec.get('cantidad_nm3',0.0), source_route_id)

    for event in drop_events or []:
        for delivery in event.get('clientes_entregas_gasbc',[]) or []:
            _add_shared_client_delivery(
                acc,
                delivery.get('cliente_id'),
                delivery.get('cantidad_gasbc_nm3',0.0),
                None,
            )

    deliveries=list(acc.values())
    deliveries.sort(
        key=lambda row:(
            0 if str(row.get('cliente_id'))==final_client else 1,
            -_finite(row.get('cantidad_gasbc_nm3',0.0)),
            str(row.get('cliente_id')),
        )
    )
    for row in deliveries:
        row['cantidad_gasbc_nm3']=float(row['cantidad_gasbc_nm3'])
        row['route_ids']=_ordered_unique(row.get('route_ids',[]))
    return deliveries


route_payload = []
visual_route_ids_by_source=defaultdict(list)

if len(df_rutas_estadisticas):
    for _, row in df_rutas_estadisticas.iterrows():
        base_rec=row.to_dict()
        for variant in _visual_route_variants(row):
            rec=dict(base_rec)
            fraction=float(variant.get('fraction',1.0))
            for field in _VISUAL_ADDITIVE_FIELDS:
                if field in rec and isinstance(rec[field],(int,float,np.integer,np.floating)) and not isinstance(rec[field],bool):
                    rec[field]=float(rec[field])*fraction

            route_id=str(variant['route_id'])
            source_route_id=str(variant['source_route_id'])
            tour_ids=_ordered_unique(variant.get('tour_ids',[]))
            item={
                k:(None if isinstance(v,float) and not math.isfinite(v) else v)
                for k,v in rec.items() if not str(k).startswith('_')
            }
            item['route_id']=route_id
            item['source_route_id']=source_route_id
            item['visual_split_fraction']=fraction
            item['transport_tour_ids']=tour_ids
            item['intermediate_drop_events']=_route_intermediate_drop_events(base_rec)
            shared_deliveries=_shared_client_deliveries_for_route(
                rec,source_route_id,tour_ids,item['intermediate_drop_events']
            )
            visual_segments=_route_visual_segment_keys(rec,variant.get('segments',[]))
            item['transport_packs']=_transport_pack_sequence(rec,visual_segments,tour_ids=tour_ids)
            item['segmentos']=_non_tour_visual_segments(visual_segments,tour_ids)
            item['segment_defs']=_segment_defs(rec,item['segmentos'],tour_ids=tour_ids)
            item['usa_gasoducto_convencional']=_uses_conventional_pipeline(item['segmentos'])
            item['clientes_compartidos']=shared_deliveries
            item['clientes_compartidos_texto']='; '.join(
                f"{d['cliente_id']} · {d['cliente']}: {d['cantidad_gasbc_nm3']:.3f} Nm3"
                for d in shared_deliveries
            )
            panel_delivery=shared_deliveries[0] if shared_deliveries else {
                'cliente_id':str(rec.get('cliente_id')),
                'cliente':str(rec.get('cliente','')),
            }
            item['panel_cliente_id']=str(panel_delivery.get('cliente_id'))
            item['panel_cliente']=str(panel_delivery.get('cliente'))
            rec['clientes_compartidos']=shared_deliveries
            rec['panel_cliente_id']=item['panel_cliente_id']
            rec['panel_cliente']=item['panel_cliente']
            item['node_interactions']=_route_node_interactions(
                rec,item['segmentos'],tour_ids,item['intermediate_drop_events']
            )
            _visual_node_keys=set(_route_node_keys(rec,item['segmentos']))
            _visual_node_keys.update(item['node_interactions'].keys())
            _final_client_key=f"client|{rec.get('cliente_id')}"
            _shared_client_keys={
                f"client|{d.get('cliente_id')}" for d in shared_deliveries
                if d.get('cliente_id') is not None
            }
            _shared_client_keys.add(_final_client_key)
            item['node_keys']=sorted(
                k for k in _visual_node_keys
                if not str(k).startswith('client|') or str(k) in _shared_client_keys
            )
            item['familia_corta']=' → '.join(
                str(pack.get('label','')) for pack in item.get('transport_packs',[]) if pack.get('label')
            )
            if not item['familia_corta']:
                item['familia_corta']=_route_family_label(rec,item['segmentos'],tour_ids=tour_ids)
            # El panel muestra exclusivamente la secuencia de transporte real.
            item['proceso']=item['familia_corta']
            item['etapas']=[item['familia_corta']] if item['familia_corta'] else []
            item['cadena']=item['familia_corta']
            item['color']=(
                item['transport_packs'][0]['color'] if item.get('transport_packs')
                else _route_family_color(rec,item['segmentos'],tour_ids=tour_ids)
            )
            route_payload.append(item)
            visual_route_ids_by_source[source_route_id].append(route_id)

client_totals = {}
for cliente_id, sub in df_rutas_estadisticas.groupby("cliente_id", sort=False) if len(df_rutas_estadisticas) else []:
    cliente_id = str(cliente_id)
    demand = _finite(CL.loc[cliente_id, "demanda_max_nm3"])
    price = _finite(precio_promedio_venta_cliente.get(cliente_id, CL.loc[cliente_id, "precio_venta_usd_nm3"]))
    distance_road = _finite(sub["distancia_vial_km"].sum())
    distance_pipe = _finite(sub["distancia_interna_gasoducto_referencial_km"].sum())
    transport_cost = _finite(sub["costo_transporte_usd"].sum())
    client_totals[cliente_id] = {
        "cliente": str(CL.loc[cliente_id, "nombre"]),
        "demanda": demand,
        "venta": _finite(sub["cantidad_nm3"].sum()),
        "precio": price,
        "ingresos": _finite(sub["ingreso_usd"].sum()),
        "costos": _finite(sub["costo_total_usd"].sum()),
        "costos_variables": _finite(sub["costo_variable_total_usd"].sum()),
        "costos_fijos_asignados": _finite(sub["costos_fijos_asignados_usd"].sum()),
        "margen_antes_costos_fijos": _finite(sub["margen_ruta_antes_costos_fijos_usd"].sum()),
        "margen_sin_fijos": _finite(sub["margen_operativo_por_ruta_usd"].sum()),
        "margen_operativo_total": _finite(sub["margen_operativo_por_ruta_usd"].sum()),
        # Definición solicitada: suma de los beneficios de todas las rutas del cliente.
        "beneficio": _finite(sub["beneficio_por_ruta_usd"].sum()),
        "distancia_vial": distance_road,
        "distancia_gasoducto": distance_pipe,
        "distancia_total": distance_road + distance_pipe,
        "costo_transporte": transport_cost,
        "costos_fijos_empresa": _finite(sub["costo_empresa_fijo_usd"].sum()),
        "costos_fijos_productoras": _finite(sub["costo_productoras_fijo_usd"].sum()),
        "costos_fijos_gasoducto": _finite(sub["costo_gasoducto_fijo_usd"].sum()),
        "costo_promedio_km": transport_cost / distance_road if distance_road > EPS else 0.0,
        "rutas": int(len(sub)),
    }


systems_payload=[]
if len(df_sistemas_independientes):
    for _,system_row in df_sistemas_independientes.iterrows():
        rec=system_row.to_dict()
        original_ids=[str(x) for x in (rec.get('route_ids',[]) or [])]
        visual_ids=[]
        for rid in original_ids:
            visual_ids.extend(visual_route_ids_by_source.get(rid,[rid]))
        rec['route_ids']=_ordered_unique(visual_ids)
        rec['numero_rutas']=len(rec['route_ids'])
        systems_payload.append({
            k:(v.tolist() if isinstance(v,np.ndarray) else v)
            for k,v in rec.items()
        })

# Sitios enriquecidos con los tipos de planta realmente activos. Un sitio puede
# contener varias unidades y varios tipos; el mapa los muestra en el tooltip.
active_types_by_site = defaultdict(list)
for _, plant_row in df_plantas_resultado.iterrows():
    site_id = str(plant_row["sitio_id"])
    t = str(plant_row["tipo"])
    label = TYPE_NAME.get(t, t)
    if label not in active_types_by_site[site_id]:
        active_types_by_site[site_id].append(label)
site_payload = clean_records(sitios, ["id", "nombre", "lat", "lon", "region", "clase_nodo", "referencia_local", "tecnologias_permitidas"])
for site_item in site_payload:
    labels = active_types_by_site.get(str(site_item["id"]), [])
    site_item["tipos_planta_activos"] = labels
    site_item["tipos_planta_activos_texto"] = " · ".join(labels)
    if str(site_item.get('clase_nodo',''))=='ENTRADA_GASODUCTO':
        referencia=str(site_item.get('referencia_local') or site_item.get('id'))
        site_item['nombre_visual']=f'Ubicación logística {referencia}'
        site_item['clase_visual']='UBICACION_LOGISTICA'
    else:
        site_item['nombre_visual']=site_item.get('nombre')
        site_item['clase_visual']=site_item.get('clase_nodo')

gasoductos_biogas_cortos_html=gasoductos_biogas_cortos_todos.copy()
if len(gasoductos_biogas_cortos_html):
    _h_disponibles=set(H_CORTO)
    gasoductos_biogas_cortos_html['disponible_en_milp']=gasoductos_biogas_cortos_html['id'].astype(str).isin(_h_disponibles)
    gasoductos_biogas_cortos_html['activado']=gasoductos_biogas_cortos_html['id'].astype(str).map(
        lambda h: int(round(value('BCP',h))) if h in _h_disponibles else 0
    )
    gasoductos_biogas_cortos_html['flujo_total_nm3']=gasoductos_biogas_cortos_html['id'].astype(str).map(
        lambda h: value('QBCP',h) if h in _h_disponibles else 0.0
    )
    gasoductos_biogas_cortos_html['estado_operacional']=gasoductos_biogas_cortos_html.apply(
        lambda r: (
            'ACTIVADO' if int(r['activado'])==1 and float(r['flujo_total_nm3'])>TOL_FLUJO_CERO_NM3
            else 'CANDIDATO_NO_UTILIZADO'
        ),axis=1
    )
    gasoductos_biogas_cortos_html['costo_variable_total_usd']=gasoductos_biogas_cortos_html.apply(
        lambda r: float(r['flujo_total_nm3'])*float(r['distancia_recta_km'])*float(r['costo_variable_usd_nm3_km']),axis=1
    )
else:
    gasoductos_biogas_cortos_html=pd.DataFrame(columns=list(gasoductos_biogas_cortos_todos.columns)+[
        'disponible_en_milp','activado','flujo_total_nm3','estado_operacional','costo_variable_total_usd'
    ])

# Compatibilidad con una sesión de Colab donde la celda de resultados PHUB
# pudiera haber quedado ejecutada con el nombre antiguo. El HTML siempre usa
# la convención semántica gasoducto_corto_id.
df_gasoductos_biogas_cortos_hub_p_html = df_gasoductos_biogas_cortos_hub_p_resultado.copy()
if (
    len(df_gasoductos_biogas_cortos_hub_p_html)
    and 'gasoducto_corto_id' not in df_gasoductos_biogas_cortos_hub_p_html.columns
    and 'gasoducto_id' in df_gasoductos_biogas_cortos_hub_p_html.columns
):
    df_gasoductos_biogas_cortos_hub_p_html = df_gasoductos_biogas_cortos_hub_p_html.rename(
        columns={'gasoducto_id':'gasoducto_corto_id'}
    )

payload = {
    "companies": clean_records(empresas, [
        "id", "nombre", "lat", "lon", "capacidad_max_biogas_nm3",
        "costo_activacion_biogas_usd", COL_PRECIO_BIOGAS_CRUDO
    ]),
    "clients": clean_records(clientes, [
        "id", "nombre", "lat", "lon", "demanda_max_nm3", "precio_venta_usd_nm3"
    ]),
    "sites": site_payload,
    "pipelines": clean_records(gasoductos, [
        "id", "nombre", "punto_a_nombre", "punto_a_zona", "punto_a_lat", "punto_a_lon",
        "punto_b_nombre", "punto_b_zona", "punto_b_lat", "punto_b_lon",
        "distancia_red_estimada_km", "costo_fijo_usd_dia", "costo_variable_usd_km_nm3"
    ]),
    "pipelinePairDistances": clean_records(distancias_entre_gasoductos_36, [
        "gasoducto_1_id","gasoducto_1_nombre","gasoducto_2_id","gasoducto_2_nombre",
        "distancia_km","metodo_distancia"
    ]),
    "pipelineTransfers": clean_records(df_transferencias_gasoducto_gasoducto, [
        "gasoducto_origen_id","gasoducto_origen","gasoducto_destino_id","gasoducto_destino",
        "distancia_km","flujo_nm3","costo_variable_usd_km_nm3","costo_variable_total_usd"
    ]) if len(df_transferencias_gasoducto_gasoducto) else [],
    "shortBiogasPipelines": clean_records(gasoductos_biogas_cortos_html, [
        "id", "nombre", "empresa_id", "empresa",
        "destino_tipo", "destino_id", "destino_nombre", "nodo_destino_id",
        "cliente_id", "cliente", "ubicacion_id", "ubicacion",
        "punto_a_lat", "punto_a_lon", "punto_b_lat", "punto_b_lon",
        "distancia_recta_km", "costo_fijo_usd_dia",
        "costo_variable_usd_nm3_km", "capacidad_max_nm3_dia",
        "empresa_disponible_en_milp", "disponible_en_milp",
        "activado", "flujo_total_nm3", "estado_operacional",
        "costo_variable_total_usd"
    ]) if len(gasoductos_biogas_cortos_html) else [],
    "shortBiogasHubPipelines": clean_records(df_gasoductos_biogas_cortos_cliente_cliente_resultado, [
        "gasoducto_id","cliente_origen_id","nodo_origen_id","cliente_destino_id","nodo_destino_id",
        "distancia_recta_km","capacidad_efectiva_nm3","costo_fijo_usd_dia","costo_variable_usd_nm3_km",
        "activado","flujo_total_nm3","punto_a_lat","punto_a_lon","punto_b_lat","punto_b_lon"
    ]) if len(df_gasoductos_biogas_cortos_cliente_cliente_resultado) else [],

    "shortBiogasPHubPipelines": clean_records(df_gasoductos_biogas_cortos_hub_p_html, [
        "gasoducto_corto_id", "origen_tipo", "origen_id", "nodo_origen_id", "destino_tipo",
        "destino_id", "nodo_destino_id", "distancia_recta_km", "capacidad_efectiva_nm3",
        "costo_fijo_usd_dia", "costo_variable_usd_nm3_km", "activado", "flujo_total_nm3",
        "punto_a_lat", "punto_a_lon", "punto_b_lat", "punto_b_lon"
    ]) if len(df_gasoductos_biogas_cortos_hub_p_html) else [],
    "shortGasBCInletPipelines": clean_records(df_gasoductos_gasbc_cortos_empresa_entrada_resultado, [
        "gasoducto_corto_id", "empresa_id", "nodo_origen_id", "direccion_id", "gasoducto_id",
        "distancia_recta_km", "capacidad_efectiva_nm3", "costo_fijo_usd_dia",
        "costo_variable_usd_nm3_km", "activado", "flujo_total_nm3",
        "punto_a_lat", "punto_a_lon", "punto_b_lat", "punto_b_lon"
    ]) if len(df_gasoductos_gasbc_cortos_empresa_entrada_resultado) else [],
    "shortGasBCDirectPipelines": clean_records(df_gasoductos_gasbc_cortos_empresa_destino_resultado, [
        "gasoducto_corto_id", "origen_tipo", "origen_id", "origen", "empresa_id", "nodo_origen_id", "destino_tipo",
        "destino_id", "nodo_destino_id", "cliente_id", "ubicacion_id",
        "distancia_recta_km", "capacidad_efectiva_nm3", "costo_fijo_usd_dia",
        "costo_variable_usd_nm3_km", "activado", "flujo_total_nm3",
        "punto_a_lat", "punto_a_lon", "punto_b_lat", "punto_b_lon"
    ]) if len(df_gasoductos_gasbc_cortos_empresa_destino_resultado) else [],
    "shortGasBCPipelines": clean_records(df_gasoductos_gasbc_cortos_salida_cliente_resultado, [
        "gasoducto_corto_id", "direccion_id", "gasoducto_id", "destino_tipo", "destino_id",
        "destino", "nodo_destino_id", "cliente_id", "empresa_id",
        "distancia_recta_km", "capacidad_efectiva_nm3", "costo_fijo_usd_dia",
        "costo_variable_usd_nm3_km", "activado", "flujo_total_nm3",
        "punto_a_lat", "punto_a_lon", "punto_b_lat", "punto_b_lon"
    ]) if len(df_gasoductos_gasbc_cortos_salida_cliente_resultado) else [],
    "straightLines": {
        **{
            f"PIPEINTRA|{str(r.direccion_id)}":[
                [float(r.entrada_lat),float(r.entrada_lon)],
                [float(r.salida_lat),float(r.salida_lon)],
            ]
            for r in direcciones_gasoductos.itertuples(index=False)
        },
        **{
            f"PIPEPAIR|{str(r.gasoducto_1_id)}|{str(r.gasoducto_2_id)}":[
                [float(r.referencia_1_lat),float(r.referencia_1_lon)],
                [float(r.referencia_2_lat),float(r.referencia_2_lon)],
            ]
            for r in distancias_entre_gasoductos_36.itertuples(index=False)
        },
        **{
            f"PIPEPAIR|{str(r.gasoducto_2_id)}|{str(r.gasoducto_1_id)}":[
                [float(r.referencia_2_lat),float(r.referencia_2_lon)],
                [float(r.referencia_1_lat),float(r.referencia_1_lon)],
            ]
            for r in distancias_entre_gasoductos_36.itertuples(index=False)
        },
        **{
            f"SHORTBIO|{h}":[
                [float(GCORTO.loc[h,'punto_a_lat']),float(GCORTO.loc[h,'punto_a_lon'])],
                [float(GCORTO.loc[h,'punto_b_lat']),float(GCORTO.loc[h,'punto_b_lon'])],
            ]
            for h in H_CORTO
            if (value('QBCP',h)+(value('QBCP_HUB',h) if h in H_EHUB_SET else 0.0))>TOL_FLUJO_CERO_NM3
        },
        **{
            f"SHORTBIOHUBEMP|{h}":[
                [float(EHUB.loc[h,'punto_a_lat']),float(EHUB.loc[h,'punto_a_lon'])],
                [float(EHUB.loc[h,'punto_b_lat']),float(EHUB.loc[h,'punto_b_lon'])],
            ]
            for h in H_EHUB_SOLO
            if value('QBCP_HUB',h)>TOL_FLUJO_CERO_NM3
        },
        **{
            f"SHORTBIOC2C|{h}":[
                [float(BIOC2C.loc[h,'punto_a_lat']),float(BIOC2C.loc[h,'punto_a_lon'])],
                [float(BIOC2C.loc[h,'punto_b_lat']),float(BIOC2C.loc[h,'punto_b_lon'])],
            ]
            for h in H_BIOC2C
            if value('QBC2C',h)>TOL_FLUJO_CERO_NM3
        },

        **{
            f"SHORTBIOPHUB|{h}":[
                [float(PHUB.loc[h,'punto_a_lat']),float(PHUB.loc[h,'punto_a_lon'])],
                [float(PHUB.loc[h,'punto_b_lat']),float(PHUB.loc[h,'punto_b_lon'])],
            ]
            for h in H_PHUB
            if value('QBCP_P',h)>TOL_FLUJO_CERO_NM3
        },
        **{
            f"SHORTGASIN|{h}":[
                [float(GECORTO.loc[h,'punto_a_lat']),float(GECORTO.loc[h,'punto_a_lon'])],
                [float(GECORTO.loc[h,'punto_b_lat']),float(GECORTO.loc[h,'punto_b_lon'])],
            ]
            for h in H_GECORTO
            if value('QGCE',h)>TOL_FLUJO_CERO_NM3
        },
        **{
            f"SHORTGASDIRECT|{h}":[
                [float(GDCORTO.loc[h,'punto_a_lat']),float(GDCORTO.loc[h,'punto_a_lon'])],
                [float(GDCORTO.loc[h,'punto_b_lat']),float(GDCORTO.loc[h,'punto_b_lon'])],
            ]
            for h in H_GDCORTO
            if (value('QGCD',h)+value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h))>TOL_FLUJO_CERO_NM3
        },
        **{
            f"SHORTGAS|{h}":[
                [float(GSCORTO.loc[h,'punto_a_lat']),float(GSCORTO.loc[h,'punto_a_lon'])],
                [float(GSCORTO.loc[h,'punto_b_lat']),float(GSCORTO.loc[h,'punto_b_lon'])],
            ]
            for h in H_GSCORTO
            if (value('QGCP',h)+value('QGCP_HUB',h))>TOL_FLUJO_CERO_NM3
        },
    },
    "routes": route_payload,
    "transportTours": transport_tours_payload,
    "clientTotals": client_totals,
    "systems": systems_payload,
    "polylines": {
        rk: poly_cache[rk].get("encoded_polyline")
        for rk in active_road_keys
        if poly_valid(poly_cache.get(rk), rk)
    },
    "familyColors": FAMILY_COLORS,
    "segmentColors": SEGMENT_COLORS,
    "visibleFlowTolerance": float(TOL_FLUJO_CERO_NM3),
    "dataHtml": make_model_data_html(),
    "statsHtml": make_stats_html(),
    "chartSpecs": chart_specs,
}

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Optimización de Biometano — Rutas locales</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root{--ink:#172033;--muted:#64748b;--line:#dbe4ee;--panel:#ffffff;--soft:#f6f8fb}
html,body,#map{width:100%;height:100%;margin:0;padding:0;overflow:hidden;background:#eef2f6;font-family:Inter,Arial,sans-serif;color:var(--ink)}*{box-sizing:border-box}
#map{position:absolute;inset:0}.leaflet-control-scale{display:none}.leaflet-overlay-pane svg path{transition:stroke-opacity .16s ease,stroke-width .16s ease}
.map-action{position:absolute;z-index:1100;top:14px;border:1px solid #cbd5e1;background:rgba(255,255,255,.98);color:var(--ink);border-radius:11px;padding:9px 13px;font:750 13px Arial,sans-serif;box-shadow:0 8px 24px rgba(15,23,42,.14);cursor:pointer}.map-action:hover{background:#f8fafc;transform:translateY(-1px)}.flow-action{left:14px}.systems-action{left:14px;top:58px}.data-action{right:116px}.stats-action{right:14px}.visibility-controls{position:absolute;z-index:1100;top:58px;right:14px;display:flex;justify-content:flex-end;flex-wrap:wrap;gap:6px;max-width:min(760px,calc(100% - 540px));pointer-events:none}.visibility-action{pointer-events:auto;border:1px solid #cbd5e1;background:rgba(255,255,255,.98);color:#334155;border-radius:10px;padding:7px 10px;font:750 11px Arial,sans-serif;box-shadow:0 6px 18px rgba(15,23,42,.12);cursor:pointer;white-space:nowrap}.visibility-action:hover{background:#f8fafc}.visibility-action.is-hidden{background:#172033;color:#fff;border-color:#172033}
.drawer{position:absolute;z-index:1090;top:102px;bottom:14px;background:rgba(255,255,255,.99);border:1px solid #cbd5e1;border-radius:16px;box-shadow:0 18px 48px rgba(15,23,42,.22);overflow:hidden;will-change:transform;transition:transform 300ms cubic-bezier(.22,.8,.22,1)}
.flow-drawer{left:14px;width:min(430px,calc(100% - 28px));transform:translateX(calc(-100% - 30px))}.flow-drawer.open{transform:translateX(0)}#flowPanel{width:min(500px,calc(100% - 28px))}
.side-drawer{right:14px;width:min(1080px,calc(100% - 28px));transform:translateX(calc(100% + 30px))}.side-drawer.open{transform:translateX(0)}
.drawer-head{display:flex;align-items:center;justify-content:space-between;gap:12px;padding:13px 14px;border-bottom:1px solid #e2e8f0;background:#fff}.drawer-head>div{display:flex;min-width:0;flex-direction:column}.drawer-head strong{font-size:16px}.drawer-head small{color:var(--muted);margin-top:2px}.drawer-head-actions{display:flex;align-items:center;gap:7px}.icon-square{min-width:32px;height:32px;border:1px solid #cbd5e1;background:#fff;color:#334155;border-radius:8px;font:800 12px Arial,sans-serif;cursor:pointer;display:inline-flex;align-items:center;justify-content:center;padding:0 10px}.icon-square:hover{background:#f8fafc}.drawer-close,.bubble-close{border:0;background:#f1f5f9;color:#334155;border-radius:9px;width:32px;height:32px;font-size:22px;cursor:pointer;flex:0 0 auto}
.groups-scroll{height:calc(100% - 61px);overflow-y:auto;overscroll-behavior:contain;scrollbar-width:thin;background:#fff}.client-group{border-bottom:1px solid #e7ebf0}.client-toggle{width:100%;border:0;background:#fff;display:grid;grid-template-columns:minmax(0,1fr) auto 20px;gap:9px;align-items:center;text-align:left;padding:11px 12px;cursor:pointer;color:var(--ink)}.client-toggle:hover{background:#f8fafc}.client-copy{min-width:0;display:flex;flex-direction:column}.client-copy strong{white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.client-copy small{color:var(--muted);margin-top:2px}.client-badge{min-width:25px;height:25px;border-radius:999px;background:#e8eef6;color:#334155;display:inline-flex;align-items:center;justify-content:center;font-size:11px;font-weight:800}.client-chevron{color:var(--muted);font-size:17px;transition:transform 220ms ease}.client-group.expanded .client-chevron{transform:rotate(180deg)}
.route-list{display:none;padding:8px 9px 11px;background:#f8fafc;max-height:none;overflow:visible}.client-group.expanded .route-list{display:grid;gap:7px}.route-select{width:100%;min-width:0;display:grid;grid-template-columns:22px minmax(260px,46%) minmax(0,1fr);grid-template-areas:'check family process';gap:5px 10px;text-align:left;border:1px solid #dbe2ea;border-left:5px solid var(--family);background:#fff;border-radius:11px;padding:9px 10px;cursor:pointer;color:var(--ink);overflow:hidden}.route-select:hover{border-color:#94a3b8;box-shadow:0 3px 10px rgba(15,23,42,.06)}.route-select.active{background:#f8fbff;box-shadow:inset 0 0 0 1px var(--family)}.route-check{grid-area:check;align-self:center;width:17px;height:17px;accent-color:var(--family);cursor:pointer}.route-family{grid-area:family;align-self:center;min-width:0}.route-pack-chain{display:flex;align-items:center;gap:5px;flex-wrap:wrap;line-height:1.2}.route-pack{font-size:11px;font-weight:900;color:var(--pack);white-space:nowrap}.route-pack-arrow{font-size:11px;font-weight:900;color:#94a3b8}.route-process{grid-area:process;min-width:0;color:var(--ink);font-weight:750;font-size:12px;line-height:1.38;white-space:normal;overflow-wrap:anywhere}.system-route-button{grid-template-columns:minmax(260px,46%) minmax(0,1fr);grid-template-areas:'family process';border-left-width:5px}.system-route-button.active{background:#eef6ff}.system-summary{font-size:11px;color:#475569;margin-top:3px;line-height:1.35}
.route-bubble{position:absolute;z-index:1085;right:14px;top:132px;width:min(445px,calc(100% - 28px));max-height:calc(100% - 74px);overflow:auto;background:rgba(255,255,255,.995);border:1px solid #cbd5e1;border-left:7px solid var(--route-color,#1565C0);border-radius:15px;box-shadow:0 14px 38px rgba(15,23,42,.22);font:13px/1.4 Arial,sans-serif;color:var(--ink);padding:14px 14px 16px;display:none;scrollbar-width:thin}.route-bubble.open{display:block}.bubble-close{position:absolute;right:8px;top:7px}.bubble-content h2{font-size:15px;margin:0 38px 5px 0}.bubble-chain{font-weight:800;color:#334155;padding:9px 10px;background:#f8fafc;border-radius:9px;margin:7px 0 11px}.bubble-part{border:1px solid #dbe2ea;border-radius:11px;background:#fff;padding:10px 11px 11px;margin-top:10px}.bubble-part-title{font-size:12px;font-weight:900;text-transform:uppercase;letter-spacing:.06em;color:var(--ink);margin:0 0 2px}.bubble-part-note{display:block;color:var(--muted);font-size:11px;margin-bottom:7px}.metric-section{border-top:1px solid #e2e8f0;padding-top:9px;margin-top:9px}.bubble-part .metric-section:first-of-type{border-top:0;padding-top:2px;margin-top:2px}.metric-row{display:grid;grid-template-columns:minmax(0,1fr) auto;gap:12px;padding:4px 0}.metric-row span{color:#475569}.metric-row b{text-align:right}.positive{color:#067647!important}.negative{color:#b42318!important}
.drawer-content{height:calc(100% - 61px);overflow:auto;padding:20px;background:#f4f7fb}.data-hero{padding:20px 22px;border:1px solid #dce5ef;border-radius:16px;background:linear-gradient(135deg,#ffffff,#f7f9fc);box-shadow:0 3px 14px rgba(15,23,42,.05)}.data-hero h2{margin:6px 0 7px;font-size:25px}.data-hero p{margin:0;color:#526174;line-height:1.55}.data-hero code{background:#eef2f7;border-radius:6px;padding:2px 5px}.data-kicker{font-size:11px;font-weight:900;letter-spacing:.13em;color:#64748b}.data-block{margin-top:28px}.section-heading{display:flex;align-items:flex-start;gap:12px;margin:0 0 12px}.section-heading>span{width:34px;height:34px;border-radius:10px;background:#eaf0f7;color:#334155;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:900}.section-heading h2{margin:0;font-size:19px}.section-heading p{margin:3px 0 0;color:#64748b;font-size:13px}.data-table{border-collapse:separate;border-spacing:0;width:100%;font-size:12px;margin:12px 0;background:#fff;border:1px solid #e2e8f0;border-radius:10px;overflow:hidden}.data-table th,.data-table td{border:0;border-bottom:1px solid #edf1f5;padding:8px 9px;text-align:left;vertical-align:top}.data-table th{background:#f5f7fa;position:sticky;top:0;color:#475569;font-size:11px;text-transform:none}.data-table tr:last-child td{border-bottom:0}details{border:1px solid #dfe6ee;border-radius:12px;padding:0;margin:10px 0;background:#fff;overflow:hidden}summary{font-weight:800;cursor:pointer;padding:12px 14px;list-style:none;background:#fbfcfe}summary::-webkit-details-marker{display:none}summary:after{content:'+';float:right;color:#64748b;font-size:18px;line-height:1}details[open] summary:after{content:'−'}details>.data-table,details>p{margin:0;border:0;border-radius:0}.summary-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:11px;margin:14px 0}.summary-card{background:#fff;border:1px solid #e1e8f0;border-radius:13px;padding:13px 14px;box-shadow:0 2px 8px rgba(15,23,42,.035)}.summary-card span{display:block;color:#64748b;font-size:12px;line-height:1.35}.summary-card b{display:block;font-size:20px;margin-top:5px;color:#0f172a}.summary-card small{font-size:10px}.data-summary{margin-top:14px}.summary-report-section>h2{font-size:17px;margin:0 0 9px}.summary-major-gap{height:32px}.section-gap{height:44px}.chart-card{background:linear-gradient(180deg,#fff 0%,#fbfdff 100%);border:1px solid #e2e8f0;border-radius:18px;padding:18px 18px 12px;margin:18px 0;box-shadow:0 10px 30px rgba(15,23,42,.055);overflow:hidden}.chart-card>h3{margin:0 0 5px;font-size:17px;line-height:1.3;letter-spacing:-.015em;color:#0f172a}.chart-note{display:block;color:#64748b;line-height:1.45;margin:0 0 10px;font-size:12px}.chart-scroll{width:100%;max-height:820px;overflow:auto;border:0;border-radius:14px;background:#f8fafc;padding:6px}.plotly-chart{width:100%;min-height:310px}.modebar{opacity:.18}.chart-scroll:hover .modebar{opacity:.85}.note{background:#f8fafc;border:1px solid #e2e8f0;border-left:4px solid #94a3b8;padding:11px;margin:12px 0;border-radius:9px}
.node-div-icon{background:transparent!important;border:0!important}.node-pin{--node:#334155;position:relative;width:38px;height:38px;border-radius:12px 12px 12px 4px;transform:rotate(-45deg);background:var(--node);border:3px solid #fff;box-shadow:0 4px 12px rgba(15,23,42,.28);display:flex;align-items:center;justify-content:center}.node-pin svg{width:21px;height:21px;transform:rotate(45deg);fill:none;stroke:#fff;stroke-width:1.9;stroke-linecap:round;stroke-linejoin:round}.node-pin.productor{border-radius:50% 50% 50% 6px}.node-pin.pipeline{border-radius:9px 9px 9px 3px}
.tour-stop-div{background:transparent!important;border:0!important;z-index:100000!important}.tour-stop-badge{width:27px;height:27px;border-radius:999px;background:#fff;border:3px solid var(--tour-color,#334155);color:#0f172a;display:flex;align-items:center;justify-content:center;font:900 11px Arial,sans-serif;box-shadow:0 3px 10px rgba(15,23,42,.3)}.tour-stop-badge.origin{background:var(--tour-color,#334155);color:#fff}
.icon-legend,.line-legend{position:absolute;z-index:900;background:rgba(255,255,255,.97);border:1px solid #cbd5e1;border-radius:13px;box-shadow:0 8px 24px rgba(15,23,42,.14);font:12px/1.35 Arial,sans-serif;color:var(--ink)}.icon-legend{right:14px;bottom:14px;padding:10px 12px;min-width:240px}.icon-grid{display:grid;grid-template-columns:1fr 1fr;gap:8px 12px;margin-top:8px}.legend-node{display:flex;align-items:center;gap:8px}.legend-node b{width:24px;height:24px;border-radius:8px;background:var(--c);display:flex;align-items:center;justify-content:center}.legend-node svg{width:15px;height:15px;fill:none;stroke:#fff;stroke-width:2;stroke-linecap:round;stroke-linejoin:round}.line-legend{left:14px;bottom:14px;padding:10px 12px;min-width:210px}.line-legend>div{display:flex;align-items:center;gap:8px;margin-top:7px}.line-legend i{display:inline-block;width:44px;border-top:7px solid var(--c);border-radius:4px}.line-legend .dash-lng{border-top-style:dashed}.line-legend .dash-gnc{border-top-style:dotted}.line-legend .dash-pipe{border-top-style:double}
@media(max-width:850px){.data-action{right:105px}.visibility-controls{left:7px;right:7px;top:102px;max-width:none}.route-bubble{top:172px}#flowPanel{width:auto}.route-bubble{left:7px;right:auto;top:auto;bottom:7px;width:calc(100% - 14px);max-height:48%}.flow-drawer,.side-drawer{left:7px;right:7px;width:auto}.icon-legend{display:none}.line-legend{left:7px;bottom:7px}.summary-grid{grid-template-columns:1fr 1fr}}
@media(max-width:520px){.summary-grid{grid-template-columns:1fr}.line-legend{display:none}}

/* Corrección visual: panel ancho y cada trayecto físico en su propia fila. */
#flowPanel{width:min(690px,calc(100% - 28px))!important}
.route-select{grid-template-columns:22px minmax(0,1fr)!important;grid-template-areas:"check family" "check process"!important;align-items:start!important}
.route-select input{grid-area:check;margin-top:4px}.route-family{grid-area:family}.route-process{grid-area:process;color:#64748b;font-size:11px;white-space:normal!important;overflow:visible!important;text-overflow:clip!important}
.route-pack-chain{display:flex;flex-wrap:wrap;align-items:center;gap:5px}.route-pack{display:inline-flex;align-items:center;padding:4px 7px;border:2px solid var(--pack);border-radius:999px;background:#fff;font-weight:800;color:#172033}.route-pack-arrow{font-weight:900;color:#64748b}
</style>
</head>
<body>
<div id="map"></div>
<button id="flowButton" type="button" class="map-action flow-action">☰ Ruta Óptima</button>
<button id="systemsButton" type="button" class="map-action systems-action">◫ Sistemas Independientes</button>
<button id="dataButton" type="button" class="map-action data-action">Datos</button>
<button id="statsButton" type="button" class="map-action stats-action">Estadísticas</button>
<div id="visibilityControls" class="visibility-controls" aria-label="Visibilidad del mapa">
  <button type="button" class="visibility-action" data-visibility="companies">Ocultar empresas</button>
  <button type="button" class="visibility-action" data-visibility="clients">Ocultar clientes</button>
  <button type="button" class="visibility-action" data-visibility="producerSites">Mostrar Productoras</button>
  <button type="button" class="visibility-action" data-visibility="sites">Ocultar ubicaciones posibles</button>
  <button type="button" class="visibility-action" data-visibility="shortPipelines">Ocultar gasoductos cortos</button>
  <button type="button" class="visibility-action" data-visibility="conventionalPipelines">Ocultar gasoductos convencionales</button>
</div>

<aside id="flowPanel" class="drawer flow-drawer" aria-hidden="true">
  <div class="drawer-head"><div><strong>Ruta Óptima</strong><small>Marca una o varias rutas para mostrarlas en el mapa</small></div>
    <div class="drawer-head-actions"><button id="clearRoutesButton" type="button" class="icon-square" title="Ocultar todas las rutas">Limpiar</button><button id="flowClose" type="button" class="drawer-close">×</button></div></div>
  <div id="groups" class="groups-scroll"></div>
</aside>

<aside id="systemsPanel" class="drawer flow-drawer" aria-hidden="true">
  <div class="drawer-head"><div><strong>Sistemas Independientes</strong><small>Rutas conectadas por empresas, clientes o activos compartidos</small></div>
    <div class="drawer-head-actions"><button id="clearSystemsButton" type="button" class="icon-square" title="Ocultar selección">Limpiar</button><button id="systemsClose" type="button" class="drawer-close">×</button></div></div>
  <div id="systemGroups" class="groups-scroll"></div>
</aside>

<aside id="dataDrawer" class="drawer side-drawer" aria-hidden="true"><div class="drawer-head"><div><strong>Datos</strong><small>Parámetros y entradas del modelo</small></div><button class="drawer-close" data-close="dataDrawer">×</button></div><div id="dataContent" class="drawer-content"></div></aside>
<aside id="statsDrawer" class="drawer side-drawer" aria-hidden="true"><div class="drawer-head"><div><strong>Estadísticas</strong><small>Resultados de la optimización</small></div><button class="drawer-close" data-close="statsDrawer">×</button></div><div id="statsContent" class="drawer-content"></div></aside>

<div id="routeBubble" class="route-bubble" aria-hidden="true"><button id="bubbleClose" type="button" class="bubble-close">×</button><div class="bubble-content"></div></div>

<div class="line-legend"><strong>Transporte real</strong><div><i class="dash-lng" style="--c:#7C3AED"></i>BioGNL</div><div><i class="dash-gnc" style="--c:#F97316"></i>BioGNC</div><div><i style="--c:#16A34A"></i>GasBC · gasoducto corto</div><div><i style="--c:#059669"></i>Biometano Gaseoso · vehículo</div><div><i class="dash-pipe" style="--c:#38BDF8"></i>Gasoducto Convencional</div><div><i class="dash-pipe" style="--c:#1D4ED8"></i>BioGás Crudo</div></div>
<div class="icon-legend"><strong>Tipos de nodo</strong><div class="icon-grid">
<span class="legend-node"><b style="--c:#2563EB"><svg viewBox="0 0 24 24"><path d="M3 21V9l6 3V8l6 3V4h6v17H3Z"/><path d="M7 17h2m3 0h2m3 0h2"/></svg></b>Empresa</span>
<span class="legend-node"><b style="--c:#DC2626"><svg viewBox="0 0 24 24"><path d="M5 8h14l-1 13H6L5 8Z"/><path d="M8 8a4 4 0 0 1 8 0"/></svg></b>Cliente</span>
<span class="legend-node"><b style="--c:#16A34A"><svg viewBox="0 0 24 24"><path d="M4 21V10h6v11M10 14h5v7m0-11h5v11"/><path d="M6 7c4 0 6-2 7-5 2 4 1 7-3 8"/></svg></b>Productora</span>
<span class="legend-node"><b style="--c:#0284C7"><svg viewBox="0 0 24 24"><path d="M3 8h6v8H3zM15 8h6v8h-6zM9 12h6"/><path d="M6 5v3m12-3v3M6 16v3m12-3v3"/></svg></b>Gasoducto</span>
</div></div>

<script>
const P=__PAYLOAD__;
const map=L.map('map',{zoomControl:false,preferCanvas:true});
L.control.zoom({position:'bottomright'}).addTo(map);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{maxZoom:19,subdomains:'abcd',attribution:'&copy; OpenStreetMap contributors &copy; CARTO'}).addTo(map);
map.createPane('routeNumberPane');
map.getPane('routeNumberPane').style.zIndex=640;
map.getPane('routeNumberPane').style.pointerEvents='auto';
const routeNumberMarkerOptions={pane:'routeNumberPane',zIndexOffset:100000,riseOnHover:true};

const routeLayers={},routeBounds={},tourLayers={},tourBounds={},nodeBounds=L.latLngBounds([]),state={active:new Set(),detail:null,expanded:new Set(),systemExpanded:new Set(),flowOpen:false,systemOpen:false,detailSystem:null,visibility:{companies:true,clients:true,producerSites:false,sites:true,shortPipelines:true,conventionalPipelines:true}};
const renderedPanels={data:false,stats:false};
document.getElementById('dataContent').innerHTML=P.dataHtml;
document.getElementById('statsContent').innerHTML=P.statsHtml;

function esc(v){return String(v==null?'':v).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]))}
function num(v,d=0){return Number(v||0).toLocaleString('es-US',{minimumFractionDigits:d,maximumFractionDigits:d})}
function usd(v){return 'US$ '+Number(v||0).toLocaleString('es-US',{minimumFractionDigits:2,maximumFractionDigits:2})}
function decodePath(str){let i=0,lat=0,lng=0,path=[];while(i<str.length){let b,shift=0,result=0;do{if(i>=str.length)throw new Error('Polilínea truncada');b=str.charCodeAt(i++)-63;result|=(b&31)<<shift;shift+=5}while(b>=32);lat+=(result&1)?~(result>>1):(result>>1);shift=0;result=0;do{if(i>=str.length)throw new Error('Polilínea truncada');b=str.charCodeAt(i++)-63;result|=(b&31)<<shift;shift+=5}while(b>=32);lng+=(result&1)?~(result>>1):(result>>1);path.push([lat/1e5,lng/1e5])}return path}
const iconSvg={
 company:'<svg viewBox="0 0 24 24"><path d="M3 21V9l6 3V8l6 3V4h6v17H3Z"/><path d="M7 17h2m3 0h2m3 0h2"/></svg>',
 client:'<svg viewBox="0 0 24 24"><path d="M5 8h14l-1 13H6L5 8Z"/><path d="M8 8a4 4 0 0 1 8 0"/></svg>',
 productor:'<svg viewBox="0 0 24 24"><path d="M4 21V10h6v11M10 14h5v7m0-11h5v11"/><path d="M6 7c4 0 6-2 7-5 2 4 1 7-3 8"/></svg>',
 pipeline:'<svg viewBox="0 0 24 24"><path d="M3 8h6v8H3zM15 8h6v8h-6zM9 12h6"/><path d="M6 5v3m12-3v3M6 16v3m12-3v3"/></svg>'
};
function nodeIcon(kind,color){const anchor=kind==='productor'?[19,52]:[19,39];return L.divIcon({className:'node-div-icon',html:'<div class="node-pin '+kind+'" style="--node:'+color+'">'+iconSvg[kind]+'</div>',iconSize:[38,43],iconAnchor:anchor,tooltipAnchor:[0,-34]})}
const markerByKey={},allNodeKeys=[],pipeById={},siteById=Object.fromEntries((P.sites||[]).map(x=>[String(x.id),x])),clientById=Object.fromEntries((P.clients||[]).map(x=>[String(x.id),x]));
const NODE_OVERLAP_TOL=1e-7;
function sameNodeCoordinate(lat1,lon1,lat2,lon2){return Math.abs(Number(lat1)-Number(lat2))<=NODE_OVERLAP_TOL&&Math.abs(Number(lon1)-Number(lon2))<=NODE_OVERLAP_TOL}
function siteOverlapsCompanyClientOrPipeline(site){
  if((P.companies||[]).some(x=>sameNodeCoordinate(site.lat,site.lon,x.lat,x.lon)))return true;
  if((P.clients||[]).some(x=>sameNodeCoordinate(site.lat,site.lon,x.lat,x.lon)))return true;
  return (P.pipelines||[]).some(x=>sameNodeCoordinate(site.lat,site.lon,x.punto_a_lat,x.punto_a_lon)||sameNodeCoordinate(site.lat,site.lon,x.punto_b_lat,x.punto_b_lon));
}
const overlappingCandidateSiteKeys=new Set((P.sites||[]).filter(siteOverlapsCompanyClientOrPipeline).map(x=>'site|'+String(x.id)));
function siteHasActiveProducer(key){const siteId=String(key||'').replace(/^site\|/,'');const site=siteById[siteId];return !!site&&String(site.tipos_planta_activos_texto||'').trim()!==''}
function markerCategoryForKey(key){key=String(key||'');if(key.startsWith('company|'))return 'companies';if(key.startsWith('client|'))return 'clients';if(key.startsWith('site|'))return siteHasActiveProducer(key)?'producerSites':'sites';if(key.startsWith('pipeA|')||key.startsWith('pipeB|'))return 'conventionalPipelines';return null}
function registerMarker(key,lat,lon,kind,color,title,detail){const m=L.marker([lat,lon],{icon:nodeIcon(kind,color),riseOnHover:true});m._visibilityCategory=markerCategoryForKey(key);m._baseTooltipHtml='<b>'+esc(title)+'</b><br>'+detail;m._nodeKind=kind;m._baseLatLng=L.latLng(Number(lat),Number(lon));m.bindTooltip(m._baseTooltipHtml,{direction:'top',sticky:true});markerByKey[key]=m;allNodeKeys.push(key);nodeBounds.extend([lat,lon]);return m}
function routeNodeTooltipHtml(key){
  const r=P.routes.find(x=>String(x.route_id)===String(state.detail));
  if(!r)return null;
  const entries=(r.node_interactions&&r.node_interactions[String(key)])||[];
  if(!entries.length)return null;
  let title='';
  if(String(key).startsWith('company|'))title='<b>Empresa · '+esc(r.empresa_id)+' · '+esc(r.empresa)+'</b>';
  else if(String(key).startsWith('client|')){const cid=String(key).split('|')[1]||'',c=clientById[cid]||{},principal=cid===String(r.cliente_id);title='<b>'+(principal?'Cliente principal':'Cliente compartido')+' · '+esc(cid)+' · '+esc(c.nombre||r.cliente||'')+'</b>';}
  else if(String(key).startsWith('pipeA|')||String(key).startsWith('pipeB|')){const parts=String(key).split('|'),g=pipeById[parts[1]],end=String(key).startsWith('pipeA|')?'A':'B';title='<b>Gasoducto convencional · '+esc(parts[1])+' · punto '+end+'</b>'+(g?'<br><small>'+esc(g.nombre||'')+'</small>':'')}
  else title='<b>Productora / ubicación · '+esc(String(key).replace(/^site\|/,''))+'</b>';
  const rows=entries.map(e=>{
    const qty=num(e.quantity_nm3||0,3)+' Nm³';
    if(e.kind==='plant')return '<div style="margin-top:6px"><b>'+esc(e.plant_type||'Productora')+'</b><br>'+esc(e.note||'Cantidad que ingresó')+': <b>'+qty+'</b><br>Producto: <b>'+esc(e.product||'')+'</b></div>';
    if(e.kind==='company')return '<div style="margin-top:6px">'+esc(e.note||'Biogás crudo extraído')+': <b>'+qty+'</b></div>';
    if(e.kind==='client')return '<div style="margin-top:6px">'+esc(e.note||'Biometano gaseoso entregado')+': <b>'+qty+'</b></div>';
    if(e.kind==='pipeline')return '<div style="margin-top:6px">'+esc(e.note||'Gasoducto utilizado')+': <b>'+qty+'</b><br>Producto: <b>'+esc(e.product||'')+'</b></div>';
    return '<div style="margin-top:6px">'+esc(e.note||'Interacción')+': <b>'+qty+'</b><br>Producto: <b>'+esc(e.product||'')+'</b></div>';
  }).join('');
  return title+'<br><small>Ruta '+esc(r.route_id)+'</small>'+rows;
}
function refreshMarkerTooltips(){Object.keys(markerByKey).forEach(key=>{const m=markerByKey[key];const html=routeNodeTooltipHtml(key)||m._baseTooltipHtml;if(m.getTooltip())m.setTooltipContent(html)})}

P.companies.forEach(x=>registerMarker('company|'+x.id,x.lat,x.lon,'company','#2563EB',x.id+' · '+x.nombre,'Empresa · Capacidad '+num(x.capacidad_max_biogas_nm3,0)+' Nm³'));
P.clients.forEach(x=>registerMarker('client|'+x.id,x.lat,x.lon,'client','#DC2626',x.id+' · '+x.nombre,'Cliente · Demanda '+num(x.demanda_max_nm3,0)+' Nm³'));
P.sites.forEach(x=>{const active=String(x.tipos_planta_activos_texto||'').trim();const clase=String(x.clase_visual||x.clase_nodo||'Nodo candidato').replaceAll('_',' ');const ref=String(x.referencia_local||'').trim();const permit=String(x.tecnologias_permitidas||'').trim();const visualName=String(x.nombre_visual||x.nombre||x.id);let detail=(active?'Plantas activas: '+esc(active):'Ubicaci\u00f3n posible sin planta activa')+'<br>Clase: '+esc(clase);if(ref)detail+='<br>Referencia local: '+esc(ref);if(permit)detail+='<br>Tecnolog\u00edas permitidas: '+esc(permit);registerMarker('site|'+x.id,x.lat,x.lon,'productor',active?'#16A34A':'#64748B',x.id+' ? '+visualName,detail)});
P.pipelines.forEach(x=>{pipeById[x.id]=x;const detail=esc(x.nombre)+' · '+num(x.distancia_red_estimada_km,0)+' km · '+esc(x.punto_a_zona)+' ↔ '+esc(x.punto_b_zona);registerMarker('pipeA|'+x.id,x.punto_a_lat,x.punto_a_lon,'pipeline','#0284C7',x.punto_a_nombre,detail);registerMarker('pipeB|'+x.id,x.punto_b_lat,x.punto_b_lon,'pipeline','#0369A1',x.punto_b_nombre,detail)});
// Los gasoductos cortos no crean iconos en el punto medio. Se representan
// exclusivamente mediante sus polilíneas de extremo a extremo.

const dashBySegment={Biogas:'9 7',GasBC:null,BiometanoGaseoso:'7 5',BioGNL:'15 8',BioGNC:'3 8',Gasoducto:'14 7 3 7',GasoductoBiogas:'10 5 2 5'};
function layerCategoryForKey(key){key=String(key||'');if(key.startsWith('PIPE|')||key.startsWith('PIPEINTRA|')||key.startsWith('PIPEPAIR|'))return 'conventionalPipelines';if(key.startsWith('SHORTBIO|')||key.startsWith('SHORTBIOPHUB|')||key.startsWith('SHORTBIOHUBEMP|')||key.startsWith('SHORTBIOC2C|')||key.startsWith('SHORTGASIN|')||key.startsWith('SHORTGAS|')||key.startsWith('SHORTGASDIRECT|'))return 'shortPipelines';return null}
const tourById=Object.fromEntries((P.transportTours||[]).map(t=>[String(t.transport_route_id),t]));
const tourRoutesById={};
P.routes.forEach(r=>(r.transport_tour_ids||[]).map(String).forEach(tourId=>{
  (tourRoutesById[tourId]=tourRoutesById[tourId]||[]).push(String(r.route_id));
}));
Object.keys(tourRoutesById).forEach(tourId=>{
  tourRoutesById[tourId]=Array.from(new Set(tourRoutesById[tourId]));
});

function segmentCoordinates(seg){
  const key=String(seg.key||'');let coords;
  // Compatibilidad defensiva: las exportaciones antiguas podían dejar SHORTGAS
  // accidentalmente en P[key] en vez de P.straightLines[key].
  const straight=((P.straightLines||{})[key]||P[key]);
  if(straight){
    coords=straight.map(c=>[Number(c[0]),Number(c[1])]);
  }else if(key.startsWith('PIPE|')){
    const g=pipeById[key.split('|')[1]];
    if(!g)return null;
    coords=[[g.punto_a_lat,g.punto_a_lon],[g.punto_b_lat,g.punto_b_lon]];
  }else{
    const encoded=P.polylines[key];
    if(!encoded)return null;
    coords=decodePath(encoded);
  }
  return seg.reverse?coords.slice().reverse():coords;
}

// Una gira física se construye una sola vez, aunque varias rutas económicas la
// referencien. Al seleccionar rutas se dibuja la unión de sus IDs de gira.
(P.transportTours||[]).forEach(tour=>{
  const tourId=String(tour.transport_route_id);
  if(tourLayers[tourId])return;
  const layers=[],bounds=L.latLngBounds([]),relatedRoutes=tourRoutesById[tourId]||[];
  let originPlaced=false;
  (tour.segments||[]).forEach((seg,idx)=>{
    const coords=segmentCoordinates(seg);if(!coords||!coords.length)return;
    const key=String(seg.key||''),segType=String(seg.segment_type||'');
    const color=seg.color||((P.segmentColors||{})[segType])||'#334155';
    const opts={color,weight:(key.startsWith('PIPE|')||key.startsWith('PIPEINTRA|')||key.startsWith('PIPEPAIR|')||key.startsWith('SHORTBIO|')||key.startsWith('SHORTBIOPHUB|')||key.startsWith('SHORTBIOHUBEMP|')||key.startsWith('SHORTBIOC2C|')||key.startsWith('SHORTGASIN|')||key.startsWith('SHORTGAS|'))?8:6,opacity:.95,lineCap:'round',lineJoin:'round',dashArray:dashBySegment[segType]||null};
    const line=L.polyline(coords,opts);line._visibilityCategory=layerCategoryForKey(key);
    const routeText=relatedRoutes.length?relatedRoutes.join(', '):'sin ruta económica asociada';
    line.bindTooltip('<b>Gira física '+esc(tourId)+'</b><br>Rutas económicas asociadas: <b>'+esc(routeText)+'</b><br>'+esc(seg.segment_label||segType)+'<br>Carga en el tramo: <b>'+num(seg.load_nm3||0,3)+'</b><br>'+esc(tour.sequence||'')+'<br><small>Polilínea: '+esc(seg.polyline_source_key||tour.polyline_source||'ZIP')+'</small>',{sticky:true});
    layers.push(line);coords.forEach(c=>bounds.extend(c));
    if(!originPlaced){
      const mk=L.marker(coords[0],Object.assign({},routeNumberMarkerOptions,{icon:L.divIcon({className:'tour-stop-div',html:'<div class="tour-stop-badge origin" style="--tour-color:'+esc(color)+'">O</div>',iconSize:[27,27],iconAnchor:[13,13]})}));
      mk.bindTooltip('Origen · '+esc(tour.origin_name||tour.origin_code));layers.push(mk);originPlaced=true;
    }
    if(idx<(tour.segments||[]).length-1){
      const stopNo=idx+1,decision=(tour.stop_decisions||[])[idx]||{},stopId=(tour.stops||[])[idx]||seg.to||'';
      const stopSite=siteById[String(stopId)]||null,stopClient=String(stopId).startsWith('NC_')?clientById[String(stopId).slice(3)]:null;
      const stopCoord=stopSite&&stopSite.lat!=null&&stopSite.lon!=null?[Number(stopSite.lat),Number(stopSite.lon)]:(stopClient&&stopClient.lat!=null&&stopClient.lon!=null?[Number(stopClient.lat),Number(stopClient.lon)]:coords[coords.length-1]);
      const mk=L.marker(stopCoord,Object.assign({},routeNumberMarkerOptions,{icon:L.divIcon({className:'tour-stop-div',html:'<div class="tour-stop-badge" style="--tour-color:'+esc(color)+'">'+stopNo+'</div>',iconSize:[27,27],iconAnchor:[13,13]})}));
      mk.bindTooltip('Parada '+stopNo+' · '+esc(stopId)+'<br>Llega: <b>'+num(decision.incoming_load||0,3)+'</b><br>Transforma/entrega: <b>'+num(decision.processed_load||0,3)+'</b><br>Remanente: <b>'+num(decision.remaining_load||0,3)+'</b> → '+esc(decision.remaining_destination||''));
      layers.push(mk);
    }
  });
  tourLayers[tourId]=layers;tourBounds[tourId]=bounds;
});

// Cada ruta visual contiene el trayecto físico cargado de la ruta. Si el viaje
// real atiende a varios clientes, se agrega el tramo positivo que continúa al
// segundo cliente para que ambos iconos queden explicados por la trayectoria.
//
// IMPORTANTE: los círculos 0,1,2,... pertenecen a la RUTA FÍSICA COMPLETA.
// Los extremos/referencias de gasoductos convencionales vuelven a formar parte
// de la enumeración. Si dos pasos lógicos comparten coordenadas (p. ej. E7 y C8
// en Planta Cervecera), el badge muestra ambos números: "1/4".
function sameRoutePoint(a,b){return !!a&&!!b&&Math.abs(Number(a[0])-Number(b[0]))<=1e-7&&Math.abs(Number(a[1])-Number(b[1]))<=1e-7}
function pushRouteWaypoint(waypoints,coord,color,label){
  if(!coord||coord.length<2)return;
  const p=[Number(coord[0]),Number(coord[1])];
  const last=waypoints.length?waypoints[waypoints.length-1]:null;
  if(last&&sameRoutePoint(last.coord,p))return;
  waypoints.push({coord:p,color:color||'#334155',label:String(label||'')});
}
function routeDropEventsAt(r,coord){
  return (r.intermediate_drop_events||[]).filter(ev=>{
    if(ev.lat==null||ev.lon==null)return false;
    return sameRoutePoint(coord,[Number(ev.lat),Number(ev.lon)]);
  });
}
function routeDropTooltipHtml(r,coord){
  const events=routeDropEventsAt(r,coord);
  if(!events.length)return '';
  return events.map(ev=>{
    const clients=(ev.clientes_entregas_gasbc||[]).map(c=>'Cliente '+esc(c.cliente_id||'')+': <b>'+num(c.cantidad_gasbc_nm3||0,3)+' Nm³ GasBC</b>').join('<br>');
    return '<div style="margin-top:6px;padding-top:6px;border-top:1px solid #cbd5e1">'
      +'<b>Descarga intermedia compartida</b><br>'
      +'Ubicación productora: '+esc(ev.site_id||'')+'<br>'
      +'Biogás crudo dejado aquí: <b>'+num(ev.cantidad_dejada_nm3||0,3)+' Nm³</b><br>'
      +'GasBC producido/entregado localmente: <b>'+num(ev.gasbc_vendido_local_nm3||0,3)+' Nm³</b>'
      +(clients?'<br>'+clients:'')
      +'</div>';
  }).join('');
}
function groupedRouteWaypoints(waypoints){
  const groups=[];
  waypoints.forEach((wp,idx)=>{
    let g=groups.find(x=>sameRoutePoint(x.coord,wp.coord));
    if(!g){g={coord:wp.coord,color:wp.color,items:[]};groups.push(g)}
    g.items.push({number:idx,label:wp.label,color:wp.color});
  });
  return groups;
}
P.routes.forEach(r=>{
  const routeId=String(r.route_id),layers=[],bounds=L.latLngBounds([]),waypoints=[];
  (r.segment_defs||[]).forEach((seg,segIdx)=>{
    const key=String(seg.key||'');
    const coords=segmentCoordinates(seg);if(!coords||!coords.length)return;
    const segType=String(seg.segment_type||'');
    const color=seg.color||((P.segmentColors||{})[segType])||r.color||'#334155';
    const line=L.polyline(coords,{color,weight:(key.startsWith('PIPE|')||key.startsWith('PIPEINTRA|')||key.startsWith('PIPEPAIR|')||key.startsWith('SHORTBIO|')||key.startsWith('SHORTBIOPHUB|')||key.startsWith('SHORTBIOHUBEMP|')||key.startsWith('SHORTBIOC2C|')||key.startsWith('SHORTGASIN|')||key.startsWith('SHORTGAS|')||key.startsWith('SHORTGASDIRECT|'))?8:6,opacity:.95,lineCap:'round',lineJoin:'round',dashArray:dashBySegment[segType]||null});line._visibilityCategory=layerCategoryForKey(key);
    const isShortPipeline=key.startsWith('SHORTBIO|')||key.startsWith('SHORTBIOPHUB|')||key.startsWith('SHORTBIOHUBEMP|')||key.startsWith('SHORTBIOC2C|')||key.startsWith('SHORTGASIN|')||key.startsWith('SHORTGAS|')||key.startsWith('SHORTGASDIRECT|');
    if(isShortPipeline){
      const product=key.startsWith('SHORTBIO')?'Biogás Crudo':'Biometano Gaseoso';
      line.bindTooltip('<b>Gasoducto corto · '+esc(r.route_id)+'</b><br>Producto: <b>'+esc(product)+'</b><br>Distancia: <b>'+num(seg.distance_km||0,2)+' km</b><br>Cantidad transportada en esta ruta: <b>'+num(seg.quantity_nm3||0,3)+' Nm³</b>',{sticky:true});
    }else if(key.startsWith('PIPEINTRA|')){
      line.bindTooltip('<b>Gasoducto Convencional · '+esc(r.route_id)+'</b><br>'+esc(seg.segment_label||segType)+'<br>Distancia interna A/B: <b>'+num(seg.distance_km||0,2)+' km</b><br>Cantidad transportada en esta ruta: <b>'+num(seg.quantity_nm3||0,3)+' Nm³</b>',{sticky:true});
    }else if(key.startsWith('PIPEPAIR|')){
      line.bindTooltip('<b>Gasoducto Convencional · '+esc(r.route_id)+'</b><br>'+esc(seg.segment_label||segType)+'<br>Distancia física entre gasoductos: <b>'+num(seg.distance_km||0,2)+' km</b><br>Cantidad transportada en esta ruta: <b>'+num(seg.quantity_nm3||0,3)+' Nm³</b>',{sticky:true});
    }else{
      line.bindTooltip('<b>'+esc(r.route_id)+' · '+esc(r.familia_corta)+'</b><br>'+esc(seg.segment_label||segType)+'<br>'+esc(r.proceso)+'<br><small>Tramo físico de esta ruta hasta su destino</small>',{sticky:true});
    }
    layers.push(line);coords.forEach(c=>bounds.extend(c));

    // Todos los tramos físicos cuentan en la numeración, incluidos PIPEPAIR.
    // Si dos tramos consecutivos comparten extremo, el punto se numera una vez.
    pushRouteWaypoint(waypoints,coords[0],color,seg.segment_label||segType);
    pushRouteWaypoint(waypoints,coords[coords.length-1],color,seg.segment_label||segType);
  });
  groupedRouteWaypoints(waypoints).forEach(group=>{
    const nums=group.items.map(x=>x.number),badgeText=nums.join('/');
    const first=nums[0]===0,labels=Array.from(new Set(group.items.map(x=>x.label).filter(Boolean)));
    const mk=L.marker(group.coord,Object.assign({},routeNumberMarkerOptions,{icon:L.divIcon({className:'tour-stop-div',html:'<div class="tour-stop-badge'+(first?' origin':'')+'" style="--tour-color:'+esc(group.color)+'">'+esc(badgeText)+'</div>',iconSize:[Math.max(27,18+badgeText.length*7),27],iconAnchor:[13,13]})}));
    const pointTitle=nums.length>1?'Puntos '+nums.join(' y '):'Punto '+nums[0];
    mk.bindTooltip('<b>'+esc(pointTitle)+'</b> · '+esc(r.route_id)+(labels.length?'<br>'+labels.map(esc).join('<br>'):'')+routeDropTooltipHtml(r,group.coord));
    layers.push(mk);bounds.extend(group.coord);
  });
  routeLayers[routeId]=layers;routeBounds[routeId]=bounds;
});

const VISIBLE_INTERACTION_TOL=Math.max(1e-9,Number(P.visibleFlowTolerance||0));
function addVisibleNodeKey(visible,key){
  key=String(key||'').trim();if(!key||!markerByKey[key])return;
  visible.add(key);
  if(key.startsWith('site|')){
    const siteId=key.slice(5),site=siteById[siteId];if(!site)return;
    const ref=String(site.referencia_local||'').trim(),clase=String(site.clase_nodo||'');
    // Nunca convertir automáticamente una ubicación intermedia en un segundo
    // icono de cliente. El cliente final ya viene explícitamente en node_keys.
    if(ref&&clase.includes('EMPRESA')&&markerByKey['company|'+ref])visible.add('company|'+ref);
  }
}
function addInteractionNodeKey(visible,rawCode){
  const code=String(rawCode||'').trim();if(!code)return;
  let candidates=[];
  if(code.startsWith('SITE:'))candidates=['site|'+code.slice(5)];
  else if(code.startsWith('CLIENT:'))candidates=['client|'+code.slice(7)];
  else if(code.startsWith('COMPANY:'))candidates=['company|'+code.slice(8)];
  else if(code.startsWith('PIPE:')){
    const direction=code.slice(5),gasId=direction.replace(/_(AB|BA)$/,'');
    candidates=['pipeA|'+gasId,'pipeB|'+gasId];
  }else candidates=['site|'+code,'client|'+code,'company|'+code];
  candidates.forEach(k=>addVisibleNodeKey(visible,k));
}
function addTourInteractionNodes(visible,tour){
  if(!tour)return;
  if(Number(tour.origin_load||0)>VISIBLE_INTERACTION_TOL)addInteractionNodeKey(visible,tour.origin_code);
  (tour.stop_decisions||[]).forEach(decision=>{
    // Una parada puramente de paso (procesado/descarga = 0) NO agrega icono.
    if(Number(decision.processed_load||0)<=VISIBLE_INTERACTION_TOL)return;
    addInteractionNodeKey(visible,decision.stop_id);
  });
}
function routeUsesProducerSite(route,key){
  key=String(key||'');
  if(!key.startsWith('site|'))return true;
  const entries=(route&&route.node_interactions&&route.node_interactions[key])||[];
  return entries.some(e=>{
    const kind=String(e.kind||'');
    return (kind==='plant'||kind==='shared_drop')&&Number(e.quantity_nm3||0)>VISIBLE_INTERACTION_TOL;
  });
}
function currentVisibleNodeKeys(){
  if(!state.active.size){
    const all=new Set();
    allNodeKeys.forEach(k=>{
      const key=String(k);
      // Una ubicación meramente posible no se dibuja encima de una empresa,
      // cliente o extremo de gasoducto. Si allí existe una productora activa,
      // el icono se conserva porque ya representa un activo real de la solución.
      if(key.startsWith('site|')&&overlappingCandidateSiteKeys.has(key)&&!siteHasActiveProducer(key))return;
      addVisibleNodeKey(all,key);
    });
    return all;
  }
  const visible=new Set();
  state.active.forEach(id=>{
    const route=P.routes.find(r=>String(r.route_id)===String(id));
    if(!route)return;
    (route.node_keys||[]).forEach(k=>{
      const key=String(k);
      // Al seleccionar una ruta, los nodos site|... sólo aparecen si esa ruta
      // realmente usa/interactúa con una productora en ese sitio. Se eliminan
      // los iconos de ubicaciones candidatas y simples extremos físicos.
      if(key.startsWith('site|')&&!routeUsesProducerSite(route,key))return;
      addVisibleNodeKey(visible,key);
    });
  });
  return visible;
}
function visibilityEnabled(category){return category?state.visibility[category]!==false:true}
function spreadVisibleNodeMarkers(visible){
  // Sólo desplaza ICONOS en pantalla; nunca cambia geometría, distancias ni rutas.
  const groups={};
  allNodeKeys.forEach(key=>{const m=markerByKey[key];if(m&&m._baseLatLng)m.setLatLng(m._baseLatLng)});
  if(!state.active.size)return;
  visible.forEach(key=>{
    const m=markerByKey[String(key)];if(!m||!m._baseLatLng||!visibilityEnabled(m._visibilityCategory))return;
    const ll=m._baseLatLng,k=ll.lat.toFixed(7)+'|'+ll.lng.toFixed(7);(groups[k]=groups[k]||[]).push(m);
  });
  Object.values(groups).forEach(arr=>{
    if(arr.length<2)return;
    const center=map.latLngToLayerPoint(arr[0]._baseLatLng),radius=22;
    arr.forEach((m,i)=>{const a=-Math.PI/2+(2*Math.PI*i/arr.length),pt=L.point(center.x+radius*Math.cos(a),center.y+radius*Math.sin(a));m.setLatLng(map.layerPointToLatLng(pt))});
  });
}
function updateNodeVisibility(){
  const visible=currentVisibleNodeKeys();
  allNodeKeys.forEach(key=>{
    const m=markerByKey[key];if(!m)return;
    const allowed=visible.has(String(key))&&visibilityEnabled(m._visibilityCategory);
    if(allowed){if(!map.hasLayer(m))m.addTo(map)}else if(map.hasLayer(m)){map.removeLayer(m)}
  });
  spreadVisibleNodeMarkers(visible);
}
map.on('zoomend',()=>{if(state.active.size)spreadVisibleNodeMarkers(currentVisibleNodeKeys())});

function transportPackHtml(r){const packs=Array.isArray(r.transport_packs)?r.transport_packs:[];if(!packs.length)return esc(r.familia_corta||'');return '<span class="route-pack-chain">'+packs.map((p,i)=>(i?'<span class="route-pack-arrow">→</span>':'')+'<span class="route-pack" style="--pack:'+esc(p.color||'#334155')+'">'+esc(p.label||'')+'</span>').join('')+'</span>'}
function groupRoutes(){const groups={};P.routes.forEach(r=>{const k=String(r.panel_cliente_id||r.cliente_id);(groups[k]=groups[k]||[]).push(r)});return groups}
function routeClientSummary(r){const ds=Array.isArray(r.clientes_compartidos)?r.clientes_compartidos:[];if(ds.length>1)return ds.map(d=>esc(d.cliente_id||'')+': '+num(d.cantidad_gasbc_nm3||0,0)).join(' · ');return esc(r.cliente_id||'')}
function renderGroups(){const groups=groupRoutes(),box=document.getElementById('groups');box.innerHTML='';Object.keys(groups).sort((a,b)=>a.localeCompare(b,undefined,{numeric:true})).forEach(client=>{const routes=groups[client].sort((a,b)=>(a.familia+a.empresa+a.route_id).localeCompare(b.familia+b.empresa+b.route_id));const section=document.createElement('section');section.className='client-group'+(state.expanded.has(client)?' expanded':'');section.dataset.client=client;const name=routes[0]&&routes[0].panel_cliente?routes[0].panel_cliente:(routes[0]&&routes[0].cliente?routes[0].cliente:client);section.innerHTML='<button type="button" class="client-toggle" data-client="'+esc(client)+'" aria-expanded="'+(state.expanded.has(client)?'true':'false')+'"><span class="client-copy"><strong>'+esc(client)+' · '+esc(name)+'</strong><small>'+routes.length+' ruta(s)</small></span><span class="client-badge">'+routes.length+'</span><span class="client-chevron">⌄</span></button><div class="route-list"></div>';const list=section.querySelector('.route-list');routes.forEach(r=>{const id=String(r.route_id),active=state.active.has(id);const label=document.createElement('label');label.className='route-select'+(active?' active':'');label.style.setProperty('--family',r.color);label.innerHTML='<input class="route-check" type="checkbox" data-route="'+esc(id)+'" '+(active?'checked':'')+'><span class="route-family">'+transportPackHtml(r)+'</span><span class="route-process">'+esc(r.route_id)+' · '+esc(r.empresa_id)+' → '+routeClientSummary(r)+((r.transport_tour_ids||[]).length?' · '+esc((r.transport_tour_ids||[]).join(', ')):'')+'</span>';list.appendChild(label)});box.appendChild(section)});if(!P.routes.length)box.innerHTML='<p style="padding:14px;color:#64748b">La solución no utiliza rutas.</p>'}
function renderSystems(){const box=document.getElementById('systemGroups');box.innerHTML='';(P.systems||[]).forEach(s=>{const sid=String(s.sistema_id),expanded=state.systemExpanded.has(sid),section=document.createElement('section');section.className='client-group'+(expanded?' expanded':'');section.dataset.system=sid;const margin=Number(s.margen_operativo_total_usd||0),benefit=Number(s.beneficio_total_sistema_usd||0);section.innerHTML='<button type="button" class="client-toggle system-toggle" data-system="'+esc(sid)+'" aria-expanded="'+(expanded?'true':'false')+'"><span class="client-copy"><strong>'+esc(sid)+' · '+esc(s.nombre_sistema||'Sistema independiente')+'</strong><small>'+Number(s.numero_rutas||0)+' ruta(s) · Margen '+usd(margin)+' · Beneficio '+usd(benefit)+'</small></span><span class="client-badge">'+Number(s.numero_rutas||0)+'</span><span class="client-chevron">⌄</span></button><div class="route-list"></div>';const list=section.querySelector('.route-list');(s.route_ids||[]).forEach(id=>{const r=P.routes.find(x=>String(x.route_id)===String(id));if(!r)return;const button=document.createElement('button');button.type='button';button.className='route-select system-route-button'+(String(state.detail)===String(id)&&String(state.detailSystem)===sid?' active':'');button.style.setProperty('--family',r.color);button.dataset.system=sid;button.dataset.route=String(id);button.innerHTML='<span class="route-family">'+transportPackHtml(r)+'</span><span class="route-process">'+esc(r.route_id)+' · '+esc(r.proceso)+'</span>';list.appendChild(button)});box.appendChild(section)});if(!(P.systems||[]).length)box.innerHTML='<p style="padding:14px;color:#64748b">No hay sistemas independientes con rutas activas.</p>'}
function selectSystem(sid){sid=String(sid);const s=(P.systems||[]).find(x=>String(x.sistema_id)===sid);if(!s)return;state.active.clear();(s.route_ids||[]).forEach(id=>state.active.add(String(id)));state.detail=null;state.detailSystem=sid;state.systemExpanded.add(sid);updateMap();renderGroups();renderSystems();renderBubble();fitSelection()}
function selectSystemRoute(sid,rid){sid=String(sid);rid=String(rid);const s=(P.systems||[]).find(x=>String(x.sistema_id)===sid);if(!s)return;state.active.clear();(s.route_ids||[]).forEach(id=>state.active.add(String(id)));state.detail=rid;state.detailSystem=sid;state.systemExpanded.add(sid);updateMap();renderGroups();renderSystems();renderBubble();fitSelection()}

function activeTourIds(){const ids=new Set();state.active.forEach(routeId=>{const route=P.routes.find(r=>String(r.route_id)===String(routeId));if(route)(route.transport_tour_ids||[]).forEach(id=>ids.add(String(id)))});return ids}
function setLayerCollectionVisible(layers,on){(layers||[]).forEach(layer=>{const allowed=on&&visibilityEnabled(layer._visibilityCategory);if(allowed&&!map.hasLayer(layer)){layer.addTo(map);if(typeof layer.bringToFront==='function')layer.bringToFront()}else if(!allowed&&map.hasLayer(layer)){map.removeLayer(layer)}})}
const visibilityLabels={companies:'empresas',clients:'clientes',producerSites:'Productoras',sites:'ubicaciones posibles',shortPipelines:'gasoductos cortos',conventionalPipelines:'gasoductos convencionales'};
function updateVisibilityButtons(){document.querySelectorAll('[data-visibility]').forEach(b=>{const key=b.dataset.visibility,on=state.visibility[key]!==false,label=visibilityLabels[key]||key;b.textContent=(on?'Ocultar ':'Mostrar ')+label;b.classList.toggle('is-hidden',!on);b.setAttribute('aria-pressed',on?'false':'true')})}
function toggleVisibilityCategory(key){if(!(key in state.visibility))return;state.visibility[key]=!state.visibility[key];updateMap();updateVisibilityButtons()}
function updateMap(){const activeTours=activeTourIds();P.routes.forEach(r=>{const id=String(r.route_id);setLayerCollectionVisible(routeLayers[id],state.active.has(id))});Object.keys(tourLayers).forEach(id=>setLayerCollectionVisible(tourLayers[id],activeTours.has(String(id))));refreshMarkerTooltips();updateNodeVisibility()}
function fitSelection(){let b=L.latLngBounds([]);if(state.active.size){state.active.forEach(id=>{const rb=routeBounds[id];if(rb&&rb.isValid())b.extend(rb)});activeTourIds().forEach(id=>{const tb=tourBounds[id];if(tb&&tb.isValid())b.extend(tb)})}else if(nodeBounds.isValid())b.extend(nodeBounds);if(b.isValid())map.fitBounds(b.pad(.08),{animate:true,maxZoom:12})}
function clearSelection(){state.active.clear();state.detail=null;state.detailSystem=null;updateMap();renderGroups();renderSystems();renderBubble()}
function toggleRoute(id,checked){id=String(id);const route=P.routes.find(r=>String(r.route_id)===id);if(!route)return;state.detailSystem=null;if(checked){state.active.add(id);state.detail=id;state.expanded.add(String(route.panel_cliente_id||route.cliente_id))}else{state.active.delete(id);if(state.detail===id){const remaining=Array.from(state.active);state.detail=remaining.length?remaining[remaining.length-1]:null}}updateMap();renderGroups();renderSystems();renderBubble()}
function renderBubble(){
 const bubble=document.getElementById('routeBubble');
 const r=P.routes.find(x=>String(x.route_id)===String(state.detail));
 const s=(P.systems||[]).find(x=>String(x.sistema_id)===String(state.detailSystem));
 if(!r&&!s){bubble.classList.remove('open');bubble.setAttribute('aria-hidden','true');return}
 bubble.style.setProperty('--route-color',r?r.color:'#334155');
 let title='',chain='',totalHtml='';
 if(s){
   const margin=Number(s.margen_operativo_total_usd||0),benefit=Number(s.beneficio_total_sistema_usd||0);
   title=esc(s.sistema_id)+' · '+esc(s.nombre_sistema||'Sistema independiente');
   chain=Number(s.numero_rutas||0)+' rutas conectadas · Empresas: '+esc((s.empresas_ids||[]).join(', '))+' · Clientes: '+esc((s.clientes_ids||[]).join(', '));
   totalHtml='<div class="bubble-part"><div class="bubble-part-title">Datos Totales del Sistema Independiente</div><small class="bubble-part-note">Margen Operativo Total = Ingresos por ventas − costos por venta. El beneficio total del sistema descuenta una sola vez los costos fijos de activación de las empresas, productoras y gasoductos pertenecientes al sistema.</small><section class="metric-section"><div class="metric-row"><span>Cantidad total vendida</span><b>'+num(s.cantidad_vendida_nm3,0)+' Nm³</b></div><div class="metric-row"><span>Ingresos por ventas</span><b>'+usd(s.ingresos_ventas_usd)+'</b></div><div class="metric-row"><span>Costos por venta</span><b>'+usd(s.costos_por_venta_usd)+'</b></div><div class="metric-row"><span>Margen Operativo Total</span><b class="'+(margin>=0?'positive':'negative')+'">'+usd(margin)+'</b></div><div class="metric-row"><span>Costos fijos de empresas, una vez</span><b>'+usd(s.costos_fijos_empresas_unicos_usd)+'</b></div><div class="metric-row"><span>Costos fijos de productoras y gasoductos, una vez</span><b>'+usd(s.costos_fijos_productoras_gasoductos_unicos_usd)+'</b></div><div class="metric-row"><span>Costos fijos de activación totales</span><b>'+usd(s.costos_fijos_activacion_unicos_usd)+'</b></div><div class="metric-row"><span>Costos totales del sistema</span><b>'+usd(s.costos_totales_sistema_usd)+'</b></div><div class="metric-row"><span>Beneficio Total del Sistema</span><b class="'+(benefit>=0?'positive':'negative')+'">'+usd(benefit)+'</b></div><div class="metric-row"><span>Rutas del sistema</span><b>'+num(s.numero_rutas,0)+'</b></div><div class="metric-row"><span>Viajes totales</span><b>'+num(s.numero_viajes_total,2)+'</b></div><div class="metric-row"><span>Costo total de transporte</span><b>'+usd(s.costo_transporte_total_usd)+'</b></div><div class="metric-row"><span>Distancia vial total</span><b>'+num(s.distancia_vial_total_km,1)+' km</b></div></section></div>';
 }else{
   const t=P.clientTotals[String(r.cliente_id)]||{},margin=Number(t.margen_operativo_total||t.margen_sin_fijos||0);
   title=esc(r.route_id)+' · '+esc(r.familia_corta);
   chain=esc(r.proceso);
   totalHtml='<div class="bubble-part"><div class="bubble-part-title">Datos Totales</div><small class="bubble-part-note">Todas las rutas del cliente '+esc(r.cliente_id)+' · '+esc(r.cliente)+'. Margen Operativo Total = ingresos por ventas − costos por venta; no descuenta costos fijos de activación.</small><section class="metric-section"><div class="metric-row"><span>Cantidad demandada cliente</span><b>'+num(t.demanda,0)+' Nm³</b></div><div class="metric-row"><span>Cantidad total vendida al cliente</span><b>'+num(t.venta,0)+' Nm³</b></div><div class="metric-row"><span>Precio de venta</span><b>US$ '+num(t.precio,3)+'/Nm³</b></div><div class="metric-row"><span>Ingresos por ventas</span><b>'+usd(t.ingresos)+'</b></div><div class="metric-row"><span>Costos por venta</span><b>'+usd(t.costos_variables)+'</b></div><div class="metric-row"><span>Margen Operativo Total</span><b class="'+(margin>=0?'positive':'negative')+'">'+usd(margin)+'</b></div><div class="metric-row"><span>Distancia vial total</span><b>'+num(t.distancia_vial,1)+' km</b></div><div class="metric-row"><span>Distancia interna de gasoducto</span><b>'+num(t.distancia_gasoducto,1)+' km</b></div><div class="metric-row"><span>Costo total de transporte</span><b>'+usd(t.costo_transporte)+'</b></div></section></div>';
 }
 let routeHtml='';
 if(r){
   const routeMargin=Number(r.margen_operativo_por_ruta_usd||r.margen_ruta_antes_costos_fijos_usd||0);
   const routeTours=(r.transport_tour_ids||[]).map(id=>tourById[String(id)]).filter(Boolean);
   const tourRows=routeTours.map(t=>'<div class="metric-row"><span>'+esc(t.transport_route_id)+' · '+esc(t.vehicle)+'</span><b>'+esc(t.sequence)+' · salida '+num(t.origin_load||0,3)+' · retorno '+num(t.return_load||0,3)+' · '+num(t.distance_km,1)+' km</b></div>').join('');
   const tourHtml=routeTours.length?'<section class="metric-section"><b>Trayectoria real del vehículo desde el ZIP</b>'+tourRows+'</section>':'';
   const sharedRows=(r.clientes_compartidos||[]).map(d=>'<div class="metric-row"><span>'+esc(d.cliente_id||'')+' · '+esc(d.cliente||'')+'</span><b>'+num(d.cantidad_gasbc_nm3||0,0)+' Nm³ GasBC</b></div>').join('');
   const sharedHtml=(r.clientes_compartidos||[]).length>1?'<section class="metric-section"><b>Clientes atendidos por esta ruta</b>'+sharedRows+'</section>':'';
   routeHtml='<div class="bubble-part"><div class="bubble-part-title">Datos por Ruta</div><small class="bubble-part-note">Margen Operativo por Ruta = ingresos por venta al cliente final − costos variables atribuibles exclusivamente a esta ruta. No se restan costos fijos.</small><section class="metric-section"><div class="metric-row"><span>Ruta</span><b>'+esc(r.route_id)+'</b></div><div class="metric-row"><span>Empresa</span><b>'+esc(r.empresa_id)+' · '+esc(r.empresa)+'</b></div><div class="metric-row"><span>Cliente económico</span><b>'+esc(r.cliente_id)+' · '+esc(r.cliente)+'</b></div><div class="metric-row"><span>Biogás crudo de entrada</span><b>'+num(r.biogas_crudo_entrada_BG_nm3||0,0)+' Nm³</b></div><div class="metric-row"><span>GasBC final vendido</span><b>'+num(r.cantidad_nm3,0)+' Nm³</b></div><div class="metric-row"><span>Ingresos por venta</span><b>'+usd(r.ingreso_usd)+'</b></div><div class="metric-row"><span>Costo variable del biogás</span><b>'+usd(r.costo_empresa_variable_usd)+'</b></div><div class="metric-row"><span>Costo variable de productoras</span><b>'+usd(r.costo_productoras_variable_usd)+'</b></div><div class="metric-row"><span>Costo variable del gasoducto</span><b>'+usd(r.costo_gasoducto_variable_usd||0)+'</b></div><div class="metric-row"><span>Costo de transporte vial</span><b>'+usd(r.costo_transporte_usd)+'</b></div><div class="metric-row"><span>Costos por venta de esta ruta</span><b>'+usd(r.costo_variable_total_usd)+'</b></div><div class="metric-row"><span>Margen Operativo por Ruta</span><b class="'+(routeMargin>=0?'positive':'negative')+'">'+usd(routeMargin)+'</b></div><div class="metric-row"><span>Viajes asignados</span><b>'+num(r.numero_viajes_ida_vuelta||0,2)+'</b></div><div class="metric-row"><span>Camiones utilizados</span><b>'+num(r.numero_camiones_utilizados||0,0)+'</b></div><div class="metric-row"><span>Distancia vial</span><b>'+num(r.distancia_vial_km,1)+' km</b></div><div class="metric-row"><span>Distancia interna de gasoducto</span><b>'+num(r.distancia_interna_gasoducto_referencial_km,1)+' km</b></div></section>'+sharedHtml+tourHtml+'</div>';
   if(s) title=esc(s.sistema_id)+' · '+esc(r.route_id)+' · '+esc(r.familia_corta);
 }
 bubble.querySelector('.bubble-content').innerHTML='<h2>'+title+'</h2><div class="bubble-chain">'+chain+'</div>'+totalHtml+routeHtml;
 bubble.classList.add('open');bubble.setAttribute('aria-hidden','false')
}

function renderPanelCharts(panel){const drawer=document.getElementById(panel+'Drawer');if(!drawer||renderedPanels[panel])return;if(typeof Plotly==='undefined'){drawer.querySelectorAll('.plotly-chart').forEach(d=>d.innerHTML='<p style="padding:18px;color:#b00020">No se pudo cargar Plotly.</p>');return}drawer.querySelectorAll('.plotly-chart').forEach(div=>{const spec=P.chartSpecs[div.id];if(!spec)return;Plotly.newPlot(div,spec.data,spec.layout,spec.config||{responsive:true,displaylogo:false})});renderedPanels[panel]=true;setTimeout(()=>drawer.querySelectorAll('.plotly-chart').forEach(div=>{try{Plotly.Plots.resize(div)}catch(e){}}),120)}
function closeSideDrawers(){document.querySelectorAll('.side-drawer').forEach(x=>x.classList.remove('open'))}
function openSideDrawer(name){closeSideDrawers();const d=document.getElementById(name+'Drawer');d.classList.add('open');setTimeout(()=>renderPanelCharts(name),40)}

document.getElementById('flowButton').onclick=()=>{state.flowOpen=!state.flowOpen;state.systemOpen=false;document.getElementById('flowPanel').classList.toggle('open',state.flowOpen);document.getElementById('systemsPanel').classList.remove('open')};
document.getElementById('flowClose').onclick=()=>{state.flowOpen=false;document.getElementById('flowPanel').classList.remove('open')};
document.getElementById('systemsButton').onclick=()=>{state.systemOpen=!state.systemOpen;state.flowOpen=false;document.getElementById('systemsPanel').classList.toggle('open',state.systemOpen);document.getElementById('flowPanel').classList.remove('open')};
document.getElementById('systemsClose').onclick=()=>{state.systemOpen=false;document.getElementById('systemsPanel').classList.remove('open')};
document.getElementById('clearRoutesButton').onclick=clearSelection;
document.getElementById('clearSystemsButton').onclick=clearSelection;
document.getElementById('bubbleClose').onclick=()=>{state.detail=null;state.detailSystem=null;renderBubble()};
document.getElementById('dataButton').onclick=()=>openSideDrawer('data');document.getElementById('statsButton').onclick=()=>openSideDrawer('stats');document.getElementById('visibilityControls').addEventListener('click',ev=>{const b=ev.target.closest('[data-visibility]');if(b)toggleVisibilityCategory(b.dataset.visibility)});
document.querySelectorAll('[data-close]').forEach(b=>b.onclick=()=>document.getElementById(b.dataset.close).classList.remove('open'));
document.getElementById('groups').addEventListener('change',ev=>{const cb=ev.target.closest('.route-check');if(cb)toggleRoute(cb.dataset.route,cb.checked)});
document.getElementById('groups').addEventListener('click',ev=>{const client=ev.target.closest('.client-toggle');if(client){const id=String(client.dataset.client);state.expanded.has(id)?state.expanded.delete(id):state.expanded.add(id);renderGroups()}});
document.getElementById('systemGroups').addEventListener('click',ev=>{const routeButton=ev.target.closest('.system-route-button');if(routeButton){selectSystemRoute(routeButton.dataset.system,routeButton.dataset.route);return}const systemButton=ev.target.closest('.system-toggle');if(systemButton){const sid=String(systemButton.dataset.system);if(state.systemExpanded.has(sid)){state.systemExpanded.delete(sid);renderSystems()}else{state.systemExpanded.add(sid);selectSystem(sid)}}});
document.addEventListener('keydown',ev=>{if(ev.key==='Escape'){state.flowOpen=false;state.systemOpen=false;state.detail=null;state.detailSystem=null;document.getElementById('flowPanel').classList.remove('open');document.getElementById('systemsPanel').classList.remove('open');closeSideDrawers();renderBubble()}});

renderGroups();renderSystems();updateVisibilityButtons();updateMap();fitSelection();setTimeout(()=>map.invalidateSize(),0);
</script>
</body>
</html>"""

html_text = HTML_TEMPLATE.replace("__PAYLOAD__", escape_script_json(payload))
HTML_PATH = DIR_RESULTADOS / "Visualizador_Biometano_Modelo_PDF.html"
HTML_PATH.write_text(html_text, encoding="utf-8")
HTML_CONTENT_PATH = CONTENT / HTML_PATH.name
shutil.copy2(HTML_PATH, HTML_CONTENT_PATH)
print(HTML_PATH, f"({HTML_PATH.stat().st_size/1024/1024:.2f} MiB)")
print(f"Rutas dibujadas: {len(route_payload):,}; giras reales: {len(transport_tours_payload):,}; segmentos viales únicos usados: {len(active_road_keys):,}")
display(HTML("<b>HTML generado con las giras reales y las trayectorias precargadas del ZIP.</b> Las entregas directas desde empresas usan E2C; los tramos entre clientes usan C2C; no se reconstruyen con líneas rectas."))


Advertencia visualizador: costos atribuidos a rutas=258857.337011, costo MILP=263350.377011
Advertencia visualizador: beneficio contable atribuido a rutas=46065.375811, beneficio MILP=41572.335811
/content/Resultados_Biometano_Modelo_PDF/Visualizador_Biometano_Modelo_PDF.html (1.54 MiB)
Rutas dibujadas: 15; giras reales: 12; segmentos viales únicos usados: 23


# 11

In [ ]:
# ============================================================
# 11. EXPORTACIÓN COMPLETA DESDE LA CACHÉ LOCAL
# ============================================================
# Resultados auditables.
configuracion_modelo = {
    'cadena_gasoducto_salida_a_LC': {
        'habilitada': True,
        'descripcion': 'Gasoducto convencional -> L/C exactamente en salida -> BioGNL/BioGNC -> vehículo -> RL/RC -> GasBC -> cliente(s)',
        'sin_gasoducto_corto_intermedio': True,
        'maximo_destinos_por_vehiculo': 2,
    },
    'funcion_objetivo': {
        'formula': 'maximizar beneficio económico total: ingresos - costos',
        'beneficio_usd': float(beneficio),
        'sin_bonificacion_diversificacion_en_objetivo': True,
        'igualdades_porcentuales_son_restricciones_estructurales': True,
        'costos_fijos_gasoductos_cortos_en_objetivo': True,
        'costo_fijo_gasoducto_biogas_corto_usd_dia': float(COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA),
        'costo_fijo_gasoducto_gasbc_corto_usd_dia': float(COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA),
    },
    'solver_profesional': {
        'nombre': 'Gurobi Optimizer',
        'api': 'gurobipy matrix API',
        'licencia_modo': str(GUROBI_LICENSE_MODE),
        'algoritmo': 'MILP nativo con presolve, heuristicas, cortes y branch-and-cut',
        'una_sola_llamada_optimize': True,
        'time_limit_segundos': float(TIME_LIMIT_SECONDS),
        'mip_gap_relativo_objetivo': float(MIP_REL_GAP),
        'threads': int(SOLVER_OPTIONS.get('Threads',0)),
        'mip_focus': int(SOLVER_OPTIONS.get('MIPFocus',0)),
        'status': int(resultado.status),
        'mensaje': str(resultado.message),
        'mip_gap_final': getattr(resultado,'mip_gap',None),
        'nodos_explorados': getattr(resultado,'mip_node_count',None),
        'tiempo_solver_segundos': getattr(resultado,'runtime',None),
        'soluciones_almacenadas': getattr(resultado,'sol_count',None),
        'log_incluido': bool(GUROBI_GUARDAR_LOG),
    },
    'control_numero_empresas': {
        'parametro': None if NUMERO_EMPRESAS_A_UTILIZAR is False else int(NUMERO_EMPRESAS_A_UTILIZAR),
        'modo': 'libre' if NUMERO_EMPRESAS_A_UTILIZAR is False else 'fijado_exacto_global',
        'numero_empresas_activadas_binaria': int(numero_empresas_activadas_binaria),
        'numero_empresas_con_flujo_positivo': int(N_E),
        'flujo_minimo_logico_por_empresa_activada_nm3_dia': 0.0,
        'alcance': 'global_en_toda_la_solucion_no_por_cliente',
        'atribucion_empresa_cliente_reporte': 'dispersa_best_fit_sin_cardinalidad_por_cliente',
    },
    'control_numero_clientes': {
        'parametro': None if NUMERO_CLIENTES_A_UTILIZAR is False else int(NUMERO_CLIENTES_A_UTILIZAR),
        'modo': 'libre' if NUMERO_CLIENTES_A_UTILIZAR is False else 'fijado_exacto_global',
        'numero_clientes_activados_binaria': int(numero_clientes_activados_binaria),
        'numero_clientes_con_flujo_positivo': int(N_C),
        'flujo_minimo_logico_por_cliente_activado_nm3_dia': 0.0,
        'alcance': 'global_en_toda_la_solucion_no_por_empresa_ni_ruta',
    },
    'igualdad_porcentual_compra': {
        'activada': bool(compra_porcentaje),
        'aplica_a': 'empresas_proveedoras_activadas',
        'base_porcentaje': 'capacidad_max_biogas_nm3',
        'porcentaje_comun_fraccion': porcentaje_compra_comun,
        'porcentaje_comun_pct': None if porcentaje_compra_comun is None else 100*porcentaje_compra_comun,
        'formulacion': 'MILP_linealizada_con_BE',
    },
    'igualdad_porcentual_venta': {
        'activada': bool(venta_porcentaje),
        'aplica_a': 'clientes_activados',
        'base_porcentaje': 'demanda_max_nm3',
        'porcentaje_comun_fraccion': porcentaje_venta_comun,
        'porcentaje_comun_pct': None if porcentaje_venta_comun is None else 100*porcentaje_venta_comun,
        'formulacion': 'MILP_linealizada_con_BJ',
    },
    'precios_endogenos_discretizados': {
        'columna_precio_compra_base': COL_PRECIO_BIOGAS_CRUDO,
        'columna_excedente_proveedor': COL_OFERTA_EXCEDENTE_BIOGAS,
        'columna_demanda_no_cubierta_cliente': COL_DEMANDA_NO_CUBIERTA,
        'alpha_compra_porcentaje': float(ALPHA_COMPRA_PORCENTAJE),
        'beta_venta_porcentaje': float(BETA_VENTA_PORCENTAJE),
        'precio_venta_min_usd_nm3': float(PRECIO_VENTA_MIN_USD_NM3),
        'n_tramos_precio': int(N_TRAMOS_PRECIO),
        'variables_compra': 'QBUY_SEG[i,k]',
        'variables_venta': 'QSELL_SEG[j,k]',
        'formula_compra': 'P_i(q)=P0_i*(1+alpha*max(0,q-E_i)/(Qmax_i-E_i))',
        'formula_venta': 'P_j(q)=max(P_MIN,P0_j*(1-beta*max(0,q-DNC_j)/(Dmax_j-DNC_j)))',
    },
    'definicion_margen_operativo_por_ruta': 'ingresos_por_venta_menos_costos_por_venta_sin_costos_fijos',
    'definicion_margen_operativo_total': 'ingresos_por_ventas_menos_costos_por_venta_sin_costos_fijos',
    'definicion_beneficio_sistema_independiente': 'margen_operativo_total_menos_costos_fijos_de_activacion_unicos_del_sistema',
    'prorrateo_contable_costo_fijo_empresas': {
        'costo_base': 'costo_fijo_original_por_empresa',
        'formula': 'costo_fijo_efectivo_por_flujo_biogas_crudo_ruta_dividido_flujo_total_empresa',
        'universo_denominador': 'todas_las_rutas_de_todos_los_clientes_que_usan_la_empresa',
        'no_usa': ['ingresos_por_venta','demanda_maxima_no_atendida'],
        'conservacion': 'la_suma_asignada_a_rutas_es_igual_al_costo_fijo_efectivo_activado',
        'archivo_auditoria': 'Auditoria_Prorrateo_Costos_Fijos_Empresas.csv',
    },
    'prorrateo_contable_costos_fijos_productoras': {
        'formula': 'costo_fijo_productora_por_salida_ruta_dividido_salida_total_productora',
        'universo_denominador': 'todas_las_rutas_de_todos_los_clientes_que_usan_la_productora',
        'bases_salida': {'BG':'GasBC','L':'BioGNL','RL':'GasBC','C':'BioGNC','RC':'GasBC'},
        'conservacion': 'la_suma_asignada_a_rutas_es_igual_al_costo_fijo_activado_de_cada_productora',
        'archivo_auditoria': 'Auditoria_Prorrateo_Costos_Fijos_Productoras.csv',
    },
    'gasoductos_biogas_crudo_cortos_dinamicos': {
        'habilitados': bool(HABILITAR_GASODUCTOS_BIOGAS_CORTOS),
        'tipos_destino': ['CLIENTE','UBICACION_POSIBLE','UBICACION_EMPRESA','ENTRADA_GASODUCTO'],
        'metodo_distancia': 'haversine_linea_recta_calculada_en_ipynb',
        'condicion_estricta': 'distancia_recta_km_menor_que_largo_maximo_km',
        'largo_maximo_km': float(LARGO_MAXIMO_GASODUCTO_BIOGAS_CORTO_KM),
        'costo_fijo_usd_dia': float(COSTO_FIJO_GASODUCTO_BIOGAS_CORTO_USD_DIA),
        'costo_variable_usd_nm3_km': float(COSTO_VARIABLE_GASODUCTO_BIOGAS_CORTO_USD_NM3_KM),
        'capacidad_max_nm3_dia': float(CAPACIDAD_GASODUCTO_BIOGAS_CORTO_NM3_DIA),
        'numero_candidatos_detectados_todos': int(len(gasoductos_biogas_cortos_todos)),
        'habilitar_relevo_cliente_cliente': bool(HABILITAR_RELEVO_BIOGAS_CRUDO_CLIENTE_CLIENTE),
        'habilitar_relevo_ubicaciones_P': bool(HABILITAR_RELEVO_BIOGAS_CRUDO_EN_UBICACIONES_P),
        'numero_candidatos_cliente_cliente': int(len(H_BIOC2C)),
        'numero_candidatos_empresa_cliente': int(
            (gasoductos_biogas_cortos_todos['destino_tipo']=='CLIENTE').sum()
        ) if len(gasoductos_biogas_cortos_todos) else 0,
        'numero_candidatos_empresa_ubicacion': int(
            (gasoductos_biogas_cortos_todos['destino_tipo']=='UBICACION_POSIBLE').sum()
        ) if len(gasoductos_biogas_cortos_todos) else 0,
        'numero_candidatos_empresa_ubicacion_empresa': int(
            (gasoductos_biogas_cortos_todos['destino_tipo']=='UBICACION_EMPRESA').sum()
        ) if len(gasoductos_biogas_cortos_todos) else 0,
        'numero_candidatos_empresa_entrada_gasoducto': int(
            (gasoductos_biogas_cortos_todos['destino_tipo']=='ENTRADA_GASODUCTO').sum()
        ) if len(gasoductos_biogas_cortos_todos) else 0,
        'numero_candidatos_disponibles_milp': int(len(H_CORTO)),
        'se_inyecta_en_zip_datos': False,
        'polilinea': 'dos_coordenadas_extremos_linea_recta_en_payload_HTML',
        'producto_transportado': 'biogas_crudo',
        'alternativa_GasBC_BG_empresa_a_entrada': {
            'habilitada': bool(HABILITAR_GASODUCTOS_GASBC_CORTOS_EMPRESA_ENTRADA),
            'numero_candidatos': int(len(H_GECORTO)),
            'secuencia': 'Empresa -> BG en la misma empresa -> GasBC -> gasoducto corto -> entrada del corredor',
            'producto_transportado': 'GasBC',
            'costo_fijo_usd_dia': float(COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA),
        },
        'alternativa_GasBC_BG_a_destino_directo': {
            'habilitada': bool(HABILITAR_GASODUCTOS_GASBC_CORTOS_EMPRESA_DESTINO),
            'numero_candidatos': int(len(H_GDCORTO)),
            'habilitar_reenvio_en_clientes': bool(HABILITAR_REENVIO_GASBC_EN_CLIENTES),
            'flujo_reenvio_hub_total_nm3': float(
                sum(value('QGCD_HUB',h)+value('QGCD_HUB_RL',h)+value('QGCD_HUB_RC',h) for h in H_GDCORTO)
                +sum(value('ZG_HUB',d,j) for d,j in AGC)
                +sum(value('QGCP_HUB',h) for h in H_GSCORTO)
            ),
            'reenviar_desde_salida_convencional_por_carretera_solo_transito_hub': bool(PERMITIR_VEHICULO_GASBC_SALIDA_CONVENCIONAL_A_CLIENTE_HUB),
            'reenviar_desde_salida_convencional_por_gasoducto_corto': True,
            'segundo_salto_desde_RL_RC': True,
            'secuencia_cliente': 'Una o varias empresas -> BG concentradora en cualquier nodo -> GasBC -> uno o varios gasoductos cortos -> clientes',
            'secuencia_ubicacion': 'Empresa -> BG en la misma empresa -> GasBC -> gasoducto corto directo -> ubicacion -> cadena posterior',
            'usa_gasoducto_convencional': False,
            'producto_transportado': 'GasBC',
            'costo_fijo_usd_dia': float(COSTO_FIJO_GASODUCTO_GASBC_CORTO_USD_DIA),
        },
        'proceso_cliente': 'empresas_a_BG_por_gasoducto_corto_o_transferencia_local_y_salida_GasBC_por_gasoductos',
        'proceso_ubicacion': 'empresa_a_P_por_gasoducto_BG_reparto_GasBC_sin_vehiculos',
        'restriccion_salida': 'sin_restriccion_artificial_ZD; puede_usar_GasBC_BioGNL_BioGNC_o_corredor',
    },
    'transporte_biogas_crudo_vehiculos': {
        'habilitado': False,
        'motivo': 'retirado_completamente; biogas_crudo_solo_local_o_gasoducto_corto',
    },
    'concentracion_biogas_crudo': {
        'varias_empresas_en_una_BG': True,
        'modos_entrada': ['gasoducto_corto_biogas_crudo','transferencia_local'],
        'salidas_posibles': ['GasBC','BioGNL','BioGNC','gasoducto_convencional','gasoductos_cortos_GasBC_a_clientes'],
    },
    'modo_numero_total_plantas': 'libre' if NUMERO_TOTAL_PLANTAS is False else 'fijado_exacto',
    'numero_total_plantas_solicitado': None if NUMERO_TOTAL_PLANTAS is False else int(NUMERO_TOTAL_PLANTAS),
    'exclusion_empresas_mayor_oferta': {
        'parametro': (
            None if NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR is False
            else int(NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR)
        ),
        'modo': (
            'considerar_todas' if NUMERO_EMPRESAS_MAYOR_OFERTA_A_EXCLUIR is False
            else 'excluir_mayores_ofertas'
        ),
        'columna_ranking_original': 'capacidad_max_biogas_nm3',
        'numero_empresas_originales': int(len(empresas_completas_originales)),
        'numero_empresas_disponibles_milp': int(len(empresas)),
        'numero_empresas_excluidas': int(len(empresas_excluidas_mayor_oferta)),
        'ids_empresas_excluidas': sorted(ids_empresas_excluidas_mayor_oferta),
        'desempate': 'id ascendente',
    },
    'numero_total_plantas_activas': int(round(numero_total_plantas_activas)),
    'criterio_transferencia_local': CRITERIO_TRANSFERENCIA_LOCAL,
    'relaciones_empresa_nodo_local': dict(EMPRESA_A_NODO),
    'relaciones_cliente_nodo_local': dict(CLIENTE_A_NODO),
    'tecnologias_en_nodo_cliente': list(TECNOLOGIAS_EN_NODO_CLIENTE),
    'sin_capacidad_maxima_artificial_transferencia_local': True,
    'viajes_y_camiones_en_transferencia_local': 0,
    'costo_vial_transferencia_local_usd': 0.0,
    'gasoductos_convencionales': {
        'distancias_desde_zip': True,
        'archivo_distancias_entre_gasoductos': 'distancias_entre_gasoductos_36.csv',
        'archivo_distancia_interna_AB_BA': 'gasoductos.csv/distancia_red_estimada_km',
        'recorrido_interno_AB_BA_habilitado': True,
        'variable_recorrido_interno': 'YGINTRA[d]',
        'no_consulta_ide_energia': True,
        'no_reconstruye_ni_reemplaza_distancias_gasoductos': True,
    },
    'transporte_directo_y_compartido': {
        'formulacion': 'balance explicito QSEG: en cada parada llega = transforma_o_entrega + remanente',
        'productos': ['BioGNL','BioGNC'],
        'activado_por_PERMITIR_RUTAS_COMPARTIDAS': bool(PERMITIR_RUTAS_COMPARTIDAS),
        'forzar_exactamente_dos_paradas': bool(FORZAR_EXACTAMENTE_DOS_PARADAS),
        'rutas_directas_NTR_permitidas': not bool(FORZAR_EXACTAMENTE_DOS_PARADAS),
        'viajes_directos_NIG_a_gasoducto_permitidos': not bool(FORZAR_EXACTAMENTE_DOS_PARADAS),
        'consecuencia_modo_forzado': 'los flujos AIG viales se fijan en cero; AIG local sin camion permanece permitido',
        'camiones_NTR': 'variables_enteras_no_negativas_sin_cota_superior',
        'transformaciones_entregas_QTR': 'variables_continuas no negativas; en rutas compartidas de dos paradas QTR >= descarga_minima*NTR',
        'cargas_por_tramo_QSEG': 'variables_continuas_no_negativas para salida, continuacion y retorno',
        'balance_carga': 'QSEG_entrada = QTR_parada + QSEG_salida_en_cada_parada',
        'capacidad': 'QSEG_de_cada_tramo_menor_o_igual_capacidad_por_NTR',
        'transformacion_minima_por_parada': float(DESCARGA_MINIMA_POR_PARADA_POR_VIAJE or 0.0),
        'limite_superior_artificial_camiones': False,
        'limite_superior_camiones_derivado_del_ZIP': False,
        'sin_capacidad_maxima_por_ruta': True,
        'binaria_activacion_viaje': bool(ACTIVAR_BINARIA_GIRA_TRANSPORTE),
        'maximo_paradas_por_viaje': 2,
        'maximo_paradas_es_variable_configurable': False,
        'una_orientacion_por_pareja': 'se_conserva_el_orden_de_menor_distancia',
        'retorno_obligatoriamente_vacio': not bool(PERMITIR_RETORNO_CON_REMANENTE),
        'retorno_con_remanente_permitido': bool(PERMITIR_RETORNO_CON_REMANENTE),
        'penalizacion_retorno_remanente_usd_unidad': float(PENALIZACION_RETORNO_REMANENTE_USD_UNIDAD),
        'distancias_cliente_cliente_desde_google_routes': True,
        'resolucion_solver': 'una_sola_llamada_gurobi_optimize_desde_cero',
        'submodelo_directo_previo': False,
        'mip_start_separado': False,
        'giras_gnl_gnc_dos_paradas': 'remanente_continua_entre_nodos_compatibles_y_el_ultimo_remanente_regresa_al_origen',
        'gasbc_desde_BG_hacia_L_C': 'sin_transporte_vial; solo_transferencia_local_o_alternativas_de_gasoducto',
        'cadena_gasoducto_L_C_vehiculo_RL_RC_cliente': True,
        'cadena_multicliente_GNL_GNC_RL_RC_GasBC_gasoducto_corto_cliente': True,
        'detalle_cadena_multicliente': 'L/C -> un viaje GNL/GNC con una o dos paradas -> RL/RC -> GasBC -> uno o varios gasoductos cortos -> clientes adicionales',
        'secuencia_entrada_reconvertida': 'L_o_C_en_una_o_varias_empresas_ubicaciones -> vehiculo_GNL_o_GNC -> RL_o_RC_en_entrada -> GasBC -> gasoducto_convencional',
        'permite_salida_y_nueva_conversion': 'GasBC_por_gasoducto -> L_o_C_en_salida -> vehiculo_GNL_o_GNC -> RL_o_RC_en_otra_ubicacion_o_cliente',
        'gasbc_corto_P_E_C_hacia_NGI': True,
        'gasbc_corto_red_P_E_C_NGI_generalizada': True,
        'casos_gasbc_generalizados': [
            'P -> NGI -> C', 'P/C/NGI -> E -> uno_o_varios_clientes',
            'C -> P', 'C -> E', 'C -> NGI', 'NGI -> P/E/C/NGI',
        ],
        'biogas_crudo_cliente_hacia_P_E_NGI': True,
        'vehiculo_L_C_desde_cliente_hacia_RL_RC_en_P_E_NGI': True,
        'vehiculo_maximo_destinos': 2,
        'mezcla_GasBC_corredor_y_gasoducto_corto_en_L_C_NGI': True,
        'detalle_mezcla_NGI': 'GasBC extraido del corredor + GasBC recibido por gasoducto corto desde P/E/C/NGI -> mismo pool NGI -> misma L/C, sujeto a una sola capacidad total de planta',
        'multiples_BG_hacia_una_misma_L_C': True,
    },
}
(DIR_RESULTADOS / 'Configuracion_Modelo.json').write_text(
    json.dumps(configuracion_modelo, ensure_ascii=False, indent=2), encoding='utf-8'
)
factores_conversion.to_csv(DIR_RESULTADOS / "Factores_Conversion_Utilizados.csv", index=False)
df_clientes_resultado.to_csv(DIR_RESULTADOS / "Clientes_Demanda_Satisfecha.csv", index=False)
df_empresas_resultado.to_csv(DIR_RESULTADOS / "Empresas_Produccion_Activacion.csv", index=False)
empresas_excluidas_mayor_oferta.to_csv(
    DIR_RESULTADOS / "Empresas_Excluidas_Mayor_Oferta.csv", index=False
)
df_plantas_resultado.to_csv(DIR_RESULTADOS / "Plantas_Activas.csv", index=False)
df_gasbc_bg_a_l_c.to_csv(DIR_RESULTADOS / "GasBC_desde_BG_hacia_L_C.csv", index=False)
df_gasbc_gasoducto_a_l_c.to_csv(
    DIR_RESULTADOS / "GasBC_desde_Gasoducto_hacia_L_C_en_Salida.csv", index=False
)
df_gasoductos_resultado.to_csv(DIR_RESULTADOS / "Gasoductos_Utilizacion.csv", index=False)
distancias_entre_gasoductos_36.to_csv(DIR_RESULTADOS / "Distancias_Entre_9_Gasoductos_36.csv", index=False)
df_transferencias_gasoducto_gasoducto.to_csv(DIR_RESULTADOS / "Transferencias_Gasoducto_Gasoducto.csv", index=False)
df_recorridos_internos_gasoducto.to_csv(DIR_RESULTADOS / "Recorridos_Internos_Gasoducto_AB_BA.csv", index=False)
gasoductos_biogas_cortos_todos.to_csv(
    DIR_RESULTADOS / "Gasoductos_Biogas_Crudo_Candidatos_Calculados_2km.csv",
    index=False,
)
df_gasoductos_biogas_cortos_resultado.to_csv(
    DIR_RESULTADOS / "Gasoductos_Biogas_Crudo_Utilizacion.csv",
    index=False,
)
gasoductos_biogas_cortos_hub_empresa.to_csv(
    DIR_RESULTADOS / "Gasoductos_Biogas_Crudo_HUB_Empresa_Candidatos.csv", index=False
)
df_gasoductos_biogas_hub_empresa_solo_resultado.to_csv(
    DIR_RESULTADOS / "Gasoductos_Biogas_Crudo_HUB_Empresa_Utilizacion_Exclusiva.csv", index=False
)
gasoductos_gasbc_cortos_empresa_entrada_todos.to_csv(
    DIR_RESULTADOS / "Gasoductos_GasBC_Empresa_Entrada_Candidatos.csv",
    index=False,
)
gasoductos_biogas_cortos_cliente_cliente.to_csv(
    DIR_RESULTADOS / 'gasoductos_biogas_crudo_cliente_cliente_candidatos.csv',index=False,encoding='utf-8-sig'
)
df_gasoductos_biogas_cortos_cliente_cliente_resultado.to_csv(
    DIR_RESULTADOS / 'gasoductos_biogas_crudo_cliente_cliente_resultado.csv',index=False,encoding='utf-8-sig'
)
gasoductos_biogas_cortos_hub_p.to_csv(
    DIR_RESULTADOS / 'gasoductos_biogas_crudo_hub_P_candidatos.csv',index=False,encoding='utf-8-sig'
)
df_gasoductos_biogas_cortos_hub_p_resultado.to_csv(
    DIR_RESULTADOS / 'gasoductos_biogas_crudo_hub_P_resultado.csv',index=False,encoding='utf-8-sig'
)

df_gasoductos_gasbc_cortos_empresa_entrada_resultado.to_csv(
    DIR_RESULTADOS / "Gasoductos_GasBC_Empresa_Entrada_Utilizacion.csv",
    index=False,
)
gasoductos_gasbc_cortos_empresa_destino_todos.to_csv(
    DIR_RESULTADOS / "Gasoductos_Cortos_GasBC_Empresa_Destino_Candidatos.csv",
    index=False,
)
df_gasoductos_gasbc_cortos_empresa_destino_resultado.to_csv(
    DIR_RESULTADOS / "Gasoductos_Cortos_GasBC_Empresa_Destino_Resultados.csv",
    index=False,
)
df_gasoductos_gasbc_cortos_salida_cliente_resultado.to_csv(
    DIR_RESULTADOS / "Gasoductos_GasBC_Salida_Cliente_Utilizacion.csv",
    index=False,
)
df_viajes_transporte.to_csv(DIR_RESULTADOS / "Viajes_Transporte_Directos_y_Dos_Paradas.csv", index=False)
df_arcos_transporte.to_csv(DIR_RESULTADOS / "Arcos_Viajes_Transporte.csv", index=False)
df_decisiones_transporte.to_csv(DIR_RESULTADOS / "Descargas_por_Viaje_Transporte.csv", index=False)
df_sistemas_independientes.to_csv(DIR_RESULTADOS / "Sistemas_Independientes.csv", index=False)
df_reparto_arcos.to_csv(DIR_RESULTADOS / "Reparto_Multicliente_Arcos.csv", index=False)
df_decisiones_reparto.to_csv(DIR_RESULTADOS / "Reparto_Multicliente_Decisiones_Continuar_o_Regresar.csv", index=False)
df_giras_reparto.to_csv(DIR_RESULTADOS / "Reparto_Multicliente_Giras_Reconstruidas.csv", index=False)
df_rutas.drop(columns=["segmentos","segmentos_gasbc_intermedio","segmentos_pre_reconversion"], errors="ignore").to_csv(
    DIR_RESULTADOS / "Rutas_Optimas_Descompuestas.csv", index=False
)
if "df_rutas_estadisticas" in globals():
    df_rutas_estadisticas.drop(
        columns=["_producer_keys", "_producer_weights", "_producer_cost_parts", "_producer_variable_parts", "_producer_fixed_parts", "segmentos_gasbc_intermedio", "segmentos_pre_reconversion"],
        errors="ignore",
    ).to_csv(DIR_RESULTADOS / "Rutas_Optimas_Con_Economia_Atribuida.csv", index=False)
if "df_auditoria_prorrateo_costos_fijos_empresas" in globals():
    df_auditoria_prorrateo_costos_fijos_empresas.to_csv(
        DIR_RESULTADOS / "Auditoria_Prorrateo_Costos_Fijos_Empresas.csv", index=False
    )
if "df_auditoria_prorrateo_costos_fijos_productoras" in globals():
    df_auditoria_prorrateo_costos_fijos_productoras.to_csv(
        DIR_RESULTADOS / "Auditoria_Prorrateo_Costos_Fijos_Productoras.csv", index=False
    )
if "df_metricas_empresas" in globals():
    df_metricas_empresas.to_csv(DIR_RESULTADOS / "Metricas_Empresas_Visualizador.csv", index=False)
if "df_metricas_clientes" in globals():
    df_metricas_clientes.to_csv(DIR_RESULTADOS / "Metricas_Clientes_Visualizador.csv", index=False)
if "df_metricas_productoras" in globals():
    df_metricas_productoras.to_csv(
        DIR_RESULTADOS / "Metricas_Productoras_Gasoductos_Visualizador.csv", index=False
    )
pd.DataFrame([kpis]).to_csv(DIR_RESULTADOS / "Estadisticas_Generales.csv", index=False)
(DIR_RESULTADOS / "Estadisticas_Generales.json").write_text(
    json.dumps(kpis, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

# Copia la precarga completa utilizada por esta solución.
cache_result = DIR_RESULTADOS / "Cache_Google_Routes_COMPLETA"
if cache_result.exists():
    shutil.rmtree(cache_result)
shutil.copytree(DIR_CACHE, cache_result)

# Empaqueta nuevamente la caché local y los datos utilizados.
# IMPORTANTE: los gasoductos cortos empresa-cliente y empresa-ubicación
# posible NO se añaden a BASE_DATOS,
# al directorio de caché ni a DATA_WITH_CACHE. Sólo se exportan como resultados
# auditables y se vuelven a calcular en cada ejecución del notebook.
CACHE_ZIP = CONTENT / NOMBRE_ZIP_CACHE_API
if CACHE_ZIP.exists():
    CACHE_ZIP.unlink()
with zipfile.ZipFile(CACHE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in DIR_CACHE.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(DIR_CACHE))

DATA_WITH_CACHE = CONTENT / NOMBRE_ZIP_DATOS_CON_CACHE
staging = CONTENT / "_staging_data_cache_final"
if staging.exists():
    shutil.rmtree(staging)
shutil.copytree(BASE_DATOS, staging / CARPETA_RAIZ_DATOS)
api_dst = staging / CARPETA_RAIZ_DATOS / "Archivos_Datos_API"
api_dst.mkdir(parents=True, exist_ok=True)
for p in DIR_CACHE.iterdir():
    if p.is_file():
        shutil.copy2(p, api_dst / p.name)
if DATA_WITH_CACHE.exists():
    DATA_WITH_CACHE.unlink()
with zipfile.ZipFile(DATA_WITH_CACHE, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in staging.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(staging))
shutil.rmtree(staging)

RESULT_ZIP = CONTENT / "Resultados_Biometano_Modelo_PDF_COMPLETOS.zip"
if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()
with zipfile.ZipFile(RESULT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in DIR_RESULTADOS.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(DIR_RESULTADOS))

print("Archivos finales:")
for p in [HTML_PATH, RESULT_ZIP, CACHE_ZIP, DATA_WITH_CACHE]:
    print(f" - {p} ({p.stat().st_size / 1024 / 1024:.2f} MiB)")

if EN_COLAB:
    print("Descargando resultados completos...")
    files.download(str(RESULT_ZIP))
    print("El HTML y el ZIP de datos actualizado quedan disponibles en /content.")


Archivos finales:
 - /content/Resultados_Biometano_Modelo_PDF/Visualizador_Biometano_Modelo_PDF.html (1.54 MiB)
 - /content/Resultados_Biometano_Modelo_PDF_COMPLETOS.zip (286.93 MiB)
 - /content/Cache_Google_Routes_Biometano_COMPLETA.zip (286.11 MiB)
 - /content/Datos_Biometano_Corredores_Bidireccionales_TRANSPORTE_VARIABLE_COMPLETO.zip (286.50 MiB)
Descargando resultados completos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

El HTML y el ZIP de datos actualizado quedan disponibles en /content.


# FInal



1. Sube el ZIP de datos desde el panel **Archivos** de Colab.
2. Ejecuta las celdas en orden.
3. El notebook valida las matrices y trayectorias E2E, E2C y C2C del ZIP; los 120 pares E2C deben estar completos antes de construir el MILP.
4. El solver recibe una sola vez el MILP completo.
5. Con la configuración incluida, los vehículos quedan disponibles para BioGNL/BioGNC; biogás crudo y GasBC no pueden usar vehículos.
6. Cada ruta dispone de variables `QSEG` para la carga de salida, la carga entre paradas y la carga de retorno.
7. En cada parada se verifica exactamente: `carga que llega = cantidad transformada/entregada + remanente`.
8. La cantidad transformada o entregada en una parada es continua y mayor o igual que cero.
9. El remanente pasa a la siguiente parada; después de la última, puede regresar al origen.
10. Los alias viales de GasBC se conservan estructuralmente para no alterar la formulación, pero su uso vehicular queda fijado a cero; GasBC sólo se transporta por gasoducto corto o convencional, además de transferencias locales.
11. Los viajes compartidos entre clientes usan las claves dirigidas `C2C|cliente_a|cliente_b`.
12. El CSV `Descargas_por_Viaje_Transporte.csv` informa carga de entrada, transformación/entrega, remanente y destino del remanente en cada parada.
13. El CSV `Arcos_Viajes_Transporte.csv` informa la carga real `QSEG` en todos los tramos, incluido el retorno.
14. El HTML muestra la gira real del vehículo, la carga de cada tramo, la cantidad procesada en cada parada y el remanente que continúa o retorna.

15. En rutas compartidas activas se exige descarga mínima por parada; cada `QTR` sigue sujeto al balance físico de carga.


El notebook calcula los candidatos de gasoducto corto de biogás crudo antes del MILP y exporta su auditoría en los resultados. No modifica el ZIP de datos ni agrega estas líneas a la caché de Google Routes.

16. También se permite `Empresa -> BG en la misma empresa -> GasBC -> gasoducto corto -> entrada del corredor principal`; ese gasoducto corto transporta **GasBC**, no biogás crudo.

17. En el panel de rutas, cada ruta representa **un único cliente final**. Los clientes intermedios por los que pasa una infraestructura compartida no se dibujan como un segundo cliente de esa ruta.
18. La economía mostrada por ruta es: **ingreso por venta al cliente final − costos variables atribuibles a esa ruta**. Los costos fijos no se restan del margen operativo por ruta.
19. Si un gasoducto de biogás crudo comparte un tramo y deja parte del flujo en una ubicación intermedia con BG y cliente local, el punto numerado de esa parada muestra al pasar el mouse la cantidad de biogás crudo dejada allí y la venta local asociada, sin sumar esa venta a los ingresos de la ruta cuyo cliente final está aguas abajo.
